<a href="https://colab.research.google.com/github/eameyaw/CD133-Machine-Learning-Models./blob/main/prostate_surfaceome_pipeline_(33).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Surfaceome-Wide Multi-Omic Pipeline and Ensemble Machine Learning Framework for Prioritising Actionable Prostate Cancer Cell-Surface Antigens

**End-to-end reproducible pipeline — Google Colab**

---

### What this notebook does

Integrates primary prostate adenocarcinoma, metastatic castration-resistant prostate cancer (mCRPC), proteomic, and normal-tissue cohorts to prioritise cell-surface antigens that are selectively targetable in advanced prostate cancer — with the clinical gap (PSMA-negative, heterogeneous and neuroendocrine disease) as the organising question.

**Implements the integrated eight-point methodological core plus four validation safeguards, per the Final AACR Research Proposal (18 Nov 2026) and its predecessor (17 Nov 2026).**

### The eight-point core, and where each lives

| # | Component | Section |
|---|---|---|
| 1 | Harmonised expression; explicit log2(TPM+0.001) inversion before TPM ratios | 1.3, `xena_log_to_tpm` |
| 2 | Prespecified biological screen — DE tests, BH-FDR family, effect and prevalence thresholds | 2.6 |
| 3 | Extracellular accessibility required; binder availability tracked separately as readiness | 2.6 |
| 4 | Protein evidence as a grade in primary tumour; metastatic findings labelled transcript-only | tiers |
| 5 | Positive-unlabelled modelling; AUPRC/ROC-AUC reported as proxy-label metrics only | 3.3 |
| 6 | Evaluation by held-out benchmark recovery against baselines and a permutation null | 3.6, 3.11 |
| 7 | Full ranking and novel shortlist published separately, with ranks, uncertainty and tiers | 3.10 |
| 8 | Descriptive ancestry analysis with group counts and intervals; no unsupported subgroup Cox | 4.1–4.3 |

### The four validation safeguards

| # | Safeguard | Section | When the data is absent |
|---|---|---|---|
| S1 | **Verified clinical-stage labels** (trial ID, modality, phase, status, freeze date) | 0 | Reports the unverified fraction; a STOP condition in LIVE mode |
| S2 | **Comparison with simple biological baselines** | 3.11, Fig 11 | Always runs — needs no extra data |
| S5 | **Benchmark provenance** (accessibility + modality evidence, frozen set) | 0 | Reports the unverified fraction |
| S3 | **Independent cohort testing with a frozen model** | 3.9 | Skips loudly; recorded as not performed |
| S4 | **Single-cell / surface-protein assessment** | 3.12 | Skips loudly; recorded as not performed |

S2 answers the first question a methods reviewer asks: *what does the ensemble add over sorting by fold-change?* Four transparent baselines — differential expression alone, statistical confidence alone, tumour selectivity alone, and an equal-weight score — are scored on the **same** primary endpoint.

**The improvement rule is predefined in `Config`, before any outcome is examined**, and requires *both* a margin on the continuous primary metric (default ≥ 3 percentile points) *and* more than a one-target gain in top-decile recovery (default ≥ 2). With only six held-out positives a single-target difference is noise, so a margin alone is not evidence. The comparison is also not adjudicated at all unless ranking stability clears ρ ≥ 0.80 — comparing two unstable rankings answers nothing. **If the ensemble does not meet the rule, the gate STOPS and tells you to report the simplest adequate baseline as the primary method.**

In the synthetic test run the ensemble *failed* this rule (+2.99 points, +1 target), which is the correct and useful outcome: the rule is strict enough to bite.

S3 and S4 skip loudly rather than silently producing nothing, and a skipped safeguard is recorded in `run_summary.json` as not performed. An analysis that was never run and one that returned nothing look identical in a results table, and only one is honest to describe as a limitation.

### Evidence status — read before quoting any number

This notebook has been executed end to end in **DEMO** mode against synthetic data. That proves the software runs; it establishes **nothing** about biology. The demo's ROC-AUC, candidate rankings, benchmark ranks, cohort sizes and survival estimates must never appear in an abstract or proposal as empirical findings. A LIVE run with independently verified data provenance is a prerequisite for any results claim.

### How to run it

1. `Runtime → Change runtime type → High-RAM` if available (the LIVE transcriptome matrices are large).
2. **Run every cell in order with `RUN_MODE = "DEMO"` first.** On the default `smoke` profile this takes about 3–5 minutes on synthetic stand-ins with the same schema as the real data. It proves the analysis code works before you spend an hour on downloads.
3. Then set `RUN_MODE = "LIVE"` and re-run. Verify the URLs in `DATA_URLS` first — public portals move their files without notice.
4. For the run you report, set `PROFILE = "manuscript"`.

### Compute profiles — measured, not estimated

| Profile | Bags | Permutations | Runtime (2-vCPU Colab) | Use for |
|---|---|---|---|---|
| `smoke` | 3 | 30 | **~3–5 min** | Proving the code runs. Default in DEMO |
| `standard` | 12 | 200 | **~35–50 min** | A first LIVE run; usable intervals |
| `manuscript` | 50 | 1000 | **~3–5 h** | The run you actually report |

Smoke-profile statistics are deliberately under-powered — wide intervals, unstable rankings — and the notebook says so at startup. **Never report smoke numbers.**

> An earlier version of this notebook claimed a 4-minute demo while its settings actually took ~37 minutes on hardware faster than Colab's. The profiles exist so the claim and the code cannot drift apart again.

### What DEMO mode is and is not

DEMO generates synthetic cohorts with planted structure: known targets are *enriched for* tumour upregulation and vital-organ sparing, but they overlap the background substantially, and about a quarter of them are deliberately given no advantage at all. This makes DEMO a fair rehearsal of the analysis rather than a flattering one.

**DEMO mode produces no biological results.** Its outputs are a test of the code, nothing more. Three properties make it useful as a test:

- Four control genes are each engineered to fail exactly one gatekeeping filter, so the filter block is unit-tested on every run.
- Four more controls (`KLK3`, `ACTB`, `TP53`, `AR`) must be removed at surfaceome bounding, and bounding asserts it.
- No expression–survival association is planted, so the Cox models should come back null. If they do not, the survival code is manufacturing significance.

### Outputs

Everything lands under `outputs/` — raw downloads, interim files, the master feature table, ten figures, twelve tables, and a machine-readable `run_summary.json` that records every headline number plus which components were skipped.

### Stop/go gates

Section 7 produces **five separate dispositions**, not one verdict — GO for LIVE analysis, GO for the ensemble as primary method, GO for biological claims, GO for stronger generalisation claims, and GO for an AACR results abstract. A run can be sound enough to proceed with analysis while remaining nowhere near ready for a results-based abstract, and collapsing those into a single verdict is how an exploratory run turns into a submission.

It also evaluates the declared STOP conditions automatically: wrong expression scale, gene-ID loss, failed filter unit tests, missing declared organs, unverifiable positive labels, training/held-out contamination, near-null benchmark enrichment, performance collapse after removing fame features, ranking instability, and insufficient subgroup events. A STOP means the evidence does not support a discovery claim — the correct response is to fix the input or report the null result, not to loosen a threshold until the gate opens.

---
## Section -1 — Colab setup (run this first)

Installs the packages Colab does not ship, and optionally mounts Drive so a
dropped session does not force a re-download of several GB of cohort data.
Safe to run anywhere: outside Colab it installs nothing and mounts nothing.

In [1]:
# =============================================================================
# COLAB SETUP - run this cell first
# =============================================================================
# Colab ships numpy/pandas/scipy/sklearn/statsmodels/seaborn/matplotlib/joblib,
# but NOT lifelines (survival), pydeseq2 (count-based DE) or cptac (proteomics).
# Without lifelines the survival section cannot run at all; without pydeseq2 the
# pipeline falls back to an unregularised per-gene NB GLM and says so in the
# output. Versions are pinned to the ones the archived DEMO release was built
# with, so a Colab run and the archive are comparable.
import importlib, subprocess, sys, os

# =============================================================================
# CPU THREAD BUDGET - FIRST, before anything imports numpy
# =============================================================================
# BUG FIX (both external reviews, correct, and my own fix last round was
# only half-effective because of exactly this). Section 0 sets these
# variables before ITS scientific imports - but _ensure() below probes for
# lifelines by IMPORTING it, and lifelines imports numpy. BLAS reads these
# variables once, at load time, so by the time Section 0 ran they were
# already too late. The measured 305s -> 243s improvement last round came
# from removing the nested joblib pools, NOT from thread pinning, which was
# silently doing nothing in the notebook.
#
# Explicit assignment, NOT setdefault: Colab sometimes pre-sets these to the
# full core count, and setdefault preserves exactly the oversubscription
# this is meant to prevent.
for _tv in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_tv] = os.environ.get("BLAS_THREADS", "1")
os.environ.setdefault("N_JOBS", str(min(os.cpu_count() or 2, 8)))
# Record whether the pinning could take effect, HERE, where it happens -
# Section 0 runs in a later cell and cannot observe this, because the
# dependency probe below legitimately imports numpy.
_pinned_early = not any(m in sys.modules
                        for m in ("numpy", "scipy", "sklearn"))
os.environ["BLAS_PINNED_EARLY"] = "1" if _pinned_early else "0"

# BUG FIX, found by actually running this cell on Colab rather than only in
# a plain kernel: Colab's own managed kernel has numpy/scipy/sklearn ALREADY
# in sys.modules before this cell - the FIRST cell in the notebook - ever
# executes. That means _pinned_early is False and the env-var approach above
# is a no-op on every single Colab run, with no ordering fix available: the
# "restart the runtime and run this cell first" advice previously printed
# here is not a remedy, because the precondition it asks for (no numeric
# library loaded yet) is already false before the user's own first
# keystroke. A check built on "were the variables set before import" can
# never pass in the one environment this notebook exists to run in.
#
# Environment variables only work because BLAS backends read them ONCE, at
# library-load time - which is exactly what makes them fragile to load
# order. threadpoolctl instead walks the libraries that are ALREADY loaded
# and rewrites their live thread-count entry points, so it works regardless
# of what imported numpy first or when. Applying it here, keeping the
# returned controller alive for the rest of the process, makes the actual
# thing this section cares about - how many threads BLAS calls will use -
# true regardless of Colab's preloading, instead of merely true when the
# cell happens to run in a pristine kernel.
try:
    importlib.import_module("threadpoolctl")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "threadpoolctl"], check=True)
import threadpoolctl
_THREAD_LIMIT_CONTROLLER = threadpoolctl.threadpool_limits(
    limits=int(os.environ.get("BLAS_THREADS", "1")))
if not _pinned_early:
    print("NOTE: numpy/scipy/sklearn were ALREADY imported by Colab's own "
          "kernel before this\n"
          "      cell ran, so the OMP_NUM_THREADS-style variables above are "
          "a no-op here.\n"
          "      threadpoolctl was used instead to cap the already-loaded "
          "libraries directly;\n"
          "      Section 7's regression check verifies the REAL effective "
          "thread count, not\n"
          "      import order, for exactly this reason.")
print(f"threads: BLAS={os.environ['OMP_NUM_THREADS']} "
      f"joblib workers={os.environ['N_JOBS']} "
      f"cpus={os.cpu_count()}")

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

_PINNED = {           # import name -> pip requirement
    "lifelines":  "lifelines==0.30.3",
    # pydeseq2 is OPTIONAL and intentionally unpinned: it resolves against
    # the local numpy/pandas, which Colab updates often. Note for anyone
    # reading the lockfile: environment_lock.txt records pydeseq2==0.6.0
    # because that is what the archived DEMO was built with, and pydeseq2
    # 0.6.x requires Python >= 3.12. That is a constraint on the COUNT-BASED
    # differential-expression path, not on the pipeline: without pydeseq2
    # the pipeline falls back to an unregularised per-gene NB GLM and
    # labels every affected output as such. An external review read the
    # lockfile as evidence that the pipeline requires 3.12+; it does not.
    "pydeseq2":   "pydeseq2",
    # cptac is NOT installed by default. Found by executing this notebook:
    # the pipeline never imports cptac - CPTAC proteomics is supplied as a
    # file - but probing for it has a SIDE EFFECT. A cptac import without
    # network raises NoInternetError and leaves a warning filter holding
    # cptac.exceptions.CptacWarning. In a notebook the parallel ML stage is
    # serialised by cloudpickle, which then fails with
    #   PicklingError: Can't pickle cptac.exceptions.CptacWarning
    # and the whole run dies at [3.3] Bagged PU learning. Installing an
    # optional package that nothing imports should not be able to break the
    # model fitting, so it is opt-in.
    # Set INSTALL_CPTAC = True only if you intend to pull proteomics through
    # the package rather than supplying PROTEOMICS_LOCAL_FILE.
}
INSTALL_CPTAC = False
if INSTALL_CPTAC:
    _PINNED["cptac"] = "cptac"
_OPTIONAL = {"cptac", "pydeseq2"}

def _ensure(pkgs=_PINNED):
    # Probing for a package IMPORTS it, and a failed import can leave
    # residue behind - cptac registers a warning filter referencing an
    # unpicklable exception class, which later breaks joblib's workers.
    # The filter list is snapshotted and restored so a presence check
    # cannot change how the rest of the run behaves.
    import warnings as _w
    _filters_before = list(_w.filters)
    missing = {}
    for mod, req in pkgs.items():
        try:
            importlib.import_module(mod)
        except Exception:
            missing[mod] = req
    _w.filters[:] = _filters_before
    if not missing:
        print("all analysis packages already present")
        return
    print(f"installing: {', '.join(missing.values())}")
    for mod, req in missing.items():
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", req],
                           capture_output=True, text=True)
        if r.returncode != 0:
            level = "OPTIONAL" if mod in _OPTIONAL else "REQUIRED"
            tail = r.stderr.strip().splitlines()[-1] if r.stderr else ""
            print(f"  {level} package {mod} FAILED to install:\n    {tail}")
            if level == "REQUIRED":
                # BUG FIX (all three external reviews, correct): a failed
                # REQUIRED install only PRINTED. On Colab that message
                # scrolls away in seconds, the run proceeds, and the
                # survival analysis is silently absent from a run that
                # otherwise looks complete - the exact "cannot distinguish
                # clean from absent" failure this codebase has been pruning
                # everywhere else. Setup must stop.
                raise RuntimeError(
                    f"REQUIRED package {mod!r} could not be installed "
                    f"({req}). The survival analysis cannot run without it, "
                    f"and a run that silently omits it would look complete. "
                    f"Fix the environment and re-run this cell.\n"
                    f"  pip said: {tail}")
        else:
            print(f"  installed {req}")

_ensure()

# --- Optional: cache raw downloads and outputs on Drive ----------------------
# The LIVE cohorts total several GB. Colab sessions are ephemeral, so without
# a persistent cache a disconnect means downloading them again. Mounting Drive
# makes a re-run resume from the cached files instead.
USE_DRIVE = False          # <-- set True on Colab to cache on Drive
DRIVE_DIR = "/content/drive/MyDrive/surfaceome_pipeline"

# SPEED FIX (both external reviews, citing Google's own Colab guidance, and
# correct): this used to point SURFACEOME_OUT_DIR at Drive as well, so every
# figure, table, log and model was written through the Drive FUSE mount. That
# mount is an order of magnitude slower than the VM's local disk for the many
# small writes this pipeline makes, and intermittent FUSE stalls on a long
# run look exactly like a hung pipeline.
#
# The right split is: large, expensive, REUSED inputs persist on Drive
# (downloading 5 GB again after a disconnect is the cost worth avoiding);
# everything the run computes goes to fast local disk; the finished archive
# is copied to Drive at the end, where it is small and written once.
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/00_raw", exist_ok=True)
    # Raw downloads and the filtered-matrix cache persist across sessions.
    os.environ["SURFACEOME_RAW_DIR"] = f"{DRIVE_DIR}/00_raw"
    # Outputs stay on LOCAL disk. SYNC_ZIP_TO_DRIVE copies the finished
    # archive over at the end.
    os.environ["SURFACEOME_OUT_DIR"] = "/content/surfaceome_outputs"
    os.makedirs("/content/surfaceome_outputs", exist_ok=True)
    SYNC_ZIP_TO_DRIVE = True
    print(f"Drive cache (inputs, persistent): {DRIVE_DIR}/00_raw")
    print("Outputs (fast local disk):        /content/surfaceome_outputs")
    print(f"Finished archive will be copied to {DRIVE_DIR}")
elif USE_DRIVE:
    SYNC_ZIP_TO_DRIVE = False
    print("USE_DRIVE is set but this is not Colab - ignoring")
else:
    SYNC_ZIP_TO_DRIVE = False

# --- Run mode ----------------------------------------------------------------
# DEMO (default): synthetic data, ~3 min, no network. Proves the code runs.
# LIVE: downloads the real cohorts. Requires network access to
#       rest.uniprot.org, toil-xena-hub.s3.us-east-1.amazonaws.com,
#       cbioportal-datahub.s3.amazonaws.com and www.proteinatlas.org.
#       These are reachable from Colab; they are NOT reachable from many
#       corporate networks or restricted sandboxes.
os.environ["RUN_MODE"] = os.environ.get("RUN_MODE", "DEMO")   # "DEMO" | "LIVE"

# BUG FIX (all three external reviews, correct). This cell used to read
#     os.environ["PROFILE"] = os.environ.get("PROFILE", "smoke")
# which HARDCODED the smoke profile into the environment before the pipeline
# ran. The pipeline's own default is already correct -
#     PROFILE = os.environ.get("PROFILE", "smoke" if RUN_MODE=="DEMO"
#                              else "standard")
# - so this cell was actively overriding a correct default with a wrong one,
# and a user who set RUN_MODE=LIVE and nothing else got a LIVE run on 3 PU
# bags and 30 permutations. That run completes, produces plausible-looking
# numbers, and is not reportable. The comment also offered "full", which is
# not a valid profile at all and raises on startup.
#
# The default is now left UNSET so the pipeline decides, and anything the
# user does set is validated here, before the long run starts.
_VALID_PROFILES = {"smoke", "standard", "manuscript"}
if os.environ.get("PROFILE"):
    if os.environ["PROFILE"].lower() not in _VALID_PROFILES:
        raise ValueError(
            f"PROFILE={os.environ['PROFILE']!r} is not valid. Choose one of "
            f"{sorted(_VALID_PROFILES)}. (There is no 'full' profile.)")
    _profile_shown = os.environ["PROFILE"].lower()
else:
    _profile_shown = "smoke" if os.environ["RUN_MODE"] == "DEMO" else "standard"

# "analysis" (default) runs the whole pipeline. "data_only" acquires and
# validates the real datasets, then stops before any model is fitted - use it
# to shake down the LIVE loaders while the label registry is still being
# curated. Nothing a data_only run writes is reportable.
os.environ["RUN_PURPOSE"] = os.environ.get("RUN_PURPOSE", "analysis")

print(f"RUN_MODE={os.environ['RUN_MODE']}  PROFILE={_profile_shown}"
      f"  RUN_PURPOSE={os.environ['RUN_PURPOSE']}")
if os.environ["RUN_MODE"] == "LIVE" and _profile_shown == "smoke":
    print("  WARNING: smoke profile on real data is a software diagnostic, "
          "not a reportable analysis.")

if os.environ["RUN_MODE"] == "LIVE":
    print(
        "\nLIVE preflight - read before running:\n"
        "  * Network: the four hosts above must be reachable. If any is\n"
        "    blocked the run stops at the first fetch with a message naming\n"
        "    the host and what to do.\n"
        "  * Disk: ~5 GB of downloads. The Toil matrix is streamed in chunks\n"
        "    and filtered to surfaceome rows, so peak RAM stays modest, but\n"
        "    the compressed files themselves need room.\n"
        "  * Time: substantially longer than DEMO, dominated by downloads.\n"
        "  * Provenance: the run begins with a label and benchmark audit. It\n"
        "    will report ACTION REQUIRED until LABEL_PROVENANCE is populated\n"
        "    and BENCHMARK_SET_FROZEN_ON is set. The pipeline still runs, but\n"
        "    the stop/go gate will not pass the run as reportable - which is\n"
        "    the intended behaviour, not a bug.\n"
        "  * Set USE_DRIVE = True above if you want downloads cached across\n"
        "    session restarts.\n")

NOTE: numpy/scipy/sklearn were ALREADY imported by Colab's own kernel before this
      cell ran, so the OMP_NUM_THREADS-style variables above are a no-op here.
      threadpoolctl was used instead to cap the already-loaded libraries directly;
      Section 7's regression check verifies the REAL effective thread count, not
      import order, for exactly this reason.
threads: BLAS=1 joblib workers=2 cpus=2
installing: lifelines==0.30.3, pydeseq2
  installed lifelines==0.30.3
  installed pydeseq2
RUN_MODE=DEMO  PROFILE=smoke  RUN_PURPOSE=analysis


---
## Section 0 — Environment, configuration and benchmark gene sets

Every threshold used anywhere in the analysis is declared once, in the `Config` dataclass below. Nothing downstream hard-codes a cutoff. This is what makes the numbers in a manuscript traceable to a single declared source rather than scattered through the code.

In [2]:
# =============================================================================
# SECTION 0 - ENVIRONMENT SETUP
# =============================================================================
# Surfaceome-Wide Multi-Omic Pipeline for Prostate Cancer Cell-Surface Antigens
#
# RUN MODE
#   RUN_MODE = "DEMO" -> generates structurally faithful synthetic stand-ins for
#                        every cohort. Runs in ~3 min, no downloads, no auth.
#                        Use this FIRST to verify the whole pipeline executes.
#   RUN_MODE = "LIVE" -> downloads the real public cohorts. TCGA/GTEx/SU2C/HPA
#                        are open access and download without credentials.
#                        CPTAC proteomics is fetched via the `cptac` package.
#
# WHY A DEMO MODE EXISTS: the real matrices are 1-4 GB and the download step is
# the single most common point of failure (URL rot, Colab disk limits, portal
# throttling). Separating "does my code work" from "did my download work" means
# a failure tells you which of the two broke.
# =============================================================================

import os
import sys
import time
import gzip
import json
import shutil
import tarfile
import warnings
import subprocess
from pathlib import Path
from dataclasses import dataclass, field

# =============================================================================
# CPU THREAD BUDGET - must be set BEFORE numpy/scipy/sklearn are imported,
# because the BLAS backends read these variables once, at load time.
# =============================================================================
# SPEED FIX (both external reviews raised oversubscription; confirmed by
# reading the call sites). There are five n_jobs=-1 sites, and two of them
# NEST: a GridSearchCV with n_jobs=-1 whose estimator is a RandomForest with
# n_jobs=-1, each of whose workers then runs BLAS on all cores. On a 2-vCPU
# Colab instance that is 2 x 2 x 2 = 8 threads contending for 2 cores. The
# result is not a speed-up: the workers fight for cache and the scheduler,
# and every joblib worker gets its own copy of the data, so PEAK MEMORY
# rises too - which on a free Colab instance is the difference between
# finishing and being killed.
#
# One thread per BLAS call, and joblib parallelism bounded to the real core
# count, is both faster and lighter here. Overridable for a big machine.
# Whether the pinning below can actually take effect. BLAS backends read
# these variables once, when the numeric stack loads, so if numpy is ALREADY
# imported the pinning is a no-op. This flag records that fact at the only
# moment it can be observed, and a regression test asserts it - which is how
# the notebook's silently-ineffective pinning was caught: Section 0 ran
# after _ensure() had imported lifelines, and lifelines imports numpy.
# SELF-REVIEW, caught by this very check failing on a CORRECT notebook.
# The first version computed
#     not any(m in sys.modules for m in ("numpy","scipy","sklearn"))
# right here, in Section 0. But in the notebook Section 0 runs in a LATER
# cell than the setup cell, and the setup cell legitimately imports numpy
# while probing for lifelines. So the flag read False even though the
# pinning had been applied correctly, one cell earlier, before any import -
# the check was measuring "was numpy loaded when Section 0 ran", not "were
# the variables set before numpy loaded". A guard that cannot distinguish
# correct from incorrect behaviour is worse than none: this one blocked a
# good release.
#
# The fact can only be observed where the pinning happens, so whoever pins
# the variables records it. The notebook's first cell sets
# BLAS_PINNED_EARLY after checking sys.modules itself; when that is absent
# (the standalone runner, where this file IS the first thing loaded) the
# check is computed here, which is then the correct place for it.
if os.environ.get("BLAS_PINNED_EARLY") in ("0", "1"):
    THREADS_PINNED_BEFORE_NUMPY = os.environ["BLAS_PINNED_EARLY"] == "1"
else:
    THREADS_PINNED_BEFORE_NUMPY = not any(
        _m in sys.modules for _m in ("numpy", "scipy", "sklearn"))
_N_CPU = os.cpu_count() or 2
N_JOBS = int(os.environ.get("N_JOBS", str(max(1, min(_N_CPU, 8)))))
_BLAS_THREADS = os.environ.get("BLAS_THREADS", "1")
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, _BLAS_THREADS)

# BUG FIX, found by actually executing the shipped NOTEBOOK on Colab rather
# than only this standalone path: THREADS_PINNED_BEFORE_NUMPY is an inference
# from import order, and on Colab numpy/scipy/sklearn are already loaded by
# the kernel itself before even the notebook's first cell runs - so the
# inference is False on every Colab run, with no ordering fix available, even
# though threading can still be capped. Environment variables only take
# effect at library LOAD time, so once a BLAS backend is loaded they do
# nothing further; threadpoolctl instead walks the libraries already loaded
# in this process and rewrites their live thread-count entry points, so it
# works regardless of load order. Apply it here too (idempotent - harmless
# if the notebook's first cell, or an earlier call in this same process,
# already did) so the standalone path gets the same real protection against
# oversubscription, not just the same env-var gesture.
try:
    import threadpoolctl as _threadpoolctl
    _THREAD_LIMIT_CONTROLLER = _threadpoolctl.threadpool_limits(
        limits=int(_BLAS_THREADS))
except ImportError:
    _threadpoolctl = None
    _THREAD_LIMIT_CONTROLLER = None


def effective_blas_thread_counts():
    """The ACTUAL thread cap each loaded BLAS/OpenMP pool is using right now.

    This is what Section 7's regression check asserts against, instead of
    THREADS_PINNED_BEFORE_NUMPY: an import-order inference can be false in an
    environment (Colab) where threading is nonetheless correctly capped by
    threadpoolctl, and true in one where it was never capped at all if
    nothing downstream ever calls BLAS. Querying the live pools is the only
    way to state the fact this section actually cares about - how many
    threads a BLAS call will use - rather than a proxy for it.

    Returns {} (not a per-pool failure) when threadpoolctl itself is
    unavailable - that absence is a missing dependency, which is a different
    condition from "present but reporting too many threads", and the two are
    not collapsed into the same False.
    """
    if _threadpoolctl is None:
        return None
    return {
        f"{info['internal_api']}:{info.get('filepath', info.get('prefix', '?'))}":
        info["num_threads"]
        for info in _threadpoolctl.threadpool_info()
    }

# BUG FIX (raised in external review): this file previously called
# warnings.filterwarnings("ignore"), which hid EVERYTHING - including solver
# non-convergence, divide-by-zero in ratio features, and pandas warnings about
# silent dtype changes. Those are exactly the signals that distinguish "the
# model fitted and found nothing" from "the model never fitted".
#
# Only the known-benign, high-volume categories are filtered, and each is
# named so the choice is auditable. Convergence and numerical warnings are
# deliberately left VISIBLE.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*use_inf_as_na.*")
warnings.filterwarnings("ignore", message=".*is_sparse is deprecated.*")
# Convergence warnings are counted rather than silenced, so a run that is
# quietly failing to fit reports the fact.
CONVERGENCE_LOG = []
# Statistical fits that FAILED, kept distinct from fits that found nothing.
FIT_FAILURES = []


def _record_convergence_warnings():
    """Collect solver-convergence warnings instead of discarding them."""
    import warnings as _w
    from sklearn.exceptions import ConvergenceWarning

    def _hook(message, category, filename, lineno, file=None, line=None):
        if issubclass(category, ConvergenceWarning):
            CONVERGENCE_LOG.append(str(message)[:200])
        else:
            _w._showwarning_orig(message, category, filename, lineno, file, line)

    if not hasattr(_w, "_showwarning_orig"):
        _w._showwarning_orig = _w.showwarning
    _w.showwarning = _hook


_record_convergence_warnings()

# ----------------------------------------------------------------------------
# 0.1 Install dependencies (Colab-safe; no-ops if already present)
# ----------------------------------------------------------------------------
def _pip(pkgs):
    """Install only what is missing, quietly."""
    import importlib
    missing = []
    for pkg, mod in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {', '.join(missing)}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                       check=False)

_pip([
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("scikit-learn", "sklearn"),
    ("statsmodels", "statsmodels"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("lifelines", "lifelines"),
    ("requests", "requests"),
])

import numpy as np
import pandas as pd
import scipy.stats as st
# ----------------------------------------------------------------------------
# NOTEBOOK DETECTION AND PLOTTING BACKEND
# ----------------------------------------------------------------------------
# BUG FIX: an earlier version called matplotlib.use("Agg") unconditionally.
# Agg is a headless, file-only backend, so NOTHING rendered in the notebook -
# every figure was written to outputs/04_figures and never shown. On Colab
# that directory is ephemeral and hidden behind the file browser, so from the
# reader's seat the pipeline appeared to produce no figures at all.
#
# Agg is now used only when there is no notebook to draw into.
def _in_notebook():
    """True inside a Jupyter/Colab kernel, False in a plain script or REPL."""
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            return False
        # ZMQInteractiveShell is the kernel used by Jupyter and Colab.
        # TerminalInteractiveShell is plain IPython, which cannot draw inline.
        return type(ip).__name__ == "ZMQInteractiveShell"
    except Exception:
        return False


IN_NOTEBOOK = _in_notebook()
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from statsmodels.stats.multitest import multipletests

# ----------------------------------------------------------------------------
# 0.2 Global configuration
# ----------------------------------------------------------------------------
RUN_MODE = os.environ.get("RUN_MODE", "DEMO").upper()   # "DEMO" or "LIVE"
SEED = 42

np.random.seed(SEED)

# Output root. The Colab setup cell may point these at Drive so a dropped
# session resumes from cached downloads instead of re-fetching several GB.
# SURFACEOME_OUT_DIR moves the whole tree; SURFACEOME_RAW_DIR moves only the
# raw-download cache, which is the part worth persisting.
OUT = Path(os.environ.get("SURFACEOME_OUT_DIR", "outputs"))
DIRS = {
    "raw":    Path(os.environ.get("SURFACEOME_RAW_DIR", OUT / "00_raw")),
    "interim": OUT / "01_interim",
    "features": OUT / "02_features",
    "models": OUT / "03_models",
    "figures": OUT / "04_figures",
    "tables": OUT / "05_tables",
    "logs":   OUT / "06_logs",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)



# ============================================================================
# CANONICAL EXPRESSION SCALE
# ============================================================================
# EVERY expression matrix downstream of loading is log2(TPM + 1). Nothing else.
#
# BUG FIX (critical, raised in external review and confirmed): these two
# converters existed but were applied ONLY to GTEx. The TCGA-PRAD and SU2C
# matrices went into cleaning, prevalence and differential expression still on
# Xena's native log2(TPM + 0.001) scale, while three separate call sites
# computed prevalence as `2**x - 1`, which assumes log2(TPM + 1).
#
# The error is a systematic -1 TPM offset on the recovered linear scale. A
# gene truly at 1 TPM was read as 0.001 TPM; one at 5 TPM was read as 4 TPM.
# It is worst exactly at the threshold that matters, shifting the effective
# "expressed" cut from 1 TPM to about 2 TPM, and it is invisible because every
# number downstream still looks entirely reasonable.
#
# The fix is structural rather than a patched call site: `to_canonical_log`
# converts once, at load, and `linear_tpm` is the ONLY sanctioned way to get
# back to TPM. No function below re-derives a linear scale by hand.
# ============================================================================
CANONICAL_PSEUDOCOUNT = 1.0      # log2(TPM + 1)
XENA_PSEUDOCOUNT = 0.001         # log2(TPM + 0.001), as Xena Toil ships it


def xena_log_to_tpm(df):
    """Invert Xena Toil's log2(TPM + 0.001) to linear TPM."""
    return np.power(2.0, df) - XENA_PSEUDOCOUNT


def tpm_to_log(df):
    """Linear TPM to the canonical log2(TPM + 1)."""
    return np.log2(np.clip(df, 0, None) + CANONICAL_PSEUDOCOUNT)


def to_canonical_log(df, source_scale, label="matrix"):
    """Put any supported input on the canonical log2(TPM + 1) scale.

    source_scale: "xena_log"  -> log2(TPM + 0.001)   (Xena Toil)
                  "linear_tpm"-> TPM                  (raw abundance)
                  "log1p_tpm" -> log2(TPM + 1)        (already canonical)
    """
    if source_scale == "log1p_tpm":
        print(f"    {label}: already canonical log2(TPM+1)")
        return df
    if source_scale == "xena_log":
        out = tpm_to_log(xena_log_to_tpm(df))
        print(f"    {label}: converted log2(TPM+0.001) -> log2(TPM+1)")
        return out
    if source_scale == "linear_tpm":
        print(f"    {label}: converted linear TPM -> log2(TPM+1)")
        return tpm_to_log(df)
    raise ValueError(f"Unknown source_scale '{source_scale}' for {label}.")


def linear_tpm(df_log, label="matrix", strict=True):
    """The ONLY sanctioned inverse. Input must already be canonical.

    BUG FIX (critical, found by external review with a concrete example):
    a log2(TPM+1) value CANNOT be negative, because TPM >= 0 implies
    TPM + 1 >= 1 implies log2(...) >= 0. Any negative value on this scale is
    therefore impossible data, and inverting it produces a negative "TPM".
    That silently broke the safety filter: F2 tests
    `safety_ratio_vs_psma <= 1.0`, which a negative ratio passes trivially.
    In the previous run 254 of 2,792 genes carried a negative ratio and every
    one of them auto-passed F2.

    This function now refuses to return impossible values. The check is here,
    at the single sanctioned inverse, so that no call site can skip it.
    """
    bad = (df_log < 0)
    n_bad = int(np.asarray(bad).sum())
    if n_bad:
        frac = n_bad / max(np.asarray(df_log).size, 1)
        msg = (f"{label}: {n_bad:,} values ({frac:.2%}) are NEGATIVE on a "
               f"log2(TPM+1) scale, which is impossible - TPM >= 0 forces "
               f"log2(TPM+1) >= 0. Inverting them yields negative TPM, which "
               f"silently passes any upper-bound safety threshold. The matrix "
               f"is either on a different scale than declared, or z-scored, "
               f"or synthetic data generated on the wrong scale.")
        if strict:
            raise ValueError(msg + " Fix the scale at load rather than "
                                   "clipping here, which would hide it.")
        print(f"    WARNING {msg}")
    out = np.power(2.0, df_log) - CANONICAL_PSEUDOCOUNT
    # Floating-point can leave a tiny negative at exactly zero expression.
    return np.clip(out, 0.0, None)


# ----------------------------------------------------------------------------
# 0.4 INLINE DISPLAY HELPERS
# ----------------------------------------------------------------------------
# Every figure and table is BOTH written to disk AND rendered in the notebook.
# Writing only to disk was the earlier bug: on Colab the output directory is
# ephemeral and tucked behind the file browser, so a reader running the
# notebook saw console text and nothing else.
#
# save_figure() keeps the file (needed for the manuscript, and for the zip at
# the end) and shows the figure where the reader is actually looking.
# ----------------------------------------------------------------------------
FIGURE_REGISTRY = []
TABLE_REGISTRY = []


# ----------------------------------------------------------------------------
# 0.8b PRODUCTION-WRITE GUARD (NEW - publication-blocking defect, external review)
# ----------------------------------------------------------------------------
# CRITICAL DEFECT, confirmed in the shipped archive: the regression suite added
# last round calls production functions (apply_gatekeeping, preflight_label_audit)
# on small test fixtures, and those functions write their tables straight into
# the publication output directory. Three real artifacts were silently replaced
# by six-gene and forty-gene test fixtures:
#
#   05_tables/table1_filter_attrition.csv        2,792 genes -> 6 fixture genes
#   02_features/feature_table_with_filters.csv   2,792 genes -> 6 fixture genes
#   05_tables/table16_preflight_label_audit.csv  real audit  -> fixture audit
#
# The run completed, reported 20/20 tests passed, and exported a corrupted
# archive. A reader reconciling Table 1 against the manuscript would find six
# genes where 2,792 were claimed. This is exactly the failure class the tests
# were added to prevent, and the tests caused it.
#
# Two independent defences are now in place, because one was evidently not
# enough:
#   1. An explicit `save_outputs` argument on every function that both
#      computes and writes, so a caller must opt IN to writing.
#   2. This module-level guard. While writes are suppressed, save_table and
#      save_figure refuse to touch the output directory at all, whatever any
#      call site asks for. Defence 1 can be forgotten at a new call site;
#      defence 2 cannot.
_WRITES_SUPPRESSED = False
SUPPRESSED_WRITE_LOG = []


class suppress_outputs:
    """Context manager: block all artifact writes inside the block.

    Used to wrap the regression suite so a test can call production code
    freely without any possibility of overwriting a publication artifact.
    """

    def __enter__(self):
        global _WRITES_SUPPRESSED
        self._prev = _WRITES_SUPPRESSED
        _WRITES_SUPPRESSED = True
        return self

    def __exit__(self, *exc):
        global _WRITES_SUPPRESSED
        _WRITES_SUPPRESSED = self._prev
        return False


def writes_suppressed():
    return _WRITES_SUPPRESSED


def guarded_to_csv(df, path, index=False, label=None):
    """Write a non-table artifact, honouring the production-write guard.

    Several functions write feature tables and QC logs directly rather than
    through save_table. Those call sites were the ones the regression suite
    corrupted, so they go through this helper and are suppressed too.
    """
    if _WRITES_SUPPRESSED:
        SUPPRESSED_WRITE_LOG.append(("file", str(label or path)))
        return None
    df.to_csv(path, index=index)
    register_artifact(path, label or "file")
    return path


def save_figure(fig, name, caption=None, dpi=300, show=True):
    """Write a figure to outputs/04_figures AND display it inline."""
    path = DIRS["figures"] / name
    if _WRITES_SUPPRESSED:
        SUPPRESSED_WRITE_LOG.append(("figure", name))
        plt.close(fig)
        return None
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    register_artifact(path, "figure")
    FIGURE_REGISTRY.append({"name": name, "path": str(path), "caption": caption})
    if show and IN_NOTEBOOK:
        try:
            from IPython.display import display, Markdown
            if caption:
                display(Markdown(f"**{name}** — {caption}"))
            display(fig)
        except Exception:
            pass
    plt.close(fig)
    if not IN_NOTEBOOK:
        print(f"    figure saved: {path.name}")
    return path


def save_table(df, name, caption=None, show=True, max_rows=15, index=False):
    """Write a table to outputs/05_tables AND display it inline."""
    path = DIRS["tables"] / name
    if _WRITES_SUPPRESSED:
        SUPPRESSED_WRITE_LOG.append(("table", name))
        return df
    df.to_csv(path, index=index)
    register_artifact(path, "table")
    TABLE_REGISTRY.append({"name": name, "path": str(path), "caption": caption,
                           "rows": len(df), "cols": df.shape[1]})
    if show and IN_NOTEBOOK:
        try:
            from IPython.display import display, Markdown
            head = f"**{name}**" + (f" — {caption}" if caption else "")
            if len(df) > max_rows:
                head += f"  _(showing {max_rows} of {len(df):,} rows)_"
            display(Markdown(head))
            shown = df.head(max_rows)
            # Round floats so wide numeric tables stay readable on screen.
            num = shown.select_dtypes("number").columns
            display(shown.style.format({c: "{:,.4g}" for c in num})
                    .hide(axis="index") if not index else shown)
        except Exception:
            try:
                from IPython.display import display
                display(df.head(max_rows))
            except Exception:
                pass
    if not IN_NOTEBOOK:
        print(f"    table saved: {path.name} ({len(df):,} rows)")
    return path


def show_heading(text, level=2):
    """A visible section heading in the notebook, not just console text."""
    if IN_NOTEBOOK:
        try:
            from IPython.display import display, Markdown
            display(Markdown(f"{'#' * level} {text}"))
            return
        except Exception:
            pass
    print(f"\n{text}\n{'-' * len(text)}")


@dataclass
class Config:
    """All analysis thresholds in one object, so every number in the manuscript
    is traceable to a single declared value rather than a magic constant buried
    in a function."""

    # --- Gatekeeping Filter 1: differential expression -----------------------
    fdr_alpha: float = 0.05          # Benjamini-Hochberg FDR ceiling
    min_log2fc: float = 1.0          # tumour vs normal prostate, log2 scale
    min_expr_tpm: float = 1.0        # expression floor (TPM) to call a gene "on"
    min_frac_expressing: float = 0.25  # fraction of tumours above the floor

    # --- Gatekeeping Filter 2: normal-tissue expression comparator ----------
    # IMPORTANT NAMING DECISION: this is NOT a validated safety filter and must
    # not be described as one. A ratio relative to PSMA is an EXPRESSION
    # COMPARATOR. Actual tolerability depends on the therapeutic modality
    # (naked antibody vs ADC vs CAR-T vs radioligand), payload potency,
    # normal-cell context, target localisation and receptor density - none of
    # which this ratio captures. Calling F2 a "safety filter" in a manuscript
    # would be a claim the data cannot support. Absolute per-organ TPM values
    # are reported alongside the ratio for exactly this reason.
    benchmark_gene: str = "FOLH1"
    # BUG FIX (raised in external review): salivary gland was ABSENT from
    # this list while present in the tissue matrix. Xerostomia from salivary
    # uptake is the signature dose-limiting toxicity of PSMA-directed
    # radioligand therapy, so omitting it from a PSMA-benchmarked comparator
    # left out the organ most likely to decide whether a target is tolerable.
    vital_organs: tuple = (
        "Brain - Cortex", "Heart - Left Ventricle", "Lung",
        "Liver", "Kidney - Cortex", "Whole Blood",
        "Nerve - Tibial", "Artery - Aorta",
        "Salivary Gland",
    )

    # MODALITY-SPECIFIC normal-tissue risk. Tolerable normal-tissue exposure
    # is not one number: it depends on what the antigen is engaged WITH.
    # A naked antibody, a potent-payload ADC, a CAR-T product and a
    # radioligand have materially different requirements, so a single
    # general-purpose "safety" ranking cannot serve all four.
    modality_risk_tissues: dict = field(default_factory=lambda: {
        "ADC": ("Liver", "Whole Blood", "Lung", "Kidney - Cortex",
                "Nerve - Tibial"),
        "CAR_T": ("Heart - Left Ventricle", "Lung", "Brain - Cortex",
                  "Whole Blood", "Liver", "Kidney - Cortex",
                  "Artery - Aorta"),
        "T_cell_engager": ("Lung", "Whole Blood", "Liver",
                           "Heart - Left Ventricle", "Brain - Cortex"),
        "radioligand": ("Salivary Gland", "Kidney - Cortex",
                        "Whole Blood", "Liver"),
    })
    # CLINICALLY UNMEASURABLE ORGANS AT RISK.
    # Declared explicitly, because their ABSENCE is not neutral - it
    # systematically flatters exactly the modality whose dose-limiting
    # toxicities they carry.
    #   Bone marrow: myelosuppression is the dose-limiting toxicity of
    #     177Lu-PSMA radioligand therapy and of alpha-emitter programmes, and
    #     bone is the dominant metastatic site in prostate cancer - so marrow
    #     is simultaneously the tumour compartment and the organ at risk.
    #     GTEx contains no bone marrow sample. Whole Blood is NOT a surrogate:
    #     circulating leukocytes are not CD34+ progenitors, and an antigen on
    #     progenitors can be entirely invisible in blood.
    #   Lacrimal and minor salivary glands: xerostomia and dry eye are the
    #     signature on-target toxicities of PSMA-directed radioligands.
    #     Lacrimal gland is absent from GTEx.
    # No ratio computed by this pipeline covers these organs. Any candidate
    # advanced toward a radioligand or alpha-emitter programme needs
    # independent marrow and exocrine-gland assessment.
    unmeasurable_risk_organs: dict = field(default_factory=lambda: {
        "Bone marrow": ("dose-limiting myelosuppression for radioligand and "
                        "alpha-emitter therapy; also the dominant metastatic "
                        "compartment. Absent from GTEx; Whole Blood is not a "
                        "surrogate for CD34+ progenitors."),
        "Lacrimal gland": ("dry eye is a signature on-target PSMA radioligand "
                           "toxicity. Absent from GTEx."),
    })

    # Ratio ceiling per modality, relative to PSMA on that modality's own
    # tissue set. Stricter where a single off-tumour hit is less recoverable.
    modality_ratio_max: dict = field(default_factory=lambda: {
        "ADC": 1.0, "CAR_T": 0.5, "T_cell_engager": 0.5, "radioligand": 1.0,
    })
    safety_ratio_max: float = 1.0    # candidate_vital_max / PSMA_vital_max

    # --- Gatekeeping Filter 3: membrane topology ----------------------------
    require_extracellular_domain: bool = True
    min_ecd_length_aa: int = 20      # shortest ECD worth targeting with an mAb

    # --- Gatekeeping Filter 4: reagent availability -------------------------
    # PROPOSAL REFINEMENT (Step 2): F1-F3 are BIOLOGICAL screening criteria.
    # F4 is an EXPERIMENTAL-READINESS tier, not a biological veto. Antibody
    # availability reflects study intensity, so a strict veto suppresses
    # exactly the understudied antigens the study exists to find. The strict
    # four-gate pipeline is still run, as a declared sensitivity analysis, so
    # the original design is reproduced rather than abandoned.
    f4_mode: str = "tier"            # "tier" (primary) | "strict" (sensitivity)
    require_validated_antibody: bool = True   # used only when f4_mode="strict"

    # --- Differential expression method -------------------------------------
    # PROPOSAL REFINEMENT (Section 6): a count-based negative-binomial model is
    # the appropriate primary analysis when raw counts and design metadata are
    # available. Welch and Mann-Whitney on transformed abundance are then
    # SENSITIVITY analyses, not the primary test. The pipeline auto-selects:
    # if a counts matrix is supplied it uses the NB GLM; otherwise it falls
    # back to Welch on log-abundance and says so in the output.
    de_method: str = "auto"          # "auto" | "nb_counts" | "welch"

    # --- Proteomic corroboration -------------------------------------------
    protein_fdr_alpha: float = 0.10  # looser: proteomics is lower-powered
    protein_min_log2fc: float = 0.26 # ~1.2-fold; MS dynamic range is compressed

    # MULTIPLE-TESTING FAMILY for the proteomics arm. The correction family is
    # defined by the ORIGINAL hypothesis-testing universe, not by whichever
    # subset happens to reach this pipeline.
    #
    #   "supplied"           use q-values from the original proteome-wide
    #                        analysis. PREFERRED - the family is whatever the
    #                        source analysis tested, and it is already right.
    #   "proteome_wide"      recompute BH across the FULL proteome p-value
    #                        table supplied, then subset to the surfaceome.
    #   "surfaceome_subset"  correct within the surfaceome only. Valid ONLY as
    #                        a separately justified, PRESPECIFIED hypothesis
    #                        family, and must be declared as such.
    #
    # There is deliberately no automatic fallback to "surfaceome_subset":
    # correcting over a post-hoc subset of a larger family inflates
    # significance, and doing it silently is how an under-powered proteomic
    # arm starts looking convincing.
    protein_fdr_family: str = "supplied"
    protein_fdr_family_prespecified: bool = False

    # --- Machine learning ---------------------------------------------------
    # --- COMPUTE PROFILE ----------------------------------------------------
    # Set by PROFILE below; these are the values it writes. Do not edit here -
    # edit PROFILE, or override a single field after CFG is built.
    #
    # HONEST RUNTIMES, measured rather than guessed. The dominant cost is the
    # saga solver inside nested grid search: bags x outer folds x 3 learners x
    # (inner folds x grid size), run twice for the leakage ablation.
    #   smoke       ~3-5 min    enough to prove the code runs
    #   standard    ~35-50 min  usable confidence intervals
    #   manuscript  ~3-5 h      the final run you report
    # Colab's free tier has 2 vCPUs and is roughly 2-3x slower than a typical
    # workstation, so treat the upper end of each range as the expectation.
    n_outer_folds: int = 5
    # NOTE: there is deliberately no `n_outer_repeats`. An earlier version
    # declared one, never used it, and figure captions claimed "10x5 repeated
    # nested CV" that the code did not perform. Repetition here comes from the
    # BAGS: each bag draws a fresh pseudo-negative sample and runs its own
    # outer CV, so the effective repetition is n_pu_bags x n_outer_folds. That
    # is what the captions now say.
    n_inner_folds: int = 3
    n_pu_bags: int = 12              # bagged positive-unlabeled replicates
    enet_grid_size: str = "full"     # "compact" | "full"
    run_stability: bool = True
    stability_bag_multipliers: tuple = (2, 3, 5)
    # Minimum log2 separation between the PSMA-low and PSMA-high strata before
    # the persistence contrast is considered interpretable (see
    # psma_low_persistence); guards against tied/zero-inflated PSMA.
    min_psma_strata_separation_log2: float = 0.5
    # Normal-tissue detection floor, in TPM. Bulk RNA cannot distinguish "not
    # expressed" from "below detection", and a vital-organ value of exactly 0
    # would otherwise give an unbounded therapeutic window (see
    # safety_features). Values below the floor are CENSORED AT the floor.
    vital_tpm_detection_floor: float = 0.5
    # Fraction of declared vital organs that must actually be MEASURED before
    # an F2 result may be called definitive. GTEx and other normal-tissue
    # panels differ in both coverage and naming, and a missing organ is not a
    # safe organ.
    # NOTE (self-review): with 9 declared vital organs this threshold is
    # NOT a 10% tolerance - it is equivalent to requiring ALL NINE, because
    # 8/9 = 88.9% < 90%. The config appeared to document a tolerance that
    # does not exist at this granularity. The policy is deliberately strict
    # (a silently absent vital organ is indistinguishable from a safe one),
    # but the equivalence is now stated and asserted at import, so changing
    # the organ list cannot silently open a tolerance nobody chose.
    # NEW (external review): coverage thresholds that were previously
    # implicit. A zero-overlap endpoint join already raises, but a very small
    # NON-zero overlap proceeded silently - an analysis on 4 of 500 patients
    # is not the analysis the Methods describe. These are prespecified here
    # rather than discovered after seeing a result.
    min_bcr_match_rate: float = 0.50
    min_ancestry_match_rate: float = 0.80
    # Both reviews left the ancestry threshold's ENFORCEMENT conditional on
    # intent, which is the right call: a descriptive ancestry breakdown at
    # 60% coverage is a reported limitation, while a PRESPECIFIED
    # ancestry-stratified inference at 60% coverage is a different analysis
    # from the one the Methods describe. The intent is therefore declared
    # here rather than inferred, and the threshold blocks only when
    # inference was prespecified.
    ancestry_inference_prespecified: bool = False
    # Conflicting patient-level endpoints across a patient's specimens are a
    # source-reconciliation problem. Strict by default: an analysis must not
    # silently resolve them. Real longitudinal records do get updated, so a
    # cohort with a documented source-precedence rule can set this False to
    # report rather than stop.
    strict_patient_outcome_consistency: bool = True
    min_vital_organ_coverage: float = 0.90
    # Where the PSMA comparator should sit within the vital-organ expression
    # distribution it is compared against. This governs the DEMO generator
    # only - on LIVE data the comparator is whatever PSMA actually measures.
    # It exists because an anchor's discriminating power is a property of its
    # POSITION in the distribution, not its absolute TPM: a comparator at the
    # median admits half the surfaceome and filters nothing.
    comparator_target_percentile: float = 25.0
    unlabeled_per_bag_multiplier: int = 3   # pseudo-negatives per positive
    n_permutations: int = 200        # label-permutation null for AUC
    n_bootstrap: int = 1000          # bootstrap CI for performance metrics

    # --- Survival -----------------------------------------------------------
    # SELF-REVIEW: this field existed but was READ NOWHERE - grep for
    # CFG.survival_endpoint returned no consumer. The config therefore
    # declared "BCR" while the loader selected DFS or PFS by preference
    # order, so the configuration documented an analysis the code did not
    # perform. It is now honoured by select_survival_endpoint():
    #   "BCR"  - prefer a validated BCR file; without one, fall back to the
    #            documented order with the existing loud warning
    #   "DFS" / "PFS" - request that endpoint ONLY; refuse rather than
    #            substitute the other, which is what a prespecified
    #            analysis requires
    #   "auto" - the documented preference order, no declared preference
    survival_endpoint: str = "BCR"
    # PROPOSAL REFINEMENT (Section 6): the 10-events-per-variable rule is a
    # SCREENING HEURISTIC, not a universal guarantee of unbiased estimation.
    # It is used here as a conservative admissibility gate, and models that
    # clear it still require inspection of covariate burden, missingness,
    # proportional hazards and multiplicity before any claim is made.
    min_events_per_covariate: int = 10
    expr_split: str = "median"       # tertile|median for KM stratification
    # Minimum patients per Kaplan-Meier group. A prespecified comparison
    # whose groups are too small is not estimable; fitting it anyway yields
    # a p-value that describes nothing. Prespecified rather than discovered.
    min_km_group_size: int = 5

    # --- Ranking stability and external transfer ----------------------------
    # PROPOSAL REFINEMENT (Step 4 / Step 5): test alternative bagging
    # assumptions and seeds, and score an independent cohort with the FROZEN
    # model. Internal benchmark enrichment is not external validation.
    stability_seeds: tuple = (42, 7)

    # --- PREDEFINED ensemble-vs-baseline improvement rule -------------------
    # Fixed BEFORE any LIVE outcome is examined. With only six held-out
    # positives, a one-target difference is noise: the rule therefore requires
    # a margin on the continuous primary metric AND more than a single-target
    # improvement in top-decile recovery, AND that the ranking be stable
    # enough for the comparison to mean anything.
    min_meaningful_margin_pct: float = 3.0   # percentile points on the primary metric
    min_top_decile_gain: int = 2             # >1 target: one target is noise at n=6
    min_rank_stability_rho: float = 0.80     # below this, do not adjudicate at all
    precision_at_k: tuple = (10, 25, 50, 100)

    # --- Reproducibility ----------------------------------------------------
    seed: int = SEED


# ----------------------------------------------------------------------------
# COMPUTE PROFILE
# ----------------------------------------------------------------------------
# "smoke"      prove the code runs, end to end, in minutes. The statistics are
#              deliberately under-powered: confidence intervals will be wide
#              and the ranking will be unstable. Never report smoke numbers.
# "standard"   the default for a first LIVE run. Usable intervals.
# "manuscript" the run you actually report. 50 bags, full permutation null.
#
# DEMO defaults to "smoke" because the whole point of the demo is a fast check
# that the software works before you spend an hour downloading data. An
# earlier version of this notebook claimed a 4-minute demo while the settings
# actually took ~37 minutes; the profiles exist so the claim and the code
# cannot drift apart again.
PROFILE = os.environ.get("PROFILE", "smoke" if RUN_MODE == "DEMO" else "standard").lower()

_PROFILES = {
    "smoke":      dict(n_pu_bags=3,  n_outer_folds=3, n_inner_folds=2,
                       n_permutations=30,  n_bootstrap=200,
                       enet_grid_size="compact", run_stability=True,
                       stability_seeds=(42, 7), stability_bag_multipliers=(3,)),
    "standard":   dict(n_pu_bags=12, n_outer_folds=5, n_inner_folds=3,
                       n_permutations=200, n_bootstrap=1000,
                       enet_grid_size="full", run_stability=True,
                       stability_seeds=(42, 7), stability_bag_multipliers=(2, 3, 5)),
    "manuscript": dict(n_pu_bags=50, n_outer_folds=5, n_inner_folds=3,
                       n_permutations=1000, n_bootstrap=2000,
                       enet_grid_size="full", run_stability=True,
                       stability_seeds=(42, 7, 2026),
                       stability_bag_multipliers=(2, 3, 5)),
}
if PROFILE not in _PROFILES:
    raise ValueError(
        f"PROFILE must be one of {list(_PROFILES)}, got '{PROFILE}'. "
        f"(There is no 'full' profile. An earlier revision of the notebook "
        f"header and the standalone runner's usage line both named one; "
        f"following either produced this error. Use 'standard' for a first "
        f"LIVE run and 'manuscript' for the run you report.)")

CFG = Config(**_PROFILES[PROFILE])

# Escape hatch for shaking down the LIVE loaders without committing to a
# multi-hour fit. Deliberately NOT a profile: a profile is a statistical
# setting, and this is a statement about what the run is FOR.
ALLOW_SMOKE_LIVE = os.environ.get("ALLOW_SMOKE_LIVE", "") in ("1", "true",
                                                             "True")

# RUN PURPOSE - what this invocation is for.
# NEW (suggested by two external reviews; adopted because it resolves a real
# deadlock). The provenance gate blocks model fitting until the clinical
# label registry is curated. That is correct for anything reportable - but
# it also made it impossible to debug the LIVE data loaders against real
# TCGA/GTEx/SU2C files while the curation is still in progress, which is
# exactly the work that has to happen first. Curation and loader debugging
# were serialised by a gate that only needed to guard the modelling.
#
#   analysis  (default) - full pipeline; the provenance gate applies.
#   data_only           - acquire, validate and cache the real datasets,
#                         then stop before any model is fitted. Produces NO
#                         ranking, NO benchmark comparison and NO stop/go
#                         verdict, and every artifact it writes is stamped
#                         non-reportable, so it cannot be mistaken for a
#                         scientific result.
RUN_PURPOSE = os.environ.get("RUN_PURPOSE", "analysis").lower()
if RUN_PURPOSE not in ("analysis", "data_only"):
    raise ValueError(
        f"RUN_PURPOSE must be 'analysis' or 'data_only', got "
        f"'{RUN_PURPOSE}'.")
REPORTABLE = (RUN_PURPOSE == "analysis")

# RUN IDENTITY - stamped onto every QC artifact.
# BUG FIX (raised in external review): the stop/go gate checked whether
# filter_unit_tests.csv EXISTED, so a stale file left by an earlier run could
# satisfy QC for a run that never produced one. Artifacts now carry the run id
# and the gate requires a match.
import uuid as _uuid
RUN_ID = os.environ.get("RUN_ID") or (
    pd.Timestamp.now().strftime("%Y%m%dT%H%M%S") + "-" + _uuid.uuid4().hex[:6])

_EST_RUNTIME = {"smoke": "~3-5 min", "standard": "~35-50 min",
                "manuscript": "~3-5 h"}

# ----------------------------------------------------------------------------
# 0.3 Benchmark gene sets
# ----------------------------------------------------------------------------
# =============================================================================
# POSITIVE LABEL SET - REQUIRES AUDIT BEFORE THE LIVE RUN
# =============================================================================
# PROPOSAL REFINEMENT (Data sources / Step 4): these labels are the model's
# ENTIRE supervision, so an error here propagates into every result. The
# proposal requires auditing each one against ClinicalTrials.gov and recording
# agent, trial ID, modality, phase, discontinuation status and a freeze date.
#
# The structure below exists to make that audit a tracked task rather than an
# assumption. Fill LABEL_PROVENANCE for every gene you keep, then run
# audit_labels() - it will refuse to proceed silently with unverified labels.
#
# A NOTE ON DISCONTINUED PROGRAMMES, which the proposal rightly raises:
# a target whose programme failed on TOXICITY is arguably a negative example,
# not a positive one. Training on it teaches the model the wrong lesson.
# Decide your rule explicitly and record it in LABEL_INCLUSION_RULE below.

LABEL_INCLUSION_RULE = {
    "min_phase": "Phase 1",
    "modalities_included": ["mAb", "ADC", "CAR-T", "radioligand", "bispecific",
                            "T-cell engager"],
    "tumour_types": "solid tumours (deliberately not prostate-restricted - the "
                    "model must learn general surface druggability, not "
                    "prostate identity)",
    "discontinued_programmes": "UNDECIDED - set to 'exclude', 'include', or "
                               "'toxicity_failures_as_negatives' before the "
                               "LIVE run and record the choice",
    "freeze_date": "UNSET - set to the date you audited the registry",
    "registry": "ClinicalTrials.gov",
}

# LABEL_PROVENANCE: gene -> dict(agent, trial_id, modality, phase, status).
# Left empty by design. Populating it is a required pre-LIVE task, and
# audit_labels() reports exactly how much of it is outstanding.
LABEL_PROVENANCE: dict = {}


def load_label_provenance(path=None, verbose=True):
    """Populate LABEL_PROVENANCE from a CSV instead of hand-editing code.

    Added for the Colab LIVE run. The registry was an empty dict literal in
    the source, so the one task that gates a reportable LIVE result -
    recording what clinical evidence each training label rests on - required
    editing Python inside a notebook cell and re-running everything. That is
    a poor fit for work done against a registry in a browser, and it makes
    the evidence impossible to review as a standalone supplementary file.

    Expected columns: gene, agent, trial_id, modality, phase, status.
    Rows whose gene is blank, or whose trial_id is blank, are loaded but
    still count as unverified in audit_labels() - partial evidence is
    recorded honestly rather than silently upgraded.

    Set LABEL_PROVENANCE_CSV in the environment, or pass a path.
    """
    import csv as _csv
    path = path or os.environ.get("LABEL_PROVENANCE_CSV", "")
    if not path:
        if verbose:
            print("  LABEL_PROVENANCE_CSV not set - the label registry "
                  "stays empty and the LIVE run will report it as "
                  "unverified.")
        return LABEL_PROVENANCE
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(
            f"LABEL_PROVENANCE_CSV={path!r} was set but no such file "
            f"exists. A declared evidence registry that cannot be read must "
            f"stop the run rather than leave the labels silently "
            f"unverified.")
    # The required columns ARE the fields the stop/go gate reads, so a
    # fully curated file necessarily clears the audit.
    required = {"gene"} | set(VERIFICATION_REQUIRED_FIELDS)
    with open(p, newline="", encoding="utf-8") as fh:
        rows = list(_csv.DictReader(fh))
    if not rows:
        raise ValueError(f"{p.name} has no rows.")
    missing_cols = required - set(rows[0])
    if missing_cols:
        raise ValueError(
            f"{p.name} is missing required column(s) "
            f"{sorted(missing_cols)}. Expected: {sorted(required)}.")
    n_filled = 0
    for r in rows:
        g = (r.get("gene") or "").strip()
        if not g:
            continue
        LABEL_PROVENANCE[g] = {
            k: (r.get(k) or "").strip()
            for k in (("agent", "trial_id", "modality")
                      + tuple(VERIFICATION_REQUIRED_FIELDS))}
        if LABEL_PROVENANCE[g]["trial_id"]:
            n_filled += 1
    if verbose:
        print(f"  label registry loaded from {p.name}: {len(rows)} row(s), "
              f"{n_filled} with a trial ID "
              f"({100 * n_filled / max(len(rows), 1):.0f}%)")
    return LABEL_PROVENANCE


def _template_role_of(gene):
    """The gene's actual standing in THIS run, for the provenance template.

    Derived from the same constants the model reads (PRIMARY_LABEL_CLASS,
    the three label-class lists, PROSTATE_HELDOUT_POSITIVES) so that the
    curation record and the training mask cannot disagree.
    """
    if gene in set(PROSTATE_HELDOUT_POSITIVES):
        return "held_out_benchmark"
    cls = label_class_of(gene)
    if cls == PRIMARY_LABEL_CLASS:
        return "training_positive"
    if cls == "immune_compartment":
        return "off_class_excluded_immune_compartment"
    if cls == "unclassified":
        return "unclassified"
    return f"withheld_from_primary_model_{cls}"


def write_label_provenance_template(path=None, verbose=True):
    """Emit a fillable registry CSV listing every gene that needs evidence.

    Produces one row per training positive and per held-out benchmark, with
    the evidence columns blank, so the curation work is a spreadsheet task
    against ClinicalTrials.gov rather than a code edit.
    """
    import csv as _csv
    path = Path(path or (DIRS["logs"] / "label_provenance_TEMPLATE.csv"))
    path.parent.mkdir(parents=True, exist_ok=True)
    genes = [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
    rows = []
    for g in genes:
        cur = LABEL_PROVENANCE.get(g, {})
        rows.append({
            "gene": g,
            # BUG FIX (two external reviews, correct and independently
            # reproduced against the shipped CSV): this emitted 69
            # "training_positive" rows while the primary model trains on 57.
            # It excluded the 5 immune-compartment genes but silently folded
            # the 12 SIGNAL-BLOCKADE genes in with the 57 payload-delivery
            # positives, although PRIMARY_LABEL_CLASS selects payload
            # delivery and the blockade genes are withheld from the primary
            # model. The CSV is meant to be the scientific record of what
            # trained the model, so a role column that misdescribes 12 of its
            # rows is a defect in the record itself - and a curator would
            # have spent real effort sourcing trial evidence under a label
            # that was wrong.
            #
            # The role now reports the gene's ACTUAL standing in this run,
            # derived from the same constants the model reads, so the two
            # cannot drift apart.
            "role": _template_role_of(g),
            # SELF-REVIEW, caught by round-tripping a fully filled
            # template: the columns I first emitted (agent/phase/status/
            # date_checked/reviewer) are NOT the fields the stop/go gate
            # reads. A registry could be curated to 100% and still leave
            # pct_verified at 0, wasting the entire curation effort. The
            # template now emits exactly VERIFICATION_REQUIRED_FIELDS, so
            # filling it in is what clears the gate.
            "agent": cur.get("agent", ""),
            "trial_id": cur.get("trial_id", ""),
            "modality": cur.get("modality", ""),
            "source_url": cur.get("source_url", ""),
            "verified_on": cur.get("verified_on", ""),
            "verified_by": cur.get("verified_by", ""),
            "development_phase": cur.get("development_phase", ""),
            "programme_status": cur.get("programme_status", ""),
        })
    for g in PROSTATE_HELDOUT_POSITIVES:
        if not any(r["gene"] == g for r in rows):
            rows.append({"gene": g, "role": "held_out_benchmark",
                         "agent": "", "trial_id": "", "modality": "",
                         "source_url": "", "verified_on": "",
                         "verified_by": "", "development_phase": "",
                         "programme_status": ""})
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = _csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    if verbose:
        print(f"  label registry template written: {path} ({len(rows)} "
              f"gene(s) to curate). Fill it in, then set "
              f"LABEL_PROVENANCE_CSV to its path and re-run.")
    return path

# The gene list itself. Treat as a CANDIDATE label set pending the audit above.
CLINICAL_STAGE_TARGETS = [
    # --- validated/approved solid-tumour surface targets --------------------
    "ERBB2", "EGFR", "MET", "TACSTD2", "CEACAM5", "MSLN", "FOLR1", "MUC16",
    "MUC1", "CD276", "DLL3", "NECTIN4", "EPCAM", "ROR1", "ROR2", "LRRC15",
    "TNFRSF10B", "CDH6", "CDH17", "CLDN18", "CLDN6", "GPC3", "ALPP",
    "PTK7", "LY6E", "SLC34A2", "SLC39A6", "TPBG", "AXL", "FGFR2", "FGFR3",
    "KDR", "PDGFRA", "IGF1R", "NTRK1", "ERBB3", "NRP1", "ITGAV", "ITGB6",
    "ANTXR1", "ENPP3", "CD47", "SIRPA", "CD70", "TNFRSF8", "CD44",
    "L1CAM", "ALCAM", "MCAM", "CA9", "GPNMB", "SDC1", "CD74", "HLA-DRA",
    "EPHA2", "EPHB4", "TFRC", "FOLH1_HELD", "GUCY2C", "GFRA1", "ULBP2",
    "CD24", "CD151", "PVR", "CD200", "SEZ6", "GRM8", "FZD7", "LGR5",
    "TM4SF1", "ITGA3", "PLXNB1", "SEMA4D", "VSIR", "LGALS9",
]
# ----------------------------------------------------------------------------
# PAYLOAD CLASS - the clinical distinction the single label set was ignoring
# ----------------------------------------------------------------------------
# CLINICAL DEFECT in the label definition: CLINICAL_STAGE_TARGETS mixes two
# therapeutic classes with OPPOSITE selectivity requirements.
#
#   PAYLOAD DELIVERY (ADC, CAR-T, T-cell engager, radioligand): a cytotoxic
#     payload is deposited wherever the antigen is, so tumour restriction is
#     the binding constraint. NECTIN4, TACSTD2, CEACAM5, DLL3, MSLN, FOLR1.
#
#   SIGNAL BLOCKADE (naked antibody, kinase inhibition): the drug interrupts
#     signalling, so broad normal expression is routinely tolerated. EGFR,
#     MET, KDR, PDGFRA, IGF1R, FGFR2/3, ERBB3 - and most starkly CD47
#     (essentially universal; dose-limiting anaemia) and TFRC (erythroid
#     precursors). These are real clinical targets, but they are NOT examples
#     of tumour-selective antigens.
#
# Training ONE classifier on both teaches contradictory requirements: that
# near-universal normal expression is a positive feature, and simultaneously
# that tumour selectivity is. This is a concrete candidate explanation for the
# central result - log_safety_ratio takes a strong negative coefficient, yet
# the ensemble cannot beat a pure therapeutic-window baseline, because the
# LABELS do not respect a therapeutic window.
#
# A third group is worse than either: antigens displayed mainly on INFILTRATING
# IMMUNE CELLS. In bulk tumour-versus-normal these look "tumour-upregulated"
# because tumours are inflamed, not because cancer cells carry them.
PAYLOAD_DELIVERY_TARGETS = [
    "ERBB2", "TACSTD2", "CEACAM5", "MSLN", "FOLR1", "MUC16", "MUC1",
    "CD276", "DLL3", "NECTIN4", "EPCAM", "ROR1", "ROR2", "LRRC15",
    "CDH6", "CDH17", "CLDN18", "CLDN6", "GPC3", "ALPP", "PTK7", "LY6E",
    "SLC34A2", "SLC39A6", "TPBG", "AXL", "ANTXR1", "ENPP3", "CD70",
    "TNFRSF8", "L1CAM", "ALCAM", "MCAM", "CA9", "GPNMB", "SDC1",
    "EPHA2", "EPHB4", "GUCY2C", "GFRA1", "SEZ6", "GRM8", "FZD7", "LGR5",
    "TM4SF1", "ITGA3", "ITGB6", "ITGAV", "CD24", "CD151", "PVR", "CD200",
    "NRP1", "PLXNB1", "SEMA4D", "TNFRSF10B", "ULBP2",
]
SIGNAL_BLOCKADE_TARGETS = [
    "EGFR", "MET", "FGFR2", "FGFR3", "KDR", "PDGFRA", "IGF1R", "NTRK1",
    "ERBB3", "TFRC", "CD47", "SIRPA",
]
IMMUNE_COMPARTMENT_TARGETS = [
    "CD74", "HLA-DRA", "CD44", "VSIR", "LGALS9",
]
# Which class trains the PRIMARY model. Payload delivery is the right default
# for this study: every modality named in the proposal (ADC, CAR-T, engager,
# radioligand) deposits a payload, so tumour restriction is the requirement
# the model should learn. The other classes are reported, and the choice is
# testable via the label-class ablation rather than asserted.
PRIMARY_LABEL_CLASS = os.environ.get("PRIMARY_LABEL_CLASS", "payload_delivery")


def label_class_of(gene):
    if gene in set(PAYLOAD_DELIVERY_TARGETS):
        return "payload_delivery"
    if gene in set(SIGNAL_BLOCKADE_TARGETS):
        return "signal_blockade"
    if gene in set(IMMUNE_COMPARTMENT_TARGETS):
        return "immune_compartment"
    return "unclassified"


def audit_label_classes(verbose=True):
    """Report the payload-class composition of the positive label set."""
    genes = [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
    rows = [{"gene": g, "label_class": label_class_of(g)} for g in genes]
    df = pd.DataFrame(rows)
    counts = df.label_class.value_counts().to_dict()
    if verbose:
        print(f"  label-class composition of {len(genes)} positives: {counts}")
        print(f"  PRIMARY_LABEL_CLASS = {PRIMARY_LABEL_CLASS} "
              f"({counts.get(PRIMARY_LABEL_CLASS, 0)} positives)")
        if counts.get("immune_compartment"):
            print(f"  NOTE: {counts['immune_compartment']} positive(s) are "
                  f"displayed mainly on infiltrating immune cells. In bulk "
                  f"tumour-vs-normal they appear tumour-upregulated because "
                  f"tumours are inflamed. They are excluded from the primary "
                  f"label class.")
        if counts.get("unclassified"):
            unc = sorted(df.loc[df.label_class == "unclassified", "gene"])
            print(f"  ACTION: classify before the LIVE run: {unc}")
    return {"counts": counts, "table": df,
            "primary_class": PRIMARY_LABEL_CLASS}


# NOTE: "FOLH1_HELD" is a sentinel, not a gene. FOLH1 (PSMA) is deliberately
# excluded from training and used as a held-out positive control. The sentinel
# is dropped before training; it exists only to make that choice explicit in
# the code rather than silent.

# HELD-OUT PROSTATE BENCHMARKS: never seen during training. If the model is
# learning anything real, these should rank near the top of the prostate
# surfaceome. This is the single most informative evaluation in the study.
# PROPOSAL REFINEMENT (comparison document): the benchmark set is SIX eligible
# surface-associated candidates. KLK3 is NOT one of them - it is an EXCLUSION
# CONTROL that must be removed at surfaceome bounding because PSA is secreted.
# Counting seven would silently include a gene that cannot be recovered by
# design, depressing the apparent recovery rate and misrepresenting the
# endpoint. The two sets are therefore declared separately.
# CLINICAL REVISION: two of the previous six were not defensible as
# clinical-stage prostate SURFACE-THERAPEUTIC targets, and they were exactly
# the two the model missed - so the headline recall metric was being set by
# benchmark-set composition rather than by model quality.
#   TMPRSS2: no clinical-stage programme targets it as a surface antigen. It
#     is known as the 5' partner of the TMPRSS2-ERG fusion and as the
#     SARS-CoV-2 entry protease. Failing to rank it is not a model failure.
#   SLC45A3 (prostein): a multi-pass transporter with minimal ectodomain,
#     used as an IHC lineage marker. Prostate-RESTRICTED, but not
#     surface-ACCESSIBLE, so it does not test what this model claims to do.
# Both are retained in a declared restriction-only category below so the
# earlier endpoint remains reproducible, and two targets with genuine
# clinical-stage surface programmes take their place.
PROSTATE_HELDOUT_POSITIVES = [
    "FOLH1",    # PSMA - radioligand (177Lu-PSMA-617) and ADC programmes
    "STEAP1",   # T-cell engager (xaluritamig) and prior ADC programmes
    "STEAP2",   # ADC programmes
    "TMEFF2",   # ADC target
    "PSCA",     # CAR-T programmes in mCRPC
    "KLK2",     # hK2 - radioligand and CAR programmes
]

# Prostate-RESTRICTED but not surface-accessible. Reported as a secondary,
# clearly-labelled endpoint; never counted in the primary recall metric,
# because a model that ranks surface targets should not be penalised for
# down-ranking a protein with no targetable ectodomain.
# BENCHMARK-SET PROVENANCE, stated plainly (external review, correct):
# TMPRSS2 and SLC45A3 were removed from the primary endpoint AFTER their
# model performance had been observed - they were the two the model missed.
# The biological rationale is defensible on its own terms (neither is a
# clinical-stage surface-therapeutic programme), but a set revised with
# knowledge of the outcome is NOT a prospectively frozen independent test.
# The original six are therefore retained below as a HISTORICAL SENSITIVITY
# endpoint and reported alongside the revised set, so the effect of the
# substitution on every headline metric is visible rather than implicit.
# Neither set may be modified again before BENCHMARK_SET_FROZEN_ON is recorded.
BENCHMARK_SET_REVISED_AFTER_SEEING_RESULTS = True
BENCHMARK_REVISION_RATIONALE = {
    "TMPRSS2": ("removed: no clinical-stage programme targets it as a surface "
                "antigen; known as the TMPRSS2-ERG fusion partner and the "
                "SARS-CoV-2 entry protease"),
    "SLC45A3": ("removed: multi-pass transporter with minimal ectodomain, "
                "used as an IHC lineage marker rather than a surface target"),
    "PSCA": "added: CAR-T programmes in mCRPC",
    "KLK2": "added: hK2 radioligand and CAR programmes",
}
HISTORICAL_HELDOUT_POSITIVES = [
    "FOLH1", "STEAP1", "STEAP2", "TMEFF2", "SLC45A3", "TMPRSS2",
]

PROSTATE_RESTRICTION_ONLY_MARKERS = [
    "SLC45A3",  # prostein - IHC lineage marker, minimal ectodomain
    "TMPRSS2",  # fusion partner / entry protease, not a surface programme
]

# CRITICAL (external review, correct): the retrospective comparison between
# the revised and historical benchmark sets was NOT training-equivalent. The
# revised six were withheld from fitting; TMPRSS2 and SLC45A3, dropped from
# the revised set, remained eligible to be drawn as PSEUDO-NEGATIVES. A model
# that has been trained to push two genes down cannot then be asked, on the
# same ranking, how well it ranks them - so the +0.9 percentile difference
# between the sets was not an unbiased effect of the substitution.
#
# EVALUATION_EXCLUDE is the UNION of both sets: all eight unique genes are
# withheld from positives and from pseudo-negative sampling, so both sets meet
# the model under identical conditions. This removes the avoidable asymmetry.
# It does NOT create prospective independence - neither set has that - and the
# revised six remain the declared developmental endpoint while the historical
# six are a retrospective sensitivity endpoint.
EVALUATION_EXCLUDE = sorted(set(PROSTATE_HELDOUT_POSITIVES)
                            | set(HISTORICAL_HELDOUT_POSITIVES))

# NEPC-specific surface targets, for the lineage-plastic arm. DLL3 and SSTR2
# are the clinically actionable NEPC antigens; DLL3 is a training positive
# (tarlatamab, pan-tumour), so SSTR2 is held out for the NEPC context.
NEPC_CONTEXT_BENCHMARKS = ["SSTR2"]

# Exclusion control: must be removed by surfaceome bounding, never counted as
# an evaluable benchmark. If it survives bounding, the topology logic is wrong.
SURFACEOME_EXCLUSION_CONTROLS = ["KLK3"]

# BENCHMARK PROVENANCE - audit required before the LIVE run.
# Each benchmark must be justified on TWO grounds, not one: that it is
# genuinely extracellularly accessible, and that it is relevant to the
# therapeutic modality you intend. A gene that is prostate-restricted but has
# no targetable ectodomain is not a fair test of a surface-target model.
# Fill `accessibility_evidence` and `modality` for each, then freeze the set.
# DO NOT modify this set after results are observed.
BENCHMARK_PROVENANCE = {
    g: {"accessibility_evidence": "UNVERIFIED", "modality": "UNVERIFIED",
        "trial_id": None}
    for g in PROSTATE_HELDOUT_POSITIVES
}
BENCHMARK_SET_FROZEN_ON = "UNSET - record the date you froze this set"


# EXTERNAL REVIEW, correct: the audits below measure METADATA COMPLETENESS,
# not independent verification. A populated trial_id proves a field was
# filled, not that a registry record was checked; and the benchmark audit
# never inspected the `modality` field at all. Both now report completeness
# and verification SEPARATELY, and neither may be read as evidence that a
# source was consulted.
# Provenance a POSITIVE surface-accessibility claim must carry before it can
# establish the experimental F3 route or evidence grade A. A boolean alone
# records that someone typed "True"; these record what was actually done.
SURFACE_EVIDENCE_REQUIRED_FIELDS = (
    "assay",            # e.g. flow cytometry on viable cells
    "cell_context",     # cell line, organoid or patient-derived material
    "source",           # publication, accession or internal report ID
    "evidence_date",    # when the experiment was performed or published
    "validated_by",     # who checked it
)

VERIFICATION_REQUIRED_FIELDS = (
    "trial_id", "source_url", "verified_on", "verified_by",
    "development_phase", "programme_status",
)


def audit_benchmarks(verbose=True):
    """Report outstanding verification on the held-out benchmark set."""
    unver = [g for g, v in BENCHMARK_PROVENANCE.items()
             if v.get("accessibility_evidence") == "UNVERIFIED"]
    if verbose:
        print(f"  benchmark audit: {len(PROSTATE_HELDOUT_POSITIVES)} eligible "
              f"benchmarks, {len(SURFACEOME_EXCLUSION_CONTROLS)} exclusion "
              f"control(s) ({', '.join(SURFACEOME_EXCLUSION_CONTROLS)})")
        if unver:
            print(f"  ACTION REQUIRED: extracellular-accessibility and "
                  f"modality evidence unverified for {len(unver)} benchmark(s): "
                  f"{', '.join(unver)}")
            print(f"  Freeze the set and record BENCHMARK_SET_FROZEN_ON before "
                  f"modelling. Never modify it after observing results.")
    # Completeness vs verification, reported separately.
    no_modality = [g for g, v in BENCHMARK_PROVENANCE.items()
                   if not v.get("modality")
                   or v.get("modality") == "UNVERIFIED"]
    no_source = [g for g, v in BENCHMARK_PROVENANCE.items()
                 if not v.get("source_url")]
    if verbose and no_modality:
        print(f"  benchmark modality field unverified for "
              f"{len(no_modality)} benchmark(s): {', '.join(no_modality)} - "
              f"the audit previously never checked this field at all")
    return {"n_eligible": len(PROSTATE_HELDOUT_POSITIVES),
            "n_unverified": len(unver),
            "n_modality_unverified": len(no_modality),
            "n_without_source_url": len(no_source),
            "frozen_on": BENCHMARK_SET_FROZEN_ON,
            "revised_after_seeing_results":
                BENCHMARK_SET_REVISED_AFTER_SEEING_RESULTS,
            "historical_set": list(HISTORICAL_HELDOUT_POSITIVES),
            "revision_rationale": dict(BENCHMARK_REVISION_RATIONALE),
            "audit_measures": ("metadata completeness only - a populated "
                               "field is not independently verified evidence"),
            "exclusion_controls": list(SURFACEOME_EXCLUSION_CONTROLS)}

# NEGATIVE-CONTROL GENES: housekeeping / intracellular. These must be removed
# by SURFACEOME BOUNDING, before any filter is applied.
NEGATIVE_CONTROLS = ["ACTB", "GAPDH", "TUBB", "RPL13A", "MKI67", "TP53", "AR"]

# PER-FILTER CONTROL GENES (DEMO mode only): each is constructed to fail
# exactly one gatekeeping filter, so the filter block is unit-tested rather
# than assumed correct. In LIVE mode these are absent and the checks are
# skipped automatically.
FILTER_CONTROLS = ["CTRL_NOT_DE", "CTRL_HIGH_VITAL", "CTRL_NO_ECD", "CTRL_NO_AB"]

# Every artificial identifier the demo injects. These exist ONLY to test the
# software and must never appear in a discovery, survival or reporting table.
#
# BUG FIX (critical, raised in external review): CTRL_NO_AB reached the novel
# candidate shortlist at the 98th percentile AND entered survival modelling.
# It is a synthetic gene constructed to fail the antibody filter - presenting
# it as a therapeutic candidate is exactly the kind of artefact that destroys
# confidence in everything around it.
# Scope note: only ENGINEERED CONTROLS are excluded, not the demo's anonymous
# background genes. `SURF####` and `BG####` are legitimate stand-ins for
# unnamed surfaceome and non-surfaceome genes - they are the background
# population the screen needs, and removing them would empty the demo. The
# controls are different in kind: each was constructed to FAIL a specific
# filter, so a control appearing as a candidate is a pipeline defect, whereas
# an anonymous filler appearing there is just the demo working.
# `NONSURF####` is proteome padding used only to size the FDR family.
# ----------------------------------------------------------------------------
# NORMAL-TISSUE LABEL HARMONISATION
# ----------------------------------------------------------------------------
# BUG FIX (external review, correct): the vital-organ list and the
# modality risk sets name "Salivary Gland", but GTEx v8 labels that tissue
# "Minor Salivary Gland". On LIVE data the organ would simply be absent -
# a warning would print, F2 would still be declared assessable, and the
# organ behind PSMA radioligand therapy's signature xerostomia would be
# silently missing from the safety comparator.
#
# Aliases are resolved explicitly, and the resolution is reported, so a
# label mismatch becomes visible instead of becoming a missing organ.
TISSUE_ALIASES = {
    "Salivary Gland": ["Minor Salivary Gland", "Salivary Gland - Minor",
                       "Salivary gland"],
    "Kidney - Cortex": ["Kidney_Cortex", "Kidney - Cortex (GTEx)", "Kidney"],
    "Heart - Left Ventricle": ["Heart_Left_Ventricle",
                               "Heart - Left ventricle"],
    "Brain - Cortex": ["Brain_Cortex", "Brain - Frontal Cortex (BA9)"],
    "Lung": ["Lung (GTEx)"],
    "Liver": ["Liver (GTEx)"],
    "Whole Blood": ["Blood", "Whole_Blood"],
    "Nerve - Tibial": ["Nerve_Tibial"],
}


def harmonise_tissue_labels(columns, verbose=True):
    """Map a normal-tissue panel's column labels onto the declared names.

    Returns {declared_name: actual_column}. Reports every resolution and
    every declared organ that could not be found, because a silently absent
    organ is indistinguishable from a safe one.
    """
    present = {str(c): str(c) for c in columns}
    lower = {str(c).strip().lower(): str(c) for c in columns}
    resolved, renamed, missing = {}, [], []
    for declared in list(dict.fromkeys(list(CFG.vital_organs))):
        if declared in present:
            resolved[declared] = declared
            continue
        hit = None
        for alias in TISSUE_ALIASES.get(declared, []):
            if alias in present:
                hit = alias
                break
            if alias.strip().lower() in lower:
                hit = lower[alias.strip().lower()]
                break
        if hit is None and declared.strip().lower() in lower:
            hit = lower[declared.strip().lower()]
        if hit is not None:
            resolved[declared] = hit
            renamed.append(f"{declared} <- {hit}")
        else:
            missing.append(declared)
    # SELF-REVIEW: the external review's Fix A specified both halves -
    # resolve the aliases AND "detect ambiguous or duplicate mappings" - and
    # I implemented only the resolution. Two declared organs resolving to the
    # SAME actual column would be counted as two organs present, inflating
    # vital-organ coverage and letting one measurement stand in for two
    # distinct safety questions. No alias in the current table overlaps, so
    # this is latent rather than active - but coverage is now a GATE input
    # (it decides whether F2 is definitive and whether a modality screen is
    # assessable), so a future one-line alias edit could silently open that
    # gate. A collision is a configuration error and is refused outright.
    _by_column = {}
    for _declared, _actual in resolved.items():
        _by_column.setdefault(_actual, []).append(_declared)
    _collisions = {c: d for c, d in _by_column.items() if len(d) > 1}
    if _collisions:
        raise ValueError(
            f"Ambiguous tissue harmonisation: {_collisions}. Each listed "
            f"column resolves to more than one declared vital organ, so one "
            f"measurement would be counted as several organs present - "
            f"inflating vital-organ coverage and answering two distinct "
            f"safety questions with the same data. Fix TISSUE_ALIASES so "
            f"each declared organ maps to a distinct column.")
    if verbose:
        if renamed:
            print(f"    tissue labels harmonised: {'; '.join(renamed)}")
        if missing:
            print(f"    VITAL ORGANS NOT MEASURED in this panel: {missing}. "
                  f"A missing organ is NOT a safe organ - F2 coverage is "
                  f"reduced accordingly.")
    return resolved, missing


# ----------------------------------------------------------------------------
# LINEAGE MARKERS RETAINED OUTSIDE THE SURFACEOME
# ----------------------------------------------------------------------------
# The metastatic matrix is bounded to the surfaceome everywhere else, which is
# correct for CANDIDATES. But the lineage markers needed to STRATIFY that
# cohort - AR for AR-pathway activity, the neuroendocrine panel for
# transdifferentiation - are not surface proteins and were therefore absent
# from the matrix in both DEMO and LIVE. Any analysis that splits mCRPC by
# lineage state would have been silently unrunnable on real data, not just in
# the demo. These genes are carried through the metastatic cohort as
# STRATIFYING VARIABLES ONLY; they are excluded from the surfaceome and can
# never enter the candidate ranking.
# Compartment markers for the epithelial-restriction score. Like the lineage
# markers below, these are NOT surface candidates and are stripped by
# surfaceome bounding - which is why the compartment analysis silently
# reported "epithelial=1, immune=0" and never ran. They are retained through
# the primary tumour matrix as STRATIFYING variables only.
COMPARTMENT_MARKER_GENES = [
    "EPCAM", "KRT8", "KRT18", "CDH1", "CLDN3", "KRT19",      # epithelial
    "PTPRC", "CD3E", "CD2", "LCK", "CD53", "ITGAM", "CSF1R",  # immune
    "COL1A1", "COL1A2", "ACTA2", "PDGFRB", "THY1",            # stromal
]

LINEAGE_MARKER_GENES = [
    "AR",                                   # AR-pathway activity
    "CHGA", "SYP", "INSM1", "ASCL1", "NCAM1",   # neuroendocrine programme
    "FOLH1",                                # PSMA strata (also a candidate)
]

SYNTHETIC_IDENTIFIER_PREFIXES = ("CTRL_", "NONSURF")


def is_synthetic_gene(genes):
    """True for identifiers the demo generator invented."""
    g = pd.Series(genes, dtype="object").astype(str)
    mask = g.isin(FILTER_CONTROLS)
    for pref in SYNTHETIC_IDENTIFIER_PREFIXES:
        mask |= g.str.startswith(pref)
    return mask.values


def drop_synthetic(df, col="gene", label="table", verbose=True):
    """Remove synthetic identifiers from anything biological.

    Applied to discovery, survival and reporting outputs. The software-
    validation artifacts (filter unit tests, bounding controls) deliberately
    keep them, because that is what they are for.
    """
    if col not in df.columns:
        return df
    mask = is_synthetic_gene(df[col])
    n = int(mask.sum())
    if n and verbose:
        names = sorted(set(df.loc[mask, col].astype(str)))[:6]
        print(f"    {label}: removed {n} synthetic identifier(s) "
              f"({', '.join(names)}{' ...' if n > len(names) else ''})")
    return df.loc[~mask].copy()
EXPECTED_CONTROL_FAILURES = {
    "CTRL_NOT_DE":     "F1_differential",
    "CTRL_HIGH_VITAL": "F2_safety",
    "CTRL_NO_ECD":     "F3_topology",
    "CTRL_NO_AB":      "F4_reagent",
}

def audit_labels(verbose=True):
    """Report how much of the positive-label set has verified provenance.

    PROPOSAL REFINEMENT (Stop/go criteria): 'unverifiable positive labels' is
    a declared STOP condition. This function makes that condition measurable
    rather than rhetorical. It does not block a DEMO run, and it does not fake
    verification - it reports the gap so the gap is visible.
    """
    genes = [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
    # EXTERNAL REVIEW, correct: a populated trial_id proves only that a field
    # was filled in. Verification requires the record to name its source and
    # when it was checked. Completeness and verification are now distinct, and
    # pct_verified - the number the STOP gate reads - uses the STRICTER
    # definition, so filling in a field cannot unlock the gate by itself.
    has_trial_id = [g for g in genes if g in LABEL_PROVENANCE
                    and LABEL_PROVENANCE[g].get("trial_id")]
    verified = [g for g in has_trial_id
                if all(LABEL_PROVENANCE[g].get(k)
                       for k in VERIFICATION_REQUIRED_FIELDS)]
    pct_complete = 100 * len(has_trial_id) / max(len(genes), 1)
    pct = 100 * len(verified) / max(len(genes), 1)
    unset = [k for k, v in LABEL_INCLUSION_RULE.items()
             if isinstance(v, str) and v.startswith(("UNSET", "UNDECIDED"))]
    if verbose:
        print(f"  label audit: {len(verified)}/{len(genes)} genes with a "
              f"recorded trial ID ({pct:.0f}%)")
        if unset:
            print(f"  inclusion rule fields still unresolved: {', '.join(unset)}")
        if pct < 100:
            print("  ACTION REQUIRED before the LIVE run: populate "
                  "LABEL_PROVENANCE from ClinicalTrials.gov, record the freeze "
                  "date, and decide the discontinued-programme rule. Deposit "
                  "the completed table as a supplementary file.")
    return {"n_labels": len(genes),
            "n_with_trial_id": len(has_trial_id),
            "pct_metadata_complete": round(pct_complete, 1),
            "n_verified": len(verified),
            "pct_verified": pct,
            "verification_requires": list(VERIFICATION_REQUIRED_FIELDS),
            "note": ("pct_verified counts labels carrying ALL verification "
                     "fields (source and date included), not merely a "
                     "populated trial_id; a filled field is not evidence "
                     "that a registry record was checked"),
            "unresolved_rule_fields": unset}


print(f"Run mode: {RUN_MODE} | profile: {PROFILE} "
      f"(expected runtime {_EST_RUNTIME[PROFILE]} on a 2-vCPU Colab instance)")
if PROFILE == "smoke":
    print("  SMOKE PROFILE: deliberately under-powered. Confidence intervals "
          "will be wide and rankings unstable. This profile proves the code "
          "runs; it does not produce reportable statistics.")
print(f"Output root: {OUT.resolve()}")
# SELF-REVIEW (round 25), found while fixing the same defect in the
# provenance template and NOT raised by any reviewer: this banner printed
# "Positives for training: 74", the size of the whole declared collection,
# while the PRIMARY model trains on the payload-delivery class alone. It is
# the first line of every run, so it was the most-read wrong number in the
# pipeline, and it is the number a reader would quote. The counts are now
# derived from the same constants the model reads.
_primary_pos = [g for g in CLINICAL_STAGE_TARGETS
                if not g.endswith("_HELD")
                and g not in set(PROSTATE_HELDOUT_POSITIVES)
                and label_class_of(g) == PRIMARY_LABEL_CLASS]
_withheld = [g for g in CLINICAL_STAGE_TARGETS
             if not g.endswith("_HELD")
             and g not in set(PROSTATE_HELDOUT_POSITIVES)
             and label_class_of(g) != PRIMARY_LABEL_CLASS]
print(f"Positives for training: {len(_primary_pos)} "
      f"({PRIMARY_LABEL_CLASS} class)")
print(f"  declared clinical-stage targets: "
      f"{len([g for g in CLINICAL_STAGE_TARGETS if not g.endswith('_HELD')])}"
      f"; {len(_withheld)} withheld from the primary model as off-class "
      f"({', '.join(sorted({label_class_of(g) for g in _withheld})) or 'none'})")
print(f"Held-out prostate benchmarks: {len(PROSTATE_HELDOUT_POSITIVES)}")


# ----------------------------------------------------------------------------
# 0.9 REPRODUCIBILITY MANIFEST (NEW - raised in external review)
# ----------------------------------------------------------------------------
# External review, correctly: "pin package versions, archive exact data
# releases, store source checksums, and preserve cohort/sample inclusion
# manifests". A run that cannot be reconstructed cannot be defended, and a
# figure regenerated two months later against a silently updated cBioPortal
# release is not the same figure.
#
# This writes ONE machine-readable manifest per run recording the interpreter,
# every scientific package version actually loaded, the resolved config, the
# SHA-256 of every input file read from disk, and the sample inclusion lists.
# It is written at the END of the run so it reflects what was really used, not
# what was intended.
# Wall-clock start of this run. Used to decide which files in the run-scoped
# output directories were actually written by THIS execution: a file older
# than the run start is stale and must not enter the archive.
RUN_START_TIME = time.time()

# EXPLICIT per-run artifact registry (external review P2). Modification-time
# filtering worked for the delivered archive but is fragile: RUN_START_TIME is
# set at import, so running run_pipeline() twice in one notebook session would
# let the FIRST run's artifacts qualify for the SECOND run's archive. Every
# write now records itself here, and the archive is built from this registry
# rather than from timestamps.
ARTIFACT_REGISTRY = {}


class isolated_input_registry:
    """Snapshot the input-file registry and restore it on exit.

    Tests must be able to call real loaders - that is the point of the
    LIVE-path suite - without their fixtures entering the provenance record
    of the scientific run. Registration is global by design, so the
    isolation belongs at the test boundary.
    """

    def __enter__(self):
        self._inputs = dict(INPUT_FILE_REGISTRY)
        self._samples = dict(SAMPLE_MANIFEST)
        return self

    def __exit__(self, *exc):
        INPUT_FILE_REGISTRY.clear()
        INPUT_FILE_REGISTRY.update(self._inputs)
        SAMPLE_MANIFEST.clear()
        SAMPLE_MANIFEST.update(self._samples)
        return False


def register_artifact(path, kind="file"):
    """Record that THIS run wrote this file."""
    try:
        rp = str(Path(path).resolve())
        ARTIFACT_REGISTRY[rp] = {"kind": kind, "run_id": RUN_ID}
    except Exception:
        pass
    return path


def new_run_identity():
    """Mint a fresh RUN_ID for this invocation.

    BUG FIX (both external reviews, correct). RUN_ID was bound once, when
    the module loaded. In a notebook - where re-running the pipeline cell is
    the normal way to work - a second invocation carried the FIRST run's
    identity, so two different analyses were stamped with the same run_id
    and the second silently overwrote the first's artifacts. Any later
    attempt to trace a number back to the run that produced it would find
    two candidates with one name.

    An explicitly supplied RUN_ID (from the environment) is honoured, so a
    caller that wants to pin the identity still can.
    """
    global RUN_ID
    if os.environ.get("RUN_ID"):
        RUN_ID = os.environ["RUN_ID"]
        return RUN_ID
    RUN_ID = (pd.Timestamp.now().strftime("%Y%m%dT%H%M%S") + "-"
              + _uuid.uuid4().hex[:6])
    return RUN_ID


def reset_artifact_registry():
    """Clear the registry at the start of a pipeline invocation, so a second
    run in the same session cannot inherit the first run's artifacts."""
    ARTIFACT_REGISTRY.clear()
    TABLE_REGISTRY.clear()
    FIGURE_REGISTRY.clear()
    # BUG FIX (external review, confirmed): the INPUT registry was not
    # cleared, so a second run in the same session inherited the first run's
    # input checksums - and worse, the LIVE-path tests register their own
    # temporary fixtures as analysis inputs, so the shipped manifest listed
    # three files in /tmp/livefix_* as the provenance of the scientific run
    # and nothing else. A provenance manifest that names test fixtures as its
    # inputs is worse than an empty one.
    INPUT_FILE_REGISTRY.clear()
    SAMPLE_MANIFEST.clear()

INPUT_FILE_REGISTRY = {}     # path -> {"sha256":..., "bytes":..., "role":...}
SAMPLE_MANIFEST = {}         # cohort -> list of sample identifiers used


def register_input_file(path, role="", sha256=None):
    """Record the SHA-256 of an input file so the run can be reconstructed.

    SPEED FIX (self-review of round 25, a regression I introduced): the new
    cache validator computes a SHA-256 to compare against the sidecar, and
    then this function computed the SAME digest again. On a LIVE run that is
    ~5 GB hashed twice; through a Drive FUSE mount, where reads are an order
    of magnitude slower than local disk, that is minutes of pure waste on
    every re-run. `sha256` lets the caller hand over a digest it has
    already computed over the identical bytes.
    """
    import hashlib
    try:
        p = Path(path)
        if not p.is_file():
            return None
        if sha256:
            digest = sha256
        else:
            h = hashlib.sha256()
            with open(p, "rb") as fh:
                for chunk in iter(lambda: fh.read(1 << 20), b""):
                    h.update(chunk)
            digest = h.hexdigest()
        INPUT_FILE_REGISTRY[str(p)] = {
            "sha256": digest,
            "bytes": p.stat().st_size,
            "modified_utc": pd.Timestamp(p.stat().st_mtime, unit="s",
                                         tz="UTC").isoformat(),
            "role": role,
        }
        return INPUT_FILE_REGISTRY[str(p)]["sha256"]
    except Exception as e:
        # BUG FIX (both external reviews, correct). This swallowed every
        # hashing failure and returned None. In LIVE that means an analysis
        # input with NO recorded provenance, while the run proceeds and the
        # manifest simply has one fewer entry - unobservable unless someone
        # counts. A LIVE run whose inputs cannot be checksummed cannot be
        # reconstructed, so it cannot support a published number.
        #
        # DEMO stays non-fatal: its inputs are synthetic and regenerated
        # from a seed, so a checksum adds nothing there.
        if RUN_MODE == "LIVE":
            raise RuntimeError(
                f"Could not compute the SHA-256 of a LIVE analysis input: "
                f"{path}\n"
                f"  Cause: {type(e).__name__}: {e}\n"
                f"  A LIVE run with an unchecksummed input cannot be "
                f"reconstructed, so it is stopped here rather than "
                f"producing results whose provenance is incomplete. Check "
                f"the file is readable and the volume is healthy (on Colab, "
                f"a stalled Drive mount is the usual cause)."
            ) from e
        print(f"    checksum skipped for {path}: {e}")
        return None


def register_samples(cohort, samples):
    """Record exactly which samples entered an analysis."""
    try:
        SAMPLE_MANIFEST[cohort] = [str(s) for s in list(samples)]
    except Exception:
        pass


def write_run_manifest(extra=None):
    """Write outputs/06_logs/run_manifest.json. Call at the end of a run."""
    import platform
    import importlib
    pkgs = {}
    for mod in ["numpy", "pandas", "scipy", "sklearn", "statsmodels",
                "matplotlib", "seaborn", "lifelines", "pydeseq2",
                "requests", "joblib"]:
        try:
            pkgs[mod] = getattr(importlib.import_module(mod), "__version__",
                                "unknown")
        except Exception:
            pkgs[mod] = "not installed"
    man = {
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "profile": PROFILE,
        "written_utc": pd.Timestamp.utcnow().isoformat(),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "packages": pkgs,
        "config": {k: (list(v) if isinstance(v, tuple) else v)
                   for k, v in vars(CFG).items()
                   if not k.startswith("_")},
        "input_files_sha256": INPUT_FILE_REGISTRY,
        "n_input_files_checksummed": len(INPUT_FILE_REGISTRY),
        "sample_manifest_sizes": {k: len(v) for k, v in SAMPLE_MANIFEST.items()},
        "sample_manifest": SAMPLE_MANIFEST,
        "environment_overrides": {k: v for k, v in os.environ.items()
                                  if k.startswith(("SU2C_", "COUNTS_",
                                                   "SURFACE_", "CPTAC_",
                                                   "PROT_"))},
    }
    if extra:
        man.update(extra)
    dest = OUT / "06_logs" / "run_manifest.json"
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "w") as fh:
        json.dump(man, fh, indent=1, default=str)
    # BUG FIX (all three external reviews, correct and important): the
    # manifest was WRITTEN but never REGISTERED. export_outputs() ships the
    # artifact registry only - the modification-time fallback was removed
    # precisely so that nothing unregistered could ride into the archive -
    # so run_manifest.json was silently dropped from every release while
    # run_summary.json recorded "run_manifest_written": true. The archive
    # checksum audit cannot catch this: it verifies that every listed file
    # matches, and a file that was never listed is not a mismatch. This is
    # the same failure shape as the mtime fallback - a mechanism that cannot
    # tell "clean" from "absent" - and it cost the release its single most
    # important reproducibility artifact.
    register_artifact(dest, kind="run_manifest")
    print(f"    run manifest: {dest} "
          f"({len(pkgs)} package versions, "
          f"{len(INPUT_FILE_REGISTRY)} input checksums, "
          f"{len(SAMPLE_MANIFEST)} cohort sample lists)")
    if RUN_MODE == "LIVE" and not INPUT_FILE_REGISTRY:
        # BLOCKING in LIVE: a real-data run with no input checksums cannot be
        # reconstructed, so the archive cannot support a published claim.
        raise RuntimeError(
            "LIVE run recorded ZERO input-file checksums. The analysis is not "
            "reconstructible: a silently updated portal release would produce "
            "different results from the same code with no way to detect it. "
            "Ensure every loader calls register_input_file() on the files it "
            "resolves, then re-run. (DEMO runs are exempt because their data "
            "are generated, not read.)")
    return man

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Run mode: DEMO | profile: smoke (expected runtime ~3-5 min on a 2-vCPU Colab instance)
  SMOKE PROFILE: deliberately under-powered. Confidence intervals will be wide and rankings unstable. This profile proves the code runs; it does not produce reportable statistics.
Output root: /content/outputs
Positives for training: 57 (payload_delivery class)
  declared clinical-stage targets: 74; 17 withheld from the primary model as off-class (immune_compartment, signal_blockade)
Held-out prostate benchmarks: 6


---
## Section 1 — Data acquisition

Each loader returns a documented schema and works identically in DEMO and LIVE mode, so code written against DEMO runs unchanged against the real cohorts.

**Why UCSC Xena Toil rather than the primary portals.** Filter 2 compares tumour expression against normal vital-organ expression. TCGA and GTEx were sequenced years apart with different library preparation and different quantification pipelines, so comparing their native processed files confounds biology with batch. The Toil recompute re-aligns and re-quantifies TCGA, TARGET and GTEx through one pipeline, which is what makes a tumour-to-vital-organ ratio interpretable at all.

In [3]:
# =============================================================================
# SECTION 1 - DATA ACQUISITION
# =============================================================================
# Every loader returns a DataFrame with a documented schema and is wrapped in a
# DEMO/LIVE switch. The schemas are identical in both modes, so code written
# against DEMO runs unchanged against LIVE.
#
# CRITICAL DESIGN DECISION - why UCSC Xena Toil and not raw TCGA + raw GTEx:
#   The study's second gatekeeping filter compares tumour expression to normal
#   vital-organ expression. TCGA and GTEx were sequenced years apart with
#   different library prep and different quantification pipelines. Comparing
#   them directly confounds biology with batch. The Toil recompute (Vivian et
#   al., Nat Biotechnol 2017) re-aligns and re-quantifies TCGA, TARGET and GTEx
#   through ONE pipeline, which is what makes a tumour-vs-vital-organ ratio
#   interpretable. Using the primary portals' own processed files instead would
#   make Filter 2 uninterpretable.
# =============================================================================

# ----------------------------------------------------------------------------
# 1.1 Download helper with integrity checking
# ----------------------------------------------------------------------------
def _network_failure(exc, url, what):
    """Turn a transport exception into an actionable message.

    Found by actually attempting a LIVE run: the first network call failed
    with a raw 20-line urllib3/requests traceback ending in ProxyError. It
    named no host a reader would recognise, gave no cause and no remedy. A
    user on Colab hitting a transient outage, a VPN, a corporate proxy or a
    sandbox allowlist gets exactly that wall of text and no idea which of
    those it is - and this is the FIRST thing the LIVE path does, so it is
    the first impression of the whole run.
    """
    from urllib.parse import urlparse
    host = urlparse(url).netloc or url
    kind = type(exc).__name__
    blocked = "proxy" in str(exc).lower() or "403" in str(exc)
    return RuntimeError(
        f"LIVE data acquisition failed while fetching {what} from {host} "
        f"({kind}).\n"
        f"  What this means: the pipeline could not reach {host}. "
        + ("The connection was refused by a PROXY, which usually means the "
           "environment restricts outbound network access to an allowlist "
           "that does not include this host.\n"
           if blocked else
           "The host did not respond, which is usually a transient outage, "
           "a VPN/firewall, or no internet access.\n")
        + f"  What to do:\n"
          f"    1. Confirm this machine can reach {host} at all, e.g. "
          f"curl -I {url.split('?')[0][:90]}\n"
          f"    2. On Google Colab these hosts are normally reachable; in a "
          f"restricted sandbox or behind a corporate proxy they are often "
          f"not.\n"
          f"    3. If the host is unreachable by policy, download the file "
          f"elsewhere and place it in {DIRS['raw']} under the expected "
          f"name - the loaders reuse a cached copy and skip the download.\n"
          f"    4. Set RUN_MODE='DEMO' to exercise the code without "
          f"network access (synthetic data; NOT reportable as biology).\n"
          f"  Original error: {str(exc)[:300]}")


_CACHE_INTEGRITY = os.environ.get("CACHE_INTEGRITY", "1") not in ("0", "false",
                                                                   "False")

# Digests computed while validating a cached file, reused by
# register_input_file so the same bytes are never hashed twice in one run.
_VALIDATED_DIGESTS = {}

# How many times the full Toil matrix has been parsed from source in this
# process. The filtered-matrix cache exists to keep this at 1.
TOIL_PARSE_COUNT = {"n": 0}


def _magic_ok(path):
    """Reject a cached file whose first bytes are not the format its name
    claims. The failure this catches is specific and common: a portal that
    answers a download URL with an HTML error page, or a session that saved
    a login redirect under the data file's name. Both are the right SIZE."""
    name = str(path).lower()
    with open(path, "rb") as fh:
        head = fh.read(512)
    if not head:
        return False, "file is empty"
    if head[:14].lstrip()[:1] in (b"<",) or b"<html" in head[:512].lower():
        return False, "file begins with HTML, not data (an error page)"
    if name.endswith(".gz") and head[:2] != b"\x1f\x8b":
        return False, "missing gzip magic bytes"
    if name.endswith(".zip") and head[:2] not in (b"PK",):
        return False, "missing zip magic bytes"
    return True, ""


def validate_cached_input(path, expect_min_bytes=10_000,
                          transfer_length_verified=False):
    """Decide whether an existing file may be REUSED instead of downloaded.

    BUG FIX (all three external reviews, correct; reproduced locally with a
    junk file of sufficient size): reuse was gated on
        dest.exists() and dest.stat().st_size >= expect_min_bytes
    alone, so any large file sitting at the expected path was accepted as
    the real dataset. On Colab that is not hypothetical - an interrupted
    session, a half-synced Drive folder or a stale copy from another project
    all produce exactly that. The failure then surfaces hundreds of lines
    later as an unintelligible parser error, or worse, as plausible numbers
    computed from the wrong file.

    The checks are deliberately BOUNDED. Fully decompressing a multi-GB
    expression matrix on every run to prove it is intact would cost more
    than re-downloading it, so this reads magic bytes, then asks the archive
    library for the central directory / member list - which is cheap and is
    precisely the structure a truncated transfer destroys.

    Returns (ok: bool, reason: str). Never raises on a corrupt file; the
    caller decides whether to re-download or stop.
    """
    path = Path(path)
    # BUG FIX (both external reviews, independently reproduced, and I
    # confirmed it here first). _VALIDATED_DIGESTS memoised a digest per
    # PATH, and nothing invalidated it when the bytes at that path changed.
    # The reproduction: validate a file (digest A memoised), delete its
    # sidecar, replace the contents with different valid data B, validate
    # again -> returns success while the memo still holds A. download()
    # then hands A to register_input_file(), so
    #   - the run manifest records digest A for bytes B, and
    #   - the filtered-matrix cache key is built from A,
    # which means a CHANGED source matrix silently reuses the OLD processed
    # matrix. Plausible numbers, wrong data, and a manifest that attests to
    # the wrong input. That is a scientific-validity failure, and it was
    # introduced by my own speed fix last round - the memo existed only to
    # avoid hashing the same bytes twice.
    #
    # The memo is dropped at the START of every validation. It is
    # repopulated only on the path that actually hashes the CURRENT bytes,
    # so a digest can never outlive the content it describes.
    _VALIDATED_DIGESTS.pop(str(path), None)
    if not path.is_file():
        return False, "no cached copy"
    size = path.stat().st_size
    if size < expect_min_bytes:
        return False, (f"cached copy is {size/1e6:.2f} MB, below the "
                       f"{expect_min_bytes/1e6:.2f} MB minimum for this file")
    if not _CACHE_INTEGRITY:
        return True, "size only (CACHE_INTEGRITY=0)"
    ok, why = _magic_ok(path)
    if not ok:
        return False, why
    name = str(path).lower()
    sidecar = path.with_suffix(path.suffix + ".sha256")
    # BUG FIX (both external reviews, independently reproduced, and I
    # reproduced it here before changing anything - this is the most serious
    # defect of the round because it affects CORRECTNESS, not speed).
    #
    # The structural check decompressed only the first 64 KB. A gzip file
    # truncated by its last EIGHT bytes - losing the CRC32 and ISIZE
    # trailer, which is exactly what an interrupted transfer or a half-synced
    # Drive folder produces - has a perfectly valid 64 KB prefix, so it was
    # ACCEPTED, while full decompression raises EOFError. My own regression
    # test claimed to cover this and did not: it truncated the file to a
    # third of its length, which breaks the stream inside the first 64 KB.
    # The test was correct about what it tested and never reached the real
    # case, so it produced false confidence rather than coverage.
    #
    # The rule now depends on whether the file's identity is ESTABLISHED:
    #   - sidecar present: the SHA-256 covers every byte, so a bounded
    #     structural check is enough and the expensive full decompression is
    #     unnecessary. A checksum match is strictly stronger evidence than
    #     any structural walk.
    #   - sidecar absent: nothing attests to these bytes, so the whole
    #     compressed stream is walked to the end. That costs a full read,
    #     which is the correct price for admitting an unattested file as a
    #     scientific input - and it happens once, because a sidecar is
    #     written afterwards.
    # Attested means "something independent vouches for every byte": a
    # stored checksum, or a transfer whose received length matched the
    # length the server declared. Either makes the full structural walk
    # redundant. Neither is assumed - the caller has to have established it.
    _attested = sidecar.is_file() or bool(transfer_length_verified)
    _bound = (1 << 16) if _attested else None
    try:
        if name.endswith(".tar.gz") or name.endswith(".tgz"):
            with tarfile.open(path) as tf:
                if _attested:
                    if tf.next() is None:
                        return False, "tar archive contains no members"
                else:
                    _members = tf.getmembers()
                    if not _members:
                        return False, "tar archive contains no members"
        elif name.endswith(".zip"):
            import zipfile as _zf
            with _zf.ZipFile(path) as za:
                if not za.namelist():
                    return False, "zip archive contains no members"
                if not _attested:
                    _bad = za.testzip()
                    if _bad is not None:
                        return False, (f"zip member failed its CRC: {_bad} - "
                                       f"the archive is damaged")
        elif name.endswith(".gz"):
            import gzip as _gz
            with _gz.open(path, "rb") as fh:
                if _bound is not None:
                    if not fh.read(_bound):
                        return False, "gzip stream decompresses to nothing"
                else:
                    # Walk to the end. gzip raises EOFError/BadGzipFile on a
                    # truncated stream only when the trailer is reached, so
                    # a bounded read cannot detect end-truncation at all.
                    _total = 0
                    while True:
                        _blk = fh.read(1 << 22)
                        if not _blk:
                            break
                        _total += len(_blk)
                    if _total == 0:
                        return False, "gzip stream decompresses to nothing"
    except Exception as exc:
        return False, (f"archive structure is unreadable "
                       f"({type(exc).__name__}: {str(exc)[:120]}) - the "
                       f"transfer was probably truncated")
    sidecar = path.with_suffix(path.suffix + ".sha256")
    if sidecar.is_file():
        import hashlib as _hl
        h = _hl.sha256()
        with open(path, "rb") as fh:
            for blk in iter(lambda: fh.read(1 << 22), b""):
                h.update(blk)
        got = h.hexdigest()
        want = sidecar.read_text().split()[0].strip()
        if got != want:
            return False, (f"checksum mismatch against {sidecar.name}: the "
                           f"cached file is not the one this pipeline "
                           f"downloaded")
        # Hand the digest back so the caller does not hash these bytes a
        # second time. See register_input_file()'s sha256 parameter.
        _VALIDATED_DIGESTS[str(path)] = got
        return True, "validated (checksum)"
    return True, "validated"


def sha256_file(path, block=1 << 22):
    """SHA-256 of a file's bytes."""
    import hashlib as _hl
    h = _hl.sha256()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(block), b""):
            h.update(blk)
    return h.hexdigest()


def confirm_input_validated(path, what=""):
    """Promote a provisional digest once the file has been fully read.

    Called by a loader AFTER it has parsed the file end to end. Until then
    the file carries only a provisional digest and validate_cached_input()
    walks its whole compressed stream on every run - the correct price for a
    file nothing has yet vouched for.
    """
    path = Path(path)
    prov = path.with_suffix(path.suffix + ".sha256.provisional")
    final = path.with_suffix(path.suffix + ".sha256")
    if final.is_file():
        return False
    if not prov.is_file():
        return False
    try:
        final.write_text(prov.read_text())
        prov.unlink()
        print(f"    validated end to end{(' (' + what + ')') if what else ''}"
              f": {path.name} - later runs reuse it on its checksum")
        return True
    except Exception:
        return False


_HTTP_PERMANENT = {400, 401, 403, 404, 405, 410, 451}


def download(url, dest, expect_min_bytes=10_000, retries=3):
    """Stream a URL to disk, with bounded retry and actionable failures.

    BUG FIX (all three external reviews, correct): only the preliminary HEAD
    probe was wrapped. The streaming GET and raise_for_status() sat outside
    any handler, so the single most likely real-world failure - the transfer
    itself dying, or the host answering 403/429/500 - escaped as a raw
    requests traceback, bypassing the actionable message entirely. The HEAD
    probe was also treated as a gate: a failed HEAD aborted the download,
    although plenty of object stores answer HEAD incorrectly while serving
    GET perfectly. HEAD is now advisory only.
    """
    dest = Path(dest)
    ok, why = validate_cached_input(dest, expect_min_bytes)
    if ok:
        print(f"  cached: {dest.name} ({dest.stat().st_size/1e6:.1f} MB, "
              f"{why})")
        register_input_file(dest, role=f"downloaded input: {url}",
                            sha256=_VALIDATED_DIGESTS.get(str(dest)))
        return dest
    if dest.exists():
        quarantine = dest.with_suffix(dest.suffix + ".rejected")
        print(f"  REJECTED cached {dest.name}: {why}")
        print(f"    moved to {quarantine.name}; re-downloading")
        try:
            if quarantine.exists():
                quarantine.unlink()
            dest.rename(quarantine)
        except Exception:
            pass
    print(f"  downloading: {url}")
    try:
        _probe = requests.head(url, timeout=60, allow_redirects=True)
        _probe.close()
    except Exception as _e:
        # ADVISORY ONLY. Some object stores answer HEAD with an error while
        # serving GET correctly, so a HEAD failure must not decide the run.
        print(f"    (HEAD probe failed: {type(_e).__name__}; "
              f"continuing to GET, which is what actually matters)")
    tmp = dest.with_suffix(dest.suffix + ".part")
    last = None
    for attempt in range(1, int(retries) + 1):
        try:
            with requests.get(url, stream=True, timeout=600) as r:
                if r.status_code in _HTTP_PERMANENT:
                    raise RuntimeError(
                        f"LIVE data acquisition failed: {url} returned HTTP "
                        f"{r.status_code} ({r.reason}). This is a PERMANENT "
                        f"error and retrying will not help. The usual causes "
                        f"are a moved object (update DATA_URLS), a portal "
                        f"that now requires authentication, or an "
                        f"environment whose allowlist blocks this host. "
                        f"Download the file elsewhere and place it at "
                        f"{dest} - the loader reuses a cached copy.")
                r.raise_for_status()
                ctype = r.headers.get("Content-Type", "")
                if "text/html" in ctype:
                    raise RuntimeError(
                        f"URL returned HTML, not data: {url}\n"
                        f"The portal path has probably changed. Resolve the "
                        f"current download path from the portal UI and "
                        f"update DATA_URLS.")
                _declared = r.headers.get("Content-Length")
                _written = 0
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                        _written += len(chunk)
                # SPEED + PRECISION FIX (self-review of this round's own
                # change). Hardening the cache validator to walk a whole
                # gzip stream was right for a file of unknown origin, but
                # download() calls that validator immediately after a
                # transfer, BEFORE the sidecar exists - so a fresh LIVE
                # download of the Toil matrix paid a full ~4 GB decompress,
                # then a full hash pass, and then the chunked pandas parse:
                # three passes over multi-gigabyte data where one was
                # needed. On a Colab vCPU that is minutes added to the very
                # run this round was meant to speed up.
                #
                # Content-Length is exact, costs nothing, and answers the
                # only question the full walk was asked here: did every
                # byte arrive? When the server declares a length and the
                # bytes written match it, the transfer is complete by
                # construction and the expensive walk is redundant. When
                # the server declares nothing (chunked encoding), there is
                # no such evidence and the walk still runs.
                if _declared is not None:
                    try:
                        _want = int(_declared)
                    except ValueError:
                        _want = None
                    if _want is not None and _written != _want:
                        raise RuntimeError(
                            f"Incomplete transfer: received "
                            f"{_written:,} bytes but the server declared "
                            f"{_want:,} for {url}. The partial file has "
                            f"been discarded; re-run to resume.")
                    _length_verified = _want is not None
                else:
                    _length_verified = False
            if tmp.stat().st_size < expect_min_bytes:
                raise RuntimeError(
                    f"Downloaded file implausibly small "
                    f"({tmp.stat().st_size/1e6:.3f} MB < "
                    f"{expect_min_bytes/1e6:.3f} MB): {url}. The transfer "
                    f"was cut short or the object is not what was expected.")
            tmp.rename(dest)
            ok, why = validate_cached_input(
                dest, expect_min_bytes,
                transfer_length_verified=_length_verified)
            if not ok:
                dest.rename(tmp.with_suffix(".corrupt"))
                raise RuntimeError(
                    f"Downloaded file failed integrity validation ({why}): "
                    f"{url}. It has NOT been accepted as an analysis input.")
            break
        except RuntimeError:
            _cleanup_partial(tmp)
            raise
        except Exception as exc:
            # Transport-level: transient by assumption, so retry with
            # backoff. A permanent HTTP status was already raised above as
            # RuntimeError and does not reach here.
            last = exc
            _cleanup_partial(tmp)
            if attempt < int(retries):
                wait = 2 ** attempt
                print(f"    attempt {attempt}/{retries} failed "
                      f"({type(exc).__name__}); retrying in {wait}s")
                time.sleep(wait)
            else:
                raise _network_failure(exc, url, Path(dest).name) from exc
    print(f"  done: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
    _fresh_digest = None
    try:
        import hashlib as _hl
        h = _hl.sha256()
        with open(dest, "rb") as fh:
            for blk in iter(lambda: fh.read(1 << 22), b""):
                h.update(blk)
        _fresh_digest = h.hexdigest()
        # BUG FIX (external review, correct). The sidecar used to be written
        # the moment the bytes landed, which made it attest to "a transfer
        # finished here" - and validate_cached_input() then treated it as
        # grounds to SKIP the full-stream check on every later run. A
        # complete transfer of a file that is itself damaged upstream, or
        # whose schema is not what this pipeline needs, would be blessed
        # permanently by its own download.
        #
        # The digest is now PROVISIONAL until a loader has read the whole
        # file and parsed it. confirm_input_validated() promotes it. So the
        # promoted sidecar means "every byte of this file was read and
        # understood", which is what the attestation was supposed to claim -
        # and it is cheaper, because the expensive walk then never repeats.
        dest.with_suffix(dest.suffix + ".sha256.provisional").write_text(
            f"{_fresh_digest}  {dest.name}\n")
    except Exception:
        pass
    register_input_file(dest, role=f"downloaded input: {url}",
                        sha256=_fresh_digest)
    return dest


def _cleanup_partial(tmp):
    """Remove a partial transfer. A .part file left behind is harmless to
    this run - the name differs from the destination - but it fills a Colab
    disk across retries and can be mistaken for a usable cache."""
    try:
        if Path(tmp).exists():
            Path(tmp).unlink()
    except Exception:
        pass


# Download paths are isolated here so that URL rot is a one-line fix rather
# than a scavenger hunt. VERIFY THESE BEFORE A LIVE RUN - public portals
# reorganise their object stores without notice.
# =============================================================================
# DATA INPUTS - the single declaration of url, filename and minimum size
# =============================================================================
# BUG FIX (self-review, round 30). The disk preflight added last round
# derived each file's name from its URL, while the loaders pass explicit
# names. They disagreed for FOUR of six sources: the URL basename for the
# expression matrix is TcgaTargetGtex_rsem_gene_tpm.gz, the loader writes
# toil_tpm.gz. So on a fully warm run the preflight saw four sources as
# missing and demanded about 3.1 GB it did not need - breaking the warm-run
# fix in the very round that introduced it. The minimum sizes disagreed too
# (preflight 500 MB against the loader's 100 MB for the same file).
#
# The cause is not the wrong string in one place; it is that two places
# independently decided the same fact with nothing checking they agreed.
# Both now read this table, and a regression test asserts that every
# download() call site goes through input_path()/input_min(), so the two
# cannot drift apart again.
DATA_INPUTS = {
    "toil_tpm": {
        "url": "https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/TcgaTargetGtex_rsem_gene_tpm.gz",
        "dest": "toil_tpm.gz", "min_bytes": int(1e8), "approx_gb": 2.6},
    "toil_pheno": {
        "url": "https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/TcgaTargetGTEX_phenotype.txt.gz",
        "dest": "toil_pheno.txt.gz", "min_bytes": int(1e5),
        "approx_gb": 0.01},
    "su2c": {
        "url": "https://cbioportal-datahub.s3.amazonaws.com/prad_su2c_2019.tar.gz",
        "dest": "prad_su2c_2019.tar.gz", "min_bytes": int(1e6),
        "approx_gb": 0.35},
    "tcga_prad": {
        "url": "https://cbioportal-datahub.s3.amazonaws.com/prad_tcga_pan_can_atlas_2018.tar.gz",
        "dest": "prad_tcga.tar.gz", "min_bytes": int(1e6), "approx_gb": 0.45},
    "hpa": {
        "url": "https://www.proteinatlas.org/download/proteinatlas.tsv.zip",
        "dest": "proteinatlas.tsv.zip", "min_bytes": int(1e6),
        "approx_gb": 0.09},
    "toil_probemap": {
        "url": "https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/probeMap%2Fgencode.v23.annotation.gene.probemap",
        "dest": "gencode_probemap.tsv", "min_bytes": 1_000_000,
        "approx_gb": 0.01},
}


def input_path(key):
    """Where this input lives on disk. Reads DIRS at call time, so a test
    that redirects DIRS['raw'] is followed."""
    return DIRS["raw"] / DATA_INPUTS[key]["dest"]


def input_min(key):
    """The minimum plausible size for this input, in bytes."""
    return DATA_INPUTS[key]["min_bytes"]


DATA_URLS = {_k: _v["url"] for _k, _v in DATA_INPUTS.items()}

# (The former literal DATA_URLS dict lived here; it is now derived from
# DATA_INPUTS above so the url, the filename and the size floor cannot
# disagree.)

# UniProt REST is queried programmatically rather than downloaded as a file.
UNIPROT_STREAM = "https://rest.uniprot.org/uniprotkb/stream"


# ----------------------------------------------------------------------------
# 1.2 Surfaceome bounding
# ----------------------------------------------------------------------------
# Schema returned:
#   gene(str) uniprot(str) n_transmem(int) ecd_len_aa(int) has_gpi(bool)
#   is_single_pass(bool) seq_len(int)
#
# The ECD length is computed from UniProt TOPO_DOM "Extracellular" spans.
# This matters because Filter 3 is not "is it a membrane protein" - plenty of
# membrane proteins have no targetable outside. It is "is there enough protein
# outside the cell for an antibody to bind", which is a length question.
# ----------------------------------------------------------------------------
def load_surfaceome_live():
    """Query UniProt for reviewed human proteins with membrane topology."""
    params = {
        "query": ("(organism_id:9606) AND (reviewed:true) AND "
                  "((ft_transmem:*) OR (ft_lipid:*) OR (cc_scl_term:SL-0112))"),
        "fields": "accession,gene_primary,length,ft_transmem,ft_topo_dom,ft_lipid",
        "format": "tsv",
    }
    cache = DIRS["raw"] / "uniprot_membrane_raw.tsv"
    qfile = DIRS["raw"] / "uniprot_membrane_query.json"
    # BUG FIX (two external reviews, correct - and the consequence is larger
    # than either stated). The response was written to disk but never read
    # back, so every run re-queried UniProt. Two separate harms:
    #   1. Needless network dependence on a host that may be slow or blocked.
    #   2. SCIENTIFIC: this query defines the GENE UNIVERSE the whole study
    #      analyses. It was neither cached nor checksummed, so the surfaceome
    #      was silently whatever UniProt served that day, and the run
    #      manifest recorded no evidence of which release that was. Two runs
    #      of the identical code months apart would analyse different gene
    #      sets with nothing in the archive to show it.
    # The cache is keyed on the query itself, so changing the query
    # invalidates it rather than silently reusing the previous universe.
    import hashlib as _hl
    qkey = _hl.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()
    if cache.is_file() and qfile.is_file():
        try:
            prev = json.loads(qfile.read_text())
        except Exception:
            prev = {}
        if prev.get("query_sha256") == qkey:
            raw = pd.read_csv(cache, sep="\t")
            print(f"  cached UniProt membrane proteome: {len(raw):,} rows "
                  f"(retrieved {prev.get('retrieved_utc', 'unknown')})")
            register_input_file(cache, role="UniProt membrane proteome "
                                            "(surfaceome gene universe)")
            return _parse_uniprot_topology(raw)
        print("  UniProt query changed since the cached copy; re-querying")
    print("  querying UniProt REST for human membrane proteome ...")
    try:
        r = requests.get(UNIPROT_STREAM, params=params, timeout=600)
        r.raise_for_status()
    except Exception as _e:
        # SELF-REVIEW: raise_for_status() was OUTSIDE the handler here too,
        # the same defect the reviews found in download(). An HTTP 503 from
        # UniProt produced a bare HTTPError on the very first LIVE call.
        raise _network_failure(_e, UNIPROT_STREAM,
                               "the UniProt human membrane proteome") from _e
    from io import StringIO
    raw = pd.read_csv(StringIO(r.text), sep="\t")
    if raw.empty:
        raise RuntimeError(
            "UniProt returned an EMPTY membrane proteome. The gene universe "
            "for the entire analysis would be empty, so the run is stopped "
            "here rather than producing an empty ranking. Check the query "
            "and UniProt service status.")
    raw.to_csv(cache, sep="\t", index=False)
    qfile.write_text(json.dumps({
        "query_sha256": qkey,
        "params": params,
        "retrieved_utc": pd.Timestamp.utcnow().isoformat(),
        "n_rows": int(len(raw)),
        "uniprot_release": r.headers.get("X-UniProt-Release", "not reported"),
        "release_date": r.headers.get("X-UniProt-Release-Date",
                                      "not reported"),
    }, indent=1))
    register_input_file(cache, role="UniProt membrane proteome "
                                    "(surfaceome gene universe)")
    register_input_file(qfile, role="UniProt query + release provenance")
    print(f"  UniProt membrane proteome: {len(raw):,} rows "
          f"(release {r.headers.get('X-UniProt-Release', 'not reported')})")
    return _parse_uniprot_topology(raw)


def _parse_uniprot_topology(raw):
    """Turn UniProt's free-text feature strings into numeric topology features."""
    import re

    def n_tm(s):
        return 0 if not isinstance(s, str) else s.count("TRANSMEM ")

    def ecd_len(s):
        """Sum the lengths of all spans annotated Extracellular."""
        if not isinstance(s, str):
            return 0
        total = 0
        # UniProt TSV encodes features as: TOPO_DOM 1..50; /note="Extracellular"
        for m in re.finditer(r"TOPO_DOM\s+(\d+)\.\.(\d+);\s*/note=\"([^\"]+)\"", s):
            start, end, note = int(m.group(1)), int(m.group(2)), m.group(3)
            if "extracellular" in note.lower():
                total += end - start + 1
        return total

    def has_gpi(s):
        return isinstance(s, str) and "GPI-anchor" in s

    out = pd.DataFrame({
        "gene": raw["Gene Names (primary)"],
        "uniprot": raw["Entry"],
        "seq_len": raw["Length"],
        "n_transmem": raw["Transmembrane"].map(n_tm),
        "ecd_len_aa": raw["Topological domain"].map(ecd_len),
        "has_gpi": raw["Lipidation"].map(has_gpi),
    })
    out = out.dropna(subset=["gene"])
    out["is_single_pass"] = out["n_transmem"] == 1
    # One row per gene: keep the isoform with the longest annotated ECD, since
    # that is the one an antibody programme would target.
    out = (out.sort_values("ecd_len_aa", ascending=False)
              .drop_duplicates("gene", keep="first")
              .reset_index(drop=True))
    return out


def load_surfaceome_demo(n_surface=2800, n_background=4200):
    """Synthetic protein annotation over a WHOLE transcriptome, not just the
    surfaceome.

    Background genes are included deliberately, with zero transmembrane
    domains, so that the surfaceome-bounding step in the pipeline is a real
    operation that can succeed or fail rather than a no-op. A bounding step
    tested only against genes that are already surface proteins tests nothing.

    Known clinical-stage targets and prostate benchmarks are guaranteed present,
    so the ML and evaluation sections have labels to work with.
    """
    rng = np.random.default_rng(CFG.seed)
    must_have = sorted(set(
        [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
        + PROSTATE_HELDOUT_POSITIVES + PROSTATE_RESTRICTION_ONLY_MARKERS
        + NEPC_CONTEXT_BENCHMARKS + SURFACEOME_EXCLUSION_CONTROLS
        + NEGATIVE_CONTROLS + FILTER_CONTROLS
        + COMPARTMENT_MARKER_GENES + LINEAGE_MARKER_GENES
    ))
    n_filler = n_surface - len(must_have)
    surface_genes = must_have + [f"SURF{i:04d}" for i in range(n_filler)]
    bg_genes = [f"BG{i:04d}" for i in range(n_background)]
    genes = surface_genes + bg_genes

    n = len(genes)
    seq_len = rng.integers(150, 1400, n)
    # Surface proteins: 1-7 TM domains. Background: zero.
    n_tm = np.where(np.arange(n) < n_surface,
                    rng.choice([1, 1, 1, 2, 4, 7], n, p=[.45, .15, .10, .10, .10, .10]),
                    0)
    # ECD length scales with sequence length for single-pass receptors and is
    # near-zero for multi-pass transporters, mirroring real topology.
    ecd = np.where(n_tm == 1,
                   (seq_len * rng.uniform(0.25, 0.80, n)).astype(int),
                   np.where(n_tm > 1, rng.integers(5, 60, n), 0))
    df = pd.DataFrame({
        "gene": genes,
        "uniprot": [f"P{i:05d}" for i in range(n)],
        "seq_len": seq_len,
        "n_transmem": n_tm,
        "ecd_len_aa": ecd,
        "has_gpi": rng.random(n) < np.where(n_tm == 0, 0.0, 0.06),
    })
    # Secreted control: KLK3 (PSA) must fail the topology filter.
    df.loc[df.gene == "KLK3", ["n_transmem", "ecd_len_aa"]] = [0, 0]
    # Intracellular controls must fail too.
    df.loc[df.gene.isin(["ACTB", "GAPDH", "TUBB", "RPL13A", "MKI67", "TP53", "AR"]),
           ["n_transmem", "ecd_len_aa"]] = [0, 0]
    # Guarantee the clinical-stage positives have targetable ECDs, as they do
    # in reality - otherwise the synthetic labels would be incoherent.
    pos = [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
    tgt = df.gene.isin(pos + ["FOLH1", "STEAP1", "STEAP2", "TMEFF2", "SLC45A3", "TMPRSS2"])
    df.loc[tgt & (df.ecd_len_aa < 60), "ecd_len_aa"] = rng.integers(80, 500, (tgt & (df.ecd_len_aa < 60)).sum())
    df.loc[tgt & (df.n_transmem == 0), "n_transmem"] = 1

    # --- Per-filter control genes -------------------------------------------
    # Each is engineered to fail exactly one gatekeeping filter, which turns
    # the filter block into a testable unit rather than an article of faith.
    df.loc[df.gene == "CTRL_NO_ECD", ["n_transmem", "ecd_len_aa"]] = [6, 5]
    for g in ["CTRL_HIGH_VITAL", "CTRL_NO_AB", "CTRL_NOT_DE"]:
        df.loc[df.gene == g, ["n_transmem", "ecd_len_aa"]] = [1, 250]

    df["is_single_pass"] = df.n_transmem == 1
    return df


def load_protein_annotation():
    """Topology annotation for every gene, surface or not. Bounding happens
    later, as an explicit and auditable pipeline step."""
    print("[1.2] Protein topology annotation")
    if RUN_MODE == "LIVE":
        df = load_surfaceome_live()
    else:
        df = load_surfaceome_demo()
    df["is_surface"] = (df.n_transmem >= 1) | df.has_gpi.fillna(False)
    guarded_to_csv(df, DIRS["interim"] / "protein_annotation.csv")
    print(f"  genes annotated: {len(df):,} | membrane-localised: {int(df.is_surface.sum()):,}")
    print(f"  single-pass: {int(df.is_single_pass.sum()):,} | "
          f"with ECD >= {CFG.min_ecd_length_aa} aa: "
          f"{int((df.ecd_len_aa >= CFG.min_ecd_length_aa).sum()):,}")
    return df


def bound_to_surfaceome(annot):
    """SURFACEOME BOUNDING - the study's first scoping decision.

    Restricting to membrane-localised proteins before any statistical testing
    cuts the multiple-testing burden by roughly an order of magnitude, which
    directly buys power for the FDR-controlled comparison in Filter 1. Doing it
    afterwards would be a form of selective reporting.
    """
    n0 = len(annot)
    surf = annot[annot.is_surface].reset_index(drop=True)
    print(f"[1.2b] Surfaceome bounding: {n0:,} annotated genes -> "
          f"{len(surf):,} membrane-localised ({100*len(surf)/n0:.1f}%)")
    checks = [(g, "declared exclusion control") for g in SURFACEOME_EXCLUSION_CONTROLS]
    checks += [("ACTB", "cytoskeletal"), ("TP53", "nuclear"),
               ("AR", "nuclear receptor")]
    for ctrl, why in checks:
        if ctrl in set(annot.gene):
            inside = ctrl in set(surf.gene)
            flag = "FAIL" if inside else "ok"
            print(f"    bounding control {ctrl} ({why}): excluded={not inside} [{flag}]")
    guarded_to_csv(surf, DIRS["interim"] / "surfaceome.csv")
    return surf




# ----------------------------------------------------------------------------
# 1.2b GENE-IDENTIFIER HARMONISATION
# ----------------------------------------------------------------------------
# BUG FIX (critical, LIVE path): the surfaceome is defined by HUGO SYMBOL
# (from UniProt), but the Xena Toil expression matrix is indexed by ENSEMBL
# GENE ID. An earlier version filtered Ensembl-indexed rows against a set of
# symbols, which matches nothing: the expression matrix came back EMPTY and
# the failure only surfaced much later as "fewer than 10 positives" - after a
# multi-gigabyte download. The comment in that code even said "collapse
# duplicate symbols" while the line above it stripped Ensembl version
# suffixes, which is the tell that it was never run end to end.
#
# This layer detects the identifier type and maps explicitly, then reports the
# mapping rate so a silent failure becomes a loud one.
# ----------------------------------------------------------------------------
def detect_id_type(index_values, sample_n=200):
    """Classify an index as 'ensembl', 'symbol' or 'unknown'."""
    import re
    vals = [str(v) for v in list(index_values)[:sample_n]]
    if not vals:
        return "unknown"
    ens = sum(bool(re.match(r"^ENS[A-Z]*G\d{6,}", v)) for v in vals)
    if ens / len(vals) > 0.8:
        return "ensembl"
    sym = sum(bool(re.match(r"^[A-Za-z][A-Za-z0-9\-\.@_]{0,20}$", v)) for v in vals)
    if sym / len(vals) > 0.8:
        return "symbol"
    return "unknown"


def load_ensembl_to_symbol():
    """Xena's gencode probeMap, as a dict of bare Ensembl ID -> HUGO symbol."""
    f = download(DATA_URLS["toil_probemap"], input_path("toil_probemap"),
                 expect_min_bytes=input_min("toil_probemap"))
    pm = pd.read_csv(f, sep="\t", low_memory=False)
    idc = pm.columns[0]
    symc = next((c for c in pm.columns if c.lower() in ("gene", "symbol", "genesymbol")),
                pm.columns[1])
    pm[idc] = pm[idc].astype(str).str.split(".").str[0]
    mapping = dict(zip(pm[idc], pm[symc].astype(str)))
    print(f"    probeMap loaded: {len(mapping):,} Ensembl -> symbol entries")
    return mapping


def to_symbols(df, mapping=None, label="matrix", min_rate=0.50,
               scale=None):
    """Return `df` re-indexed by HUGO symbol, whatever it arrived as.

    Raises if the mapping rate is implausibly low, because continuing with a
    near-empty matrix wastes hours and produces a confident null result.
    """
    kind = detect_id_type(df.index)
    print(f"    {label}: identifier type detected = {kind}")
    if kind == "symbol":
        # BUG FIX (both external reviews, correct): this returned immediately,
        # so duplicate SYMBOLS in an already-symbol-indexed matrix bypassed
        # the aggregation logic entirely and were carried downstream as
        # duplicate rows - in a function presented as a general gene
        # harmonisation utility.
        n_dup_sym = int(df.index.duplicated().sum())
        if not n_dup_sym:
            return df
        if scale is None:
            raise ValueError(
                f"{label} is already indexed by gene symbol but contains "
                f"{n_dup_sym:,} duplicate symbol(s), e.g. "
                f"{sorted(set(df.index[df.index.duplicated()]))[:5]}. "
                f"Collapsing them needs the declared expression scale and an "
                f"explicit policy; carrying duplicate rows forward breaks "
                f"every per-gene join downstream. Pass scale=...")
        print(f"    {label}: already symbol-indexed, collapsing "
              f"{n_dup_sym:,} duplicate symbol(s) on the linear scale from "
              f"declared scale {scale}")
        out = collapse_duplicate_symbols(df, scale, how="sum_linear")
        out.attrs["scale"] = "log1p_tpm"
        return out
    if kind == "unknown":
        raise ValueError(
            f"Could not classify the identifiers in {label}: "
            f"examples {list(df.index[:5])}. Inspect the file and map to HUGO "
            f"symbols explicitly before continuing.")
    if mapping is None:
        mapping = load_ensembl_to_symbol()
    bare = pd.Index(df.index.astype(str).str.split(".").str[0])
    mapped = bare.map(lambda v: mapping.get(v))
    rate = float(pd.notna(mapped).mean())
    print(f"    {label}: mapped {rate:.1%} of Ensembl IDs to symbols")
    if rate < min_rate:
        raise ValueError(
            f"Only {rate:.1%} of identifiers in {label} mapped to symbols "
            f"(need >= {min_rate:.0%}). The probeMap and the expression matrix "
            f"are probably from different annotation releases. Resolve this "
            f"before continuing - a low mapping rate silently becomes a null "
            f"result.")
    out = df.copy()
    out.index = mapped
    out = out[pd.notna(out.index)]
    # Several Ensembl IDs map to one symbol. Taking the MAXIMUM per sample -
    # as an earlier version did - can stitch a single gene's profile together
    # from different transcripts in different samples, which is a chimera that
    # exists in no biological sample. Gene-level TPM is additive over its
    # transcripts, so summing is the defensible collapse. The input here is
    # linear-scale Xena output, so the sum is taken on recovered TPM and
    # returned on the same scale the caller supplied.
    n_dup = int(out.index.duplicated().sum())
    if n_dup:
        # CRITICAL BUG FIX (external review, confirmed): the inverse transform
        # was hard-coded to the XENA pseudocount, so duplicate symbols in a
        # LINEAR TPM or log2(TPM+1) matrix were collapsed with the wrong
        # arithmetic. Worse, load_transcriptomes_live() called this BEFORE
        # detecting the declared scale, so the assumption was never checked.
        # The caller must now declare the scale; collapsing on an undeclared
        # scale raises rather than guessing.
        if scale is None:
            raise ValueError(
                f"{label} has {n_dup:,} duplicate gene symbol(s) and no "
                f"declared expression scale. Collapsing them needs the "
                f"correct inverse transform - assuming Xena-log would "
                f"corrupt linear or log1p input. Detect the scale first and "
                f"pass scale=... to to_symbols().")
        print(f"    {label}: summing {n_dup:,} duplicate symbol rows on the "
              f"linear scale from declared scale {scale} (gene TPM is "
              f"additive over transcripts); returned as canonical log2(TPM+1)")
        out = collapse_duplicate_symbols(out, scale, how="sum_linear")
        out.attrs["scale"] = "log1p_tpm"
    else:
        out = out.groupby(level=0).first()
    print(f"    {label}: {df.shape[0]:,} rows -> {out.shape[0]:,} unique symbols")
    return out



# ----------------------------------------------------------------------------
# 1.2c EXPRESSION-SCALE DETECTION
# ----------------------------------------------------------------------------
# Earlier versions decided whether a matrix was log-scaled with bare inline
# heuristics (`if max < 25`, `if max > 50`). Those are silent-wrong-answer
# paths: a genuinely linear matrix of low-expressed genes would be
# exponentiated, and a log matrix of highly-expressed genes would not be, with
# nothing in the output to say which happened. A fold-change computed on a
# mis-scaled matrix is wrong by an exponent and still looks entirely
# plausible.
#
# This function makes the decision explicit, states its evidence, and refuses
# to guess when the evidence is ambiguous.
# ----------------------------------------------------------------------------
# ----------------------------------------------------------------------------
# ONE canonical scale converter, used by every aggregation path
# ----------------------------------------------------------------------------
# CRITICAL BUG FIX (external review, reproduced): the SU2C duplicate-collapse
# branch compared the detected scale against the literal "linear" while
# detect_expression_scale() returns "linear_tpm". The comparison was therefore
# NEVER true for a correctly identified linear matrix, so linear TPM was
# exponentiated as though it were log expression. Two measurements of 2 and
# 3 TPM should combine to log2(5+1) = 2.585; the broken path returned 3.459.
# The xena_log branch compounded it by subtracting the canonical pseudocount
# of 1.0 instead of Xena's 0.001, which drove 0.1 + 0.2 TPM to 0.000 - the
# signal destroyed outright.
#
# The root cause was string comparison against scale literals scattered across
# call sites. There is now exactly ONE converter. Every aggregation path goes
# through it, the scale names are validated against the enumeration, and an
# unrecognised scale raises instead of silently taking a branch.
SCALE_PSEUDOCOUNT = {
    "linear_tpm": None,          # already linear
    "xena_log": XENA_PSEUDOCOUNT,    # log2(TPM + 0.001)
    "log1p_tpm": CANONICAL_PSEUDOCOUNT,  # log2(TPM + 1)
}


def to_linear_tpm(df, scale):
    """Convert a matrix on any supported scale to LINEAR TPM."""
    if scale not in SCALE_PSEUDOCOUNT:
        raise ValueError(
            f"Unsupported expression scale {scale!r}. Supported: "
            f"{sorted(SCALE_PSEUDOCOUNT)}. Aggregating on an unknown scale "
            f"would silently corrupt every abundance in the matrix.")
    if scale == "linear_tpm":
        out = df.copy()
    else:
        out = np.power(2.0, df) - SCALE_PSEUDOCOUNT[scale]
    neg = int((out < -1e-6).sum().sum()) if hasattr(out, "sum") else 0
    if neg:
        print(f"    {neg:,} value(s) below zero after converting from "
              f"{scale} to linear TPM; clipped at 0. A large count means the "
              f"declared scale is wrong.")
    return out.clip(lower=0)


def collapse_duplicate_symbols(expr, scale, how="sum_linear"):
    """Combine duplicate gene symbols on the LINEAR scale, then re-log once.

    `how="sum_linear"` is correct only when duplicate rows are genuinely
    additive transcript quantities. Technical replicates require a different,
    source-justified policy, which is why the caller must declare it.
    """
    linear = to_linear_tpm(expr, scale)
    if how == "sum_linear":
        combined = linear.groupby(level=0).sum()
    elif how == "mean":
        combined = linear.groupby(level=0).mean()
    elif how == "first":
        combined = linear.groupby(level=0).first()
    else:
        raise ValueError(f"Unsupported duplicate policy {how!r}")
    # Convert ONCE to the canonical scale. Never transform again downstream.
    return np.log2(combined + CANONICAL_PSEUDOCOUNT)


def detect_expression_scale(df, label="matrix", assume=None):
    """Return 'xena_log', 'log1p_tpm', 'linear_tpm' or raise.

    BUG FIX (raised in external review): an earlier version returned only
    'log' or 'linear', and treated ANY negative values as proof of a log
    scale. A z-scored matrix is full of negatives, so it was silently
    classified as log expression - after which `2**x - 1` produced numbers
    with no TPM meaning at all, and a "fraction expressing above 1 TPM"
    statistic that was pure noise. cBioPortal ships z-scored matrices
    routinely, so this was a live hazard on the SU2C path.

    Z-scores are now detected on their own signature (per-gene mean near 0 and
    SD near 1) and REFUSED, because no pseudocount can recover abundance from
    them.
    """
    if assume in ("xena_log", "log1p_tpm", "linear_tpm"):
        # BUG FIX (external review, correct): a declared scale short-circuited
        # ALL validation, and to_canonical_log then clipped negatives to zero.
        # A z-scored matrix mislabelled "linear_tpm" was therefore converted
        # into plausible-looking abundances with its negative half flattened
        # to zero - invalid input laundered into valid-looking output. A
        # declaration says which transform to invert; it cannot make the
        # numbers mean something they do not.
        vals = df.to_numpy(dtype=float, copy=False)
        if not np.isfinite(vals).all():
            raise ValueError(
                f"{label}: declared scale {assume!r} but the matrix holds "
                f"{int((~np.isfinite(vals)).sum()):,} non-finite value(s). "
                f"Abundances must be numeric and finite; handle missingness "
                f"explicitly rather than pushing NaN/inf through a scale "
                f"conversion.")
        if assume == "xena_log":
            # BUG FIX (external review, correct): the declared path validated
            # finiteness and the non-negativity of linear/log1p, but never the
            # LOWER BOUND of Xena-log. Because TPM >= 0, log2(TPM + 0.001) >=
            # log2(0.001) = -9.966. Values below that are impossible, and
            # to_linear_tpm would clip the reconstructed negative TPM to zero,
            # turning invalid input into plausible zero expression.
            xena_min = float(np.log2(XENA_PSEUDOCOUNT))
            n_below = int((vals < xena_min - 1e-6).sum())
            if n_below:
                raise ValueError(
                    f"{label}: declared scale 'xena_log' but {n_below:,} "
                    f"value(s) fall below log2(0.001) = {xena_min:.3f} "
                    f"(min {float(vals.min()):.3f}). That is impossible on "
                    f"this scale - TPM cannot be negative - and inverting "
                    f"them would be clipped to zero, laundering invalid "
                    f"input into plausible absence of expression.")
        if assume in ("linear_tpm", "log1p_tpm"):
            n_neg = int((vals < 0).sum())
            if n_neg:
                raise ValueError(
                    f"{label}: declared scale {assume!r} requires "
                    f"non-negative values, but {n_neg:,} "
                    f"({n_neg / vals.size:.1%}) are negative (min "
                    f"{float(vals.min()):.3f}). Linear TPM and log2(TPM+1) "
                    f"are both bounded below by 0, so this is most likely "
                    f"z-scored or log-ratio data. Clipping the negatives "
                    f"would invent abundances that were never measured.")
        print(f"    {label}: scale DECLARED as {assume} (domain validated)")
        return assume
    # In LIVE mode the scale must come from source metadata, not inference.
    # Inference is a convenience for synthetic data, not a substitute for
    # reading the portal's documentation.
    if RUN_MODE == "LIVE" and not os.environ.get("ALLOW_SCALE_INFERENCE"):
        raise ValueError(
            f"{label}: LIVE mode requires an explicitly DECLARED expression "
            f"scale. Inference is available for synthetic data but is not an "
            f"acceptable substitute for the source documentation, because a "
            f"wrong call makes every fold-change wrong by an exponent while "
            f"still looking plausible.\n"
            f"Set the matching environment variable (TOIL_SCALE, SU2C_SCALE, "
            f"GTEX_SCALE) to one of xena_log | log1p_tpm | linear_tpm, or set "
            f"ALLOW_SCALE_INFERENCE=1 to accept inference and its risk.")
    if assume in ("log", "linear"):   # legacy values, mapped forward
        mapped = "log1p_tpm" if assume == "log" else "linear_tpm"
        print(f"    {label}: scale DECLARED as {assume} -> {mapped}")
        return mapped

    # --- z-score / standardised matrix: refuse, do not guess ---------------
    num = df.select_dtypes("number") if hasattr(df, "select_dtypes") else df
    row_mean = np.nanmean(num.values, axis=1)
    row_sd = np.nanstd(num.values, axis=1)
    frac_centered = float(np.mean(np.abs(row_mean) < 0.15))
    frac_unit_sd = float(np.mean(np.abs(row_sd - 1.0) < 0.25))
    if frac_centered > 0.7 and frac_unit_sd > 0.5:
        raise ValueError(
            f"{label} looks Z-SCORED ({frac_centered:.0%} of genes centred on "
            f"zero, {frac_unit_sd:.0%} with unit SD). Z-scores carry no "
            f"abundance information, so prevalence and TPM thresholds are "
            f"meaningless on them. Supply an abundance matrix (TPM, FPKM or "
            f"counts), or set the scale explicitly if you are certain this is "
            f"not standardised data.")
    v = df.values[np.isfinite(df.values)]
    if v.size == 0:
        raise ValueError(f"{label}: no finite values; cannot determine scale.")
    vmax, vmin = float(np.nanmax(v)), float(np.nanmin(v))
    frac_neg = float((v < 0).mean())
    p99 = float(np.nanpercentile(v, 99))

    # A large negative floor is the signature of Xena's 0.001 pseudocount:
    # log2(0 + 0.001) = -9.97, so unexpressed genes pile up near -10.
    if vmin < -5 and frac_neg > 0.01:
        verdict = "xena_log"
        why = (f"minimum {vmin:.2f} near log2(0.001) = -9.97, "
               f"{frac_neg:.1%} negative")
    elif frac_neg > 0.01:
        # BUG FIX (raised in external review): a valid log2(TPM+1) matrix
        # CANNOT hold negative values, so this branch previously certified an
        # impossible scale. Negatives without a deep Xena-style floor mean the
        # matrix is something else - centred, standardised or differently
        # pseudocounted - and guessing is how a wrong scale propagates.
        raise ValueError(
            f"{label}: {frac_neg:.1%} of values are negative, but the minimum "
            f"({vmin:.2f}) is not near log2(0.001) = -9.97, so this is "
            f"neither log2(TPM+1) (which cannot be negative) nor Xena's "
            f"log2(TPM+0.001). It may be centred, standardised or use another "
            f"pseudocount. Declare the scale explicitly once you have checked "
            f"the source documentation.")
    elif vmax > 1000:
        verdict = "linear_tpm"
        why = f"maximum {vmax:,.0f} is far beyond any log scale"
    elif vmax <= 30 and p99 <= 25:
        verdict = "log1p_tpm"
        why = f"maximum {vmax:.1f}, 99th percentile {p99:.1f}, no negatives"
    else:
        raise ValueError(
            f"{label}: expression scale is AMBIGUOUS (min {vmin:.2f}, "
            f"max {vmax:.2f}, 99th pct {p99:.2f}, {frac_neg:.1%} negative). "
            f"Refusing to guess, because guessing wrong makes every "
            f"fold-change wrong by an exponent while still looking plausible. "
            f"Pass assume='log' or assume='linear' once you have checked the "
            f"source documentation.")
    print(f"    {label}: scale inferred as {verdict} ({why})")
    return verdict


# ----------------------------------------------------------------------------
# 1.3 Transcriptomes: TCGA-PRAD tumour, TCGA normal prostate, GTEx tissues
# ----------------------------------------------------------------------------
# Schema returned by load_transcriptomes():
#   prad_tumour : DataFrame genes x samples, log2(TPM+0.001) as Xena ships it
#   prad_normal : DataFrame genes x samples (TCGA adjacent-normal prostate)
#   gtex_median : DataFrame genes x tissues, median log2(TPM+0.001)
# ----------------------------------------------------------------------------
def load_transcriptomes_live(gene_universe):
    tpm_f = download(DATA_URLS["toil_tpm"], input_path("toil_tpm"),
                     input_min("toil_tpm"))
    phe_f = download(DATA_URLS["toil_pheno"], input_path("toil_pheno"),
                     input_min("toil_pheno"))

    pheno = pd.read_csv(phe_f, sep="\t", index_col=0, low_memory=False)
    pheno.columns = [c.strip().lower().replace(" ", "_") for c in pheno.columns]
    site_col = next(c for c in pheno.columns if "primary_site" in c or "site" in c)
    type_col = next(c for c in pheno.columns if "sample_type" in c)
    study_col = next(c for c in pheno.columns if "study" in c)

    # Read the 60k x 19k matrix in chunks, keeping only surfaceome rows. The
    # full matrix is ~4 GB decompressed and will exhaust a free Colab instance.
    #
    # The matrix is indexed by ENSEMBL GENE ID, while `gene_universe` holds
    # HUGO SYMBOLS, so the Ensembl IDs are translated BEFORE filtering. The
    # inverse map is built once so the per-chunk filter stays cheap.
    # Retain the compartment and lineage markers alongside the surfaceome.
    # They are not candidates, but without them the epithelial-restriction
    # score and the AR/NEPC strata cannot be computed at all - which is
    # exactly what happened: the compartment analysis reported
    # "epithelial=1, immune=0" and skipped itself silently.
    keep = (set(gene_universe) | set(COMPARTMENT_MARKER_GENES)
            | set(LINEAGE_MARKER_GENES))
    mapping = load_ensembl_to_symbol()
    wanted_ens = {e for e, sym in mapping.items() if sym in keep}
    print(f"  {len(wanted_ens):,} Ensembl IDs correspond to the surfaceome")
    if len(wanted_ens) < 0.5 * len(keep):
        print(f"  WARNING: only {len(wanted_ens):,} Ensembl IDs matched "
              f"{len(keep):,} surfaceome symbols. Check annotation releases.")

    # ------------------------------------------------------------------
    # FILTERED-MATRIX CACHE
    # ------------------------------------------------------------------
    # SPEED FIX (both external reviews recommended this, and it is the
    # single largest LIVE win). The chunked read below is the right way to
    # get a 60k x 19k matrix through a small Colab instance - but it ran
    # from scratch on EVERY invocation, decompressing ~4 GB and parsing it
    # in Python to keep ~3k rows. The chunking bounds peak memory; it does
    # nothing for the repeated cost. Re-running the pipeline after any
    # change - which is exactly what debugging a first LIVE run involves -
    # paid that parse again each time.
    #
    # The filtered matrix is a pure function of (source bytes, requested
    # gene set, Ensembl->symbol mapping), so the cache key is the hash of
    # all three. Any change to any of them misses the cache and re-parses;
    # the cache can never silently serve a matrix built for a different
    # gene universe or annotation release.
    import hashlib as _hl
    # BUG FIX (both external reviews, correct): the key fell back to
    # f"size:{bytes}" when no digest was available. Two different matrices
    # of identical byte length then share a cache entry - and more to the
    # point, a file size is not an identity, so a publication-grade derived
    # cache must not be keyed on one. Combined with the stale-memo defect
    # above, this was a second route by which a changed source could reuse
    # an old processed matrix. Without a real digest the cache is SKIPPED:
    # a slower correct run beats a fast wrong one.
    _src_digest = (_VALIDATED_DIGESTS.get(str(tpm_f))
                   or (INPUT_FILE_REGISTRY.get(str(tpm_f), {}) or {}
                       ).get("sha256"))
    _cacheable = bool(_src_digest)
    if not _cacheable:
        print("  filtered-matrix cache DISABLED for this run: no verified "
              "SHA-256 for the source matrix, and a file size is not an "
              "identity. Correct results, slower re-runs.")
    _key = _hl.sha256("|".join([
        str(_src_digest or "NO-DIGEST"),
        _hl.sha256(",".join(sorted(wanted_ens)).encode()).hexdigest(),
        _hl.sha256(",".join(f"{k}={v}" for k, v in
                            sorted(mapping.items())).encode()).hexdigest(),
    ]).encode()).hexdigest()[:16]
    _cache_pq = DIRS["raw"] / f"toil_surfaceome_filtered_{_key}.parquet"
    # Parquet needs pyarrow, which Colab ships but a bare local interpreter
    # may not. .npz is in numpy itself and is SAFE to read - unlike pickle,
    # which would mean unpickling a file sitting in the same directory as
    # untrusted downloads. The cache must never be a code-execution path.
    _cache_npz = DIRS["raw"] / f"toil_surfaceome_filtered_{_key}.npz"
    _manifest_f = (DIRS["raw"]
                   / f"toil_surfaceome_filtered_{_key}.manifest.json")
    expr, first_index = None, None
    # BUG FIX (external review, independently reproduced, and the defect is
    # mine from last round). The loader used to try Parquet first and fall
    # back to NPZ, and only afterwards did _cache_is_sane() hash the file
    # NAMED IN THE MANIFEST. With both files present for the same cache key -
    # an ordinary state, since a run that falls back to NPZ leaves any
    # earlier Parquet in place - verification could attest one file while
    # the DataFrame came from the other. Shape and identifiers matched, so
    # every check passed, and the analysis then computed on unattested
    # values.
    #
    # Detecting the mismatch would have been enough to close it; preventing
    # it is better. The manifest is read FIRST and the loader opens ONLY the
    # format the manifest declares, so there is no longer a path by which
    # the verified file and the loaded file can differ. The format
    # preference is gone with it.
    _declared, _declared_path = None, None
    if _cacheable and _manifest_f.is_file():
        try:
            _declared = json.loads(_manifest_f.read_text())
        except Exception as _e:
            print(f"  cache manifest unreadable ({type(_e).__name__}); "
                  f"re-parsing from source")
            _declared = None
    if _declared:
        _name = _declared.get("cache_filename", "")
        # A manifest is data from disk, so treat its filename as untrusted:
        # a bare name only, inside the raw directory, never a path.
        if not _name or Path(_name).name != _name:
            print("  cache manifest names no usable cache file; re-parsing")
        else:
            _declared_path = DIRS["raw"] / _name
            if not _declared_path.is_file():
                print(f"  manifest attests {_name}, which is absent; "
                      f"re-parsing from source")
                _declared_path = None
    if _declared_path is not None:
        try:
            if _declared_path.suffix == ".parquet":
                expr = pd.read_parquet(_declared_path)
            elif _declared_path.suffix == ".npz":
                _z = np.load(_declared_path, allow_pickle=False)
                expr = pd.DataFrame(_z["values"],
                                    index=[str(g) for g in _z["index"]],
                                    columns=[str(c) for c in _z["columns"]])
            else:
                print(f"  manifest attests an unsupported cache format "
                      f"{_declared_path.suffix!r}; re-parsing from source")
                _declared_path = None
            if expr is not None:
                print(f"  cached filtered Toil matrix: {expr.shape[0]:,} "
                      f"genes x {expr.shape[1]:,} samples "
                      f"({_declared_path.name}, the file the manifest "
                      f"attests)")
        except Exception as _e:
            print(f"  attested cache unreadable ({type(_e).__name__}); "
                  f"re-parsing from source")
            expr = None
    # A cache that loads is not yet a cache that is USABLE. Both reviews
    # asked for the loaded matrix to be checked against what was requested,
    # and they are right: a cache hit that returns the wrong genes is worse
    # than a cache miss, because nothing downstream would notice.
    # STRENGTHENED (external review, correct). The first version accepted any
    # overlap at all with the requested universe, two sample columns and one
    # finite value. A matrix truncated to its first few hundred genes, or one
    # whose write was cut off, satisfies all three - and a cache hit that
    # returns a partial matrix is worse than a miss, because the run
    # continues on quietly reduced data and nothing downstream can tell.
    # "Readable" was being used as a proxy for "complete".
    #
    # A cache is now accepted only against a manifest written beside it at
    # build time, recording the source digest, the universe and mapping
    # hashes, the shape, and the identifier lists. The matrix must match that
    # manifest, and the manifest must match THIS run's request. Anything else
    # re-parses from the verified source.
    # (defined with the cache paths above, not here: the manifest-first load
    # needs it before _cache_is_sane is reached.)

    def _cache_is_sane(df, loaded_path):
        if df is None or df.empty:
            return False, "cached matrix is empty"
        if not _manifest_f.is_file():
            return False, ("no cache manifest beside it, so completeness "
                           "cannot be established")
        try:
            man = json.loads(_manifest_f.read_text())
        except Exception as _e:
            return False, f"cache manifest unreadable ({type(_e).__name__})"
        # The manifest has to describe THIS request, not some other one.
        if man.get("cache_key") != _key:
            return False, (f"manifest is for cache key "
                           f"{str(man.get('cache_key'))[:12]}, not {_key[:12]}")
        if man.get("source_sha256") != str(_src_digest):
            return False, "manifest records a different source checksum"
        # BUG FIX (both external reviews, independently reproduced). The
        # manifest recorded the SOURCE digest, the shape and the identifier
        # lists - the cache's IDENTITY - but never a digest of the cached
        # file itself, so a matrix whose expression VALUES had changed while
        # its genes, samples and dimensions stayed identical passed every
        # check. Identity was being verified and contents were not, which is
        # the wrong half for a file the analysis then computes on: every
        # downstream number would come from the altered values and nothing
        # would report it.
        _want_cache = man.get("cache_sha256")
        if not _want_cache:
            return False, ("manifest records no digest of the cache file "
                           "itself, so its contents cannot be verified")
        # The digest is checked against the file the DataFrame was ACTUALLY
        # loaded from, not against whichever file the manifest names. Those
        # were allowed to differ; that was the defect. Both conditions are
        # asserted, so the verified file and the loaded file are provably
        # one file.
        if loaded_path is None:
            return False, ("no attested cache path was loaded, so nothing "
                           "can be verified")
        _declared_name = man.get("cache_filename", "")
        if Path(loaded_path).name != _declared_name:
            return False, (f"loaded {Path(loaded_path).name} but the "
                           f"manifest attests {_declared_name!r} - the "
                           f"verified file is not the file in use")
        if not Path(loaded_path).is_file():
            return False, "the loaded cache file is no longer present"
        if sha256_file(loaded_path) != _want_cache:
            return False, ("the loaded cache file's contents do not match "
                           "its manifest digest - values may have changed")
        # ...and the matrix on disk has to be the one the manifest describes.
        if list(df.shape) != list(man.get("shape", [])):
            return False, (f"shape {list(df.shape)} does not match the "
                           f"manifest's {man.get('shape')}")
        if sorted(str(i) for i in df.index) != sorted(man.get("genes", [])):
            return False, "gene identifiers do not match the manifest"
        if sorted(str(c) for c in df.columns) != sorted(
                man.get("samples", [])):
            return False, "sample identifiers do not match the manifest"
        # Completeness against what was actually asked for.
        _idx = {str(i).split(".")[0] for i in df.index}
        _overlap = len(_idx & set(wanted_ens))
        _expected = man.get("n_requested_found")
        if _expected is not None and _overlap != _expected:
            return False, (f"{_overlap:,} requested identifiers present, but "
                           f"the manifest recorded {_expected:,}")
        if _overlap == 0:
            return False, ("shares no gene identifiers with the requested "
                           "universe")
        _arr = df.to_numpy(dtype="float64")
        _finite = float(np.isfinite(_arr).mean()) if _arr.size else 0.0
        if _finite < 0.5:
            return False, (f"only {_finite:.1%} of values are finite; a "
                           f"usable expression matrix is almost entirely "
                           f"finite")
        return True, (f"{_overlap:,} requested identifiers, shape "
                      f"{df.shape[0]}x{df.shape[1]}, {_finite:.1%} finite, "
                      f"manifest matched")

    # The separate NPZ fallback that stood here is gone: there is now ONE
    # load path, opening the single format the manifest declares. A fallback
    # that could pick a different file from the one being verified was the
    # defect itself.
    if expr is not None:
        _sane, _sane_why = _cache_is_sane(expr, _declared_path)
        if not _sane:
            print(f"  cached filtered matrix REJECTED ({_sane_why}); "
                  f"re-parsing from source")
            expr = None
        else:
            print(f"    cache contents validated: {_sane_why}")
    if expr is None:
        # Counted so that "the cache was used" is OBSERVABLE. This is the
        # expensive operation the cache exists to avoid - not the download()
        # call, which happens on every run and is a cheap cache hit on the
        # file. My fixture counted download() calls and so could not tell a
        # working cache from a silently broken one; the npz cache was
        # unreadable for five revisions behind exactly that blind spot.
        TOIL_PARSE_COUNT["n"] += 1
        chunks = []
        print("  streaming Toil TPM matrix (surfaceome rows only) ...")
        for ch in pd.read_csv(tpm_f, sep="\t", index_col=0, chunksize=2000,
                              low_memory=False):
            if first_index is None:
                first_index = list(ch.index[:20])
            bare = ch.index.astype(str).str.split(".").str[0]
            ch.index = bare
            _hit = ch[bare.isin(wanted_ens)]
            if len(_hit):
                # float32 halves the memory of the retained matrix. TPM
                # values carry nowhere near 15 significant digits, and every
                # downstream statistic is computed in float64 after pandas
                # promotes on arithmetic, so this is a storage width, not a
                # change in the arithmetic.
                chunks.append(_hit.astype("float32"))
        expr = pd.concat(chunks) if chunks else pd.DataFrame()
        if not expr.empty:
            # The chunked reader has just walked the whole gzip stream and
            # parsed every row, so this is the strongest evidence the file
            # will ever get - stronger than the structural walk it replaces.
            confirm_input_validated(tpm_f, "full matrix parsed")
        if not expr.empty and _cacheable:
            # NOT register_artifact(): packaging is registry-only, so
            # registering this would ship a multi-hundred-MB intermediate
            # cache inside the release archive. It is a performance
            # artifact, not a scientific one, and the run manifest already
            # records the SOURCE checksum from which it is derived.
            # BUG FIX (self-review, and this one had never worked at all).
            # The npz branch wrote the gene and sample names via
            #     np.asarray(expr.index.astype(str))
            # which is an OBJECT array, and the reader opens the file with
            # allow_pickle=False - so every npz cache this pipeline wrote was
            # unreadable by the pipeline that wrote it. The failure was
            # invisible because the reader catches the error, prints one line
            # and re-parses from source: a broken cache and a cold start look
            # identical from the outside. pyarrow is absent on a bare Python,
            # so on those machines the headline speed benefit of the
            # filtered-matrix cache was simply never delivered.
            #
            # My own fixture missed it because it asserted that a cache FILE
            # EXISTS and that a re-run returns identical values - both true
            # when the cache is silently discarded and the matrix re-parsed.
            # The test now asserts the cache was USED (see the LIVE suite).
            #
            # Written to a temporary name and committed with os.replace, so
            # an interrupted write cannot leave a half-file that later looks
            # like a cache; the manifest carries the digest and is written
            # only after the data file is in place.
            _written = None
            try:
                _tmp_pq = _cache_pq.with_suffix(".parquet.tmp")
                expr.to_parquet(_tmp_pq, compression="snappy")
                os.replace(_tmp_pq, _cache_pq)
                _written = _cache_pq
            except Exception:
                try:
                    # np.savez_compressed APPENDS ".npz" when the name does
                    # not already end in it, so a temp called "....npz.tmp"
                    # is written as "....npz.tmp.npz" and the os.replace
                    # below then fails on a path that does not exist. The
                    # temp name therefore ends in .npz.
                    _tmp_npz = _cache_npz.with_suffix(".tmp.npz")
                    np.savez_compressed(
                        _tmp_npz,
                        values=expr.to_numpy(dtype="float32"),
                        # dtype="U" gives a fixed-width unicode array, which
                        # np.load reads with allow_pickle=False. Reading a
                        # cache must never be a code-execution path, so
                        # allow_pickle stays off and the WRITER adapts.
                        index=np.asarray([str(i) for i in expr.index],
                                         dtype="U"),
                        columns=np.asarray([str(c) for c in expr.columns],
                                           dtype="U"))
                    os.replace(_tmp_npz, _cache_npz)
                    _written = _cache_npz
                except Exception as _e:
                    print(f"  (filtered-matrix cache not written: "
                          f"{type(_e).__name__}: {str(_e)[:80]}) - correct "
                          f"results, slower re-runs")
            if _written is not None:
                # Only the attested file should exist for this key. The
                # reader now opens solely what the manifest declares, so a
                # leftover file in the other format is already inert - but
                # leaving it on disk preserves the ambiguity that caused the
                # defect, and wastes a few hundred MB on a Colab volume.
                for _other in (_cache_pq, _cache_npz):
                    if _other != _written and _other.is_file():
                        try:
                            _other.unlink()
                            print(f"    removed superseded cache in the "
                                  f"other format: {_other.name}")
                        except Exception:
                            pass
                try:
                    _manifest_f.write_text(json.dumps({
                        "cache_key": _key,
                        "source_sha256": str(_src_digest),
                        "source_file": Path(tpm_f).name,
                        "cache_filename": _written.name,
                        "cache_sha256": sha256_file(_written),
                        "shape": list(expr.shape),
                        "genes": [str(i) for i in expr.index],
                        "samples": [str(c) for c in expr.columns],
                        "n_requested_found": len(
                            {str(i).split(".")[0] for i in expr.index}
                            & set(wanted_ens)),
                        "dtype": str(expr.to_numpy().dtype),
                        "written_utc": pd.Timestamp.utcnow().isoformat(),
                        "format": _written.suffix.lstrip("."),
                    }, indent=1))
                except Exception as _e:
                    # No manifest means the cache will not be trusted on the
                    # next run. That is the safe direction: it re-parses.
                    print(f"  (cache manifest not written: "
                          f"{type(_e).__name__}; the cache will be ignored "
                          f"next run and the matrix re-parsed)")
                print(f"  cached filtered matrix for re-runs: "
                      f"{_written.name} "
                      f"({_written.stat().st_size/1e6:.1f} MB) - a re-run "
                      f"skips the full-matrix parse entirely")
    if expr.empty:
        raise ValueError(
            f"No surfaceome rows matched the expression matrix. First index "
            f"values were {first_index}. If these are not Ensembl gene IDs, "
            f"the download path in DATA_URLS points at a different file than "
            f"this code expects.")
    # BUG FIX (external review, confirmed): symbol collapsing ran BEFORE the
    # expression scale was detected, so the collapse silently assumed Xena-log.
    # The scale is now determined FIRST and passed in, and a mismatch raises.
    toil_scale = detect_expression_scale(expr, "Toil TPM (pre-collapse)",
                                         assume=os.environ.get("TOIL_SCALE"))
    expr = to_symbols(expr, mapping, label="Toil TPM", scale=toil_scale)
    # to_symbols returns canonical log2(TPM+1) whenever it collapsed rows.
    toil_scale = expr.attrs.get("scale", toil_scale)

    shared = [s for s in expr.columns if s in pheno.index]
    expr, pheno = expr[shared], pheno.loc[shared]

    is_prad = pheno[study_col].astype(str).str.contains("TCGA", na=False) & \
              pheno[site_col].astype(str).str.contains("Prostate", case=False, na=False)
    stype = pheno[type_col].astype(str)
    is_gtex = pheno[study_col].astype(str).str.contains("GTEX", na=False)

    # BUG FIX (raised in external review): normals were selected as
    # `~is_tumour`, i.e. "anything not labelled tumour". That is not the same
    # as "normal": metastatic, recurrent, cell-line and unlabelled sample
    # types would all have been swept into the normal reference, which is the
    # denominator of every fold-change in the study.
    is_tumour = stype.str.contains("Primary Tumor", case=False, na=False)
    is_normal = stype.str.contains(
        "Solid Tissue Normal|Normal Tissue", case=False, na=False)
    excluded = is_prad & ~is_tumour & ~is_normal
    if int(excluded.sum()):
        kinds = sorted(set(stype[excluded]))
        print(f"  excluded {int(excluded.sum())} prostate samples whose type "
              f"is neither primary tumour nor solid-tissue normal: {kinds}")
    print(f"  sample types kept -> tumour: '{sorted(set(stype[is_prad & is_tumour]))}' "
          f"| normal: '{sorted(set(stype[is_prad & is_normal]))}'")

    prad_tumour = expr.loc[:, is_prad & is_tumour]
    prad_normal = expr.loc[:, is_prad & is_normal]
    if prad_normal.shape[1] == 0:
        raise ValueError(
            "No TCGA prostate samples matched an explicit normal sample type. "
            "Inspect the phenotype file's sample_type values and update the "
            "match rather than falling back to 'not tumour', which would "
            "contaminate the normal reference.")
    gtex = expr.loc[:, is_gtex]
    gtex_site = pheno.loc[is_gtex, site_col]
    # GTEx detailed tissue label lives in a separate column in some releases
    detail = next((c for c in pheno.columns if "detailed" in c), None)
    labels = pheno.loc[is_gtex, detail] if detail else gtex_site
    gtex_median = gtex.T.groupby(labels.values).median().T

    # Put every matrix on the canonical log2(TPM+1) scale HERE, once, so no
    # downstream function has to know which pseudocount the portal used.
    # The scale was determined BEFORE symbol collapsing and may have been
    # changed to canonical by that collapse. Re-detecting here would discard
    # what was actually applied and could disagree with it.
    if toil_scale in SCALE_PSEUDOCOUNT:
        print(f"    scale carried forward from pre-collapse detection: "
              f"{toil_scale}")
    else:
        toil_scale = detect_expression_scale(
            prad_tumour, "Toil PRAD tumour",
            assume=os.environ.get("TOIL_SCALE"))
    prad_tumour = to_canonical_log(prad_tumour, toil_scale, "PRAD tumour")
    prad_normal = to_canonical_log(prad_normal, toil_scale, "PRAD normal")
    gtex_median = to_canonical_log(gtex_median, toil_scale, "GTEx medians")
    return prad_tumour, prad_normal, gtex_median


def load_transcriptomes_demo(surf):
    """Synthetic transcriptomes with planted, biologically-shaped signal.

    The generator deliberately builds in the structure the pipeline is supposed
    to detect, so that a pipeline failure is distinguishable from an absent
    signal. It is a positive control for the analysis code, NOT a simulation
    study and NOT evidence about real biology.
    """
    rng = np.random.default_rng(CFG.seed + 1)
    genes = surf.gene.tolist()
    g = len(genes)
    n_t, n_n = 480, 52       # TCGA-PRAD: ~500 tumours, ~52 adjacent normals
    # Deduplicated: salivary gland is now BOTH a declared vital organ and a
    # member of the extra tissue list, and a duplicate column label makes
    # DataFrame.columns.get_loc() return an array rather than a position,
    # which breaks positional assignment.
    tissues = list(dict.fromkeys(
        [TISSUE_ALIASES.get(t, [t])[0] if t == "Salivary Gland" else t
         for t in CFG.vital_organs] + [
        "Prostate", "Testis", "Thyroid", "Spleen", "Pancreas", "Stomach",
        "Colon - Transverse", "Skin - Sun Exposed", "Muscle - Skeletal",
        "Adipose - Subcutaneous", "Breast - Mammary Tissue", "Esophagus - Mucosa",
        # Deliberately the GTEx v8 label, NOT the declared config name, so
        # the alias harmonisation is exercised in DEMO. A mapping that only
        # runs on LIVE data is untested code.
        "Pituitary", "Adrenal Gland", "Bladder", "Minor Salivary Gland",
    ]))
    assert len(tissues) == len(set(tissues)), "duplicate tissue labels"

    # ------------------------------------------------------------------
    # GENERATE ON A LINEAR TPM SCALE, THEN CONVERT
    # ------------------------------------------------------------------
    # BUG FIX (critical): an earlier generator drew baselines from
    # N(2.0, 2.2) clipped to [-3, 11] and treated them as log2(TPM+1). About
    # 18% of those values are negative, which is impossible on that scale, and
    # inverting them produced negative TPM that auto-passed the safety filter.
    #
    # Expression is now drawn on the LINEAR TPM scale from a log-normal, which
    # is both the right support (non-negative) and a realistic shape for
    # transcript abundance. Conversion to the canonical log scale happens once,
    # at the end, through the same helper the LIVE path uses.
    base_tpm = rng.lognormal(mean=1.0, sigma=1.8, size=g).clip(0, 5000)
    base = np.log2(base_tpm + CANONICAL_PSEUDOCOUNT)     # canonical log2(TPM+1)
    tumour_lfc = np.zeros(g)

    pos = [x for x in CLINICAL_STAGE_TARGETS if not x.endswith("_HELD")]
    idx = {gn: i for i, gn in enumerate(genes)}

    # A realistic minority of the surfaceome is up in tumour.
    n_up = int(0.08 * g)
    up_idx = rng.choice(g, n_up, replace=False)
    tumour_lfc[up_idx] = rng.gamma(2.0, 0.8, n_up)

    # Known targets and prostate benchmarks are ENRICHED FOR, BUT NOT DEFINED
    # BY, upregulation. The enrichment is partial and the effect sizes overlap
    # the background, which is what makes the demo a fair rehearsal: a
    # generator that makes positives perfectly separable produces AUC = 1.00
    # and tells you nothing about whether the model would work on real data.
    for gn in pos + PROSTATE_HELDOUT_POSITIVES + NEPC_CONTEXT_BENCHMARKS:
        if gn in idx and rng.random() < 0.72:      # 28% of positives are NOT up
            tumour_lfc[idx[gn]] = max(tumour_lfc[idx[gn]], rng.uniform(0.5, 2.2))
            base[idx[gn]] = max(base[idx[gn]], rng.uniform(2.5, 6.5))

    gene_sd = rng.uniform(0.45, 1.5, g)
    # Noise is added on the log scale, then the result is clipped to the
    # VALID SUPPORT of log2(TPM+1), which is [0, inf). Clipping at zero is
    # correct here rather than a fudge: zero means "not expressed".
    tum = np.clip(base[:, None] + tumour_lfc[:, None]
                  + rng.normal(0, 1, (g, n_t)) * gene_sd[:, None], 0, None)
    nor = np.clip(base[:, None]
                  + rng.normal(0, 1, (g, n_n)) * gene_sd[:, None], 0, None)

    prad_tumour = pd.DataFrame(tum, index=genes,
                               columns=[f"TCGA-T{i:04d}" for i in range(n_t)])
    prad_normal = pd.DataFrame(nor, index=genes,
                               columns=[f"TCGA-N{i:04d}" for i in range(n_n)])

    # GTEx: vital organs are held LOW for good targets (that is what makes them
    # good), and random elsewhere.
    gt = np.clip(base[:, None] + rng.normal(0, 1.0, (g, len(tissues))), 0, None)
    gtex_median = pd.DataFrame(gt, index=genes, columns=tissues)
    vital = [t for t in CFG.vital_organs if t in gtex_median.columns]
    # A substantial slice of the BACKGROUND is also vital-organ-sparing. Good
    # safety is common; it is good safety PLUS tumour expression PLUS topology
    # that is rare, and the model has to find that conjunction rather than
    # reading one giveaway feature.
    # CRITICAL BUG FIX (found auditing this generator): vital-organ "safety"
    # was planted as NEGATIVE values on the log2(TPM+1) scale - N(-0.8, 1.0)
    # for background and N(-1.0, 0.9) for positives, which are 79% and 87%
    # negative. That scale is bounded below by zero, so the clip at the end
    # of this function collapsed the entire intended gradient of "low but
    # measurable" normal expression onto exactly 0.000.
    #
    # This is the same impossible-scale error the comment above documents
    # fixing for `base`; it was never applied here, where it is far worse.
    # Its consequence is the comparator itself: FOLH1 drew from N(-1.0, 0.9)
    # and ended at exactly 0.0 TPM, which is why F2 reported NOT ASSESSABLE
    # for all 2,792 genes, all four modality screens were unassessable, every
    # eligibility was provisional, and the detection-floor sensitivity
    # analysis swung nine-fold. Three review rounds each identified a
    # downstream symptom; the cause was here.
    #
    # Safety is now planted on the LINEAR TPM scale as genuinely low but
    # MEASURABLE abundance, converted once at the end like everything else.
    def _plant_low_tpm(lo, hi, shape):
        """Low but measurable vital-organ TPM, as canonical log2(TPM+1)."""
        return np.log2(rng.uniform(lo, hi, shape) + CANONICAL_PSEUDOCOUNT)

    bg_safe = rng.choice(g, int(0.22 * g), replace=False)
    _vcols = [gtex_median.columns.get_loc(v) for v in vital]
    gtex_median.iloc[bg_safe, _vcols] = _plant_low_tpm(
        0.02, 0.45, (len(bg_safe), len(vital)))
    for gn in pos + PROSTATE_HELDOUT_POSITIVES:
        if gn in idx and rng.random() < 0.78:      # 22% of positives are NOT safe
            gtex_median.loc[gn, vital] = _plant_low_tpm(0.02, 0.40, len(vital))

    # BIOLOGICAL FIX: PSMA is NOT vital-organ-sparing. The Human Protein Atlas
    # reports roughly 3 nTPM in kidney cortex and 10 nTPM in minor salivary
    # gland, and those are exactly the organs behind PSMA radioligand
    # therapy's dose-limiting renal exposure and xerostomia. PSMA serves as
    # the comparator BECAUSE that exposure is clinically tolerated, not
    # because the antigen is absent. Rolling it into the "safe" group inverts
    # the premise of Filter 2 and guarantees a degenerate anchor.
    if CFG.benchmark_gene in idx:
        bg = CFG.benchmark_gene
        # CALIBRATION FIX (found reviewing this round's own fix): the first
        # version took real-world nTPM figures (~3 kidney cortex, ~10 minor
        # salivary gland) and dropped them into a synthetic distribution whose
        # scale is not comparable. This generator's vital-organ background has
        # a median of ~5.5 TPM, so a 6.6 TPM anchor sat at its 54th percentile
        # and Filter 2 admitted 53.9% of the surfaceome - no longer a
        # selectivity filter at all. One degenerate regime (everything at the
        # floor) had been traded for another (the bar above the median).
        #
        # What matters is the comparator's POSITION in the distribution it is
        # compared against, not its absolute value: PSMA's normal-tissue
        # exposure is clinically TOLERATED, not permissive. The anchor is
        # therefore placed at a target percentile of this generator's own
        # vital-organ background, which keeps the filter discriminating while
        # remaining well above the detection floor so the censoring logic is
        # still exercised.
        _vital_bg = np.power(2.0, gtex_median[vital].values) - CANONICAL_PSEUDOCOUNT
        _anchor = float(np.nanpercentile(np.clip(_vital_bg, 0, None),
                                         CFG.comparator_target_percentile))
        _anchor = max(_anchor, 4 * float(CFG.vital_tpm_detection_floor))
        gtex_median.loc[bg, vital] = _plant_low_tpm(
            0.25 * _anchor, 0.60 * _anchor, len(vital))
        # Kidney cortex and salivary gland carry PSMA's real dose-limiting
        # exposure, so they hold the maximum that defines the anchor.
        for organ, frac_lo, frac_hi in [("Kidney - Cortex", 0.45, 0.70),
                                        ("Salivary Gland", 0.85, 1.15),
                                        ("Minor Salivary Gland", 0.85, 1.15)]:
            if organ in gtex_median.columns:
                gtex_median.loc[bg, organ] = float(np.log2(
                    rng.uniform(frac_lo * _anchor, frac_hi * _anchor)
                    + CANONICAL_PSEUDOCOUNT))
        print(f"    comparator {bg} planted with a MEASURABLE normal-tissue "
              f"profile peaking in kidney cortex and salivary gland, as the "
              f"real antigen does; anchored at the "
              f"{CFG.comparator_target_percentile}th percentile of this "
              f"generator's vital-organ background (~{_anchor:.2f} TPM) so "
              f"Filter 2 discriminates instead of admitting the majority")
    # Prostate-restricted genes are high in GTEx prostate specifically.
    for gn in (PROSTATE_HELDOUT_POSITIVES + PROSTATE_RESTRICTION_ONLY_MARKERS
               + ["KLK3"]):
        if gn in idx:
            gtex_median.loc[gn, "Prostate"] = rng.uniform(6, 10)
    # AR is carried through the DEMO cohorts (as a NON-surface lineage marker,
    # excluded from the surfaceome at bounding) so that the AR-low persistence
    # axis is actually EXERCISED in DEMO rather than silently skipped. A new
    # analysis that never runs in the only executable mode is untested code.
    if "AR" in idx:
        gtex_median.loc["AR", :] = rng.uniform(1.5, 5.0, gtex_median.shape[1])
    # Housekeeping controls: high and flat everywhere.
    for gn in NEGATIVE_CONTROLS:
        if gn in idx:
            gtex_median.loc[gn, :] = rng.uniform(6.5, 10.0, gtex_median.shape[1])
            prad_tumour.loc[gn, :] = rng.normal(8.5, .5, n_t)
            prad_normal.loc[gn, :] = rng.normal(8.4, .5, n_n)

    # --- Per-filter controls ------------------------------------------------
    # CTRL_NOT_DE: membrane protein, targetable ECD, antibodies, safe - but
    # identical in tumour and normal. Must fail F1 only.
    if "CTRL_NOT_DE" in idx:
        prad_tumour.loc["CTRL_NOT_DE", :] = rng.normal(6.0, 0.6, n_t)
        prad_normal.loc["CTRL_NOT_DE", :] = rng.normal(6.0, 0.6, n_n)
        gtex_median.loc["CTRL_NOT_DE", :] = _plant_low_tpm(
            0.05, 0.60, gtex_median.shape[1])
    # CTRL_HIGH_VITAL: strongly up in tumour, but saturating in brain/heart/
    # liver. Must fail F2 only. This is the pattern that kills real programmes.
    if "CTRL_HIGH_VITAL" in idx:
        prad_tumour.loc["CTRL_HIGH_VITAL", :] = rng.normal(8.0, 0.6, n_t)
        prad_normal.loc["CTRL_HIGH_VITAL", :] = rng.normal(4.0, 0.6, n_n)
        gtex_median.loc["CTRL_HIGH_VITAL", :] = rng.normal(2.0, 0.5, gtex_median.shape[1])
        gtex_median.loc["CTRL_HIGH_VITAL", vital] = rng.normal(9.0, 0.4, len(vital))
    # CTRL_NO_ECD and CTRL_NO_AB are good on expression and safety; they fail
    # on topology and reagents respectively, which are annotation-driven.
    for gn in ["CTRL_NO_ECD", "CTRL_NO_AB"]:
        if gn in idx:
            prad_tumour.loc[gn, :] = rng.normal(7.5, 0.6, n_t)
            prad_normal.loc[gn, :] = rng.normal(4.0, 0.6, n_n)
            gtex_median.loc[gn, :] = _plant_low_tpm(
                0.30, 1.50, gtex_median.shape[1])
            # Same impossible-scale fix: plant low MEASURABLE TPM, not
            # negative log values that clip to exactly zero.
            gtex_median.loc[gn, vital] = _plant_low_tpm(0.02, 0.30, len(vital))

    # ------------------------------------------------------------------
    # ENFORCE THE VALID SUPPORT, once, after all planted signal
    # ------------------------------------------------------------------
    # Several signal blocks above draw from distributions centred at or below
    # zero in order to make a gene look unexpressed in a tissue. On a
    # log2(TPM+1) scale the floor is exactly 0, which means 0 TPM - precisely
    # the intended meaning. Clipping here is therefore the correct encoding of
    # "not expressed", not a cover-up for out-of-range draws.
    prad_tumour = prad_tumour.clip(lower=0.0)
    prad_normal = prad_normal.clip(lower=0.0)
    gtex_median = gtex_median.clip(lower=0.0)

    for name, m in [("PRAD tumour", prad_tumour), ("PRAD normal", prad_normal),
                    ("GTEx medians", gtex_median)]:
        assert (m.values >= 0).all(), f"{name} still holds negative log values"
    print(f"    synthetic matrices verified on log2(TPM+1) support "
          f"[0, {max(prad_tumour.values.max(), gtex_median.values.max()):.1f}]")
    return prad_tumour, prad_normal, gtex_median


def load_transcriptomes(surf):
    print("[1.3] Transcriptomes (TCGA-PRAD tumour/normal, GTEx tissues)")
    if RUN_MODE == "LIVE":
        out = load_transcriptomes_live(surf.gene.tolist())
    else:
        out = load_transcriptomes_demo(surf)
    t, n, gx = out
    print(f"  PRAD tumours: {t.shape[1]:,} | PRAD normals: {n.shape[1]:,} | "
          f"GTEx tissues: {gx.shape[1]:,} | genes: {t.shape[0]:,}")
    if n.shape[1] < 10:
        print("  WARNING: fewer than 10 adjacent normals. Differential "
              "expression will be underpowered; consider GTEx prostate as the "
              "normal reference and report both.")
    return t, n, gx


# ----------------------------------------------------------------------------
# 1.4 SU2C/PCF metastatic CRPC cohort
# ----------------------------------------------------------------------------
# Schema: expr (genes x samples, log2), pheno (sample, is_nepc, ar_signalling)
# ----------------------------------------------------------------------------
def extract_archive_once(tar_f, base_dir, label):
    """Extract an archive into a directory KEYED ON THE ARCHIVE'S DIGEST.

    BUG FIX (both external reviews, correct, and the same failure shape as
    the stale-digest defect): the pattern was

        ex_dir = DIRS["raw"] / "<name>_extracted"
        if not ex_dir.exists():
            ...extract...

    so once the directory existed it was reused FOREVER. If the source
    archive changed - a portal re-release, a corrected cohort, a re-download
    after the cache was rejected - the run continued reading the OLD
    extracted files while the input manifest recorded the checksum of the
    NEW archive. The archive says one thing, the analysis used another, and
    nothing anywhere reports a disagreement.

    The directory name now contains the archive's SHA-256 prefix, so a
    different archive is a different directory and re-extraction is
    automatic. A completion sentinel is written only after extraction
    finishes, so an interrupted extraction is not mistaken for a complete
    one - on Colab, a disconnect mid-extraction is an ordinary event.
    """
    digest = (_VALIDATED_DIGESTS.get(str(tar_f))
              or (INPUT_FILE_REGISTRY.get(str(tar_f), {}) or {}).get("sha256"))
    if not digest:
        import hashlib as _hl
        h = _hl.sha256()
        with open(tar_f, "rb") as fh:
            for blk in iter(lambda: fh.read(1 << 22), b""):
                h.update(blk)
        digest = h.hexdigest()
    ex_dir = Path(base_dir).with_name(f"{Path(base_dir).name}_{digest[:12]}")
    done = ex_dir / ".extraction_complete"
    if done.is_file():
        return ex_dir
    if ex_dir.exists():
        print(f"  {label}: previous extraction was incomplete; redoing it")
        shutil.rmtree(ex_dir, ignore_errors=True)
    ex_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_f) as tf:
        # Extract only safe members; refuse absolute and parent paths.
        safe = [m for m in tf.getmembers()
                if not m.name.startswith(("/", "..")) and ".." not in m.name]
        tf.extractall(ex_dir, members=safe)
    done.write_text(f"{digest}  {Path(tar_f).name}\n")
    # A complete extraction IS a full read of the compressed stream, so the
    # archive has now earned its checksum attestation.
    confirm_input_validated(tar_f, "archive extracted in full")
    print(f"  {label}: extracted to {ex_dir.name} "
          f"(keyed on the archive's SHA-256)")
    return ex_dir


def load_mcrpc_live(gene_universe):
    tar_f = download(DATA_URLS["su2c"], input_path("su2c"), input_min("su2c"))
    ex_dir = extract_archive_once(tar_f, DIRS["raw"] / "su2c_extracted",
                                  "SU2C mCRPC archive")
    cand = list(ex_dir.rglob("data_mrna*.txt")) + list(ex_dir.rglob("data_RNA*.txt"))
    if not cand:
        raise FileNotFoundError(
            f"No expression matrix found in {ex_dir}. Inspect the archive "
            f"contents and update the glob pattern."
        )
    expr_f = sorted(cand, key=lambda p: p.stat().st_size)[-1]
    expr = pd.read_csv(expr_f, sep="\t", comment="#", low_memory=False)
    sym = "Hugo_Symbol" if "Hugo_Symbol" in expr.columns else expr.columns[0]
    expr = expr.set_index(sym)
    expr = expr.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in expr.columns])
    expr = expr.apply(pd.to_numeric, errors="coerce")
    # Keep the lineage markers alongside the surfaceome: they are needed to
    # stratify the cohort and were previously filtered out here, which would
    # have made every lineage-stratified analysis unrunnable in LIVE.
    keep_universe = set(gene_universe) | set(LINEAGE_MARKER_GENES)
    expr = expr[expr.index.isin(keep_universe)]
    # BUG FIX (raised in external code review, confirmed): duplicate
    # Hugo_Symbol rows were collapsed with .groupby(level=0).max(). That is
    # wrong twice over. First, a per-sample maximum assembles one profile out
    # of DIFFERENT source rows - sample A's value from probe 1 and sample B's
    # from probe 2 - producing an expression vector that was never measured
    # from any single assay. Second, `to_symbols()` elsewhere in this module
    # correctly sums duplicates ON THE LINEAR SCALE, so the two loaders
    # disagreed about the same operation and max on a log scale is not even
    # the right summary for additive transcript quantities.
    #
    # Duplicate symbols in a cBioPortal mRNA file are not self-describing:
    # they may be technical repeats, alternative assays, or distinct
    # transcripts. Rather than guess, the pipeline reports them and refuses
    # unless the aggregation is declared via SU2C_DUPLICATE_POLICY.
    scale_already_canonical = False
    n_dup = int(expr.index.duplicated().sum())
    if n_dup:
        policy = os.environ.get("SU2C_DUPLICATE_POLICY", "").strip().lower()
        dup_names = sorted(set(expr.index[expr.index.duplicated()]))[:8]
        if policy == "sum_linear":
            # Correct for additive transcript-level quantities: sum on the
            # LINEAR scale, never the log scale.
            # BUG FIX (external review, correct): this passed `declared=`,
            # but the parameter is `assume=`. The branch would have raised
            # TypeError on the first real SU2C file with duplicate symbols -
            # a LIVE-only crash the DEMO can never reach.
            sc = detect_expression_scale(expr, "SU2C mRNA (pre-collapse)",
                                         assume=os.environ.get("SU2C_SCALE"))
            # One converter, correct pseudocount per scale, canonical output.
            expr = collapse_duplicate_symbols(expr, sc, how="sum_linear")
            scale_already_canonical = True
            print(f"    {n_dup:,} duplicate gene symbol(s) SUMMED on the "
                  f"linear scale (source scale {sc}) and returned as "
                  f"canonical log2(TPM+1)")
        elif policy in ("mean", "first"):
            sc = detect_expression_scale(expr, "SU2C mRNA (pre-collapse)",
                                         assume=os.environ.get("SU2C_SCALE"))
            expr = collapse_duplicate_symbols(expr, sc, how=policy)
            scale_already_canonical = True
            print(f"    {n_dup:,} duplicate gene symbol(s) combined with "
                  f"policy={policy} on the linear scale (source scale {sc}) "
                  f"and returned as canonical log2(TPM+1)")
        else:
            raise ValueError(
                f"The SU2C mRNA matrix contains {n_dup:,} duplicate gene "
                f"symbol(s), e.g. {dup_names}. How to combine them depends on "
                f"what they are in THIS file release - technical repeats, "
                f"alternative assays, or distinct transcripts - and the file "
                f"does not say. The previous per-sample maximum built profiles "
                f"from different source rows and is not used. Inspect the "
                f"source annotation, then set SU2C_DUPLICATE_POLICY to one of: "
                f"'sum_linear' (additive transcript quantities), 'mean' "
                f"(technical replicates), or 'first' (keep the primary row). "
                f"Record the choice in the run manifest.")
    # cBioPortal mRNA files vary: some are log-scaled, some linear, some
    # z-scored. Decide explicitly, then put the matrix on the canonical scale.
    # detect_expression_scale REFUSES z-scored input rather than guessing.
    # CRITICAL BUG FIX (both external reviews, confirmed): the duplicate
    # collapse above already returns canonical log2(TPM+1) and sets
    # `scale_already_canonical`, but that flag was never READ here, so the
    # matrix was detected and converted a SECOND time. With
    # SU2C_SCALE=linear_tpm, two additive measurements of 2 and 3 TPM give
    # log2(5+1) = 2.585 after the collapse and then 1.842 after the second
    # transformation. No exception is raised - the abundances are simply
    # wrong, and they feed mCRPC prevalence, persistence, disease-context
    # eligibility and the PSMA-low strata.
    if scale_already_canonical:
        su2c_scale = "log1p_tpm"
        print("    scale: already canonical after duplicate collapse - NO "
              "second transformation applied (a second conversion here is "
              "what double-logged the matrix)")
    else:
        su2c_scale = detect_expression_scale(expr, "SU2C mRNA",
                                             assume=os.environ.get("SU2C_SCALE"))
        expr = to_canonical_log(expr, su2c_scale, "SU2C mRNA")

    cl_f = next(iter(ex_dir.rglob("data_clinical_sample.txt")), None)
    pheno = pd.DataFrame({"sample": expr.columns})
    if cl_f is not None:
        cl = pd.read_csv(cl_f, sep="\t", comment="#", low_memory=False)
        key = next((c for c in cl.columns if "SAMPLE_ID" in c.upper()), cl.columns[0])
        cl = cl.set_index(key)
        hist = next((c for c in cl.columns
                     if "HISTOLOG" in c.upper() or "SUBTYPE" in c.upper()), None)
        if hist:
            lab = cl[hist].reindex(expr.columns).astype(str)
            pheno["is_nepc"] = lab.str.contains("neuro|NEPC|small cell",
                                                case=False, na=False).values
    if "is_nepc" not in pheno:
        print("  NOTE: no histology column found; NEPC features will be skipped.")
        pheno["is_nepc"] = False
    return expr, pheno


def load_mcrpc_demo(surf):
    rng = np.random.default_rng(CFG.seed + 2)
    # Candidates come from the surfaceome; the lineage markers used to
    # STRATIFY the cohort (AR, the neuroendocrine panel) are not surface
    # proteins and must be carried separately or the stratified analyses
    # cannot run. They never enter the candidate ranking.
    genes = surf.gene.tolist() + [g for g in LINEAGE_MARKER_GENES
                                  if g not in set(surf.gene)]
    g, n = len(genes), 208      # SU2C 2019 RNA-seq cohort is ~200 samples
    idx = {gn: i for i, gn in enumerate(genes)}
    # Linear TPM first, then the canonical log scale - same reasoning as the
    # primary generator above.
    base = np.log2(rng.lognormal(1.1, 1.8, g).clip(0, 5000) + CANONICAL_PSEUDOCOUNT)
    expr = np.clip(base[:, None] + rng.normal(0, 1.1, (g, n)), 0, None)
    cols = [f"SU2C-{i:04d}" for i in range(n)]
    is_nepc = rng.random(n) < 0.17      # ~17% NEPC in SU2C is realistic

    expr = pd.DataFrame(expr, index=genes, columns=cols)
    # AR-pathway targets fall in NEPC; DLL3/SEZ6 rise. This is the key clinical
    # motivation for the study, so the demo must reproduce it.
    for gn in ["FOLH1", "KLK3", "TMPRSS2", "SLC45A3", "KLK2", "PSCA"]:
        if gn in idx:
            expr.loc[gn, is_nepc] -= rng.uniform(2.5, 4.5)
            expr.loc[gn, ~is_nepc] += rng.uniform(1.0, 2.5)
    for gn in ["DLL3", "SEZ6", "CD276", "SSTR2"]:
        if gn in idx:
            expr.loc[gn, is_nepc] += rng.uniform(2.5, 4.0)
    # AR signalling is suppressed in NEPC and spans a wide range in
    # AR-positive disease - the structure the AR-low axis splits on.
    if "AR" in expr.index:
        ar = rng.normal(5.0, 1.6, n)
        ar[is_nepc] -= rng.uniform(3.0, 4.5)        # AR-low / NEPC
        expr.loc["AR"] = np.clip(ar, 0, None)
    for gn in ["CHGA", "SYP", "INSM1", "ASCL1", "NCAM1"]:
        if gn in expr.index:
            v = rng.normal(2.0, 1.2, n)
            v[is_nepc] += rng.uniform(3.0, 4.5)
            expr.loc[gn] = np.clip(v, 0, None)
    expr = expr.clip(lower=0.0)
    assert (expr.values >= 0).all(), "mCRPC matrix holds negative log values"
    pheno = pd.DataFrame({"sample": cols, "is_nepc": is_nepc})
    return expr, pheno


def load_mcrpc(surf):
    print("[1.4] SU2C/PCF metastatic CRPC transcriptomes")
    if RUN_MODE == "LIVE":
        expr, pheno = load_mcrpc_live(surf.gene.tolist())
    else:
        expr, pheno = load_mcrpc_demo(surf)
    print(f"  mCRPC samples: {expr.shape[1]:,} (NEPC: {int(pheno.is_nepc.sum())}) | "
          f"genes: {expr.shape[0]:,}")
    return expr, pheno


# ----------------------------------------------------------------------------
# 1.5 CPTAC proteomics - orthogonal protein-level corroboration
# ----------------------------------------------------------------------------
# HONEST LIMITATION, STATED IN CODE: CPTAC's prostate proteogenomic cohort is
# PRIMARY tumour. There is no CPTAC mCRPC proteome. Protein-level corroboration
# therefore applies to the primary-disease arm of this study only; mCRPC and
# NEPC candidates remain transcript-level until validated in-house or against
# an external mCRPC proteomics set (e.g. a published metastatic proteome
# supplementary table supplied via PROTEOMICS_LOCAL_FILE).
#
# The loader does NOT hard-code a portal URL, because the prostate proteome's
# file path has moved repeatedly. Supply the matrix yourself and point
# PROTEOMICS_LOCAL_FILE at it; the expected schema is printed on failure.
# ----------------------------------------------------------------------------
PROTEOMICS_LOCAL_FILE = os.environ.get("PROTEOMICS_LOCAL_FILE", "")

def load_proteomics_live(gene_universe):
    """Load tumour-vs-normal proteomics in either supported form.

    (a) PRECOMPUTED results: columns gene, prot_log2fc, prot_pval (plus an
        optional source q-value, which is the preferred FDR route because its
        family is already the one the source analysis tested).
    (b) GENE x SAMPLE abundance matrix: a gene/symbol column plus sample
        columns. Requires PROTEOMICS_SAMPLE_ANNOTATION (sample -> sample_type)
        and PROTEOMICS_SCALE ('log2' or 'linear'). Differential testing is
        Welch on log2 abundance with BH correction over this analysis's own
        testing universe.

    Both branches are implemented. Nothing about the groups, the scale or the
    testing family is inferred from column names.
    """
    if not PROTEOMICS_LOCAL_FILE or not Path(PROTEOMICS_LOCAL_FILE).exists():
        raise FileNotFoundError(
            "CPTAC proteomics not found.\n"
            "Set PROTEOMICS_LOCAL_FILE to either:\n"
            "  (a) a gene x sample abundance matrix plus a sample annotation, or\n"
            "  (b) a pre-computed table with columns: gene, prot_log2fc, prot_pval\n"
            "Obtain the prostate proteome from the NCI Proteomic Data Commons "
            "(pdc.cancer.gov) or the corresponding publication's supplementary "
            "tables, then re-run. Do not substitute a different tumour type."
        )
    p = Path(PROTEOMICS_LOCAL_FILE)
    register_input_file(p, role="locally supplied proteomics")
    sep = "\t" if p.suffix in (".tsv", ".txt") else ","
    df = pd.read_csv(p, sep=sep, low_memory=False)
    cols = {c.lower(): c for c in df.columns}
    if {"gene", "prot_log2fc", "prot_pval"} <= set(cols):
        ren = {cols["gene"]: "gene", cols["prot_log2fc"]: "prot_log2fc",
               cols["prot_pval"]: "prot_pval"}
        keep = ["gene", "prot_log2fc", "prot_pval"]
        # Adjusted p-values from the ORIGINAL proteome-wide analysis, if the
        # source provided them. This is the preferred route: the family is
        # already whatever that analysis tested.
        for cand in ("prot_qval", "prot_qval_supplied", "padj", "qvalue",
                     "adj_pval", "fdr"):
            if cand in cols:
                ren[cols[cand]] = "prot_qval_supplied"
                keep.append("prot_qval_supplied")
                break
        out = df.rename(columns=ren)[keep]

        # Record the testing universe so the FDR family is documented rather
        # than inferred. PROTEOMICS_N_TESTED is the number of proteins the
        # SOURCE analysis tested, which is usually far larger than the
        # surfaceome subset that reaches this pipeline.
        n_tested = os.environ.get("PROTEOMICS_N_TESTED", "")
        out.attrs["n_tested_proteome"] = int(n_tested) if n_tested.isdigit() else 0
        out.attrs["source_file"] = p.name
        full_f = os.environ.get("PROTEOMICS_FULL_PVALUES", "")
        if full_f and Path(full_f).exists():
            fsep = "\t" if Path(full_f).suffix in (".tsv", ".txt") else ","
            register_input_file(full_f, role="full proteome p-value family")
            fp = pd.read_csv(full_f, sep=fsep, low_memory=False)
            fcols = {c.lower(): c for c in fp.columns}
            fp = fp.rename(columns={fcols.get("gene", "gene"): "gene",
                                    fcols.get("prot_pval", "prot_pval"): "prot_pval"})
            out.attrs["full_proteome_pvalues"] = fp[["gene", "prot_pval"]]
            print(f"  full proteome p-value table supplied: {len(fp):,} proteins")
        if "prot_qval_supplied" in out.columns:
            print(f"  source q-values present ({out.prot_qval_supplied.notna().sum():,} "
                  f"proteins) - preferred FDR route available")
        return out
    # NEW (both external reviews: the docstring advertised a gene-by-sample
    # abundance matrix and the implementation only accepted a precomputed
    # results table, so a documented input format could not be processed).
    # The matrix workflow is now implemented rather than the documentation
    # quietly narrowed, because sample-level input is what an investigator
    # downloading CPTAC actually has.
    #
    # Requires PROTEOMICS_SAMPLE_ANNOTATION: a two-column file mapping sample
    # -> sample_type, with tumour/normal spelled however the source spells
    # them. Nothing is guessed from column names.
    ann_f = os.environ.get("PROTEOMICS_SAMPLE_ANNOTATION", "")
    gene_col = cols.get("gene") or cols.get("symbol") or cols.get("hugo_symbol")
    if gene_col and ann_f and Path(ann_f).exists():
        register_input_file(ann_f, role="proteomics sample annotation")
        asep = "\t" if Path(ann_f).suffix in (".tsv", ".txt") else ","
        ann = pd.read_csv(ann_f, sep=asep, low_memory=False)
        acols = {c.lower(): c for c in ann.columns}
        skey = acols.get("sample") or acols.get("sample_id") or list(ann.columns)[0]
        stype = acols.get("sample_type") or acols.get("group")
        if stype is None:
            raise ValueError(
                f"{Path(ann_f).name} needs a `sample_type` (or `group`) "
                f"column naming tumour and normal samples. Inferring the "
                f"groups from column-name patterns is how a tumour-vs-normal "
                f"contrast silently becomes the wrong contrast.")
        ann = ann.rename(columns={skey: "sample", stype: "sample_type"})
        lab = ann.set_index(ann["sample"].astype(str))["sample_type"].astype(str)
        mat = df.set_index(df[gene_col].astype(str)).drop(columns=[gene_col])
        mat = mat.apply(pd.to_numeric, errors="coerce")
        shared = [c for c in mat.columns if str(c) in lab.index]
        if not shared:
            raise ValueError(
                f"No sample in {p.name} matched {Path(ann_f).name}. The "
                f"abundance matrix columns and the annotation keys must use "
                f"the same identifiers; an unmatched merge would yield an "
                f"empty contrast rather than an error.")
        mat = mat[shared]
        lab = lab.loc[[str(c) for c in shared]]
        is_t = lab.str.contains("tumou?r|primary|case", case=False, regex=True).values
        is_n = lab.str.contains("normal|adjacent|benign|control", case=False,
                                regex=True).values
        if int(is_t.sum()) < 3 or int(is_n.sum()) < 3:
            raise ValueError(
                f"Proteomics tumour/normal split is too small to test: "
                f"{int(is_t.sum())} tumour, {int(is_n.sum())} normal "
                f"(need >= 3 each). Values in `sample_type` were "
                f"{sorted(set(lab))[:8]}.")
        if int((is_t & is_n).sum()):
            raise ValueError(
                "A sample was classified as BOTH tumour and normal by its "
                "`sample_type` value. Fix the annotation vocabulary.")
        # BUG FIX (found reviewing this round's own new code): samples whose
        # `sample_type` matched NEITHER pattern were silently excluded from
        # the contrast with no record - the identical defect class fixed
        # earlier in the Toil loader, where normals had been selected as
        # "anything not labelled tumour". A CPTAC cohort containing
        # metastatic or cell-line samples would have been analysed as a
        # subset, with the denominator of every protein fold change quietly
        # different from what the file contains.
        unmatched = ~(is_t | is_n)
        if int(unmatched.sum()):
            kinds = sorted(set(lab.values[unmatched]))
            print(f"  EXCLUDED {int(unmatched.sum())} proteomics sample(s) "
                  f"whose sample_type matched neither tumour nor normal: "
                  f"{kinds[:8]}. They are NOT in the contrast. If any of "
                  f"these should be included, extend the annotation "
                  f"vocabulary deliberately rather than relying on pattern "
                  f"matching.")
        # Abundance matrices arrive variously as log2 ratios, log2 intensities
        # or linear intensities. A log-scale difference of means is the
        # estimand for log2FC; a LINEAR matrix must be logged first, and that
        # decision is declared, never inferred.
        pscale = os.environ.get("PROTEOMICS_SCALE", "").strip().lower()
        if pscale not in ("log2", "linear"):
            raise ValueError(
                "Set PROTEOMICS_SCALE to 'log2' (abundances already on a log "
                "scale - the usual CPTAC form) or 'linear'. Guessing is how a "
                "fold change ends up on the wrong scale.")
        # BUG FIX (both external reviews, correct): clip(lower=0) silently
        # replaced any negative value with zero before the log transform. On
        # a declared LINEAR abundance scale a negative value is not an
        # outlier to tame - it is evidence that the matrix is not linear
        # abundance at all (a log-ratio mislabelled, or a background
        # subtraction left in), and zeroing it manufactures a plausible
        # measurement from an invalid one. NaN is preserved, because
        # missingness has its own documented rules downstream.
        if pscale == "linear":
            _pv = mat.to_numpy(dtype=float)
            if np.isinf(_pv).any():
                raise ValueError(
                    "Infinite values in the proteomics abundance matrix. "
                    "Check for a division by zero in the source "
                    "normalisation.")
            _pobs = _pv[np.isfinite(_pv)]
            if _pobs.size and (_pobs < 0).any():
                _n_neg = int((_pobs < 0).sum())
                raise ValueError(
                    f"PROTEOMICS_SCALE='linear' but the matrix contains "
                    f"{_n_neg:,} negative value(s) (min "
                    f"{_pobs.min():.4g}). Linear protein abundance cannot be "
                    f"negative, so this matrix is either already on a log or "
                    f"ratio scale - set PROTEOMICS_SCALE='log2' - or carries "
                    f"an uncorrected background subtraction. Clipping these "
                    f"to zero would fabricate measurements.")
            work = np.log2(mat + 1.0)
        else:
            work = mat
        t_vals, n_vals = work.loc[:, is_t], work.loc[:, is_n]
        enough = (t_vals.notna().sum(axis=1) >= 3) & (n_vals.notna().sum(axis=1) >= 3)
        work_t, work_n = t_vals.loc[enough], n_vals.loc[enough]
        stat, pv = st.ttest_ind(work_t.values, work_n.values, axis=1,
                                equal_var=False, nan_policy="omit")
        lfc = np.nanmean(work_t.values, axis=1) - np.nanmean(work_n.values, axis=1)
        out = pd.DataFrame({
            "gene": work_t.index.astype(str),
            "prot_log2fc": lfc,
            "prot_pval": np.nan_to_num(np.asarray(pv, dtype=float), nan=1.0),
        })
        # The FDR family is THIS analysis's own testing universe: every
        # protein tested here, not the surfaceome subset that survives later.
        out["prot_qval_supplied"] = multipletests(
            out.prot_pval.values, method="fdr_bh")[1]
        out.attrs["n_tested_proteome"] = int(len(out))
        out.attrs["source_file"] = p.name
        out.attrs["de_method"] = "welch_t_on_log2_abundance"
        print(f"  proteomics MATRIX workflow: {int(enough.sum()):,} protein(s) "
              f"tested, {int(is_t.sum())} tumour vs {int(is_n.sum())} normal "
              f"({pscale} input); BH q-values computed over this analysis's "
              f"own {len(out):,}-protein family")
        print(f"  NOTE: Welch t-tests on log2 abundance. A moderated test "
              f"(limma-style empirical Bayes) is better powered at this "
              f"sample size; supply a precomputed table if you have one.")
        return out

    raise ValueError(
        f"Could not interpret {p.name}. Columns found: {list(df.columns)[:12]}.\n"
        f"Supported inputs:\n"
        f"  (a) precomputed results: columns gene, prot_log2fc, prot_pval\n"
        f"  (b) gene x sample abundance matrix: a gene/symbol column plus "
        f"sample columns, WITH PROTEOMICS_SAMPLE_ANNOTATION set to a "
        f"sample->sample_type file and PROTEOMICS_SCALE set to log2 or linear"
    )


def load_proteomics_demo(surf, rna_lfc):
    """Synthetic proteome correlated with, but noisier than, the transcriptome.

    Correlation is set near r=0.5, which is the realistic ceiling for
    mRNA-protein agreement in tumour tissue. Making it higher would flatter the
    pipeline; making it lower would make corroboration impossible by
    construction.
    """
    rng = np.random.default_rng(CFG.seed + 3)
    lfc = rna_lfc.reindex(surf.gene).fillna(0).values
    noise = rng.normal(0, 1.0, len(lfc))
    prot = 0.55 * lfc + 0.85 * noise
    # Measured proteome coverage is ~8-11k proteins, and membrane proteins are
    # systematically under-detected by bulk MS. Drop ~45% at random.
    detected = rng.random(len(lfc)) > 0.45
    se = rng.uniform(0.25, 0.65, len(lfc))
    z = prot / se
    pval = 2 * st.norm.sf(np.abs(z))
    out = pd.DataFrame({"gene": surf.gene.values,
                        "prot_log2fc": prot, "prot_pval": pval})
    out = out[detected].reset_index(drop=True)

    # The demo emits a FULL proteome-wide family, not just the surfaceome
    # rows, so the declared-FDR-family code path is actually exercised rather
    # than merely present. A real MS experiment quantifies roughly 9,000-11,000
    # proteins, of which the surfaceome is a minority.
    n_non_surface = 8200
    noise_p = 2 * st.norm.sf(np.abs(rng.normal(0, 1, n_non_surface)))
    full = pd.concat([
        out[["gene", "prot_pval"]],
        pd.DataFrame({"gene": [f"NONSURF{i:05d}" for i in range(n_non_surface)],
                      "prot_pval": noise_p}),
    ], ignore_index=True)
    out.attrs["full_proteome_pvalues"] = full
    out.attrs["n_tested_proteome"] = len(full)
    # Supply source q-values computed over that full family - the preferred
    # route, and what a real source analysis would hand you.
    q_full = multipletests(full.prot_pval.values, method="fdr_bh")[1]
    qmap = dict(zip(full.gene.values, q_full))
    out["prot_qval_supplied"] = out.gene.map(qmap)
    print(f"    demo proteome-wide family: {len(full):,} proteins tested, "
          f"{len(out):,} within the surfaceome")
    return out


def load_proteomics(surf, rna_lfc):
    print("[1.5] Proteomics (protein-level corroboration)")
    if RUN_MODE == "LIVE":
        out = load_proteomics_live(surf.gene.tolist())
    else:
        out = load_proteomics_demo(surf, rna_lfc)
    print(f"  proteins quantified: {len(out):,} / {len(surf):,} surfaceome genes "
          f"({100*len(out)/len(surf):.0f}% coverage)")
    return out


# ----------------------------------------------------------------------------
# 1.6 Human Protein Atlas - reagent availability and IHC
# ----------------------------------------------------------------------------
# Schema: gene, subcellular, antibody_available(bool), n_antibodies(int),
#         hpa_membrane(bool), ihc_prostate_high(bool)
# ----------------------------------------------------------------------------
def load_hpa_live():
    import zipfile
    z = download(DATA_URLS["hpa"], input_path("hpa"), input_min("hpa"))
    with zipfile.ZipFile(z) as zf:
        name = zf.namelist()[0]
        with zf.open(name) as f:
            hpa = pd.read_csv(f, sep="\t", low_memory=False)
    cols = {c.lower(): c for c in hpa.columns}
    def pick(*keys):
        for k in keys:
            for lc, orig in cols.items():
                if k in lc:
                    return orig
        return None
    gcol = pick("gene")
    sub = pick("subcellular main location", "subcellular location")
    ab = pick("antibody")
    out = pd.DataFrame({"gene": hpa[gcol].astype(str)})
    out["subcellular"] = hpa[sub].astype(str) if sub else ""
    out["n_antibodies"] = (hpa[ab].astype(str).str.count(",") + 1) if ab else 0
    out.loc[hpa[ab].isna() if ab else slice(None), "n_antibodies"] = 0
    out["antibody_available"] = out.n_antibodies > 0
    out["hpa_membrane"] = out.subcellular.str.contains(
        "plasma membrane|cell junction", case=False, na=False)
    ihc = pick("prostate")
    out["ihc_prostate_high"] = (hpa[ihc].astype(str).str.contains(
        "High|Medium", case=False, na=False) if ihc else False)
    return out.drop_duplicates("gene").reset_index(drop=True)


def load_hpa_demo(surf):
    rng = np.random.default_rng(CFG.seed + 4)
    n = len(surf)
    pos = set([g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
              + PROSTATE_HELDOUT_POSITIVES + NEPC_CONTEXT_BENCHMARKS)
    # Reagent coverage is not random in reality: well-studied targets have more
    # antibodies. That is exactly the confounder the review section discusses,
    # and the leakage ablation is what measures how much it matters. The gap is
    # deliberately moderate rather than absolute, since in reality plenty of
    # undrugged proteins are well-reagented.
    n_ab = np.where(surf.gene.isin(pos), rng.integers(2, 12, n), rng.integers(0, 8, n))
    out = pd.DataFrame({
        "gene": surf.gene.values,
        "n_antibodies": n_ab,
        "antibody_available": n_ab > 0,
        "hpa_membrane": (surf.n_transmem.values > 0) & (rng.random(n) < 0.88),
        "subcellular": np.where(surf.n_transmem.values > 0, "Plasma membrane", "Nucleoplasm"),
        "ihc_prostate_high": rng.random(n) < 0.3,
    })
    out.loc[out.gene.isin(pos), "ihc_prostate_high"] = True
    # Per-filter control: no antibody exists, so F4 must reject it.
    out.loc[out.gene == "CTRL_NO_AB", ["n_antibodies", "antibody_available"]] = [0, False]
    for g in ["CTRL_NOT_DE", "CTRL_HIGH_VITAL", "CTRL_NO_ECD"]:
        out.loc[out.gene == g, ["n_antibodies", "antibody_available"]] = [4, True]
        out.loc[out.gene == g, "hpa_membrane"] = True
    return out


def load_hpa(surf):
    print("[1.6] Human Protein Atlas annotation")
    hpa = load_hpa_live() if RUN_MODE == "LIVE" else load_hpa_demo(surf)
    print(f"  HPA records: {len(hpa):,} | with >=1 antibody: {hpa.antibody_available.sum():,}")
    return hpa


# ----------------------------------------------------------------------------
# 1.7 Clinical follow-up with genetic-ancestry annotation
# ----------------------------------------------------------------------------
# Schema: sample, surv_time (months), surv_event (0/1), ancestry, gleason,
#         stage_t, psa, age
#
# CRITICAL: ancestry must be GENETIC ancestry, not self-reported race, if the
# manuscript claims ancestry-stratified biology. TCGA-PRAD genetic ancestry
# calls are published (Carrot-Zhang et al., Cancer Cell 2020 ancestry
# annotation). Self-reported race is a social variable and a poor proxy; using
# it while writing "ancestry" is a common and serious misstatement.
# ----------------------------------------------------------------------------
# CRITICAL BUG FIX (external review, confirmed): the event indicator was
# parsed with `str.contains("1|Recur|Progress|DECEASED")`, a regex with two
# failure modes that both produce FALSE EVENTS:
#   * the alternative "1" matches the digit anywhere in the string, so
#     "10:DISEASE FREE" and any coded value containing a 1 became an event;
#   * "Recur" matches inside "NO RECURRENCE", turning an explicit NON-event
#     into an event.
# A false-positive event changes Kaplan-Meier curves, Cox hazard ratios, event
# counts and every power calculation derived from them - silently, because the
# parse always succeeds.
#
# Status codes are now decoded from an explicit dictionary of documented
# values, and an unrecognised code RAISES instead of being guessed. The map
# below covers the cBioPortal/TCGA conventions this loader targets; a
# different endpoint source needs its own documented map, which is why the
# error names the offending values rather than defaulting them to zero.
EVENT_STATUS_MAP = {
    "0:DISEASEFREE": 0, "1:RECURRED/PROGRESSED": 1, "1:RECURRED": 1,
    "1:PROGRESSED": 1, "0:CENSORED": 0, "1:PROGRESSION": 1,
    "0:LIVING": 0, "1:DECEASED": 1, "0:ALIVE": 0, "1:DEAD": 1,
    "0:NOTPROGRESSED": 0, "0:NORECURRENCE": 0,
    "DISEASEFREE": 0, "RECURRED": 1, "PROGRESSED": 1, "CENSORED": 0,
    "LIVING": 0, "DECEASED": 1, "NORECURRENCE": 0,
    "0": 0, "1": 1,
}


def parse_event_status(col, col_name="event column"):
    """Decode a survival event indicator from documented status codes only."""
    status = (col.astype("string").str.strip().str.upper()
              .str.replace(" ", "", regex=False)
              .str.replace("_", "", regex=False))
    known = status.isin(EVENT_STATUS_MAP)
    unknown = status.notna() & ~known
    if unknown.any():
        bad = sorted(set(status[unknown].dropna()))[:12]
        raise ValueError(
            f"Unrecognised survival status code(s) in {col_name}: {bad}. "
            f"These are NOT decoded by pattern matching, because a regex that "
            f"matched '1' anywhere turned '10:DISEASE FREE' into an event and "
            f"'NO RECURRENCE' into a recurrence. Add each documented code to "
            f"EVENT_STATUS_MAP with its correct 0/1 meaning, taken from the "
            f"data dictionary of THIS endpoint source.")
    out = status.map(EVENT_STATUS_MAP)
    n_ev = int((out == 1).sum())
    n_cen = int((out == 0).sum())
    n_na = int(out.isna().sum())
    print(f"  event status decoded from {col_name}: {n_ev} event(s), "
          f"{n_cen} censored, {n_na} missing "
          f"(explicit code map, no pattern matching)")
    return pd.to_numeric(out, errors="coerce")


SURVIVAL_ENDPOINT_PAIRS = (
    ("DFS_MONTHS", "DFS_STATUS", "disease_free_survival"),
    ("PFS_MONTHS", "PFS_STATUS", "progression_free_survival"),
)


def survival_eligibility_mask(df):
    """The SINGLE definition of an analysable survival observation.

    Both external reviews, and the newly identified Bug 3: the BCR validator
    counted an observation usable when the time was merely non-missing,
    while every downstream model filtered `surv_time > 0`. Zero-duration
    records therefore passed the endpoint-coverage gate at 100% and were
    then dropped by the analysis, so quality control could approve a file
    that yields no analysed observations. The rule was implemented in five
    places with two meanings; it is now implemented once.

    A zero-duration record is not necessarily a source error - it is a
    record this pipeline's models exclude, so it must not be counted as
    usable by the gate that admits the file.
    """
    for _c in ("surv_time", "surv_event"):
        if _c not in df.columns:
            raise ValueError(
                f"survival_eligibility_mask: missing required column "
                f"{_c!r}. Eligibility cannot be decided from an incomplete "
                f"survival table.")
    time = pd.to_numeric(df["surv_time"], errors="coerce")
    event = pd.to_numeric(df["surv_event"], errors="coerce")
    # SELF-REVIEW: coercion here turned a malformed value into NaN, which
    # the mask then reported as merely INELIGIBLE - so a record carrying
    # "unknown" where a time should be was dropped from the analysis with no
    # message, indistinguishable from a legitimately censored patient. The
    # LIVE validators raise on malformed input, but this mask is also the
    # filter applied to DEMO and to any table assembled downstream, so it
    # was the one place where bad data could leave silently. A value that is
    # present but unparseable is a source error wherever it is met.
    _bad_time = df["surv_time"].notna() & time.isna()
    _bad_event = df["surv_event"].notna() & event.isna()
    if _bad_time.any() or _bad_event.any():
        raise ValueError(
            f"survival_eligibility_mask: {int(_bad_time.sum())} "
            f"unparseable follow-up time(s) and {int(_bad_event.sum())} "
            f"unparseable event code(s) are present but not numeric, e.g. "
            f"time={sorted(set(df['surv_time'][_bad_time].astype(str)))[:3]}, "
            f"event={sorted(set(df['surv_event'][_bad_event].astype(str)))[:3]}. "
            f"Treating them as ineligible would drop those records from the "
            f"analysis silently, indistinguishable from censored patients.")
    return time.notna() & (time > 0) & event.isin([0, 1])


TCGA_SAMPLE_TYPE_CODES = {
    "01": "Primary Solid Tumor", "02": "Recurrent Solid Tumor",
    "03": "Primary Blood Derived Cancer - Peripheral Blood",
    "05": "Additional - New Primary", "06": "Metastatic",
    "07": "Additional Metastatic", "10": "Blood Derived Normal",
    "11": "Solid Tissue Normal", "12": "Buccal Cell Normal",
    "14": "Bone Marrow Normal",
}

TCGA_BARCODE_RE = r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})-(\d{2})[A-Z]?(?:-|$)"


def parse_tcga_barcodes(sample_ids, label="TCGA samples"):
    """Parse TCGA sample barcodes into patient ID and sample-type code.

    External review, correct: validate_identifier_column(truncate_to=12)
    checks the identifier's LENGTH, not its format, so "BAD-INPUT-1234-01A"
    was accepted and silently yielded the patient ID "BAD-INPUT-12". A
    length test is not a format test, and my claim that malformed prefixes
    were refused was only half true.

    This parser is TCGA-SPECIFIC by design and is applied only to TCGA
    inputs. SU2C and other cohorts use different identifier systems, so
    imposing a TCGA pattern on them would reject valid data - the same class
    of mistake as my earlier barcode-collision policy.

    Returns a frame of patient and sample_type. Several specimens per
    patient are legitimate and preserved; an unrecognised sample-type code
    is reported rather than silently given a fallback rank.
    """
    ids = pd.Series(list(sample_ids)).astype("string").str.strip().str.upper()
    parsed = ids.str.extract(TCGA_BARCODE_RE)
    invalid = parsed[0].isna()
    if invalid.any():
        raise ValueError(
            f"{label}: {int(invalid.sum())} malformed TCGA barcode(s), e.g. "
            f"{ids[invalid].head(5).tolist()}. Expected "
            f"TCGA-XX-XXXX-NN[A]...; a length check alone accepts strings "
            f"like 'BAD-INPUT-1234-01A' and yields the nonsense patient ID "
            f"'BAD-INPUT-12'.")
    unknown = sorted(set(parsed[1].dropna()) - set(TCGA_SAMPLE_TYPE_CODES))
    if unknown:
        print(f"    {label}: sample-type code(s) {unknown} are not in the "
              f"TCGA data dictionary; those specimens get the lowest "
              f"selection preference rather than being assumed comparable "
              f"to a primary tumour.")
    n_pat = int(parsed[0].nunique())
    if len(ids) > n_pat:
        print(f"    {label}: {len(ids):,} specimen(s) from {n_pat:,} "
              f"patient(s) - repeated specimens are expected and are "
              f"resolved by the prespecified selection rule.")
    return pd.DataFrame({"patient": parsed[0].to_numpy(),
                         "sample_type": parsed[1].to_numpy()},
                        index=pd.Series(list(sample_ids)).index)


def validate_patient_outcome_consistency(clinical, strict=True):
    """Confirm repeated specimen records AGREE on the patient's outcome.

    Both external reviews, independently, on the reduction I wrote last
    round: selecting the longest follow-up among a patient's records makes
    specimen selection depend on the OUTCOME. Demonstrated with their
    example - records (12 months, recurrence) and (99 months, censored) for
    one patient selected the 99-month row, and the patient's recurrence
    event disappeared from the analysis. Both records are numerically
    eligible, so nothing raised.

    That is selection bias, not a reporting detail: a specimen chosen
    because its follow-up is long or its event absent conditions the
    analysis on the very quantity being modelled. My rule was deterministic,
    which is not the same as unbiased.

    Conflicting patient-level endpoints are a source-reconciliation problem
    and are reported as such. In real longitudinal records follow-up is
    legitimately updated, so this is a conservative gate: with strict=False
    it reports rather than raises, for cohorts where a documented
    source-precedence rule resolves the disagreement.
    """
    required = ["patient", "surv_time", "surv_event"]
    missing = [c for c in required if c not in clinical.columns]
    if missing:
        raise ValueError(
            f"validate_patient_outcome_consistency: missing column(s) "
            f"{missing}")
    cols = [c for c in ("surv_time", "surv_event", "endpoint_name")
            if c in clinical.columns]
    disagree = (clinical.groupby("patient")[cols].nunique(dropna=False)
                .gt(1).any(axis=1))
    bad = sorted(disagree[disagree].index.tolist())
    if bad:
        msg = (f"{len(bad)} patient(s) have CONFLICTING endpoint records "
               f"across their specimens, e.g. {bad[:5]}. Selecting one "
               f"specimen by follow-up duration or event status would "
               f"resolve the conflict by conditioning on the outcome - a "
               f"recurrence event can disappear entirely. Reconcile these "
               f"at the source, or set a documented precedence rule.")
        if strict:
            raise ValueError(msg)
        print(f"  WARNING: {msg}")
    return bad


def select_one_specimen_per_patient(clin, label="survival analysis"):
    """Reduce to one record per patient WITHOUT consulting the outcome.

    Replaces the longest-follow-up rule, which both reviews correctly
    identified as outcome-dependent. Patient-level outcomes are reconciled
    first (validate_patient_outcome_consistency), so every remaining
    specimen for a patient carries the SAME endpoint - which means the
    choice of specimen can no longer change the patient's outcome at all.

    The specimen is then chosen by a prespecified rule that uses only
    specimen metadata, in this order:
      1. specimen-type preference, where a TCGA sample-type code is present
         (01 primary tumour before 06 metastatic before 11 normal), because
         the analysis is of primary disease;
      2. the lowest sample identifier, as a deterministic tie-break.
    Neither key can see surv_time or surv_event. Changing a patient's
    follow-up therefore cannot change which specimen is analysed - the
    property the review asked to be testable.
    """
    # BUG FIX (external review, correct): this returned the frame unchanged
    # when the patient column was absent - a silent no-op in a function
    # whose entire purpose is to GUARANTEE one record per patient. The LIVE
    # loader now supplies validated identifiers, so this is a contract
    # weakness rather than a live defect, but a direct caller could bypass
    # the whole patient-level safeguard and obtain sample-level survival
    # estimates that look patient-level.
    if "patient" not in clin.columns:
        raise ValueError(
            f"{label}: a patient-level analysis was requested but the table "
            f"has no 'patient' column (columns: "
            f"{sorted(map(str, clin.columns))[:12]}). Returning the table "
            f"unchanged would let sample-level records be analysed and "
            f"reported as patients, inflating the event count and narrowing "
            f"every interval.")
    n_before = len(clin)
    if n_before == int(clin.patient.nunique()):
        return clin, 0
    work = clin.copy()
    if "sample" in work.columns:
        # SELF-REVIEW: the first version matched the sample-type code with
        # the regex r"-(\d{2})[A-Z]?$", anchored to the END of the barcode.
        # On a REAL full-length TCGA aliquot barcode
        # (TCGA-AA-0001-01A-11R-A056-07) that captures "07" - the plate or
        # centre code - not the sample type, and it does so for the tumour
        # and the normal barcode alike. So the specimen-type preference
        # silently stopped applying on real data, and a barcode ending in
        # -01 or -11 would have been read as a sample type it does not
        # carry, potentially ranking a NORMAL specimen as primary tumour.
        #
        # It passed its test because my fixture ("TCGA-AA-0001-01A") was
        # shaped to match my implementation rather than the data: a
        # four-field barcode is a sample ID, not an aliquot ID. The code is
        # now read POSITIONALLY from field 4, which is where the TCGA
        # specification puts sample+vial, so every barcode length parses the
        # same field.
        if "sample_type" in work.columns:
            # Parsed once at the loader boundary by parse_tcga_barcodes();
            # re-deriving it here is how the two came to disagree.
            _code = work["sample_type"].astype("string")
        else:
            _parts = work["sample"].astype(str).str.split("-")
            _field4 = _parts.str[3].fillna("")
            _code = (_field4.str.replace(r"[^0-9]", "", regex=True).str[:2]
                     .where(lambda c: c.str.len() == 2))
        _rank = {"01": 0, "03": 1, "06": 2, "11": 3}
        work["_type_rank"] = _code.map(_rank).fillna(9).astype(int)
        order, asc = ["_type_rank", "sample"], [True, True]
    else:
        work["_type_rank"] = 0
        order, asc = ["_type_rank"], [True]
    kept = (work.sort_values(order, ascending=asc)
            .drop_duplicates(subset="patient", keep="first")
            .sort_index().drop(columns="_type_rank"))
    n_dropped = n_before - len(kept)
    print(f"    {label}: {n_before:,} specimen record(s) -> {len(kept):,} "
          f"unique patient(s); {n_dropped:,} additional specimen(s) set "
          f"aside by the prespecified rule (specimen type, then sample ID - "
          f"never follow-up or event status, which would make selection "
          f"outcome-dependent).")
    return kept, n_dropped


def enforce_one_sample_per_patient(clin, label="survival analysis"):
    """Reconcile patient outcomes, then select one specimen per patient.

    Retained as the single entry point used across the pipeline. It now
    performs the two steps in the order the reviews require: outcomes are
    validated at the patient level FIRST, so specimen selection cannot
    change an outcome, and the selection itself never reads the outcome.
    """
    # BUG FIX (external review, correct): this returned the frame unchanged
    # when the patient column was absent - a silent no-op in a function
    # whose entire purpose is to GUARANTEE one record per patient. The LIVE
    # loader now supplies validated identifiers, so this is a contract
    # weakness rather than a live defect, but a direct caller could bypass
    # the whole patient-level safeguard and obtain sample-level survival
    # estimates that look patient-level.
    if "patient" not in clin.columns:
        raise ValueError(
            f"{label}: a patient-level analysis was requested but the table "
            f"has no 'patient' column (columns: "
            f"{sorted(map(str, clin.columns))[:12]}). Returning the table "
            f"unchanged would let sample-level records be analysed and "
            f"reported as patients, inflating the event count and narrowing "
            f"every interval.")
    # BUG FIX (external review, correct): with
    # strict_patient_outcome_consistency=False the validator warned and the
    # pipeline carried on, so specimen selection still decided which of two
    # contradictory patient-level outcomes became authoritative - safer than
    # the old longest-follow-up rule, but the conflict was still resolved by
    # a rule that has no clinical standing.
    #
    # Non-strict mode is now an AUDIT mode: it reports every conflicting
    # patient without raising inside the validator, and modelling is refused
    # here regardless. A documented source-precedence rule belongs upstream,
    # applied to the patient-level endpoint table, not inferred from which
    # specimen happens to be preferred.
    _conflicts = validate_patient_outcome_consistency(clin, strict=False)
    if _conflicts:
        raise ValueError(
            f"Unresolved patient-level endpoint conflict(s) for "
            f"{len(_conflicts)} patient(s), e.g. {_conflicts[:5]}. Clinical "
            f"source reconciliation is required before any survival model is "
            f"fitted: specimen selection must not decide which contradictory "
            f"outcome is authoritative. Resolve these in the source, or apply "
            f"a documented precedence rule to the patient-level endpoint "
            f"table before calling this function. "
            f"(CFG.strict_patient_outcome_consistency controls whether the "
            f"validator itself raises; it does not permit modelling on "
            f"unreconciled records.)")
    return select_one_specimen_per_patient(clin, label=label)


def validate_identifier_column(df, column, label, truncate_to=None):
    """One identifier contract for every patient- and sample-level join.

    Both external reviews reproduced this: pandas matches a null join key to
    another null join key, so merge_clinical_to_samples() reported a 100%
    clinical match rate while linking a missing identifier to a missing
    identifier. validate="many_to_one" does not catch it, because null keys
    are unique. Placeholder strings are rejected too, since .astype(str)
    upstream turns a missing value into the literal "nan".

    With truncate_to set, a collision introduced BY the truncation (two
    distinct source barcodes collapsing to one patient key) is reported -
    a real TCGA hazard that duplicate checks on the truncated column alone
    attribute to the source file.
    """
    if column not in df.columns:
        raise ValueError(f"{label}: missing required column {column!r}")
    ids = df[column].astype("string").str.strip()
    invalid = (ids.isna() | ids.eq("")
               | ids.str.upper().isin(["NAN", "NONE", "NULL", "NA", "<NA>"]))
    if invalid.fillna(True).any():
        _n = int(invalid.fillna(True).sum())
        raise ValueError(
            f"{label}: {_n} missing or placeholder {column!r} "
            f"identifier(s). A null join key matches other null join keys in "
            f"pandas, so these would be recorded as successful record "
            f"linkages and inflate every coverage statistic. Resolve or "
            f"exclude these records in the source data.")
    if truncate_to:
        # SELF-REVIEW + both external reviews, correctly: my first version
        # raised whenever two full identifiers shared a truncated key - but
        # for TCGA that is the NORMAL relationship. Two specimens of one
        # patient (TCGA-AA-0001-01A and TCGA-AA-0001-11A) legitimately
        # truncate to TCGA-AA-0001, and the check would have rejected valid
        # data. It only ever passed because truncate_to was used nowhere in
        # production, which made a wrong policy look like a working one.
        #
        # The real contract: many specimens per patient is legal and is
        # handled by the specimen-selection rule; a MALFORMED barcode, one
        # that does not carry a resolvable patient prefix, is not.
        short = ids.str[:truncate_to]
        _too_short = ids.str.len() < truncate_to
        if _too_short.any():
            _bad = sorted(set(ids[_too_short]))[:5]
            raise ValueError(
                f"{label}: {int(_too_short.sum())} {column!r} value(s) are "
                f"shorter than the {truncate_to}-character patient prefix, "
                f"e.g. {_bad}. A truncated key cannot be resolved to a "
                f"patient, and padding or accepting it would merge "
                f"unrelated records.")
        _n_patients = int(short.nunique())
        _n_specimens = int(len(short))
        if _n_specimens > _n_patients:
            print(f"    {label}: {_n_specimens:,} specimen(s) map to "
                  f"{_n_patients:,} patient(s) - repeated specimens are "
                  f"expected and are resolved by the prespecified specimen "
                  f"selection rule, not treated as an error.")
        return short
    return ids


def validate_ancestry_request(cfg, ancestry_file):
    """Enforce the genetic-ancestry contract BEFORE any clinical data loads.

    Both external reviews reproduced this: the coverage gate added last
    round lives inside the branch that LOADS an ancestry file, so with
    ancestry_inference_prespecified=True and no ANCESTRY_FILE at all the
    loader completed and returned self-reported race as the ancestry
    variable. That is the same structural defect as the join guards I fixed
    in the same round - a safeguard conditional on the optional input being
    present - and asking for genetic ancestry while silently receiving race
    is the one substitution this pipeline must never make.
    """
    if getattr(cfg, "ancestry_inference_prespecified", False):
        if not ancestry_file:
            raise ValueError(
                "CFG.ancestry_inference_prespecified is True, but "
                "ANCESTRY_FILE is not supplied. The analysis would fall back "
                "to SELF-REPORTED RACE, a different variable with a "
                "different estimand, and report it as ancestry. Supply the "
                "TCGA genetic-ancestry annotation, or set "
                "ancestry_inference_prespecified=False to report a "
                "race-based breakdown as explicitly exploratory.")
        if not Path(ancestry_file).is_file():
            raise FileNotFoundError(
                f"ancestry_inference_prespecified is True and ANCESTRY_FILE="
                f"{ancestry_file!r}, but that path is not a file.")
    return True


def validate_survival_fields(raw_time, event, label):
    """Validate follow-up times and event codes for ANY endpoint source.

    Shared by the external BCR branch and the built-in DFS/PFS branch,
    because the defect reproduced in both external reviews - non-numeric
    times coerced to missing and then skipped - was fixed on one path and
    left on the other. One contract, both paths.

    `event` must already have passed parse_event_status(). Returns the
    parsed numeric time series.
    """
    _parsed = pd.to_numeric(raw_time, errors="coerce")
    _malformed = pd.Series(raw_time).notna() & _parsed.isna()
    if _malformed.any():
        _bad = sorted(set(pd.Series(raw_time)[_malformed].astype(str)))[:5]
        raise ValueError(
            f"{label}: {int(_malformed.sum())} non-numeric follow-up "
            f"time(s), e.g. {_bad}. A time that cannot be parsed is a "
            f"source-data error, not a censored observation; coercing it to "
            f"missing would report endpoint coverage on data with no "
            f"analysable follow-up.")
    if np.isinf(_parsed.to_numpy(dtype=float)).any():
        raise ValueError(f"{label}: infinite follow-up time(s).")
    _neg = _parsed.notna() & (_parsed < 0)
    if _neg.any():
        raise ValueError(
            f"{label}: {int(_neg.sum())} negative follow-up time(s). Time "
            f"to event cannot be negative; check the time origin and units.")
    _ev = pd.Series(event).dropna()
    if len(_ev) and not _ev.isin([0, 1]).all():
        raise ValueError(
            f"{label}: event indicator(s) outside {{0, 1}} after decoding: "
            f"{sorted(set(_ev) - {0, 1})}.")
    return _parsed


def merge_clinical_to_samples(smap, clinical):
    """Join sample-level records to patient-level clinical records.

    SELF-REVIEW + both external reviews: the clinical-side duplicate checks I
    added last round live INSIDE the BCR and ancestry branches, so they are
    conditional on those optional inputs being supplied. With neither file
    present nothing validated patient uniqueness, and this final merge could
    still multiply sample rows. Uniqueness is now enforced unconditionally,
    the cardinality is declared, and the sample count is asserted unchanged -
    a join that invents or loses samples fails here rather than surfacing as
    an inflated event count in a hazard ratio.
    """
    # BUG FIX (both external reviews, each reproduced it): the duplicate
    # checks below do not catch NULL keys, because null keys are unique -
    # and pandas matches a null join key to another null join key, unlike
    # SQL. The function reported a 100% clinical match rate while linking a
    # missing identifier to a missing identifier. validate="many_to_one"
    # does not help for the same reason. Keys are validated first.
    smap = smap.copy()
    clinical = clinical.copy()
    smap["sample"] = validate_identifier_column(
        smap, "sample", "Expression samples")
    smap["patient"] = validate_identifier_column(
        smap, "patient", "Expression samples (patient key)")
    clinical["patient"] = validate_identifier_column(
        clinical, "patient", "Clinical records")
    if smap["sample"].duplicated().any():
        _d = sorted(smap["sample"][smap["sample"].duplicated()].unique())[:5]
        raise ValueError(
            f"Duplicate sample identifier(s) in the expression cohort, e.g. "
            f"{_d}. Resolve which record is authoritative before joining "
            f"clinical data.")
    if clinical["patient"].duplicated().any():
        _d = sorted(
            clinical["patient"][clinical["patient"].duplicated()].unique())[:5]
        raise ValueError(
            f"Duplicate patient-level clinical record(s), e.g. {_d}. A left "
            f"join on duplicated patient keys multiplies sample rows and "
            f"inflates every statistic derived from them.")
    n_before = len(smap)
    merged = smap.merge(clinical, on="patient", how="left",
                        validate="many_to_one", indicator=True)
    if len(merged) != n_before:
        raise AssertionError(
            f"The clinical merge changed the sample count: {n_before} -> "
            f"{len(merged)}. Multiple samples per patient are expected; "
            f"multiple clinical records per patient are not.")
    match_rate = float(merged["_merge"].eq("both").mean())
    print(f"  clinical join: {len(merged):,} sample(s), "
          f"{match_rate:.1%} matched to a patient record")
    return merged.drop(columns="_merge"), match_rate


def validate_endpoint_configuration(requested, bcr_file):
    """Resolve the endpoint CONTRACT before any clinical column is read.

    SELF-REVIEW + both external reviews, which each reproduced this by
    executing the function: last round I made CFG.survival_endpoint live but
    treated "BCR" as non-strict - prefer BCR, else fall back to DFS/PFS with
    a loud warning. I documented that as a deliberate "lenient reading". It
    was wrong, and my own rationale was the weak part: the field was added
    precisely so a prespecified endpoint could not be silently replaced, and
    a warning is not a refusal. Requesting BCR and receiving disease-free
    survival invalidates every Kaplan-Meier curve, hazard ratio and sentence
    that names the endpoint.

    BEHAVIOURAL CHANGE: with the default CFG.survival_endpoint="BCR", a LIVE
    run that supplies no BCR_ENDPOINT_FILE now RAISES instead of running on
    DFS with a warning. That is the intended direction - an invalid analysis
    should fail visibly - and the deliberate escape hatch is to set
    CFG.survival_endpoint="auto", which states in the configuration that any
    complete endpoint is acceptable.
    """
    requested = str(requested or "auto").strip().upper()
    if requested not in {"BCR", "DFS", "PFS", "AUTO"}:
        raise ValueError(
            f"CFG.survival_endpoint={requested!r} is not recognised. Use "
            f"'BCR', 'DFS', 'PFS' or 'auto'.")
    if requested == "BCR":
        if not bcr_file:
            raise ValueError(
                "CFG.survival_endpoint='BCR' declares biochemical "
                "recurrence, but BCR_ENDPOINT_FILE is unset. DFS and PFS are "
                "different clinical endpoints with study-specific "
                "definitions; substituting one would invalidate every "
                "survival figure and every sentence naming the endpoint. "
                "Supply a validated BCR file, or set "
                "CFG.survival_endpoint='auto' to accept the documented "
                "preference order deliberately.")
        if not Path(bcr_file).is_file():
            raise FileNotFoundError(
                f"CFG.survival_endpoint='BCR' and BCR_ENDPOINT_FILE="
                f"{bcr_file!r}, but that path is not a file.")
    elif requested in {"DFS", "PFS"} and bcr_file:
        raise ValueError(
            f"Conflicting endpoint configuration: "
            f"CFG.survival_endpoint={requested!r} but BCR_ENDPOINT_FILE="
            f"{bcr_file!r} is also set. The BCR branch would take precedence "
            f"and silently override the declared endpoint. Choose one.")
    return requested


ENDPOINT_BY_NAME = {
    "DFS": ("DFS_MONTHS", "DFS_STATUS", "disease_free_survival"),
    "PFS": ("PFS_MONTHS", "PFS_STATUS", "progression_free_survival"),
}


def select_survival_endpoint(cl, pairs=SURVIVAL_ENDPOINT_PAIRS,
                             requested=None):
    """Choose a survival endpoint as a MATCHED time/status pair.

    Returns (time_col, event_col, endpoint_name), or None when the file
    carries no complete pair. Raises when a pair is HALF present, because
    that is the case the previous independent-substring selection silently
    recombined into an endpoint that exists in no source document.

    Column names are matched by substring on the upper-cased headers, as
    before, but a time column is only ever paired with the status column of
    the SAME endpoint.
    """
    cols = [str(c) for c in cl.columns]
    # SELF-REVIEW / external review ("make the requested endpoint explicit"):
    # CFG.survival_endpoint existed but was read nowhere, so the declared
    # endpoint and the selected one could differ silently. A NAMED endpoint
    # is now a requirement: the pipeline supplies that pair or refuses,
    # rather than substituting the other composite. "BCR" and "auto" keep the
    # documented preference order ("BCR" because a validated BCR file is
    # handled by the caller before this function is reached).
    # SELF-REVIEW: defaulting `requested` to CFG.survival_endpoint made a
    # bare select_survival_endpoint(cl) RAISE under the default config
    # ("BCR"), because this function legitimately refuses BCR. The CFG
    # default belongs to the loader, which passes it explicitly after
    # validate_endpoint_configuration() has resolved the contract; this
    # helper's own default is the only thing it can always honour.
    if requested is None:
        requested = "AUTO"
    requested = str(requested or "AUTO").strip().upper()
    # "BCR" never reaches this function as a fallback any more:
    # validate_endpoint_configuration() has already refused a BCR request
    # without a BCR file, and a BCR request WITH one is served by the
    # external-BCR branch before the selector is called. So only AUTO may
    # choose between endpoints; DFS and PFS are strict.
    # BUG FIX (both external reviews, correct): this helper tolerated
    # requested="BCR" and returned DFS. I had argued the case could not
    # arise because validate_endpoint_configuration() runs first in the
    # loader - but a helper that permits what the contract forbids is one
    # direct call away from the original defect, and a researcher reusing
    # the selector would get the wrong endpoint with no warning. The
    # selector now refuses the endpoint it cannot select, so the contract
    # holds at every entry point rather than only on the validated path.
    if requested == "BCR":
        raise ValueError(
            "select_survival_endpoint() cannot select biochemical "
            "recurrence: BCR is supplied as an external endpoint file and "
            "must be loaded through validate_external_bcr(). Requesting it "
            "here previously returned DFS or PFS - a different clinical "
            "endpoint. Use 'DFS', 'PFS' or 'auto'.")
    strict_request = requested in ENDPOINT_BY_NAME
    if strict_request:
        pairs = (ENDPOINT_BY_NAME[requested],)
    elif requested != "AUTO":
        raise ValueError(
            f"CFG.survival_endpoint={requested!r} is not recognised. Use "
            f"'BCR', 'DFS', 'PFS' or 'auto'.")

    def _find(key):
        # SELF-REVIEW: a substring match returning the FIRST hit chooses
        # silently between candidates. A cBioPortal release carrying both
        # DFS_MONTHS and DFS_MONTHS_IMPUTED would hand the analysis whichever
        # happens to come first in the header, and the run would name the
        # endpoint identically either way. An exact match is unambiguous and
        # is preferred; otherwise several candidates is an error, not a
        # coin toss.
        hits = [c for c in cols if key in c]
        if key in hits:
            return key
        if len(hits) > 1:
            raise ValueError(
                f"Ambiguous survival column: {len(hits)} columns contain "
                f"{key!r} ({hits}) and none matches it exactly. Which one "
                f"carries the endpoint cannot be inferred from the header - "
                f"rename the intended column to {key} or supply "
                f"BCR_ENDPOINT_FILE.")
        return hits[0] if hits else None

    # SELF-REVIEW of this round's own fix: preferring a later COMPLETE pair
    # over raising on an earlier HALF-present one is right - a complete PFS
    # pair is a well-defined endpoint - but doing it silently reintroduces
    # the very failure this function exists to stop. Someone who supplied
    # DFS_MONTHS expecting disease-free survival would receive
    # progression-free survival with no notice. The substitution is allowed;
    # it is not allowed to be quiet.
    half_present = []
    for time_key, event_key, name in pairs:
        t, e = _find(time_key), _find(event_key)
        if t and e:
            if half_present:
                _earlier = ", ".join(
                    f"{nm} (only {tk if tt else ek} present)"
                    for nm, tk, tt, ek, _ee in half_present)
                print(f"  NOTE: an earlier endpoint was INCOMPLETE and has "
                      f"been skipped: {_earlier}. Using {name} "
                      f"({t} + {e}) instead. If you intended the earlier "
                      f"endpoint, supply its missing column - this run's "
                      f"figures and Methods must name {name}.")
            return t, e, name
        if t or e:
            half_present.append((name, time_key, t, event_key, e))
        elif strict_request:
            # SELF-REVIEW of this round's own fix, caught by its own test:
            # a NAMED endpoint that is entirely absent fell through to
            # `return None`, which the caller treats as "no endpoint
            # available" and continues with an empty survival analysis. For
            # an explicit request that is the substitution the field exists
            # to prevent - absence of the requested endpoint is an error,
            # not a silent downgrade.
            raise ValueError(
                f"CFG.survival_endpoint={requested!r} was requested but "
                f"neither {time_key} nor {event_key} is present in this "
                f"clinical file (columns: {cols[:12]}...). A prespecified "
                f"endpoint that the data cannot supply must stop the "
                f"analysis rather than fall back to another composite. "
                f"Supply the endpoint, or set CFG.survival_endpoint='auto' "
                f"to accept the documented preference order.")
    if half_present:
        detail = "; ".join(
            f"{nm}: {tk}={'present' if t else 'MISSING'}, "
            f"{ek}={'present' if e else 'MISSING'}"
            for nm, tk, t, ek, e in half_present)
        raise ValueError(
            f"Incomplete survival endpoint pair(s) and no complete "
            f"alternative: {detail}. Pairing a follow-up time with another "
            f"endpoint's event status produces a survival outcome that was "
            f"never defined in the source data. Supply the matching column "
            f"or set BCR_ENDPOINT_FILE to a validated endpoint.")
    return None


def validate_external_bcr(bcr_file, cohort_patients):
    """Read and VALIDATE an externally supplied biochemical-recurrence file.

    Factored out of load_clinical_live() (self-review): the validation lived
    inline inside a loader whose first statement downloads a TCGA archive, so
    it could not be executed in a test. The regression suite therefore
    asserted the validation's presence by SEARCHING THE LOADER'S SOURCE TEXT
    for phrases like "duplicate patient" - a check that passes if the raise
    is deleted and a comment mentioning it survives. Behaviour that matters
    this much needs a test that can actually run it.

    Returns (bcr_frame, n_matched). Raises on any condition that would let a
    malformed endpoint file enter the survival analysis silently.
    """
    bcr = pd.read_csv(bcr_file)
    kc = next(c for c in bcr.columns
              if "patient" in c.lower() or "barcode" in c.lower())
    tc = next(c for c in bcr.columns
              if "time" in c.lower() or "month" in c.lower())
    ec = next(c for c in bcr.columns
              if "event" in c.lower() or "status" in c.lower())
    bcr = bcr.rename(columns={kc: "patient", tc: "surv_time",
                              ec: "surv_event"})
    bcr["patient"] = bcr.patient.astype(str).str[:12]
    # Externally supplied data get the SAME explicit code decoding as the
    # built-in DFS/PFS path; previously the stricter endpoint had the weaker
    # validation and free text entered the analysis unchallenged.
    bcr["surv_event"] = parse_event_status(
        bcr["surv_event"], f"{Path(bcr_file).name}:surv_event")
    # BUG FIX (both external reviews, each reproduced by executing this
    # function, and confirmed here: 2/2 IDs matched, 100% "coverage", 0/2
    # usable times, ACCEPTED). Two distinct defects:
    #
    #  (a) errors="coerce" turned non-numeric text into NaN, and the negative
    #      check then skipped NaN, so follow-up times of "unknown" and
    #      "missing" passed validation entirely. This is the SAME defect
    #      class I fixed in the raw-count validator last round - missing
    #      values excluded from validation rather than rejected - left
    #      unfixed one module over. A malformed time is a source-data error,
    #      not a missing observation.
    #
    #  (b) coverage measured PATIENT-ID MATCHING, not usable survival
    #      observations, so an endpoint file could report full coverage while
    #      containing no analysable follow-up at all. The four quantities the
    #      review asks for are now reported separately, and the gate is
    #      applied to the usable rate.
    # SELF-REVIEW: an inline copy of the malformed/infinite/negative/event
    # checks stood here. I wrote it in the SAME round that I created
    # validate_survival_fields() as the single shared contract - so the
    # round whose purpose was "one validated input contract" shipped two
    # implementations of it, with the BCR path using its own. The two did
    # not differ yet, but divergence between two copies of one rule is the
    # single most recurrent defect in this codebase: the tissue aliases, the
    # detection floor, the mask semantics, the join-cardinality guards and
    # the NaN policy each reached production as a rule applied in one place
    # and not the other. One implementation, called from both paths.
    bcr["surv_time"] = validate_survival_fields(
        bcr["surv_time"], bcr["surv_event"], Path(bcr_file).name)
    n_dup_pat = int(bcr.patient.duplicated().sum())
    if n_dup_pat:
        raise ValueError(
            f"{Path(bcr_file).name}: {n_dup_pat} duplicate patient "
            f"identifier(s), e.g. "
            f"{sorted(bcr.patient[bcr.patient.duplicated()].unique())[:5]}. "
            f"A left merge on duplicated keys silently MULTIPLIES rows, "
            f"inflating the event count and every statistic derived from "
            f"it. Resolve which record is authoritative in the source file.")
    n_matched = int(pd.Series(cohort_patients).astype(str)
                    .isin(set(bcr.patient)).sum())
    if n_matched == 0:
        raise ValueError(
            f"{Path(bcr_file).name}: no patient identifier matched the "
            f"expression cohort. A silent left merge would produce an "
            f"all-missing endpoint and an empty survival analysis. Check "
            f"the identifier format (expected a 12-character TCGA "
            f"barcode prefix).")
    # NEW (external review): a ZERO-overlap join already raises, but a very
    # small non-zero overlap proceeded silently. A survival analysis on 4 of
    # 500 patients is not the analysis the Methods would describe, and the
    # figure caption would not say so. The acceptable minimum is
    # prespecified in CFG rather than judged after seeing the result.
    # The four quality-control measures the review asks for, reported
    # separately. The GATE is the usable rate: a high identifier match rate
    # is not evidence of an analysable endpoint.
    _cohort = pd.Series(cohort_patients).astype(str).drop_duplicates()
    _n_cohort = max(len(_cohort), 1)
    # BUG FIX (external review, newly identified): usability was defined
    # here as a non-missing time, while every model filters surv_time > 0.
    # Zero-duration records therefore passed this gate at 100% and were then
    # dropped by the analysis. One rule, shared with the models.
    _usable_ids = set(bcr.loc[survival_eligibility_mask(bcr), "patient"])
    _n_usable = int(_cohort.isin(_usable_ids).sum())
    _id_rate = n_matched / _n_cohort
    # SELF-REVIEW, caught by this round's own single-implementation
    # invariant: my first version of this line re-coerced surv_time, adding
    # a SECOND parsing site in the very round whose purpose was to collapse
    # them to one. validate_survival_fields() has already returned the
    # parsed column, so it is used as-is.
    _time_rate = float((bcr.surv_time > 0).mean()) if len(bcr) else 0.0
    _event_rate = (float(bcr.surv_event.isin([0, 1]).mean())
                   if len(bcr) else 0.0)
    _usable_rate = _n_usable / _n_cohort
    print(f"  BCR endpoint QC: patient-ID match {n_matched}/{_n_cohort} "
          f"({_id_rate:.1%}) | valid follow-up {_time_rate:.1%} | "
          f"valid event codes {_event_rate:.1%} | USABLE (time AND event) "
          f"{_n_usable}/{_n_cohort} ({_usable_rate:.1%})")
    if _usable_rate < CFG.min_bcr_match_rate:
        raise ValueError(
            f"{Path(bcr_file).name}: only {_n_usable} of {_n_cohort} cohort "
            f"patients ({_usable_rate:.1%}) have BOTH a valid follow-up time "
            f"and a decoded event indicator, below the prespecified minimum "
            f"of {CFG.min_bcr_match_rate:.0%}. The patient-identifier match "
            f"rate is {_id_rate:.1%}, which is why identifier coverage must "
            f"not be reported as endpoint coverage. Every survival statistic "
            f"would describe that small subset while the Methods described "
            f"the cohort.")
    return bcr, n_matched


# Approximate compressed download sizes, used only to warn about disk before
# a LIVE run starts. Deliberately generous; these are planning figures, not
# measurements, and the code never fails a run on the basis of them alone.
_LIVE_DOWNLOAD_BUDGET_GB = 6.0

# (The per-source size tables that sat here are now fields of DATA_INPUTS,
# so the preflight and the loaders read one declaration.)


def preflight_resources(verbose=True, require_disk=True):
    """Check free disk and report RAM BEFORE a multi-gigabyte LIVE download.

    NEW (both external reviews asked for this). A LIVE run acquired several
    gigabytes with no idea whether the instance could hold them: the failure
    mode was a disk-full error part-way through a download, twenty minutes
    in, with a partial file left behind. Checking first costs microseconds.

    Disk is checked and can block. RAM is REPORTED, not enforced: the peak
    memory of this pipeline on real data has never been measured, so any
    threshold here would be a number I invented, and failing a run against
    an invented number is worse than letting it run and watching it.
    """
    import shutil as _sh
    free_gb = _sh.disk_usage(str(DIRS["raw"])).free / 1e9
    # BUG FIX (external review, correct, and I introduced this last round):
    # the budget was a flat ~6 GB demanded of every LIVE run regardless of
    # what was already on disk. The second run - the warm one, with every
    # source downloaded and verified and nothing left to fetch - was refused
    # for lacking room it did not need. That is the opposite of the intent:
    # the check exists so a first download does not die half-way, and it was
    # blocking exactly the fast path this work was meant to enable.
    #
    # The requirement is now computed from what is actually MISSING. A file
    # that passes validate_cached_input() needs no space at all.
    _needed_gb, _have, _missing = 0.0, [], []
    for _k in DATA_INPUTS:
        _dest = input_path(_k)
        _min = input_min(_k)
        _ok_c, _ = validate_cached_input(_dest, _min)
        if _ok_c:
            _have.append(_k)
        else:
            _missing.append(_k)
            _needed_gb += DATA_INPUTS[_k].get("approx_gb", 0.5)
    # Headroom for extraction and the filtered-matrix cache, only when
    # something still has to be fetched or re-extracted.
    _scratch_gb = 2.0 if _missing else 0.5
    ram_txt = "unknown (psutil not installed)"
    try:
        import psutil as _ps
        ram_txt = f"{_ps.virtual_memory().available / 1e9:.1f} GB available"
    except Exception:
        try:
            # Linux without psutil: read MemAvailable directly rather than
            # leaving the figure unreported.
            with open("/proc/meminfo") as _fh:
                for _line in _fh:
                    if _line.startswith("MemAvailable:"):
                        ram_txt = (f"{int(_line.split()[1]) / 1e6:.1f} GB "
                                   f"available")
                        break
        except Exception:
            pass
    _required_gb = _needed_gb + _scratch_gb
    # BUG FIX (both external reviews, correct - and the situation is one I
    # created). Round 26 deliberately split the volumes for speed: raw inputs
    # persist on Drive, everything computed goes to fast local /content. The
    # preflight then checked only DIRS["raw"], so on the intended Colab
    # layout it reported on Drive's free space and said nothing whatever
    # about the local disk that every figure, table, model and the release
    # ZIP are written to. Drive having 50 GB free tells you nothing about
    # /content having room for the outputs.
    _out_dir = DIRS.get("logs", DIRS["raw"])
    try:
        _same_volume = (Path(_out_dir).stat().st_dev
                        == Path(DIRS["raw"]).stat().st_dev)
    except Exception:
        _same_volume = True
    _out_free_gb = _sh.disk_usage(str(_out_dir)).free / 1e9
    # Outputs are small next to the inputs: the archived DEMO release is
    # ~8 MB and a manuscript run's figures and tables are of that order. The
    # budget is for the uncompressed intermediates and the ZIP built beside
    # them, with headroom.
    _out_required_gb = 2.0
    if verbose:
        print(f"  resources: {free_gb:.1f} GB free on "
              f"{DIRS['raw']}; RAM {ram_txt}; "
              f"{os.cpu_count() or '?'} CPU(s); "
              f"BLAS threads {os.environ.get('OMP_NUM_THREADS', 'default')}; "
              f"joblib workers {N_JOBS}")
        if _missing:
            print(f"  {len(_have)}/{len(DATA_INPUTS)} sources already cached "
                  f"and validated; {len(_missing)} to fetch "
                  f"({', '.join(sorted(_missing))}) needing about "
                  f"{_required_gb:.1f} GB including scratch")
        else:
            print(f"  all {len(_have)} sources already cached and validated: "
                  f"no downloads needed, {_required_gb:.1f} GB of scratch "
                  f"headroom wanted")
    if verbose and not _same_volume:
        print(f"  outputs are on a SEPARATE volume: "
              f"{_out_free_gb:.1f} GB free on {_out_dir}, "
              f"{_out_required_gb:.1f} GB wanted")
    if require_disk and not _same_volume and _out_free_gb < _out_required_gb:
        raise RuntimeError(
            f"Only {_out_free_gb:.1f} GB free on the OUTPUT volume "
            f"({_out_dir}), and this run needs about "
            f"{_out_required_gb:.1f} GB there for intermediates, figures, "
            f"tables and the release archive.\n"
            f"  The raw-input volume has {free_gb:.1f} GB free, which is a "
            f"different filesystem and does not help. On Colab this is the "
            f"usual split: inputs cached on Drive, outputs on local "
            f"/content.\n"
            f"  Free space under {_out_dir}, or point SURFACEOME_OUT_DIR at "
            f"a volume with room.")
    if require_disk and free_gb < _required_gb:
        raise RuntimeError(
            f"Only {free_gb:.1f} GB free on {DIRS['raw']}, and this run "
            f"needs about {_required_gb:.1f} GB "
            f"({_needed_gb:.1f} GB to fetch "
            f"{len(_missing)} missing source(s), {_scratch_gb:.1f} GB "
            f"scratch). Stopping now rather than failing part-way through a "
            f"download and leaving a partial file behind.\n"
            f"  Already-validated sources are reused, so freeing space and "
            f"re-running resumes rather than restarting"
            + (f" ({len(_have)} of {len(DATA_INPUTS)} are already in place)."
               if _have else ".") + "\n"
            f"  Options: free space; point SURFACEOME_RAW_DIR at a larger "
            f"volume (on Colab, a mounted Drive folder); or use a runtime "
            f"with more disk.")
    return {"free_disk_gb": round(free_gb, 2), "ram": ram_txt,
            "n_cpu": os.cpu_count(), "n_jobs": N_JOBS,
            "sources_cached": len(_have), "sources_missing": len(_missing),
            "required_gb": round(_required_gb, 2),
            "output_free_gb": round(_out_free_gb, 2),
            "output_required_gb": _out_required_gb,
            "output_same_volume": bool(_same_volume)}


def validate_live_configuration(verbose=True):
    """Check everything that can be checked WITHOUT the network, first.

    BUG FIX (all three external reviews, correct). The endpoint and ancestry
    contracts were validated at the top of load_clinical_live(), which was
    already an improvement on validating them after the TCGA tarball was
    parsed - but load_clinical_live() is the FOURTH acquisition step. A LIVE
    run with an invalid endpoint configuration still downloaded the Toil
    expression matrix, the SU2C archive and the HPA table - several
    gigabytes and many minutes on a Colab instance - before failing on a
    contract that costs microseconds to check.

    Everything here is a pure configuration check. It is called as the first
    statement of the LIVE path, before any network access.

    Returns the resolved endpoint request; raises on any invalid contract.
    """
    # BUG FIX (both external reviews, correct, and it blocked the very
    # workflow I added last round for them). The strict BCR contract is
    # right for a REPORTABLE analysis: substituting DFS or PFS for BCR
    # would invalidate every survival figure and every sentence naming the
    # endpoint. But it was enforced unconditionally, before the data_only
    # branch was reached - so the recommended first Colab diagnostic
    # ("acquire and validate the real data, fit nothing") could not run on
    # untouched defaults, because CFG.survival_endpoint defaults to "BCR"
    # and no BCR file exists yet. Acquisition was gated on a contract that
    # only the modelling needs.
    #
    # A data_only run therefore RESOLVES the endpoint permissively, reports
    # which one it found, and is barred from producing reportable output by
    # REPORTABLE=False. The strict contract is untouched for analysis runs.
    # Nothing is silently substituted: the diagnostic says what it used.
    _requested = getattr(CFG, "survival_endpoint", "auto")
    if not REPORTABLE and _requested.upper() == "BCR" \
            and not os.environ.get("BCR_ENDPOINT_FILE", ""):
        print("  data_only: CFG.survival_endpoint='BCR' needs a "
              "BCR_ENDPOINT_FILE, which is not set. This diagnostic run "
              "will INSPECT whatever clinical endpoint fields the cohort "
              "actually provides instead.")
        print("  No BCR result is produced, claimed, or reportable from "
              "this run. A reportable BCR analysis still requires the file.")
        endpoint = validate_endpoint_configuration(
            "auto", os.environ.get("BCR_ENDPOINT_FILE", ""))
    else:
        endpoint = validate_endpoint_configuration(
            _requested, os.environ.get("BCR_ENDPOINT_FILE", ""))
    validate_ancestry_request(CFG, os.environ.get("ANCESTRY_FILE", ""))
    # The profile sets bags, folds and permutations - all of which are
    # modelling parameters. A data_only run fits nothing, so refusing it on
    # profile grounds would block the diagnostic for a reason that cannot
    # apply to it. REPORTABLE is the correct condition here, not the profile.
    if RUN_MODE == "LIVE" and PROFILE == "smoke" and REPORTABLE \
            and not ALLOW_SMOKE_LIVE:
        raise RuntimeError(
            "LIVE run requested with PROFILE='smoke'. The smoke profile is a "
            "software diagnostic: it uses 3 PU bags, 3x2 cross-validation "
            "and 30 permutations, which is far too little to support any "
            "reported effect size or confidence interval. Real data deserve "
            "at least PROFILE='standard'; use 'manuscript' for the run you "
            "report. Set ALLOW_SMOKE_LIVE=1 only to shake down the LIVE "
            "loaders, and do not report anything from such a run.")
    # BUG FIX (external review, correct and important). load_proteomics_live()
    # raises FileNotFoundError when PROTEOMICS_LOCAL_FILE is unset - and it is
    # called in the REPORTABLE path only, after every download, the whole
    # harmonisation and the feature build. So the recommended sequence
    # (data_only succeeds, switch to analysis) crashed after all the
    # acquisition work, on a condition knowable before the first byte moved.
    # data_only stops before that stage, so a clean diagnostic run gave no
    # warning at all. This is the fail-late pattern removed everywhere else
    # in this file, surviving in the one input the pipeline cannot download.
    #
    # No implicit bypass: proteomics is a declared input of the reportable
    # analysis, and quietly continuing without it would change what the study
    # measured while the outputs still read as complete. A transcript-only
    # study is a scientific decision to be taken deliberately, not a
    # side-effect of a missing file.
    if REPORTABLE and RUN_MODE == "LIVE":
        _pf = os.environ.get("PROTEOMICS_LOCAL_FILE", "")
        if not _pf or not Path(_pf).is_file():
            raise FileNotFoundError(
                "A reportable LIVE analysis needs the prostate proteomics "
                "file, and PROTEOMICS_LOCAL_FILE is "
                + (f"set to {_pf!r}, which does not exist."
                   if _pf else "unset.") + "\n"
                "  This is checked HERE, before any download, because the "
                "loader that needs it runs after several gigabytes of "
                "acquisition and the whole feature build.\n"
                "  Supply either (a) a gene x sample abundance matrix plus "
                "PROTEOMICS_SAMPLE_ANNOTATION and PROTEOMICS_SCALE, or "
                "(b) a precomputed table with columns gene, prot_log2fc, "
                "prot_pval. The prostate proteome is at the NCI Proteomic "
                "Data Commons (pdc.cancer.gov) or in the corresponding "
                "paper's supplementary tables.\n"
                "  To validate the real cohorts WITHOUT this file, run "
                "RUN_PURPOSE=data_only, which stops before the proteomics "
                "stage and produces nothing reportable.")
        print(f"  proteomics input present: {Path(_pf).name} "
              f"({Path(_pf).stat().st_size/1e6:.2f} MB)")
    if verbose:
        print(f"  configuration validated before network access: "
              f"endpoint={endpoint}, profile={PROFILE}, "
              f"purpose={RUN_PURPOSE}")
        preflight_resources(verbose=True, require_disk=True)
    return endpoint


def load_clinical_live(samples):
    # BUG FIX (both external reviews, correct): the endpoint and ancestry
    # contracts were validated AFTER the TCGA archive was downloaded and
    # parsed, so an invalid configuration wasted the download and failed
    # late. Configuration is cheap to check and is checked first.
    bcr_file = os.environ.get("BCR_ENDPOINT_FILE", "")
    ancestry_file = os.environ.get("ANCESTRY_FILE", "")
    # Re-checked here as well as in validate_live_configuration(): this
    # function is reachable on its own from the LIVE-path tests, and a
    # contract that is only enforced by its caller is not enforced.
    endpoint_request = validate_live_configuration(verbose=False)
    tar_f = download(DATA_URLS["tcga_prad"], input_path("tcga_prad"),
                    input_min("tcga_prad"))
    ex = extract_archive_once(tar_f, DIRS["raw"] / "prad_tcga_extracted",
                              "TCGA-PRAD archive")
    pf = next(iter(ex.rglob("data_clinical_patient.txt")))
    cl = pd.read_csv(pf, sep="\t", comment="#", low_memory=False)
    cl.columns = [c.strip().upper() for c in cl.columns]

    def pick(*keys):
        for k in keys:
            for c in cl.columns:
                if k in c:
                    return c
        return None

    # ----------------------------------------------------------------
    # ENDPOINT PROVENANCE - do not call this biochemical recurrence
    # ----------------------------------------------------------------
    # BUG FIX (raised in external review): an earlier version read
    # DFS_MONTHS / PFS_MONTHS straight into `surv_time` / `surv_event` and
    # every figure, table and sentence downstream then said "biochemical
    # recurrence-free survival".
    #
    # They are not the same endpoint. Disease-free and progression-free
    # survival are composite endpoints whose definitions vary by study and
    # are not necessarily PSA-based recurrence after prostatectomy. A PSA
    # rise, a radiographic progression and a death are different events with
    # different clinical meaning and different time origins.
    #
    # The loader now RECORDS which column it used and labels the endpoint
    # accordingly. If you want a true BCR endpoint, supply it via
    # BCR_ENDPOINT_FILE with a documented definition and time origin.
    pid = pick("PATIENT_ID")
    endpoint_col = endpoint_name = None

    out = pd.DataFrame({"patient": cl[pid].astype(str)})
    # BUG FIX (both external reviews, confirmed): a REQUESTED but missing BCR
    # file fell through to the DFS/PFS branch silently, so an analysis asked
    # for biochemical recurrence could quietly run on a different composite
    # endpoint and label every figure accordingly. Asking for BCR and getting
    # PFS without being told is worse than a crash.
    if bcr_file and not Path(bcr_file).exists():
        raise FileNotFoundError(
            f"BCR_ENDPOINT_FILE was set to {bcr_file!r} but that path does "
            f"not exist. Falling back to DFS/PFS would silently substitute a "
            f"different clinical endpoint for the one requested. Fix the path "
            f"or unset BCR_ENDPOINT_FILE to use DFS/PFS deliberately.")
    if bcr_file and Path(bcr_file).exists():
        register_input_file(bcr_file, role="biochemical-recurrence endpoint")
        out["patient"] = out.patient.str[:12]
        bcr, n_matched = validate_external_bcr(bcr_file, out["patient"])
        print(f"  BCR endpoint validated: {n_matched} patient(s) matched, "
              f"event codes decoded explicitly, no duplicate keys")
        # SELF-REVIEW: last round I added a clinical-side duplicate check and
        # validate="one_to_one" to the ANCESTRY merge and left this one - its
        # immediate sibling, three lines of the same function - unguarded.
        # validate_external_bcr() checks the ENDPOINT file for duplicate
        # keys, but nothing checked the clinical table, and a left merge on a
        # duplicated clinical key multiplies patient rows exactly as it would
        # on a duplicated endpoint key. It also silently inflates the
        # denominator of the coverage rate computed just above. Same policy,
        # both sides, asserted by pandas as well.
        if int(out.patient.duplicated().sum()):
            raise ValueError(
                f"The clinical table has "
                f"{int(out.patient.duplicated().sum())} duplicate patient "
                f"identifier(s) before the endpoint merge, e.g. "
                f"{sorted(out.patient[out.patient.duplicated()].unique())[:5]}. "
                f"The join would multiply patient rows and inflate every "
                f"survival statistic. This indicates a defect upstream in "
                f"clinical loading.")
        out = out.merge(bcr[["patient", "surv_time", "surv_event"]],
                        on="patient", how="left", validate="one_to_one")
        endpoint_name = "biochemical_recurrence"
        endpoint_col = Path(bcr_file).name
        print(f"  endpoint: TRUE BCR supplied via {endpoint_col}")
    else:
        # BUG FIX (both external reviews, independently correct): time and
        # event status were picked by INDEPENDENT substring searches, so a
        # clinical file carrying DFS_MONTHS and PFS_STATUS but no DFS_STATUS
        # yielded disease-free follow-up TIME paired with progression-free
        # EVENT indicators - a survival outcome that exists in no source
        # document, labelled "disease_free_survival" because the time column
        # happened to say DFS. Every hazard ratio downstream would describe a
        # composite nobody defined.
        #
        # Endpoints are now selected as MATCHED PAIRS, in a declared order of
        # preference, and a file offering only half a pair is an error rather
        # than a silent recombination.
        sel = select_survival_endpoint(cl, requested=endpoint_request)
        if sel is None:
            endpoint_col, endpoint_name = "none", "unavailable"
            out["surv_time"] = np.nan
            out["surv_event"] = np.nan
            print("  endpoint: UNAVAILABLE - no complete time/status pair")
        else:
            time_col, event_col, endpoint_name = sel
            endpoint_col = f"{time_col} + {event_col}"
            out["surv_event"] = parse_event_status(cl[event_col], event_col)
            # Same validation as the external BCR path: a non-numeric
            # follow-up time in a cBioPortal column is a source-data error,
            # not a censored observation.
            out["surv_time"] = validate_survival_fields(
                cl[time_col], out["surv_event"],
                f"{endpoint_name} ({time_col} + {event_col})")
        print(f"  endpoint: {endpoint_name} (from column {endpoint_col})")
        # NOTE: an "endpoint mismatch" warning stood here for the case
        # CFG.survival_endpoint="BCR" with no BCR file. It is now
        # unreachable - validate_endpoint_configuration() refuses that
        # combination before any column is read - so the dead branch is
        # removed rather than left to suggest a fallback that no longer
        # exists. Reaching this point means the configuration asked for
        # "auto".
        print("  WARNING: this is NOT biochemical recurrence. DFS and PFS are")
        print("           composite endpoints with study-specific definitions.")
        print("           Every figure, table and manuscript sentence must name")
        print("           the endpoint actually used. Set BCR_ENDPOINT_FILE to")
        print("           supply a validated BCR endpoint instead.")
    out["endpoint_name"] = endpoint_name
    out["endpoint_source_column"] = endpoint_col
    for src, dst in [("RACE", "race_reported"), ("GLEASON", "gleason"),
                     ("AJCC_PATHOLOGIC_TUMOR_STAGE", "stage_t"),
                     ("PSA", "psa"), ("AGE", "age")]:
        c = pick(src)
        out[dst] = cl[c].values if c else np.nan

    print("  NOTE: cBioPortal ships SELF-REPORTED RACE, not genetic ancestry.\n"
          "        To make an ancestry claim, merge the TCGA genetic-ancestry\n"
          "        annotation (Carrot-Zhang et al. 2020) on patient barcode and\n"
          "        set ANCESTRY_FILE. Falling back to self-reported race and\n"
          "        labelling the variable honestly.")
    af = ancestry_file
    # BUG FIX (both external reviews, correct - and the unfixed sibling of
    # the BCR_ENDPOINT_FILE defect already corrected above): a REQUESTED but
    # missing ancestry file fell through to the self-reported-race fallback.
    # The fallback itself is honest - the variable is relabelled - but an
    # analysis that prespecified GENETIC ancestry would then run on
    # self-reported race and report an ancestry-stratified result. Asking for
    # ancestry and silently getting race is the one substitution this
    # pipeline must never make quietly. Same policy, same reason, same
    # failure mode: a requested input that is absent is an error.
    if af and not Path(af).exists():
        raise FileNotFoundError(
            f"ANCESTRY_FILE was set to {af!r} but that path does not exist. "
            f"Falling back to self-reported race would substitute a "
            f"different variable for the one requested and every "
            f"ancestry-stratified result would be mislabelled. Fix the path "
            f"or unset ANCESTRY_FILE to use self-reported race "
            f"deliberately.")
    if af and Path(af).exists():
        register_input_file(af, role="genetic ancestry annotation")
        anc = pd.read_csv(af)
        kc = next(c for c in anc.columns if "patient" in c.lower() or "barcode" in c.lower())
        vc = next(c for c in anc.columns if "ancestry" in c.lower() or "consensus" in c.lower())
        anc = anc.rename(columns={kc: "patient", vc: "ancestry"})
        anc["patient"] = anc.patient.astype(str).str[:12]
        out["patient"] = out.patient.str[:12]
        # BUG FIX (external review, correct - the third branch of the same
        # join-cardinality defect, after BCR and the expression merge): the
        # BCR path rejects duplicate patient keys because a left merge on
        # duplicated keys MULTIPLIES rows, but the ancestry path merged
        # unchecked. Two ancestry records for one patient would silently
        # duplicate that patient in every subsequent survival or subgroup
        # analysis, inflating the event count and narrowing every confidence
        # interval. Validated explicitly, with the cardinality asserted by
        # pandas as well, and the match rate reported so a near-empty
        # annotation cannot pass for a successful merge.
        _n_dup_anc = int(anc.patient.duplicated().sum())
        if _n_dup_anc:
            raise ValueError(
                f"{Path(af).name}: {_n_dup_anc} duplicate patient "
                f"identifier(s), e.g. "
                f"{sorted(anc.patient[anc.patient.duplicated()].unique())[:5]}. "
                f"A left merge on duplicated keys multiplies patient rows "
                f"and inflates every statistic derived from them. Resolve "
                f"which record is authoritative in the source file.")
        if int(out.patient.duplicated().sum()):
            raise ValueError(
                f"The clinical table has duplicate patient identifiers "
                f"before the ancestry merge; the join would be ambiguous. "
                f"This indicates a defect upstream in clinical loading.")
        _n_anc_matched = int(out.patient.isin(set(anc.patient)).sum())
        _anc_rate = _n_anc_matched / max(len(out), 1)
        out = out.merge(anc[["patient", "ancestry"]], on="patient",
                        how="left", validate="one_to_one")
        print(f"  genetic ancestry merged: {_n_anc_matched}/{len(out)} "
              f"patient(s) annotated ({_anc_rate:.1%})")
        if (_anc_rate < CFG.min_ancestry_match_rate
                and getattr(CFG, "ancestry_inference_prespecified", False)):
            raise ValueError(
                f"CFG.ancestry_inference_prespecified is True, but only "
                f"{_anc_rate:.1%} of the cohort carries a genetic-ancestry "
                f"annotation (prespecified minimum "
                f"{CFG.min_ancestry_match_rate:.0%}). A prespecified "
                f"ancestry-stratified inference on this coverage is a "
                f"different analysis from the one declared. Either supply a "
                f"more complete annotation, or set "
                f"ancestry_inference_prespecified=False to report the "
                f"breakdown as exploratory with its reduced sample size.")
        if _anc_rate < CFG.min_ancestry_match_rate:
            print(f"  EXPLORATORY ONLY: only {_anc_rate:.1%} of the cohort "
                  f"carries a "
                  f"genetic-ancestry annotation (below the prespecified "
                  f"{CFG.min_ancestry_match_rate:.0%}). An "
                  f"ancestry-stratified analysis on this coverage would "
                  f"describe a small, possibly unrepresentative subset - "
                  f"report the annotated fraction with any such result.")
        out["ancestry_source"] = "genetic"
    else:
        out["ancestry"] = out["race_reported"]
        out["ancestry_source"] = "self_reported_race"

    # Map tumour barcodes (TCGA-XX-XXXX-01) back to patient barcodes.
    # BUG FIX (both external reviews): the collision/format policy existed
    # in a helper that production never called - the loader truncated
    # barcodes with a bare .str[:12]. The helper is now applied at the
    # boundary where the conversion actually happens, so a malformed barcode
    # is caught here rather than surfacing as a mysteriously unmatched
    # patient later.
    smap = pd.DataFrame({"sample": samples})
    smap["sample"] = validate_identifier_column(
        smap, "sample", "Expression samples")
    # TCGA-specific parsing at the TCGA boundary: the format is validated,
    # not merely the length, and the sample-type code is read from the field
    # the specification defines rather than re-derived downstream.
    _tcga = parse_tcga_barcodes(smap["sample"], "Expression sample barcodes")
    smap["patient"] = _tcga["patient"].to_numpy()
    smap["sample_type"] = _tcga["sample_type"].to_numpy()
    out, _clin_match_rate = merge_clinical_to_samples(smap, out)
    return out


def load_clinical_demo(samples):
    """Synthetic BCR follow-up with realistic event rates and censoring.

    TCGA-PRAD's real BCR event count is low (~90 events in ~430 informative
    patients). The demo reproduces that scarcity on purpose: it is the binding
    constraint on every survival claim in this study.
    """
    rng = np.random.default_rng(CFG.seed + 5)
    n = len(samples)
    gleason = rng.choice([6, 7, 8, 9, 10], n, p=[.12, .50, .16, .18, .04])
    stage = rng.choice(["T2a", "T2b", "T2c", "T3a", "T3b", "T4"], n,
                       p=[.14, .10, .28, .30, .16, .02])
    psa = np.exp(rng.normal(1.5, 0.9, n)).clip(0.1, 120)
    age = rng.normal(61, 7, n).clip(38, 82).round()
    ancestry = rng.choice(["EUR", "AFR", "EAS", "AMR"], n, p=[.73, .14, .09, .04])

    # Hazard driven by Gleason and stage, with a modest AFR effect, then
    # exponential times and administrative censoring at 120 months.
    lp = 0.42 * (gleason - 7) + 0.30 * pd.Series(stage).str.startswith("T3").astype(int).values \
         + 0.22 * (ancestry == "AFR") + 0.012 * (psa - 8)
    rate = 0.0045 * np.exp(lp)
    t_event = rng.exponential(1 / rate)
    t_cens = rng.uniform(6, 120, n)
    time = np.minimum(t_event, t_cens)
    event = (t_event <= t_cens).astype(int)
    return pd.DataFrame({
        "sample": samples, "patient": [s[:12] for s in samples],
        "surv_time": time.round(1), "surv_event": event,
        "ancestry": ancestry, "ancestry_source": "genetic_simulated",
        "endpoint_name": "simulated_recurrence",
        "endpoint_source_column": "synthetic",
        "gleason": gleason, "stage_t": stage, "psa": psa.round(1), "age": age,
    })


def load_clinical(samples):
    print("[1.7] Clinical follow-up")
    cl = load_clinical_live(samples) if RUN_MODE == "LIVE" else load_clinical_demo(samples)
    # SELF-REVIEW: the reduction to one record per patient was applied
    # inside run_survival_suite(), but the COUNTS were reported here, from
    # the un-reduced table - so the number printed as "patients with usable
    # follow-up" was a SAMPLE count, and survival_power_report() received
    # the same un-reduced frame, inflating its event count and every
    # min_detectable_HR derived from it. I fixed the analysis last round and
    # left the reporting and the power calculation on sample rows, which is
    # the pseudoreplication the review raised, surfacing as a number the
    # reader would take for patients.
    #
    # The DEMO generator emits one sample per patient, so the reduction is a
    # no-op there and no DEMO test could have caught this - TCGA PRAD and
    # SU2C both carry patients with multiple specimens. The reduction now
    # happens ONCE, at this single entry point, so the report, the power
    # calculation and the models all see the same patient-level table. The
    # call in run_survival_suite() is retained as an idempotent guard.
    _n_samples = len(cl)
    cl, _n_repeat = enforce_one_sample_per_patient(cl, "clinical cohort")
    ok = cl[survival_eligibility_mask(cl)]
    # BUG FIX (both external reviews, correct): this generic wrapper printed
    # "usable BCR follow-up" whatever endpoint was actually loaded - DFS, PFS
    # or the simulated DEMO endpoint. It changes no number, but it is the
    # same class of claim the endpoint-provenance safeguards exist to
    # prevent: a label asserting an endpoint the run did not use. The label
    # is derived from the recorded metadata instead.
    _ep_label = (str(cl.endpoint_name.iloc[0]).replace("_", " ")
                 if "endpoint_name" in cl.columns and len(cl)
                 else "unknown endpoint")
    print(f"  cohort: {_n_samples:,} sample-level record(s) -> "
          f"{len(cl):,} unique patient(s)"
          + (f" ({_n_repeat:,} repeated specimen(s) dropped)"
             if _n_repeat else ""))
    print(f"  PATIENTS with usable {_ep_label} follow-up: {len(ok):,} | "
          f"events: {int(ok.surv_event.sum())} | "
          f"median OBSERVED time: "
          f"{ok.surv_time.median():.1f} mo")
    # BUG FIX (external review, correct): this was labelled "median
    # follow-up", which is a different statistic. The median of observed
    # times mixes event times with censoring times and understates follow-up
    # whenever events are common; median follow-up is estimated by reverse
    # Kaplan-Meier, which survival_power_report() reports separately. Two
    # names for two quantities, so a manuscript cannot borrow one number for
    # the other.
    print(f"  ancestry source: {cl.ancestry_source.iloc[0]}")
    print("  ancestry counts (with events):")
    for a, grp in ok.groupby("ancestry"):
        print(f"    {a}: n={len(grp)}, events={int(grp.surv_event.sum())}")
    return cl

---
## Section 2 — Merge, clean, feature engineering, and the four gatekeeping filters

**Surfaceome bounding comes first**, before any statistical test. Restricting to membrane-localised proteins cuts the multiple-testing burden roughly tenfold, which buys power for the FDR-controlled comparison in Filter 1. Bounding after testing would be selective reporting.

**Filter 2 is benchmarked against PSMA rather than against an absolute expression cap.** The TPM at which a target becomes toxic is unknown and tissue-dependent. What is known is that PSMA-directed radioligand therapy is tolerated in humans at PSMA's observed normal-tissue expression. Expressing every candidate as a ratio to PSMA turns an unanswerable question — *how much is too much?* — into an answerable one: *is this worse than something already given to patients?*

**Proteomic corroboration is tracked but does not veto.** Mass spectrometry systematically under-detects membrane proteins; non-detection is a coverage artefact, not evidence of absence. Treating it as a fifth filter would discard real targets for being hydrophobic.

**F1–F3 are the biological screen; F4 is an experimental-readiness tier.** Antibody availability reflects how much a protein has been studied, so a strict reagent veto suppresses exactly the understudied antigens a discovery study exists to find. The original strict four-gate pipeline is still computed in full, as a declared sensitivity analysis (`passed_strict_four_gate`), so nothing is discarded — set `CFG.f4_mode = "strict"` to make it primary.

**Differential expression prefers a count-based model when counts exist.** Supply `COUNTS_FILES` and the pipeline uses **pyDESeq2** when installed — the established implementation with empirical-Bayes dispersion shrinkage — falling back to an unregularised per-gene NB GLM, explicitly labelled as such, when it is not. At these sample sizes per-gene dispersion estimates are noisy, and shrinkage is the difference between a validated method and an approximation of one, so `pip install pydeseq2` before the final run.

A **hard guard** refuses to run a count model on data that is not raw counts: it checks for negative values, for a plausible integer fraction, and for plausible magnitude. Applying a negative-binomial model to TPM or log-transformed values is invalid because the mean–variance relationship the family assumes is not there, and it is an easy mistake to make silently.

**The PSMA ratio is an expression benchmark, not proof of clinical safety.** Normal-cell context, target localisation, therapeutic modality and organ-specific toxicity all remain unresolved by it, so absolute per-organ TPM values are reported alongside the ratio.

In [4]:
# =============================================================================
# SECTION 2 - MERGE, CLEAN, FEATURE ENGINEERING, GATEKEEPING FILTERS
# =============================================================================

# ----------------------------------------------------------------------------
# 2.1 Cleaning utilities
# ----------------------------------------------------------------------------
def harmonise_genes(*frames):
    """Intersect gene indices across cohorts and report attrition.

    Silent index mismatch produces an all-NaN merge that looks like "no
    significant hits" rather than like a bug, so attrition is printed.
    """
    sets = [set(f.index if isinstance(f, pd.DataFrame) else f) for f in frames]
    common = set.intersection(*sets)
    for i, s in enumerate(sets):
        print(f"    frame {i}: {len(s):,} genes -> {len(common):,} shared "
              f"({100*len(common)/max(len(s),1):.0f}% retained)")
    return sorted(common)


def clean_expression(df, label, min_expr=CFG.min_expr_tpm,
                     min_frac=0.0, drop_zero_var=False):
    """QC an expression matrix. Input and output are log2(TPM+1).

    CRITICAL BUG FIX (raised in external code review, confirmed by a minimal
    reproduction): `drop_zero_var` used to default to True and was applied to
    the tumour and normal matrices INDEPENDENTLY, before the gene index was
    intersected. An antigen that is expressed in most tumours but undetectable
    in every adjacent normal has EXACTLY zero variance in the normal matrix,
    so it was dropped there and then removed from the tumour matrix at the
    intersection.

    That is the single most desirable phenotype in a surfaceome
    target-discovery study - a perfectly tumour-restricted surface antigen -
    and it was the one phenotype guaranteed to be deleted before testing.
    The defect is invisible in DEMO because synthetic normal samples carry
    sampling noise and therefore never have exactly zero variance; it would
    only have appeared on real GTEx/TCGA normals.

    The default is now False, and per-group variance filtering is no longer
    used anywhere in the pipeline. Variance eligibility, where it is wanted
    at all, is assessed JOINTLY across tumour and normal after harmonisation
    by `joint_variance_filter()` below, which can never remove a gene for
    being constant in one group alone.
    """
    n0 = df.shape
    df = df.loc[:, df.notna().mean() > 0.5]          # drop empty samples
    df = df.loc[df.notna().mean(axis=1) > 0.8, :]    # drop sparse genes
    df = df.apply(pd.to_numeric, errors="coerce")
    # Impute the few remaining gaps with the gene's own median, not zero:
    # zero-filling a log-expression matrix invents strong negative signal.
    df = df.T.fillna(df.median(axis=1)).T
    if drop_zero_var:
        # Retained only for explicit opt-in on a SINGLE-group matrix where
        # cross-group rescue is impossible (e.g. GTEx tissue medians). Never
        # use it on one arm of a two-group contrast.
        v = df.var(axis=1)
        df = df.loc[v > 1e-8]
    if min_frac > 0:
        frac_on = (linear_tpm(df) >= min_expr).mean(axis=1)
        df = df.loc[frac_on >= min_frac]
    print(f"    {label}: {n0[0]:,}x{n0[1]:,} -> {df.shape[0]:,}x{df.shape[1]:,}")
    return df


def joint_variance_filter(df_t, df_n, label="tumour vs normal"):
    """Drop only genes that are constant across the POOLED tumour+normal data.

    A gene constant in one group but not the other carries the strongest
    possible differential signal and is retained. A gene constant across both
    groups at the same value carries no information for any test and is
    dropped. Returns the surviving gene index plus a report.
    """
    common = sorted(set(df_t.index) & set(df_n.index))
    pooled = pd.concat([df_t.loc[common], df_n.loc[common]], axis=1)
    v_pooled = pooled.var(axis=1)
    keep = v_pooled > 1e-8
    v_t, v_n = df_t.loc[common].var(axis=1), df_n.loc[common].var(axis=1)
    # Genes that a per-group filter WOULD have deleted but that are retained
    # here - i.e. the population the old defect was silently discarding.
    rescued = int(((v_t <= 1e-8) | (v_n <= 1e-8)).reindex(keep.index).fillna(False)
                  .astype(bool).values.sum())
    dropped = int((~keep).sum())
    print(f"    joint variance filter ({label}): {len(common):,} genes -> "
          f"{int(keep.sum()):,} retained; {dropped:,} constant in POOLED data")
    if rescued:
        print(f"    {rescued:,} gene(s) are constant within one group only and "
              f"are RETAINED - a per-group variance filter would have deleted "
              f"them, including any perfectly tumour-restricted antigen")
    return ([g for g in common if keep.loc[g]],
            {"n_input": len(common), "n_retained": int(keep.sum()),
             "n_dropped_constant_in_pooled": dropped,
             "n_retained_constant_within_one_group_only": rescued})


# ----------------------------------------------------------------------------
# 2.2 Differential expression, tumour vs normal prostate
# ----------------------------------------------------------------------------
def differential_expression_nb(counts_t, counts_n, alpha=CFG.fdr_alpha):
    """Count-based negative-binomial GLM - the PRIMARY test when raw counts are
    available.

    PROPOSAL REFINEMENT (Section 6): RNA-seq counts are over-dispersed and
    heteroscedastic. A t-test on transformed abundance is a reasonable
    sensitivity analysis but is not the appropriate primary model when counts
    and design metadata exist. This fits a per-gene NB GLM with a library-size
    offset, which is the same family DESeq2 and edgeR use.

    PREFERENCE ORDER, and why it matters:
      1. pyDESeq2 - used automatically when installed. It applies
         empirical-Bayes dispersion shrinkage, which an unregularised per-gene
         NB fit does not. At these sample sizes, per-gene dispersion estimates
         are noisy, and shrinkage is the difference between a validated method
         and an approximation of one.
      2. This statsmodels NB GLM - a DEPENDENCY-FREE FALLBACK, explicitly
         labelled as such in run_summary.json. Usable, but do not describe it
         as DESeq2-equivalent in a manuscript.

    HARD GUARD: count models require RAW COUNTS. Applying them to TPM or to
    log-transformed values is invalid - the mean-variance relationship the NB
    family assumes simply is not there. The function checks and refuses.
    """
    import statsmodels.api as sm

    # --- Validate that these really are raw counts --------------------------
    # BUG FIX (both external reviews, correct on both counts): the check
    # inspected only the FIRST 50,000 values of each matrix and accepted the
    # input if >= 95% of those were integer-like. The pyDESeq2 branch then
    # applied .round(), so up to 5% of the sampled values - and an unbounded
    # fraction of the unsampled remainder - could be normalised or imputed
    # abundances silently coerced into apparently valid raw counts. A
    # negative-binomial dispersion estimated from rounded TPM is not a
    # count model, and nothing downstream would reveal it.
    #
    # Validation now covers EVERY non-missing value and admits no tolerance:
    # a single fractional count is a wrong input file, not noise to round
    # away. The rounding in the pyDESeq2 branch therefore becomes a no-op
    # type cast on data already proven integral.
    # SELF-REVIEW of this round's own fix: the strict validation I wrote
    # last round was correct but materialised the WHOLE matrix three times -
    # a float64 view, a non-NaN copy, and an np.floor temporary. On LIVE TCGA
    # PRAD (~60,000 genes x ~550 samples) that is ~825 MB per matrix and
    # ~2.2 GB across both, on top of the GTEx, SU2C and TPM matrices already
    # resident. In Colab that is a credible MemoryError, and because it is
    # reachable only on the LIVE path no DEMO test can ever hit it. The
    # external review flagged exactly this ("implement equivalent validation
    # in chunks to avoid unnecessary memory copies") and I implemented the
    # correctness half while dropping the engineering half.
    #
    # Validation is now streamed one column block at a time, so peak extra
    # memory is a few MB regardless of cohort size, and the coverage is
    # unchanged: EVERY non-missing value is still checked, with no tolerance.
    _CHUNK = 64
    _global_max = -np.inf
    for _lbl, _mat in (("tumour", counts_t), ("normal", counts_n)):
        _ncol = _mat.shape[1]
        if _ncol == 0:
            continue
        _n_checked = 0
        for _lo in range(0, _ncol, _CHUNK):
            _block = _mat.iloc[:, _lo:_lo + _CHUNK].to_numpy(dtype=float)
            # BUG FIX (external review, correct, and a defect in MY OWN
            # rewrite of last round): missing values were EXCLUDED from
            # validation rather than rejected, so a count matrix containing
            # NaN passed every check and then failed inside pyDESeq2 with
            # IntCastingNaNError - a crash far from its cause, on the LIVE
            # path only. A missing count and a true zero are different
            # events, biologically and technically, so NaN is neither
            # silently skipped nor silently zero-filled: it is refused here,
            # where the reason is still visible.
            _finite = np.isfinite(_block)
            if not _finite.all():
                _isnan = np.isnan(_block)
                _n_nan = int(_isnan.sum())
                _n_inf = int((~_finite & ~_isnan).sum())
                raise ValueError(
                    f"{_lbl} counts: {_n_nan:,} missing and {_n_inf:,} "
                    f"infinite value(s) in columns "
                    f"{_lo}-{min(_lo + _CHUNK, _ncol) - 1}. A raw-count "
                    f"matrix must be complete and finite: a missing count is "
                    f"not a zero count, so it cannot be filled in, and an "
                    f"inf indicates a prior division. Resolve the gaps in "
                    f"the source matrix, or set CFG.de_method='welch' to use "
                    f"a model that tolerates missingness explicitly.")
            _obs = _block[_finite]
            if not _obs.size:
                continue
            _n_checked += int(_obs.size)
            if (_obs < 0).any():
                _neg = _obs[_obs < 0]
                raise ValueError(
                    f"{_lbl} counts: {_neg.size:,} negative value(s) "
                    f"(min {_neg.min():.4g}) - these are not raw counts. "
                    f"Count models must not be applied to TPM, FPKM or "
                    f"log-transformed values.")
            _frac = _obs[_obs != np.floor(_obs)]
            if _frac.size:
                raise ValueError(
                    f"{_lbl} counts: {_frac.size:,} fractional value(s) in "
                    f"columns {_lo}-{min(_lo + _CHUNK, _ncol) - 1} "
                    f"(e.g. {_frac[:3].tolist()}). These are normalised or "
                    f"imputed abundances, not raw counts; rounding them "
                    f"would fabricate a count matrix. Supply raw counts, or "
                    f"set CFG.de_method='welch'.")
            _global_max = max(_global_max, float(_obs.max()))
        print(f"    {_lbl} counts validated: {_n_checked:,} value(s), all "
              f"finite, non-negative and integral")
    # The implausibility check reuses the maximum already seen, so neither
    # matrix is materialised a second time.
    if np.isfinite(_global_max) and _global_max < 50:
        raise ValueError(
            f"Maximum value {_global_max:.2f} is implausibly small for raw "
            f"counts. Check that COUNTS_FILES points at a counts matrix.")

    # --- Prefer pyDESeq2 when available -------------------------------------
    try:
        from pydeseq2.dds import DeseqDataSet
        from pydeseq2.ds import DeseqStats
        print("    using pyDESeq2 (empirical-Bayes dispersion shrinkage)")
        genes = harmonise_genes(counts_t, counts_n)
        t, n = counts_t.loc[genes], counts_n.loc[genes]
        # SELF-REVIEW: .round() is now provably a no-op - validation above
        # rejects any fractional, missing or infinite value - but leaving it
        # here is a trap twice over. A reader infers that fractional input is
        # tolerated, which is precisely the behaviour that was removed; and
        # if the validation is ever relaxed, the rounding silently
        # reactivates the original defect of fabricating a count matrix from
        # normalised abundances. The cast now asserts integrality instead of
        # imposing it.
        mat = pd.concat([t, n], axis=1).T
        _mv = mat.to_numpy(dtype=float)
        if not (np.isfinite(_mv).all()
                and np.equal(_mv, np.floor(_mv)).all()):
            raise AssertionError(
                "pyDESeq2 input is not integral despite validation - the "
                "count validation above and this cast have diverged. Do not "
                "round here: find why a non-integral value reached this "
                "point.")
        mat = mat.astype(int)
        meta = pd.DataFrame(
            {"condition": ["tumour"] * t.shape[1] + ["normal"] * n.shape[1]},
            index=mat.index)
        # pyDESeq2 renamed `design_factors` to `design` at 0.5. Support both
        # rather than pinning a version the user may not have.
        try:
            dds = DeseqDataSet(counts=mat, metadata=meta, design="~condition",
                               refit_cooks=True, quiet=True)
        except TypeError:
            dds = DeseqDataSet(counts=mat, metadata=meta,
                               design_factors="condition", refit_cooks=True)
        dds.deseq2()
        res = DeseqStats(dds, contrast=["condition", "tumour", "normal"],
                         quiet=True)
        res.summary()
        r = res.results_df
        out = pd.DataFrame({
            "gene": r.index,
            "rna_log2fc_nb": r["log2FoldChange"].values,
            "rna_pval_nb": r["pvalue"].fillna(1.0).values,
            "rna_qval_nb": r["padj"].fillna(1.0).values,
        })
        print(f"    pyDESeq2: q<{alpha}: {(out.rna_qval_nb < alpha).sum():,}")
        out.attrs["method"] = "pydeseq2"
        return out
    except ImportError:
        print("    pyDESeq2 not installed - falling back to an UNREGULARISED "
              "per-gene NB GLM. This lacks dispersion shrinkage; install "
              "pydeseq2 for the final manuscript run "
              "(pip install pydeseq2) and do not describe the fallback as "
              "DESeq2-equivalent.")

    genes = harmonise_genes(counts_t, counts_n)
    t, n = counts_t.loc[genes], counts_n.loc[genes]
    y_all = np.hstack([t.values, n.values])
    group = np.hstack([np.ones(t.shape[1]), np.zeros(n.shape[1])])
    lib = y_all.sum(axis=0)
    offset = np.log(np.maximum(lib, 1) / np.median(np.maximum(lib, 1)))
    X = sm.add_constant(group)

    lfc, pvals = np.zeros(len(genes)), np.ones(len(genes))
    print(f"    fitting NB GLM for {len(genes):,} genes (library-size offset) ...")
    for i in range(len(genes)):
        y = y_all[i]
        if y.sum() < 10:
            continue
        try:
            mu = max(y.mean(), 1e-6)
            var = max(y.var(ddof=1), mu * 1.000001)
            alpha_disp = float(np.clip((var - mu) / (mu ** 2), 1e-6, 10))
            m = sm.GLM(y, X, family=sm.families.NegativeBinomial(alpha=alpha_disp),
                       offset=offset).fit()
            lfc[i] = m.params[1] / np.log(2)      # natural log -> log2
            pvals[i] = m.pvalues[1]
        except Exception as ex:
            # A failed fit is NOT a non-significant result. Recording p = 1
            # without a marker makes a convergence failure indistinguishable
            # from a gene that was tested and found unchanged.
            pvals[i] = np.nan
            FIT_FAILURES.append({"stage": "nb_glm", "gene": genes[i],
                                 "error": f"{type(ex).__name__}: {ex}"[:160]})
    ok_fit = np.isfinite(pvals)
    n_failed = int((~ok_fit).sum())
    if n_failed:
        print(f"    {n_failed} genes FAILED to fit (recorded as missing, not "
              f"as non-significant); see fit_failures.csv")
    qv = np.full(len(pvals), np.nan)
    if ok_fit.sum() > 1:
        qv[ok_fit] = multipletests(pvals[ok_fit], alpha=alpha,
                                   method="fdr_bh")[1]
    print(f"    NB GLM (unregularised fallback): q<{alpha}: {(qv < alpha).sum():,}")
    out = pd.DataFrame({"gene": genes, "rna_log2fc_nb": lfc,
                        "rna_pval_nb": pvals, "rna_qval_nb": qv})
    out.attrs["method"] = "statsmodels_nb_glm_unregularised"
    return out


def differential_expression(tumour, normal, alpha=CFG.fdr_alpha):
    """Welch t-test per gene on log2 abundance + Benjamini-Hochberg FDR.

    STATUS: this is the primary test only when raw counts are unavailable
    (which is the case for the Xena TPM matrices). When counts ARE available,
    differential_expression_nb() is the primary analysis and this becomes a
    sensitivity analysis - see CFG.de_method.

    Welch (not Student) because tumour and normal variances differ materially
    and the group sizes are very unbalanced (~480 vs ~52). A pooled-variance
    test here would be anti-conservative.

    A rank-based test (Mann-Whitney) is computed alongside as a distributional
    sensitivity check, because log-TPM is not exactly normal and reviewers ask.
    """
    genes = harmonise_genes(tumour, normal)
    t, n = tumour.loc[genes], normal.loc[genes]
    tv, nv = t.values, n.values

    stat, p = st.ttest_ind(tv, nv, axis=1, equal_var=False, nan_policy="omit")
    # Mann-Whitney over genes (vectorised across rows is unavailable, so loop
    # only if the gene count is manageable; it is, after surfaceome bounding).
    p_mw = np.array([st.mannwhitneyu(tv[i], nv[i], alternative="two-sided").pvalue
                     for i in range(len(genes))])

    lfc = tv.mean(axis=1) - nv.mean(axis=1)   # already log2 -> difference is LFC
    pooled_sd = np.sqrt((tv.var(axis=1, ddof=1) + nv.var(axis=1, ddof=1)) / 2)
    cohen_d = lfc / np.where(pooled_sd > 1e-9, pooled_sd, np.nan)

    p = np.nan_to_num(p, nan=1.0)
    p_mw = np.nan_to_num(p_mw, nan=1.0)
    qv = multipletests(p, alpha=alpha, method="fdr_bh")[1]
    qv_mw = multipletests(p_mw, alpha=alpha, method="fdr_bh")[1]

    frac_on = (linear_tpm(t) >= CFG.min_expr_tpm).mean(axis=1).values

    out = pd.DataFrame({
        "gene": genes,
        "rna_log2fc": lfc,
        "rna_pval": p,
        "rna_qval": qv,
        "rna_pval_mw": p_mw,
        "rna_qval_mw": qv_mw,
        "cohen_d": cohen_d,
        "tumour_mean_log2": tv.mean(axis=1),
        "normal_mean_log2": nv.mean(axis=1),
        "frac_tumours_expressing": frac_on,
    })
    print(f"    tested {len(out):,} genes | "
          f"q<{alpha}: {(out.rna_qval < alpha).sum():,} | "
          f"q<{alpha} & LFC>{CFG.min_log2fc}: "
          f"{((out.rna_qval < alpha) & (out.rna_log2fc > CFG.min_log2fc)).sum():,}")
    concord = ((out.rna_qval < alpha) == (out.rna_qval_mw < alpha)).mean()
    print(f"    parametric/rank-test concordance at q<{alpha}: {100*concord:.1f}%")
    return out


# ----------------------------------------------------------------------------
# 2.3 Vital-organ safety features, benchmarked against PSMA
# ----------------------------------------------------------------------------
def safety_features(gtex_log, benchmark=CFG.benchmark_gene):
    """Compute vital-organ exposure, expressed relative to PSMA.

    The absolute TPM at which a target becomes toxic is unknown and
    tissue-dependent. What IS known is that PSMA-directed radioligand therapy
    is tolerated in humans at its observed normal-tissue expression. Expressing
    every candidate as a ratio to PSMA converts an unanswerable question
    ("how much is too much?") into an answerable one ("is this worse than
    something we already give to patients?").
    """
    # By this point every matrix is on the canonical log2(TPM+1) scale, set
    # once at load. Converting again here - as an earlier version did, with
    # the Xena pseudocount - would apply the wrong inverse a second time.
    tpm = linear_tpm(gtex_log, label="GTEx tissue medians")
    # Belt and braces: the inverse already refuses negative input, but a
    # ratio is only interpretable if its numerator and denominator are both
    # finite and non-negative, so assert that here too.
    if not np.isfinite(tpm.values).all():
        n = int((~np.isfinite(tpm.values)).sum())
        raise ValueError(
            f"{n:,} non-finite values in the recovered normal-tissue TPM "
            f"matrix. A tissue-exposure ratio computed from them is "
            f"meaningless, so the pipeline stops rather than producing one.")
    # BUG FIX (external review, correct): organs were matched by LITERAL
    # name, but GTEx v8 labels the relevant salivary tissue "Minor Salivary
    # Gland" while this config declares "Salivary Gland". On LIVE data the
    # organ behind PSMA radioligand xerostomia would simply be absent: a
    # warning would print, F2 would still be declared assessable, and the
    # comparator would quietly omit a dose-limiting tissue.
    resolved, missing = harmonise_tissue_labels(tpm.columns)
    vital = list(dict.fromkeys(resolved.values()))
    organ_coverage = len(resolved) / max(len(set(CFG.vital_organs)), 1)
    out_coverage_ok = organ_coverage >= CFG.min_vital_organ_coverage
    if missing:
        print(f"    WARNING: vital organs absent after harmonisation: "
              f"{missing} (coverage {organ_coverage:.0%}; "
              f"{CFG.min_vital_organ_coverage:.0%} required for a DEFINITIVE "
              f"F2 result). A missing organ is NOT a safe organ.")
    # Declare the organs no transcriptome in this study can cover. Their
    # absence is directional, not neutral: it favours the modality whose
    # dose-limiting toxicity they carry.
    for organ, why in CFG.unmeasurable_risk_organs.items():
        print(f"    UNMEASURABLE ORGAN AT RISK - {organ}: {why}")
    if not vital:
        raise ValueError("No vital-organ columns matched. Inspect GTEx tissue labels.")

    vt = tpm[vital]
    out = pd.DataFrame({
        "gene": tpm.index,
        "vital_max_tpm": vt.max(axis=1).values,
        "vital_mean_tpm": vt.mean(axis=1).values,
        "vital_n_organs_over_5tpm": (vt > 5).sum(axis=1).values,
        "normal_breadth_n_tissues_over_5tpm": (tpm > 5).sum(axis=1).values,
        "worst_organ": vt.idxmax(axis=1).values,
    })
    if benchmark not in tpm.index:
        raise ValueError(
            f"Benchmark gene {benchmark} absent from GTEx matrix; Filter 2 "
            f"cannot be calibrated. Check gene-symbol harmonisation."
        )
    bench_max_raw = float(vt.loc[benchmark].max())
    # CRITICAL BUG FIX, found by the new detection-floor sensitivity analysis:
    # the floor was applied to the NUMERATOR (each candidate's vital-organ
    # maximum) but NOT to the DENOMINATOR (PSMA's own). With PSMA's vital-organ
    # max at 0.00 TPM in this matrix, `bench_max` fell back to the 1e-3 epsilon
    # and every ratio was divided by 0.001 - so the whole
    # safety_ratio_vs_psma scale was arbitrary, and at any floor above 0 the
    # filter admitted nothing. One floor policy, applied to both sides.
    floor_cmp = float(CFG.vital_tpm_detection_floor)
    bench_max = max(bench_max_raw, floor_cmp)
    if bench_max_raw < floor_cmp:
        # The anchor carries no information in this case: PSMA's own exposure
        # is itself at or below the detection limit, so "relative to an
        # exposure tolerated in patients" cannot be evaluated here.
        print(f"    WARNING: the comparator gene {benchmark} has a "
              f"vital-organ maximum of {bench_max_raw:.4f} TPM, at or below "
              f"the {floor_cmp} TPM detection floor. Filter 2 is then "
              f"comparing every candidate against a quantity that was not "
              f"measurably expressed, so the ratio scale is NOT interpretable "
              f"as 'relative to an exposure already tolerated clinically'. "
              f"Both numerator and denominator are censored at the floor so "
              f"the filter degrades gracefully instead of silently admitting "
              f"nothing, but on LIVE data check the comparator's normal-tissue "
              f"values before interpreting any F2 count.")

    # NEW (found in our own review of the exported ranking): bulk RNA cannot
    # distinguish "not expressed in this organ" from "below the detection
    # limit of bulk RNA-seq". 25% of the eligible primary candidates in the
    # last run had a vital-organ maximum of EXACTLY 0.000 TPM, which gave them
    # a therapeutic window unbounded by construction and an automatic pass on
    # Filter 2 - the strongest possible safety claim resting on a measurement
    # at the floor of the assay.
    #
    # This is the GTEx-bulk-zero problem, and it is the failure mode that
    # kills antibody-drug conjugates in the clinic: expression confined to a
    # rare cell type within a vital organ is invisible in bulk tissue and is
    # precisely what produces unexpected organ toxicity.
    #
    # Vital-organ values are therefore CENSORED AT the detection floor rather
    # than taken as true zeros, which bounds the best achievable ratio, and
    # every candidate whose window depends on an at-limit measurement is
    # flagged so a reader can see which safety claims are assay-limited.
    floor = float(CFG.vital_tpm_detection_floor)
    raw_vital_max = out.vital_max_tpm.copy()
    out["vital_max_tpm_raw"] = raw_vital_max.values
    out["vital_max_at_detection_floor"] = (raw_vital_max < floor).values
    out["vital_max_tpm_censored"] = np.maximum(raw_vital_max.values, floor)
    out["vital_tpm_detection_floor"] = floor
    out["safety_ratio_vs_psma"] = out.vital_max_tpm_censored / bench_max
    # Keep the uncensored ratio for transparency; it must never gate a filter.
    out["safety_ratio_vs_psma_uncensored"] = raw_vital_max.values / bench_max
    # CRITICAL (external review, correct): the modality screens were taught to
    # refuse a censored comparator, but the PRIMARY F2 pathway was not. With
    # PSMA's own vital-organ maximum at or below the floor, F2 kept dividing
    # by the floor and producing PASS/FAIL verdicts that are arithmetic, not
    # evidence of a PSMA-relative therapeutic window - while all four modality
    # screens reported NOT ASSESSABLE from the same comparator. One rule now
    # governs both.
    out["f2_comparator_raw_tpm"] = bench_max_raw
    out["f2_comparator_censored_tpm"] = bench_max
    # Assessability now requires BOTH a measurable comparator AND adequate
    # organ coverage. A comparator measured against two thirds of the declared
    # organs is not the comparator that was prespecified.
    out["f2_vital_organ_coverage"] = float(organ_coverage)
    out["f2_organ_coverage_adequate"] = bool(out_coverage_ok)
    out["f2_comparator_measurable"] = bool(bench_max_raw > floor)
    out["f2_comparator_assessable"] = bool(bench_max_raw > floor) and bool(out_coverage_ok)
    if not out_coverage_ok:
        print(f"    F2 NOT DEFINITIVE: only {organ_coverage:.0%} of declared "
              f"vital organs were measured ({CFG.min_vital_organ_coverage:.0%} "
              f"required). The exploratory ratio still orders the ranking, but "
              f"no candidate may claim a demonstrated margin against an "
              f"incompletely measured organ panel.")
    if bench_max_raw <= floor:
        print(f"    F2 COMPARATOR NOT ASSESSABLE: {benchmark} vital-organ "
              f"maximum is {bench_max_raw:.4f} TPM, at or below the {floor} "
              f"TPM floor. The ratio is still computed as an EXPLORATORY "
              f"ranking diagnostic, but F2 cannot certify a PSMA-relative "
              f"safety margin, and every shortlist derived from it is "
              f"provisional and comparator-censored.")
    n_floor = int(out.vital_max_at_detection_floor.sum())
    n_zero = int((raw_vital_max == 0).sum())
    if n_floor:
        print(f"    {n_floor:,} gene(s) have a vital-organ maximum below the "
              f"{floor} TPM detection floor ({n_zero:,} at exactly 0.000); "
              f"their exposure is CENSORED AT the floor so the therapeutic "
              f"window is bounded, and they are flagged "
              f"`vital_max_at_detection_floor` - the window rests on an "
              f"at-limit bulk measurement, not on demonstrated absence")
    # PROPOSAL REFINEMENT (Step 2): report ABSOLUTE tissue values alongside the
    # PSMA ratio. The ratio is an expression benchmark, not proof of clinical
    # safety: normal-cell context, target localisation, therapeutic modality
    # and organ-specific toxicity all remain unresolved by it. A reader needs
    # the absolute number to judge that for themselves.
    out["psma_benchmark_vital_max_tpm"] = bench_max
    for organ in vital:
        out[f"tpm__{organ.replace(' ', '_').replace('-', '')}"] = tpm[organ].values
    # Tissue-specificity: how concentrated is expression in prostate?
    if "Prostate" in tpm.columns:
        out["prostate_tpm"] = tpm["Prostate"].values
        out["prostate_specificity"] = (
            tpm["Prostate"].values / (tpm.drop(columns=["Prostate"]).max(axis=1).values + 1e-3))
    print(f"    PSMA ({benchmark}) vital-organ max = {bench_max:.2f} TPM "
          f"(worst organ: {vt.loc[benchmark].idxmax()})")
    print(f"    candidates at or below the PSMA safety ceiling: "
          f"{(out.safety_ratio_vs_psma <= CFG.safety_ratio_max).sum():,} / {len(out):,}")
    return out


# ----------------------------------------------------------------------------
# 2.4 mCRPC / NEPC features
# ----------------------------------------------------------------------------
def experimental_surface_evidence(f):
    """Attach intact-cell surface evidence from SURFACE_ASSAY_FILE.

    This is the ONLY route by which a protein lacking a transmembrane or GPI
    anchor may pass F3. It requires a positive assay result, parsed by the
    same strict rules used for the evidence grade - a declaration that a
    protein is "secreted" or "extracellular" is not evidence of accessibility.
    """
    f = f.copy()
    f["surface_detected_experimentally"] = False
    if not (SURFACE_ASSAY_FILE and Path(SURFACE_ASSAY_FILE).exists()):
        print("    experimental surface-accessibility route: NO assay file "
              "supplied, so no gene can pass F3 without a membrane anchor. "
              "Surface-associated targets without transmembrane domains "
              "(e.g. KLK2) remain ineligible until intact-cell binding "
              "evidence is provided.")
        return f
    register_input_file(SURFACE_ASSAY_FILE, role="surface assay evidence")
    sa = pd.read_csv(SURFACE_ASSAY_FILE)
    if "gene" not in sa.columns or "surface_detected" not in sa.columns:
        raise ValueError(
            "SURFACE_ASSAY_FILE needs `gene` and `surface_detected` columns "
            "to support the experimental accessibility route.")
    # BUG FIX (external review, correct): a boolean alone certified that
    # SOMEBODY ENTERED a positive result, not that the result came from a
    # qualifying intact-cell experiment - and the same evidence can confer
    # evidence grade A and change biological eligibility. Every positive
    # claim must now carry provenance, and a positive without it is REFUSED
    # rather than downgraded, because a half-documented claim that still
    # unlocks the gate is the failure mode being fixed.
    pos_rows = sa.loc[sa.surface_detected.map(
        lambda v: str(v).strip().lower() in
        {"true", "t", "yes", "y", "1", "positive", "detected"})].copy()
    missing_meta = []
    for _, row in pos_rows.iterrows():
        gaps = [c for c in SURFACE_EVIDENCE_REQUIRED_FIELDS
                if not str(row.get(c, "")).strip()
                or str(row.get(c, "")).strip().lower() in {"nan", "none"}]
        if gaps:
            missing_meta.append((str(row.gene), gaps))
    if missing_meta:
        raise ValueError(
            f"{len(missing_meta)} positive surface-accessibility claim(s) in "
            f"SURFACE_ASSAY_FILE lack required provenance, e.g. "
            f"{missing_meta[:3]}. A positive flag alone cannot establish "
            f"intact-cell accessibility or evidence grade A. Every positive "
            f"row needs: {list(SURFACE_EVIDENCE_REQUIRED_FIELDS)}. Rows "
            f"recording a NEGATIVE result need no provenance.")
    pos = set(pos_rows.gene.astype(str))
    f["surface_detected_experimentally"] = f.gene.astype(str).isin(pos)
    for c in SURFACE_EVIDENCE_REQUIRED_FIELDS:
        if c in sa.columns:
            f[f"surface_evidence_{c}"] = f.gene.astype(str).map(
                dict(zip(sa.gene.astype(str), sa[c].astype(str))))
    print(f"    experimental surface-accessibility evidence supplied for "
          f"{int(f.surface_detected_experimentally.sum())} gene(s)")
    return f


def unmeasurable_organ_report():
    """Export the organs at risk that no transcriptome here can assess."""
    rows = [{"organ": o, "why_it_matters": w,
             "covered_by_this_pipeline": False,
             "required_before_clinical_claim":
                 "independent protein-level or imaging assessment"}
            for o, w in CFG.unmeasurable_risk_organs.items()]
    out = pd.DataFrame(rows)
    save_table(out, "table24_unmeasurable_risk_organs.csv",
               caption="Organs at risk that this pipeline CANNOT assess. "
                       "Their absence systematically flatters the modality "
                       "whose dose-limiting toxicity they carry.")
    return out


def mcrpc_features(expr, pheno):
    """Expression in metastatic CRPC, and specifically in NEPC.

    This is the feature block that addresses the stated clinical gap. A target
    that is excellent in primary disease but absent in mCRPC does not solve the
    problem the study is posed against.
    """
    s = pheno.set_index("sample").reindex(expr.columns)
    nepc = s.is_nepc.fillna(False).values.astype(bool)
    out = pd.DataFrame({
        "gene": expr.index,
        "mcrpc_mean_log2": expr.mean(axis=1).values,
        "mcrpc_frac_expressing": (linear_tpm(expr) >= CFG.min_expr_tpm).mean(axis=1).values,

    })
    if nepc.sum() >= 5 and (~nepc).sum() >= 5:
        a, b = expr.loc[:, nepc].values, expr.loc[:, ~nepc].values
        _, p = st.ttest_ind(a, b, axis=1, equal_var=False, nan_policy="omit")
        p = np.nan_to_num(p, nan=1.0)
        out["nepc_log2fc"] = a.mean(axis=1) - b.mean(axis=1)
        out["nepc_qval"] = multipletests(p, method="fdr_bh")[1]
        out["nepc_mean_log2"] = a.mean(axis=1)
        out["retained_in_nepc"] = (out.nepc_log2fc > -1.0) & (out.nepc_mean_log2 > 2.0)
        print(f"    NEPC comparison: n={int(nepc.sum())} vs {int((~nepc).sum())} | "
              f"targets retained in NEPC: {int(out.retained_in_nepc.sum()):,}")
    else:
        print(f"    NEPC subgroup too small (n={int(nepc.sum())}); NEPC features set to NaN. "
              f"Any NEPC claim must be dropped from the manuscript.")
        for c in ["nepc_log2fc", "nepc_qval", "nepc_mean_log2"]:
            out[c] = np.nan
        out["retained_in_nepc"] = np.nan

    # RAISED IN EXTERNAL REVIEW, and correct on both counts:
    #   - kurtosis is NOT a bimodality test. Negative excess kurtosis is
    #     consistent with bimodality but also with many unimodal flat
    #     distributions, so the old name "mcrpc_bimodality" claimed more than
    #     the statistic delivers.
    #   - dividing SD by the MEAN OF LOG values is not a coefficient of
    #     variation; on a log scale the mean can approach zero and the ratio
    #     explodes. "mcrpc_cv" was simply the wrong name for the quantity.
    # Both are renamed to what they measure, and a real bimodality coefficient
    # is added alongside.
    out["mcrpc_neg_excess_kurtosis"] = expr.apply(
        lambda r: float(-st.kurtosis(r.dropna(), fisher=True))
        if r.notna().sum() > 10 else np.nan, axis=1).values

    def _bimodality_coefficient(row):
        """Sarle's bimodality coefficient: (skew^2 + 1) / kurtosis.
        Values above ~0.555 (the uniform-distribution reference) are
        consistent with bimodality. Still a heuristic, not a test."""
        v = row.dropna().values
        if v.size < 20:
            return np.nan
        n = v.size
        g1, g2 = st.skew(v), st.kurtosis(v, fisher=True)
        denom = g2 + 3 * ((n - 1) ** 2) / ((n - 2) * (n - 3))
        return float((g1 ** 2 + 1) / denom) if denom > 0 else np.nan

    out["mcrpc_bimodality_coefficient"] = expr.apply(_bimodality_coefficient, axis=1).values
    # Dispersion on the log scale, named for what it is.
    out["mcrpc_log_sd"] = expr.std(axis=1).values
    return out


# ----------------------------------------------------------------------------
# 2.5 Assemble the master feature table
# ----------------------------------------------------------------------------
def build_feature_table(surf, de, safety, mcrpc, prot, hpa):
    print("[2.5] Assembling master feature table")
    f = surf.merge(de, on="gene", how="inner")
    for name, d in [("safety", safety), ("mcrpc", mcrpc), ("hpa", hpa)]:
        before = len(f)
        f = f.merge(d, on="gene", how="left")
        print(f"    + {name}: {before:,} -> {len(f):,} rows")
    f = f.merge(prot, on="gene", how="left")
    f["protein_detected"] = f.prot_log2fc.notna()
    f = apply_proteomic_fdr(f, prot)

    # --- Derived / engineered features --------------------------------------
    # Therapeutic window: tumour signal over worst vital organ, in log space.
    # BUG FIX (external review, confirmed): this used the RAW vital-organ
    # maximum while the rest of the pipeline censors that quantity at the
    # detection floor, so a gene measured at 0.000 TPM in every vital organ
    # received a window computed against zero - the unbounded-window artifact
    # the floor exists to prevent. 291 of the 338 below-floor genes had
    # windows inflated by more than 0.1 log2 units.
    #
    # This matters more than its size suggests: therapeutic_window_log2 IS
    # baseline B3 (tumour selectivity), the transparent baseline the ensemble
    # is measured against, so the headline comparison was being made against
    # a baseline computed under a different censoring policy from everything
    # else in the analysis.
    if "vital_max_tpm_censored" not in f.columns:
        raise KeyError(
            "therapeutic_window_log2 requires vital_max_tpm_censored. "
            "safety_features() must run before build_feature_table(); "
            "falling back to the raw maximum silently reintroduces the "
            "uncensored window this fix removes.")
    f["therapeutic_window_log2"] = (
        f.tumour_mean_log2 - np.log2(f.vital_max_tpm_censored.clip(lower=0) + 1))
    f["therapeutic_window_log2_uncensored"] = (
        f.tumour_mean_log2 - np.log2(f.vital_max_tpm.clip(lower=0) + 1))
    _infl = ((f.therapeutic_window_log2_uncensored
              - f.therapeutic_window_log2) > 0.1)
    print(f"    therapeutic window uses the FLOOR-CENSORED vital-organ "
          f"maximum; {int(_infl.sum()):,} gene(s) had windows inflated by "
          f">0.1 log2 units under the raw value. This quantity is baseline "
          f"B3, so the ensemble-vs-baseline comparison moves with it.")
    # Agreement between primary and metastatic disease.
    f["primary_to_mcrpc_delta"] = f.mcrpc_mean_log2 - f.tumour_mean_log2
    f["expressed_in_both"] = ((f.frac_tumours_expressing > 0.5) &
                              (f.mcrpc_frac_expressing > 0.5)).astype(float)
    # Transcript-protein concordance, where protein was measured.
    f["rna_protein_concordant"] = np.where(
        f.protein_detected, np.sign(f.rna_log2fc) == np.sign(f.prot_log2fc), np.nan)
    # Topology score: single-pass with a long ECD is the ideal mAb/ADC target.
    f["topology_score"] = (f.is_single_pass.astype(int) * 2
                           + (f.ecd_len_aa >= 100).astype(int)
                           + (f.ecd_len_aa >= CFG.min_ecd_length_aa).astype(int)
                           + f.has_gpi.astype(int) * 0.5)
    f["log_ecd_len"] = np.log1p(f.ecd_len_aa)
    f["log_n_antibodies"] = np.log1p(f.n_antibodies.fillna(0))
    f["neg_log10_rna_q"] = -np.log10(f.rna_qval.clip(lower=1e-300))
    f["log_safety_ratio"] = np.log2(f.safety_ratio_vs_psma.clip(lower=1e-4))
    f["log_prostate_specificity"] = np.log2(f.get("prostate_specificity", pd.Series(1, index=f.index)).clip(lower=1e-4))

    f = f.drop_duplicates("gene").reset_index(drop=True)
    guarded_to_csv(f, DIRS["features"] / "master_feature_table.csv")
    print(f"    master table: {f.shape[0]:,} genes x {f.shape[1]:,} columns")
    return f



# ----------------------------------------------------------------------------
# 2.5b PROTEOMIC MULTIPLE-TESTING FAMILY
# ----------------------------------------------------------------------------
# The correction family is the ORIGINAL hypothesis-testing universe.
#
# An earlier version recomputed Benjamini-Hochberg over the proteins that both
# were detected AND survived surfaceome bounding - a post-hoc subset of a much
# larger family. BH adjusts the i-th smallest of m p-values by m/i, so
# shrinking m from a whole proteome (~9,000-11,000 quantified proteins) to a
# surfaceome subset (~1,200) inflates significance by roughly the ratio of
# those counts. Every proteomic "corroboration" would have been easier to earn
# than the source analysis intended, and nothing in the output would say so.
#
# Three sanctioned routes, in order of preference, with no silent fallback.
# ----------------------------------------------------------------------------
def apply_proteomic_fdr(f, prot):
    """Attach prot_qval under an explicitly declared testing family."""
    family = CFG.protein_fdr_family
    n_det = int(f.protein_detected.sum())
    f["prot_qval"] = np.nan
    f["prot_fdr_family"] = family
    f["prot_fdr_family_size"] = np.nan

    # --- Route 1: q-values from the original analysis -----------------------
    if family == "supplied":
        if "prot_qval_supplied" in f.columns and f.prot_qval_supplied.notna().any():
            # Validate supplied q-values before trusting them. A q-value out
            # of [0,1], or one smaller than its own p-value, means the column
            # is not what it claims - and silently adopting it would import
            # someone else's error as this study's statistics.
            q = pd.to_numeric(f.prot_qval_supplied, errors="coerce")
            pv = pd.to_numeric(f.get("prot_pval"), errors="coerce")
            bad_range = int(((q < 0) | (q > 1)).sum())
            if bad_range:
                raise ValueError(
                    f"{bad_range} supplied q-values fall outside [0, 1]. The "
                    f"column is not an adjusted p-value; check the source.")
            if pv is not None and pv.notna().any():
                viol = int((q + 1e-9 < pv).sum())
                if viol > 0.01 * len(f):
                    raise ValueError(
                        f"{viol} supplied q-values are SMALLER than their own "
                        f"p-values, which no monotone FDR procedure produces. "
                        f"The q and p columns are probably mismatched or the "
                        f"q-values are not adjusted.")
            cov = float(q.notna().mean())
            print(f"    supplied q-values validated: range OK, monotone vs p, "
                  f"coverage {cov:.0%} of surfaceome rows")
            if cov < 0.5:
                print(f"      CAUTION: under half the rows carry a source "
                      f"q-value; the rest will be missing rather than imputed")
            f["prot_qval"] = q
            m = int(prot.attrs.get("n_tested_proteome", 0)) or np.nan
            f["prot_fdr_family_size"] = m
            print(f"    proteomic FDR: using q-values from the SOURCE analysis "
                  f"(family size {m if m == m else 'documented by the source'})")
            print(f"      {n_det:,} surfaceome proteins carry a source q-value; "
                  f"no re-correction applied")
            return f
        raise ValueError(
            "CFG.protein_fdr_family='supplied' but no 'prot_qval_supplied' "
            "column was provided.\n"
            "Supply the adjusted p-values from the original proteome-wide "
            "analysis (column: prot_qval_supplied), which is the correct and "
            "preferred route because the family is already whatever that "
            "analysis tested.\n"
            "Alternatives: set protein_fdr_family='proteome_wide' and supply "
            "the FULL proteome p-value table, or "
            "protein_fdr_family='surfaceome_subset' WITH "
            "protein_fdr_family_prespecified=True if a surfaceome-restricted "
            "family is separately justified and was declared in advance.")

    # --- Route 2: recompute BH over the full proteome, then subset ----------
    if family == "proteome_wide":
        full = prot.attrs.get("full_proteome_pvalues")
        if full is None:
            raise ValueError(
                "CFG.protein_fdr_family='proteome_wide' requires the FULL "
                "proteome p-value table, not just the surfaceome rows. BH is "
                "rank-based, so it cannot be reconstructed from a subset: the "
                "rank of each p-value within the complete family is exactly "
                "the information that was discarded.\n"
                "Attach it as prot.attrs['full_proteome_pvalues'] = DataFrame "
                "with columns gene, prot_pval.")
        full = full.dropna(subset=["prot_pval"]).drop_duplicates("gene")
        m = len(full)
        q_all = multipletests(full.prot_pval.values, method="fdr_bh")[1]
        qmap = dict(zip(full.gene.values, q_all))
        f["prot_qval"] = f.gene.map(qmap)
        f["prot_fdr_family_size"] = m
        print(f"    proteomic FDR: BH recomputed across the FULL proteome "
              f"({m:,} proteins tested), then subset to the surfaceome")
        print(f"      {n_det:,} surfaceome proteins detected; "
              f"{int(f.prot_qval.notna().sum()):,} received a family-wide q-value")
        return f

    # --- Route 3: prespecified surfaceome family ----------------------------
    if family == "surfaceome_subset":
        if not CFG.protein_fdr_family_prespecified:
            raise ValueError(
                "CFG.protein_fdr_family='surfaceome_subset' requires "
                "protein_fdr_family_prespecified=True.\n"
                "Correcting within the surfaceome is defensible ONLY as a "
                "separately justified hypothesis family declared BEFORE "
                "looking at the data. Applied post hoc to a subset of a "
                "proteome-wide analysis it inflates significance by roughly "
                "the ratio of the two family sizes. Set the flag to record "
                "that this was prespecified, and state the justification in "
                "the Methods.")
        det = f.protein_detected.values
        if det.sum() > 1:
            f.loc[det, "prot_qval"] = multipletests(
                f.loc[det, "prot_pval"].fillna(1.0).values, method="fdr_bh")[1]
        f["prot_fdr_family_size"] = int(det.sum())
        print(f"    proteomic FDR: PRESPECIFIED surfaceome-restricted family "
              f"({int(det.sum()):,} detected surfaceome proteins)")
        print("      This is a narrower family than the source analysis. The "
              "Methods must state the justification and that it was declared "
              "in advance.")
        return f

    raise ValueError(f"Unknown protein_fdr_family '{family}'.")


# ----------------------------------------------------------------------------
# 2.6 The four fixed gatekeeping filters
# ----------------------------------------------------------------------------
def apply_gatekeeping(f, save_outputs=True):
    """Apply the four declared filters and record a transparent audit trail.

    `save_outputs=False` computes without writing any artifact. Tests MUST
    pass False: this function writes the filter-attrition table and the
    filtered feature table, and calling it on a fixture with writing enabled
    is what overwrote two publication artifacts in the previous build.

    Each gene keeps a per-filter boolean, so the attrition table in the
    manuscript is generated from data rather than retyped - and so a reviewer
    asking "why did gene X drop out?" gets an exact answer.
    """
    print("[2.6] Gatekeeping filters")
    g = f.copy()

    # Filter 1 - differential expression with FDR and fold-change control
    g["F1_differential"] = (
        (g.rna_qval < CFG.fdr_alpha) &
        (g.rna_log2fc >= CFG.min_log2fc) &
        (g.frac_tumours_expressing >= CFG.min_frac_expressing)
    )

    # F2 assessability travels with the data: when the comparator was
    # censored, F2 yields an exploratory ranking signal, never a certified
    # safety verdict. Recorded per gene so no downstream table can lose it.
    _f2_assessable = bool(g.get("f2_comparator_assessable",
                                pd.Series([True])).iloc[0]) \
        if "f2_comparator_assessable" in g.columns else True
    g["f2_assessable"] = _f2_assessable

    # Filter 2 - normal-tissue EXPRESSION COMPARATOR relative to PSMA.
    # Not a validated safety threshold - see the Config note on f2 naming.
    #
    # BUG FIX (critical): the test was `ratio <= 1.0`, which a NEGATIVE ratio
    # passes trivially. Negative ratios arose from impossible negative TPM and
    # 254 of 2,792 genes auto-passed the filter in the previous run. A ratio
    # must now be FINITE AND NON-NEGATIVE to be evaluated at all; anything
    # else fails the filter and is counted, so an upstream scale problem
    # surfaces as a visible filter failure instead of silent admission.
    ratio = pd.to_numeric(g.safety_ratio_vs_psma, errors="coerce")
    valid_ratio = np.isfinite(ratio) & (ratio >= 0)
    n_invalid = int((~valid_ratio).sum())
    if n_invalid:
        print(f"    F2: {n_invalid:,} genes have an INVALID exposure ratio "
              f"(negative or non-finite) and FAIL the filter rather than "
              f"passing it. Investigate the expression scale upstream.")
    g["F2_ratio_valid"] = valid_ratio
    g["F2_safety"] = valid_ratio & (ratio <= CFG.safety_ratio_max)
    # Three-state outcome (external review): a boolean cannot distinguish
    # "comparator measured and criterion met" from "comparator unmeasurable,
    # so nothing was determined". NOT ASSESSABLE must never be read as a pass.
    # BUG FIX (found by reconciling f2_assessable against F2_outcome in the
    # exported ranking): `_f2_assessable` is a PYTHON bool, and `~` on a
    # Python bool is bitwise NOT on the integer - `~True` is -2 and `~False`
    # is -1, BOTH truthy. np.where therefore selected "NOT ASSESSABLE" for
    # every gene regardless of assessability, while F2_safety_definitive
    # (which uses `& _f2_assessable`) was computed correctly. The two columns
    # contradicted each other in the same table. Use `not` for a scalar bool.
    g["F2_outcome"] = np.where(
        (not _f2_assessable), "NOT ASSESSABLE",
        np.where(g.F2_safety, "PASS", "FAIL"))
    # A DEFINITIVE pass needs both the criterion and an assessable
    # comparator. F2_safety is retained for the exploratory ranking, so the
    # ranking is unchanged while eligibility claims become honest.
    g["F2_safety_definitive"] = g.F2_safety & _f2_assessable
    if not _f2_assessable:
        print(f"    F2 outcome recorded as NOT ASSESSABLE for all "
              f"{len(g):,} genes: the comparator is censored, so "
              f"F2_safety_definitive is False everywhere and no candidate "
              f"can claim a demonstrated PSMA-relative margin. The "
              f"exploratory ratio still orders the ranking.")

    # Filter 3 - confirmed membrane topology with a targetable ECD
    # RAISED IN EXTERNAL REVIEW, and correct: a transmembrane domain does not
    # mean PLASMA membrane. ER, mitochondrial, lysosomal and Golgi proteins
    # are all transmembrane and all untargetable by an antibody, so topology
    # alone admits intracellular-membrane proteins to a cell-surface screen.
    # CLINICAL GAP (external review, correct): F3 recognised only
    # transmembrane and GPI anchoring, but surface ACCESSIBILITY is not
    # limited to those mechanisms. KLK2/hK2 is the clearest case - a secreted
    # kallikrein with flow-cytometrically demonstrated cell-surface
    # localisation on prostate cancer cells, and active radioligand and CAR
    # programmes - and it is excluded by topology alone. In the current run
    # KLK2 is one of our own held-out BENCHMARKS yet is ineligible by
    # construction, so no model could ever recover it into a shortlist.
    #
    # A third, EXPERIMENTALLY ESTABLISHED route is therefore allowed. It is
    # deliberately narrow: it requires intact-cell binding evidence supplied
    # in SURFACE_ASSAY_FILE, never an annotation that a protein is secreted
    # or extracellular, and every gene admitted by it is flagged so the basis
    # is auditable and separable in any table.
    _anchored = (g.n_transmem >= 1) | g.has_gpi.fillna(False)
    _experimental = g.get("surface_detected_experimentally",
                          pd.Series(False, index=g.index)).fillna(False)
    n_exp = int((_experimental & ~_anchored).sum())
    if n_exp:
        print(f"    {n_exp} gene(s) pass F3 by EXPERIMENTALLY ESTABLISHED "
              f"surface accessibility rather than membrane anchoring "
              f"(intact-cell binding evidence from SURFACE_ASSAY_FILE). "
              f"Flagged in F3_route so the basis is auditable.")
    # BUG FIX (external review, correct): the ECD-length rule was applied to
    # BOTH F3 routes, so a protein with demonstrated intact-cell surface
    # binding could still be excluded by an annotation-derived ectodomain
    # length - which is exactly the annotation the experimental route exists
    # to supplement. KLK2 shows the failure mode: synthetic annotations give
    # it 7 transmembrane segments and a 5-aa ectodomain, so even a positive
    # surface assay would not have rescued it.
    #
    # The two routes are now independent. The topology-inferred route keeps
    # the ECD requirement, because for an inferred call the ectodomain IS the
    # evidence. The experimental route does not, because a measured
    # intact-cell binding result supersedes an inferred length.
    _ecd_ok = pd.Series(True, index=g.index)
    if CFG.require_extracellular_domain:
        _ecd_ok = (g.ecd_len_aa >= CFG.min_ecd_length_aa)
    route_anchored = _anchored & _ecd_ok
    route_experimental = _experimental
    g["F3_topology"] = route_anchored | route_experimental
    # Distinguish the route ATTEMPTED from the gate PASSED, so no exported
    # table can say a route was established while the gate actually failed.
    g["F3_route_attempted"] = np.where(
        _anchored, "membrane-anchored (transmembrane or GPI)",
        np.where(_experimental,
                 "experimentally established surface accessibility",
                 "none"))
    g["F3_route"] = np.where(
        route_experimental, "experimentally established surface accessibility",
        np.where(route_anchored, "membrane-anchored (transmembrane or GPI)",
                 "not established"))
    g["F3_failed_on_ecd_length"] = _anchored & ~_ecd_ok & ~route_experimental
    n_ecd_fail = int(g.F3_failed_on_ecd_length.sum())
    if n_ecd_fail:
        print(f"    {n_ecd_fail} membrane-anchored gene(s) fail F3 on "
              f"ectodomain length alone (< {CFG.min_ecd_length_aa} aa). An "
              f"intact-cell binding result would admit them by the "
              f"experimental route; an annotation alone would not.")

    # Plasma-membrane confidence, graded rather than binary. HPA localisation
    # is antibody-dependent and missing for understudied proteins, so it
    # raises confidence where present and never vetoes where absent.
    intracellular = g.get("subcellular", pd.Series("", index=g.index)).astype(str)
    g["intracellular_membrane_flag"] = intracellular.str.contains(
        "endoplasmic|mitochond|lysosom|golgi|peroxisom|nuclear membrane",
        case=False, na=False)
    # ORDER MATTERS: the intracellular annotation is tested BEFORE the HPA
    # membrane flag. A protein carrying both annotations is excluded from the
    # screen as intracellular, so labelling it "high - HPA plasma membrane"
    # in the same table contradicted the exclusion it had just received.
    g["plasma_membrane_confidence"] = np.select(
        [g.intracellular_membrane_flag,
         g.hpa_membrane.fillna(False),
         g.F3_topology],
        ["low - annotated to an intracellular membrane",
         "high - HPA plasma membrane",
         "moderate - topology only, localisation unconfirmed"],
        default="not applicable")
    n_intra = int((g.F3_topology & g.intracellular_membrane_flag).sum())
    if n_intra:
        print(f"    {n_intra:,} topology-passing genes are annotated to an "
              f"INTRACELLULAR membrane - flagged, not vetoed, because HPA "
              f"localisation is itself antibody-dependent")
    # HPA localisation is used as corroboration, not as a veto: HPA membrane
    # annotation is antibody-dependent and absent for poorly studied proteins,
    # so vetoing on it would systematically remove novel candidates - the very
    # thing the study is looking for.
    g["F3_hpa_corroborated"] = g.F3_topology & g.hpa_membrane.fillna(False)

    # Filter 4 - reagent availability, as a READINESS TIER rather than a veto
    # PROPOSAL REFINEMENT (Step 2): F1-F3 are the biological screen. Antibody
    # availability reflects how much a protein has been studied, so vetoing on
    # it removes understudied antigens by default - the opposite of what a
    # discovery study wants. It is retained as experimental-readiness evidence.
    g["F4_reagent"] = g.antibody_available.fillna(False)

    # PRIMARY definition: biological screening only.
    g["passed_biological_screen"] = g.F1_differential & g.F2_safety & g.F3_topology
    # Definitive variant: identical, except it requires an ASSESSABLE safety
    # comparator. Any claim of clinical readiness or therapeutic
    # prioritisation uses this; the exploratory ranking keeps the other.
    g["passed_biological_screen_definitive"] = (
        g.F1_differential & g.F2_safety_definitive & g.F3_topology)

    # ------------------------------------------------------------------
    # DISEASE-CONTEXT SCREENS
    # ------------------------------------------------------------------
    # RAISED IN EXTERNAL REVIEW, and correct: the study is motivated by
    # advanced disease - PSMA-low, heterogeneous and neuroendocrine tumours -
    # but the screen tested primary-tumour differential expression only. A
    # candidate could pass every filter while being absent from metastatic
    # disease, which is precisely the population the work is posed against.
    #
    # Eligibility is therefore declared PER DISEASE CONTEXT, before ranking.
    # A gene can qualify for one context and not another, and the context is
    # reported rather than averaged away.
    # BUG FIX (raised in external review): intracellular-membrane proteins
    # were FLAGGED but not excluded, so a protein with plausible topology and
    # explicit annotation to the ER, mitochondrion, lysosome or Golgi could
    # remain a high-ranked "surface" target. For an ADC, CAR-T, bispecific or
    # radioligand that is not a marginal concern - the antigen is simply not
    # reachable.
    #
    # They are now excluded from the biological screen, but retained in a
    # separate DISCOVERY-ONLY category rather than deleted, because HPA
    # localisation is antibody-dependent and a confident intracellular call
    # for an understudied protein can itself be wrong.
    intra_flag = g.get("intracellular_membrane_flag",
                       pd.Series(False, index=g.index)).fillna(False)
    n_excluded = int((g.passed_biological_screen & intra_flag).sum())
    g["excluded_intracellular"] = g.passed_biological_screen & intra_flag
    g["passed_biological_screen"] = g.passed_biological_screen & ~intra_flag
    if n_excluded:
        print(f"    {n_excluded:,} genes EXCLUDED from the biological screen "
              f"for annotation to an intracellular membrane; retained in the "
              f"discovery-only category (`excluded_intracellular`) because "
              f"HPA localisation is antibody-dependent and can be wrong for "
              f"understudied proteins")

    g["eligible_primary"] = g.passed_biological_screen
    g["eligibility_basis"] = ("comparator-censored (exploratory)"
                              if not _f2_assessable
                              else "PSMA-relative comparator measured")
    # BUG FIX (both external reviews, independently reproduced, and I
    # reproduced it before changing anything). Last round a static check
    # flagged the _elig_state lambda as defined-but-never-called, and I made
    # it called. I did not check whether it was RIGHT. It was not:
    #
    #     np.where(not _f2_assessable, "PROVISIONAL", np.where(...))
    #
    # The first condition is a SCALAR, so when the F2 comparator is
    # unassessable it is true for every row and every gene becomes
    # PROVISIONAL - including genes that definitively fail F1 differential
    # expression or F3 topology, or are excluded as intracellular. The
    # three-state column collapsed to one state, destroying the exact
    # distinction it was added to make. The DEMO run hides it because organ
    # coverage is 100%, so _f2_assessable is True; a LIVE cohort missing
    # vital-organ references takes the other branch and would have labelled
    # all 2,792 genes provisional.
    #
    # The lesson is about method, not this line: a static checker reports
    # that a value is discarded, never that it was wrong. "Unused" is not
    # "ready to use", and I treated it as such.
    #
    # The lambda is deleted rather than patched - a dead-but-wrong helper is
    # a trap for whoever next tries to use it. A definitive failure of a
    # gate that does not depend on F2 remains a failure when F2 cannot be
    # assessed; only a gene that clears those gates is genuinely pending.
    _other_gates_pass = (g.F1_differential.fillna(False).astype(bool)
                         & g.F3_topology.fillna(False).astype(bool)
                         & ~intra_flag.astype(bool))
    if _f2_assessable:
        g["eligibility_state"] = np.where(
            _other_gates_pass & g.F2_safety.fillna(False).astype(bool),
            "ELIGIBLE", "NOT ELIGIBLE")
    else:
        g["eligibility_state"] = np.where(
            _other_gates_pass, "NOT ASSESSABLE - F2", "NOT ELIGIBLE")

    # SELF-REVIEW (round 35): eligible_primary and eligibility_state answer
    # DIFFERENT questions, and on the unassessable path they disagree on the
    # same row - a gene clearing F1, F3 and the intracellular exclusion but
    # failing the censored F2 comparator reads NOT ASSESSABLE - F2 in one
    # column and False in the other. Both are defensible (eligible_primary
    # is explicitly exploratory when the comparator is censored, as
    # eligibility_basis records), but nothing stated the relationship, so a
    # reviewer filtering on eligible_primary would silently drop genes the
    # state column calls merely pending.
    #
    # On the ASSESSABLE path the two must agree exactly, and that is worth
    # asserting at runtime rather than trusting: it is a cross-column
    # invariant, the kind that breaks quietly when either definition is
    # edited. Verified as holding on the delivered DEMO run (0 violations
    # across 2,792 genes) before being enforced here.
    if _f2_assessable:
        _agree = ((g.eligibility_state == "ELIGIBLE")
                  == g.eligible_primary.fillna(False).astype(bool))
        if not bool(_agree.all()):
            _bad = g.loc[~_agree, "gene"].head(5).tolist()
            raise AssertionError(
                f"eligibility_state and eligible_primary disagree for "
                f"{int((~_agree).sum())} gene(s) with an ASSESSABLE F2 "
                f"comparator, e.g. {_bad}. On that path they must be the "
                f"same decision expressed two ways; a disagreement means "
                f"one of the two definitions has drifted.")
    g["eligibility_state_authority"] = (
        "definitive (F2 comparator measured)" if _f2_assessable
        else "pending F2 evidence - eligibility_state is authoritative here; "
             "eligible_primary is exploratory because the comparator is "
             "censored")
    if not _f2_assessable:
        n_prov = int(g.passed_biological_screen.sum())
        print(f"    {n_prov:,} gene(s) pass the biological screen, but the F2 "
              f"comparator was censored, so their eligibility is PROVISIONAL "
              f"and must not be read as demonstrated PSMA-relative safety.")

    # CRITICAL BUG FIX (raised in external code review, confirmed): the
    # intracellular-membrane exclusion was applied to the primary screen only,
    # via passed_biological_screen. The mCRPC and NEPC eligibility rules were
    # built directly from F2/F3 and so skipped it entirely - meaning a protein
    # annotated to an INTRACELLULAR membrane could be shortlisted as an
    # advanced-disease therapeutic target, in the very shortlists (mCRPC,
    # NEPC, PSMA-low) that are this study's primary deliverable. An antibody,
    # CAR-T cell or radioligand cannot reach an intracellular membrane at all,
    # so this is a hard accessibility error, not a ranking nuance.
    # The exclusion is now applied in EVERY disease context and to the
    # modality classifications, from one shared flag.
    g["intracellular_excluded_flag"] = intra_flag.astype(bool)

    mcrpc_ok = (g.get("mcrpc_frac_expressing", pd.Series(np.nan, index=g.index))
                >= CFG.min_frac_expressing)
    g["eligible_mcrpc"] = (g.F2_safety & g.F3_topology & ~intra_flag
                           & mcrpc_ok.fillna(False))

    if "retained_in_nepc" in g.columns and g.retained_in_nepc.notna().any():
        g["eligible_nepc"] = (g.F2_safety & g.F3_topology & ~intra_flag &
                              g.retained_in_nepc.fillna(False).astype(bool))
    else:
        g["eligible_nepc"] = np.nan   # not assessable; never silently False

    g["n_contexts_eligible"] = (g[["eligible_primary", "eligible_mcrpc"]]
                                .fillna(False).astype(int).sum(axis=1)
                                + g.eligible_nepc.fillna(False).astype(int))

    # BUG FIX (raised in external review): the context label omitted NEPC
    # entirely, so an antigen eligible ONLY in neuroendocrine disease was
    # classified "neither" - the exact phenotype the study is posed against,
    # labelled as of no interest.
    ep = g.eligible_primary.fillna(False)
    em = g.eligible_mcrpc.fillna(False)
    en = g.eligible_nepc.fillna(False)
    g["disease_context"] = np.select(
        [ep & em & en, ep & em, ep & en, em & en,
         ep, em, en],
        ["primary + mCRPC + NEPC", "primary + mCRPC", "primary + NEPC",
         "mCRPC + NEPC", "primary only", "mCRPC only", "NEPC only"],
        default="none")
    ctx = g.disease_context.value_counts()
    print("    disease-context composition:")
    for k, v in ctx.items():
        if k != "none":
            print(f"      {k}: {v:,}")
    print(f"    eligible in primary: {int(g.eligible_primary.sum()):,} | "
          f"mCRPC: {int(g.eligible_mcrpc.sum()):,} | "
          f"both: {int((g.eligible_primary & g.eligible_mcrpc).sum()):,}")
    if g.eligible_nepc.notna().any():
        print(f"    eligible in NEPC: {int(g.eligible_nepc.fillna(False).sum()):,}")
    else:
        print("    NEPC eligibility NOT ASSESSABLE (subgroup too small) - "
              "left as missing rather than False, so it cannot be mistaken "
              "for an assessed negative")
    # SENSITIVITY definition: the original strict four-gate pipeline, retained
    # in full so the original design is reproduced rather than discarded.
    g["passed_strict_four_gate"] = g.passed_biological_screen & g.F4_reagent

    if CFG.f4_mode == "strict":
        g["passed_all_filters"] = g.passed_strict_four_gate
    else:
        g["passed_all_filters"] = g.passed_biological_screen

    print(f"    F4 mode: {CFG.f4_mode} "
          f"({'reagent availability is a hard gate' if CFG.f4_mode=='strict' else 'reagent availability is a readiness tier, not a gate'})")
    print(f"    passed biological screen (F1-F3): "
          f"{int(g.passed_biological_screen.sum()):,}")
    print(f"    passed strict four-gate (sensitivity analysis): "
          f"{int(g.passed_strict_four_gate.sum()):,}")
    lost = int((g.passed_biological_screen & ~g.F4_reagent).sum())
    print(f"    biologically-qualified but reagent-poor: {lost:,} "
          f"(these are retained as Tier 3; a strict veto would discard them)")

    # Protein-level corroboration is tracked separately from the four filters,
    # because MS non-detection is a coverage artefact and not evidence of
    # absence. Treating it as a fifth veto would discard real targets for the
    # crime of being hydrophobic.
    g["protein_corroborated"] = (
        g.protein_detected &
        (g.prot_qval < CFG.protein_fdr_alpha) &
        (g.prot_log2fc >= CFG.protein_min_log2fc)
    ).fillna(False)

    rows, remaining = [], len(g)
    rows.append({"step": "Surfaceome-bounded, expression-measured", "n": remaining, "removed": 0})
    cum = pd.Series(True, index=g.index)
    for col, label in [
        ("F1_differential", "F1: BH-FDR + fold-change (tumour vs normal)"),
        ("F2_safety",       f"F2: normal-tissue expression comparator "
                            f"(vital-organ ratio <= {CFG.safety_ratio_max}x PSMA; "
                            f"NOT a validated safety threshold)"),
        ("F3_topology",     f"F3: membrane topology + ECD >= {CFG.min_ecd_length_aa} aa"),
        ("F4_reagent",      "F4: reagent available (READINESS TIER, not a biological gate)"),
    ]:
        prev = int(cum.sum())
        cum &= g[col].fillna(False)
        rows.append({"step": label, "n": int(cum.sum()), "removed": prev - int(cum.sum())})
    rows.append({"step": "+ protein-level corroboration (reported, not vetoed)",
                 "n": int((cum & g.protein_corroborated).sum()),
                 "removed": int((cum & ~g.protein_corroborated).sum())})

    attrition = pd.DataFrame(rows)
    if save_outputs:
        save_table(attrition, "table1_filter_attrition.csv",
                   caption="Sequential attrition through the four "
                           "gatekeeping filters")
    if not IN_NOTEBOOK:
        print(attrition.to_string(index=False))

    # --- Filter unit tests --------------------------------------------------
    # Each control gene must fail its designated filter and pass the other
    # three. This catches filter-logic inversion and column-name drift, which
    # are otherwise invisible because the output still looks like a gene list.
    # BUG FIX (raised in external review): a missing control was silently
    # skipped, so the unit-test file could be written with FEWER rows than
    # controls and still read as "all passed". The expected count is now
    # required, and a missing control is itself a FAILURE rather than an
    # absence.
    filt_cols = ["F1_differential", "F2_safety", "F3_topology", "F4_reagent"]
    audit = []
    for ctrl, should_fail in EXPECTED_CONTROL_FAILURES.items():
        if ctrl not in set(g.gene):
            audit.append({"control": ctrl, "should_fail": should_fail,
                          **{c: None for c in filt_cols},
                          "passed_all": None, "test": "FAIL",
                          "note": "control ABSENT from the feature table - "
                                  "the filter it guards is untested"})
            continue
        r = g.loc[g.gene == ctrl].iloc[0]
        got = {c: bool(r[c]) for c in filt_cols}
        ok = (got[should_fail] is False) and all(
            v for k, v in got.items() if k != should_fail)
        audit.append({"control": ctrl, "should_fail": should_fail,
                      **got, "passed_all": bool(r.passed_all_filters),
                      "test": "PASS" if ok else "FAIL", "note": ""})
    adf = pd.DataFrame(audit)
    n_expected = len(EXPECTED_CONTROL_FAILURES)
    # Stamp the artifact with this run's identity so a stale file from an
    # earlier run cannot satisfy the gate.
    adf["run_id"] = RUN_ID
    adf["n_expected_controls"] = n_expected
    if save_outputs:
        guarded_to_csv(adf, DIRS["logs"] / "filter_unit_tests.csv")
    print(f"    filter unit tests ({len(adf)}/{n_expected} controls present):")
    print(adf[["control", "should_fail", "test", "note"]].to_string(index=False))
    n_fail = int((adf.test == "FAIL").sum())
    if len(adf) != n_expected or n_fail:
        print(f"    *** FILTER UNIT TEST FAILURE: {n_fail} failed, "
              f"{n_expected - len(adf)} missing. Do not interpret results "
              f"until the filter logic is corrected. ***")

    if save_outputs:
        guarded_to_csv(g, DIRS["features"] / "feature_table_with_filters.csv")
    return g, attrition


# ============================================================================
# 2.7 SURFACE-TARGETABILITY EVIDENCE GRADE  [NEW]
# ============================================================================
# RAISED IN EXTERNAL REVIEW: protein abundance in a bulk tumour specimen is
# not the same quantity as accessible antigen density on a malignant cell
# surface. For an ADC, bispecific, CAR-T or radioligand the biologically
# decisive measurements are accessibility, surface density, normal-cell
# expression, internalisation where applicable, and heterogeneity - none of
# which bulk transcriptomics or bulk proteomics can supply.
#
# Without a separate grade, a well-expressed but poorly accessible membrane
# protein - or an intracellular-membrane protein that slipped through topology
# filtering - can receive an overstated therapeutic designation simply because
# its mRNA and total protein are high.
#
# This grade is deliberately CONSERVATIVE and mostly reports what is NOT
# known. Every candidate starts at "inferred" and can only reach higher grades
# with evidence this pipeline does not generate. The point is to make the gap
# explicit in the output rather than leaving a reader to assume it was closed.
# ============================================================================
SURFACE_EVIDENCE_GRADES = {
    "A": "Direct surface measurement on intact malignant cells "
         "(flow cytometry, surface biotinylation or equivalent)",
    "B": "Plasma-membrane localisation independently supported "
         "(HPA plasma-membrane annotation plus protein-level detection)",
    "C": "Plasma-membrane localisation annotated, protein not measured",
    "D": "Topology inferred from sequence only; localisation unconfirmed",
    "E": "Topology passes but the protein is annotated to an intracellular "
         "membrane - surface accessibility doubtful",
}

SURFACE_ASSAY_FILE = os.environ.get("SURFACE_ASSAY_FILE", "")


def surface_targetability_grade(g):
    """Assign a surface-accessibility evidence grade, separate from abundance.

    Optional input: SURFACE_ASSAY_FILE, a CSV with columns
    gene, surface_detected (bool), surface_density_per_cell (optional),
    assay (free text). Only this file can earn grade A - the pipeline cannot
    infer it, and does not pretend to.
    """
    hpa_pm = g.get("hpa_membrane", pd.Series(False, index=g.index)).fillna(False)
    intra = g.get("intracellular_membrane_flag",
                  pd.Series(False, index=g.index)).fillna(False)
    prot_det = g.get("protein_detected", pd.Series(False, index=g.index)).fillna(False)
    topo = g.get("F3_topology", pd.Series(False, index=g.index)).fillna(False)

    direct = pd.Series(False, index=g.index)
    density = pd.Series(np.nan, index=g.index)
    assay = pd.Series("", index=g.index, dtype="object")
    if SURFACE_ASSAY_FILE and Path(SURFACE_ASSAY_FILE).exists():
        register_input_file(SURFACE_ASSAY_FILE, role="surface assay evidence")
        sa = pd.read_csv(SURFACE_ASSAY_FILE)
        # CRITICAL BUG FIX (external review, correct and dangerous): the
        # previous code did
        #     sa.get("surface_detected", pd.Series(True, index=sa.index))
        #     direct = g.gene.map(lambda x: bool(key.get(str(x), False)))
        # which had two independent ways to award grade A - the HIGHEST
        # evidence grade, the only one that claims demonstrated surface
        # accessibility - without a positive assay result:
        #   1. If the file lacked a `surface_detected` column, EVERY row
        #      defaulted to True.
        #   2. bool() on the strings "False"/"false"/"no"/"0" and on np.nan
        #      is True in Python, so a text or missing value became a
        #      positive result.
        # A malformed assay file would therefore have promoted candidates to
        # "surface accessibility established" and straight into the
        # therapeutic shortlists. The column is now REQUIRED, values are
        # parsed strictly, missing results are refused, and duplicate gene
        # rows are rejected rather than silently resolved by dict order.
        if "gene" not in sa.columns:
            raise ValueError(
                f"SURFACE_ASSAY_FILE ({SURFACE_ASSAY_FILE}) has no `gene` "
                f"column. Columns found: {list(sa.columns)}")
        if "surface_detected" not in sa.columns:
            raise ValueError(
                f"SURFACE_ASSAY_FILE ({SURFACE_ASSAY_FILE}) has no "
                f"`surface_detected` column. This column is REQUIRED: it is "
                f"the only evidence that can earn surface-evidence grade A. "
                f"Defaulting it would award the study's highest evidence "
                f"grade on no evidence. Columns found: {list(sa.columns)}")
        dups = sa.gene.astype(str)[sa.gene.astype(str).duplicated()].unique()
        if len(dups):
            raise ValueError(
                f"SURFACE_ASSAY_FILE has duplicate gene rows: "
                f"{sorted(dups)[:10]}. Resolve them in the source file - "
                f"which replicate is authoritative is a scientific decision, "
                f"not one this code may make silently.")

        def _strict_bool(v):
            """Parse an assay result. Anything not clearly true/false fails."""
            if isinstance(v, (bool, np.bool_)):
                return bool(v)
            if isinstance(v, (int, float, np.integer, np.floating)):
                if pd.isna(v):
                    return None
                if float(v) in (0.0, 1.0):
                    return bool(v)
                return None
            t = str(v).strip().lower()
            if t in {"true", "t", "yes", "y", "1", "positive", "detected"}:
                return True
            if t in {"false", "f", "no", "n", "0", "negative", "not detected"}:
                return False
            return None

        parsed = sa.surface_detected.map(_strict_bool)
        bad = sa.loc[parsed.isna(), ["gene", "surface_detected"]]
        if len(bad):
            raise ValueError(
                f"SURFACE_ASSAY_FILE has {len(bad)} unparseable or missing "
                f"`surface_detected` value(s), e.g. "
                f"{bad.head(5).to_dict('records')}. A missing assay result is "
                f"NOT a negative and must not be treated as a positive "
                f"either, so the pipeline stops. Record an explicit "
                f"true/false for every gene, or remove the row.")
        key = dict(zip(sa.gene.astype(str), parsed.astype(bool)))
        # Unassayed genes are False (no evidence), never True.
        direct = g.gene.astype(str).map(key).fillna(False).astype(bool)
        n_pos, n_neg = int(parsed.sum()), int((~parsed.astype(bool)).sum())
        print(f"    surface assay file parsed strictly: {n_pos} positive, "
              f"{n_neg} negative result(s); genes absent from the file are "
              f"treated as NO EVIDENCE, not as positive")
        if "surface_density_per_cell" in sa.columns:
            density = g.gene.map(dict(zip(sa.gene.astype(str),
                                          sa.surface_density_per_cell)))
        if "assay" in sa.columns:
            assay = g.gene.map(dict(zip(sa.gene.astype(str),
                                        sa.assay.astype(str)))).fillna("")
        print(f"    surface assay evidence supplied for "
              f"{int(pd.Series(direct).sum()):,} genes")
    else:
        print("    surface-targetability: NO assay file supplied "
              "(SURFACE_ASSAY_FILE). No candidate can reach grade A, and the "
              "decisive measurement for every therapeutic modality is "
              "therefore UNMEASURED. State this in the limitations.")

    g["surface_evidence_grade"] = np.select(
        [direct,
         topo & hpa_pm & prot_det,
         topo & hpa_pm,
         topo & intra,
         topo],
        ["A", "B", "C", "E", "D"],
        default="not applicable")
    g["surface_evidence_description"] = g.surface_evidence_grade.map(
        SURFACE_EVIDENCE_GRADES).fillna("not a membrane protein")
    g["surface_density_per_cell"] = density
    g["surface_assay"] = assay
    # A single boolean a reader can act on, kept deliberately strict.
    g["surface_accessibility_established"] = g.surface_evidence_grade.eq("A")

    counts = g.loc[topo, "surface_evidence_grade"].value_counts().sort_index()
    print("    surface-targetability grades among topology-passing genes:")
    for grade, n in counts.items():
        print(f"      {grade}: {n:,} - {SURFACE_EVIDENCE_GRADES.get(grade, '')[:58]}")
    n_a = int(g.surface_accessibility_established.sum())
    print(f"    candidates with ESTABLISHED surface accessibility: {n_a:,}"
          + ("" if n_a else " - no therapeutic designation is justified on "
                            "this evidence alone"))
    return g


# ============================================================================
# 2.8 PSMA-HIGH vs PSMA-LOW PERSISTENCE  [NEW]
# ============================================================================
# RAISED IN EXTERNAL REVIEW, and the single most useful addition suggested:
# the study's stated unmet need is targets for PSMA-LOW disease, yet nothing
# in the pipeline measured whether a candidate is actually retained when PSMA
# is low. Every other analysis treats mCRPC as one population.
#
# This stratifies the metastatic cohort by FOLH1 expression and asks, per
# candidate, whether expression persists in the PSMA-low stratum. A target
# that tracks PSMA is useless for the patients the work is aimed at; a target
# that is retained, or enriched, when PSMA falls is exactly what it is for.
#
# This connects the computational output to the clinical question far more
# directly than any model metric, and it costs one extra comparison.
# ============================================================================
def psma_low_persistence(mcrpc_expr, benchmark=CFG.benchmark_gene,
                         low_quantile=0.25):
    """Per-gene expression in PSMA-low vs PSMA-high metastatic tumours."""
    if benchmark not in mcrpc_expr.index:
        print(f"    PSMA-low analysis SKIPPED: {benchmark} not measured in the "
              f"metastatic cohort, so PSMA strata cannot be defined. Recorded "
              f"as not performed.")
        return None

    psma = mcrpc_expr.loc[benchmark]
    thr_lo = psma.quantile(low_quantile)
    thr_hi = psma.quantile(1 - low_quantile)
    low = psma <= thr_lo
    high = psma >= thr_hi
    if int(low.sum()) < 5 or int(high.sum()) < 5:
        print(f"    PSMA-low analysis SKIPPED: strata too small "
              f"(low n={int(low.sum())}, high n={int(high.sum())}).")
        return None

    # BUG FIX (raised in external code review, confirmed): the two strata were
    # built with inclusive comparisons against the 25th and 75th percentiles
    # of PSMA, with no check that they are disjoint. If PSMA expression has
    # many tied values - which is exactly what happens in a zero-inflated or
    # low-PSMA cohort, the clinical population this analysis exists to serve -
    # then thr_lo == thr_hi and the SAME samples enter both groups. The t-test
    # would then compare a group against itself, returning p ~ 1 and delta ~ 0
    # for every gene, and the pipeline would report "no target persists in
    # PSMA-low disease" as a biological finding rather than as a degenerate
    # stratification. The n >= 5 guard does not catch this.
    overlap = int((low & high).sum())
    if overlap or not (thr_hi > thr_lo):
        print(f"    PSMA-low analysis SKIPPED: strata are NOT disjoint "
              f"(overlap n={overlap}; q{int(low_quantile*100)}={thr_lo:.4f}, "
              f"q{int((1-low_quantile)*100)}={thr_hi:.4f}). PSMA expression is "
              f"too tied in this cohort to define low and high strata at this "
              f"quantile. Recorded as not performed rather than reported as a "
              f"null result.")
        return None
    # Require a minimum separation so the contrast is interpretable, not just
    # technically disjoint by one tied observation.
    sep = float(thr_hi - thr_lo)
    if sep < CFG.min_psma_strata_separation_log2:
        print(f"    PSMA-low analysis SKIPPED: strata separated by only "
              f"{sep:.3f} log2 units (need >= "
              f"{CFG.min_psma_strata_separation_log2}). A contrast this "
              f"narrow cannot distinguish PSMA-low from PSMA-high biology.")
        return None
    print(f"    PSMA strata: low n={int(low.sum())} (<= {thr_lo:.3f}), "
          f"high n={int(high.sum())} (>= {thr_hi:.3f}), disjoint, "
          f"separation {sep:.3f} log2 units")

    lo, hi = mcrpc_expr.loc[:, low].values, mcrpc_expr.loc[:, high].values
    _, p = st.ttest_ind(lo, hi, axis=1, equal_var=False, nan_policy="omit")
    p = np.nan_to_num(np.asarray(p, dtype=float), nan=1.0)
    delta = lo.mean(axis=1) - hi.mean(axis=1)

    out = pd.DataFrame({
        "gene": mcrpc_expr.index,
        "psma_low_mean_log2": lo.mean(axis=1),
        "psma_high_mean_log2": hi.mean(axis=1),
        "psma_low_vs_high_delta": delta,
        "psma_low_vs_high_qval": multipletests(p, method="fdr_bh")[1],
        "psma_low_frac_expressing": (linear_tpm(
            mcrpc_expr.loc[:, low], label="mCRPC PSMA-low")
            >= CFG.min_expr_tpm).mean(axis=1).values,
    })
    # The clinically useful categories. "Retained" is the minimum bar for a
    # PSMA-alternative; "enriched" is the ideal.
    out["psma_low_status"] = np.select(
        [(out.psma_low_vs_high_delta >= 1.0) & (out.psma_low_frac_expressing >= 0.5),
         (out.psma_low_vs_high_delta > -1.0) & (out.psma_low_frac_expressing >= 0.5),
         out.psma_low_frac_expressing >= 0.5],
        ["enriched in PSMA-low", "retained in PSMA-low",
         "expressed but reduced in PSMA-low"],
        default="not retained in PSMA-low")
    out["suitable_for_psma_low"] = out.psma_low_status.isin(
        ["enriched in PSMA-low", "retained in PSMA-low"])

    for k, v in out.psma_low_status.value_counts().items():
        print(f"      {k}: {v:,}")
    # Sanity check worth printing: PSMA itself must separate the strata.
    own = out.loc[out.gene == benchmark, "psma_low_vs_high_delta"]
    if len(own):
        print(f"      control: {benchmark} itself differs by "
              f"{own.iloc[0]:+.2f} log2 between strata (should be strongly "
              f"negative by construction)")
    return out


# ============================================================================
# 2.9 MODALITY-SPECIFIC TARGET SUITABILITY  [NEW]
# ============================================================================
# RAISED IN EXTERNAL REVIEW: the pipeline applied ONE general-purpose
# normal-tissue comparator as a biological exclusion criterion, but acceptable
# normal-tissue exposure differs substantially by therapeutic modality:
#
#   ADC              surface density, internalisation, trafficking, payload
#                    sensitivity; liver and marrow exposure dominate
#   CAR-T            very low expression in essential normal cells; a single
#                    off-tumour hit is not recoverable, so the bar is highest
#   T-cell engager   accessible extracellular epitope, antigen density,
#                    off-tumour immune activation
#   radioligand      ligand binding and retention, absorbed organ dose;
#                    salivary gland and kidney dominate
#
# A candidate can therefore be plausible for one modality and clearly
# unsuitable for another, and reporting a single "safe / unsafe" verdict
# hides that. This computes a per-modality comparator and states, explicitly,
# what each modality still requires that no transcriptome can supply.
# ============================================================================
# External review, correct and worth stating in the artefact rather than
# only in a reply: the unattainability guard is mathematically sound, but
# CFG.modality_ratio_max is still an ASSUMED RNA-expression screening
# criterion, not an established biological safety boundary. No ratio of bulk
# transcript abundance defines a therapeutic window. The ceilings therefore
# need independent justification and a sensitivity analysis before any
# modality claim rests on them, and that limitation travels with every row
# of the modality audit.
MODALITY_THRESHOLD_PROVENANCE = (
    "ratio ceiling is an ASSUMED expression-screening criterion, not a "
    "validated safety boundary; requires independent justification and "
    "sensitivity analysis")

MODALITY_UNMEASURED_REQUIREMENTS = {
    "ADC": "surface density, internalisation rate, payload sensitivity",
    "CAR_T": "normal-cell expression at single-cell resolution, antigen "
             "coverage across malignant cells",
    "T_cell_engager": "epitope accessibility on intact cells, antigen density",
    "radioligand": "ligand binding affinity, tumour retention, absorbed "
                   "organ dose",
}


def modality_suitability(f, gtex_log, benchmark=CFG.benchmark_gene):
    """Per-modality normal-tissue comparator and an explicit evidence gap."""
    tpm = linear_tpm(gtex_log, label="GTEx (modality analysis)")
    if benchmark not in tpm.index:
        print("    modality analysis SKIPPED: benchmark gene absent")
        return f

    # BUG FIX (self-review): the modality screen matched tissue names
    # LITERALLY while safety_features() and detection_floor_sensitivity()
    # both resolve them through harmonise_tissue_labels(). One alias table,
    # applied in two of three normal-tissue comparators - the same
    # one-policy-applied-in-one-place defect class as the detection floor.
    #
    # The consequence was visible in the DEMO run and would be worse on LIVE
    # GTEx v8: the radioligand screen ran WITHOUT "Salivary Gland" (GTEx
    # labels it "Minor Salivary Gland") and still reported assessable=True
    # with 849 passing candidates. Salivary gland is the organ that defines
    # PSMA radioligand therapy's dose-limiting xerostomia, and the PSMA
    # comparator maximum was taken over the remaining tissues, so the
    # denominator was not "PSMA in this modality's risk tissues" at all.
    resolved_mod, _unresolved_mod = harmonise_tissue_labels(
        tpm.columns, verbose=False)
    rows = []
    for modality, tissues in CFG.modality_risk_tissues.items():
        present = [resolved_mod[t] for t in tissues if t in resolved_mod]
        missing = [t for t in tissues if t not in resolved_mod]
        # Coverage gate, matching the F2 policy. A ratio over a degraded
        # tissue set is not a safety screen for the modality as declared.
        mod_coverage = (len(present) / len(tissues)) if tissues else 0.0
        coverage_ok = mod_coverage >= CFG.min_vital_organ_coverage
        if present and not coverage_ok:
            f[f"passes_expression_screen_{modality}"] = False
            f[f"screen_assessable_{modality}"] = False
            f[f"screen_outcome_{modality}"] = "NOT ASSESSABLE"
            print(f"    {modality}: NOT ASSESSABLE - only "
                  f"{len(present)}/{len(tissues)} declared risk tissues are "
                  f"in the matrix ({mod_coverage:.0%} < "
                  f"{CFG.min_vital_organ_coverage:.0%}); missing {missing}. "
                  f"A ratio over a degraded organ set is not a safety screen "
                  f"for this modality as declared.")
            rows.append({
                "modality": modality,
                "risk_tissues_used": ", ".join(present),
                "risk_tissues_missing": ", ".join(missing) or "none",
                "tissue_coverage": round(mod_coverage, 3),
                "psma_reference_tpm_raw": np.nan,
                "psma_reference_tpm_censored": np.nan,
                "comparator_at_or_below_floor": np.nan,
                "ratio_ceiling": CFG.modality_ratio_max[modality],
                "min_possible_ratio": np.nan,
                "threshold_attainable": np.nan,
                "n_passing_expression_screen": np.nan,
                "n_failing_expression_screen": np.nan,
                "assessable": False,
                "why_not_assessable": (
                    f"only {len(present)}/{len(tissues)} declared risk "
                    f"tissues present ({mod_coverage:.0%}); missing "
                    f"{', '.join(missing)}"),
                "still_unmeasured": MODALITY_UNMEASURED_REQUIREMENTS[modality],
            "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
                "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
            })
            continue
        if not present:
            # BUG FIX (external review, correct): this branch printed a
            # notice and `continue`d WITHOUT appending an audit row, so a
            # modality whose entire organ panel was absent vanished from
            # table19 altogether. A reader comparing the table against
            # CFG.modality_risk_tissues would see a modality that looks
            # simply not requested, which is indistinguishable from one that
            # could not be assessed. Every configured modality now produces
            # exactly one row, whatever happened to it.
            f[f"passes_expression_screen_{modality}"] = False
            f[f"screen_assessable_{modality}"] = False
            f[f"screen_outcome_{modality}"] = "NOT ASSESSABLE"
            print(f"    {modality}: NOT ASSESSABLE - none of its "
                  f"{len(tissues)} declared risk tissues are in the matrix "
                  f"({missing})")
            rows.append({
                "modality": modality,
                "risk_tissues_used": "none",
                "risk_tissues_missing": ", ".join(missing) or "none",
                "tissue_coverage": 0.0,
                "psma_reference_tpm_raw": np.nan,
                "psma_reference_tpm_censored": np.nan,
                "comparator_at_or_below_floor": np.nan,
                "ratio_ceiling": CFG.modality_ratio_max[modality],
                "min_possible_ratio": np.nan,
                "threshold_attainable": np.nan,
                "n_passing_expression_screen": np.nan,
                "n_failing_expression_screen": np.nan,
                "assessable": False,
                "why_not_assessable": (
                    f"none of the {len(tissues)} declared risk tissues for "
                    f"this modality is present in the normal-tissue matrix"),
                "still_unmeasured": MODALITY_UNMEASURED_REQUIREMENTS[modality],
            "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
                "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
            })
            continue
        sub = tpm[present]
        bench_raw_mod = float(sub.loc[benchmark].max())
        bench = max(bench_raw_mod, CFG.vital_tpm_detection_floor, 1e-3)
        # CRITICAL BUG FIX (external review, correct - and a direct
        # consequence of the two-sided censoring added last round): when BOTH
        # the candidate maximum and the comparator are censored at the same
        # floor, the smallest achievable ratio is exactly 1.0. Any modality
        # whose ceiling is below 1.0 then becomes MATHEMATICALLY UNPASSABLE,
        # and its screen silently reported zero candidates - which reads as a
        # biological finding ("no gene is safe enough for CAR-T") when it is
        # an arithmetic artifact. CAR_T and T_cell_engager, both ceiling 0.5,
        # returned 0 for exactly this reason.
        #
        # A ratio test is only meaningful when the comparator is measured
        # above the floor. When it is not, the modality is recorded as NOT
        # ASSESSABLE rather than scored as all-fail.
        ceiling = CFG.modality_ratio_max[modality]
        # BUG FIX (external review, correct on both counts):
        #  (a) the policy says the comparator must be measured ABOVE the
        #      floor, but the test used >=, so a value exactly AT the floor -
        #      i.e. censored - counted as measurable;
        #  (b) the unassessable rule fired only when ceiling < 1.0, so ADC and
        #      radioligand kept reporting pass counts off a censored
        #      comparator. A ratio against a quantity that was not measurably
        #      expressed is uninterpretable whatever the ceiling is.
        # One rule now applies to every modality, and the table records the
        # RAW and CENSORED comparator separately so a reader can see which.
        comparator_measurable = bench_raw_mod > CFG.vital_tpm_detection_floor
        # CRITICAL BUG FIX (external review, correct, and the THIRD time this
        # defect class has appeared inside the previous round's own fix):
        # the guard above asked only whether the comparator sits AT OR BELOW
        # the floor, because that is the case where the minimum possible
        # ratio is exactly 1.0. But the general condition for an unattainable
        # screen is
        #
        #     floor / comparator > ceiling
        #
        # and T_cell_engager fell precisely in the gap the narrow guard left:
        # comparator 0.967 TPM is ABOVE the 0.5 floor, so the guard did not
        # fire, yet the smallest achievable ratio is 0.5/0.967 = 0.517
        # against a ceiling of 0.5. No gene could pass whatever its real
        # expression, and the table reported "assessable: True, 0 passing" -
        # which reads as the biological finding that no antigen in the
        # surfaceome is safe enough for a T-cell engager. It is an arithmetic
        # consequence of the floor-and-ceiling combination.
        #
        # Attainability is now tested directly, so the audit records WHY the
        # screen cannot discriminate and the passing count is NA rather than
        # a misleading zero. The scientific remedy is to reconsider how a
        # modality ceiling is defined relative to assay sensitivity - not to
        # lower the detection floor until candidates pass.
        _min_possible_ratio = (float(CFG.vital_tpm_detection_floor)
                               / bench if bench > 0 else np.inf)
        _threshold_attainable = _min_possible_ratio <= float(ceiling)
        if comparator_measurable and not _threshold_attainable:
            f[f"passes_expression_screen_{modality}"] = False
            f[f"screen_assessable_{modality}"] = False
            f[f"screen_outcome_{modality}"] = "NOT ASSESSABLE"
            print(f"    {modality}: NOT ASSESSABLE - the screen is "
                  f"mathematically unattainable under the configured "
                  f"thresholds. The {CFG.vital_tpm_detection_floor} TPM "
                  f"detection floor against a comparator of "
                  f"{bench_raw_mod:.3f} TPM gives a minimum possible ratio "
                  f"of {_min_possible_ratio:.3f}, above the "
                  f"{ceiling} ceiling, so NO gene can pass whatever its "
                  f"expression. Reporting zero candidates here would be an "
                  f"arithmetic artifact presented as a biological finding. "
                  f"Revisit the ceiling relative to assay sensitivity.")
            rows.append({
                "modality": modality,
                "risk_tissues_used": ", ".join(present),
                "risk_tissues_missing": ", ".join(missing) or "none",
                "tissue_coverage": round(mod_coverage, 3),
                "psma_reference_tpm_raw": round(bench_raw_mod, 3),
                "psma_reference_tpm_censored": round(bench, 3),
                "comparator_at_or_below_floor": False,
                "ratio_ceiling": ceiling,
                "min_possible_ratio": round(_min_possible_ratio, 4),
                "threshold_attainable": False,
                "n_passing_expression_screen": np.nan,
                "n_failing_expression_screen": np.nan,
                "assessable": False,
                "why_not_assessable": (
                    f"detection floor {CFG.vital_tpm_detection_floor} TPM "
                    f"against comparator {bench_raw_mod:.3f} TPM gives a "
                    f"minimum possible ratio of {_min_possible_ratio:.3f} > "
                    f"ceiling {ceiling}; the screen cannot discriminate"),
                "still_unmeasured": MODALITY_UNMEASURED_REQUIREMENTS[modality],
            "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
                "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
            })
            continue
        if not comparator_measurable:
            f[f"passes_expression_screen_{modality}"] = False
            f[f"screen_assessable_{modality}"] = False
            f[f"screen_outcome_{modality}"] = "NOT ASSESSABLE"
            print(f"    {modality}: NOT ASSESSABLE - the comparator "
                  f"({benchmark}) has a maximum of {bench_raw_mod:.3f} TPM on "
                  f"this modality's risk tissues, at or below the "
                  f"{CFG.vital_tpm_detection_floor} TPM floor. With both sides "
                  f"censored the minimum possible ratio is 1.0, so a ceiling "
                  f"of {ceiling} cannot be met by ANY gene. Reporting zero "
                  f"candidates here would be an arithmetic artifact, not a "
                  f"safety finding. Validate the comparator on LIVE data.")
            rows.append({
                "modality": modality,
                "risk_tissues_used": ", ".join(present),
                "risk_tissues_missing": ", ".join(missing) or "none",
                "tissue_coverage": round(mod_coverage, 3),
                "psma_reference_tpm_raw": round(bench_raw_mod, 3),
                "psma_reference_tpm_censored": round(bench, 3),
                "comparator_at_or_below_floor": True,
                "ratio_ceiling": ceiling,
                "min_possible_ratio": round(_min_possible_ratio, 4),
                "threshold_attainable": False,
                "n_passing_expression_screen": np.nan,
                "n_failing_expression_screen": np.nan,
                "assessable": False,
                "why_not_assessable": (
                    f"comparator ({benchmark}) at or below the "
                    f"{CFG.vital_tpm_detection_floor} TPM detection floor "
                    f"(raw {bench_raw_mod:.3f}); a ratio against a quantity "
                    f"that was not measurably expressed is uninterpretable, "
                    f"and with both sides censored the minimum ratio is 1.0"),
                "still_unmeasured": MODALITY_UNMEASURED_REQUIREMENTS[modality],
            "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
                "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
            })
            continue
        # BUG FIX (external review, correct): the detection-floor censoring
        # added to safety_features() was NOT applied here, so a gene with an
        # undetectable bulk measurement in this modality's risk tissues got a
        # ratio of exactly 0 and passed the modality screen automatically -
        # the same unbounded-window artefact the floor was introduced to
        # remove, left in place on the per-modality path. One floor policy,
        # applied in every normal-tissue comparator.
        floor = float(CFG.vital_tpm_detection_floor)
        raw_mod_max = sub.max(axis=1)
        mod_max = np.maximum(raw_mod_max, floor)
        ratio = (mod_max / bench).reindex(f.gene).values
        f[f"at_detection_floor_{modality}"] = (
            (raw_mod_max < floor).reindex(f.gene).fillna(False).values)
        f[f"screen_assessable_{modality}"] = True
        col = f"ratio_{modality}"
        f[col] = ratio
        ok = np.isfinite(ratio) & (ratio >= 0) & (ratio <= ceiling)
        # BUG FIX: an intracellular-membrane protein is not reachable by ANY
        # of these modalities, so it can never pass a modality screen.
        intra = (f.get("intracellular_excluded_flag",
                       pd.Series(False, index=f.index))
                 .fillna(False).astype(bool).values)
        _pass = ok & f.F3_topology.fillna(False).values & ~intra
        f[f"passes_expression_screen_{modality}"] = _pass
        # Three-state outcome, so "could not be evaluated" is never recorded
        # as "failed" anywhere downstream.
        f[f"screen_outcome_{modality}"] = np.where(_pass, "PASS", "FAIL")
        # Back-compatible alias, explicitly NOT a suitability claim. The screen
        # uses normal-tissue RNA ratio + topology ONLY; every measurement that
        # would actually establish modality suitability (surface density,
        # internalisation, epitope accessibility, absorbed dose) is unmeasured
        # here and listed in `still_unmeasured`.
        f[f"suitable_{modality}"] = f[f"passes_expression_screen_{modality}"]
        worst = sub.idxmax(axis=1).reindex(f.gene).values
        f[f"worst_tissue_{modality}"] = worst
        rows.append({
            "modality": modality,
            "risk_tissues_used": ", ".join(present),
            "risk_tissues_missing": ", ".join(missing) or "none",
            "tissue_coverage": round(mod_coverage, 3),
            "psma_reference_tpm_raw": round(bench_raw_mod, 3),
            "psma_reference_tpm_censored": round(bench, 3),
            "comparator_at_or_below_floor": False,
            "ratio_ceiling": ceiling,
            "min_possible_ratio": round(_min_possible_ratio, 4),
            "threshold_attainable": True,
            "n_passing_expression_screen": int(
                f[f"passes_expression_screen_{modality}"].sum()),
            "n_failing_expression_screen": int(
                (~f[f"passes_expression_screen_{modality}"]).sum()),
            "assessable": True,
            "why_not_assessable": "",
            "still_unmeasured": MODALITY_UNMEASURED_REQUIREMENTS[modality],
            "threshold_provenance": MODALITY_THRESHOLD_PROVENANCE,
        })
        print(f"    {modality}: "
              f"{int(f[f'passes_expression_screen_{modality}'].sum()):,} "
              f"pass the EXPRESSION SCREEN at or below {ceiling}x PSMA on "
              f"{len(present)} risk tissue(s)"
              + (f"; MISSING tissues: {missing}" if missing else ""))

    if rows:
        mod = pd.DataFrame(rows)
        save_table(mod, "table19_modality_expression_screen.csv",
                   caption="Per-modality normal-tissue RNA comparator plus "
                           "topology - a PRELIMINARY EXPRESSION-BASED SCREEN, "
                           "not a modality suitability determination. The "
                           "measurements each modality still requires are "
                           "listed in the final column.")
        # BUG FIX (found reviewing this round's own change): making an
        # unassessable modality return False fixed its own count but then
        # propagated the SAME conflation one level up. The profile compared
        # `n_passed` against the number of modality COLUMNS, and the
        # unassessable modalities still had columns - so with two of four
        # unassessable, "all modalities" became unreachable by construction,
        # and 418 genes that passed BOTH assessable screens were labelled
        # "modality-dependent", asserting a constraint that was never tested.
        #
        # The denominator is now the number of ASSESSABLE modalities, and the
        # vocabulary distinguishes "not assessable" from "fails".
        assessable = [m for m in CFG.modality_risk_tissues
                      if f.get(f"screen_assessable_{m}",
                               pd.Series(False, index=f.index)).fillna(False).all()]
        unassessable = [m for m in CFG.modality_risk_tissues
                        if m not in assessable
                        and f"passes_expression_screen_{m}" in f.columns]
        cols = [f"passes_expression_screen_{m}" for m in assessable
                if f"passes_expression_screen_{m}" in f.columns]
        if unassessable:
            print(f"    {len(unassessable)} modality screen(s) NOT ASSESSABLE "
                  f"({', '.join(unassessable)}) are EXCLUDED from the modality "
                  f"profile denominator: a screen that could not be evaluated "
                  f"is not a screen the gene failed")
        if cols:
            n_mod = f[cols].sum(axis=1)
            n_assessable = len(cols)
            f["n_modalities_passing_screen"] = n_mod
            f["n_modalities_failing_screen"] = n_assessable - n_mod
            f["n_modalities_assessable"] = n_assessable
            f["n_modalities_not_assessable"] = len(unassessable)
            f["n_modalities_suitable"] = n_mod   # back-compatible alias
            f["modalities_not_assessable"] = ", ".join(unassessable) or "none"
            suffix = (f" (of {n_assessable} assessable; "
                      f"{len(unassessable)} not assessable)"
                      if unassessable else "")
            # "no assessable modality" - never "no modality" - because two
            # screens could not be evaluated and a gene failing the two that
            # could is not a gene with no therapeutic route.
            f["modality_profile"] = np.where(
                n_mod == 0, "no assessable modality passed" + suffix,
                np.where(n_mod == n_assessable,
                         "all assessable modalities" + suffix,
                         "modality-dependent" + suffix))
            print(f"    candidates passing the screen for all "
                  f"{n_assessable} ASSESSABLE "
                  f"modalities: "
                  f"{int((n_mod == n_assessable).sum()):,} | for none: "
                  f"{int((n_mod == 0).sum()):,} | modality-dependent: "
                  f"{int(((n_mod > 0) & (n_mod < n_assessable)).sum()):,}")
            print("      A modality-dependent candidate is not a weaker "
                  "candidate - it is one whose development route is "
                  "constrained, which is actionable information.")
        else:
            # BUG FIX (found by running the corrected code): when NO modality
            # is assessable the column list is empty, so every modality field
            # silently disappeared from the ranking. A reader then sees no
            # modality information at all and no reason why - strictly worse
            # than being told it could not be assessed. The three-state
            # vocabulary is written explicitly instead.
            f["n_modalities_passing_screen"] = 0
            f["n_modalities_failing_screen"] = 0
            f["n_modalities_assessable"] = 0
            f["n_modalities_not_assessable"] = len(unassessable)
            f["n_modalities_suitable"] = 0
            f["modalities_not_assessable"] = ", ".join(unassessable) or "none"
            f["modality_profile"] = "NOT ASSESSABLE - no modality screen "\
                                    "could be evaluated"
            print(f"    NO modality screen was assessable "
                  f"({', '.join(unassessable)}). Every modality field is set "
                  f"to NOT ASSESSABLE rather than omitted: absence of a "
                  f"column would read as absence of risk.")
    return f


# ----------------------------------------------------------------------------
# 2.9 LINEAGE-PLASTICITY AXIS (NEW - raised in external review)
# ----------------------------------------------------------------------------
# External review, correctly: defining PSMA-low from the lowest quartile of
# bulk FOLH1 RNA does not identify patients who are clinically PSMA-negative
# or PSMA-PET-ineligible, and the study's clinical premise rests on several
# DISTINCT resistance phenotypes, not one. Lineage plasticity - loss of AR
# signalling, with or without neuroendocrine transdifferentiation - is the
# dominant route to surface-antigen loss after androgen-receptor pathway
# inhibition, and a target that vanishes in AR-low disease does not address
# the gap the proposal is written against.
#
# This adds a SECOND, independent persistence axis on AR signalling, built the
# same way as the PSMA axis and with the same disjointness guards, so a
# candidate can be assessed across the phenotypes that actually matter:
#   AR-positive adenocarcinoma / AR-low or double-negative / NEPC / PSMA-low.
#
# It remains an RNA-based molecular stratification, NOT a clinical phenotype.
# That limitation is stated in the output caption rather than left implicit.
def lineage_axis_persistence(mcrpc_expr, axis_gene="AR", axis_name="AR",
                             low_quantile=0.25, nepc_mask=None):
    """Per-gene expression in axis-low vs axis-high metastatic tumours.

    Mirrors psma_low_persistence() for an arbitrary lineage marker (AR by
    default) so target persistence can be assessed across the resistance
    phenotypes, not only across PSMA strata.
    """
    if axis_gene not in mcrpc_expr.index:
        print(f"    {axis_name}-axis analysis SKIPPED: {axis_gene} not "
              f"measured in the metastatic cohort. Recorded as not performed.")
        return None

    v = mcrpc_expr.loc[axis_gene]
    thr_lo, thr_hi = v.quantile(low_quantile), v.quantile(1 - low_quantile)
    low, high = v <= thr_lo, v >= thr_hi
    if int(low.sum()) < 5 or int(high.sum()) < 5:
        print(f"    {axis_name}-axis analysis SKIPPED: strata too small "
              f"(low n={int(low.sum())}, high n={int(high.sum())}).")
        return None
    # Same disjointness and separation guards as the PSMA axis: a tied marker
    # would otherwise compare a group against itself and return a null result
    # that looks biological.
    overlap = int((low & high).sum())
    if overlap or not (thr_hi > thr_lo):
        print(f"    {axis_name}-axis analysis SKIPPED: strata NOT disjoint "
              f"(overlap n={overlap}). Recorded as not performed.")
        return None
    sep = float(thr_hi - thr_lo)
    if sep < CFG.min_psma_strata_separation_log2:
        print(f"    {axis_name}-axis analysis SKIPPED: strata separated by "
              f"only {sep:.3f} log2 units.")
        return None

    lo, hi = mcrpc_expr.loc[:, low].values, mcrpc_expr.loc[:, high].values
    _, p = st.ttest_ind(lo, hi, axis=1, equal_var=False, nan_policy="omit")
    p = np.nan_to_num(np.asarray(p, dtype=float), nan=1.0)
    delta = lo.mean(axis=1) - hi.mean(axis=1)
    pref = axis_name.lower()
    out = pd.DataFrame({
        "gene": mcrpc_expr.index,
        f"{pref}_low_mean_log2": lo.mean(axis=1),
        f"{pref}_high_mean_log2": hi.mean(axis=1),
        f"{pref}_low_vs_high_delta": delta,
        f"{pref}_low_vs_high_qval": multipletests(p, method="fdr_bh")[1],
        f"{pref}_low_frac_expressing": (
            linear_tpm(mcrpc_expr.loc[:, low],
                       label=f"mCRPC {axis_name}-low")
            >= CFG.min_expr_tpm).mean(axis=1).values,
    })
    # "Retained" = still expressed in a majority of axis-low tumours AND not
    # significantly lower than in axis-high disease. Both conditions, because
    # a target can be nominally expressed and still be strongly depleted.
    out[f"retained_in_{pref}_low"] = (
        (out[f"{pref}_low_frac_expressing"] >= CFG.min_frac_expressing)
        & ~((out[f"{pref}_low_vs_high_delta"] < -1.0)
            & (out[f"{pref}_low_vs_high_qval"] < CFG.fdr_alpha)))
    n_ret = int(out[f"retained_in_{pref}_low"].sum())
    print(f"    {axis_name}-axis strata: low n={int(low.sum())}, "
          f"high n={int(high.sum())}, separation {sep:.3f} log2 units -> "
          f"{n_ret:,} gene(s) retained in {axis_name}-low disease")
    save_table(out, f"table20_{pref}_low_persistence.csv",
               caption=f"Target persistence in {axis_name}-low vs "
                       f"{axis_name}-high metastatic disease. RNA-based "
                       f"molecular stratification on {axis_gene} expression "
                       f"- NOT a clinical phenotype and not equivalent to "
                       f"clinically confirmed {axis_name}-pathway-independent "
                       f"disease.")
    return out


# ----------------------------------------------------------------------------
# 2.10 DETECTION-FLOOR SENSITIVITY (NEW - raised in external review)
# ----------------------------------------------------------------------------
# External review, correctly: the 0.5 TPM floor is a MODELLING ASSUMPTION, not
# a measured detection limit for every dataset, tissue and cell type. A filter
# whose output depends on an unvalidated constant must report how much it
# depends on it, otherwise the constant is doing scientific work invisibly.
def detection_floor_sensitivity(gtex_log, f, benchmark=CFG.benchmark_gene,
                                floors=(0.0, 0.1, 0.5, 1.0, 2.0)):
    """Re-derive Filter 2 across plausible normal-tissue detection floors."""
    tpm = linear_tpm(gtex_log, label="GTEx (floor sensitivity)")
    # Harmonise labels BEFORE selecting organs: GTEx v8 calls the relevant
    # salivary tissue "Minor Salivary Gland", so a literal name match drops
    # the organ behind PSMA radioligand xerostomia without failing anything.
    _resolved, _missing_organs = harmonise_tissue_labels(tpm.columns)
    vital = list(dict.fromkeys(_resolved.values()))
    _coverage = len(_resolved) / max(len(set(CFG.vital_organs)), 1)
    if not vital or benchmark not in tpm.index:
        print("    detection-floor sensitivity SKIPPED: comparator unavailable")
        return None
    vt = tpm[vital]
    bench_raw = float(vt.loc[benchmark].max())
    raw = vt.max(axis=1).reindex(f.gene)
    base_genes = None
    rows = []
    for fl in floors:
        # Censor BOTH sides at the same floor - the defect this analysis found.
        bench = max(bench_raw, fl, 1e-3)
        ratio = np.maximum(raw.values, fl) / bench
        passes = np.isfinite(ratio) & (ratio >= 0) & (ratio <= CFG.safety_ratio_max)
        genes = set(f.gene[passes])
        if base_genes is None:
            base_genes = genes if fl == CFG.vital_tpm_detection_floor else None
        rows.append({"detection_floor_tpm": fl,
                     "n_passing_F2": int(passes.sum()),
                     "n_at_or_below_floor": int((raw.values < fl).sum()),
                     "is_pipeline_default":
                         bool(abs(fl - CFG.vital_tpm_detection_floor) < 1e-9)})
    out = pd.DataFrame(rows)
    # SELF-REVIEW, found by a static check flagging _coverage as computed and
    # discarded. The MAIN F2 screen records organ coverage, gates
    # f2_comparator_assessable on it, and prints NOT DEFINITIVE below the
    # threshold. This sensitivity analysis computed the same fraction and
    # threw it away, so its table said nothing about the panel it rests on -
    # a reader could take its ranges as firm while the main screen had
    # already declared the comparator non-definitive on the same data. The
    # numbers were never wrong; the table was not self-describing.
    out["vital_organ_coverage"] = round(float(_coverage), 3)
    out["organ_coverage_adequate"] = bool(
        _coverage >= CFG.min_vital_organ_coverage)
    out["organs_missing"] = ("; ".join(sorted(_missing_organs))
                             if _missing_organs else "none")
    if _coverage < CFG.min_vital_organ_coverage:
        print(f"    NOTE: this sensitivity analysis rests on "
              f"{_coverage:.0%} of the declared vital organs "
              f"({CFG.min_vital_organ_coverage:.0%} required for a "
              f"definitive F2 call), missing "
              f"{'; '.join(sorted(_missing_organs))}. Its ranges are "
              f"conditional on that incomplete panel.")
    # How much does the eligible set move between the loosest and strictest
    # floor? That number is the honest statement of how assumption-dependent
    # Filter 2 is.
    lo, hi = out.n_passing_F2.max(), out.n_passing_F2.min()
    span_pct = 100 * (lo - hi) / max(lo, 1)
    out["pct_change_vs_loosest"] = (
        100 * (out.n_passing_F2 - lo) / max(lo, 1)).round(1)
    save_table(out, "table23_detection_floor_sensitivity.csv",
               caption=f"Filter 2 eligibility across plausible normal-tissue "
                       f"detection floors. The floor is an assumption, not a "
                       f"measured limit; the eligible set varies by "
                       f"{span_pct:.1f}% across this range.")
    print(f"    detection-floor sensitivity: F2 passers range "
          f"{hi:,}-{lo:,} across floors {min(floors)}-{max(floors)} TPM "
          f"({span_pct:.1f}% variation). Report this range alongside any "
          f"eligibility count.")
    return {"floors": list(floors),
            "n_passing_by_floor": dict(zip(out.detection_floor_tpm,
                                           out.n_passing_F2)),
            "pct_variation_across_floors": round(float(span_pct), 1),
            "default_floor": CFG.vital_tpm_detection_floor,
            "vital_organ_coverage": round(float(_coverage), 3),
            "organ_coverage_adequate": bool(
                _coverage >= CFG.min_vital_organ_coverage),
            "organs_missing": sorted(_missing_organs)}


# ----------------------------------------------------------------------------
# 2.11 EPITHELIAL RESTRICTION (NEW - clinical review)
# ----------------------------------------------------------------------------
# CLINICAL DEFECT this addresses: the screen compares BULK tumour against BULK
# adjacent normal, with no tumour-purity adjustment anywhere. Antigens carried
# mainly by INFILTRATING IMMUNE CELLS therefore appear "tumour-upregulated"
# because tumours are more inflamed than benign prostate - not because cancer
# cells display them. TCGA-PRAD purity runs roughly 0.6-0.7, so a meaningful
# share of every bulk measurement is stroma and leukocyte.
#
# Nominating such an antigen as a cancer-cell surface target is not a ranking
# error - the resulting therapy would deplete the immune infiltrate.
#
# Without per-sample deconvolution this cannot be resolved exactly, but it can
# be MEASURED: across tumours, a cancer-cell antigen co-varies with epithelial
# markers, while a leukocyte antigen co-varies with immune markers. The
# difference between those two correlations is a per-gene compartment score.
EPITHELIAL_MARKERS = ["EPCAM", "KRT8", "KRT18", "CDH1", "CLDN3", "KRT19"]
IMMUNE_MARKERS = ["PTPRC", "CD3E", "CD2", "LCK", "CD53", "ITGAM", "CSF1R"]
STROMAL_MARKERS = ["COL1A1", "COL1A2", "ACTA2", "PDGFRB", "THY1"]


def epithelial_restriction(tumour_log, f):
    """Per-gene compartment score: epithelial vs immune/stromal co-expression.

    Returns a frame to merge on `gene`. Positive `epithelial_restriction_score`
    means the gene tracks epithelial content; negative means it tracks the
    immune/stromal compartment and is a likely infiltrate antigen.
    """
    def panel_mean(markers):
        present = [m for m in dict.fromkeys(markers) if m in tumour_log.index]
        if len(present) < 2:
            return None, present
        return tumour_log.loc[present].mean(axis=0), present

    epi, epi_used = panel_mean(EPITHELIAL_MARKERS)
    imm, imm_used = panel_mean(IMMUNE_MARKERS)
    strom, strom_used = panel_mean(STROMAL_MARKERS)
    if epi is None or imm is None:
        print("    epithelial-restriction SKIPPED: marker panels not "
              "measurable in this matrix (epithelial="
              f"{len(epi_used)}, immune={len(imm_used)}). Compartment "
              "confounding therefore remains UNASSESSED - state it.")
        return None

    X = tumour_log.reindex(f.gene).values
    def corr_with(v):
        vv = np.asarray(v, dtype=float)
        vc = vv - vv.mean()
        Xc = X - np.nanmean(X, axis=1, keepdims=True)
        num = np.nansum(Xc * vc, axis=1)
        den = (np.sqrt(np.nansum(Xc ** 2, axis=1)) * np.sqrt(np.sum(vc ** 2)))
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(den > 0, num / den, np.nan)

    r_epi, r_imm = corr_with(epi), corr_with(imm)
    # np.nanmax over an all-NaN column emits a RuntimeWarning and returns NaN,
    # which would silently turn the compartment score into NaN for every gene
    # when the stromal panel is unavailable. Stack only panels that exist.
    stack = [r_imm]
    if strom is not None:
        stack.append(corr_with(strom))
        r_strom = stack[-1]
    else:
        r_strom = np.full(len(f), np.nan)
    arr = np.vstack(stack)
    with np.errstate(invalid="ignore"):
        non_epi = np.where(np.isnan(arr).all(axis=0), np.nan,
                           np.nanmax(np.where(np.isnan(arr), -np.inf, arr),
                                     axis=0))
    out = pd.DataFrame({
        "gene": f.gene.values,
        "corr_epithelial_panel": r_epi,
        "corr_immune_panel": r_imm,
        "corr_stromal_panel": r_strom,
        "epithelial_restriction_score": r_epi - non_epi,
    })
    # EXTERNAL REVIEW, correct: in the previous run all five flagged genes
    # (CD2, CSF1R, LCK, CD53, PTPRC) were themselves members of the immune
    # panel used to BUILD the score. That demonstrates the score can recognise
    # its own inputs - it is not independent evidence that the score detects
    # infiltrating-cell antigens. Panel members are now excluded from the flag
    # and reported separately as a positive control: if the construction
    # markers do NOT score as immune-tracking, the score is broken.
    panel_members = (set(EPITHELIAL_MARKERS) | set(IMMUNE_MARKERS)
                     | set(STROMAL_MARKERS))
    out["is_panel_marker"] = out.gene.isin(panel_members)
    _flag = ((out.epithelial_restriction_score < -0.2)
             & (out.corr_immune_panel > 0.3))
    out["panel_positive_control_recovered"] = _flag & out.is_panel_marker
    out["likely_infiltrate_antigen"] = _flag & ~out.is_panel_marker
    n_ctrl = int(out.panel_positive_control_recovered.sum())
    n_imm_panel = len(set(IMMUNE_MARKERS) & set(out.gene))
    print(f"    positive control: {n_ctrl}/{n_imm_panel} immune-panel "
          f"construction markers score as immune-tracking, as they must; "
          f"they are EXCLUDED from the infiltrate flag because recognising "
          f"its own inputs is not independent evidence")
    n_inf = int(out.likely_infiltrate_antigen.sum())
    print(f"    epithelial restriction scored on {len(epi_used)} epithelial / "
          f"{len(imm_used)} immune markers; {n_inf:,} gene(s) flagged as "
          f"LIKELY INFILTRATE ANTIGENS (they track the immune compartment, "
          f"so bulk tumour-vs-normal enrichment may reflect inflammation "
          f"rather than cancer-cell display)")
    save_table(out.sort_values("epithelial_restriction_score"),
               "table25_epithelial_restriction.csv",
               caption="Per-gene compartment score. A negative score means "
                       "the gene tracks immune/stromal rather than epithelial "
                       "content - bulk enrichment may not reflect cancer-cell "
                       "surface display. Not a substitute for deconvolution "
                       "or single-cell data.")
    return out

---
## Section 3 — Machine learning

Read the header block in the cell below before interpreting any number this section produces. The short version:

The proposed design — *train a classifier on 60–100 clinical-stage targets* — is not standard supervised classification. It is **positive-unlabeled learning with a very small positive set**, and three things follow from that:

1. **The unlabeled genes are not negatives.** They contain every target that is good but not yet drugged — exactly what the study is hunting. Training against them as negatives teaches the model to reject its own discoveries. Hence bagged PU learning with repeated subsampling.
2. **A single AUC is uninformative at this sample size.** With fewer than 100 positives, one train/test split gives a 95% CI roughly ±0.10 wide. Hence 10×5 repeated nested CV with bootstrap intervals, and a label-permutation null so the AUC is read against what this design yields on noise rather than against 0.5.
3. **"Is it a drug target?" is confounded with "has it been studied?"** Antibody counts and annotation depth are proxies for research attention. A model fed those features scores well by learning fame. Hence the declared leakage ablation: the model is fitted twice, with and without research-intensity features, and both are reported.

**The primary endpoint is not cross-validated AUC.** It is where **six eligible** held-out prostate benchmarks — never seen in training — rank among ~2,800 surfaceome genes, with uncertainty and a permutation comparison. That is the only internal evaluation here that is not circular.

*KLK3 is an exclusion control, not a seventh benchmark.* PSA is secreted, so it must be removed at surfaceome bounding and cannot be recovered by design. Counting it as an evaluable benchmark would silently depress the recovery rate and misrepresent the endpoint, so the two sets are declared separately.

**A poor PSMA rank triggers a documented failure investigation**, not a silent exclusion and not a conclusion that the framework is invalid. The stop/go gate names the four things to check: symbol mapping through the merge, absence from the training positives, the gene's raw fold-change and vital-organ profile, and whether the normal reference is appropriate.

**Treat ROC-AUC and AUPRC as PU proxy discrimination metrics**, computed against pseudo-negatives — not as clinical target-identification accuracy. Unlabelled genes scoring highly may be discoveries rather than false positives, which is precisely the point of the design. Precision@K is reported because it matches how the output is actually used: a wet-lab team takes the top K to the bench.

**Feature groups are kept independent.** `therapeutic_window_log2` is computed from `tumour_mean_log2` and `log_safety_ratio`, so including all three made the elastic-net coefficients uninterpretable — in testing the composite took a *negative* coefficient, reading as "a wider therapeutic window is bad". The components are modelled; the composite is reported only.

**Ranking stability is measured, not assumed** (3.8). The model is re-scored under alternative seeds and pseudo-negative ratios, and the pipeline reports rank correlation and top-K overlap. If the median Spearman ρ falls below 0.8, report rank intervals rather than a point-ranked top-20.

**External transfer uses a frozen model** (3.9). Internal benchmark enrichment tests whether the model recovers known biology; it does not test whether the ranking transfers to a cohort processed elsewhere. Only scoring an independent cohort with the frozen model does that.

In [5]:
# =============================================================================
# SECTION 3 - MACHINE LEARNING: FEATURE SELECTION, REGULARISATION, ENSEMBLE
# =============================================================================
# THE CENTRAL STATISTICAL PROBLEM, STATED PLAINLY
#
# The proposed design - "train a classifier on 60-100 clinical-stage targets" -
# is NOT a standard supervised classification problem. It is positive-unlabeled
# (PU) learning with a very small positive set:
#
#   * There are ~60-80 positives and ~2,700 unlabeled genes.
#   * The unlabeled set is NOT a negative set. It contains every target that is
#     good but not yet drugged - which is precisely what the study is hunting.
#     Calling them negatives trains the model to reject its own discoveries.
#   * With n_pos < 100, a single train/test split gives an AUC with a 95% CI
#     roughly +/-0.10 wide. A single reported AUC is therefore uninformative.
#   * "Is it a known drug target?" is confounded with "has it been studied?".
#     Antibody counts, publication-driven annotation depth and even HPA
#     membrane calls are all proxies for research attention. A model fed those
#     features will achieve a high AUC by learning fame, not biology.
#
# WHAT THIS CODE DOES ABOUT IT
#   1. Bagged PU learning: unlabeled genes are repeatedly subsampled as
#      pseudo-negatives; predictions are averaged over bags.
#   2. Repeated stratified nested CV (10 x 5 outer, 3 inner) so the reported
#      performance carries an honest variance estimate.
#   3. A label-permutation null, so the AUC is interpreted against what this
#      design yields on noise rather than against 0.5.
#   4. A declared leakage ablation: the model is fit twice, with and without
#      research-intensity features, and both are reported.
#   5. The primary endpoint is NOT cross-validated AUC. It is the ranking of
#      seven held-out prostate benchmarks that the model never saw.
# =============================================================================

import re
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (RepeatedStratifiedKFold, GridSearchCV,
                                     StratifiedKFold)
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve,
                             brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance


# ----------------------------------------------------------------------------
# 3.0 VERSION-AGNOSTIC ELASTIC-NET BUILDER
# ----------------------------------------------------------------------------
# BUG FIX (critical, found by execution on Colab): scikit-learn's handling of
# elastic-net logistic regression differs across the versions in active use,
# and the pipeline crashed on Colab while running clean locally.
#
#   sklearn <= 1.7 : LogisticRegression(penalty="elasticnet", l1_ratio=<float>)
#                    REQUIRES l1_ratio; passing None raises
#                    "l1_ratio must be specified when penalty is elasticnet."
#   sklearn >= 1.8 : `penalty` is deprecated in favour of `l1_ratio` alone and
#                    emits a FutureWarning; it is REMOVED in 1.10.
#
# Code written against either one alone breaks on the other. This builder
# constructs the right estimator for the installed version and always supplies
# an explicit l1_ratio, so no call site can forget it.
# ----------------------------------------------------------------------------
from sklearn import __version__ as _SKL_VERSION

def _skl_at_least(major, minor):
    try:
        parts = _SKL_VERSION.split(".")
        return (int(parts[0]), int(parts[1])) >= (major, minor)
    except Exception:
        return False

_SKL_NEW_PENALTY_API = _skl_at_least(1, 8)


def make_elasticnet(C=1.0, l1_ratio=0.5, seed=CFG.seed, max_iter=8000):
    """Elastic-net logistic regression that works on every supported sklearn.

    l1_ratio is ALWAYS explicit. 0 is ridge, 1 is lasso, in between is a mix.
    """
    kw = dict(solver="saga", max_iter=max_iter, class_weight="balanced",
              random_state=seed, C=C, l1_ratio=l1_ratio)
    if not _SKL_NEW_PENALTY_API:
        kw["penalty"] = "elasticnet"
    return LogisticRegression(**kw)


def enet_param_grid():
    """Hyperparameter grid, sized by the active compute profile.

    The saga solver dominates runtime, and grid size multiplies directly into
    it: a 15-point grid inside nested CV is 15x the inner-loop cost of a
    1-point one. The compact grid keeps the span of regularisation strengths
    that matters and drops the redundant intermediate points.
    """
    if CFG.enet_grid_size == "compact":
        return {"clf__C": [0.05, 1.0], "clf__l1_ratio": [0.5]}
    return {"clf__C": [0.01, 0.05, 0.2, 1.0, 5.0],
            "clf__l1_ratio": [0.1, 0.5, 0.9]}



def permutation_p(null_values, observed, n_resamples=None):
    """Monte-Carlo permutation p-value with the (b+1)/(B+1) correction.

    BUG FIX (raised in external review): three call sites computed a raw
    exceedance fraction, which reports p = 0 when no draw exceeds the observed
    statistic. A finite resampling scheme cannot establish a zero probability;
    the smallest p it can support is 1/(B+1). The correction is standard
    (Davison & Hinkley; North et al.) and also keeps the test valid rather
    than anti-conservative.

    Returns the p-value and the Monte-Carlo resolution, so a reported p can be
    read against the smallest value the design could have produced.
    """
    nulls = np.asarray(null_values, dtype=float)
    nulls = nulls[np.isfinite(nulls)]
    B = len(nulls) if n_resamples is None else int(n_resamples)
    if B == 0 or not np.isfinite(observed):
        return {"p": np.nan, "n_resamples": B, "resolution": np.nan,
                "p_is_at_resolution_floor": False}
    b = int((nulls >= observed).sum())
    p = (b + 1) / (B + 1)
    res = 1.0 / (B + 1)
    return {"p": float(p), "n_resamples": B, "resolution": float(res),
            "n_exceeding": b,
            "p_is_at_resolution_floor": bool(b == 0)}


# ----------------------------------------------------------------------------
# 3.1 Feature matrix definition
# ----------------------------------------------------------------------------
# Features are grouped so that the leakage ablation is a one-line change
# rather than a hand-edited list.

# PROPOSAL REFINEMENT (Step 3): features are organised into INDEPENDENT
# GROUPS, and redundant composite/component pairs are avoided so that
# coefficients remain interpretable.
#
# RESOLVED COLLINEARITY: `therapeutic_window_log2` is tumour_mean_log2 minus
# log(vital_max), so including it alongside both of its own components made
# the elastic-net coefficients uninterpretable - in testing it took a NEGATIVE
# coefficient, which reads as "a wider therapeutic window is bad". The
# components are kept (they are measured quantities); the composite is dropped
# from the model and retained in the output tables for reporting only.
FEATURE_GROUPS = {
    "tumour_enrichment":   ["rna_log2fc", "neg_log10_rna_q", "cohen_d"],
    "prevalence":          ["tumour_mean_log2", "frac_tumours_expressing"],
    "normal_distribution": ["log_safety_ratio", "vital_n_organs_over_5tpm",
                            "normal_breadth_n_tissues_over_5tpm",
                            "log_prostate_specificity"],
    "metastatic_persistence": ["mcrpc_mean_log2", "mcrpc_frac_expressing",
                               "primary_to_mcrpc_delta", "expressed_in_both",
                               "mcrpc_bimodality_coefficient", "mcrpc_log_sd"],
    "extracellular_accessibility": ["n_transmem", "log_ecd_len",
                                    "topology_score", "seq_len"],
    # Compartment: does the gene track epithelial or immune/stromal content?
    # Added so the model can learn that an infiltrate antigen is not a
    # cancer-cell surface target (clinical review).
    "cellular_compartment": ["epithelial_restriction_score"],
}
BIOLOGY_FEATURES = [f for grp in FEATURE_GROUPS.values() for f in grp]

# Reported in tables, deliberately NOT given to the model (see above).
REPORTING_ONLY_FEATURES = ["therapeutic_window_log2"]

# Features that encode HOW MUCH A GENE HAS BEEN STUDIED rather than what it is.
# These are the leakage risk. They are informative for the stated practical
# purpose (a target with no reagents is hard to develop) but they make any
# claim of "biological discovery" circular.
RESEARCH_INTENSITY_FEATURES = [
    "log_n_antibodies",
]

# NEW (found in our own review of the exported feature importances): a THIRD
# ablation arm, because the existing biology/research-intensity split does not
# catch the dominant source of circularity.
#
# In the last run `log_ecd_len` was the single most important feature (0.076),
# eight times `rna_log2fc` (0.0095) and ~180x `neg_log10_rna_q` (0.0004), and
# extracellular-domain length ALONE separated the training positives from
# background at AUC 0.631 (p = 1.2e-4). That is not prostate cancer biology.
# Clinical-stage surface targets are drawn disproportionately from large
# single-pass receptor families, because those are the proteins against which
# binders are practical to raise - so ECD length is a PRIOR ON THE LABEL SET,
# a property of what people have chosen to drug, coded here as "biology" and
# therefore invisible to the research-intensity ablation.
#
# PROTEIN_ARCHITECTURE_FEATURES are the structural covariates that carry that
# prior. The model is now fitted a third time without them. If performance
# survives, the ranking rests on expression biology; if it collapses, the
# model is substantially a detector of druggable protein architecture, which
# must be stated rather than discovered by a reviewer.
PROTEIN_ARCHITECTURE_FEATURES = [
    "n_transmem", "log_ecd_len", "topology_score", "seq_len",
]

# ----------------------------------------------------------------------------
# LATE-FUSION FEATURE SETS
# ----------------------------------------------------------------------------
# RAISED IN EXTERNAL REVIEW, and correct: PROTEIN_FEATURES was declared and
# then never used, and the NEPC variables were computed but never modelled. So
# the "multi-omic ensemble" was, in the primary model, transcriptomic and
# topological only, with proteomics applied downstream as corroboration.
#
# That is a defensible PRIMARY design - protein data are missing for roughly
# half the surfaceome, and imputing them would invent evidence - but it has to
# be described accurately, and the fusion model has to actually exist so the
# choice is tested rather than asserted.
#
# Primary model  : biology features, every gene, no imputation of missing omics.
# Fusion model   : biology + protein (+ NEPC where assessable), fitted ONLY on
#                  the subset where those measurements exist, and reported as
#                  a clearly-labelled sensitivity analysis on a smaller N.
PROTEIN_FEATURES = ["prot_log2fc"]
NEPC_FEATURES = ["nepc_log2fc"]
FUSION_FEATURES = PROTEIN_FEATURES + NEPC_FEATURES


def evaluation_exclude_mask(f):
    """Genes withheld from training so BOTH benchmark sets are comparable.

    The union of the revised and historical benchmark sets. Withholding only
    the revised set let the two historical-only genes be sampled as
    pseudo-negatives, which biased the retrospective comparison against them.
    """
    return f.gene.isin(set(EVALUATION_EXCLUDE)).values


def off_class_mask(f, label_class=None):
    """Genes that are clinical-stage targets OUTSIDE the primary label class.

    Excluded from TRAINING (never used as pseudo-negatives, because they are
    known targets) but NOT evaluation benchmarks. Kept separate from
    `is_heldout` for exactly that reason.
    """
    label_class = label_class or PRIMARY_LABEL_CLASS
    if label_class in ("all", None):
        return np.zeros(len(f), dtype=bool)
    all_pos = set(g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD"))
    excluded = {g for g in all_pos if label_class_of(g) != label_class}
    return f.gene.isin(excluded).values


def make_labels(f, verbose=True, label_class=None):
    """Positives = clinical-stage targets. Everything else is UNLABELED.

    CLINICAL REVISION: positives are now restricted to ONE payload class by
    default (see PAYLOAD_DELIVERY_TARGETS). The previous label set mixed
    payload-delivery targets, which require tumour restriction, with
    signal-blockade targets whose clinical use tolerates near-universal
    normal expression, and with antigens carried mainly by infiltrating
    immune cells. A single classifier trained across all three is taught
    contradictory selectivity requirements.

    Genes outside the primary class are NOT made negatives - that would
    assert they are unsuitable targets, which is false. They are removed
    from training entirely, exactly as the held-out benchmarks are.
    """
    label_class = label_class or PRIMARY_LABEL_CLASS
    all_pos = set(g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD"))
    if label_class in ("all", None):
        pos, excluded = all_pos, set()
    else:
        pos = {g for g in all_pos if label_class_of(g) == label_class}
        excluded = all_pos - pos
    heldout = set(PROSTATE_HELDOUT_POSITIVES)
    y = pd.Series(0, index=f.index, dtype=int)
    y[f.gene.isin(pos)] = 1
    # The PRIMARY endpoint mask is the revised set only.
    is_heldout = f.gene.isin(heldout).values
    # Off-class positives are withheld from BOTH classes, never used as
    # pseudo-negatives. They are returned as a SEPARATE mask: folding them
    # into is_heldout would make them count as evaluation benchmarks and
    # corrupt the held-out recall metric, which is the primary endpoint.
    if not verbose:
        return y.values, is_heldout
    # Held-out benchmarks are removed from training entirely - neither positive
    # nor pseudo-negative - so they cannot contaminate either class.
    print(f"    label class: {label_class} "
          f"({len(pos)} declared positives of {len(all_pos)} clinical-stage "
          f"targets; {len(excluded)} off-class target(s) withheld from BOTH "
          f"classes rather than used as negatives)")
    print(f"    positives present in feature table: {int(y.sum())} "
          f"/ {len(pos)} declared")
    print(f"    held-out prostate benchmarks present: {int(is_heldout.sum())} "
          f"/ {len(heldout)} declared")
    print(f"    unlabeled pool: {int(((y == 0) & ~is_heldout).sum()):,}")
    if int(y.sum()) < 25:
        print("    WARNING: fewer than 25 positives. Treat all model metrics "
              "as exploratory; the CI will be very wide.")
    return y.values, is_heldout


def build_design(f, feature_cols):
    X = f[feature_cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    # Replace infinities created by log/ratio transforms before imputation.
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


# ----------------------------------------------------------------------------
# 3.2 Model zoo with inner-loop hyperparameter grids
# ----------------------------------------------------------------------------
def model_zoo(seed=CFG.seed):
    """Three complementary learners, each regularised, each tuned internally.

    Elastic-net logistic regression supplies interpretable, sparse coefficients
    and performs the feature selection. The two tree ensembles capture
    non-additive structure (e.g. a long ECD only matters if the gene is
    single-pass). Soft voting over the three is the ensemble.
    """
    enet = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", make_elasticnet(C=1.0, l1_ratio=0.5, seed=seed)),
    ])
    enet_grid = enet_param_grid()

    rf = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        # SPEED FIX: n_jobs=1, NOT -1. This estimator is always fitted
        # INSIDE a GridSearchCV that is itself parallel, so a -1 here
        # nested joblib pools - each grid worker spawning a full forest
        # pool, each of those running threaded BLAS. Outer parallel, inner
        # serial is the correct split. A RandomForest is deterministic given
        # random_state regardless of worker count, so this changes speed and
        # peak memory only, never a single predicted value.
        ("clf", RandomForestClassifier(n_estimators=400,
                                       class_weight="balanced_subsample",
                                       min_samples_leaf=2, n_jobs=1,
                                       random_state=seed)),
    ])
    rf_grid = ({"clf__max_depth": [5, None]} if CFG.enet_grid_size == "compact"
               else {"clf__max_depth": [3, 5, None],
                     "clf__max_features": ["sqrt", 0.5]})

    # early_stopping is DISABLED deliberately. With ~75 positives, a bag is a
    # few hundred rows and an outer training fold smaller still; carving off
    # another 20% as an internal validation set can leave a split with one
    # class present, which either errors or silently stops after one
    # iteration. A fixed, modest max_iter with L2 regularisation is the safer
    # control at this sample size.
    hgb = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06,
                                               early_stopping=False,
                                               random_state=seed)),
    ])
    hgb_grid = ({"clf__max_leaf_nodes": [7, 31]} if CFG.enet_grid_size == "compact"
                else {"clf__max_leaf_nodes": [7, 15, 31],
                      "clf__l2_regularization": [0.0, 1.0]})

    return {"ElasticNet": (enet, enet_grid),
            "RandomForest": (rf, rf_grid),
            "HistGradBoost": (hgb, hgb_grid)}


# ----------------------------------------------------------------------------
# 3.3 Bagged PU learning with repeated nested cross-validation
# ----------------------------------------------------------------------------
def pu_bagged_nested_cv(X, y, is_heldout, label="full", seed=CFG.seed,
                        genes=None):
    """Core training routine.

    Returns per-gene ensemble scores (out-of-fold for labeled genes, averaged
    over bags for unlabeled genes) plus per-model CV performance with CIs.
    """
    rng = np.random.default_rng(seed)
    pos_idx = np.where((y == 1) & ~is_heldout)[0]
    unl_idx = np.where((y == 0) & ~is_heldout)[0]
    n_pos = len(pos_idx)
    n_neg_per_bag = min(len(unl_idx), CFG.unlabeled_per_bag_multiplier * n_pos)

    print(f"  [{label}] positives={n_pos}, unlabeled={len(unl_idx):,}, "
          f"pseudo-negatives per bag={n_neg_per_bag}, bags={CFG.n_pu_bags}")

    zoo = model_zoo(seed)
    # Accumulators: sum of scores and count of predictions, per gene per model.
    score_sum = {m: np.zeros(len(y)) for m in zoo}
    score_cnt = {m: np.zeros(len(y)) for m in zoo}
    fold_metrics = {m: [] for m in zoo}
    best_params = {m: [] for m in zoo}
    coef_paths = []

    for bag in range(CFG.n_pu_bags):
        neg_sample = rng.choice(unl_idx, n_neg_per_bag, replace=False)
        bag_idx = np.concatenate([pos_idx, neg_sample])
        Xb, yb = X.iloc[bag_idx], y[bag_idx]
        # The genes not drawn into this bag are this bag's out-of-bag set,
        # scored by the bag's models. Over 50 bags every unlabeled gene is
        # scored many times by models that never saw it.
        oob_idx = np.setdiff1d(np.arange(len(y)), bag_idx)

        outer = StratifiedKFold(n_splits=CFG.n_outer_folds, shuffle=True,
                                random_state=seed + bag)
        for tr, te in outer.split(Xb, yb):
            for mname, (est, grid) in zoo.items():
                inner = StratifiedKFold(n_splits=CFG.n_inner_folds, shuffle=True,
                                        random_state=seed + bag)
                gs = GridSearchCV(clone(est), grid, scoring="roc_auc",
                                  cv=inner, n_jobs=N_JOBS, refit=True,
                                  error_score=np.nan)
                gs.fit(Xb.iloc[tr], yb[tr])
                mdl = gs.best_estimator_
                best_params[mname].append(gs.best_params_)

                # Honest held-out metric for this fold
                pte = mdl.predict_proba(Xb.iloc[te])[:, 1]
                if len(np.unique(yb[te])) == 2:
                    fold_metrics[mname].append({
                        "bag": bag,
                        "auc": roc_auc_score(yb[te], pte),
                        "ap": average_precision_score(yb[te], pte),
                        # Named explicitly: this is the Brier score on the
                        # BALANCED bag (prevalence ~1/3), not on the true
                        # prevalence of ~0.020 (57/2792) used by the calibration
                        # figure. Two quantities previously shared the name
                        # "brier" an order of magnitude apart in what "good"
                        # means, so a reader comparing them drew the opposite
                        # conclusion to the truth.
                        "brier_balanced_bag": brier_score_loss(yb[te], pte),
                    })
                # Score the fold's test genes (out-of-fold) ...
                gi = bag_idx[te]
                score_sum[mname][gi] += pte
                score_cnt[mname][gi] += 1
                # ... and every gene outside this bag (out-of-bag), including
                # the held-out prostate benchmarks.
                if len(oob_idx):
                    poob = mdl.predict_proba(X.iloc[oob_idx])[:, 1]
                    score_sum[mname][oob_idx] += poob
                    score_cnt[mname][oob_idx] += 1

                if mname == "ElasticNet":
                    lr = mdl.named_steps["clf"]
                    coef_paths.append(pd.Series(lr.coef_.ravel(), index=X.columns))

        if (bag + 1) % 10 == 0:
            print(f"    bag {bag+1}/{CFG.n_pu_bags} complete")

    # Average scores; genes never scored (should be none) stay NaN.
    scores = {}
    for m in zoo:
        with np.errstate(invalid="ignore", divide="ignore"):
            s = np.where(score_cnt[m] > 0, score_sum[m] / np.maximum(score_cnt[m], 1), np.nan)
        scores[m] = s
    score_df = pd.DataFrame(scores)
    # Soft-voting ensemble = mean of the three calibrated-ish probabilities.
    score_df["Ensemble"] = score_df.mean(axis=1)

    # KNOWN ASYMMETRY, exposed rather than hidden. A training positive sits in
    # every bag, so it receives one out-of-fold score per bag. An unlabeled
    # gene is usually out-of-bag, so it is scored by every fold model - about
    # n_outer_folds times more often. Averaging over more predictions makes
    # unlabeled scores smoother than positives' scores, which slightly
    # disadvantages positives in any rank comparison. It is inherent to bagged
    # PU learning rather than a bug, but it is invisible unless reported, so
    # the per-gene prediction counts are written out.
    counts = pd.DataFrame({m: score_cnt[m] for m in zoo})
    # BUG FIX (found in our own audit): this QC file shipped with NO gene
    # identifier, in feature-table order. It could only be joined to anything
    # by assuming positional alignment - and that assumption fails, because
    # the candidate ranking is sorted by rank. Joining the two positionally
    # produced a clean, symmetric "22 training positives are mislabelled in
    # the export" that looked like a serious labelling bug and was purely an
    # artifact of the missing key. A QC artifact that invites that error is
    # worse than no artifact.
    counts.insert(0, "gene",
                  list(genes) if genes is not None else list(X.index))
    counts["gene_is_positive"] = (y == 1)
    counts["gene_is_heldout"] = is_heldout
    # Sanitise the label fully: it reaches here as free text like
    # "fusion: biology + protein/NEPC", and a '/' in a filename silently
    # becomes a directory that does not exist.
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", label).strip("_")
    guarded_to_csv(counts, DIRS["logs"] / f"pu_score_counts_{safe}.csv")
    med_pos = float(np.median(score_cnt["ElasticNet"][(y == 1) & ~is_heldout])) if (y == 1).any() else np.nan
    med_unl = float(np.median(score_cnt["ElasticNet"][(y == 0) & ~is_heldout]))
    print(f"  [{label}] median predictions averaged per gene: "
          f"positives {med_pos:.0f}, unlabeled {med_unl:.0f} "
          f"(ratio {med_unl/max(med_pos,1):.1f}x - see note in code)")

    perf = []
    for m in zoo:
        d = pd.DataFrame(fold_metrics[m])
        if len(d) == 0:
            continue
        # "brier_balanced_bag", not "brier": the per-fold value is computed
        # on the balanced bag, not at the true prevalence used by the
        # calibration figure. The names are now distinct so the two cannot
        # be compared by accident.
        for metric in ["auc", "ap", "brier_balanced_bag"]:
            v = d[metric].dropna().values
            # NOT a confidence interval. These are the 2.5th and 97.5th
            # percentiles of the FOLD-LEVEL estimates, and folds reuse the
            # same genes across bags, so the values are dependent. Treating
            # them as a CI for the mean AUC would understate uncertainty.
            # Named for what they are, and reported with the dependence.
            lo, hi = np.percentile(v, [2.5, 97.5]) if len(v) > 5 else (np.nan, np.nan)
            perf.append({"model": m, "metric": metric, "mean": v.mean(),
                         "sd": v.std(ddof=1) if len(v) > 1 else np.nan,
                         "fold_spread_lo": lo, "fold_spread_hi": hi,
                         "n_fold_estimates": len(v),
                         "n_unique_positives": int(((y == 1) & ~is_heldout).sum()),
                         "interval_type": "fold-level percentile spread, "
                                          "NOT a confidence interval "
                                          "(folds share genes across bags)"})
    perf = pd.DataFrame(perf)

    coefs = pd.DataFrame(coef_paths) if coef_paths else pd.DataFrame()
    return score_df, perf, coefs, best_params


# ----------------------------------------------------------------------------
# 3.4 Resampled selection frequency (the feature-selection step)
# ----------------------------------------------------------------------------
def stability_selection(coefs, threshold=0.6):
    """Fraction of elastic-net fits in which each feature received a non-zero
    coefficient - a RESAMPLED SELECTION FREQUENCY.

    TERMINOLOGY (external review, correct): this was described as
    Meinshausen-Buhlmann stability selection. It is not. That procedure adds
    a prescribed subsampling scheme and a threshold calibrated to control the
    expected number of falsely selected variables; neither is implemented
    here. The quantity below is a selection frequency over the PU bags, which
    is a legitimate stability report and far more informative than a single
    model's coefficients, which at this sample size reshuffle completely
    between seeds - but it carries no false-selection guarantee.
    """
    if coefs.empty:
        return pd.DataFrame()
    sel = (coefs.abs() > 1e-8).mean(axis=0).sort_values(ascending=False)
    mean_coef = coefs.mean(axis=0)
    sd_coef = coefs.std(axis=0)
    out = pd.DataFrame({
        "feature": sel.index,
        "selection_frequency": sel.values,
        "mean_coef": mean_coef.reindex(sel.index).values,
        "sd_coef": sd_coef.reindex(sel.index).values,
    })
    out["sign_consistency"] = out.apply(
        lambda r: np.nan if r.sd_coef == 0 or np.isnan(r.sd_coef)
        else max((coefs[r.feature] > 0).mean(), (coefs[r.feature] < 0).mean()), axis=1)
    out["stable"] = out.selection_frequency >= threshold
    save_table(out, "table3_stability_selection.csv",
               caption="Fraction of elastic-net fits retaining each feature")
    print(f"    stable features (selected in >={threshold:.0%} of fits): "
          f"{int(out.stable.sum())}/{len(out)}")
    return out


# ----------------------------------------------------------------------------
# 3.5 Permutation null for AUC
# ----------------------------------------------------------------------------
def permutation_null_auc(X, y, is_heldout, n_perm=CFG.n_permutations,
                         seed=CFG.seed, estimator="ensemble"):
    """Reference distribution of CV AUC under permuted labels.

    Reporting an AUC of 0.85 is meaningless without knowing what this design
    produces on noise. With tiny positive sets and bagged resampling, null AUCs
    can sit well above 0.5, and a naive reader will over-interpret.

    BUG FIX (raised in external review): an earlier version permuted a FIXED,
    untuned elastic-net while the observed statistic came from the tuned
    three-learner ensemble. Null and observed were therefore produced by
    different algorithms, so the comparison did not test what it claimed to.
    The null now runs the SAME learners that produce the observed value, and
    returns the matching observed statistic computed under an identical
    procedure, so the two are directly comparable.

    estimator: "ensemble" (all three learners, matching the reported score)
               "elasticnet" (fast approximation; reported as such)
    """
    rng = np.random.default_rng(seed)
    pos_idx = np.where((y == 1) & ~is_heldout)[0]
    unl_idx = np.where((y == 0) & ~is_heldout)[0]
    n_neg = min(len(unl_idx), CFG.unlabeled_per_bag_multiplier * len(pos_idx))

    # Build the SAME learner set used for the observed statistic. Tuning is
    # omitted inside the permutation loop for tractability, and fixed at the
    # grid midpoint for both null AND observed, so the comparison stays
    # like-for-like. That trade-off is stated rather than hidden.
    zoo = model_zoo(seed)
    if estimator == "ensemble":
        members = [(n, clone(e).set_params(**{k: v[len(v) // 2]
                                              for k, v in g.items()}))
                   for n, (e, g) in zoo.items()]
    else:
        e, g = zoo["ElasticNet"]
        members = [("ElasticNet", clone(e))]

    def _cv_auc(Xs, ys, fold_seed):
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=fold_seed)
        aucs = []
        for tr, te in cv.split(Xs, ys):
            if len(np.unique(ys[tr])) < 2 or len(np.unique(ys[te])) < 2:
                continue
            preds = []
            for _, m in members:
                mm = clone(m).fit(Xs.iloc[tr], ys[tr])
                preds.append(mm.predict_proba(Xs.iloc[te])[:, 1])
            aucs.append(roc_auc_score(ys[te], np.mean(preds, axis=0)))
        return float(np.mean(aucs)) if aucs else np.nan

    print(f"    running {n_perm} label permutations on the "
          f"{'full ensemble' if estimator == 'ensemble' else 'elastic-net'} "
          f"(same learners as the observed statistic) ...")
    nulls, observed = [], []
    for i in range(n_perm):
        neg = rng.choice(unl_idx, n_neg, replace=False)
        idx = np.concatenate([pos_idx, neg])
        Xs, y_true = X.iloc[idx], y[idx].copy()
        # Matched observed value on the same draw, same folds, same learners.
        if i < max(20, n_perm // 10):
            observed.append(_cv_auc(Xs, y_true, i))
        y_perm = y_true.copy()
        rng.shuffle(y_perm)                   # destroy the label-feature link
        a = _cv_auc(Xs, y_perm, i)
        if np.isfinite(a):
            nulls.append(a)
    nulls = np.array(nulls)
    observed = np.array([o for o in observed if np.isfinite(o)])
    obs_mean = float(observed.mean()) if observed.size else np.nan
    pstat = permutation_p(nulls, obs_mean)
    p_emp = pstat["p"]
    print(f"    null AUC: mean={nulls.mean():.3f}, "
          f"95th pct={np.percentile(nulls, 95):.3f}, max={nulls.max():.3f}")
    print(f"    matched observed AUC (same procedure): {obs_mean:.3f} "
          f"-> permutation p = {p_emp:.4g} "
          f"(B={pstat['n_resamples']}, resolution {pstat['resolution']:.4g})")
    if pstat.get("p_is_at_resolution_floor"):
        print(f"      no permutation reached the observed value, so p is at "
              f"the resolution FLOOR: report as p < {pstat['resolution']:.3g}, "
              f"never as p = 0")
    return {"null": nulls, "matched_observed": obs_mean,
            "empirical_p": p_emp, "estimator": estimator,
            "permutation_detail": pstat}


# ----------------------------------------------------------------------------
# 3.6 Held-out benchmark evaluation - the study's primary endpoint
# ----------------------------------------------------------------------------
def evaluate_heldout_benchmarks(f, score_df, is_heldout, seed=CFG.seed):
    """Where do the six withheld prostate benchmarks rank?

    The only internal evaluation in the study that is not circular. If the
    model has learned something transferable, FOLH1/STEAP1/STEAP2 should
    appear in the top decile of ~2,800 genes. A rank-sum test against the
    unlabeled background gives a p-value, and a Monte-Carlo permutation test
    is computed alongside because the benchmark set has only six members.
    """
    d = f.copy()
    d["ensemble_score"] = score_df["Ensemble"].values
    d["rank"] = d.ensemble_score.rank(ascending=False, method="average")
    d["percentile"] = 100 * (1 - d["rank"] / len(d))

    hb = d.loc[is_heldout, ["gene", "ensemble_score", "rank", "percentile"]] \
          .sort_values("rank")
    # Background = unlabeled, non-held-out genes only. Comparing against
    # training positives would make the test trivially significant.
    #
    # BUG FIX: clinical-stage targets outside the primary label class also
    # carry y=0, so they were entering the BACKGROUND pool. 15 of the 17 rank
    # in the top decile, so the benchmark-vs-background contrast was being
    # made against a pool salted with known drug targets - which biases the
    # enrichment statistic downward for every method equally, but for no
    # defensible reason. They are excluded from both arms.
    y, _ = make_labels(f, verbose=False)
    off = off_class_mask(f)
    bg_mask = (y == 0) & ~is_heldout & ~off
    bg = d.loc[bg_mask, "ensemble_score"].dropna().values
    hs = d.loc[is_heldout, "ensemble_score"].dropna().values

    res = {"n_heldout": len(hs), "n_background": len(bg),
           "median_heldout_percentile": float(hb.percentile.median()),
           "n_in_top_decile": int((hb.percentile >= 90).sum())}

    # ==================================================================
    # RANKING METRICS - named for exactly what they measure
    # ==================================================================
    # BUG FIX (raised in external review): the earlier "precision@K" removed
    # training positives from the DENOMINATOR as well as the numerator, so it
    # was not precision among the original top K. The name promised a
    # conventional metric and delivered a different one, which is worse than
    # reporting an imperfect metric honestly.
    #
    # Three quantities are now reported, each named for its own definition:
    #
    #   heldout_recall_at_K    of the 6 withheld benchmarks, how many appear
    #                          in the top K. This is the cleanest statement of
    #                          the primary endpoint and the one to lead with.
    #   heldout_hit_fraction_among_unlabeled_topK
    #                          among top-K genes that are NOT training
    #                          positives, the fraction that are withheld
    #                          benchmarks. Explicitly conditional, so the
    #                          denominator is part of the name.
    #   enrichment_at_K        observed benchmark density in the top K over
    #                          the density expected from a random ranking.
    #                          Interpretable without a labelled negative set.
    #
    # True target-identification precision is NOT estimable here: it needs a
    # comprehensive ground truth that does not exist, since an unlabeled gene
    # in the top K may be a discovery rather than a false positive.
    y_all, _ = make_labels(f, verbose=False)
    ranked_labels = d.assign(_y=y_all,
                             _held=d.gene.isin(PROSTATE_HELDOUT_POSITIVES)
                             ).sort_values("rank")
    n_total, n_held_all = len(ranked_labels), int(ranked_labels._held.sum())
    recall_k, hitfrac_k, enrich_k = {}, {}, {}
    for k in CFG.precision_at_k:
        topk = ranked_labels.head(k)
        n_hit = int(topk._held.sum())
        recall_k[f"K={k}"] = round(n_hit / max(n_held_all, 1), 3)
        unlab = topk[topk._y == 0]
        hitfrac_k[f"K={k}"] = round(n_hit / max(len(unlab), 1), 4)
        expected = k * (n_held_all / max(n_total, 1))
        enrich_k[f"K={k}"] = round(n_hit / expected, 2) if expected > 0 else np.nan
    res["heldout_recall_at_K"] = recall_k
    res["heldout_hit_fraction_among_unlabeled_topK"] = hitfrac_k
    res["enrichment_at_K_over_random"] = enrich_k
    print(f"    held-out recall@K (of {n_held_all} withheld benchmarks): {recall_k}")
    print(f"    enrichment@K over a random ranking: {enrich_k}")
    print(f"    hit fraction among unlabeled top-K: {hitfrac_k}")
    print("      These are PU PROXY metrics against an incomplete label set, "
          "not clinical target-identification accuracy. An unlabeled gene in "
          "the top K may be a discovery rather than a false positive, so none "
          "of the three bounds real performance.")

    # ---- significance of the benchmark enrichment -----------------------
    # Restored after an over-wide edit removed it. Both tests are reported:
    # the rank-sum test is standard, and the Monte-Carlo permutation test
    # does not rely on the asymptotics of a six-member group.
    if len(hs) >= 2 and len(bg) >= 10:
        u = st.mannwhitneyu(hs, bg, alternative="greater")
        res["mannwhitney_p"] = float(u.pvalue)
        res["rank_biserial_effect"] = float(
            2 * u.statistic / (len(hs) * len(bg)) - 1)
        rng = np.random.default_rng(seed)
        obs = hs.mean()
        pool = np.concatenate([hs, bg])
        draws = np.array([rng.choice(pool, len(hs), replace=False).mean()
                          for _ in range(10000)])
        pstat = permutation_p(draws, obs)
        res["permutation_p"] = pstat["p"]
        res["permutation_detail"] = pstat
    else:
        res["mannwhitney_p"] = np.nan
        res["permutation_p"] = np.nan
        print(f"    too few benchmarks ({len(hs)}) or background genes "
              f"({len(bg)}) for an enrichment test; reported as missing "
              f"rather than as a non-significant result")

    print("    held-out benchmark ranks:")
    print(hb.to_string(index=False))
    print(f"    median percentile = {res['median_heldout_percentile']:.1f}, "
          f"in top decile = {res['n_in_top_decile']}/{res['n_heldout']}")
    for k in ("mannwhitney_p", "permutation_p"):
        if k in res:
            print(f"    {k} = {res[k]:.2e}")
    save_table(hb, "table4_heldout_benchmarks.csv",
               caption="Where the six withheld prostate benchmarks ranked (primary endpoint)")
    return d, hb, res


# ----------------------------------------------------------------------------
# 3.7 Feature importance: permutation + coefficients
# ----------------------------------------------------------------------------
def feature_importance(X, y, is_heldout, seed=CFG.seed):
    """Permutation importance on a held-out split, for each base learner.

    Permutation importance is used rather than tree-native impurity importance,
    which is biased toward high-cardinality continuous features and would
    overstate the expression variables purely because they are continuous.
    Correlated features share credit, so the table is read as a group-level
    statement, not a per-feature ranking.
    """
    rng = np.random.default_rng(seed)
    pos = np.where((y == 1) & ~is_heldout)[0]
    unl = np.where((y == 0) & ~is_heldout)[0]
    neg = rng.choice(unl, min(len(unl), CFG.unlabeled_per_bag_multiplier * len(pos)),
                     replace=False)
    idx = np.concatenate([pos, neg])
    Xb, yb = X.iloc[idx], y[idx]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr, te = next(iter(cv.split(Xb, yb)))
    rows = []
    for mname, (est, grid) in model_zoo(seed).items():
        gs = GridSearchCV(clone(est), grid, scoring="roc_auc",
                          cv=StratifiedKFold(3, shuffle=True, random_state=seed),
                          n_jobs=N_JOBS)
        gs.fit(Xb.iloc[tr], yb[tr])
        r = permutation_importance(gs.best_estimator_, Xb.iloc[te], yb[te],
                                   n_repeats=30, random_state=seed,
                                   scoring="roc_auc", n_jobs=N_JOBS)
        for j, feat in enumerate(X.columns):
            rows.append({"model": mname, "feature": feat,
                         "importance": r.importances_mean[j],
                         "sd": r.importances_std[j]})
    imp = pd.DataFrame(rows)
    agg = (imp.groupby("feature")
              .importance.agg(["mean", "std"])
              .sort_values("mean", ascending=False)
              .reset_index())
    agg.columns = ["feature", "mean_importance_across_models", "sd_across_models"]
    save_table(imp, "table5a_permutation_importance_per_model.csv",
               caption="Permutation importance, per learner", show=False)
    save_table(agg, "table5b_permutation_importance_aggregate.csv",
               caption="Permutation importance, averaged across learners")
    print("    top features by permutation importance (AUC decrease):")
    print(agg.head(10).to_string(index=False))
    return imp, agg


# ----------------------------------------------------------------------------
# 3.8 Ranking stability across seeds and bagging assumptions
# ----------------------------------------------------------------------------
# PROPOSAL REFINEMENT (Step 4): "test alternative bagging assumptions and
# seeds". At this sample size a ranking that moves materially between seeds is
# not a ranking, and reporting a single-seed top-20 as a result would be
# irreproducible. This quantifies how much the shortlist actually moves.
# ----------------------------------------------------------------------------
def ranking_stability(X, y, train_exclude, seeds=None, bag_multipliers=None,
                      top_k=50, genes=None, scorer="frozen",
                      benchmark_mask=None):
    """Re-score under alternative seeds and pseudo-negative ratios; report how
    stable the top-K shortlist is across those choices."""
    seeds = list(seeds or CFG.stability_seeds)
    bag_multipliers = tuple(bag_multipliers or CFG.stability_bag_multipliers)
    bench_pcts = []     # benchmark percentile per scoring configuration
    # `train_exclude` governs FITTING; `benchmark_mask` governs EVALUATION.
    is_heldout = train_exclude
    print(f"    re-scoring under {len(seeds)} seeds x "
          f"{len(bag_multipliers)} bagging ratios ...")

    # BUG FIX (raised in external review): stability was measured by repeating
    # pu_bagged_nested_cv(), but the FINAL ranking comes from
    # score_with_frozen(). Those are different scoring procedures, so the
    # reported rank correlation did not describe the stability of the ranking
    # actually published.
    #
    # scorer="frozen" (default) now repeats the real final procedure - fit the
    # frozen ensemble, score every gene - under several seeds and
    # pseudo-negative ratios. scorer="oof" keeps the old behaviour for
    # comparison.
    orig_mult, orig_bags = CFG.unlabeled_per_bag_multiplier, CFG.n_pu_bags
    if scorer == "oof":
        CFG.n_pu_bags = max(3, orig_bags // 4)
    runs, labels = [], []
    try:
        for s in seeds:
            for m in bag_multipliers:
                CFG.unlabeled_per_bag_multiplier = m
                if scorer == "frozen":
                    fz = fit_frozen_model(X, y, is_heldout, seed=s)
                    if fz is None:
                        continue
                    runs.append(score_with_frozen(fz, X))
                else:
                    sc, _, _, _ = pu_bagged_nested_cv(
                        X, y, is_heldout, label=f"stability s{s} m{m}", seed=s)
                    runs.append(sc["Ensemble"].values)
                labels.append(f"seed{s}_ratio{m}")
    finally:
        CFG.unlabeled_per_bag_multiplier = orig_mult
        CFG.n_pu_bags = orig_bags
    if not runs:
        print("    stability not assessable (no model could be fitted)")
        return {}, pd.DataFrame()
    print(f"    stability measured on the {'FINAL FROZEN' if scorer=='frozen' else 'out-of-fold'} "
          f"scoring procedure - the one that produces the published ranking")

    R = pd.DataFrame(np.vstack(runs).T, columns=labels)
    ranks = R.rank(ascending=False, method="average")

    # NOISE MATCHING (found in our own audit): the spread of the BENCHMARK
    # percentile across scoring configurations is the uncertainty the baseline
    # comparison needs. B1-B4 are deterministic functions of the features, so
    # ALL estimation noise in that comparison sits on the ensemble side, and
    # reporting the gain as a bare point estimate hides it.
    # CRITICAL BUG FIX (external review, confirmed): the benchmark percentile
    # was computed over whatever mask arrived in this parameter, and the
    # caller passes `train_exclude` - the six prespecified benchmarks PLUS 17
    # off-class clinical targets PLUS 2 historical-only benchmarks, 25 genes
    # in total. The reported spread therefore described a 25-gene population
    # while the primary endpoint is the six. The two were visibly different:
    # 97.96 / 98.14 from the stability analysis against 92.34 for the actual
    # ensemble endpoint.
    #
    # This is the SAME two-masks-one-parameter defect already fixed in
    # fusion_sensitivity_model and preflight_label_audit; I introduced it here
    # when adding the noise-matching code and did not apply the lesson. The
    # masks are now separate, and benchmark_mask is MANDATORY for this
    # computation rather than silently defaulting.
    if benchmark_mask is None:
        raise ValueError(
            "ranking_stability requires an explicit benchmark_mask for the "
            "benchmark-percentile spread. Defaulting it to train_exclude is "
            "what made the spread describe 25 genes instead of the six "
            "prespecified benchmarks. Pass benchmark_mask=is_heldout.")
    if genes is not None and np.any(np.asarray(benchmark_mask)):
        hb = np.asarray(benchmark_mask, dtype=bool)
        # The evaluation population must be the prespecified benchmark set,
        # not a superset. Asserted rather than assumed, because the whole
        # defect above was an unasserted assumption.
        # BUG FIX (both external reviews, correct): this assertion checked
        # the COUNT only, so a mask selecting six entirely different genes
        # passed it. After three separate mask defects in this codebase -
        # fusion_sensitivity_model, preflight_label_audit and this very
        # function - a cardinality check is not enough: the identities are
        # what the endpoint is defined by. Checked by exact set equality.
        expected = set(PROSTATE_HELDOUT_POSITIVES) & set(genes)
        actual = set(np.asarray(genes)[hb])
        if actual != expected:
            raise ValueError(
                f"benchmark_mask does not select the prespecified benchmark "
                f"genes. Missing: {sorted(expected - actual)}; unexpected: "
                f"{sorted(actual - expected)}. The spread must be computed "
                f"over the primary endpoint population exactly - a mask of "
                f"the right SIZE but the wrong MEMBERS is how the stability "
                f"spread came to be computed over 25 training-excluded "
                f"genes instead of the 6 benchmarks.")
        # BUG FIX (external review, correct): the spread was computed by
        # ranking ALL genes, while the primary gain it is compared against
        # now excludes the training positives. Comparing a 2.27-point spread
        # measured on 2,792 genes with a 2.00-point deficit measured on
        # 2,735 is comparing quantities from different populations - the
        # same mismatch corrected for the percentile and the recall, left in
        # place one statistic over. The exclusion is applied here too, so
        # the two figures are commensurable.
        _train_pos_sb = np.zeros(len(R), dtype=bool)
        if train_exclude is not None and y is not None:
            _train_pos_sb = (np.asarray(y) == 1) & ~hb
        _common_sb = ~_train_pos_sb
        for col in R.columns:
            _sc_sb = R[col][_common_sb]
            pct = _sc_sb.rank(pct=True).values * 100.0
            bench_pcts.append(float(np.median(pct[hb[_common_sb]])))
        if bench_pcts:
            spread = max(bench_pcts) - min(bench_pcts)
            print(f"    benchmark percentile across {len(bench_pcts)} scoring "
                  f"configurations: median {np.median(bench_pcts):.2f}, range "
                  f"{min(bench_pcts):.2f}-{max(bench_pcts):.2f} (spread "
                  f"{spread:.2f} points). The transparent baselines carry NO "
                  f"such spread. This range is DESCRIPTIVE variability "
                  f"across {len(bench_pcts)} prespecified scoring "
                  f"configurations on the same common population as the "
                  f"primary gain - it is not a confidence interval and not "
                  f"an estimate of the sampling distribution of the gain, "
                  f"so it cannot establish statistical indistinguishability. "
                  f"An incremental gain smaller than this "
                  f"range cannot be resolved by this design.")

    # Spearman correlation between every pair of runs
    corr = ranks.corr(method="spearman")
    iu = np.triu_indices_from(corr.values, k=1)
    pair_rho = corr.values[iu]

    # Jaccard overlap of the top-K shortlists
    tops = [set(np.argsort(-R[c].values)[:top_k]) for c in R.columns]
    jac = [len(a & b) / len(a | b) for i, a in enumerate(tops)
           for b in tops[i + 1:]]

    res = {
        "scorer": scorer,
        "n_runs": len(labels),
        "median_spearman_rho": float(np.median(pair_rho)),
        "min_spearman_rho": float(np.min(pair_rho)),
        f"median_jaccard_top{top_k}": float(np.median(jac)),
        f"min_jaccard_top{top_k}": float(np.min(jac)),
    }
    # Per-gene rank dispersion: the honest uncertainty on a candidate's place
    # BUG FIX (raised in external review): the exported table carried numeric
    # row indices and no gene column, so per-candidate rank uncertainty could
    # not be joined back to the ranking table at all.
    disp = pd.DataFrame({
        "gene": (list(genes) if genes is not None
                 else [f"row_{i}" for i in range(ranks.shape[0])]),
        "mean_rank": ranks.mean(axis=1).values,
        "sd_rank": ranks.std(axis=1).values,
        "min_rank": ranks.min(axis=1).values,
        "max_rank": ranks.max(axis=1).values,
        "rank_range": (ranks.max(axis=1) - ranks.min(axis=1)).values,
        "scorer": scorer,
        "n_runs": len(labels),
    }).sort_values("mean_rank")
    save_table(disp, "table10_rank_stability_per_gene.csv",
               caption="Per-GENE rank dispersion across seeds and bagging "
                       "ratios, measured on the final scoring procedure",
               show=False)
    print(f"    rank correlation across runs: median rho = "
          f"{res['median_spearman_rho']:.3f}, worst = {res['min_spearman_rho']:.3f}")
    print(f"    top-{top_k} shortlist overlap: median Jaccard = "
          f"{res[f'median_jaccard_top{top_k}']:.3f}, worst = "
          f"{res[f'min_jaccard_top{top_k}']:.3f}")
    if res["median_spearman_rho"] < 0.8:
        print("    WARNING: ranking is seed-sensitive. Report rank intervals "
              "rather than point ranks, and do not present a single top-20.")
    if bench_pcts:
        res["ensemble_benchmark_percentile_by_config"] = bench_pcts
        res["ensemble_percentile_spread"] = round(
            float(max(bench_pcts) - min(bench_pcts)), 2)
        # BUG FIX (both external reviews, correct): "a gain smaller than
        # this spread is not resolvable by this design" is too strong. Two
        # scoring configurations give a descriptive range for the
        # ENSEMBLE's percentile, not an inferential interval for the
        # DIFFERENCE between the ensemble and B1, and stochastic variation
        # moves a ranking metric in either direction rather than only
        # depressing it.
        res["noise_matching_note"] = (
            f"B1-B4 are deterministic, so the estimation noise in this "
            f"comparison sits on the ensemble side. Across "
            f"{len(bench_pcts)} prespecified scoring configuration(s) the "
            f"ensemble's median benchmark percentile spanned "
            f"{float(max(bench_pcts) - min(bench_pcts)):.2f} points. That "
            f"is a descriptive sensitivity range, not a confidence interval "
            f"for the gain and not a resolution limit; establishing one "
            f"would need an uncertainty estimate for the difference, which "
            f"this design does not provide.")
    return res, disp


# ----------------------------------------------------------------------------
# 3.9 External cohort transfer with the FROZEN model
# ----------------------------------------------------------------------------
# PROPOSAL REFINEMENT (Step 5 / Aim 3): "score a truly independent prostate
# cohort using the frozen model". Internal held-out benchmarks test whether the
# model recovers known biology; they do NOT test whether the ranking transfers
# to a cohort processed elsewhere, by other people, on other platforms. Only
# this does.
# ----------------------------------------------------------------------------
def fit_frozen_model(X, y, is_heldout, seed=CFG.seed):
    """Fit the final model to be applied unchanged to an external cohort.

    Frozen means frozen: no refitting, no threshold tuning on the external
    data, no feature reselection.

    BUG FIX: an earlier version fitted the UNTUNED base estimator straight out
    of the model zoo. That was wrong twice over. Statistically, the frozen
    model would have used arbitrary default hyperparameters rather than the
    ones cross-validation selected, so it would not have been the model whose
    performance was reported. Mechanically, the zoo's elastic-net carries no
    l1_ratio (the grid supplies it), which raises on scikit-learn <= 1.7 and
    crashed the notebook on Colab.

    Hyperparameters are now selected ONCE by inner cross-validation on the
    labelled data, then held fixed across the bagged members. Tuning per bag
    would let the hyperparameters chase each bag's pseudo-negative draw, which
    is a subtle form of overfitting to the resampling scheme.
    """
    rng = np.random.default_rng(seed)
    pos = np.where((y == 1) & ~is_heldout)[0]
    unl = np.where((y == 0) & ~is_heldout)[0]
    if len(pos) < 5:
        print("    too few positives to freeze a model; skipping")
        return None
    n_neg = min(len(unl), CFG.unlabeled_per_bag_multiplier * len(pos))

    # BUG FIX (raised in external review): an earlier version froze ONLY the
    # elastic-net, while the reported candidate ranking came from the
    # three-learner soft-voting ensemble. External transfer therefore tested a
    # related but different model from the one the paper describes.
    #
    # All three learners are now tuned and frozen, and scoring averages them
    # exactly as the reported ensemble does.
    tune_idx = np.concatenate([pos, rng.choice(unl, n_neg, replace=False)])
    n_splits = max(2, min(CFG.n_inner_folds, int(min(np.bincount(y[tune_idx])))))
    frozen_members, chosen = {}, {}
    for name, (est, grid) in model_zoo(seed).items():
        gs = GridSearchCV(clone(est), grid, scoring="roc_auc",
                          cv=StratifiedKFold(n_splits, shuffle=True,
                                             random_state=seed),
                          n_jobs=N_JOBS, error_score=np.nan)
        gs.fit(X.iloc[tune_idx], y[tune_idx])
        chosen[name] = gs.best_params_
        frozen_members[name] = []
    print(f"    frozen-model hyperparameters (one CV selection per learner):")
    for k, v in chosen.items():
        print(f"      {k}: {v}")

    for b in range(CFG.n_pu_bags):
        neg = rng.choice(unl, n_neg, replace=False)
        idx = np.concatenate([pos, neg])
        for name, (est, _) in model_zoo(seed).items():
            m = clone(est).set_params(**chosen[name])
            frozen_members[name].append(m.fit(X.iloc[idx], y[idx]))
    total = sum(len(v) for v in frozen_members.values())
    print(f"    frozen ensemble: {total} fitted members "
          f"({len(frozen_members)} learners x {CFG.n_pu_bags} bags) "
          f"over {X.shape[1]} features")
    return {"members": frozen_members, "features": list(X.columns),
            "hyperparameters": chosen, "n_members": total,
            "learners": list(frozen_members)}


def export_frozen_model(frozen, path=None):
    """Persist the frozen ensemble so an external cohort can be scored
    without refitting (NEW - raised in external review).

    The archive previously shipped an EMPTY 03_models/ directory: the frozen
    ensemble existed only in memory, so no independent researcher could
    reproduce a single prediction, and an external-validation cohort could not
    be scored with the same objects. A frozen model that is not serialised is
    not frozen - it is merely unrefitted within one session.

    Saved alongside the estimators: the exact feature schema IN ORDER (the
    commonest silent failure when rescoring is a reordered design matrix), the
    selected hyperparameters, the run identity and the package versions the
    fit depends on.
    """
    if frozen is None:
        print("    frozen model NOT exported: no model was fitted")
        return None
    if _WRITES_SUPPRESSED:
        SUPPRESSED_WRITE_LOG.append(("model", "frozen_ensemble.joblib"))
        return None
    import platform
    try:
        import joblib
    except ImportError:
        print("    frozen model NOT exported: joblib unavailable")
        return None
    import sklearn
    path = Path(path or (DIRS["models"] / "frozen_ensemble.joblib"))
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "profile": PROFILE,
        "seed": CFG.seed,
        # Feature order is part of the model, not metadata: score_with_frozen
        # must receive these columns, in this order.
        "feature_names_in_order": list(frozen["features"]),
        "n_features": len(frozen["features"]),
        "learners": list(frozen["learners"]),
        "hyperparameters": frozen["hyperparameters"],
        "n_members": frozen["n_members"],
        "members": frozen["members"],
        "scoring_contract": ("average predict_proba[:,1] within each learner, "
                             "then average the three learner means; output is "
                             "a RANKING score, not a calibrated probability"),
        "versions": {"python": sys.version.split()[0],
                     "platform": platform.platform(),
                     "scikit_learn": sklearn.__version__,
                     "numpy": np.__version__,
                     "pandas": pd.__version__,
                     "joblib": joblib.__version__},
    }
    joblib.dump(payload, path, compress=3)
    # SELF-REVIEW, caught after removing the modification-time fallback:
    # the frozen model was never explicitly REGISTERED. It reached the
    # archive only because the fallback swept in anything with a recent
    # mtime - so the moment that hole was closed, a required release
    # artifact silently vanished from the package. This is what both
    # external reviews meant by "required files should be declared,
    # registered and verified": removing the fallback is only half the fix.
    try:
        register_artifact(path)
    except Exception as _e:
        print(f"    WARNING: could not register the frozen model: {_e!r}")
    size_mb = path.stat().st_size / 1e6
    # Reload immediately and verify the schema survives the round trip: an
    # artifact that cannot be read back is not a deliverable.
    ok, detail = False, ""
    try:
        back = joblib.load(path)
        ok = (back["feature_names_in_order"] == payload["feature_names_in_order"]
              and back["n_members"] == payload["n_members"])
        detail = (f"{back['n_members']} members, "
                  f"{back['n_features']} features verified")
    except Exception as e:
        detail = f"RELOAD FAILED: {e!r}"
    print(f"    frozen ensemble exported: {path.name} ({size_mb:.1f} MB); "
          f"reload check {'PASSED' if ok else 'FAILED'} - {detail}")
    manifest = pd.DataFrame([{
        "artifact": path.name, "run_id": RUN_ID, "n_members": payload["n_members"],
        "n_features": payload["n_features"],
        "learners": ", ".join(payload["learners"]),
        "scikit_learn": sklearn.__version__,
        "reload_verified": ok,
        "scoring_contract": payload["scoring_contract"],
    }])
    save_table(manifest, "table22_frozen_model_manifest.csv",
               caption="Serialised frozen ensemble: schema, hyperparameters "
                       "and the versions required to reload it")
    return {"path": str(path), "size_mb": round(size_mb, 2),
            "reload_verified": bool(ok), "n_members": payload["n_members"],
            "n_features": payload["n_features"]}


def score_with_frozen(frozen, X):
    """Score every row with the frozen ensemble - one identical procedure.

    Used for the FINAL ranking, so that no gene's score depends on how often
    it happened to fall out-of-bag. Averages within each learner first, then
    across learners, matching the reported soft-voting weight.
    """
    if frozen is None:
        return None
    Xe = X[frozen["features"]]
    per_learner = [np.mean([m.predict_proba(Xe)[:, 1] for m in models], axis=0)
                   for models in frozen["members"].values()]
    return np.mean(per_learner, axis=0)


def score_external_cohort(frozen, X_external, gene_index):
    """Apply the frozen model to an independent cohort's feature matrix.

    X_external must be built by the SAME feature pipeline, with the same
    columns in the same order. The function checks this rather than trusting
    it, because a silently reordered column matrix produces plausible nonsense.
    """
    if frozen is None:
        print("    no frozen model available; external scoring skipped")
        return None
    expected = frozen["features"]
    missing = [c for c in expected if c not in X_external.columns]
    if missing:
        raise ValueError(
            f"External cohort is missing {len(missing)} features the frozen "
            f"model requires: {missing[:8]}. Rebuild it through the same "
            f"feature pipeline; do not drop or impute columns to make it fit.")
    Xe = X_external[expected]
    # Average WITHIN each learner, then across learners, so the soft-voting
    # weighting matches the reported ensemble rather than being dominated by
    # whichever learner happens to have more fitted members.
    per_learner = []
    for name, models in frozen["members"].items():
        per_learner.append(np.mean([m.predict_proba(Xe)[:, 1] for m in models], axis=0))
    scores = np.mean(per_learner, axis=0)
    out = pd.DataFrame({"gene": gene_index, "external_score": scores})
    out["external_rank"] = out.external_score.rank(ascending=False)
    save_table(out, "table11_external_cohort_scores.csv",
               caption="Frozen-model scores on the external cohort")
    return out


def compare_internal_external(internal_ranked, external_scores, top_k=50):
    """Did the ranking transfer? Spearman plus top-K overlap."""
    m = internal_ranked[["gene", "rank"]].merge(
        external_scores[["gene", "external_rank"]], on="gene", how="inner")
    if len(m) < 20:
        print("    too few shared genes to assess transfer")
        return {}
    rho = st.spearmanr(m["rank"], m.external_rank)
    a = set(m.nsmallest(top_k, "rank").gene)
    b = set(m.nsmallest(top_k, "external_rank").gene)
    res = {"n_shared_genes": len(m), "spearman_rho": float(rho.statistic),
           "spearman_p": float(rho.pvalue),
           f"top{top_k}_overlap": len(a & b),
           f"top{top_k}_jaccard": round(len(a & b) / len(a | b), 3)}
    print(f"    external transfer: rho = {res['spearman_rho']:.3f} "
          f"(p = {res['spearman_p']:.2e}), top-{top_k} overlap = "
          f"{res[f'top{top_k}_overlap']}/{top_k}")
    return res


# ----------------------------------------------------------------------------
# 3.10 Single-cell / spatial heterogeneity assessment
# ----------------------------------------------------------------------------
# PROPOSAL REFINEMENT (Aim 3): bulk RNA cannot distinguish a target expressed
# uniformly from one expressed in half the cells, and that distinction IS the
# clinical problem. This hook consumes a single-cell summary table if you have
# one and computes the two numbers that matter.
#
# It deliberately does not fabricate single-cell data in DEMO mode: a synthetic
# answer to "what fraction of malignant cells express this?" would be actively
# misleading, so with no file supplied the analysis is SKIPPED and recorded as
# not performed.
# ----------------------------------------------------------------------------
SINGLECELL_FILE = os.environ.get("SINGLECELL_FILE", "")

def single_cell_assessment(candidate_genes):
    """Malignant-cell coverage and rare normal-cell expression per candidate.

    Expected schema (CSV): gene, pct_malignant_cells_expressing,
    max_normal_celltype_pct, max_normal_celltype_name.
    """
    if not SINGLECELL_FILE or not Path(SINGLECELL_FILE).exists():
        print("    single-cell assessment SKIPPED: no SINGLECELL_FILE supplied.")
        print("    Candidate heterogeneity and rare normal-cell expression are "
              "therefore UNASSESSED. State this in the limitations; do not "
              "imply coverage you have not measured.")
        return None
    sc = pd.read_csv(SINGLECELL_FILE)
    sc = sc[sc.gene.isin(candidate_genes)]
    sc["heterogeneity_flag"] = sc.pct_malignant_cells_expressing < 50
    sc["rare_normal_risk_flag"] = sc.max_normal_celltype_pct > 20
    save_table(sc, "table12_single_cell_assessment.csv",
               caption="Malignant-cell coverage and rare normal-cell expression")
    print(f"    single-cell: {len(sc)} candidates assessed | "
          f"{int(sc.heterogeneity_flag.sum())} expressed in <50% of malignant "
          f"cells | {int(sc.rare_normal_risk_flag.sum())} with notable "
          f"rare-normal-cell expression")
    return sc


# ----------------------------------------------------------------------------
# 3.11 SIMPLE BIOLOGICAL BASELINES
# ----------------------------------------------------------------------------
# PROPOSAL REFINEMENT (comparison document, "Simple baselines"): a complex
# ensemble must demonstrate INCREMENTAL utility beyond transparent biological
# ranking. Otherwise the complexity is unjustified and the methodological
# novelty claim fails - a reviewer will simply ask "what does the model add
# over sorting by fold-change?", and that question needs a number, not an
# argument.
#
# Four baselines, each a one-line ranking anyone could reproduce:
#   B1  differential expression alone        (rank by log2FC)
#   B2  statistical confidence alone         (rank by -log10 q)
#   B3  tumour selectivity alone             (rank by therapeutic window)
#   B4  transparent weighted score           (equal-weight z-scores of the
#                                             three biological pillars)
#
# Each is evaluated on the SAME primary endpoint as the ensemble: recovery of
# the held-out prostate benchmarks. If the ensemble does not beat these, say
# so and report the simplest adequate method as the result.
# ----------------------------------------------------------------------------
def simple_baselines(f, is_heldout, ensemble_scores=None, seed=CFG.seed,
                     exclude_from_background=None, stability=None):
    """Transparent baseline rankings, scored on the primary endpoint.

    `is_heldout` is the BENCHMARK set (the numerator of the endpoint).
    `exclude_from_background` additionally removes known clinical-stage
    targets outside the primary label class from the comparison background:
    they carry y=0 but are not negatives, and leaving them in the background
    pool compares the benchmarks against a pool salted with known drug
    targets, which biases every method's enrichment statistic downward.
    """
    rng = np.random.default_rng(seed)
    y, _ = make_labels(f, verbose=False)
    if exclude_from_background is None:
        exclude_from_background = np.zeros(len(f), dtype=bool)
    exclude_from_background = np.asarray(exclude_from_background, dtype=bool)

    def z(col, invert=False):
        v = pd.to_numeric(f.get(col, pd.Series(np.nan, index=f.index)),
                          errors="coerce").replace([np.inf, -np.inf], np.nan)
        v = v.fillna(v.median())
        s = (v - v.mean()) / (v.std(ddof=0) + 1e-9)
        return -s if invert else s

    baselines = {
        "B1 differential expression (log2FC)": z("rna_log2fc"),
        "B2 statistical confidence (-log10 q)": z("neg_log10_rna_q"),
        "B3 tumour selectivity (therapeutic window)": z("therapeutic_window_log2"),
        "B4 transparent weighted score": (
            z("rna_log2fc") + z("log_safety_ratio", invert=True)
            + z("log_ecd_len")) / 3.0,
    }
    if ensemble_scores is not None:
        baselines["PU ensemble (this study)"] = pd.Series(
            ensemble_scores, index=f.index)

    bg_mask = (y == 0) & ~is_heldout & ~exclude_from_background
    # SELF-REVIEW (found by auditing the ensemble-vs-baseline comparison
    # that the STOP verdict rests on, and quantified against the exported
    # ranking): the median benchmark percentile was computed over ALL
    # ranked genes, and that population CONTAINS the training positives the
    # ensemble was fitted on. All 57 of them score above the median
    # benchmark, so they occupy ranks above it and push the benchmarks down
    # the percentile scale - for the ensemble only. B1-B4 are unsupervised
    # functions of the features and memorise nothing, so they are not
    # depressed the same way.
    #
    # Measured: removing the training positives from the ranked population
    # lifts the ensemble from 91.55 to 93.14 and B1 from 94.50 to
    # 95.14, so the headline gain moves from -2.95 to -2.00 (figures from
    # the executed run, not the exploratory estimate).
    # This is the MIRROR IMAGE of the precision_at_25 defect already fixed
    # here: that one flattered the ensemble, this one penalises it. A
    # supervised ranker and an unsupervised score cannot be compared on a
    # population that includes the supervised method's own training set.
    #
    # Both figures are now reported. The verdict is UNCHANGED either way -
    # -2.00 is still far short of the prespecified +3.0 - so this is a
    # correction for honesty, not a result that rescues the ensemble.
    _trained_pos = (y == 1) & ~np.asarray(is_heldout, dtype=bool)
    _common_pop = ~_trained_pos
    rows = []
    for name, s in baselines.items():
        s = pd.Series(s).astype(float)
        rank = s.rank(ascending=False, method="average")
        pct = 100 * (1 - rank / len(s))
        # The same statistic on the common population: training labels
        # removed from the ranked pool for every method alike.
        _sc = s[_common_pop]
        _rc = _sc.rank(ascending=False, method="average")
        _pctc = 100 * (1 - _rc / len(_sc))
        _bmc = np.asarray(is_heldout, dtype=bool)[_common_pop]
        hs, bg = s[is_heldout].dropna(), s[bg_mask].dropna()
        if len(hs) < 2 or len(bg) < 10:
            continue
        u = st.mannwhitneyu(hs, bg, alternative="greater")
        # Exact permutation p: robust for a 6-member benchmark set
        pool = np.concatenate([hs.values, bg.values])
        obs = hs.mean()
        draws = np.array([rng.choice(pool, len(hs), replace=False).mean()
                          for _ in range(5000)])
        pstat = permutation_p(draws, obs)
        rows.append({
            "method": name,
            "median_benchmark_percentile": round(float(pct[is_heldout].median()), 2),
            "median_benchmark_percentile_common_population": round(
                float(_pctc[_bmc].median()), 2),
            # SELF-REVIEW: I moved the PERCENTILE to the common population
            # last round and left this count - the other half of the same
            # verdict - on all genes. So the prespecified rule was being
            # evaluated with its margin condition measured on 2,735 genes
            # and its top-decile condition on 2,792: two conditions, two
            # populations, one verdict. Both are now reported on both
            # populations, and the verdict reads the common one for each.
            "benchmarks_in_top_decile": int((pct[is_heldout] >= 90).sum()),
            "benchmarks_in_top_decile_common_population": int(
                (_pctc[_bmc] >= 90).sum()),
            "n_benchmarks": int(is_heldout.sum()),
            "psma_rank": (int(rank[f.gene == CFG.benchmark_gene].iloc[0])
                          if (f.gene == CFG.benchmark_gene).any() else np.nan),
            "mannwhitney_p": float(u.pvalue),
            "permutation_p": pstat["p"],
            "permutation_resolution": pstat["resolution"],
            # CRITICAL BUG FIX (raised in external code review, confirmed
            # against the exported ranking): this was named precision_at_25
            # and counted the 74 TRAINING POSITIVES as discoveries. The frozen
            # ensemble was fitted on those labels, so its top 25 contained 22
            # training positives and 1 held-out benchmark - a 0.92 "precision"
            # that was 96% label recovery. B1-B4 are untrained, so their
            # 0.04-0.32 was honest, and the comparison was structurally
            # rigged in favour of the ensemble inside the very table that
            # declares the ensemble did NOT beat the baselines.
            #
            # The quantity is now reported under its true name, split into
            # its components, and EXCLUDED from the improvement verdict. The
            # defensible validation statistic is held-out recall at K.
            "known_label_recovery_at_25": round(float(
                (pd.Series(y, index=f.index)[rank <= 25].sum()
                 + is_heldout[rank.values <= 25].sum()) / 25), 3),
            "training_positives_in_top25": int(
                pd.Series(y, index=f.index)[rank <= 25].sum()),
            "heldout_benchmarks_in_top25": int(
                is_heldout[rank.values <= 25].sum()),
            "unlabeled_in_top25": int(
                25 - pd.Series(y, index=f.index)[rank <= 25].sum()
                - is_heldout[rank.values <= 25].sum()),
            # Held-out recall at K. BUG FIX (external review, correct): the
            # comment claimed "no training label can inflate it", which is
            # true of the NUMERATOR but not of the rank. Rank is computed
            # over all genes, so 57 training positives the ensemble
            # memorised occupy top-25 slots and push benchmarks out of them
            # - the same population mismatch corrected for the percentile,
            # left in place one metric over. The reviewer recalculated it:
            # the ensemble's recall at 25 is 0/6 on all genes and 2/6 on the
            # common population (FOLH1 48->12, TMEFF2 53->16).
            #
            # Both populations are reported, named, and never mixed inside
            # one comparison. B1 recovers 0/6 under either definition.
            "heldout_recall_at_25": round(float(
                is_heldout[rank.values <= 25].sum()
                / max(int(is_heldout.sum()), 1)), 3),
            "heldout_recall_at_50": round(float(
                is_heldout[rank.values <= 50].sum()
                / max(int(is_heldout.sum()), 1)), 3),
            "heldout_recall_at_25_common_population": round(float(
                _bmc[_rc.values <= 25].sum()
                / max(int(_bmc.sum()), 1)), 3),
            "heldout_recall_at_50_common_population": round(float(
                _bmc[_rc.values <= 50].sum()
                / max(int(_bmc.sum()), 1)), 3),
            "n_reference_genes_all": int(len(s)),
            "n_reference_genes_common": int(len(_sc)),
            "is_trained_on_labels": bool(name.startswith("PU ensemble")),
        })
    out = pd.DataFrame(rows)
    save_table(out, "table13_baseline_comparison.csv",
               caption="Ensemble vs transparent baselines on the primary endpoint")
    if not IN_NOTEBOOK:
        print(out.to_string(index=False))

    verdict = {}
    if ensemble_scores is not None and len(out):
        ens = out[out.method.str.startswith("PU ensemble")]
        base = out[~out.method.str.startswith("PU ensemble")]
        if len(ens) and len(base):
            e = ens.iloc[0]
            # The PRIMARY comparison uses the common population, where the
            # ensemble's own training labels are removed from the ranked
            # pool for every method alike. On the all-genes population the
            # supervised method is penalised for memorising labels that the
            # unsupervised baselines never saw, which is not a property of
            # the ranker being tested. Both figures are reported, and the
            # prespecified margin is unchanged.
            _sp = (stability or {}).get("ensemble_percentile_spread")
            _ncfg = len((stability or {}).get(
                "ensemble_benchmark_percentile_by_config", []) or [])
            _sp_txt = (f"{float(_sp):.2f} points" if _sp is not None
                       else "an amount not measured in this run")
            _n_cfg_txt = str(_ncfg) if _ncfg else "an unrecorded number of"
            _pcol = "median_benchmark_percentile_common_population"
            best = base.loc[base[_pcol].idxmax()]
            gain = e[_pcol] - best[_pcol]
            gain_all_genes = (e.median_benchmark_percentile
                              - best.median_benchmark_percentile)
            _tdcol = "benchmarks_in_top_decile_common_population"
            td_gain = int(e[_tdcol] - best[_tdcol])
            td_gain_all_genes = int(e.benchmarks_in_top_decile
                                    - best.benchmarks_in_top_decile)
            # PREDEFINED RULE, fixed in CFG before any LIVE outcome is seen.
            # Both conditions must hold. With only six held-out positives a
            # one-target difference is noise, so a margin on the continuous
            # metric alone is not sufficient evidence of improvement.
            meets_margin = gain >= CFG.min_meaningful_margin_pct
            meets_targets = td_gain >= CFG.min_top_decile_gain
            verdict = {
                "best_baseline": best.method,
                "comparison_population": (
                    "training positives excluded from the ranked pool for "
                    "every method (primary)"),
                "best_baseline_median_pct": float(best[_pcol]),
                "ensemble_median_pct": float(e[_pcol]),
                "incremental_gain_pct": round(float(gain), 2),
                "best_baseline_median_pct_all_genes": float(
                    best.median_benchmark_percentile),
                "ensemble_median_pct_all_genes": float(
                    e.median_benchmark_percentile),
                "incremental_gain_pct_all_genes": round(
                    float(gain_all_genes), 2),
                "top_decile_gain": td_gain,
                "top_decile_gain_all_genes": td_gain_all_genes,
                "required_margin_pct": CFG.min_meaningful_margin_pct,
                "required_top_decile_gain": CFG.min_top_decile_gain,
                "meets_margin": bool(meets_margin),
                "meets_target_gain": bool(meets_targets),
                "ensemble_beats_baselines": bool(meets_margin and meets_targets),
                "rule": ("PREDEFINED: gain >= "
                         f"{CFG.min_meaningful_margin_pct} percentile points "
                         f"AND >= {CFG.min_top_decile_gain} additional "
                         f"benchmarks in the top decile"),
                # NOISE MATCHING (found in our own audit): the ensemble
                # score is a STOCHASTIC estimate - averaged over bagged
                # pseudo-negative draws - while B1-B4 are deterministic
                # functions of the features with exactly zero estimation
                # noise. Noise can only degrade the ensemble's median
                # benchmark percentile, never improve it, so comparing a
                # noisy estimate against a noiseless one on a fixed margin
                # is not a fair test of the ensemble. The margin stays
                # prespecified; what is added is the uncertainty on the
                # ensemble side, so the verdict can be stated as a verdict
                # about an interval rather than about a point.
                "ensemble_is_stochastic": True,
                "baselines_are_deterministic": True,
                # BUG FIX (both external reviews, correct): this told the
                # reader to "treat the gain as an interval" on the strength
                # of two scoring configurations. Two configurations cannot
                # establish an interval for a DIFFERENCE, and stochastic
                # variation moves a ranking metric in either direction - the
                # earlier claim that noise could only depress the ensemble
                # was wrong too. Replaced with what was actually measured.
                # SELF-REVIEW of this very edit: my first version
                # interpolated `spread` and `n_cfg`, which live in
                # ranking_stability(), not here - a NameError at runtime.
                # The stability result is now passed in explicitly and the
                # note degrades honestly when it is absent.
                # BUG FIX (both external reviews, correct): the note read
                # "varied by ensemble_percentile_spread points" - I wrote
                # the FIELD NAME into the prose instead of interpolating the
                # value, so the exported metadata told the reader a variable
                # name. It now carries the number.
                "gain_sensitivity_note": (
                    f"across the prespecified scoring configurations the "
                    f"ensemble's median benchmark percentile varied by "
                    f"{_sp_txt} on the same common population as "
                    f"the gain. This is a DESCRIPTIVE sensitivity range "
                    f"across {_n_cfg_txt} configuration(s), not a confidence "
                    f"interval for the difference between the ensemble and "
                    f"the baseline and not evidence of equivalence; two "
                    f"configurations cannot establish an inferential "
                    f"interval. The ensemble did not meet the predefined "
                    f"improvement criteria."),
                # Both verdict metrics are computed on the six WITHHELD
                # benchmarks only. known_label_recovery_at_25 is deliberately
                # excluded: it is in-sample for the ensemble and out-of-sample
                # for B1-B4, so it cannot be compared across them.
                "verdict_metrics_are_heldout_only": True,
                "excluded_from_verdict": ["known_label_recovery_at_25"],
                # BUG FIX (both external reviews, correct): these read the
                # ALL-GENE recall after the primary evaluation moved to the
                # common population, so the summary carried an unqualified
                # 0/6 while the primary table reported 2/6. Named fields for
                # both populations, no ambiguous key.
                "ensemble_recall_at_25_common": float(
                    e.get("heldout_recall_at_25_common_population", np.nan)),
                "ensemble_recall_at_25_all_genes": float(
                    e.get("heldout_recall_at_25", np.nan)),
                "best_baseline_recall_at_25_common": float(
                    best.get("heldout_recall_at_25_common_population",
                             np.nan)),
                "best_baseline_recall_at_25_all_genes": float(
                    best.get("heldout_recall_at_25", np.nan)),
            }
            print(f"    best baseline: {best.method} "
                  f"(median percentile {best.median_benchmark_percentile:.1f})")
            # Two decimal places: rounding 2.99 to "3.0" against a 3.0
            # threshold produces a message that reads as self-contradictory.
            print(f"    ensemble gain: {gain:+.2f} percentile points "
                  f"(need >= {CFG.min_meaningful_margin_pct}), "
                  f"{td_gain:+d} benchmarks in top decile "
                  f"(need >= {CFG.min_top_decile_gain})")
            if verdict["ensemble_beats_baselines"]:
                print("    -> ensemble meets the PREDEFINED improvement rule")
            else:
                why = []
                if not meets_margin:
                    why.append("margin below the predefined threshold")
                if not meets_targets:
                    why.append("top-decile gain is within one-target noise at n=6")
                print(f"    -> ensemble does NOT meet the predefined rule "
                      f"({'; '.join(why)}).")
                print("       Report the simplest adequate baseline as the "
                      "primary method. The ensemble may still be reported for "
                      "calibrated uncertainty, leakage control and feature "
                      "interpretability - but not as better at the primary "
                      "endpoint.")
    return out, verdict


# ----------------------------------------------------------------------------
# 3.12 PROTEIN / NEPC LATE-FUSION SENSITIVITY MODEL
# ----------------------------------------------------------------------------
# Fitted only where the extra measurements exist. Missing proteomic data is
# NOT imputed: mass spectrometry under-detects membrane proteins, so an
# imputed value would be a statement about detectability dressed as a
# statement about abundance.
#
# This is reported as a sensitivity analysis on a reduced N, never as the
# primary model, because restricting to protein-detected genes selects for
# abundant, well-behaved proteins - which is a different population from the
# surfaceome the study set out to screen.
# ----------------------------------------------------------------------------
def fusion_sensitivity_model(f, y, train_exclude, bio_cols, seed=CFG.seed,
                             benchmark_mask=None):
    """Does adding protein and NEPC evidence change benchmark recovery?

    CRITICAL BUG FIX (external review, confirmed): this function took ONE mask
    and used it for two different jobs - excluding genes from training AND
    identifying the evaluation benchmarks. When the caller began passing
    `train_exclude` (benchmarks PLUS 17 off-class clinical-stage targets) to
    keep known targets out of the pseudo-negative pool, those 17 genes were
    silently promoted to BENCHMARKS here. The function then reported "11 of 15
    benchmarks in the top decile" for a set that is not the six prespecified
    prostate benchmarks at all, so the fusion comparison answered a different
    question from the one in its own caption.

    The two jobs are now separate arguments. `train_exclude` governs fitting;
    `benchmark_mask` governs evaluation and nothing else. When
    `benchmark_mask` is omitted it falls back to `train_exclude`, which
    reproduces the old behaviour - so the caller must pass it explicitly, and
    a regression test asserts the reported benchmark count equals the true
    benchmark count.
    """
    # External review, correctly: leaving `benchmark_mask=None` as a silent
    # fallback to `train_exclude` preserves the exact defect this signature
    # was introduced to remove. It is now an error.
    if benchmark_mask is None:
        raise ValueError(
            "fusion_sensitivity_model requires an explicit benchmark_mask. "
            "Falling back to train_exclude is what caused 17 off-class "
            "clinical targets to be counted as held-out benchmarks. Pass "
            "benchmark_mask=is_heldout (evaluation) alongside train_exclude "
            "(fitting).")
    train_exclude = np.asarray(train_exclude, dtype=bool)
    is_heldout = np.asarray(benchmark_mask, dtype=bool)
    extra = [c for c in FUSION_FEATURES if c in f.columns]
    if not extra:
        print("    no protein/NEPC columns present; fusion analysis skipped")
        return None

    complete = f[extra].notna().all(axis=1).values
    # Positives for FITTING exclude everything withheld from training.
    n_pos_sub = int(((y == 1) & complete & ~train_exclude).sum())
    # Benchmarks for EVALUATION are the prespecified set only.
    n_held_sub = int((is_heldout & complete).sum())
    print(f"    genes with complete protein/NEPC evidence: {complete.sum():,}"
          f" / {len(f):,} ({100*complete.mean():.0f}%)")
    print(f"    positives retained: {n_pos_sub} | benchmarks retained: {n_held_sub}")
    if n_pos_sub < 15 or n_held_sub < 2:
        print("    too few labelled genes survive the completeness filter; "
              "fusion analysis not interpretable and is skipped")
        return None

    sub = f.loc[complete].reset_index(drop=True)
    # Two masks, two jobs: `excl_sub` is withheld from fitting; `held_sub`
    # measures benchmark recovery.
    y_sub = y[complete]
    excl_sub = train_exclude[complete]
    held_sub = is_heldout[complete]

    results = {}
    for name, cols in [("biology only (same subset)", bio_cols),
                       ("biology + protein/NEPC", bio_cols + extra)]:
        X = build_design(sub, [c for c in cols if c in sub.columns])
        scores, _, _, _ = pu_bagged_nested_cv(
            X, y_sub, excl_sub, label=f"fusion: {name}", seed=seed)
        s = scores["Ensemble"].values
        rank = pd.Series(s).rank(ascending=False, method="average")
        pct = 100 * (1 - rank / len(s))
        results[name] = {
            "n_genes": int(len(sub)),
            "median_benchmark_percentile": float(np.nanmedian(pct[held_sub])),
            "benchmarks_in_top_decile": int((pct[held_sub] >= 90).sum()),
            "n_benchmarks": int(held_sub.sum()),
        }

    out = pd.DataFrame(results).T.reset_index().rename(columns={"index": "model"})
    save_table(out, "table15_fusion_sensitivity.csv",
               caption="Protein/NEPC late fusion, on the complete-evidence "
                       "subset only (sensitivity analysis, reduced N)")
    a = results["biology only (same subset)"]["median_benchmark_percentile"]
    b = results["biology + protein/NEPC"]["median_benchmark_percentile"]
    print(f"    fusion effect on benchmark recovery: {b - a:+.2f} percentile "
          f"points on the SAME subset")
    print("      Compared within the subset, so the difference reflects the "
          "added features and not the change of population. The primary "
          "model remains the biology-only model on all genes.")
    return {"results": results, "delta_percentile": float(b - a),
            "n_subset": int(complete.sum()), "n_positives_subset": n_pos_sub,
            "n_benchmarks_evaluated": n_held_sub,
            "n_withheld_from_fitting": int((excl_sub).sum()),
            "benchmark_mask_supplied": bool(benchmark_mask is not None)}


# ============================================================================
# 3.13 BLOCKING PREFLIGHT LABEL-INTEGRITY AUDIT
# ============================================================================
# Runs BEFORE any model is fitted, and RAISES on a critical failure.
#
# This replaces a retrospective stop/go assessment for label integrity. The
# distinction matters: a retrospective gate tells you after several hours of
# compute that the labels were never trustworthy, by which point there is a
# results table on screen and an incentive to argue with the gate. A blocking
# preflight makes the failure cheap and the decision unarguable.
#
# SEVERITY POLICY
#   critical : structural integrity of the labels and features. Blocking in
#              BOTH demo and live, because these are code and data defects
#              rather than provenance gaps, and no run of any kind is
#              interpretable with them present.
#   provenance-critical : blocking in LIVE only. In DEMO the labels are
#              synthetic by construction, so registry verification is not
#              merely absent but meaningless.
#   warning  : recorded, never blocking.
# ============================================================================
class PreflightError(RuntimeError):
    """Raised when label integrity fails before training."""


def preflight_label_audit(f, y, train_exclude, feature_cols, label_audit=None,
                          save_outputs=True, benchmark_mask=None,
                          benchmark_audit=None, strict=None):
    """Block training unless the label set is structurally sound.

    BUG FIX (external review, confirmed): this function also conflated the two
    masks. Called with `train_exclude`, it reported "23 held-out benchmarks"
    when there are six - the other 17 are off-class clinical-stage targets
    withheld from fitting. The benchmark-availability check then validated the
    wrong quantity. Training exclusions and evaluation benchmarks are now
    separate arguments and separately reported.
    """
    if benchmark_mask is None:
        raise ValueError(
            "preflight_label_audit requires an explicit benchmark_mask. "
            "Defaulting it to train_exclude made the audit report 23 "
            "held-out benchmarks when there are six. Pass benchmark_mask "
            "separately from train_exclude.")
    train_exclude = np.asarray(train_exclude, dtype=bool)
    benchmark_mask = np.asarray(benchmark_mask, dtype=bool)
    strict = (RUN_MODE == "LIVE") if strict is None else strict
    checks = []

    def add(name, severity, passed, detail):
        checks.append({"check": name, "severity": severity,
                       "passed": bool(passed), "detail": detail})

    # Fitting population: everything withheld from training is out of both
    # classes. Evaluation benchmark count is reported from its own mask.
    pos_mask = (y == 1) & ~train_exclude
    unl_mask = (y == 0) & ~train_exclude
    n_pos, n_unl = int(pos_mask.sum()), int(unl_mask.sum())
    n_held = int(benchmark_mask.sum())
    n_excluded = int(train_exclude.sum())
    n_off_class = int((train_exclude & ~benchmark_mask).sum())
    declared_pos = [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]

    # --- 1. Missing or inconsistent labels ---------------------------------
    n_nan = int(pd.isna(y).sum()) if y.dtype.kind == "f" else 0
    add("Labels free of missing values", "critical", n_nan == 0,
        f"{n_nan} NaN labels")

    matched = set(f.gene) & set(declared_pos)
    missing_pos = sorted(set(declared_pos) - set(f.gene))
    frac_matched = len(matched) / max(len(declared_pos), 1)
    add("Declared positives present in the feature table", "critical",
        frac_matched >= 0.50,
        f"{len(matched)}/{len(declared_pos)} matched ({frac_matched:.0%})"
        + (f"; first unmatched: {missing_pos[:6]}" if missing_pos else ""))

    # --- 2. Duplicate records ----------------------------------------------
    n_dup = int(f.gene.duplicated().sum())
    add("No duplicate gene rows", "critical", n_dup == 0,
        f"{n_dup} duplicated gene symbols")

    dup_decl = len(declared_pos) - len(set(declared_pos))
    add("No duplicate declared positives", "critical", dup_decl == 0,
        f"{dup_decl} repeated entries in the positive list")

    # --- 3. Conflicting annotations ----------------------------------------
    overlap_pos_held = set(declared_pos) & set(PROSTATE_HELDOUT_POSITIVES)
    add("Positives and held-out benchmarks are disjoint", "critical",
        not overlap_pos_held,
        f"overlap: {sorted(overlap_pos_held)}" if overlap_pos_held else "disjoint")

    overlap_excl = (set(declared_pos) | set(PROSTATE_HELDOUT_POSITIVES)) & \
                   set(SURFACEOME_EXCLUSION_CONTROLS)
    add("Exclusion controls are not also labelled", "critical",
        not overlap_excl,
        f"overlap: {sorted(overlap_excl)}" if overlap_excl else "disjoint")

    both = int((pos_mask & benchmark_mask).sum())
    add("No gene is both a positive and a benchmark", "critical", both == 0,
        f"{both} genes carry both roles")

    # --- 4. Class imbalance -------------------------------------------------
    prevalence = n_pos / max(n_pos + n_unl, 1)
    add("Enough positives to fit", "critical", n_pos >= 10,
        f"{n_pos} positives, {n_unl:,} unlabeled (prevalence {prevalence:.3%})")
    add("Positives sufficient for stable estimates", "warning", n_pos >= 25,
        f"{n_pos} positives; below 25 the intervals are very wide")
    add("Benchmarks available for the primary endpoint", "critical", n_held >= 2,
        f"{n_held} eligible held-out benchmark(s); {n_off_class} further "
        f"off-class clinical target(s) withheld from fitting only "
        f"({n_excluded} excluded from training in total)")

    # --- 5. Potential data leakage -----------------------------------------
    leak_in_bio = [c for c in feature_cols if c in RESEARCH_INTENSITY_FEATURES]
    add("No research-intensity features in the biology set", "critical",
        not leak_in_bio, f"leaked: {leak_in_bio}" if leak_in_bio else "clean")

    # A single feature that separates the classes perfectly is a label proxy,
    # not biology. This catches an accidentally-derived feature before it
    # produces a triumphant AUC.
    perfect = []
    Xc = f[feature_cols].apply(pd.to_numeric, errors="coerce")
    lab_idx = np.where(pos_mask | unl_mask)[0]
    yy = y[lab_idx]
    if len(np.unique(yy)) == 2:
        for c in feature_cols:
            v = Xc[c].values[lab_idx]
            ok = np.isfinite(v)
            if ok.sum() < 20 or len(np.unique(yy[ok])) < 2:
                continue
            a = roc_auc_score(yy[ok], v[ok])
            if max(a, 1 - a) > 0.995:
                perfect.append((c, round(float(max(a, 1 - a)), 4)))
    add("No single feature separates the classes perfectly", "critical",
        not perfect, f"suspect: {perfect}" if perfect else "none above AUC 0.995")

    dup_rows = int(Xc.round(10).duplicated().sum())
    add("Feature rows are not mass-duplicated", "warning", dup_rows < 0.05 * len(f),
        f"{dup_rows} duplicate feature rows")

    # --- 6. Provenance (blocking in LIVE only) -----------------------------
    if label_audit is not None:
        add("Positive labels verified against the trial registry",
            "provenance-critical", label_audit.get("pct_verified", 0) >= 100,
            f"{label_audit.get('pct_verified', 0):.0f}% of positives carry a trial ID")
        add("Label inclusion rule fully resolved", "provenance-critical",
            not label_audit.get("unresolved_rule_fields"),
            f"unresolved: {label_audit.get('unresolved_rule_fields')}")
    if benchmark_audit is not None:
        # BUG FIX (external review, correct): this gate read ONLY
        # `n_unverified`, which counts records whose accessibility_evidence
        # literally equals "UNVERIFIED". Typing any other text into that one
        # field - including text that asserts nothing - made the gate pass
        # while every `modality` stayed UNVERIFIED and every `source_url`
        # stayed empty. The audit had become descriptive while the
        # ENFORCEMENT stayed behind it.
        #
        # The gate now requires every evidence field to be populated, and a
        # missing field is never read as verification.
        n_unv = benchmark_audit.get("n_unverified", 1)
        n_mod_unv = benchmark_audit.get("n_modality_unverified", 1)
        n_no_src = benchmark_audit.get("n_without_source_url", 1)
        gaps = []
        if n_unv:
            gaps.append(f"{n_unv} without accessibility evidence")
        if n_mod_unv:
            gaps.append(f"{n_mod_unv} without a verified modality")
        if n_no_src:
            gaps.append(f"{n_no_src} without a source URL")
        add("Benchmark accessibility, modality AND source verified",
            "provenance-critical",
            (n_unv == 0) and (n_mod_unv == 0) and (n_no_src == 0),
            "; ".join(gaps) if gaps else "all evidence fields populated")
        # A set revised after its performance was observed is not a
        # prospectively independent endpoint, and recording a freeze date now
        # cannot restore independence retrospectively.
        if benchmark_audit.get("revised_after_seeing_results"):
            add("Benchmark set independent of observed model performance",
                "provenance-critical", False,
                "the current set was revised AFTER model performance was "
                "observed; it is a development endpoint, not an independent "
                "test set. Freeze a set chosen without reference to scores, "
                "or report both sets and claim no independence.")
        add("Benchmark set frozen with a recorded date", "provenance-critical",
            not str(benchmark_audit.get("frozen_on", "UNSET")).startswith("UNSET"),
            f"frozen_on = {benchmark_audit.get('frozen_on')}")

    audit = pd.DataFrame(checks)
    # Tests MUST pass save_outputs=False: writing a fixture audit here is what
    # replaced the real label audit in the previous build, and the corrupted
    # file then reported "0/74 positives matched" as though it described the
    # actual analysis.
    if save_outputs:
        save_table(audit, "table16_preflight_label_audit.csv",
                   caption="Blocking label-integrity audit, evaluated BEFORE "
                           "training")

    blocking = audit[(~audit.passed) & (
        (audit.severity == "critical") |
        ((audit.severity == "provenance-critical") & strict))]
    warned = audit[(~audit.passed) & (audit.severity == "warning")]

    show_heading("Preflight label-integrity audit", 3)
    for r in audit.itertuples():
        if not r.passed:
            print(f"    [{'BLOCK' if r.check in set(blocking.check) else 'note '}] "
                  f"{r.check}: {r.detail}")
    print(f"    {int(audit.passed.sum())}/{len(audit)} checks passed | "
          f"{len(blocking)} blocking | {len(warned)} warnings")

    if len(blocking):
        raise PreflightError(
            "Training BLOCKED by the preflight label-integrity audit.\n\n"
            + "\n".join(f"  - {r.check}: {r.detail}" for r in blocking.itertuples())
            + "\n\nThese are critical checks: a model fitted on labels that "
              "fail them is not interpretable, so the run stops here rather "
              "than producing a results table that would have to be withdrawn. "
              "Resolve each item and re-run. Full audit: "
              "outputs/05_tables/table16_preflight_label_audit.csv")

    if strict:
        print("    LIVE mode: provenance checks were enforced as blocking.")
    else:
        n_prov = int((~audit.passed & (audit.severity == "provenance-critical")).sum())
        if n_prov:
            print(f"    DEMO mode: {n_prov} provenance check(s) failed but did "
                  f"not block, because demo labels are synthetic and registry "
                  f"verification of them would be meaningless. These WILL "
                  f"block a LIVE run.")
    return audit


# ----------------------------------------------------------------------------
# 3.x HISTORICAL BENCHMARK SENSITIVITY (NEW - external review Finding F)
# ----------------------------------------------------------------------------
# The historical six-gene set was DECLARED last round but never evaluated, so
# the effect of substituting TMPRSS2/SLC45A3 for PSCA/KLK2 on the headline
# metrics was invisible. Because the substitution was made with knowledge of
# model performance, that effect is exactly what a reader needs to see.
#
# This reports every primary endpoint metric for BOTH sets on the SAME
# ranking. It is a sensitivity analysis, not a second endpoint: neither set is
# independent of development, and the historical set must never be relabelled
# as a prospectively held-out endpoint.
def historical_benchmark_sensitivity(ranked, score_col="ensemble_score"):
    """Primary endpoint metrics for the revised and historical benchmark sets."""
    if score_col not in ranked.columns or "percentile" not in ranked.columns:
        print("    historical benchmark sensitivity SKIPPED: ranking absent")
        return None
    sets = {
        "revised (current primary endpoint)": list(PROSTATE_HELDOUT_POSITIVES),
        "historical (pre-revision)": list(HISTORICAL_HELDOUT_POSITIVES),
    }
    rows = []
    for name, genes in sets.items():
        present = [g for g in genes if g in set(ranked.gene)]
        sub = ranked[ranked.gene.isin(present)]
        if not len(sub):
            continue
        rows.append({
            "benchmark_set": name,
            "n_declared": len(genes),
            "n_present_in_ranking": len(present),
            "median_percentile": round(float(sub.percentile.median()), 2),
            "in_top_decile": int((sub.percentile >= 90).sum()),
            "in_top_25_ranks": int((sub["rank"] <= 25).sum()),
            "in_top_50_ranks": int((sub["rank"] <= 50).sum()),
            "n_eligible_primary": int(
                sub.get("eligible_primary", pd.Series(False, index=sub.index))
                .fillna(False).sum()),
            "members": ", ".join(sorted(present)),
        })
    if len(rows) < 2:
        print("    historical benchmark sensitivity SKIPPED: a set is absent")
        return None
    out = pd.DataFrame(rows)
    rev = out.iloc[0]
    his = out.iloc[1]
    delta = float(rev.median_percentile - his.median_percentile)
    # BUG FIX (external review, correct): recoverability was reported against
    # eligible_primary alone and described as "no model can rank it into an
    # eligible shortlist", which over-generalised. STEAP1 fails the PRIMARY
    # screen yet qualifies in mCRPC and NEPC - a benchmark can be clinically
    # relevant in one disease context while failing a filter designed for
    # another. Recoverability is now reported PER CONTEXT.
    #
    # Two distinct endpoints are also separated, because they answer
    # different questions and neither should silently stand in for the other:
    #   unconditional - how the model ranks ALL prespecified benchmarks,
    #                   including those that fail biological eligibility
    #                   (measures target recognition);
    #   conditional   - how it ranks the benchmarks that satisfy the
    #                   eligibility criteria FOR THAT CONTEXT
    #                   (measures ranking given eligibility).
    contexts = [("primary", "eligible_primary"),
                ("mcrpc", "eligible_mcrpc"), ("nepc", "eligible_nepc")]
    recoverability, endpoints = [], []
    for name, genes in sets.items():
        sub = ranked[ranked.gene.isin(genes)]
        for ctx, col in contexts:
            if col not in sub.columns:
                continue
            elig = sub[col].fillna(False).astype(bool)
            bad = sorted(sub.loc[~elig, "gene"])
            recoverability.append({
                "benchmark_set": name, "context": ctx,
                "n_eligible": int(elig.sum()), "n_total": int(len(sub)),
                "not_recoverable_in_this_context": ", ".join(bad) or "none",
            })
            endpoints.append({
                "benchmark_set": name, "context": ctx,
                "endpoint": "unconditional (all benchmarks)",
                "n": int(len(sub)),
                "median_percentile": round(float(sub.percentile.median()), 2),
                "in_top_decile": int((sub.percentile >= 90).sum()),
            })
            if int(elig.sum()):
                se = sub[elig]
                endpoints.append({
                    "benchmark_set": name, "context": ctx,
                    "endpoint": "conditional (eligible benchmarks only)",
                    "n": int(len(se)),
                    "median_percentile": round(
                        float(se.percentile.median()), 2),
                    "in_top_decile": int((se.percentile >= 90).sum()),
                })
    save_table(pd.DataFrame(recoverability),
               "table27_benchmark_recoverability_by_context.csv",
               caption="Benchmark eligibility PER DISEASE CONTEXT. A "
                       "benchmark failing the primary screen may still "
                       "qualify in mCRPC or NEPC; 'not recoverable' is a "
                       "context-specific statement, never a global one.")
    save_table(pd.DataFrame(endpoints),
               "table28_benchmark_endpoints_conditional.csv",
               caption="Unconditional benchmark recovery (target "
                       "recognition) and conditional recovery among eligible "
                       "benchmarks (ranking given eligibility). These answer "
                       "different questions; neither replaces the other.")
    unrecoverable = [f"{r['benchmark_set']} / {r['context']}: "
                     f"{r['not_recoverable_in_this_context']}"
                     for r in recoverability
                     if r["not_recoverable_in_this_context"] != "none"]
    save_table(out, "table26_historical_benchmark_sensitivity.csv",
               caption="Primary endpoint metrics for the revised and the "
                       "historical benchmark set on the SAME ranking. The "
                       "revised set was chosen AFTER observing performance, "
                       "so neither set is an independent endpoint and this "
                       "is reported as a sensitivity analysis only.")
    print(f"    benchmark-set sensitivity: revised median percentile "
          f"{rev.median_percentile:.2f} vs historical "
          f"{his.median_percentile:.2f} (delta {delta:+.2f}); top decile "
          f"{int(rev.in_top_decile)}/{int(rev.n_present_in_ranking)} vs "
          f"{int(his.in_top_decile)}/{int(his.n_present_in_ranking)}")
    if unrecoverable:
        for u in unrecoverable:
            print(f"    NOT ELIGIBLE IN THIS CONTEXT - {u}: no model can "
                  f"rank these into THAT context's eligible shortlist. They "
                  f"may still be eligible in another context; see "
                  f"table27.")
    return {"table": out.to_dict("records"),
            "delta_median_percentile": round(delta, 2),
            "revised_set": sets["revised (current primary endpoint)"],
            "historical_set": sets["historical (pre-revision)"],
            "unrecoverable_benchmarks_by_context": unrecoverable,
            "recoverability_by_context": recoverability,
            "independence": ("NEITHER set is independent of development; the "
                             "revised set was chosen after observing scores")}

---
## Section 4 — Survival analysis

**Power is computed before any model is fitted, and models the data cannot support are refused rather than reported.**

TCGA-PRAD is a surgically treated, mostly favourable-risk primary cohort with few biochemical-recurrence events. Non-European ancestry subgroups are small. Where events-per-variable falls below 10, a multivariable Cox model is not reliably estimable, and an ancestry-stratified multivariable model usually is not estimable at all. The code prints each refusal with its arithmetic, so a reviewer can see what was declined and why.

Two further safeguards: the expression split for Kaplan–Meier is **pre-declared in the config**, never optimised — choosing the cut point that maximises separation badly inflates type-I error and is among the most common irreproducibility sources in expression–survival papers. And the proportional-hazards assumption is tested, because a single hazard ratio from a model that violates it is a time-averaged artefact.

In [6]:
# =============================================================================
# SECTION 4 - SURVIVAL ANALYSIS: KAPLAN-MEIER AND MULTIVARIABLE COX
# =============================================================================
# Endpoint: biochemical recurrence-free survival (BCR-FS).
#
# THE BINDING CONSTRAINT, STATED BEFORE ANY MODEL IS FITTED
#   TCGA-PRAD is a surgically-treated, mostly favourable-risk primary cohort
#   with few BCR events. Ancestry subgroups other than European are small, and
#   the AFR subgroup typically contributes on the order of ten events. With
#   events-per-variable below 10, a multivariable Cox model is not estimable
#   with any reliability, and an ancestry-stratified multivariable model is
#   usually not estimable at all.
#
#   This module therefore computes the available power BEFORE fitting, refuses
#   to fit models the data cannot support, and reports what it refused. A
#   reviewer should be able to see the refusal, not just its absence.
# =============================================================================

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.exceptions import ConvergenceError


# ----------------------------------------------------------------------------
# 4.1 Power and feasibility accounting
# ----------------------------------------------------------------------------
def endpoint_label(clin):
    """The endpoint actually present in the data, for titles and captions.

    Never hard-code "biochemical recurrence": on the LIVE path the available
    column is usually disease-free or progression-free survival, which is a
    different clinical event.
    """
    if "endpoint_name" in clin.columns and clin.endpoint_name.notna().any():
        raw = str(clin.endpoint_name.dropna().iloc[0])
    else:
        raw = "survival"
    pretty = {"biochemical_recurrence": "BCR-free survival",
              "disease_free_survival": "Disease-free survival",
              "progression_free_survival": "Progression-free survival",
              "simulated_recurrence": "Simulated recurrence-free survival",
              "unavailable": "Survival (endpoint undocumented)"}
    return pretty.get(raw, raw.replace("_", " ").capitalize())


def _ncp_for_power(crit, df=2, power=0.80, lo=0.0, hi=200.0, tol=1e-4):
    """Non-centrality parameter giving `power` for a chi-square test.

    Solved by bisection on the non-central chi-square survival function; there
    is no closed form for df > 1, and using the 1-df z-based formula for a
    multi-group test understates the required effect.
    """
    f = lambda L: st.ncx2.sf(crit, df, L) - power
    if f(lo) > 0:
        return lo
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if f(mid) > 0:
            hi = mid
        else:
            lo = mid
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi)


def survival_power_report(clin, by="ancestry"):
    """Events, EPV ceiling, and the minimum detectable hazard ratio per group.

    The minimum detectable HR uses Schoenfeld's formula for the log-rank test:
        n_events >= (z_{1-a/2} + z_{1-b})^2 / (p(1-p) * log(HR)^2)
    solved for HR at 80% power, two-sided alpha 0.05, balanced split p=0.5.
    """
    rows = []
    z_a, z_b = st.norm.ppf(0.975), st.norm.ppf(0.80)
    # Chi-square critical value for the 3-group (2 df) log-rank actually run
    # by km_by_expression, used for the matching power statement below.
    chi2_crit_2df = st.chi2.ppf(0.95, df=2)
    for grp, d in list(clin.groupby(by)) + [("ALL", clin)]:
        d = d.dropna(subset=["surv_time", "surv_event"])
        # One eligibility rule, shared with the loader and the endpoint
        # validator (survival_eligibility_mask), so the patients counted as
        # usable by quality control are exactly the patients analysed.
        d = d[survival_eligibility_mask(d)]
        e = int(d.surv_event.sum())
        n = len(d)
        # BUG FIX 1 (found auditing this module): "median follow-up" was the
        # median of the OBSERVED time over ALL patients, which mixes
        # time-to-event with time-to-censoring. In a cohort with ~28% events
        # that understates the real median follow-up by roughly two-fold
        # (verified: 25.2 vs 55.0 months on a matched simulation). The
        # standard estimator is the reverse Kaplan-Meier (Schemper-Smith):
        # treat CENSORING as the event and take the median. This number is
        # quoted in the abstract's power sentence, so it has to be the right
        # quantity.
        med_fu = np.nan
        if n and (1 - d.surv_event).sum() > 0:
            try:
                rkm = KaplanMeierFitter().fit(d.surv_time, 1 - d.surv_event)
                med_fu = float(rkm.median_survival_time_)
            except Exception:
                med_fu = np.nan
        if not np.isfinite(med_fu) and n:
            # Reverse KM is undefined when censoring never reaches 50%;
            # fall back to the median among censored observations and say so.
            cens = d.loc[d.surv_event == 0, "surv_time"]
            med_fu = float(cens.median()) if len(cens) else np.nan
        # BUG FIX 2: the minimum detectable HR came from Schoenfeld's
        # TWO-GROUP balanced formula, but no test in the pipeline is a
        # two-group comparison of that form - km_by_expression runs a
        # THREE-group log-rank (2 df) and cox_multivariable uses CONTINUOUS
        # standardised expression. The single number therefore described a
        # test that is never performed. All three are now reported, each
        # labelled with the test it belongs to.
        # BUG FIX (external review, correct): the power report described a
        # THREE-group (2 df) log-rank as "the test as run", but
        # CFG.expr_split defaults to "median", which is a TWO-group test.
        # Last round's correction over-shot: the form must be derived from
        # the configuration, not fixed.
        split_is_tertile = str(CFG.expr_split).lower().startswith("tert")
        if e > 0:
            # Two-group balanced log-rank (reference / planning value).
            hr_2grp = float(np.exp((z_a + z_b) / np.sqrt(0.25 * e)))
            # Three-group log-rank, 2 df: the non-centrality required for 80%
            # power is larger than for a 1-df test, so the detectable effect
            # is correspondingly larger. Equal thirds -> the extreme-group
            # contrast carries 2/3 of the events at allocation 0.5.
            ncp_2df = _ncp_for_power(chi2_crit_2df, df=2, power=0.80)
            hr_3grp = float(np.exp(np.sqrt(ncp_2df / (0.25 * e * (2 / 3)))))
            # Continuous Cox on a standardised covariate: HR per 1 SD.
            hr_per_sd = float(np.exp((z_a + z_b) / np.sqrt(e)))
        else:
            hr_2grp = hr_3grp = hr_per_sd = np.nan
        rows.append({
            "group": grp, "n": n, "events": e,
            "km_test_as_configured": ("3-group log-rank (2 df)"
                                      if split_is_tertile
                                      else "2-group log-rank (1 df)"),
            "min_detectable_HR_for_configured_test": (
                (round(hr_3grp, 2) if split_is_tertile else round(hr_2grp, 2))
                if e else np.nan),
            "event_rate": round(e / n, 3) if n else np.nan,
            "max_covariates_at_EPV10": e // CFG.min_events_per_covariate,
            # BUG FIX (external review, correct): the three-group column was
            # named "_as_run" regardless of configuration, so a report from a
            # median-split analysis carried a column asserting a three-group
            # test had been performed. The configured test is named
            # separately above; these two are PLANNING SCENARIOS.
            "min_detectable_HR_2group_logrank_scenario":
                round(hr_2grp, 2) if e else np.nan,
            "min_detectable_HR_3group_logrank_scenario":
                round(hr_3grp, 2) if e else np.nan,
            "min_detectable_HR_per_SD_cox_as_run":
                round(hr_per_sd, 2) if e else np.nan,
            "median_followup_months_reverseKM":
                round(med_fu, 1) if np.isfinite(med_fu) else np.nan,
        })
    rep = pd.DataFrame(rows)
    save_table(rep, "table6_survival_power.csv",
               caption="Events, EPV ceiling, and the minimum detectable "
                       "effect for EACH test actually run: the 3-group "
                       "test named in km_test_as_configured, the continuous "
                       "per-SD Cox, and the two- and three-group log-rank "
                       "PLANNING SCENARIOS (only one of which matches the "
                       "configured split). Median follow-up is the reverse "
                       "Kaplan-Meier estimate, not the median observed time.")
    if not IN_NOTEBOOK:
        print(rep.to_string(index=False))
    return rep


# ----------------------------------------------------------------------------
# 4.2 Kaplan-Meier by candidate expression
# ----------------------------------------------------------------------------
def km_by_expression(expr_log, clin, gene, split=None, ax=None):
    """KM curves for high vs low expression of one candidate, with log-rank.

    The split threshold is declared in CFG, not optimised. Choosing the cut
    point that maximises separation (a 'minimum p-value' or maximally-selected
    rank cut) inflates the type-I error substantially, and is one of the most
    common irreproducibility sources in expression-survival papers.
    """
    # BUG FIX (external review, correct): three skip paths returned None -
    # gene absent, too few eligible patients, too few events - so some
    # skipped comparisons appeared in the audit as NOT FITTED while others
    # vanished entirely. A reader could not distinguish an unperformed test
    # from one that was never attempted. Every attempted candidate now
    # yields a structured record with its reason.
    def _nf(reason, n=0, events=0, split_used=None):
        return {"gene": gene, "status": "NOT FITTED",
                "split": split_used, "threshold_rule": split_used,
                "reason": reason, "n": int(n), "events": int(events),
                "logrank_p": np.nan}

    if gene not in expr_log.index:
        return _nf("gene absent from the expression matrix")
    e = expr_log.loc[gene]
    d = clin.dropna(subset=["surv_time", "surv_event"]).copy()
    d = d[survival_eligibility_mask(d)]
    d["expr"] = d["sample"].map(e)
    d = d.dropna(subset=["expr"])
    if len(d) < 30:
        return _nf(f"insufficient eligible patients ({len(d)} < 30)",
                   len(d), d.surv_event.sum())
    if d.surv_event.sum() < 10:
        return _nf(f"insufficient events ({int(d.surv_event.sum())} < 10)",
                   len(d), d.surv_event.sum())

    # BUG FIX (both external reviews): three separate defects here.
    #
    # (a) `split=CFG.expr_split` was a DEFAULT ARGUMENT, evaluated once when
    #     the function was defined. A later change to CFG.expr_split would
    #     not reach any call that omitted the argument, so the figure and
    #     the configuration could describe different analyses.
    # (b) every value other than "median" was treated as a tertile request,
    #     so a typo or an unsupported setting silently changed the
    #     statistical analysis instead of failing.
    # (c) with heavily tied expression - common for low-abundance surfaceome
    #     genes - the two tertile cutpoints coincide and pd.cut raised
    #     "Bin edges must be unique". Reproduced by both reviewers with 36
    #     patients and identical expression.
    #
    # The prespecified comparison is never silently replaced by a different
    # one: when the requested groups are not constructible the function
    # returns an explicit NOT FITTED record with its reason.
    if split is None:
        split = getattr(CFG, "expr_split", "median")
    split = str(split).strip().lower()
    if split not in {"median", "tertile"}:
        raise ValueError(
            f"Unsupported CFG.expr_split={split!r}. Use 'median' or "
            f"'tertile'. Treating an unrecognised value as a tertile split "
            f"would run an analysis nobody configured.")

    _not_fitted = {"gene": gene, "status": "NOT FITTED", "split": split,
                   "n": int(len(d)), "events": int(d.surv_event.sum()),
                   "p": np.nan, "test": "none"}
    if split == "median":
        thr = float(d.expr.median())
        # A constant (or near-constant) expression vector cannot be split:
        # every patient falls on one side and the "two-group" comparison is
        # fabricated.
        if not np.isfinite(thr) or d.expr.nunique() < 2:
            _not_fitted["reason"] = (
                f"expression is constant across the {len(d)} eligible "
                f"patients (nunique={int(d.expr.nunique())}); a median split "
                f"cannot define two groups")
            return _not_fitted
        d["grp"] = np.where(d.expr > thr, "High", "Low")
        if d.grp.nunique() < 2:
            _not_fitted["reason"] = (
                "the median split placed every patient in one group")
            return _not_fitted
    else:
        q = d.expr.quantile([1 / 3, 2 / 3]).to_numpy()
        if not np.isfinite(q).all() or q[0] >= q[1]:
            _not_fitted["reason"] = (
                f"tertile cutpoints are not unique (q33={q[0]:.4g}, "
                f"q67={q[1]:.4g}) because expression is heavily tied across "
                f"the {len(d)} eligible patients; the prespecified "
                f"three-group comparison is not estimable. Silently "
                f"collapsing to two groups would substitute a different "
                f"analysis.")
            return _not_fitted
        d["grp"] = pd.cut(d.expr, [-np.inf, q[0], q[1], np.inf],
                          labels=["Low", "Mid", "High"]).astype(str)

    # BUG FIX (external review, reproduced here): unique tertile cutpoints
    # do NOT guarantee three populated groups. With expression taking two
    # values (18 zeros, 18 ones) the cutpoints are 0 and 1 - distinct, so
    # last round's uniqueness check passed - but the groups are Low=18,
    # Mid=18, High=0. The code then found two present groups, ran the
    # TWO-group log-rank, and returned a record still labelled
    # threshold_rule="tertile". An internally consistent output describing a
    # different analysis from the one configured.
    #
    # The prespecified comparison must exist in full or not be attempted.
    _expected = ["Low", "High"] if split == "median" else ["Low", "Mid",
                                                           "High"]
    _counts = d.grp.value_counts()
    _empty = [g for g in _expected if int(_counts.get(g, 0)) == 0]
    if _empty:
        return _nf(
            f"the prespecified {split} split did not produce all required "
            f"groups: {', '.join(g + '=' + str(int(_counts.get(g, 0))) for g in _expected)}"
            f" (empty: {_empty}). Fitting the groups that happen to be "
            f"populated would substitute a different comparison while still "
            f"reporting a {split} rule.",
            len(d), d.surv_event.sum(), split)
    _tiny = [g for g in _expected
             if int(_counts.get(g, 0)) < CFG.min_km_group_size]
    if _tiny:
        return _nf(
            f"group(s) below the prespecified minimum size "
            f"{CFG.min_km_group_size}: "
            f"{ {g: int(_counts.get(g, 0)) for g in _tiny} }",
            len(d), d.surv_event.sum(), split)
    groups = list(_expected)
    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 4.2))
    colors = {"Low": "#2c7fb8", "Mid": "#7a7a7a", "High": "#d95f0e"}
    for g in groups:
        sub = d[d.grp == g]
        # BUG FIX (external review, correct): a group with zero observed
        # events was skipped in the PLOT while remaining in the log-rank
        # test, so the figure and the reported p-value described different
        # comparisons. A zero-event group has a perfectly valid flat KM
        # curve at survival probability 1, and it is often the interesting
        # one. Every group in the test is now drawn.
        if sub.empty:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(sub.surv_time, sub.surv_event, label=f"{g} (n={len(sub)}, e={int(sub.surv_event.sum())})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=colors.get(g, "k"), lw=1.8)

    if len(groups) == 2:
        a, b = d[d.grp == groups[0]], d[d.grp == groups[1]]
        lr = logrank_test(a.surv_time, b.surv_time, a.surv_event, b.surv_event)
        p = lr.p_value
    else:
        lr = multivariate_logrank_test(d.surv_time, d.grp, d.surv_event)
        p = lr.p_value

    ax.set_title(f"{gene} - {endpoint_label(clin)}\nlog-rank p = {p:.3g}", fontsize=10)
    # BUG FIX (raised in external review): the axis was always labelled
    # "Months from surgery", which is wrong whenever the supplied endpoint has
    # a different time origin - progression-free survival from diagnosis or
    # from treatment start, for instance. The origin is now derived from the
    # endpoint metadata, and left unstated when it is not documented.
    origins = {"biochemical_recurrence": "Months from prostatectomy",
               "disease_free_survival": "Months from diagnosis",
               "progression_free_survival": "Months from treatment start",
               "simulated_recurrence": "Months (simulated)"}
    raw_ep = (str(clin.endpoint_name.dropna().iloc[0])
              if "endpoint_name" in clin.columns and clin.endpoint_name.notna().any()
              else "unavailable")
    ax.set_xlabel(origins.get(raw_ep, "Months from the endpoint's time origin "
                                      "(origin not documented)"))
    ax.set_ylabel(f"{endpoint_label(clin)} probability")
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="lower left", frameon=False)
    sns.despine(ax=ax)
    # SELF-REVIEW: the fitted row carried no `status`, while a NOT FITTED
    # row carries one - so the exported table had a status column populated
    # only for the skipped analyses, and a reader (or a consumer applying
    # `df.status == "NOT FITTED"`) had to infer that a blank meant success.
    # Every row now states its own outcome.
    return {"gene": gene, "status": "FITTED", "logrank_p": float(p),
            "n": len(d), "events": int(d.surv_event.sum()),
            "threshold_rule": split, "reason": ""}


# ----------------------------------------------------------------------------
# 4.3 Multivariable Cox, with EPV gating and PH diagnostics
# ----------------------------------------------------------------------------
def cox_multivariable(expr_log, clin, gene, extra_covars=("gleason", "psa", "age"),
                      strata=None, label=""):
    """Fit Cox PH for one candidate adjusted for standard clinicopathology.

    Returns None - and says why - when EPV is insufficient. Reporting an
    unstable HR from an over-fitted Cox model is worse than reporting nothing,
    because the confidence interval looks authoritative.
    """
    d = clin.dropna(subset=["surv_time", "surv_event"]).copy()
    d = d[survival_eligibility_mask(d)]
    if gene in expr_log.index:
        d["expr_z"] = d["sample"].map(expr_log.loc[gene])
        d["expr_z"] = (d.expr_z - d.expr_z.mean()) / d.expr_z.std(ddof=0)
    else:
        return None

    covars = ["expr_z"] + [c for c in extra_covars if c in d.columns]
    keep = ["surv_time", "surv_event"] + covars + ([strata] if strata else [])
    d = d[keep].dropna()
    for c in covars:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna()

    n_events = int(d.surv_event.sum())
    n_params = len(covars)
    epv = n_events / max(n_params, 1)
    if epv < CFG.min_events_per_covariate:
        # The EPV rule is a CONSERVATIVE SCREENING HEURISTIC, not a
        # mathematical guarantee. What actually matters is adequate events,
        # achievable precision, controlled model complexity and explicit
        # assumptions. EPV is used here as a transparent, pre-declared gate
        # so that the decision not to fit is auditable rather than ad hoc.
        msg = (f"    NOT FITTED - Cox for {gene}{(' [' + label + ']') if label else ''}: "
               f"{n_events} events / {n_params} covariates = EPV {epv:.1f} "
               f"< {CFG.min_events_per_covariate} (pre-declared screening "
               f"heuristic). Estimates would carry unquantified small-sample "
               f"bias and uninformative precision.")
        print(msg)
        return {"gene": gene, "label": label, "status": "skipped_insufficient_EPV",
                "events": n_events, "n": len(d), "epv": round(epv, 2)}

    cph = CoxPHFitter(penalizer=0.1)   # mild ridge: stabilises small-sample fits
    try:
        cph.fit(d, duration_col="surv_time", event_col="surv_event",
                strata=[strata] if strata else None)
    except (ConvergenceError, ValueError) as ex:
        print(f"    Cox failed for {gene}: {ex}")
        return {"gene": gene, "label": label, "status": f"failed: {type(ex).__name__}",
                "events": n_events, "n": len(d)}

    s = cph.summary.loc["expr_z"]
    # Proportional-hazards assumption. If violated, the single HR is a
    # time-averaged artefact and must be reported as such.
    # BUG FIX (both external reviews, correct - a confirmed REPORTING
    # defect): ph_ok was initialised to True and the except branch was a bare
    # `pass`, so a diagnostic that CRASHED was reported as
    # "ph_assumption_met": true. An investigator reading that column could
    # not distinguish a test that passed from a test that never ran, and the
    # single hazard ratio for a violated model is a time-averaged artefact.
    #
    # Three states now, and the failure reason is recorded rather than
    # swallowed. The wording also stops short of claiming the assumption
    # HOLDS: a non-significant test is an absence of detected violation, not
    # proof of proportionality, so graphical and clinical assessment remain
    # necessary.
    ph_status, ph_p, ph_error = "NOT ASSESSED", np.nan, ""
    try:
        from lifelines.statistics import proportional_hazard_test
        pht = proportional_hazard_test(cph, d, time_transform="rank")
        ph_p = float(pht.summary.loc["expr_z", "p"])
        ph_status = ("NO VIOLATION DETECTED" if ph_p > 0.05
                     else "POSSIBLE VIOLATION")
    except Exception as exc:
        ph_error = f"{type(exc).__name__}: {exc}"
        ph_status = "NOT ASSESSED"
        print(f"    PH diagnostic unavailable for {gene}: {ph_error}")
    # Retained for downstream consumers, but NEVER True on a failed test:
    # None means not assessed, so a truthiness test cannot read it as a pass.
    ph_ok = (None if ph_status == "NOT ASSESSED"
             else ph_status == "NO VIOLATION DETECTED")

    return {
        "gene": gene, "label": label, "status": "fitted",
        "n": len(d), "events": n_events, "epv": round(epv, 2),
        "HR_per_SD": float(np.exp(s["coef"])),
        "ci_lo": float(np.exp(s["coef lower 95%"])),
        "ci_hi": float(np.exp(s["coef upper 95%"])),
        "p": float(s["p"]),
        "concordance": float(cph.concordance_index_),
        "covariates": ",".join(covars),
        "ph_assumption_p": ph_p,
        "ph_assumption_status": ph_status,
        "ph_assumption_met": ph_ok,
        "ph_diagnostic_error": ph_error,
    }


def run_survival_suite(expr_log, clin, candidate_genes, max_genes=8):
    """KM + Cox for the top candidates, overall and by ancestry, with FDR
    correction across the candidates tested."""
    print("[4.3] Survival models")
    # The analysis unit is the PATIENT. The clinical join preserves multiple
    # specimens per patient on purpose, so the reduction happens here, once,
    # before any model sees the table - rather than each model treating rows
    # as independent observations.
    clin, _n_pseudo = enforce_one_sample_per_patient(clin, "survival models")
    genes = [g for g in candidate_genes if g in expr_log.index][:max_genes]
    km_rows, cox_rows = [], []

    n = len(genes)
    if n:
        ncol = min(4, n)
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.0 * nrow),
                                 squeeze=False)
        for i, g in enumerate(genes):
            ax = axes[i // ncol][i % ncol]
            r = km_by_expression(expr_log, clin, g, ax=ax)
            # BUG FIX (external review, correct): a NOT FITTED record is a
            # dict and therefore truthy, so it was appended beside genuinely
            # fitted results and would have been read as a survival
            # comparison. It is recorded - a skipped analysis must be
            # visible, not absent - but marked, left unplotted, and excluded
            # from the FDR correction, which is computed across tests that
            # actually produced a p-value.
            if r and r.get("status") == "NOT FITTED":
                print(f"    KM NOT FITTED for {g}: {r.get('reason', '')}")
                ax.axis("off")
                ax.set_title(f"{g}: not fitted", fontsize=8)
                km_rows.append(r)
            elif r:
                km_rows.append(r)
            else:
                ax.axis("off")
        for j in range(n, nrow * ncol):
            axes[j // ncol][j % ncol].axis("off")
        fig.suptitle(f"Kaplan-Meier: {endpoint_label(clin)} by candidate expression "
                     f"({CFG.expr_split} split, threshold pre-declared)", y=1.0)
        fig.tight_layout()
        save_figure(fig, "fig6_kaplan_meier_panel.png", dpi=200,
                    # BUG FIX (external review, correct): the figure TITLE
                    # read the configured split while the saved CAPTION
                    # hard-coded "median split", so a run configured with a
                    # tertile or quantile split shipped a caption describing
                    # an analysis that was not performed. The caption is the
                    # text that travels with the figure into a manuscript.
                    caption=f"{endpoint_label(clin)} by candidate expression "
                            f"(pre-declared {CFG.expr_split} split)")

    # Overall multivariable Cox
    for g in genes:
        r = cox_multivariable(expr_log, clin, g, label="all_patients")
        if r:
            cox_rows.append(r)

    # Ancestry-stratified: single shared baseline hazard with ancestry as a
    # stratum. This is preferred over fitting one model per ancestry group,
    # which the event counts almost never support.
    # BUG FIX (both external reviews, correct): this ran an
    # "ancestry_stratified" model whenever the ancestry column held more
    # than one category - without consulting ancestry_source. When no
    # genetic-ancestry annotation is supplied that column is populated from
    # SELF-REPORTED RACE, so the pipeline could publish a result labelled
    # ancestry-stratified that was stratified by race. The two have
    # different estimands and different interpretations; relabelling one as
    # the other is exactly the provenance failure the endpoint safeguards
    # exist to prevent.
    #
    # Genetic-ancestry inference now requires a genetic source for every
    # analysed record. A race-based breakdown is still fitted when present,
    # but labelled for what it is so no downstream table can call it
    # ancestry.
    _anc_src = (set(clin.get("ancestry_source", pd.Series(dtype=object))
                    .dropna().unique()))
    # BUG FIX (both external reviews, confirmed in table8_cox_models.csv):
    # my check last round treated only the exact string "genetic" as genetic
    # ancestry, so the DEMO generator's "genetic_simulated" was labelled
    # self_reported_race_stratified_EXPLORATORY - describing a synthetic
    # ancestry variable as self-reported race. Wrong in both directions: it
    # misnames the DEMO variable, and it would let a reader believe real
    # race data had been modelled. Three distinct sources, three labels,
    # never merged.
    _ANCESTRY_LABELS = {
        "genetic": "ancestry_stratified",
        "genetic_simulated": "simulated_genetic_ancestry_DEMO",
        "self_reported_race": "self_reported_race_stratified_EXPLORATORY",
    }
    _anc_is_genetic = _anc_src == {"genetic"}
    if len(_anc_src) == 1:
        _only = next(iter(_anc_src))
        _anc_label = _ANCESTRY_LABELS.get(
            _only, f"{_only}_stratified_UNRECOGNISED_SOURCE")
    elif len(_anc_src) > 1:
        _anc_label = ("mixed_ancestry_sources_"
                      + "_".join(sorted(_anc_src)) + "_EXPLORATORY")
    else:
        _anc_label = "ancestry_source_unrecorded_EXPLORATORY"
    if _anc_src and not _anc_is_genetic:
        print(f"    stratified model labelled "
              f"{_anc_label!r}: ancestry_source={sorted(_anc_src)}, not "
              f"genetic. This is NOT a genetic-ancestry analysis and must "
              f"not be reported as one.")
    if "ancestry" in clin.columns and clin.ancestry.nunique() > 1:
        for g in genes:
            r = cox_multivariable(expr_log, clin, g, strata="ancestry",
                                  label=_anc_label)
            if r:
                cox_rows.append(r)
        # Per-ancestry fits are attempted but will usually be refused by EPV.
        for anc, d in clin.groupby("ancestry"):
            if d.dropna(subset=["surv_event"]).surv_event.sum() < 10:
                print(f"    ancestry {anc}: too few events for a subgroup model "
                      f"(events={int(d.surv_event.fillna(0).sum())}); not fitted.")
                continue
            for g in genes[:3]:
                r = cox_multivariable(expr_log, d, g, extra_covars=("gleason",),
                                      label=f"ancestry={anc}")
                if r:
                    cox_rows.append(r)

    km_df = pd.DataFrame(km_rows)
    cox_df = pd.DataFrame(cox_rows)
    # FDR across candidates within each analysis label, because several genes
    # are tested against the same endpoint.
    # NOT FITTED rows carry no p-value, so they are excluded from the FDR
    # denominator: correcting across tests that were never run would both
    # understate the correction and imply comparisons that do not exist.
    for df, pcol in [(km_df, "logrank_p"), (cox_df, "p")]:
        if len(df) and "status" in df.columns:
            df.loc[df.status == "NOT FITTED", pcol] = np.nan
        if len(df) and pcol in df.columns:
            grp = "label" if "label" in df.columns else None
            df["qval_bh"] = np.nan
            it = df.groupby(grp) if grp else [("all", df)]
            for _, sub in it:
                m = sub[pcol].notna()
                if m.sum() > 1:
                    df.loc[sub.index[m], "qval_bh"] = multipletests(
                        sub.loc[m, pcol].values, method="fdr_bh")[1]
    if len(km_df):
        save_table(km_df, "table7_kaplan_meier.csv",
                   caption="Log-rank results per candidate")
    if len(cox_df):
        save_table(cox_df, "table8_cox_models.csv",
                   caption="Cox models fitted, and those refused for low EPV")
        fitted = cox_df[cox_df.status == "fitted"]
        print(f"    Cox models fitted: {len(fitted)} | refused for low EPV: "
              f"{int((cox_df.status == 'skipped_insufficient_EPV').sum())}")
        if len(fitted):
            print(fitted[["gene", "label", "HR_per_SD", "ci_lo", "ci_hi", "p",
                          "qval_bh", "ph_assumption_status"]].to_string(index=False))
    return km_df, cox_df

---
## Section 5 — Figures

Fixed-order categorical palette (blue, orange, aqua), a single-hue blue ramp for magnitude, and a blue–red diverging pair with a grey midpoint for signed fold-changes. Every figure with two or more series carries both a legend and direct labels, so identity never rests on colour alone — which matters for colour-vision-deficient readers and for print.

In [7]:
# =============================================================================
# SECTION 5 - FIGURES AND TABLES
# =============================================================================
# Palette: fixed-order categorical slots (blue, orange, aqua), a single-hue
# blue sequential ramp for magnitude, and a blue<->red diverging pair with a
# grey midpoint for signed fold-changes. Categorical hues are assigned in fixed
# order and never cycled; every figure with two or more series carries both a
# legend and direct labels, so identity never rests on colour alone.
# =============================================================================

C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"     # categorical slots 1-3
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#8a8a85"
CRITICAL = "#d03b3b"
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

from matplotlib.colors import LinearSegmentedColormap
CMAP_SEQ = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE)
CMAP_DIV = LinearSegmentedColormap.from_list(
    "div_blue_red", ["#184f95", "#6da7ec", "#f0efec", "#e88a8a", "#d03b3b"])

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.edgecolor": MUTED, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "legend.frameon": False,
    "xtick.color": INK2, "ytick.color": INK2,
    "figure.facecolor": "white", "axes.facecolor": "white",
})


# _save delegates to save_figure so every figure is written AND displayed.
_CAPTIONS = {
    "fig1_filter_attrition.png": "Candidates remaining after each gatekeeping filter",
    "fig2_volcano.png": "Surfaceome differential expression, tumour vs adjacent normal",
    "fig3_safety_window.png": "Tumour expression against vital-organ exposure, relative to PSMA",
    "fig4_roc_pr_null.png": "Out-of-fold ROC, precision-recall, and the permutation null",
    "fig5_calibration.png": "Calibration of the ensemble's scores",
    "fig6_kaplan_meier_panel.png": "BCR-free survival by candidate expression",
    "fig7_cox_forest.png": "Multivariable Cox hazard ratios per SD of expression",
    "fig8_candidate_heatmap.png": "Multi-omic feature profile of the top novel candidates",
    "fig9_feature_importance.png": "Permutation importance and resampled selection frequency",
    "fig10_leakage_ablation.png": "Does the model learn biology or research attention?",
    "fig11_baseline_comparison.png": "Ensemble against four transparent baselines",
}


def _save(fig, name):
    return save_figure(fig, name, caption=_CAPTIONS.get(name))


# ----------------------------------------------------------------------------
# Figure 1 - filter attrition
# ----------------------------------------------------------------------------
def fig_attrition(attrition):
    fig, ax = plt.subplots(figsize=(8.4, 3.8))
    d = attrition.copy()
    ypos = np.arange(len(d))[::-1]
    # Log x-scale: the funnel spans three orders of magnitude, and on a linear
    # axis every step after the first collapses into an unreadable sliver.
    left = 0.8
    ax.barh(ypos, np.maximum(d.n, 1), left=left, color=C1, height=0.6)
    for y, n, rm in zip(ypos, d.n, d.removed):
        txt = f"{n:,}" + (f"   (-{rm:,})" if rm > 0 else "")
        ax.text(max(n, 1) * 1.12, y, txt, va="center", fontsize=8.5, color=INK)
    ax.set_xscale("log")
    ax.set_xlim(left, max(d.n) * 4)
    ax.set_yticks(ypos)
    ax.set_yticklabels(d.step, fontsize=8)
    ax.set_xlabel("Candidate genes remaining (log scale)")
    ax.set_title("Sequential attrition through the four gatekeeping filters",
                 loc="left")
    ax.grid(axis="y", visible=False)
    sns.despine(ax=ax)
    _save(fig, "fig1_filter_attrition.png")


# ----------------------------------------------------------------------------
# Figure 2 - volcano
# ----------------------------------------------------------------------------
def fig_volcano(f, label_genes):
    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    sig = (f.rna_qval < CFG.fdr_alpha) & (f.rna_log2fc.abs() >= CFG.min_log2fc)
    ax.scatter(f.loc[~sig, "rna_log2fc"], f.loc[~sig, "neg_log10_rna_q"],
               s=7, c="#d8d7d2", linewidths=0, label="Not significant")
    ax.scatter(f.loc[sig, "rna_log2fc"], f.loc[sig, "neg_log10_rna_q"],
               s=11, c=C1, linewidths=0,
               label=f"q<{CFG.fdr_alpha}, |log2FC|>={CFG.min_log2fc}")
    hl = f[f.gene.isin(label_genes)]
    ax.scatter(hl.rna_log2fc, hl.neg_log10_rna_q, s=42, facecolors="none",
               edgecolors=C2, linewidths=1.5, label="Prioritised / benchmark")
    for _, r in hl.iterrows():
        ax.annotate(r.gene, (r.rna_log2fc, r.neg_log10_rna_q),
                    textcoords="offset points", xytext=(5, 4),
                    fontsize=7.5, color=INK)
    ax.axhline(-np.log10(CFG.fdr_alpha), color=MUTED, ls="--", lw=0.8)
    ax.axvline(CFG.min_log2fc, color=MUTED, ls="--", lw=0.8)
    ax.axvline(-CFG.min_log2fc, color=MUTED, ls="--", lw=0.8)
    ax.set_xlabel("log2 fold-change (PRAD tumour vs adjacent normal prostate)")
    ax.set_ylabel("-log10 BH-adjusted q-value")
    ax.set_title("Surfaceome differential expression")
    ax.legend(fontsize=7.5, loc="upper left")
    sns.despine(ax=ax)
    _save(fig, "fig2_volcano.png")


# ----------------------------------------------------------------------------
# Figure 3 - safety window vs PSMA benchmark
# ----------------------------------------------------------------------------
def fig_safety(f, label_genes):
    fig, ax = plt.subplots(figsize=(6.2, 5.0))
    d = f.dropna(subset=["safety_ratio_vs_psma", "tumour_mean_log2"])
    passed = d.passed_all_filters if "passed_all_filters" in d else pd.Series(False, index=d.index)
    ax.scatter(d.loc[~passed, "safety_ratio_vs_psma"].clip(lower=1e-3),
               d.loc[~passed, "tumour_mean_log2"],
               s=7, c="#d8d7d2", linewidths=0, label="Filtered out")
    ax.scatter(d.loc[passed, "safety_ratio_vs_psma"].clip(lower=1e-3),
               d.loc[passed, "tumour_mean_log2"],
               s=16, c=C3, linewidths=0, label="Passed all four filters")
    hl = d[d.gene.isin(label_genes)]
    ax.scatter(hl.safety_ratio_vs_psma.clip(lower=1e-3), hl.tumour_mean_log2,
               s=44, facecolors="none", edgecolors=C2, linewidths=1.5,
               label="Prioritised / benchmark")
    for _, r in hl.iterrows():
        ax.annotate(r.gene, (max(r.safety_ratio_vs_psma, 1e-3), r.tumour_mean_log2),
                    textcoords="offset points", xytext=(5, 4), fontsize=7.5, color=INK)
    ax.axvline(CFG.safety_ratio_max, color=CRITICAL, ls="--", lw=1.1)
    ax.annotate("PSMA safety ceiling", (CFG.safety_ratio_max, ax.get_ylim()[1]),
                xytext=(-4, -12), textcoords="offset points", ha="right",
                fontsize=7.5, color=CRITICAL)
    ax.set_xscale("log")
    ax.set_xlabel("Vital-organ exposure relative to PSMA (log scale)")
    ax.set_ylabel("Mean tumour expression, log2(TPM+1)")
    ax.set_title("Therapeutic window: tumour signal against vital-organ risk")
    ax.legend(fontsize=7.5, loc="lower left")
    sns.despine(ax=ax)
    _save(fig, "fig3_safety_window.png")


# ----------------------------------------------------------------------------
# Figure 4 - ROC and precision-recall with bootstrap CI
# ----------------------------------------------------------------------------
def _bootstrap_curve(y, s, kind="roc", n_boot=None, seed=CFG.seed):
    """Bootstrap a ROC or PR curve and its AUC/AP.

    BUG FIX (raised in external review): n_boot defaulted to 400 and no call
    site passed CFG.n_bootstrap, so every compute profile silently used 400
    resamples - the declared 200/1000/2000 settings had no effect at all.
    """
    n_boot = int(CFG.n_bootstrap if n_boot is None else n_boot)
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 1, 101)
    curves, aucs = [], []
    for _ in range(n_boot):
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i])) < 2:
            continue
        if kind == "roc":
            fpr, tpr, _ = roc_curve(y[i], s[i])
            curves.append(np.interp(grid, fpr, tpr))
            aucs.append(roc_auc_score(y[i], s[i]))
        else:
            pr, rc, _ = precision_recall_curve(y[i], s[i])
            order = np.argsort(rc)
            curves.append(np.interp(grid, rc[order], pr[order]))
            aucs.append(average_precision_score(y[i], s[i]))
    cur = np.vstack(curves)
    return grid, cur.mean(0), np.percentile(cur, 2.5, 0), np.percentile(cur, 97.5, 0), \
           np.mean(aucs), np.percentile(aucs, [2.5, 97.5])


def fig_roc_pr(score_df, y, is_heldout, nulls, matched=None):
    mask = ~is_heldout
    yy = y[mask]
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.2))

    # Panel A - ROC for the three base learners plus the ensemble
    ax = axes[0]
    for (name, col) in zip(["Ensemble", "ElasticNet", "RandomForest"], [C1, C2, C3]):
        if name not in score_df.columns:
            continue
        s = score_df[name].values[mask]
        ok = ~np.isnan(s)
        g, m, lo, hi, auc, roc_ci = _bootstrap_curve(
            yy[ok], s[ok], kind="roc", n_boot=CFG.n_bootstrap)
        if name == "Ensemble":
            ens_auc, ens_roc_ci = float(auc), [float(roc_ci[0]), float(roc_ci[1])]
        ax.plot(g, m, color=col, lw=2,
                label=f"{name}  AUC {auc:.3f} [{roc_ci[0]:.3f}-{roc_ci[1]:.3f}]")
        if name == "Ensemble":
            ax.fill_between(g, lo, hi, color=col, alpha=0.16, linewidth=0)
            ax.annotate("Ensemble", (g[60], m[60]), textcoords="offset points",
                        xytext=(6, -12), fontsize=8, color=col)
    ax.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=0.9)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"A  Out-of-fold ROC (bagged PU: {CFG.n_pu_bags} bags x "
                 f"{CFG.n_outer_folds} outer folds, {CFG.n_inner_folds} inner)")
    ax.legend(fontsize=7, loc="lower right")

    # Panel B - precision-recall, the honest view under heavy class imbalance
    ax = axes[1]
    s = score_df["Ensemble"].values[mask]
    ok = ~np.isnan(s)
    # BUG FIX (raised in external review, confirmed in the saved output):
    # this call previously reused the variable `ci`, overwriting the ROC
    # interval, and the return statement then exported the PRECISION-RECALL
    # interval under the name `auc_ci`. The published summary carried
    # AUC = 0.904 with "auc_ci" = 0.432-0.680 - an interval that does not
    # contain its own point estimate, because it belonged to AP = 0.560.
    # The two intervals are now held in separate variables and returned under
    # separate, explicit names.
    g, m, lo, hi, ap, pr_ci = _bootstrap_curve(
        yy[ok], s[ok], kind="pr", n_boot=CFG.n_bootstrap)
    base = yy[ok].mean()
    ax.plot(g, m, color=C1, lw=2,
            label=f"Ensemble  AP {ap:.3f} [{pr_ci[0]:.3f}-{pr_ci[1]:.3f}]")
    ax.fill_between(g, lo, hi, color=C1, alpha=0.16, linewidth=0)
    ax.axhline(base, color=MUTED, ls="--", lw=0.9,
               label=f"Prevalence baseline {base:.3f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("B  Precision-recall")
    ax.legend(fontsize=7, loc="upper right")

    # Panel C - the permutation null, so the AUC is read against the right zero
    ax = axes[2]
    ax.hist(nulls, bins=28, color="#d8d7d2", edgecolor="white", linewidth=0.5)
    # The null was produced by permutation_null_auc() under its own fixed
    # procedure. Comparing it to the out-of-fold AUC would compare two
    # different algorithms, so the MATCHED observed statistic from that same
    # procedure is what gets plotted and tested. The out-of-fold AUC is still
    # shown, as a separate marker, clearly labelled.
    obs_oof = roc_auc_score(yy[ok], s[ok])
    obs = float(matched["matched_observed"]) if matched else obs_oof
    ax.axvline(obs, color=C2, lw=2.2)
    ax.annotate(f"Observed {obs:.3f}", (obs, ax.get_ylim()[1] * 0.92),
                xytext=(-6, 0), textcoords="offset points", ha="right",
                fontsize=8.5, color=C2)
    if matched:
        ax.axvline(obs_oof, color=MUTED, ls=":", lw=1.4)
        ax.annotate(f"out-of-fold {obs_oof:.3f}\n(different procedure)",
                    (obs_oof, ax.get_ylim()[1] * 0.45), xytext=(-6, 0),
                    textcoords="offset points", ha="right", fontsize=7,
                    color=INK2)
    p95 = np.percentile(nulls, 95)
    ax.axvline(p95, color=CRITICAL, ls="--", lw=1.1)
    ax.annotate(f"Null 95th pct {p95:.3f}", (p95, ax.get_ylim()[1] * 0.55),
                xytext=(6, 0), textcoords="offset points", fontsize=7.5, color=CRITICAL)
    ax.set_xlabel("Cross-validated AUC under permuted labels")
    ax.set_ylabel("Permutations")
    _pstat = (matched.get("permutation_detail") if matched
              else permutation_p(nulls, obs))
    _p = _pstat["p"]
    _ptxt = (f"p < {_pstat['resolution']:.3g}"
             if _pstat.get("p_is_at_resolution_floor") else f"p = {_p:.3g}")
    ax.set_title(f"C  Permutation null (B={_pstat['n_resamples']}, {_ptxt})")
    for a in axes:
        sns.despine(ax=a)
    fig.tight_layout()
    _save(fig, "fig4_roc_pr_null.png")
    return {
        # Point estimates and their OWN intervals, named unambiguously.
        "ensemble_oof_auc": float(obs_oof),
        "roc_auc_ci": ens_roc_ci,
        "ensemble_ap": float(ap),
        "average_precision_ci": [float(pr_ci[0]), float(pr_ci[1])],
        "prevalence_baseline": float(base),
        "n_bootstrap": int(CFG.n_bootstrap),
        # Null comparison, against the MATCHED statistic from the same
        # procedure, with the (b+1)/(B+1) correction applied.
        "null_auc_mean": float(np.mean(nulls)),
        "null_auc_p95": float(p95),
        "matched_observed_auc": float(obs),
        "permutation_p": float(_p),
        "permutation_n_resamples": int(_pstat["n_resamples"]),
        "permutation_resolution": float(_pstat["resolution"]),
        "permutation_p_at_resolution_floor": bool(
            _pstat.get("p_is_at_resolution_floor", False)),
        "note": ("roc_auc_ci belongs to ensemble_oof_auc and "
                 "average_precision_ci to ensemble_ap; permutation_p compares "
                 "matched_observed_auc, not ensemble_oof_auc, because the null "
                 "uses a fixed procedure"),
    }


# ----------------------------------------------------------------------------
# Figure 5 - calibration
# ----------------------------------------------------------------------------
def fig_calibration(score_df, y, is_heldout):
    mask = ~is_heldout
    s = score_df["Ensemble"].values[mask]
    yy = y[mask]
    ok = ~np.isnan(s)
    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    pt, pp = calibration_curve(yy[ok], s[ok], n_bins=8, strategy="quantile")
    ax.plot(pp, pt, "o-", color=C1, lw=2, ms=7, label="Ensemble")
    ax.plot([0, 1], [0, 1], ls="--", color=MUTED, lw=0.9, label="Perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed fraction of known targets")
    brier = brier_score_loss(yy[ok], s[ok])
    # NEW (found in our own review of the exported run summary, and flagged
    # independently in external review): a raw Brier score is uninterpretable
    # at 2.7% prevalence, and reporting it alone made a CALIBRATION FAILURE
    # read as a strength. The reference point is the no-skill constant
    # predictor, which always predicts the base rate and scores p*(1-p).
    #
    # Last run: Brier 0.0458 against a no-skill 0.0259 - a Brier SKILL SCORE
    # of -0.77, i.e. worse than predicting the prevalence for every gene. The
    # cause is structural, not a bug: bagged pseudo-negative sampling trains
    # each learner on a roughly balanced problem (one positive to three
    # sampled pseudo-negatives, so a bag is ~25% positive), so the scores are
    # calibrated to that balance and not to the true positive rate.
    #
    # The illustrative figures in this comment come from an earlier run whose
    # positive count was taken as ~75; the labelled fraction across all 2,792
    # genes is 57/2792 = 0.020, so the over-prediction factor is larger than
    # the 5.1x once quoted here. Every number the figure DRAWS is computed
    # below from the data at runtime - prev, brier_noskill and brier_skill -
    # so only this commentary was stale. Treat the quoted values as
    # illustrative of the effect, never as the current measurement.
    #
    # The scores are therefore valid as RANKS and invalid as probabilities. A
    # score of 0.9 does not mean a 90% chance of being a viable target. Both
    # numbers are now reported, the skill score is drawn on the figure, and
    # the prior-corrected score is reported alongside.
    prev = float(np.mean(yy[ok]))
    brier_noskill = prev * (1.0 - prev)
    brier_skill = 1.0 - (brier / brier_noskill if brier_noskill > 0 else np.nan)
    mean_score = float(np.mean(s[ok]))
    # Scale-free prior correction: rescale scores so their mean matches the
    # observed prevalence, which is the minimal fix that makes the magnitude
    # comparable to a probability. Reported, never used for ranking (a
    # monotone rescaling cannot change the rank order, which is the point).
    s_adj = np.clip(s[ok] * (prev / max(mean_score, 1e-9)), 0, 1)
    brier_prior_corrected = brier_score_loss(yy[ok], s_adj)
    ax.axhline(prev, color=MUTED, lw=0.8, ls=":",
               label=f"Prevalence ({prev:.3f})")
    verdict = "WORSE than no-skill" if brier_skill < 0 else "better than no-skill"
    ax.set_title(f"Calibration: Brier {brier:.3f} vs no-skill "
                 f"{brier_noskill:.3f}\nskill score {brier_skill:+.2f} "
                 f"({verdict}) - scores are RANKS, not probabilities",
                 fontsize=8.5)
    ax.legend(fontsize=7.5)
    sns.despine(ax=ax)
    _save(fig, "fig5_calibration.png")
    if brier_skill < 0:
        print(f"    CALIBRATION: Brier {brier:.4f} vs no-skill "
              f"{brier_noskill:.4f} -> skill score {brier_skill:+.2f}. The "
              f"ensemble score is WORSE CALIBRATED than always predicting the "
              f"base rate (mean score {mean_score:.3f} vs prevalence "
              f"{prev:.3f}, {mean_score/max(prev,1e-9):.1f}x over-prediction). "
              f"This is expected under balanced pseudo-negative bagging. "
              f"Interpret the output as a RANKING only; do not report the "
              f"score as a probability of target actionability.")
    return {"brier_prevalence_scale": float(brier),
            "brier": float(brier),   # legacy alias
            "brier_noskill_prevalence_rule": float(brier_noskill),
            "brier_skill_score": float(brier_skill),
            "brier_prior_corrected": float(brier_prior_corrected),
            "mean_predicted_score": mean_score,
            "prevalence": prev,
            "over_prediction_factor": float(mean_score / max(prev, 1e-9)),
            "scores_are_calibrated_probabilities": bool(brier_skill >= 0),
            "interpretation": ("ranking only - PU bagging calibrates to the "
                               "balanced training problem, not to the true "
                               "positive rate")}


# ----------------------------------------------------------------------------
# Figure 7 - Cox forest plot
# ----------------------------------------------------------------------------
def fig_forest(cox_df):
    d = cox_df[cox_df.status == "fitted"].copy() if len(cox_df) else pd.DataFrame()
    if d.empty:
        print("    forest plot skipped: no estimable Cox models")
        return
    d["lab"] = d.gene + "  [" + d.label + "]"
    d = d.sort_values("HR_per_SD")
    fig, ax = plt.subplots(figsize=(6.6, 0.42 * len(d) + 1.9))
    y = np.arange(len(d))
    sigcol = np.where(d.qval_bh.fillna(1) < 0.05, C2, C1)
    ax.hlines(y, d.ci_lo, d.ci_hi, color=sigcol, lw=2.0)
    ax.scatter(d.HR_per_SD, y, s=42, color=sigcol, zorder=3)
    ax.axvline(1.0, color=MUTED, ls="--", lw=0.9)
    ax.set_yticks(y); ax.set_yticklabels(d.lab, fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("Hazard ratio per SD of expression (BCR-free survival)")
    ax.set_title("Multivariable Cox models\n(orange = BH q<0.05; adjusted for Gleason, PSA, age)")
    for yi, r in zip(y, d.itertuples()):
        ax.text(ax.get_xlim()[1], yi, f"  e={r.events}", va="center",
                fontsize=7, color=INK2)
    ax.grid(axis="y", visible=False)
    sns.despine(ax=ax)
    _save(fig, "fig7_cox_forest.png")


# ----------------------------------------------------------------------------
# Figure 8 - candidate profile heatmap
# ----------------------------------------------------------------------------
def fig_candidate_heatmap(ranked, top_n=25):
    cols = ["rna_log2fc", "cohen_d", "therapeutic_window_log2",
            "log_safety_ratio", "mcrpc_mean_log2", "primary_to_mcrpc_delta",
            "nepc_log2fc", "prot_log2fc", "log_ecd_len", "topology_score"]
    cols = [c for c in cols if c in ranked.columns]
    d = ranked.head(top_n).set_index("gene")[cols]
    # Drop columns that are entirely missing. NEPC features are NaN whenever
    # the NEPC subgroup was too small to compare, and an all-NaN column
    # renders as a blank stripe that a reader mistakes for a real value of
    # zero rather than for absent data.
    all_nan = [c for c in d.columns if d[c].isna().all()]
    if all_nan:
        print(f"    heatmap: dropping {len(all_nan)} all-missing column(s): "
              f"{', '.join(all_nan)}")
        d = d.drop(columns=all_nan)
        cols = [c for c in cols if c not in all_nan]
    if d.empty or not cols:
        print("    heatmap skipped: no populated feature columns")
        return
    # Z-score each feature so a diverging scale is interpretable across columns
    z = (d - d.mean()) / d.std(ddof=0).replace(0, np.nan)
    z = z.clip(-3, 3)
    fig, ax = plt.subplots(figsize=(0.62 * len(cols) + 3.4, 0.30 * len(d) + 2.2))
    im = ax.imshow(z.values, cmap=CMAP_DIV, vmin=-3, vmax=3, aspect="auto")
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels([c.replace("_", " ") for c in cols], rotation=45,
                       ha="right", fontsize=7.5)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=7.5)
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Feature z-score (within the top-ranked set)", fontsize=8)
    cb.outline.set_visible(False)
    ax.set_title(f"Top {len(d)} prioritised candidates - multi-omic feature profile")
    ax.grid(False)
    _save(fig, "fig8_candidate_heatmap.png")


# ----------------------------------------------------------------------------
# Figure 9 - feature importance and resampled selection frequency, side by side
# ----------------------------------------------------------------------------
def fig_importance(imp_agg, stability, top_n=14):
    fig, axes = plt.subplots(1, 2, figsize=(11.6, 0.34 * top_n + 2.4))
    d = imp_agg.head(top_n).iloc[::-1]
    ax = axes[0]
    y = np.arange(len(d))
    ax.barh(y, d.mean_importance_across_models, height=0.62, color=C1,
            xerr=d.sd_across_models, error_kw=dict(ecolor=MUTED, lw=0.9, capsize=2))
    ax.set_yticks(y)
    ax.set_yticklabels([s.replace("_", " ") for s in d.feature], fontsize=8)
    ax.set_xlabel("Mean AUC decrease on permutation")
    ax.set_title("A  Permutation importance (averaged over learners)")
    ax.grid(axis="y", visible=False)

    ax = axes[1]
    if not stability.empty:
        s = stability.head(top_n).iloc[::-1]
        y = np.arange(len(s))
        colors = np.where(s.mean_coef >= 0, C1, C2)
        ax.barh(y, s.selection_frequency, height=0.62, color=colors)
        ax.axvline(0.6, color=CRITICAL, ls="--", lw=1.0)
        ax.annotate("stability threshold", (0.6, len(s) - 0.6),
                    xytext=(5, 0), textcoords="offset points",
                    fontsize=7.5, color=CRITICAL)
        ax.set_yticks(y)
        ax.set_yticklabels([t.replace("_", " ") for t in s.feature], fontsize=8)
        ax.set_xlim(0, 1.0)
        ax.set_xlabel("Fraction of elastic-net fits retaining the feature")
        ax.set_title("B  Resampled selection frequency (blue = positive coef, orange = negative)")
        ax.grid(axis="y", visible=False)
    for a in axes:
        sns.despine(ax=a)
    fig.tight_layout()
    _save(fig, "fig9_feature_importance.png")


# ----------------------------------------------------------------------------
# Figure 10 - leakage ablation
# ----------------------------------------------------------------------------
def fig_ablation(perf_full, perf_bio):
    a = perf_full[perf_full.metric == "auc"].set_index("model")
    b = perf_bio[perf_bio.metric == "auc"].set_index("model")
    models = [m for m in a.index if m in b.index]
    if not models:
        return
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    x = np.arange(len(models)); w = 0.36
    lo, hi = "fold_spread_lo", "fold_spread_hi"
    ax.bar(x - w/2, a.loc[models, "mean"], w, color=C1,
           yerr=[a.loc[models, "mean"] - a.loc[models, lo],
                 a.loc[models, hi] - a.loc[models, "mean"]],
           error_kw=dict(ecolor=MUTED, lw=0.9, capsize=3),
           label="Biology + research-intensity features")
    ax.bar(x + w/2, b.loc[models, "mean"], w, color=C2,
           yerr=[b.loc[models, "mean"] - b.loc[models, lo],
                 b.loc[models, hi] - b.loc[models, "mean"]],
           error_kw=dict(ecolor=MUTED, lw=0.9, capsize=3),
           label="Biology features only")
    for xi, m in zip(x, models):
        ax.text(xi - w/2, a.loc[m, "mean"] + 0.012, f"{a.loc[m,'mean']:.3f}",
                ha="center", fontsize=7.5, color=INK)
        ax.text(xi + w/2, b.loc[m, "mean"] + 0.012, f"{b.loc[m,'mean']:.3f}",
                ha="center", fontsize=7.5, color=INK)
    ax.axhline(0.5, color=MUTED, ls="--", lw=0.9)
    ax.set_xticks(x); ax.set_xticklabels(models, fontsize=8.5)
    ax.set_ylabel("Cross-validated AUC\n(mean; bars show fold-level spread, not a CI)")
    ax.set_title("Leakage ablation: does the model learn biology or research attention?")
    ax.set_ylim(0.4, 1.0)
    ax.legend(fontsize=7.5, loc="lower right")
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)
    _save(fig, "fig10_leakage_ablation.png")


# ----------------------------------------------------------------------------
# Figure 11 - does the ensemble beat transparent baselines?
# ----------------------------------------------------------------------------
# PROPOSAL REFINEMENT: this figure exists to answer, visually and in one
# glance, the question a methods reviewer will ask first - what does the
# complex model add over sorting by fold-change?
# ----------------------------------------------------------------------------
def fig_baselines(baselines):
    if baselines is None or baselines.empty:
        return
    d = baselines.copy().iloc[::-1]
    is_ens = d.method.str.startswith("PU ensemble")
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 0.5 * len(d) + 2.6))

    ax = axes[0]
    y = np.arange(len(d))
    # BUG FIX (external review, correct): this panel is titled "Primary
    # endpoint" but plotted median_benchmark_percentile - the ALL-GENE
    # metric - after the primary comparison moved to the common population.
    # So the figure showed B1 94.5 / ensemble 91.5 while run_summary.json
    # reported the primary endpoint as B1 95.14 / ensemble 93.14. A figure
    # that contradicts the summary is the kind of discrepancy a reader has
    # no way to detect. The panel now plots whichever column the verdict
    # uses, and its label states the population.
    _pcol_fig = ("median_benchmark_percentile_common_population"
                 if "median_benchmark_percentile_common_population"
                 in d.columns else "median_benchmark_percentile")
    _pop_label = ("training positives excluded"
                  if _pcol_fig.endswith("common_population")
                  else "all genes")
    ax.barh(y, d[_pcol_fig], height=0.6,
            color=np.where(is_ens, C1, "#c9c8c3"))
    for yi, v, ens in zip(y, d[_pcol_fig], is_ens):
        ax.text(v + 0.6, yi, f"{v:.1f}", va="center", fontsize=8,
                color=C1 if ens else INK2,
                fontweight="bold" if ens else "normal")
    ax.set_yticks(y)
    ax.set_yticklabels([m if len(m) < 44 else m[:41] + "..." for m in d.method],
                       fontsize=8)
    ax.set_xlabel(f"Median percentile of held-out benchmarks "
                  f"({_pop_label})")
    ax.set_title("A  Primary endpoint: benchmark recovery\n"
                 "(blue = this study's ensemble, grey = transparent baselines)",
                 loc="left")
    ax.set_xlim(0, 104)
    ax.grid(axis="y", visible=False)

    ax = axes[1]
    n_b = int(d.n_benchmarks.iloc[0]) if "n_benchmarks" in d else 6
    # Colour follows the entity, not the panel: the ensemble stays blue in
    # both panels so a reader tracks one series across the figure.
    # SELF-REVIEW: panel A was moved to the primary (common) population
    # last round and this panel was left on the all-gene count - the two
    # halves of one figure describing two populations. Both panels now read
    # the same column family.
    _tdcol_fig = ("benchmarks_in_top_decile_common_population"
                  if "benchmarks_in_top_decile_common_population" in d.columns
                  else "benchmarks_in_top_decile")
    ax.barh(y, d[_tdcol_fig], height=0.6,
            color=np.where(is_ens, C1, "#c9c8c3"))
    for yi, v in zip(y, d[_tdcol_fig]):
        ax.text(v + 0.06, yi, f"{int(v)}/{n_b}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y); ax.set_yticklabels([])
    ax.set_xlabel(f"Benchmarks recovered in the top decile (of {n_b} eligible)")
    ax.set_title("B  Top-decile recovery", loc="left")
    ax.set_xlim(0, n_b + 0.8)
    ax.grid(axis="y", visible=False)

    for a in axes:
        sns.despine(ax=a)
    fig.tight_layout()
    _save(fig, "fig11_baseline_comparison.png")

---
## Section 6 — Run the pipeline

Two ranking tables are produced and the distinction between them is the whole point:

- `table2_full_candidate_ranking.csv` — every surfaceome gene, scored.
- `table2b_novel_candidate_shortlist.csv` — **the discovery output**: genes that are neither already clinical-stage targets nor held-out benchmarks.

A ranked list with ERBB2, mesothelin and nectin-4 in the top rows is not a discovery; it is the model reciting its training labels. Positives scoring highly is a fit check, reported as such. Only the novel list belongs in a manuscript's headline result.

In [8]:
import base64 as _b64_selfcheck
PIPELINE_SOURCE_FOR_SELFCHECK = _b64_selfcheck.b64decode(
    "...").decode()


# =============================================================================
# SECTION 6 - MAIN PIPELINE
# =============================================================================

def rank_candidates(d, y, is_heldout):
    """Final prioritisation tables.

    TWO TABLES ARE PRODUCED, AND THE DISTINCTION IS THE WHOLE POINT:

      table2_full_candidate_ranking.csv  - every surfaceome gene, scored.
      table2b_novel_candidate_shortlist.csv - the DISCOVERY output: genes that
          are NOT already clinical-stage targets and NOT held-out benchmarks.

    A ranked list in which ERBB2, MSLN and NECTIN4 occupy the top rows is not a
    discovery; it is the model reciting its own training labels. Positives
    scoring highly is a sanity check, reported as such. Only the novel list can
    be described as a prioritisation of new targets, and only it should appear
    in a manuscript's headline result.
    """
    r = d.copy()
    r["is_training_positive"] = y == 1
    r["is_heldout_benchmark"] = is_heldout
    # Engineered controls are never candidates. They are retained in the
    # full ranking table for audit (so you can confirm where they landed) but
    # excluded from novelty, every shortlist, and survival analysis.
    r["is_synthetic_control"] = is_synthetic_gene(r.gene)
    # RENAMED (external review, correct): this flag means "carries no label in
    # THIS pipeline" - it does not establish that a gene is an undiscovered
    # therapeutic target, nor that no published pharmacological or biological
    # evidence exists for it. Calling it `is_novel` overstated the claim in
    # the shortlist headers, which are the most-read output in the archive.
    # Novelty is a literature and development-pipeline determination made by
    # the investigator, not a property this code can compute.
    # CRITICAL BUG FIX (external review, confirmed in the exported
    # shortlists): `is_unlabeled_candidate` excluded training positives,
    # benchmarks and synthetic controls - but NOT the clinical-stage targets
    # withheld from training because they sit outside the primary payload
    # class. Those genes are unlabeled IN THIS PIPELINE yet are known,
    # actively developed drug targets, and they were appearing in outputs
    # captioned as discovery shortlists: IGF1R, KDR, LGALS9 and SIRPA in the
    # primary list, CD44, CD74 and NTRK1 in the advanced-disease lists.
    # Presenting a known clinical-stage target as a candidate discovery is a
    # scientific reporting error, not a naming preference.
    #
    # They are RETAINED in the full ranking as known off-class comparators -
    # discarding them would hide useful context - and excluded from every
    # output called a shortlist.
    r["is_known_clinical_target"] = r.gene.isin(
        [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]
    ) | r.is_heldout_benchmark | r.gene.isin(PROSTATE_RESTRICTION_ONLY_MARKERS)
    r["label_class"] = r.gene.map(label_class_of)
    r["is_unlabeled_candidate"] = (~r.is_training_positive
                                   & ~r.is_heldout_benchmark
                                   & ~r.is_synthetic_control
                                   & ~r.is_known_clinical_target)
    n_offclass_excluded = int((r.is_known_clinical_target
                               & ~r.is_training_positive
                               & ~r.is_heldout_benchmark).sum())
    if n_offclass_excluded:
        top = r.loc[r.is_known_clinical_target & ~r.is_training_positive
                    & ~r.is_heldout_benchmark & (r.percentile >= 90),
                    "gene"].tolist()
        print(f"    {n_offclass_excluded} known clinical-stage target(s) "
              f"outside the training class are retained in the full ranking "
              f"as KNOWN COMPARATORS but excluded from every discovery "
              f"shortlist"
              + (f"; in the top decile: {sorted(top)[:8]}" if top else ""))
    # Back-compatible alias so downstream code and saved notebooks keep
    # working; the exported column name is the honest one.
    r["is_novel"] = r["is_unlabeled_candidate"]
    n_ctrl = int(r.is_synthetic_control.sum())
    if n_ctrl:
        top_ctrl = r.loc[r.is_synthetic_control & (r.percentile >= 90), "gene"].tolist()
        print(f"    excluded {n_ctrl} engineered control(s) from candidacy"
              + (f"; {len(top_ctrl)} had reached the top decile and would "
                 f"otherwise have been reported as candidates: "
                 f"{', '.join(top_ctrl)}" if top_ctrl else ""))
    # PROPOSAL REFINEMENT (Step 6): evidence tiers restated so that the tier
    # NAMES carry the evidence grade. A reader should never have to look up
    # what Tier 2 means to know a candidate lacks protein evidence.
    bio = r.passed_biological_screen
    prot_detected = r.protein_detected.fillna(False)
    prot_corrob = r.protein_corroborated.fillna(False)
    reagent = r.F4_reagent.fillna(False)
    top = r.percentile >= 90

    # ------------------------------------------------------------------
    # EVIDENCE TIERS
    # ------------------------------------------------------------------
    # BUG FIX (raised in external review): Tier 2 previously read "protein
    # evidence unavailable", which conflated two different situations -
    # the protein was never measured, versus it WAS measured and did not
    # corroborate. The first is a coverage artefact (mass spectrometry
    # under-detects membrane proteins); the second is evidence against the
    # candidate. Collapsing them hides a real negative result.

    # ==================================================================
    # PER-CONTEXT EVIDENCE TIERS
    # ==================================================================
    # BUG FIX (critical, raised in external review and confirmed in the
    # exported CSVs): the single `tier` column was driven by
    # `passed_biological_screen`, which is PRIMARY-tumour differential
    # expression, while the shortlists were built from context-specific
    # eligibility. The output therefore contradicted itself - 32 of 44 mCRPC
    # and 23 of 30 NEPC shortlisted candidates carried the label
    # "Not prioritised", which would lead a reader to discard exactly the
    # advanced-disease targets the study exists to find.
    #
    # Each context now gets its OWN tier, computed from that context's
    # eligibility. A gene is never described as unprioritised in a list it
    # qualified for.
    def _context_tier(elig_col):
        e = r.get(elig_col)
        if e is None:
            return pd.Series("not assessable", index=r.index)
        e = e.fillna(False).astype(bool) if e.dtype != bool else e
        return pd.Series(np.select(
            [e & prot_corrob & reagent & top,
             e & prot_corrob & top,
             e & ~prot_detected & reagent & top,
             e & ~prot_detected & top,
             e & prot_detected & ~prot_corrob & top,
             e],
            ["T1 - eligible + protein corroborated + binder available",
             "T1b - eligible + protein corroborated, binder to develop",
             "T2 - eligible + binder available, protein NOT MEASURED",
             "T2b - eligible, protein NOT MEASURED, binder to develop",
             "T3 - eligible, protein measured but NOT corroborating",
             "T4 - eligible, outside top decile"],
            default="not eligible in this context"), index=r.index)

    r["priority_tier_primary"] = _context_tier("eligible_primary")
    r["priority_tier_mcrpc"] = _context_tier("eligible_mcrpc")
    r["priority_tier_nepc"] = _context_tier("eligible_nepc")
    # Retained for backward compatibility, but explicitly scoped so it can no
    # longer be mistaken for a cross-context judgement.
    r["tier"] = r["priority_tier_primary"]
    r["tier_scope"] = "primary adenocarcinoma only - see priority_tier_* per context"

    # Experimental readiness is reported SEPARATELY from discovery priority,
    # so a reader can see both without one silently gating the other.
    # BUG FIX (external review, confirmed in the exported shortlists): the
    # "binder development required" branch was gated on `bio`, the PRIMARY
    # adenocarcinoma biological screen. A gene eligible in mCRPC or NEPC but
    # not in primary disease therefore received "not applicable" even when it
    # was shortlisted and had no binder - 8 mCRPC, 5 NEPC and 5 PSMA-low
    # candidates in the last run. Experimental readiness is a property of
    # reagent availability and of eligibility in ANY context, not of primary
    # disease specifically; the two are combined downstream, not conflated
    # here.
    any_eligible = (
        r.get("eligible_primary", pd.Series(False, index=r.index)).fillna(False)
        | r.get("eligible_mcrpc", pd.Series(False, index=r.index)).fillna(False)
        | r.get("eligible_nepc", pd.Series(False, index=r.index)).fillna(False)
    ).astype(bool)
    r["readiness"] = np.where(
        reagent, "binder available",
        np.where(any_eligible | bio, "binder development required",
                 "not applicable"))
    r["readiness_scope"] = np.where(
        reagent, "reagent recorded",
        np.where(any_eligible, "eligible in >=1 disease context",
                 "not eligible in any assessed context"))

    cols = ["gene", "priority_tier_primary", "priority_tier_mcrpc",
            "priority_tier_nepc", "tier_scope", "readiness", "disease_context",
            "surface_evidence_grade", "surface_accessibility_established",
            # Per-modality three-state outcome, assessability and comparator
            # values, so a downstream analyst can reconstruct every modality
            # decision from the candidate table alone (external review).
            "modality_profile", "n_modalities_suitable",
            "n_modalities_passing_screen", "n_modalities_failing_screen",
            "n_modalities_assessable", "n_modalities_not_assessable",
            "modalities_not_assessable",
            "screen_outcome_ADC", "screen_outcome_CAR_T",
            "screen_outcome_T_cell_engager", "screen_outcome_radioligand",
            "screen_assessable_ADC", "screen_assessable_CAR_T",
            "screen_assessable_T_cell_engager", "screen_assessable_radioligand",
            "ratio_ADC", "ratio_CAR_T", "ratio_T_cell_engager",
            "ratio_radioligand",
            # BUG FIX (both external reviews, correct): F2_outcome,
            # F2_safety_definitive and passed_biological_screen_definitive
            # existed only in the feature table, so a reader of the candidate
            # ranking or any shortlist could not see which eligibility
            # decisions were definitive and which were exploratory - the
            # single most important distinction in the output.
            "F2_outcome", "F2_safety_definitive",
            "passed_biological_screen_definitive",
            # BUG FIX (both external reviews, correct) - and the SAME
            # omission as the block above, one round later: eligibility_state
            # was added to the feature table last round and not to this
            # list, so the new three-state status was invisible in the
            # ranking and every shortlist. Adding the string fixes today's
            # instance; the regression test below fixes the class, by
            # failing whenever a column named eligib* exists in the feature
            # table and not here.
            "f2_assessable", "eligibility_basis", "eligibility_state",
            "eligibility_state_authority",
            "f2_comparator_raw_tpm", "f2_comparator_censored_tpm",
            "f2_vital_organ_coverage", "f2_organ_coverage_adequate",
            "f2_comparator_measurable",
            "psma_low_status", "suitable_for_psma_low",
            "plasma_membrane_confidence", "is_unlabeled_candidate",
            "is_known_clinical_target", "label_class",
            # is_novel retained as a deprecated alias so downstream code and
            # previously saved notebooks keep working; the honest name is
            # is_unlabeled_candidate and is listed first.
            "is_novel", "is_training_positive",
            "is_heldout_benchmark", "is_synthetic_control",
            "passed_biological_screen",
            # Auditability: the intracellular exclusion and its scope were
            # previously applied but never exported, so a reader could not
            # check which genes were removed or why.
            "intracellular_excluded_flag", "excluded_intracellular",
            "F3_route", "surface_detected_experimentally",
            "epithelial_restriction_score", "likely_infiltrate_antigen",
            "readiness_scope",
            "retained_in_ar_low", "ar_low_vs_high_delta",
            "eligible_primary", "eligible_mcrpc", "eligible_nepc",
            "passed_strict_four_gate", "ensemble_score", "rank", "percentile",
            "rna_log2fc", "rna_qval", "cohen_d", "frac_tumours_expressing",
            "vital_max_tpm", "vital_max_tpm_raw",
            "vital_max_at_detection_floor", "safety_ratio_vs_psma_uncensored",
            "worst_organ", "safety_ratio_vs_psma",
            # Exported so a reader can RECONCILE the at-detection-floor
            # flag: the flag alone cannot be checked against anything.
            "vital_max_tpm_censored",
            "therapeutic_window_log2", "n_transmem", "ecd_len_aa",
            "mcrpc_mean_log2", "mcrpc_frac_expressing", "nepc_log2fc",
            "mcrpc_bimodality_coefficient",
            "prot_log2fc", "prot_qval", "protein_detected", "protein_corroborated",
            "n_antibodies", "F1_differential", "F2_safety", "F3_topology",
            "F4_reagent", "passed_all_filters"]
    cols = [c for c in cols if c in r.columns]
    out = r.sort_values("rank")[cols]
    save_table(out, "table2_full_candidate_ranking.csv",
               caption="Every surfaceome gene, scored and tiered", show=False)

    # ==================================================================
    # DISCOVERY SHORTLISTS - ONE PER DISEASE CONTEXT
    # ==================================================================
    # BUG FIX (critical, raised in external review): a SINGLE shortlist
    # required `passed_biological_screen`, which is primary-tumour
    # differential expression. So an antigen strongly expressed in
    # neuroendocrine or metastatic disease, with acceptable normal-tissue
    # exposure, an accessible ectodomain and a high model score, was EXCLUDED
    # for failing a primary-tumour criterion.
    #
    # For a pipeline whose stated purpose is finding alternatives to PSMA in
    # PSMA-low, metastatic and neuroendocrine disease, that is the central
    # scientific defect rather than a coding detail: the shortlist answered a
    # different question from the one the study asks.
    #
    # Each context now gets its own shortlist, scored and reported
    # SEPARATELY. A shared ranking is available as a secondary view, but the
    # primary outputs are the three context-specific lists.
    contexts = [
        ("primary", "eligible_primary",
         "Primary prostate adenocarcinoma"),
        ("mcrpc", "eligible_mcrpc",
         "Metastatic castration-resistant prostate cancer"),
        ("nepc", "eligible_nepc",
         "Neuroendocrine / AR-low prostate cancer"),
    ]
    def _demo_stamp(d):
        """In DEMO, every exported shortlist carries a synthetic-data guard.
        The CSVs outlive the notebook session and look identical to a real
        deliverable, so the warning travels with the file, not just the log."""
        if RUN_MODE == "DEMO" and len(d):
            return d.assign(
                DEMO_WARNING="SYNTHETIC IDENTIFIERS - NOT FOR EXPERIMENTAL "
                             "TARGET SELECTION")
        return d

    shortlists, ctx_rows = {}, []
    for key, flag, title in contexts:
        if flag not in out.columns or out[flag].isna().all():
            print(f"    {title}: NOT ASSESSABLE (no {flag}); no shortlist "
                  f"produced. This is recorded as unassessed, not as empty.")
            shortlists[key] = out.head(0)
            ctx_rows.append({"context": key, "title": title,
                             "status": "not assessable", "n_eligible": 0,
                             "n_shortlisted": 0})
            continue
        elig = out[flag].fillna(False).astype(bool)
        sl = out[out.is_novel & elig & (out.percentile >= 90)].sort_values("rank")
        shortlists[key] = sl
        sl = sl.assign(context_tier=sl[f"priority_tier_{key}"])
        # BUG FIX (raised in external code review, confirmed): the exported CSV
        # was truncated to .head(50) while the run summary reported the FULL
        # shortlist size, so a reader reconciling the two found 118 mCRPC
        # candidates claimed and 50 present. The deliverable silently omitted
        # 68 shortlisted candidates. The complete shortlist is now exported;
        # only the on-screen display is truncated.
        save_table(_demo_stamp(sl), f"table2b_{key}_shortlist.csv",
                   caption=f"DISCOVERY OUTPUT - {title}: UNLABELED candidates "
                           f"eligible in this disease context. "
                           f"UNLABELED means unlabeled in this pipeline, NOT "
                           f"established as novel - verify against the "
                           f"literature and development pipelines. "
                           f"(COMPLETE shortlist, n={len(sl)}; not truncated)")
        n_nb = int((sl.readiness == "binder development required").sum())
        print(f"    {title}: {int(elig.sum()):,} eligible -> "
              f"{len(sl)} novel shortlisted ({n_nb} need binder development)")
        ctx_rows.append({"context": key, "title": title, "status": "assessed",
                         "n_eligible": int(elig.sum()),
                         "n_shortlisted": int(len(sl))})

    save_table(pd.DataFrame(ctx_rows), "table2c_context_summary.csv",
               caption="Shortlist size per disease context")

    # NEW (raised in external review): "the present synthetic shortlisted
    # entries, including identifiers such as SURF0479, are placeholders rather
    # than real genes - they must not be selected for laboratory experiments."
    # Correct, and the risk is real: the exported CSVs look exactly like a
    # real deliverable and will outlive the notebook session. Every DEMO
    # shortlist now carries an explicit machine- and human-readable guard.
    if RUN_MODE == "DEMO":
        n_synth = sum(int(len(v)) for v in shortlists.values())
        warn = pd.DataFrame([{
            "WARNING": "DO NOT USE FOR EXPERIMENTAL TARGET SELECTION",
            "run_mode": RUN_MODE,
            "run_id": RUN_ID,
            "reason": ("Gene identifiers in this run are SYNTHETIC "
                       "placeholders generated to exercise the pipeline. "
                       "They are not human genes, the expression values are "
                       "simulated, and the survival endpoint is simulated. No "
                       "ranking, tier, shortlist or p-value in this archive "
                       "refers to real prostate cancer biology."),
            "rows_affected": n_synth,
            "required_before_use": ("Re-run with RUN_MODE=LIVE against "
                                    "documented TCGA-PRAD, SU2C/PCF, CPTAC "
                                    "and GTEx/HPA releases, pass the "
                                    "blocking preflight, then validate "
                                    "candidates by cell-surface assay."),
        }])
        save_table(warn, "table00_DEMO_DO_NOT_USE_FOR_EXPERIMENTS.csv",
                   caption="DEMO RUN GUARD - synthetic identifiers, not "
                           "experimental candidates")
        print("    DEMO guard written: shortlists are stamped "
              "SYNTHETIC and table00 states they must not be used for "
              "experimental target selection")

    # Secondary, explicitly-labelled union view. Reported for convenience,
    # never as the primary result, because a single ranking across contexts
    # implies a shared clinical objective that has not been defined.
    any_ctx = (out.eligible_primary.fillna(False)
               | out.eligible_mcrpc.fillna(False)
               | out.eligible_nepc.fillna(False))
    novel = out[out.is_novel & any_ctx & (out.percentile >= 90)].sort_values("rank")
    save_table(_demo_stamp(novel), "table2d_union_shortlist_secondary.csv",
               caption=f"SECONDARY view only: union across contexts "
                       f"(COMPLETE, n={len(novel)}; not truncated). The "
                       f"per-context lists are the primary outputs")

    # NEW (raised in external code review): PSMA-low disease is the clinical
    # gap this study is posed against, yet `suitable_for_psma_low` was
    # COMPUTED and then never required by any shortlist - 8 of the 50 exported
    # mCRPC candidates were annotated "not retained in PSMA-low". A target
    # that disappears in PSMA-low disease does not address the stated gap.
    #
    # The general mCRPC list is retained unchanged (a target can be valuable
    # in PSMA-expressing metastatic disease), and a SEPARATE shortlist now
    # enforces PSMA-low persistence explicitly, so the two questions are not
    # conflated.
    psma_low_sl = out.head(0)
    if "suitable_for_psma_low" in out.columns and out.suitable_for_psma_low.notna().any():
        psma_low_sl = out[
            out.is_novel
            & out.eligible_mcrpc.fillna(False).astype(bool)
            & out.suitable_for_psma_low.fillna(False).astype(bool)
            & (out.percentile >= 90)
        ].sort_values("rank")
        n_dropped = int(len(shortlists.get("mcrpc", out.head(0))) - len(psma_low_sl))
        save_table(_demo_stamp(psma_low_sl), "table2e_psma_low_shortlist.csv",
                   caption=f"DISCOVERY OUTPUT - PSMA-low mCRPC: novel "
                           f"candidates that additionally PERSIST in "
                           f"PSMA-low disease (n={len(psma_low_sl)}); "
                           f"{n_dropped} mCRPC-shortlisted candidate(s) do "
                           f"not meet this requirement")
        print(f"    PSMA-low enforced shortlist: {len(psma_low_sl)} novel "
              f"candidates persist in PSMA-low disease "
              f"({n_dropped} mCRPC candidates excluded by this requirement)")
    else:
        print("    PSMA-low enforced shortlist: NOT ASSESSABLE (PSMA-low "
              "persistence was not computed); recorded as unassessed")
    shortlists["psma_low_mcrpc"] = psma_low_sl

    # Sanity check, reported but never presented as a result: where do the
    # training positives land? If they are not concentrated at the top, the
    # model did not fit; if they are, that is expected and uninformative.
    pos_pct = out.loc[out.is_training_positive, "percentile"]
    print(f"    training positives: median percentile {pos_pct.median():.1f} "
          f"(expected high - this is a fit check, not a finding)")
    for key, title in [("primary", "primary"), ("mcrpc", "mCRPC"), ("nepc", "NEPC")]:
        col = f"priority_tier_{key}"
        if col not in out.columns:
            continue
        top_tiers = out[col].astype(str).str.startswith(("T1", "T2", "T3"))
        print(f"    {title}: {int(top_tiers.sum())} in T1-T3, "
              f"{int((out.is_novel & top_tiers).sum())} of them novel")
    print(f"    union (secondary) shortlist: {len(novel)} candidates "
          f"eligible in at least one disease context")
    return out, novel, shortlists


# ----------------------------------------------------------------------------
# STOP / GO QUALITY GATE
# ----------------------------------------------------------------------------
# PROPOSAL REFINEMENT (Section 7): the proposal declares explicit STOP
# conditions. Declaring them in prose is easy; the value comes from evaluating
# them automatically, before anyone writes a results paragraph. Each condition
# below maps to one the proposal names.
#
# A STOP does not mean the code failed. It means the EVIDENCE does not support
# a discovery claim, and the correct response is to fix the input or report the
# null result - not to loosen a threshold until the gate opens.
# ----------------------------------------------------------------------------
# BUG FIX (external review, correct): the gate collected every failure into
# one `stops` list, and "GO for LIVE analysis" then required `not stops`. So
# the ensemble failing to beat a simple baseline - a MODEL finding, not a data
# problem - blocked LIVE analysis entirely, even with clean data, verified
# labels and passing QC. Those are different decisions: the right response to
# "the ensemble adds nothing over a baseline" is to run LIVE with the simpler
# method as primary, not to refuse to run at all. The headline verdict still
# STOPs on any failure; the matrix now says WHICH work may proceed.
MODEL_VALIDATION_STOP_MARKERS = (
    "does not meet the PREDEFINED improvement rule",
    "Biology-only AUC",
    "benchmark enrichment is not distinguishable",
)


def classify_stop(msg):
    """Classify a STOP message: model_validation or data_quality."""
    for marker in MODEL_VALIDATION_STOP_MARKERS:
        if marker in msg:
            return "model_validation"
    return "data_quality"


def stop_go_gate(summary, feature_table, label_audit):
    stops, cautions = [], []

    # 1. Incorrect expression scale
    tm = feature_table.tumour_mean_log2
    if tm.max() > 40 or tm.min() < -20:
        stops.append("Expression scale implausible for log2 abundance "
                     f"(range {tm.min():.1f} to {tm.max():.1f}). A fold-change "
                     "computed on a mis-scaled matrix is wrong by an exponent.")

    # 2. Unresolved gene-ID loss
    if summary.get("gene_retention_pct", 100) < 85:
        stops.append(f"Gene retention across cohorts "
                     f"{summary['gene_retention_pct']:.0f}% (<85%). This is "
                     "symbol harmonisation failing, not biology.")

    # 3. Failed filter unit tests
    # QC artifacts must be FRESH and COMPLETE. A missing or stale file is a
    # STOP, not a silent pass: "no evidence the tests ran" and "the tests
    # passed" are different states and must not look the same to the gate.
    ut = DIRS["logs"] / "filter_unit_tests.csv"
    if not ut.exists():
        stops.append("No filter_unit_tests.csv for this run - the filter unit "
                     "tests did not run, so filter correctness is unverified.")
    else:
        u = pd.read_csv(ut)
        stale = ("run_id" not in u.columns) or (str(u.run_id.iloc[0]) != RUN_ID)
        n_exp = int(u.n_expected_controls.iloc[0]) if "n_expected_controls" in u.columns else len(u)
        if stale:
            stops.append(f"filter_unit_tests.csv is STALE (run_id "
                         f"{u.run_id.iloc[0] if 'run_id' in u.columns else 'absent'} "
                         f"!= current {RUN_ID}). QC from a previous run cannot "
                         f"certify this one.")
        elif (u.test == "FAIL").any() or len(u) != n_exp:
            stops.append(f"Filter unit tests FAILED or incomplete "
                         f"({int((u.test=='FAIL').sum())} failed, "
                         f"{n_exp - len(u)} controls missing).")

    # 4. Missing declared normal organs
    if summary.get("missing_vital_organs"):
        stops.append(f"Declared vital organs absent from the normal-tissue "
                     f"matrix: {summary['missing_vital_organs']}. The safety "
                     "filter is weaker than stated, in the direction that "
                     "flatters candidates.")

    # 5. Unverifiable positive labels
    if label_audit["pct_verified"] < 100:
        (stops if RUN_MODE == "LIVE" else cautions).append(
            f"Positive labels verified against the registry: "
            f"{label_audit['pct_verified']:.0f}%. The label set is the model's "
            f"entire supervision.")

    # 6. Training / held-out contamination
    overlap = set(g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")) \
              & set(PROSTATE_HELDOUT_POSITIVES)
    if overlap:
        stops.append(f"Held-out benchmarks also present in the training "
                     f"positives: {sorted(overlap)}. Evaluation is circular.")

    # 7. Near-null benchmark enrichment
    hb = summary.get("heldout_benchmark_result") or {}
    perm_p = hb.get("permutation_p") if hb else None
    if perm_p is None or not np.isfinite(perm_p):
        if hb:
            cautions.append("Benchmark enrichment was not testable (too few "
                            "benchmarks or background genes), so it is "
                            "reported as missing rather than as significant.")
    elif perm_p > 0.05:
        stops.append(f"Held-out benchmark enrichment is not distinguishable "
                     f"from chance (permutation p = {perm_p:.3g}). The model "
                     "has not learned transferable target biology.")
    elif hb and hb.get("n_in_top_decile", 0) < max(1, hb.get("n_heldout", 1) // 2):
        cautions.append(f"Only {hb['n_in_top_decile']}/{hb['n_heldout']} "
                        "benchmarks reached the top decile.")

    # 7b. PSMA rank: a documented failure INVESTIGATION, not a silent pass
    # PROPOSAL REFINEMENT: a poor PSMA rank should trigger an investigation
    # rather than be silently excluded or read as proof the framework is
    # invalid. The gate therefore names the specific things to check.
    psma_rank = summary.get("psma_rank")
    if psma_rank is not None and not pd.isna(psma_rank):
        n_tested = summary.get("n_surfaceome_tested", 1)
        if psma_rank > 0.10 * n_tested:
            stops.append(
                f"PSMA (FOLH1) ranked {int(psma_rank)} of {n_tested} - outside "
                f"the top decile. REQUIRED FAILURE INVESTIGATION before any "
                f"interpretation: (a) confirm FOLH1 symbol mapping survived "
                f"the merge; (b) check it is not in the training positives; "
                f"(c) inspect its tumour/normal fold-change and vital-organ "
                f"profile directly; (d) confirm the normal reference is "
                f"appropriate. Do NOT silently drop PSMA, and do not conclude "
                f"the framework is invalid without completing (a)-(d).")

    # 7c. Incremental utility over transparent baselines, against the
    # PREDEFINED rule in CFG - fixed before any outcome was examined.
    bv = summary.get("baseline_verdict", {})
    rs_rho = summary.get("ranking_stability", {}).get("median_spearman_rho", 1.0)
    if bv:
        if rs_rho < CFG.min_rank_stability_rho:
            cautions.append(
                f"Ranking stability (rho = {rs_rho:.2f}) is below the "
                f"{CFG.min_rank_stability_rho} precondition, so the "
                f"ensemble-vs-baseline comparison cannot be adjudicated "
                f"reliably. Stabilise the ranking before claiming either way.")
        elif not bv.get("ensemble_beats_baselines", True):
            stops.append(
                f"The PU ensemble does not meet the PREDEFINED improvement "
                f"rule against {bv['best_baseline']} "
                f"(gain {bv['incremental_gain_pct']:+.2f} pct pts, need "
                f">= {bv['required_margin_pct']}; top-decile gain "
                f"{bv['top_decile_gain']:+d}, need "
                f">= {bv['required_top_decile_gain']}). Report the simplest "
                f"adequate baseline as the primary method. The ensemble may "
                f"still be reported for uncertainty, leakage control and "
                f"interpretability - but not as better at the primary endpoint.")

    # 8. Performance collapse after removing fame features
    perf = pd.DataFrame(summary.get("model_performance_auc") or [])
    if len(perf):
        full = perf[perf.feature_set == "biology+research_intensity"]["mean"].mean()
        bio = perf[perf.feature_set == "biology_only"]["mean"].mean()
        if bio < 0.60:
            stops.append(f"Biology-only AUC {bio:.3f} - the model's apparent "
                         "performance rests on research-attention features.")
        elif (full - bio) > 0.10:
            cautions.append(f"Research-attention features add {full-bio:.3f} "
                            "AUC. Lead with the biology-only model and report "
                            "the gap prominently.")

    # 9. Ranking instability
    rs = summary.get("ranking_stability", {})
    if rs and rs.get("median_spearman_rho", 1.0) < 0.80:
        cautions.append(f"Ranking is seed-sensitive (median rho = "
                        f"{rs['median_spearman_rho']:.2f}). Report rank "
                        "intervals, not point ranks.")

    # 10. Insufficient subgroup survival events
    if summary.get("cox_refused_low_epv", 0) > 0:
        cautions.append(f"{summary['cox_refused_low_epv']} Cox models refused "
                        "for insufficient events per variable - consistent "
                        "with the pre-registered descriptive ancestry plan.")

    print("=" * 78)
    print("STOP / GO QUALITY GATE")
    print("=" * 78)
    if stops:
        print(f"  VERDICT: STOP  ({len(stops)} blocking condition(s))")
        for s in stops:
            print(f"    [STOP] {s}")
    else:
        print("  VERDICT: GO - no blocking condition met")
    for c in cautions:
        print(f"    [CAUTION] {c}")
    if RUN_MODE == "DEMO":
        print("\n  DEMO MODE: this gate validates the CODE PATH only.")
        print("  A GO verdict here is NOT evidence about biology, and the demo's")
        print("  AUC, rankings, sample sizes and survival estimates must never")
        print("  appear in an abstract. Re-run with RUN_MODE='LIVE'.")
    print("=" * 78)
    # ------------------------------------------------------------------
    # FINAL STOP/GO DECISION MATRIX
    # Five separate dispositions. A single overall "GO" is not meaningful:
    # a run can be sound enough to proceed with analysis while remaining
    # nowhere near ready for a results-based abstract, and collapsing those
    # into one verdict is how an exploratory run turns into a submission.
    # ------------------------------------------------------------------
    bv = summary.get("baseline_verdict", {})
    rs = summary.get("ranking_stability", {})
    live = RUN_MODE == "LIVE"
    data_stops = [m for m in stops if classify_stop(m) == "data_quality"]
    model_stops = [m for m in stops if classify_stop(m) == "model_validation"]
    matrix = [
        {"gate": "GO for LIVE analysis",
         "required": "Design frozen, valid data scales, auditable labels, "
                     "verified joins, filters and controls",
         # Data-quality failures only. A model that adds nothing over a simple
         # baseline does not make the DATA unfit to analyse.
         "met": (not data_stops) and label_audit["pct_verified"] == 100,
         "note": ("Proceed only after QC. Model-validation failures do not "
                  "block this gate - they decide which method is primary.")},
        {"gate": "GO for ensemble as primary",
         "required": "Meaningful, robust improvement over B1-B4 on "
                     "prespecified endpoints",
         "met": bool(bv.get("ensemble_beats_baselines", False))
                and rs.get("median_spearman_rho", 0) >= CFG.min_rank_stability_rho,
         "note": "Otherwise use the simplest adequate baseline"},
        {"gate": "GO for biological claims",
         "required": "LIVE data, appropriate uncertainty, valid cohort "
                     "interpretation",
         "met": live and not stops,
         "note": "No synthetic results"},
        {"gate": "GO for stronger generalisation claims",
         "required": "Frozen external cohort and relevant single-cell / "
                     "protein evidence",
         "met": bool(summary.get("external_transfer_tested"))
                and bool(summary.get("single_cell_assessed")),
         "note": "Otherwise label discovery-stage"},
        {"gate": "GO for AACR results abstract",
         "required": "Reproducible LIVE run, archived run summary, verified "
                     "tables and defensible limitations",
         "met": live and not stops and label_audit["pct_verified"] == 100,
         "note": "Conditional"},
    ]
    print("\n  FINAL STOP/GO DECISION MATRIX")
    for m in matrix:
        print(f"    [{'GO  ' if m['met'] else 'STOP'}] {m['gate']}")
        if not m["met"]:
            print(f"            needs: {m['required']}  ({m['note']})")
    save_table(pd.DataFrame(matrix), "table14_stop_go_matrix.csv",
               caption="Five separate submission dispositions")

    return {"verdict": "STOP" if stops else "GO",
            "stops": stops, "cautions": cautions,
            "data_quality_stops": data_stops,
            "model_validation_stops": model_stops,
            "n_data_quality_stops": len(data_stops),
            "n_model_validation_stops": len(model_stops),
            "decision_matrix": matrix}


def run_pipeline():
    import time
    t0 = time.time()
    # Clear the per-run artifact registry so a second invocation in the same
    # session cannot inherit the first run's artifacts (external review P2).
    # A fresh identity per invocation, before anything is stamped with it.
    new_run_identity()
    reset_artifact_registry()
    # SELF-REVIEW: the registry above is reset per run precisely so a second
    # invocation cannot inherit the first's artifacts - and I added
    # RELEASE_APPROVAL last round without the same treatment, which is the
    # identical stale-state defect one variable over. If run 1 passed and
    # run 2 raised before reaching the gate, the flag still read "approved"
    # from run 1, and a direct export_outputs() call would have packaged run
    # 2's partial outputs as a validated release. Approval is earned by a
    # run, not inherited from the session.
    RELEASE_APPROVAL.update({
        "approved": False,
        "reason": "tests have not yet run for this invocation",
        "invocation": int(RELEASE_APPROVAL.get("invocation", 0)) + 1})
    RELEASE_STATE.update({
        "status": "RUNNING", "verified": False, "run_id": RUN_ID,
        "invocation": int(RELEASE_STATE.get("invocation", 0)) + 1,
        "reason": "pipeline running; not yet tested or verified"})
    print("=" * 78)
    print("SURFACEOME-WIDE MULTI-OMIC TARGET PRIORITISATION PIPELINE")
    print(f"Mode: {RUN_MODE} | seed: {CFG.seed} | profile: {PROFILE} "
          f"| purpose: {RUN_PURPOSE}")
    print("=" * 78)
    if not REPORTABLE:
        print("  RUN_PURPOSE=data_only: the real datasets will be acquired, "
              "validated and cached,\n"
              "  then the run STOPS before any model is fitted. No ranking, "
              "no benchmark comparison\n"
              "  and no stop/go verdict are produced. Nothing from this run "
              "is reportable.")

    # --- Stage 0a: configuration contracts, BEFORE any network access -------
    # Moved here from inside load_clinical_live(), which is the fourth
    # acquisition step: an invalid endpoint contract used to cost a
    # multi-gigabyte Toil download before it was detected.
    if RUN_MODE == "LIVE":
        validate_live_configuration()

    # --- Stage 0: pre-registration checks -----------------------------------
    print("[0] Label and benchmark provenance audit")
    # Load the evidence registry before auditing it, so a curated CSV is
    # picked up without editing code. Absent, the audit reports the labels
    # as unverified exactly as before.
    try:
        load_label_provenance()
    except Exception as _e:
        raise RuntimeError(
            f"The label provenance registry could not be loaded: {_e}") from _e
    label_audit = audit_labels()
    # Emit a fillable template whenever the registry is incomplete, so the
    # curation task is a spreadsheet rather than a code edit.
    try:
        if not LABEL_PROVENANCE or label_audit.get("pct_verified", 0) < 100:
            write_label_provenance_template()
    except Exception as _e:
        print(f"  (could not write the label template: {_e!r})")
    benchmark_audit = audit_benchmarks()

    # --- Stage 1: extract ---------------------------------------------------
    annot = load_protein_annotation()
    surf = bound_to_surfaceome(annot)
    prad_t, prad_n, gtex = load_transcriptomes(surf)
    mcrpc_expr, mcrpc_pheno = load_mcrpc(surf)
    hpa = load_hpa(surf)
    clin = load_clinical(prad_t.columns.tolist())
    # Samples AS LOADED. The post-QC manifest is recorded after cleaning
    # (external review, correct): registering only here described the samples
    # read from disk, not the samples that entered each analysis, so the
    # manifest could not be used to reconstruct a contrast.
    register_samples("TCGA-PRAD tumour (as loaded)", prad_t.columns)
    register_samples("TCGA-PRAD normal (as loaded)", prad_n.columns)
    register_samples("SU2C mCRPC (as loaded)", mcrpc_expr.columns)

    if not REPORTABLE:
        # data_only stop point. Everything above is acquisition and schema
        # validation of REAL inputs - exactly the part that has never been
        # exercised against the live hosts - and everything below fits
        # models. Stopping here lets the loaders be debugged against real
        # TCGA/GTEx/SU2C files while the label registry is still being
        # curated, without the provenance gate having to be weakened.
        _do = DIRS["logs"] / "data_only_acquisition_report.csv"
        _rows = [
            {"dataset": "surfaceome (UniProt)", "n_rows": len(surf),
             "n_cols": int(surf.shape[1])},
            {"dataset": "TCGA-PRAD tumour", "n_rows": int(prad_t.shape[0]),
             "n_cols": int(prad_t.shape[1])},
            {"dataset": "TCGA-PRAD normal", "n_rows": int(prad_n.shape[0]),
             "n_cols": int(prad_n.shape[1])},
            {"dataset": "GTEx", "n_rows": int(gtex.shape[0]),
             "n_cols": int(gtex.shape[1])},
            {"dataset": "SU2C mCRPC expression",
             "n_rows": int(mcrpc_expr.shape[0]),
             "n_cols": int(mcrpc_expr.shape[1])},
            {"dataset": "HPA annotation", "n_rows": int(hpa.shape[0]),
             "n_cols": int(hpa.shape[1])},
            {"dataset": "clinical", "n_rows": int(clin.shape[0]),
             "n_cols": int(clin.shape[1])},
        ]
        _df = pd.DataFrame(_rows)
        _df["run_purpose"] = "data_only"
        _df["reportable"] = "NO - acquisition diagnostic only"
        _df["run_id"] = RUN_ID
        _df.to_csv(_do, index=False)
        register_artifact(_do, kind="table")
        write_run_manifest({"run_purpose": "data_only",
                            "reportable": False,
                            "stopped_before": "model fitting"})
        print("\n" + "=" * 78)
        print("DATA-ONLY RUN COMPLETE - acquisition and validation only")
        print("=" * 78)
        print(_df.to_string(index=False))
        print("\nNo model was fitted. No ranking, benchmark comparison or "
              "stop/go verdict exists for this run,")
        print("and nothing here is reportable. Re-run with "
              "RUN_PURPOSE=analysis once the label registry")
        print("is curated, to produce a scientific result.")
        return None, None, None, None

    # --- Stage 2: merge and clean ------------------------------------------
    print("[2.1] Harmonising gene identifiers across cohorts")
    # ------------------------------------------------------------------
    # GENE UNIVERSE
    # ------------------------------------------------------------------
    # RAISED IN EXTERNAL REVIEW, and correct: intersecting ALL cohorts meant a
    # gene absent from the optional metastatic cohort vanished from the
    # primary-disease analysis too. Missing evidence in an optional dataset is
    # not evidence of unsuitability for a different disease context.
    #
    # The REQUIRED universe is the surfaceome measured in primary tumour and
    # normal tissue. Metastatic data is attached where it exists and recorded
    # as missing where it does not.
    n_before_merge = len(surf)
    print("  required cohorts (primary tumour, normal, GTEx):")
    common = harmonise_genes(prad_t, prad_n, gtex, set(surf.gene))
    gene_retention_pct = 100 * len(common) / max(n_before_merge, 1)
    # Lineage markers are retained alongside the shared candidate genes: they
    # stratify the cohort (AR activity, neuroendocrine programme) and are
    # never candidates. Without this they are dropped here and every
    # lineage-stratified analysis becomes unrunnable - in LIVE as well as DEMO.
    mcrpc_shared = sorted(((set(common) | set(LINEAGE_MARKER_GENES))
                           & set(mcrpc_expr.index)))
    n_missing_mcrpc = len(set(common) - set(mcrpc_shared))
    print(f"  optional cohort (mCRPC): {len(mcrpc_shared):,}/{len(common):,} "
          f"genes measured; {n_missing_mcrpc:,} retained with mCRPC evidence "
          f"recorded as MISSING rather than dropped")
    # BUG FIX (external review, correct): this block existed TWICE, and the
    # second copy ran AFTER prad_t had already been restricted to the
    # surfaceome and cleaned - so it reassigned `prad_t_markers` from a
    # matrix with no markers left in it, silently undoing the fix. DEMO did
    # not expose this because the synthetic surfaceome annotation happens to
    # include KRT8/KRT18/KRT19/COL1A1 as "surface" genes, so some markers
    # survived by accident. On real data the panels would have been lost.
    #
    # The marker-bearing matrix is now built ONCE, from the UNRESTRICTED
    # tumour matrix, cleaned on its own, and never reassigned. It is used for
    # compartment scoring only; `surf` is untouched, so a marker can never
    # become a candidate.
    marker_keep = sorted((set(common) | set(COMPARTMENT_MARKER_GENES))
                         & set(prad_t.index))
    prad_t_markers = clean_expression(
        prad_t.loc[marker_keep], "PRAD tumour (+compartment markers)",
        min_frac=0.0, drop_zero_var=False)
    # Report what matters: how many markers are PRESENT for scoring, and
    # separately how many of those are non-surface (the LIVE-relevant number,
    # since those are the ones surfaceome bounding would otherwise remove).
    _present = sorted(set(COMPARTMENT_MARKER_GENES) & set(prad_t_markers.index))
    _nonsurf = sorted(set(_present) - set(common))
    print(f"    compartment markers available for scoring: {len(_present)} of "
          f"{len(set(COMPARTMENT_MARKER_GENES))} declared "
          f"({len(_nonsurf)} of them non-surface, i.e. recoverable only "
          f"because this matrix was preserved before bounding)")
    if len(_present) < 6:
        print(f"    WARNING: only {len(_present)} compartment marker(s) are "
              f"present. The epithelial-restriction score needs at least two "
              f"epithelial and two immune markers; it will skip rather than "
              f"score on a degenerate panel.")
    prad_t, prad_n = prad_t.loc[common], prad_n.loc[common]
    gtex = gtex.loc[common]
    mcrpc_expr = mcrpc_expr.loc[mcrpc_shared]
    surf = surf[surf.gene.isin(common)].reset_index(drop=True)

    print("[2.2] Cleaning expression matrices")
    # CRITICAL BUG FIX: no per-group zero-variance filtering. A gene that is
    # undetectable in every adjacent normal has exactly zero variance there,
    # and the old per-group filter deleted it before the contrast was ever
    # tested - removing precisely the tumour-restricted antigens this study
    # exists to find. Variance is now assessed jointly, after harmonisation.
    prad_t = clean_expression(prad_t, "PRAD tumour", min_frac=0.0,
                              drop_zero_var=False)
    prad_n = clean_expression(prad_n, "PRAD normal", drop_zero_var=False)
    mcrpc_expr = clean_expression(mcrpc_expr, "mCRPC", drop_zero_var=False)
    gtex = clean_expression(gtex, "GTEx tissue medians", drop_zero_var=False)
    jv_keep, jv_report = joint_variance_filter(prad_t, prad_n,
                                               "PRAD tumour vs normal")
    prad_t = prad_t.loc[[g for g in prad_t.index if g in set(jv_keep)]]
    prad_n = prad_n.loc[[g for g in prad_n.index if g in set(jv_keep)]]
    common = harmonise_genes(prad_t, prad_n, gtex)
    # Post-QC sample manifest: exactly which samples entered each contrast,
    # with the count dropped at QC recorded so exclusions are auditable.
    register_samples("TCGA-PRAD tumour (analysed)", prad_t.columns)
    register_samples("TCGA-PRAD normal (analysed)", prad_n.columns)
    register_samples("SU2C mCRPC (analysed)", mcrpc_expr.columns)
    prad_t, prad_n = prad_t.loc[common], prad_n.loc[common]
    gtex = gtex.loc[common]
    # Retain lineage markers in the metastatic matrix even though they are
    # outside the surfaceome: they are STRATIFYING variables (AR activity,
    # neuroendocrine programme), not candidates, and dropping them here is
    # what made the lineage-stratified analyses unrunnable.
    _keep_mcrpc = (set(common) | set(LINEAGE_MARKER_GENES)) & set(mcrpc_expr.index)
    mcrpc_expr = mcrpc_expr.loc[sorted(_keep_mcrpc)]
    _lm = sorted(set(LINEAGE_MARKER_GENES) & set(mcrpc_expr.index) - set(common))
    if _lm:
        print(f"    lineage markers retained for stratification only "
              f"(never candidates): {_lm}")
    surf = surf[surf.gene.isin(common)].reset_index(drop=True)

    # --- Stage 3: features --------------------------------------------------
    print("[2.3] Differential expression (tumour vs adjacent normal)")
    # PROPOSAL REFINEMENT: prefer a count-based NB GLM when raw counts exist.
    counts_file = os.environ.get("COUNTS_FILES", "")
    if CFG.de_method in ("auto", "nb_counts") and counts_file:
        de_method_used = "negative_binomial_glm_on_counts"
        print(f"    primary test: NB GLM on raw counts ({counts_file})")
        # Expected: two CSVs separated by a comma, tumour then normal,
        # each genes x samples of raw counts.
        for _cp in counts_file.split(","):
            register_input_file(_cp.strip(), role="raw counts for NB GLM")
        ct, cn = [pd.read_csv(p.strip(), index_col=0)
                  for p in counts_file.split(",")]
        de_nb = differential_expression_nb(ct, cn)
        de = differential_expression(prad_t, prad_n)   # sensitivity analysis
        de = de.merge(de_nb, on="gene", how="left")

        # BUG FIX (raised in external review): the NB result was written over
        # the shared columns only where it existed, leaving Welch results in
        # the SAME columns for every other gene. Downstream filtering then
        # mixed two different statistical models with no record of which gene
        # used which, so a single FDR threshold was applied across two
        # distinct testing procedures.
        #
        # Coverage is now required to be near-complete for the count-based
        # analysis to be primary. Below that, the count result is retained as
        # a labelled sensitivity analysis and Welch stays primary for ALL
        # genes - one method for the whole table, never a blend.
        ok = de.rna_qval_nb.notna()
        coverage = float(ok.mean())
        print(f"    count-based coverage: {coverage:.1%} of genes")
        if coverage >= 0.95:
            de.loc[ok, "rna_log2fc"] = de.loc[ok, "rna_log2fc_nb"]
            de.loc[ok, "rna_qval"] = de.loc[ok, "rna_qval_nb"]
            de.loc[ok, "rna_pval"] = de.loc[ok, "rna_pval_nb"]
            de["de_method_per_gene"] = np.where(ok, de_method_used, "EXCLUDED")
            # Genes without a count result are dropped from the primary
            # analysis rather than silently falling back to Welch.
            n_drop = int((~ok).sum())
            if n_drop:
                print(f"    {n_drop} genes lack a count-based result and are "
                      f"EXCLUDED from the primary analysis rather than "
                      f"silently reverting to Welch")
                de = de[ok].reset_index(drop=True)
        else:
            # BUG FIX (external review, correct): when the user has
            # explicitly prespecified de_method='nb_counts', low NB coverage
            # must NOT quietly promote Welch to primary - that substitutes a
            # different primary analysis for a data-quality problem. Under an
            # explicit NB request in LIVE this is a blocking failure; under
            # 'auto' the documented fallback stands.
            cov_msg = (
                f"the count-based NB analysis covers fewer than 95% of genes, "
                f"so it cannot serve as the primary test for the whole table.")
            if CFG.de_method == "nb_counts" and RUN_MODE == "LIVE":
                raise ValueError(
                    f"de_method='nb_counts' was prespecified but {cov_msg} "
                    f"Reverting to Welch would silently change the primary "
                    f"analysis. Either fix the count inputs so coverage is "
                    f"adequate, or amend the analysis plan explicitly by "
                    f"setting CFG.de_method='welch'.")
            print(f"    coverage below 95%: {cov_msg} The count-based result "
                  f"is kept as a LABELLED SENSITIVITY analysis (columns *_nb) "
                  f"and Welch remains primary for every gene. One method for "
                  f"the whole table.")
            de_method_used = "welch_t_on_log_abundance (NB retained as sensitivity)"
            de["de_method_per_gene"] = "welch_t_on_log_abundance"
    else:
        de_method_used = "welch_t_on_log_abundance"
        if CFG.de_method == "nb_counts":
            # BUG FIX (external review, correct): a missing input file must
            # not silently change the PRIMARY statistical method of a
            # prespecified analysis. In LIVE this is now fatal; in DEMO it
            # remains a warning, because DEMO has no counts by construction.
            msg = ("de_method='nb_counts' was requested - a count-based "
                   "negative-binomial GLM as the PRIMARY differential-"
                   "expression test - but COUNTS_FILES is unset. Silently "
                   "substituting Welch on log abundance would change the "
                   "prespecified primary analysis because a file was "
                   "missing. Supply COUNTS_FILES, or set CFG.de_method to "
                   "'welch' to choose Welch deliberately.")
            if RUN_MODE == "LIVE":
                raise ValueError(msg)
            print(f"    NOTE (DEMO only): {msg}")
        print("    primary test: Welch t-test on log2 abundance. Raw counts "
              "were not supplied, so a count-based NB model (the preferable "
              "primary analysis) was not available. Record this in Methods.")
        de = differential_expression(prad_t, prad_n)
        de["de_method_per_gene"] = "welch_t_on_log_abundance"

    print("[2.3b] Vital-organ safety features")
    # BUG FIX (both external reviews, independently correct, and the most
    # serious defect of this round): this audit matched the ORIGINAL column
    # labels while safety_features() resolved them through
    # harmonise_tissue_labels(). The DEMO matrix carries GTEx v8's "Minor
    # Salivary Gland", so the audit declared "Salivary Gland" absent and
    # raised a data-quality STOP -
    #
    #   [STOP] Declared vital organs absent from the normal-tissue matrix:
    #          ['Salivary Gland']
    #
    # - in the SAME run whose feature table reported 100% vital-organ
    # coverage and an assessable F2 comparator computed over that organ.
    # One run, two contradictory statements about the same data: the same
    # defect class as the `~True` F2_outcome bug, and a false STOP is as
    # damaging to a reader's trust as a false GO.
    #
    # The audit now consumes the resolved panel, so there is ONE answer to
    # "which declared organs are present" for the whole pipeline.
    _resolved_vital, missing_vital = harmonise_tissue_labels(
        gtex.columns, verbose=False)
    organ_coverage_audit = (len(_resolved_vital) / len(set(CFG.vital_organs))
                            if CFG.vital_organs else 0.0)
    safety = safety_features(gtex)
    unmeasurable_organ_report()
    epi_rest = epithelial_restriction(prad_t_markers, surf)
    print("[2.4] mCRPC and NEPC features")
    mc = mcrpc_features(mcrpc_expr, mcrpc_pheno)
    prot = load_proteomics(surf, de.set_index("gene").rna_log2fc)

    f = build_feature_table(surf, de, safety, mc, prot, hpa)
    if epi_rest is not None:
        f = f.merge(epi_rest, on="gene", how="left")
    # Experimental surface evidence must be attached BEFORE gatekeeping: it
    # is the only route by which a non-anchored protein can pass F3.
    f = experimental_surface_evidence(f)


    f, attrition = apply_gatekeeping(f)
    floor_sens = detection_floor_sensitivity(gtex, f)

    print("[2.6b] Modality-specific target suitability")
    f = modality_suitability(f, gtex)

    print("[2.7] Surface-targetability evidence grade")
    f = surface_targetability_grade(f)

    print("[2.8] PSMA-high vs PSMA-low persistence in metastatic disease")
    psma_low = psma_low_persistence(mcrpc_expr)
    if psma_low is not None:
        f = f.merge(psma_low.drop(columns=["psma_low_frac_expressing"]),
                    on="gene", how="left")
        save_table(psma_low, "table17_psma_low_persistence.csv",
                   caption="Candidate expression in PSMA-low vs PSMA-high "
                           "metastatic tumours. RNA-based molecular "
                           "stratification on bulk FOLH1 - NOT equivalent to "
                           "clinically PSMA-negative or PSMA-PET-ineligible "
                           "disease.")

    # NEW: a second, independent resistance axis on AR signalling. Lineage
    # plasticity after AR-pathway inhibition is the dominant route to surface
    # antigen loss, and a target that disappears in AR-low disease does not
    # address the clinical gap this study is posed against.
    ar_low = lineage_axis_persistence(mcrpc_expr, axis_gene="AR", axis_name="AR")
    if ar_low is not None:
        f = f.merge(ar_low.drop(columns=["ar_low_frac_expressing"]),
                    on="gene", how="left")

    # --- Stage 4: machine learning -----------------------------------------
    print("[3.1] Constructing labels (positive-unlabeled)")
    y, is_heldout = make_labels(f)
    # Clinical-stage targets outside the primary payload class: withheld from
    # training (they are known targets, so they must not become pseudo-
    # negatives) but NOT counted as evaluation benchmarks.
    off_class = off_class_mask(f)
    # Both benchmark sets are withheld from fitting so the retrospective
    # comparison is training-equivalent; only the revised set is the
    # evaluation endpoint (`is_heldout`).
    eval_exclude = evaluation_exclude_mask(f)
    train_exclude = is_heldout | off_class | eval_exclude
    n_hist_only = int((eval_exclude & ~is_heldout).sum())
    if n_hist_only:
        print(f"    {n_hist_only} historical-only benchmark(s) also withheld "
              f"from training so both benchmark sets face the model under "
              f"identical conditions (they are NOT primary endpoints)")
    if off_class.sum():
        print(f"    {int(off_class.sum())} off-class clinical target(s) "
              f"withheld from training only (not benchmarks)")
    label_class_audit = audit_label_classes()
    if y.sum() < 10:
        raise RuntimeError(
            "Fewer than 10 positives present after filtering. The label gene "
            "symbols are not matching the expression matrix - check symbol "
            "harmonisation before interpreting anything downstream.")

    full_cols = [c for c in BIOLOGY_FEATURES + RESEARCH_INTENSITY_FEATURES if c in f.columns]
    bio_cols = [c for c in BIOLOGY_FEATURES if c in f.columns]
    X_full = build_design(f, full_cols)
    X_bio = build_design(f, bio_cols)
    print(f"    design matrix: {X_full.shape[0]:,} genes x {X_full.shape[1]} features "
          f"({len(bio_cols)} biology, {len(full_cols)-len(bio_cols)} research-intensity)")

    # ------------------------------------------------------------------
    # BLOCKING PREFLIGHT - nothing is fitted until the labels pass
    # ------------------------------------------------------------------
    print("[3.2] Preflight label-integrity audit (BLOCKING)")
    preflight = preflight_label_audit(
        f, y, train_exclude, bio_cols,
        benchmark_mask=is_heldout,
        label_audit=label_audit, benchmark_audit=benchmark_audit)

    print("[3.3] Bagged PU learning - FULL feature set")
    score_full, perf_full, coefs_full, bp = pu_bagged_nested_cv(
        X_full, y, train_exclude, label="full", genes=f.gene.values)
    print("[3.3b] Leakage ablation - BIOLOGY-ONLY feature set")
    score_bio, perf_bio, coefs_bio, _ = pu_bagged_nested_cv(
        X_bio, y, train_exclude, label="biology-only", genes=f.gene.values)

    # NEW ABLATION ARM: biology WITHOUT protein-architecture covariates.
    # ECD length was the top feature and separates positives from background
    # on its own, because clinical-stage targets are drawn from large
    # single-pass receptor families - a prior on the label set, not prostate
    # biology. This arm tests whether the ranking survives without it.
    arch_cols = [c for c in bio_cols if c in PROTEIN_ARCHITECTURE_FEATURES]
    noarch_cols = [c for c in bio_cols if c not in PROTEIN_ARCHITECTURE_FEATURES]
    perf_noarch = pd.DataFrame()
    arch_delta = None
    if arch_cols and len(noarch_cols) >= 3:
        print(f"[3.3c] Architecture-prior ablation - biology WITHOUT "
              f"{len(arch_cols)} structural covariates {arch_cols}")
        _, perf_noarch, _, _ = pu_bagged_nested_cv(
            X_bio[noarch_cols], y, train_exclude,
            label="biology-minus-architecture", genes=f.gene.values)
        perf_noarch["feature_set"] = "biology_minus_architecture"
    else:
        print("[3.3c] Architecture-prior ablation SKIPPED: too few "
              "non-architecture biology features to fit a comparable model")

    perf_full["feature_set"] = "biology+research_intensity"
    perf_bio["feature_set"] = "biology_only"
    perf = pd.concat([d for d in [perf_full, perf_bio, perf_noarch]
                      if len(d)], ignore_index=True)
    if len(perf_noarch):
        a = perf_bio[perf_bio.metric == "auc"]["mean"].mean()
        b = perf_noarch[perf_noarch.metric == "auc"]["mean"].mean()
        arch_delta = {
            "architecture_features_removed": arch_cols,
            "biology_only_auc": round(float(a), 4),
            "biology_minus_architecture_auc": round(float(b), 4),
            "auc_attributable_to_architecture_prior": round(float(a - b), 4),
        }
        print(f"    biology-only AUC {a:.3f} -> without architecture "
              f"{b:.3f} (delta {a-b:+.3f}). A large drop means the model is "
              f"substantially a detector of druggable protein architecture "
              f"rather than of prostate cancer expression biology.")
    save_table(perf, "table9_model_performance.csv",
               caption="Cross-validated performance across three feature "
                       "sets: full, biology-only, and biology without "
                       "protein-architecture covariates")
    if not IN_NOTEBOOK:
        print(perf[perf.metric == "auc"].to_string(index=False))

    print("[3.4] Resampled selection frequency")
    stability = stability_selection(coefs_bio if not coefs_bio.empty else coefs_full)

    print("[3.5] Permutation null (same learners as the observed statistic)")
    # CRITICAL BUG FIX (found reviewing this round's own changes): the
    # off-class exclusion was applied ONLY to the three pu_bagged_nested_cv
    # calls and the frozen model. Every other analysis still received
    # `is_heldout`, so the 17 clinical-stage targets outside the primary
    # payload class carried y=0 there - i.e. 17 KNOWN DRUG TARGETS were used
    # as pseudo-negatives in the permutation null, the permutation importance,
    # the ranking-stability analysis, the fusion sensitivity test and the
    # ROC/PR and calibration figures. 15 of those 17 rank in the top decile
    # (median percentile 98.2), so they appeared as high-scoring false
    # positives and depressed average precision for a reason with no
    # statistical justification.
    #
    # Worse, permutation_null_auc re-fits under permuted labels: with
    # `is_heldout` it trained on a population 17 genes larger than the
    # observed model did, breaking the null-must-match-the-observed-procedure
    # invariant established in an earlier round. Same defect class as the
    # matched-statistic bug, in a new location.
    #
    # RULE, applied consistently from here: `train_exclude` for anything that
    # fits, scores or summarises the MODEL; `is_heldout` only for the
    # benchmark-evaluation endpoint.
    null_res = permutation_null_auc(X_bio, y, train_exclude,
                                    estimator="ensemble")
    nulls = null_res["null"]

    print("[3.5b] Frozen final scorer (consistent scores for every gene)")
    frozen = fit_frozen_model(X_bio, y, train_exclude)
    frozen_export = export_frozen_model(frozen)

    # ------------------------------------------------------------------
    # SEPARATE EVALUATION FROM FINAL SCORING
    # ------------------------------------------------------------------
    # RAISED IN EXTERNAL REVIEW, and correct: out-of-fold/out-of-bag scores
    # are not generated identically for every gene. A training positive sits
    # in every bag and receives one out-of-fold prediction per bag; an
    # unlabeled gene is usually out-of-bag and is scored by every fold model,
    # about n_outer_folds times more often. More averaging means a smoother
    # score, so the two groups were measured with different precision in the
    # very comparison that produces the ranking.
    #
    # The two jobs are now split:
    #   - UNBIASED PERFORMANCE ESTIMATION keeps the out-of-fold scores, which
    #     is what they are valid for.
    #   - FINAL CANDIDATE SCORING uses the frozen ensemble to score every gene
    #     through one identical procedure, so ranks are comparable by
    #     construction.
    if frozen is not None:
        final_scores = score_with_frozen(frozen, X_bio)
        score_for_ranking = pd.DataFrame({"Ensemble": final_scores})
        print(f"    all {len(final_scores):,} genes scored by one identical "
              f"procedure; out-of-fold scores retained for performance "
              f"estimation only")
        rho = st.spearmanr(score_bio["Ensemble"].values, final_scores)
        print(f"    agreement with the out-of-fold ranking: Spearman rho = "
              f"{rho.statistic:.3f} (a low value would mean the two scoring "
              f"routes disagree and the earlier ranking was resampling-driven)")
    else:
        score_for_ranking = score_bio
        print("    frozen scorer unavailable; falling back to out-of-fold "
              "scores and noting the comparability caveat")
        rho = None
    ext_res = {}
    ext_file = os.environ.get("EXTERNAL_COHORT_FEATURES", "")
    if ext_file and Path(ext_file).exists():
        Xe_raw = pd.read_csv(ext_file)
        ext_scores = score_external_cohort(frozen, build_design(Xe_raw, bio_cols),
                                           Xe_raw.gene.values)
    else:
        ext_scores = None
        print("    SKIPPED: no EXTERNAL_COHORT_FEATURES supplied. The model is "
              "frozen and ready, but external transferability is UNTESTED - "
              "internal benchmark enrichment is not external validation. State "
              "this plainly rather than implying validation you have not done.")

    print("[3.6] Held-out prostate benchmark evaluation (primary endpoint)")
    # The biology-only model is used for prioritisation: it is the one whose
    # output can be described as a biological prediction.
    d, hb, hb_res = evaluate_heldout_benchmarks(f, score_for_ranking, is_heldout)

    print("[3.7] Feature importance")
    imp, imp_agg = feature_importance(X_bio, y, train_exclude)

    # NEW (found in our own review): the two interpretability tables DISAGREE,
    # and the disagreement needs stating rather than leaving for a reviewer.
    # In the last run `cohen_d` was selected in 100% of stability-selection
    # runs with a consistent-sign coefficient of 0.34, yet had near-zero
    # permutation importance. That is the textbook signature of collinearity:
    # cohen_d, rna_log2fc and tumour_mean_log2 measure overlapping quantities,
    # so permuting any ONE of them barely moves the prediction because the
    # others still carry the signal, while a sparse linear model still selects
    # it reliably. Neither table is wrong; reporting either alone is.
    collinearity_note = None
    try:
        both = (imp_agg.set_index("feature")["mean_importance_across_models"]
                .to_frame("permutation_importance")
                .join(stability.set_index("feature")[["selection_frequency",
                                                      "mean_coef"]],
                      how="inner"))
        if len(both):
            thr = both.permutation_importance.quantile(0.25)
            disc = both[(both.selection_frequency >= 0.9)
                        & (both.permutation_importance <= thr)]
            collinearity_note = {
                "n_features_compared": int(len(both)),
                "stably_selected_but_low_permutation_importance":
                    sorted(disc.index.tolist()),
                "interpretation": ("Features stably selected by the sparse "
                                   "linear model but with low permutation "
                                   "importance are collinear with other "
                                   "retained features. Report feature groups, "
                                   "not individual feature ranks, and do not "
                                   "read a low permutation importance as "
                                   "biological irrelevance."),
            }
            if len(disc):
                print(f"    COLLINEARITY: {sorted(disc.index.tolist())} are "
                      f"stably selected ({'/'.join('%.2f' % v for v in disc.selection_frequency)}) "
                      f"yet have low permutation importance - they share "
                      f"signal with correlated features. Interpret at the "
                      f"feature-GROUP level; the two tables are reconciled in "
                      f"the run summary.")
    except Exception as e:
        print(f"    collinearity reconciliation skipped: {e}")

    print("[3.8] Ranking stability across seeds and bagging assumptions")
    stab_res, stab_disp = ranking_stability(
        X_bio, y, train_exclude, genes=f.gene.values, scorer="frozen",
        benchmark_mask=is_heldout)

    print("[3.9] Tier reachability check")
    tier_reach = test_tier_reachability()

    print("[3.10] Candidate ranking")
    ranked, novel, shortlists = rank_candidates(d, y, is_heldout)
    hist_bench = historical_benchmark_sensitivity(ranked)
    if ext_scores is not None:
        ext_res = compare_internal_external(ranked, ext_scores)

    print("[3.10b] Protein/NEPC late-fusion sensitivity analysis")
    # Two masks, two jobs: train_exclude governs fitting, is_heldout governs
    # benchmark evaluation. Passing one mask for both is what made the fusion
    # analysis report 15 "benchmarks" when there are six.
    fusion_res = fusion_sensitivity_model(f, y, train_exclude, bio_cols,
                                          benchmark_mask=is_heldout)

    print("[3.11] Simple-baseline comparison (does the ensemble earn its complexity?)")
    baselines, baseline_verdict = simple_baselines(
        f, is_heldout, ensemble_scores=score_for_ranking["Ensemble"].values,
        exclude_from_background=off_class,
        # Passed so the sensitivity note can carry the measured spread
        # instead of the name of the variable holding it.
        stability=stab_res)

    print("[3.12] Single-cell / spatial heterogeneity assessment")
    sc_res = single_cell_assessment(
        drop_synthetic(novel, label="single-cell input", verbose=False).gene.tolist())

    # --- Stage 5: survival --------------------------------------------------
    print("[4.1] Survival power accounting")
    power = survival_power_report(clin)
    # Survival is tested on NOVEL candidates. Testing it on already-drugged
    # targets would answer a question nobody asked.
    # Survival is run on control-free novel candidates. The union view is
    # used here deliberately: survival is secondary exploratory evidence
    # about candidates, not a context-specific eligibility criterion.
    surv_pool = drop_synthetic(novel, label="survival candidate pool")
    shortlist = surv_pool.gene.head(8).tolist()
    if not shortlist:
        surv_pool = drop_synthetic(ranked[ranked.is_novel],
                                   label="survival fallback pool")
        shortlist = surv_pool.gene.head(4).tolist()
        print("    NOTE: no novel gene reached Tier 1-2; using the top-ranked "
              "novel genes for the survival illustration. These are not "
              "validated hits.")
    km_df, cox_df = run_survival_suite(prad_t, clin, shortlist)

    # --- Stage 6: figures ---------------------------------------------------
    print("[5] Figures")
    label_genes = list(dict.fromkeys(
        drop_synthetic(novel, label="figure labels", verbose=False)
        .gene.head(6).tolist() + PROSTATE_HELDOUT_POSITIVES[:4]))
    fig_attrition(attrition)
    fig_volcano(f, label_genes)
    fig_safety(f, label_genes)
    curve_res = fig_roc_pr(score_bio, y, train_exclude, nulls,
                           matched=null_res)
    calib = fig_calibration(score_bio, y, train_exclude)
    brier = calib["brier"]
    fig_forest(cox_df)
    heat_src = novel if len(novel) >= 8 else ranked[ranked.is_novel]
    fig_candidate_heatmap(drop_synthetic(heat_src, label="heatmap", verbose=False))
    fig_importance(imp_agg, stability)
    fig_ablation(perf_full, perf_bio)
    fig_baselines(baselines)

    # --- Stage 7: machine-readable run summary ------------------------------
    summary = {
        "run_mode": RUN_MODE,
        "run_id": RUN_ID,
        "profile": PROFILE,
        "seed": CFG.seed,
        "n_surfaceome_tested": int(len(f)),
        "n_positives_trained": int(y.sum()),
        "n_heldout_benchmarks": int(is_heldout.sum()),
        "attrition": attrition.to_dict("records"),
        "model_performance_auc": perf[perf.metric == "auc"].to_dict("records"),
        "curves": curve_res,
        "brier": brier,
        "calibration": calib,
        "architecture_prior_ablation": arch_delta,
        "collinearity_reconciliation": collinearity_note,
        "frozen_model_export": frozen_export,
        "historical_benchmark_sensitivity": hist_bench,
        "detection_floor_sensitivity": floor_sens,
        "label_class": PRIMARY_LABEL_CLASS,
        "label_class_counts": label_class_audit["counts"],
        "n_off_class_withheld": int(off_class.sum()),
        "n_historical_only_withheld": int((eval_exclude & ~is_heldout).sum()),
        "evaluation_exclude_set": list(EVALUATION_EXCLUDE),
        "heldout_benchmark_result": hb_res,
        "tier_counts_by_context": {
            k: ranked[f"priority_tier_{k}"].value_counts().to_dict()
            for k in ("primary", "mcrpc", "nepc")
            if f"priority_tier_{k}" in ranked.columns},
        "n_novel_shortlist_union_secondary": int(len(novel)),
        "shortlists_by_context": {k: int(len(v)) for k, v in shortlists.items()},
        "n_synthetic_controls_excluded": int(ranked.is_synthetic_control.sum()),
        "n_novel_T1_by_context": {
            k: int((ranked.is_novel
                    & ranked[f"priority_tier_{k}"].astype(str).str.startswith("T1")).sum())
            for k in ("primary", "mcrpc", "nepc")
            if f"priority_tier_{k}" in ranked.columns},
        "tier_vocabulary": "per-context T1/T1b/T2/T2b/T3/T4; the legacy "
                           "'tier' column is primary-adenocarcinoma scope only",
        "training_positive_median_percentile": float(
            ranked.loc[ranked.is_training_positive, "percentile"].median()),
        "survival_power": power.to_dict("records"),
        "cox_fitted": int((cox_df.status == "fitted").sum()) if len(cox_df) else 0,
        "cox_refused_low_epv": int((cox_df.status == "skipped_insufficient_EPV").sum()) if len(cox_df) else 0,
        "runtime_seconds": round(time.time() - t0, 1),
        # --- proposal-mandated provenance and transfer fields ---------------
        "label_audit": label_audit,
        "preflight_audit": {
            "n_checks": int(len(preflight)),
            "n_passed": int(preflight.passed.sum()),
            "failed": preflight.loc[~preflight.passed,
                                    ["check", "severity", "detail"]].to_dict("records"),
            "enforced_strict": RUN_MODE == "LIVE",
        },
        "benchmark_audit": benchmark_audit,
        "label_inclusion_rule": LABEL_INCLUSION_RULE,
        "f4_mode": CFG.f4_mode,
        "de_method_used": de_method_used,
        "n_passed_biological_screen": int(f.passed_biological_screen.sum()),
        "n_passed_strict_four_gate": int(f.passed_strict_four_gate.sum()),
        "ranking_stability": stab_res,
        "tier_reachability": tier_reach,
        "final_scorer": ("frozen_ensemble_all_genes" if frozen is not None
                         else "out_of_fold_fallback"),
        "oof_vs_frozen_spearman": (float(rho.statistic) if rho is not None else None),
        "fusion_sensitivity": fusion_res,
        "endpoint_name": (str(clin.endpoint_name.dropna().iloc[0])
                          if "endpoint_name" in clin.columns
                          and clin.endpoint_name.notna().any() else "unknown"),
        "eligible_primary": int(f.eligible_primary.sum()),
        "eligible_mcrpc": int(f.eligible_mcrpc.sum()),
        "eligible_both": int((f.eligible_primary & f.eligible_mcrpc).sum()),
        "surface_grade_counts": f.surface_evidence_grade.value_counts().to_dict(),
        "modality_suitable_counts": {
            m: int(f[f"suitable_{m}"].sum())
            for m in CFG.modality_risk_tissues if f"suitable_{m}" in f.columns},
        "modality_profile_counts": (f.modality_profile.value_counts().to_dict()
                                    if "modality_profile" in f.columns else {}),
        "surface_accessibility_established": int(
            f.surface_accessibility_established.sum()),
        "psma_low_analysis_performed": psma_low is not None,
        "psma_low_suitable": (int(f.suitable_for_psma_low.fillna(False).sum())
                              if "suitable_for_psma_low" in f.columns else None),
        "external_transfer": ext_res,
        "external_transfer_tested": ext_scores is not None,
        "single_cell_assessed": sc_res is not None,
        "gene_retention_pct": gene_retention_pct,
        "baseline_comparison": baselines.to_dict("records"),
        "baseline_verdict": baseline_verdict,
        "psma_rank": (float(ranked.loc[ranked.gene == CFG.benchmark_gene, "rank"].iloc[0])
                      if (ranked.gene == CFG.benchmark_gene).any() else None),
        "n_eligible_benchmarks": int(is_heldout.sum()),
        "exclusion_controls_removed_at_bounding": [
            g for g in SURFACEOME_EXCLUSION_CONTROLS if g not in set(f.gene)],
        "missing_vital_organs": missing_vital,
        "vital_organ_coverage_audit": round(organ_coverage_audit, 4),
    }

    # Diagnostics that distinguish "found nothing" from "never fitted".
    if CONVERGENCE_LOG:
        print(f"\n  {len(CONVERGENCE_LOG)} solver-convergence warning(s) "
              f"recorded. These were previously suppressed globally. Example: "
              f"{CONVERGENCE_LOG[0][:120]}")
        save_table(pd.DataFrame({"warning": CONVERGENCE_LOG}),
                   "table20_convergence_warnings.csv",
                   caption="Solver-convergence warnings, no longer silenced",
                   show=False)
    if FIT_FAILURES:
        save_table(pd.DataFrame(FIT_FAILURES), "table21_fit_failures.csv",
                   caption="Statistical fits that FAILED - distinct from fits "
                           "that returned a non-significant result", show=False)
        print(f"  {len(FIT_FAILURES)} statistical fit failure(s) recorded "
              f"separately from non-significant results")
    summary["n_convergence_warnings"] = len(CONVERGENCE_LOG)
    summary["n_fit_failures"] = len(FIT_FAILURES)

    gate = stop_go_gate(summary, f, label_audit)
    summary["stop_go"] = gate

    with open(DIRS["logs"] / "run_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)

    # Regression tests run AFTER the summary and tables are written, because
    # several of them inspect the exported artifacts rather than in-memory
    # state - the deliverable is what a reader gets, so that is what is tested.
    print("[7.2] Regression tests for this review round")
    reg = test_review_round_regressions()
    print("[7.2b] LIVE-path integration tests (miniature fixtures)")
    live = test_live_path_integration()
    summary["live_path_tests"] = {
        "n_tests": int(len(live)), "n_passed": int(live.passed.sum()),
        "failed": live[~live.passed][["test", "detail"]].to_dict("records"),
    }
    summary["regression_tests"] = {
        "n_tests": int(len(reg)),
        "n_passed": int(reg.passed.sum()),
        "failed": reg[~reg.passed][["test", "detail"]].to_dict("records"),
    }
    # FAIL-CLOSED RELEASE GATE. The scientific STOP on ensemble advancement
    # is a research RESULT and remains fully reportable; a failing test is a
    # software DEFECT and must stop the release. The two are kept distinct.
    try:
        require_all_tests_pass(reg, "Regression tests")
        require_all_tests_pass(live, "LIVE-path integration tests")
        # Tests passing moves the run to TESTED - NOT to approved. Both
        # external reviews: approval was granted here, before post-run
        # verification had run, and a failed verify() never revoked it.
        release_transition(
            "TESTED",
            f"{int(reg.passed.sum())}/{len(reg)} regression and "
            f"{int(live.passed.sum())}/{len(live)} LIVE-path tests passed; "
            f"awaiting post-run verification",
            require_from="RUNNING")
        RELEASE_APPROVAL.update(
            {"approved": True,
             "approved_for_invocation": int(
                 RELEASE_APPROVAL.get("invocation", 0)),
             "reason": (
                f"{int(reg.passed.sum())}/{len(reg)} regression and "
                f"{int(live.passed.sum())}/{len(live)} LIVE-path tests "
                f"passed"), "run_id": RUN_ID})
    except RuntimeError as _gate_err:
        RELEASE_STATE.update({"status": "BLOCKED", "verified": False,
                              "reason": str(_gate_err)})
        RELEASE_APPROVAL.update({"approved": False,
                                 "reason": str(_gate_err), "run_id": RUN_ID})
        summary.pop("release_approved", None)
        summary["release_approved_through"] = "none"
        summary["release_block_reason"] = str(_gate_err)
        with open(DIRS["logs"] / "run_summary.json", "w") as fh:
            json.dump(summary, fh, indent=2, default=str)
        raise
    # BUG FIX (both external reviews, P2, correct). This wrote
    # release_approved=True at the end of run_pipeline() - after the test
    # suites, but BEFORE post-run verify() and before packaging. An archive
    # whose run_summary.json says "release_approved": true could therefore
    # be one that never passed verification at all, and a reader has no way
    # to tell the stages apart. The internal state machine was always
    # correct; it was this field, the one a human actually reads, that
    # overstated what had happened.
    #
    # The three stages are now reported separately, and the composite field
    # is only true once packaging has verified the archive.
    summary["tests_passed"] = True
    summary["postrun_verified"] = False
    summary["release_packaged"] = False
    summary.pop("release_approved", None)
    summary["release_approved_through"] = "tests"
    summary["release_block_reason"] = (
        "test suites passed; post-run verification and packaging had not yet "
        "run when this summary was written. export_outputs() rewrites these "
        "fields once the archive has been built and audited.")
    with open(DIRS["logs"] / "run_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)
    # SELF-REVIEW: the summary is written with a bare json.dump, so it was
    # never registered - it too had been reaching the archive only via the
    # modification-time fallback. Registered here, where the final version
    # is written.
    register_artifact(DIRS["logs"] / "run_summary.json")

    # NEW: reproducibility manifest - interpreter, package versions, resolved
    # config, input-file checksums and sample inclusion lists.
    print("[7.1] Writing reproducibility manifest")
    summary["run_manifest_written"] = True
    write_run_manifest({"joint_variance_filter": jv_report,
                        "de_method_used": de_method_used})

    print("=" * 78)
    print(f"COMPLETE in {summary['runtime_seconds']}s")
    print(f"Tables : {DIRS['tables']}")
    print(f"Figures: {DIRS['figures']}")
    print(f"Summary: {DIRS['logs'] / 'run_summary.json'}")
    print("=" * 78)
    show = ["gene", "context_tier", "ensemble_score", "percentile", "rna_log2fc",
            "safety_ratio_vs_psma", "mcrpc_mean_log2", "nepc_log2fc",
            "protein_corroborated"]
    show = [c for c in show if c in ranked.columns]
    print("\nTOP 15 NOVEL CANDIDATES  (not already clinical-stage targets)")
    print("This is the discovery output. The full ranking, including known")
    print("targets and held-out benchmarks, is in table2.")
    for key, title in [("primary", "PRIMARY ADENOCARCINOMA"),
                       ("mcrpc", "METASTATIC CRPC"),
                       ("nepc", "NEUROENDOCRINE / AR-LOW")]:
        sl = shortlists.get(key)
        print(f"\n  {title} - top novel candidates")
        if sl is None or sl.empty:
            print("    none shortlisted (or context not assessable)")
        else:
            print(sl[[c for c in show if c in sl.columns]].head(8).to_string(index=False))
    if "ancestry" in clin.columns:
        print("\nANCESTRY NOTE: similar candidate expression across ancestry "
              "groups does NOT establish equal therapeutic efficacy. Report it "
              "as descriptive target fidelity with group counts and intervals.")
    print("\nHELD-OUT BENCHMARK PLACEMENT  (primary validation endpoint)")
    print(ranked.loc[ranked.is_heldout_benchmark,
                     ["gene", "percentile", "rank",
                      "priority_tier_primary"]].to_string(index=False))
    return summary, ranked, shortlists, f




# ----------------------------------------------------------------------------
# OUTPUT GALLERY AND EXPORT
# ----------------------------------------------------------------------------
# Colab's filesystem is ephemeral: outputs/ disappears when the session ends,
# and nothing in it is visible unless the reader opens the file browser. The
# gallery re-displays everything inline, and export_outputs() packages the lot
# so it can actually leave the session.
# ----------------------------------------------------------------------------
def show_gallery(figures=True, tables=True, max_table_rows=10):
    """Re-display every figure and table produced by the run."""
    if not IN_NOTEBOOK:
        print(f"Gallery: {len(FIGURE_REGISTRY)} figures, "
              f"{len(TABLE_REGISTRY)} tables in {OUT.resolve()}")
        return
    from IPython.display import display, Markdown, Image

    if figures:
        show_heading(f"All figures ({len(FIGURE_REGISTRY)})", 2)
        for item in FIGURE_REGISTRY:
            p = Path(item["path"])
            if not p.exists():
                continue
            cap = f" — {item['caption']}" if item.get("caption") else ""
            display(Markdown(f"**{item['name']}**{cap}"))
            display(Image(filename=str(p)))

    if tables:
        show_heading(f"All tables ({len(TABLE_REGISTRY)})", 2)
        rows = pd.DataFrame(TABLE_REGISTRY)[["name", "rows", "cols", "caption"]]
        display(rows.style.hide(axis="index"))
        for item in TABLE_REGISTRY:
            p = Path(item["path"])
            if not p.exists():
                continue
            df = pd.read_csv(p)
            cap = f" — {item['caption']}" if item.get("caption") else ""
            extra = (f"  _(showing {max_table_rows} of {len(df):,} rows)_"
                     if len(df) > max_table_rows else "")
            display(Markdown(f"**{item['name']}**{cap}{extra}"))
            display(df.head(max_table_rows).style.hide(axis="index"))


# Artifacts written AFTER the regression suite, because they record the
# suite's own results. They are therefore exempt from the pre-packaging
# existence and registration checks, and their presence is asserted against
# the FINISHED archive instead. The two collections are tied together by a
# regression test: nothing may be exempted here without a matching required
# member below, or it would have no check at all - which is exactly how
# run_manifest.json came to be missing from three consecutive releases.
WRITTEN_AFTER_TEST_SUITE = ("run_summary.json", "run_manifest.json")

RELEASE_REQUIRED_MEMBERS = {
    "06_logs/run_manifest.json": "the reproducibility manifest",
    "06_logs/run_summary.json": "the run summary",
    "06_logs/environment_lock.txt": "the dependency lockfile",
    "06_logs/pipeline_source/run_pipeline.py": "the standalone runner",
}


def verify_release_archive(zip_path, manifest_name="06_logs/archive_contents.csv",
                           require_release_members=True):
    """Reconcile a finished release ZIP against the manifest inside it.

    External review, confirmed against the delivered archive: five stale
    tables and fusion logs were present in the ZIP with no manifest entry -
    table15_fusion_sensitivity.csv (15 "benchmarks" when the endpoint has
    six), table19_modality_suitability.csv (superseded by
    table19_modality_expression_screen.csv), table20_convergence_warnings.csv
    (a solver warning the run summary says did not occur) and two
    pu_score_counts_fusion_* logs. A reader could cite an obsolete number as
    a current result.

    The root cause was a packaging step that zipped the output DIRECTORY
    instead of the registry-staged copy, so files from earlier runs were
    swept in. No amount of pre-packaging checking can catch that; only
    opening the delivered archive can.

    Raises on any unmanifested entry, missing entry or checksum mismatch.
    """
    import hashlib as _hl
    from zipfile import ZipFile as _ZF
    with _ZF(zip_path) as _a:
        # BUG FIX (both external reviews, independently reproduced): the
        # membership check built a SET of names, which silently collapses
        # duplicates - so a ZIP holding two members called a.csv with a
        # valid manifest passed. Duplicate entries make extraction
        # ambiguous: which of the two a reader gets depends on their tool.
        from collections import Counter as _Ctr
        _names = [i.filename for i in _a.infolist() if not i.is_dir()]
        _dupes = sorted(n for n, c in _Ctr(_names).items() if c > 1)
        if _dupes:
            raise RuntimeError(
                f"release archive {Path(zip_path).name}: duplicate "
                f"entr(ies) {_dupes[:6]}. Extraction order decides which "
                f"copy a reader receives, so the archive is ambiguous.")
        entries = set(_names)
        # SELF-REVIEW, prompted by the external finding on run_manifest.json:
        # reconciling the archive against its OWN manifest can only detect
        # disagreement between two things that are both inside the archive.
        # An artifact that was never registered is absent from both, so the
        # reconciliation passes while the release is incomplete. Membership
        # of the release-critical artifacts is therefore asserted against a
        # list declared HERE, in the verifier, not derived from the archive.
        _required_members = RELEASE_REQUIRED_MEMBERS
        # SELF-REVIEW, and this is the fifth consecutive round in which the
        # defect found was inside the guard added the round before. The
        # membership requirement above is correct for a RELEASE archive and
        # wrong for every archive this function is pointed at: the suite's
        # own miniature fixtures hold two files and no 06_logs/ tree, so the
        # new guard failed them and took the whole audit block down with it.
        # A guard must assert its own scope. require_release_members is
        # True by default, so the real call site in export_outputs() is
        # protected without having to remember to ask; the fixtures opt out
        # explicitly, and a dedicated fixture below proves that a
        # release-shaped archive missing the manifest still FAILS - so the
        # opt-out cannot quietly become the only path exercised.
        _absent = ({k: v for k, v in _required_members.items()
                    if k not in entries}
                   if require_release_members else {})
        if _absent:
            raise RuntimeError(
                f"release archive {Path(zip_path).name} is missing "
                f"release-critical artifact(s): "
                + "; ".join(f"{k} ({v})" for k, v in sorted(_absent.items()))
                + ". The manifest reconciliation cannot detect this, because "
                  "an artifact that was never registered is absent from both "
                  "the archive and its manifest.")
        if manifest_name not in entries:
            raise RuntimeError(
                f"release archive {Path(zip_path).name} has no checksum "
                f"manifest at {manifest_name}: its contents cannot be "
                f"verified, so it is not releasable.")
        man = pd.read_csv(_a.open(manifest_name))
        expected = set(man["archived_file"].astype(str))
        actual = entries - {manifest_name}
        unlisted, missing = sorted(actual - expected), sorted(expected - actual)
        if unlisted or missing:
            raise RuntimeError(
                f"release archive {Path(zip_path).name} does not match its "
                f"manifest. Unmanifested entr(ies) ({len(unlisted)}): "
                f"{unlisted[:8]}. Missing entr(ies) ({len(missing)}): "
                f"{missing[:8]}. An unmanifested file may be a stale result "
                f"from an earlier run that a reader would take for this one.")
        bad = []
        for row in man.itertuples():
            digest = _hl.sha256(_a.read(row.archived_file)).hexdigest()
            if digest != str(row.sha256):
                bad.append(row.archived_file)
        if bad:
            raise RuntimeError(
                f"release archive {Path(zip_path).name}: {len(bad)} checksum "
                f"mismatch(es), e.g. {bad[:5]}.")
    print(f"  archive verified: {len(expected)} entr(ies), all manifested, "
          f"all checksums match, no duplicates, no stale files")
    # SELF-REVIEW, caught by this round's own tests: the first version
    # transitioned the run to PACKAGED from inside this function. A
    # VERIFIER must not mutate release state - it made the audit
    # untestable on a probe archive (the run is not VERIFIED during the
    # regression suite) and coupled an integrity check to workflow
    # position. The transition belongs to the caller that is actually
    # packaging; this function only answers whether the archive is sound.
    return True


def export_outputs(download=True, to_drive=False):
    """Zip outputs/ and, on Colab, offer it as a browser download.

    Colab storage does not survive the session. Run this before you close the
    tab, or mount Drive and pass to_drive=True.
    """
    # The gate is checked HERE as well as in run_pipeline(), so a direct
    # call to export_outputs() cannot package an unapproved run. Both
    # external reviews asked for the exporter to verify release status
    # independently rather than infer it from the fact that it was called.
    # The exporter requires the run to have reached VERIFIED - i.e. post-run
    # verification completed successfully for THIS invocation. Both external
    # reviews: previously it accepted an approval granted when the test
    # suites passed, which is an earlier stage, so a failed verify() did not
    # stop packaging.
    if not release_is_packageable():
        raise RuntimeError(
            f"Release blocked: this run has not completed post-run "
            f"verification. Release state is "
            f"{RELEASE_STATE.get('status')!r} (expected 'VERIFIED'), "
            f"run_id {RELEASE_STATE.get('run_id')!r} vs current {RUN_ID!r}. "
            f"Reason: {RELEASE_STATE.get('reason', 'unknown')}. Passing the "
            f"test suites is not sufficient to package: verification runs "
            f"after them and must also succeed. Diagnostic outputs remain "
            f"on disk; they are not a validated release.")

    # Post-run verification has provably succeeded by this point -
    # release_is_packageable() is what we just checked - so the summary can
    # now record that stage truthfully. It records packaging as NOT
    # complete, because this file is about to be placed INSIDE the archive
    # and a file cannot honestly attest to the completion of the archive
    # containing it. The archive's own integrity is attested from outside,
    # by verify_release_archive() reconciling it against
    # 06_logs/archive_contents.csv after the ZIP is closed.
    try:
        _rs = DIRS["logs"] / "run_summary.json"
        if _rs.is_file():
            _sum = json.loads(_rs.read_text())
            _sum["tests_passed"] = True
            _sum["postrun_verified"] = True
            _sum["release_packaged"] = False
            # SELF-REVIEW of round 31, found by reading the SHIPPED summary
            # rather than the code: the three stage booleans were added as
            # asked, and then `release_approved: true` was left sitting
            # beside `release_packaged: false` - a self-contradiction in the
            # delivered file, and exactly the "stronger approval than has
            # occurred" the review warned about. At the moment this file is
            # written, approval is incomplete BY CONSTRUCTION: packaging and
            # the archive audit are still ahead of it.
            #
            # A boolean cannot express that honestly, in either direction -
            # true overclaims, false would libel a good release. So the
            # ambiguous field is gone and replaced by one that names the
            # furthest stage actually reached. A consumer that still reads
            # `release_approved` now gets nothing, which fails safe.
            _sum.pop("release_approved", None)
            _sum["release_approved_through"] = "postrun_verified"
            _sum["release_status"] = (
                "tests passed and post-run verification passed. Packaging is "
                "audited from outside this file by verify_release_archive(), "
                "which reconciles the finished ZIP against "
                "06_logs/archive_contents.csv; a file inside an archive "
                "cannot attest to that archive being complete.")
            _sum["release_block_reason"] = ""
            _rs.write_text(json.dumps(_sum, indent=2, default=str))
    except Exception as _e:
        raise RuntimeError(
            f"Could not update the release status in run_summary.json "
            f"({type(_e).__name__}: {_e}). Refusing to package an archive "
            f"whose summary would misreport which stages ran.") from _e

    import shutil
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
    base = f"surfaceome_outputs_{RUN_MODE}_{PROFILE}_{stamp}"

    # CRITICAL BUG FIX (external review, confirmed in the shipped archive):
    # this zipped the WHOLE outputs/ tree, which is reused across runs and
    # never cleared. The archive therefore carried artifacts from earlier
    # executions alongside the current ones, and they contradicted each
    # other. Two confirmed cases:
    #   * table15_fusion_sensitivity.csv reported "15 benchmarks, 11 in the
    #     top decile" from a previous run, while run_summary.json for THIS
    #     run correctly recorded that the fusion analysis was skipped. A
    #     reader reconciling the two would cite a result the run disowns.
    #   * table19_modality_suitability.csv (the old name) sat beside
    #     table19_modality_expression_screen.csv (the current one), with
    #     different numbers for different calculations.
    #
    # The archive is now assembled from the REGISTRIES - the artifacts this
    # execution actually produced - so a skipped analysis contributes no
    # table, and a renamed output leaves no predecessor behind.
    staging = Path(f".staging_{RUN_ID}")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    produced = set()
    for rec in TABLE_REGISTRY:
        produced.add(Path(rec["path"]).resolve())
    for rec in FIGURE_REGISTRY:
        produced.add(Path(rec["path"]).resolve())
    # BUG FIX (external review, correct - the previous fix was incomplete):
    # tables and figures were registry-filtered, but logs, models, features
    # and interim artifacts were still swept in WHOLESALE on the assumption
    # that they are run-scoped. They are not. The archive therefore still
    # carried pu_score_counts_fusion_*.csv from an earlier execution,
    # byte-for-byte identical to the previous run's files, even though this
    # run skipped the fusion analysis entirely. A reader could mistake them
    # for current evidence.
    #
    # Those directories are now filtered by MODIFICATION TIME against the
    # start of this run. A file not written or rewritten during the run did
    # not come from it and does not enter the archive.
    # P2 (external review): modification time is fragile - RUN_START_TIME is
    # set at import, so a SECOND run_pipeline() call in the same session would
    # inherit the FIRST run's artifacts. Membership now comes from the
    # explicit per-run ARTIFACT_REGISTRY, which reset_artifact_registry()
    # clears at the start of every invocation. mtime is retained only as a
    # secondary guard for files written by libraries that bypass the helpers.
    for rp in ARTIFACT_REGISTRY:
        fp = Path(rp)
        if fp.exists():
            produced.add(fp.resolve())
    # BUG FIX (both external reviews, independently reproduced): this
    # admitted any unregistered file whose mtime was at or after
    # RUN_START_TIME. RUN_START_TIME is bound once when the module loads, so
    # on a SECOND run_pipeline() call in the same session every file written
    # by the FIRST run satisfies it - reintroducing precisely the
    # stale-artifact problem the registry exists to prevent. A fallback that
    # admits unregistered files is a hole in the registry, not a safety net.
    #
    # Membership is now the registry alone. A stage that forgets to register
    # its output gets a loud failure from the mandatory-artifact check
    # rather than a quiet inclusion of something from another run.
    _unregistered = []
    for sub in ("06_logs", "03_models", "02_features", "01_interim"):
        d = OUT / sub
        if d.exists():
            for fp in d.rglob("*"):
                if fp.is_file() and fp.resolve() not in produced:
                    _unregistered.append(str(fp.relative_to(OUT)))
    if _unregistered:
        print(f"  {len(_unregistered)} unregistered file(s) NOT archived "
              f"(no modification-time fallback - an unregistered file may "
              f"belong to an earlier run in this session): "
              f"{sorted(_unregistered)[:6]}")
    # The manifest must be INSIDE the archive to be useful; writing it after
    # assembly left it out of the delivered ZIP (external review).
    import hashlib as _hl
    # Source fingerprint (external review P2): without it an archive cannot be
    # reconciled against the exact code that produced it.
    try:
        src_h = _hl.sha256()
        src_files = sorted(Path("src").glob("s0*.py")) or sorted(
            Path(".").glob("pipeline_flat.py"))
        for sf in src_files:
            src_h.update(sf.read_bytes())
        fingerprint = src_h.hexdigest()
        n_src = len(src_files)
    except Exception:
        fingerprint, n_src = "unavailable", 0
    fp_path = DIRS["logs"] / "source_fingerprint.json"
    with open(fp_path, "w") as fh:
        json.dump({"run_id": RUN_ID, "n_source_files": n_src,
                   "source_sha256": fingerprint,
                   "note": ("SHA-256 over the concatenated pipeline sources, "
                            "so results can be reconciled against the exact "
                            "implementation that produced them")}, fh, indent=1)
    produced.add(fp_path.resolve())
    print(f"  source fingerprint: {fingerprint[:16]}... ({n_src} file(s))")

    # NEW (both external reviews, P2 but real): the archive carried results
    # and audit tables but no importable source and no dependency lockfile,
    # so an independent analyst could verify the OUTPUTS without being able
    # to re-run the ANALYSIS. The release package now carries the pipeline
    # source and the exact resolved versions of every library the results
    # depend on. This is reproducibility packaging, not CI - a clean-room
    # rerun on pinned versions still has to be performed by the user.
    try:
        _env_path = DIRS["logs"] / "environment_lock.txt"
        # BUG FIX (both external reviews, correct on both counts): the
        # lockfile I wrote last round was NOT INSTALLABLE. It recorded
        # "sklearn==1.9.1" - sklearn is the import name, the distribution is
        # scikit-learn - and "pydeseq2==NOT INSTALLED in this run", which is
        # not valid requirement syntax at all. A file that looks like a lock
        # and cannot be installed is worse than none, because it will be
        # trusted. Import names are mapped to distribution names, absent
        # optional packages are recorded in a separate manifest rather than
        # as broken requirements, and the result is checked parseable.
        # BUG FIX (external review, correct): the pin list was a short
        # hand-maintained set and omitted seaborn and requests, both of
        # which the pipeline imports UNCONDITIONALLY - so a fresh
        # environment built from the lockfile would resolve whatever
        # versions pip happened to pick for them, and their versions
        # appeared only in run_manifest.json. A lockfile that silently
        # omits a hard dependency is a reproducibility claim it cannot keep.
        #
        # The list is now derived from the modules the pipeline has actually
        # imported, intersected with a declared distribution-name map, so a
        # new import is pinned rather than forgotten.
        _DIST_NAME = {"sklearn": "scikit-learn", "numpy": "numpy",
                      "pandas": "pandas", "scipy": "scipy",
                      "matplotlib": "matplotlib", "seaborn": "seaborn",
                      "requests": "requests",
                      "statsmodels": "statsmodels",
                      "lifelines": "lifelines", "joblib": "joblib",
                      "pydeseq2": "pydeseq2"}
        # Any third-party top-level module already imported by this run that
        # the map does not cover is reported, so the omission cannot be
        # silent the way seaborn and requests were.
        _stdlib = set(getattr(sys, "stdlib_module_names", set()))
        _loaded = {m.split(".")[0] for m in list(sys.modules)
                   if not m.startswith("_")}
        _third_party = {m for m in _loaded
                        if m not in _stdlib and m.isidentifier()
                        and getattr(sys.modules.get(m), "__file__", None)
                        and "site-packages" in str(
                            getattr(sys.modules.get(m), "__file__", ""))}
        _unmapped = sorted(_third_party - set(_DIST_NAME))
        if _unmapped:
            print(f"  NOTE: {len(_unmapped)} imported third-party module(s) "
                  f"are not in the pin map and are therefore NOT pinned: "
                  f"{_unmapped[:12]}. Add them to _DIST_NAME if the results "
                  f"depend on their versions.")
        _pins, _absent = [], []
        for _m, _dist in _DIST_NAME.items():
            try:
                _mod = __import__(_m)
                _v = getattr(_mod, "__version__", None)
                if _v:
                    _pins.append(f"{_dist}=={_v}")
                else:
                    _absent.append(f"{_dist}: installed, version "
                                   f"not introspectable")
            except Exception:
                _absent.append(f"{_dist}: not installed in this run "
                               f"(optional for the DEMO profile)")
        _req = "\n".join(sorted(_pins))
        _env_path.write_text(
            f"# Installable requirements for run {RUN_ID}\n"
            f"# python_requires == {sys.version.split()[0]}\n"
            f"# pip install -r environment_lock.txt\n"
            f"{_req}\n")
        # Every line must be a valid requirement: name plus pinned version,
        # no prose. Asserted here so an unusable lock cannot ship again.
        import re as _re2
        _bad_lines = [ln for ln in _req.splitlines()
                      if not _re2.fullmatch(
                          r"[A-Za-z0-9][A-Za-z0-9._-]*==[0-9][^\s]*", ln)]
        if _bad_lines:
            raise AssertionError(
                f"environment_lock.txt contains non-installable line(s): "
                f"{_bad_lines}")
        if _absent:
            _abs_path = DIRS["logs"] / "environment_absent_packages.txt"
            _abs_path.write_text(
                f"# Packages NOT pinned for run {RUN_ID}, with the reason.\n"
                f"# Recorded here rather than in the lockfile, where they "
                f"would make it uninstallable.\n"
                + "\n".join(f"# {a}" for a in sorted(_absent)) + "\n")
            produced.add(_abs_path.resolve())
        produced.add(_env_path.resolve())
        _src_dir = DIRS["logs"] / "pipeline_source"
        _src_dir.mkdir(parents=True, exist_ok=True)
        _n_copied = 0
        for _sf in src_files:
            try:
                (_src_dir / Path(_sf).name).write_bytes(Path(_sf).read_bytes())
                produced.add((_src_dir / Path(_sf).name).resolve())
                _n_copied += 1
            except Exception:
                pass
        # Ship a runnable entry point with the source. Verified by
        # extracting a delivered archive and running it: the seven modules
        # are SECTIONS of one flat program, not importable packages, so
        # `python s07_main.py` fails on names defined in earlier sections.
        # The flat module must also be written to DISK before exec, or
        # inspect.getsource() cannot find it and ~40 source-inspecting
        # tests fail spuriously - an independent analyst following the
        # obvious path saw 149/191 and a blocked release. With this runner
        # the archived source reproduces 236/236 and 47/47.
        _runner = _src_dir / "run_pipeline.py"
        _runner.write_text(
            '"""Standalone runner for the archived pipeline source.\n\n'
            'Usage:   python run_pipeline.py\n'
            'Env:     RUN_MODE=DEMO|LIVE\n'
            '         PROFILE=smoke|standard|manuscript\n'
            '         RUN_PURPOSE=analysis|data_only\n\n'
            'PROFILE is left UNSET by default so the pipeline chooses:\n'
            'smoke for DEMO, standard for LIVE. There is no "full"\n'
            'profile; an earlier version of this usage line named one.\n\n'
            'The seven s0*.py files are SECTIONS of one program and are\n'
            'concatenated, not imported. The flat module is written to disk\n'
            'first so inspect.getsource() can find it.\n"""\n'
            'import os, pathlib\n'
            'os.environ.setdefault("RUN_MODE", "DEMO")\n'
            '# PROFILE is deliberately NOT defaulted here. Hardcoding\n'
            '# "smoke" would override the pipeline\'s own correct default\n'
            '# and give a LIVE run 3 PU bags and 30 permutations.\n'
            'HERE = pathlib.Path(__file__).parent\n'
            'SECTIONS = ["s01_setup.py", "s02_data.py", "s03_features.py",\n'
            '            "s04_ml.py", "s05_survival.py", "s06_figures.py",\n'
            '            "s07_main.py"]\n'
            'src = "".join((HERE / f).read_text() + "\\n\\n"'
            ' for f in SECTIONS)\n'
            'src = src.replace(\'if __name__ == "__main__":\\n'
            '    summary, ranked, shortlists, feat = run_pipeline()\', "")\n'
            'flat = HERE / "pipeline_flat.py"\n'
            'flat.write_text(src)\n'
            'ns = {"__name__": "surfaceome_flat"}\n'
            'exec(compile(src, str(flat), "exec"), ns)\n'
            '# This calls the SHARED controller, not run_pipeline(): a\n'
            '# script run must pass the same release gate the notebook\n'
            '# applies - analysis, post-run verify(), packaging and the\n'
            '# archive audit - or the two paths look alike and carry\n'
            '# different guarantees.\n'
            'ns["execute_validated_analysis"](download_archive=False)\n')
        # SELF-CHECK, added because I shipped exactly this bug: an earlier
        # version of the comment block above was written with escaped
        # newlines, so the entire tail of the file - including the call -
        # became ONE comment line containing literal backslash-n. The
        # generated runner parsed cleanly, imported every definition, and
        # called NOTHING. It exited 0 with no output and no outputs. A
        # generated script that silently does nothing is the worst possible
        # failure, because success and total inaction are indistinguishable
        # from the exit code. The file is parsed and its top-level calls are
        # asserted before it is allowed into the archive.
        try:
            import ast as _rast
            _rt = _rast.parse(_runner.read_text())
            _top_calls = {
                getattr(_n.value.func, "id", None)
                or getattr(_n.value.func, "attr", None)
                or (_n.value.func.value.id
                    if isinstance(_n.value.func, _rast.Subscript) else None)
                for _n in _rt.body
                if isinstance(_n, _rast.Expr) and isinstance(_n.value, _rast.Call)
            }
            _invokes = any(
                isinstance(_n, _rast.Expr) and isinstance(_n.value, _rast.Call)
                and "execute_validated_analysis" in _rast.dump(_n.value)
                for _n in _rt.body)
            if not _invokes:
                raise RuntimeError(
                    f"The generated standalone runner {_runner.name} does "
                    f"not CALL anything at top level: it would define every "
                    f"function and exit 0 without running the pipeline. "
                    f"Top-level calls found: {sorted(c for c in _top_calls if c)}. "
                    f"Refusing to package a runner that silently does "
                    f"nothing.")
        except SyntaxError as _se:
            raise RuntimeError(
                f"The generated standalone runner {_runner.name} does not "
                f"parse ({_se}). It would fail for anyone who extracts the "
                f"archive.") from _se
        produced.add(_runner.resolve())
        _n_copied += 1
        print(f"  release package: {len(_pins)} pinned dependency "
              f"version(s), {_n_copied} source file(s) archived "
              f"(including run_pipeline.py)")
    except Exception as _e:
        # BUG FIX (external review, correct): this caught every packaging
        # failure and printed a warning, so a release could ship with no
        # lockfile and no source snapshot while the run reported success. A
        # required release artifact that fails to build must block the
        # release, not annotate it. Data-analysis outputs are already
        # written at this point; what fails here is the reproducibility
        # package, and that is exactly what a reader needs to check the
        # rest.
        raise RuntimeError(
            f"release packaging FAILED: {_e!r}. The dependency lock and "
            f"source snapshot are required release artifacts - without them "
            f"the archive cannot be independently reproduced, so the run is "
            f"not releasable. Fix the cause rather than shipping the "
            f"outputs alone.") from _e
    man_rows = []
    for fpr in sorted(produced):
        fpp = Path(fpr)
        if not fpp.exists():
            continue
        h = _hl.sha256()
        with open(fpp, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        man_rows.append({"archived_file": str(fpp.relative_to(OUT.resolve())),
                         "bytes": fpp.stat().st_size,
                         "sha256": h.hexdigest()})
    man_df = pd.DataFrame(man_rows)
    man_df.insert(0, "run_id", RUN_ID)
    # The manifest cannot list its own checksum (it does not exist until it
    # is written), so it is the single expected unlisted entry. Every other
    # file in the archive is checksummed.
    man_path = DIRS["logs"] / "archive_contents.csv"
    man_df.to_csv(man_path, index=False)
    produced.add(man_path.resolve())

    n_copied, skipped_stale = 0, []
    for sub in sorted(p.name for p in OUT.iterdir() if p.is_dir()):
        d = OUT / sub
        for fp in sorted(d.rglob("*")):
            if not fp.is_file():
                continue
            if fp.resolve() in produced:
                dest = staging / fp.relative_to(OUT)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(fp, dest)
                n_copied += 1
            else:
                skipped_stale.append(str(fp.relative_to(OUT)))
    archive = shutil.make_archive(base, "zip", root_dir=str(staging))
    shutil.rmtree(staging)
    # NEW (external review, correct and release-blocking): every integrity
    # claim here was made about the state of the code BEFORE packaging - the
    # registry, the staging copy, the skipped-stale list. None of it
    # inspected the finished ZIP, which is the only artifact a reader
    # receives. The audit now opens the archive that was just written and
    # reconciles its entries against the checksum manifest inside it, so an
    # unmanifested or stale entry fails the run instead of shipping beside a
    # passing test.
    verify_release_archive(archive)
    release_transition("PACKAGED",
                       f"archive {Path(archive).name} verified",
                       require_from="VERIFIED")
    if skipped_stale:
        print(f"  EXCLUDED {len(skipped_stale)} stale artifact(s) not produced "
              f"by run {RUN_ID}:")
        for sp in sorted(skipped_stale)[:12]:
            print(f"    - {sp}")
        if len(skipped_stale) > 12:
            print(f"    ... and {len(skipped_stale) - 12} more")
        print("  These remain on disk but are NOT in the archive: they belong "
              "to an earlier execution and would contradict this run.")

    size_mb = os.path.getsize(archive) / 1e6
    print(f"Packaged {archive} ({size_mb:.1f} MB)")
    print(f"  {len(FIGURE_REGISTRY)} figures, {len(TABLE_REGISTRY)} tables, "
          f"plus run_summary.json")

    if to_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            dest = Path("/content/drive/MyDrive/surfaceome_runs")
            dest.mkdir(parents=True, exist_ok=True)
            shutil.copy(archive, dest / Path(archive).name)
            print(f"  copied to {dest / Path(archive).name}")
        except Exception as e:
            print(f"  Drive copy skipped: {e}")

    if download and IN_COLAB:
        try:
            from google.colab import files
            files.download(archive)
            print("  browser download started")
        except Exception as e:
            print(f"  automatic download unavailable ({e}). "
                  f"Use the Files pane on the left to download {archive}.")
    elif download and not IN_COLAB:
        print(f"  not running on Colab; the archive is at {Path(archive).resolve()}")
    return archive


# ----------------------------------------------------------------------------
# TIER REACHABILITY TEST
# ----------------------------------------------------------------------------
# RAISED IN EXTERNAL REVIEW: a tier branch can be unreachable if an earlier
# condition already covers its cases, in which case the tier exists in the
# documentation but never in the output - and nobody notices, because an empty
# tier looks like "no candidates qualified" rather than "this rule is dead".
# This enumerates the boolean combinations and reports which tiers any input
# could actually produce.
# ----------------------------------------------------------------------------
# RELEASE STATE MACHINE (both external reviews, as the architectural fix
# rather than another isolated safeguard). The previous design had a single
# `approved` boolean set by run_pipeline() once the two test suites passed -
# so approval existed BEFORE post-run verification had run, and a failed
# verify() left it standing. Each safeguard protected its own stage; none
# enforced one release contract.
#
# A run now progresses CREATED -> RUNNING -> TESTED -> VERIFIED -> PACKAGED,
# and any failed required check moves it to BLOCKED. Approval is not a flag
# anyone can inherit or set early: it is the state the run has actually
# reached. The exporter requires VERIFIED, so tests passing is no longer
# sufficient to package.
RELEASE_STATES = ("CREATED", "RUNNING", "TESTED", "VERIFIED", "PACKAGED",
                  "BLOCKED")

RELEASE_STATE = {"status": "CREATED", "invocation": 0, "run_id": None,
                 "reason": "no run has started", "verified": False}


def release_transition(status, reason="", require_from=None):
    """Move the run to `status`, refusing an out-of-order transition.

    require_from names the state(s) the run must currently be in. A
    transition from the wrong state is itself a release defect - it means a
    stage ran without its prerequisite - so it raises rather than being
    silently accepted.
    """
    if status not in RELEASE_STATES:
        raise ValueError(f"unknown release state {status!r}")
    cur = RELEASE_STATE.get("status")
    if require_from is not None:
        allowed = ((require_from,) if isinstance(require_from, str)
                   else tuple(require_from))
        if cur not in allowed:
            RELEASE_STATE.update(
                {"status": "BLOCKED", "verified": False,
                 "reason": (f"out-of-order release transition: tried to "
                            f"enter {status} from {cur}, expected one of "
                            f"{allowed}")})
            raise RuntimeError(RELEASE_STATE["reason"])
    RELEASE_STATE.update({"status": status, "reason": reason,
                          "verified": status in ("VERIFIED", "PACKAGED")})
    return RELEASE_STATE["status"]


def release_is_packageable():
    """True only when this invocation has completed post-run verification."""
    return (RELEASE_STATE.get("status") == "VERIFIED"
            and RELEASE_STATE.get("verified") is True
            and RELEASE_STATE.get("run_id") == RUN_ID)


def mark_verified(ok, detail=""):
    """Record the outcome of post-run verification. Called by verify()."""
    if ok:
        return release_transition("VERIFIED", detail or "post-run "
                                  "verification passed", require_from="TESTED")
    RELEASE_STATE.update({"status": "BLOCKED", "verified": False,
                          "reason": detail or "post-run verification failed"})
    return "BLOCKED"


# Retained for the regression tests that assert the older contract; it is a
# VIEW of the state machine, never an independent source of truth.
RELEASE_APPROVAL = {"approved": False, "reason": "not yet evaluated"}


def require_output(path, why="a mandatory analysis output"):
    """Fail when a required artifact is absent or empty.

    Both external reviews, correct: the suites are full of
    `if some_table.exists():` guards with no else, so a missing required
    output makes its assertions vanish silently. That is the same defect as
    the test I wrote against a filename that never existed - it reported
    nothing and looked like a pass. A guard that can only skip is not a
    guard.

    Optional analyses record an explicit SKIPPED status instead of
    disappearing.
    """
    path = Path(path)
    if not path.is_file():
        raise AssertionError(
            f"Required output missing: {path.name} ({why}). Its checks "
            f"cannot run, and a check that does not run is not a check that "
            f"passed.")
    if path.stat().st_size == 0:
        raise AssertionError(
            f"Required output is empty: {path.name} ({why}).")
    return path


def require_all_tests_pass(test_df, suite_name):
    """Fail closed when a test suite did not fully pass.

    Both external reviews, independently, as the top P0: the pipeline ran
    both suites, recorded the pass/fail counts in run_summary.json, printed
    COMPLETE, and packaged an archive regardless. The checksum machinery
    guarantees the delivered files match the manifest; it says nothing about
    whether the code that produced them passed its tests. A perfectly
    manifested archive of results from defective code is exactly the failure
    mode the manifest cannot catch.

    An empty suite is a failure too: 'no tests executed' must never read as
    'nothing failed'.
    """
    if test_df is None or len(test_df) == 0:
        raise RuntimeError(
            f"Release blocked - {suite_name}: no tests executed. An empty "
            f"suite is not a passing suite.")
    if not {"test", "passed"}.issubset(set(test_df.columns)):
        raise RuntimeError(
            f"Release blocked - {suite_name}: unrecognised result schema "
            f"(columns {list(test_df.columns)}); pass/fail cannot be "
            f"determined, so the run cannot be approved.")
    # BUG FIX (both external reviews, independently reproduced): .astype(bool)
    # on an object column makes any NON-EMPTY STRING truthy, so a row with
    # passed="False" was read as a PASS and the gate approved the release. A
    # release gate must not infer truth from a type it did not expect.
    _pv = test_df["passed"]
    _bad_type = [repr(v) for v in _pv if not isinstance(v, (bool, np.bool_))]
    if _bad_type:
        raise TypeError(
            f"Release blocked - {suite_name}: the 'passed' column must hold "
            f"real booleans, found {len(_bad_type)} other value(s): "
            f"{_bad_type[:5]}. A string such as 'False' is truthy in "
            f"Python and would be counted as a passing test.")
    failed = test_df.loc[_pv.eq(False)]
    if len(failed):
        raise RuntimeError(
            f"Release blocked - {suite_name}: {len(failed)} of "
            f"{len(test_df)} test(s) failed: "
            + "; ".join(failed["test"].astype(str).head(8))
            + (" ..." if len(failed) > 8 else "")
            + ". The test reports are retained for diagnosis, but this run "
              "is NOT a validated release.")
    return True


def test_tier_reachability(verbose=True):
    import itertools
    rows = []
    for bio, pd_, pc, rg, top in itertools.product([False, True], repeat=5):
        if pc and not pd_:
            continue      # cannot corroborate a protein that was not measured
        tier = np.select(
            [bio and pc and rg and top,
             bio and pc and top,
             bio and (not pd_) and rg and top,
             bio and (not pd_) and top,
             bio and pd_ and (not pc) and top,
             bio],
            ["Tier 1", "Tier 1b", "Tier 2", "Tier 2b", "Tier 3", "Tier 4"],
            default="Not prioritised")
        rows.append({"bio": bio, "protein_detected": pd_,
                     "protein_corroborated": pc, "reagent": rg,
                     "top_decile": top, "tier": str(tier)})
    df = pd.DataFrame(rows)
    reached = sorted(set(df.tier))
    declared = ["Tier 1", "Tier 1b", "Tier 2", "Tier 2b", "Tier 3", "Tier 4",
                "Not prioritised"]
    unreachable = [t for t in declared if t not in reached]
    if verbose:
        print(f"    tier reachability: {len(reached)}/{len(declared)} declared "
              f"tiers are reachable")
        if unreachable:
            print(f"      UNREACHABLE (dead rules, remove or fix): {unreachable}")
        else:
            print("      every declared tier is reachable from some input")
    save_table(df, "table18_tier_reachability.csv",
               caption="Enumerated tier assignments - confirms no declared "
                       "tier is a dead rule", show=False)
    return {"reachable": reached, "unreachable": unreachable}


# ============================================================================
# 7.3 REGRESSION TESTS FOR THIS REVIEW ROUND (NEW)
# ============================================================================
# External review, correctly: "add automated unit tests for all the above
# behaviours before a LIVE run". Every defect fixed in this round was a
# SCIENTIFIC LOGIC error that ran to completion and produced attractive
# output - none of them raised an exception. A passing execution is therefore
# no evidence that they are fixed, so each one gets a test that fails if the
# defect returns. Three of them are invisible in DEMO data, so the tests
# construct the adversarial input directly rather than relying on a run.
def _raises(fn, exc=Exception):
    """True when fn() raises `exc`. Keeps assertion sites to one line.

    SELF-REVIEW: the first version swallowed EVERY other exception and
    returned False, so a broken test - a NameError from a variable scoped
    to another function, a typo in an attribute - reported a clean,
    plausible-looking assertion failure instead of an error. That happened
    twice in one round here, and both times the message read as though the
    code under test had misbehaved. Errors that indicate the TEST is wrong
    are now re-raised, so they surface as what they are.
    """
    _test_is_broken = (NameError, AttributeError, ImportError, SyntaxError)
    try:
        fn()
        return False
    except exc:
        return True
    except _test_is_broken:
        raise
    except Exception:
        return False


def test_review_round_regressions(verbose=True):
    results = []

    def check(name, passed, detail=""):
        results.append({"test": name, "passed": bool(passed), "detail": detail})

    # ----------------------------------------------------------------
    # ARTIFACT-INTEGRITY TESTS - run FIRST, against the files on disk.
    # ----------------------------------------------------------------
    # These exist because the previous build's regression suite silently
    # overwrote three publication artifacts with test fixtures while
    # reporting 20/20 passed. A test suite that cannot detect its own
    # contamination is worse than no suite: it converts a corrupted
    # archive into a certified one. These checks reconcile what is on
    # disk against the run summary and refuse to pass if any fixture
    # identifier appears in a scientific table.
    FIXTURE_IDS = {"SURFACE_OK", "INTRACELLULAR", "TUMOUR_ONLY", "VARIABLE",
                   "FLAT_BOTH", "G1", "G2"} | {f"G{i}" for i in range(40)}
    try:
        summ_p = DIRS["logs"] / "run_summary.json"
        summ = json.load(open(summ_p)) if summ_p.exists() else {}
        n_tested = summ.get("n_surfaceome_tested")

        rank_p = DIRS["tables"] / "table2_full_candidate_ranking.csv"
        feat_p = DIRS["features"] / "feature_table_with_filters.csv"
        attr_p = DIRS["tables"] / "table1_filter_attrition.csv"

        if rank_p.exists() and n_tested:
            n_rank = len(pd.read_csv(rank_p))
            check("Candidate ranking matches the analysed gene count",
                  n_rank == n_tested,
                  f"summary {n_tested:,}, table2 {n_rank:,}")

        if feat_p.exists() and n_tested:
            feat = pd.read_csv(feat_p)
            check("Filtered feature table matches the analysed gene count",
                  len(feat) == n_tested,
                  f"summary {n_tested:,}, feature table {len(feat):,}")
            leaked = sorted(set(feat.get("gene", pd.Series(dtype=str)))
                            & FIXTURE_IDS)
            check("No test fixture leaked into the feature table",
                  not leaked, f"leaked: {leaked}" if leaked else "clean")

        if attr_p.exists() and n_tested:
            attr = pd.read_csv(attr_p)
            n0 = int(attr.n.iloc[0]) if "n" in attr.columns and len(attr) else -1
            check("Filter attrition starts from the analysed gene count",
                  n0 == n_tested, f"summary {n_tested:,}, table1 first row {n0:,}")

        if rank_p.exists() and feat_p.exists():
            r = set(pd.read_csv(rank_p).gene)
            fe = set(pd.read_csv(feat_p).gene)
            check("Ranking and feature table describe the same genes",
                  r == fe,
                  f"ranking-only {len(r-fe)}, feature-only {len(fe-r)}")

        audit_p = DIRS["tables"] / "table16_preflight_label_audit.csv"
        if audit_p.exists():
            a = pd.read_csv(audit_p)
            det = " ".join(a.get("detail", pd.Series(dtype=str))
                           .astype(str).tolist())
            check("Preflight audit describes the real label set, not a fixture",
                  ("x1" not in det) and ("'x2'" not in det),
                  "fixture feature names x1/x2 absent"
                  if "x1" not in det else "FIXTURE AUDIT ON DISK")
    except Exception as e:
        check("artifact-integrity checks executed", False, repr(e))

    # 0. INVARIANT: every analysis that fits, scores or summarises the model
    #    must use the SAME population the model was trained on. This test
    #    exists because the off-class exclusion was applied to the training
    #    calls and silently missed by the null, the importance analysis, the
    #    stability analysis, the fusion test and the ROC/PR figures - turning
    #    17 known clinical-stage drug targets into pseudo-negatives in all of
    #    them. The call sites are checked by reading this module's own source,
    #    so a new call site added later with the wrong mask fails here.
    try:
        import inspect as _isp
        src_main = _isp.getsource(run_pipeline)
        MODEL_FNS = ["permutation_null_auc", "feature_importance",
                     "ranking_stability", "fusion_sensitivity_model",
                     "fig_roc_pr", "fig_calibration", "pu_bagged_nested_cv",
                     "fit_frozen_model", "preflight_label_audit"]
        BENCH_FNS = ["evaluate_heldout_benchmarks"]
        offenders = []
        for line in src_main.splitlines():
            for fn in MODEL_FNS:
                if fn + "(" in line and "is_heldout" in line \
                        and "train_exclude" not in line:
                    offenders.append((fn, line.strip()[:70]))
        check("Model analyses all use the trained-on population "
              "(train_exclude), not the benchmark mask",
              not offenders,
              f"offending call sites: {offenders}" if offenders
              else f"{len(MODEL_FNS)} model functions checked")
        bench_ok = all(("is_heldout" in line)
                       for fn in BENCH_FNS
                       for line in src_main.splitlines() if fn + "(" in line)
        check("Benchmark evaluation still uses the benchmark mask only",
              bench_ok, "evaluate_heldout_benchmarks keeps is_heldout")
    except Exception as e:
        check("train/eval population invariant checked", False, repr(e))

    # 0a-bis. MASK SEMANTICS, not call-site text. External review, correctly:
    # the source-text test guarded against passing the wrong variable but did
    # not check what each function DOES with the mask - which is why 32 tests
    # passed while the fusion analysis counted 23 genes as "benchmarks" when
    # there are six. These assertions check the meaning.
    try:
        _genes2 = ((set(CLINICAL_STAGE_TARGETS) - {"FOLH1_HELD"})
                   | set(PROSTATE_HELDOUT_POSITIVES)
                   | {"FILLER1", "FILLER2"})
        _f2 = pd.DataFrame({"gene": sorted(_genes2)})
        _y2, _hb2 = make_labels(_f2, verbose=False)
        _off2 = off_class_mask(_f2)
        _tx2 = _hb2 | _off2
        check("Benchmark mask counts ONLY the prespecified benchmarks",
              int(_hb2.sum()) == len(
                  set(PROSTATE_HELDOUT_POSITIVES) & set(_f2.gene)),
              f"benchmark mask = {int(_hb2.sum())}, declared = "
              f"{len(set(PROSTATE_HELDOUT_POSITIVES) & set(_f2.gene))}")
        check("Training exclusions are a strict superset of the benchmarks",
              bool(np.all(_hb2 <= _tx2)) and int(_tx2.sum()) >= int(_hb2.sum()),
              f"train_exclude = {int(_tx2.sum())}, benchmarks = "
              f"{int(_hb2.sum())}, off-class = {int(_off2.sum())}")
        check("Benchmarks and off-class exclusions do not overlap",
              int((_hb2 & _off2).sum()) == 0,
              f"overlap = {int((_hb2 & _off2).sum())}")
    except Exception as e:
        check("mask-semantics checks executed", False, repr(e))

    # 0a-ter. The two functions that previously conflated the masks must now
    # report a benchmark count equal to the TRUE benchmark count, not the
    # training-exclusion count. Checked on the real run's summary.
    try:
        summ2 = json.load(open(DIRS["logs"] / "run_summary.json"))
        n_true_bench = summ2.get("n_heldout_benchmarks")
        fus = summ2.get("fusion_sensitivity") or {}
        if fus and n_true_bench:
            n_eval = fus.get("n_benchmarks_evaluated")
            check("Fusion analysis evaluates the true benchmark set, not the "
                  "training exclusions",
                  n_eval is not None and n_eval <= n_true_bench,
                  f"fusion evaluated {n_eval} benchmark(s); true benchmark "
                  f"set is {n_true_bench} (a complete-evidence subset may be "
                  f"smaller, never larger)")
            check("Fusion analysis received an explicit benchmark mask",
                  bool(fus.get("benchmark_mask_supplied")),
                  "benchmark_mask passed separately from train_exclude")
        # Read the preflight TABLE, not just its failures: the
        # benchmark-availability check now passes, so its detail string is
        # absent from `failed`. The previous assertion looked only at
        # failures and so could never see the corrected text.
        pf_p = DIRS["tables"] / "table16_preflight_label_audit.csv"
        if pf_p.exists():
            pf_t = pd.read_csv(pf_p)
            row = pf_t[pf_t.check.astype(str).str.contains(
                "Benchmarks available", case=False, na=False)]
            det = str(row.detail.iloc[0]) if len(row) else ""
            n_claimed = summ2.get("n_heldout_benchmarks")
            ok = ("withheld from fitting only" in det
                  and (n_claimed is None
                       or det.strip().startswith(str(n_claimed))))
            check("Preflight reports benchmarks separately from training "
                  "exclusions", ok,
                  f"detail = {det[:90]!r}")
    except Exception as e:
        check("reported-benchmark-count checks executed", False, repr(e))

    # 0a-quater. Static keyword-argument check across the whole pipeline.
    # A LIVE-only branch called detect_expression_scale(declared=...) when the
    # parameter is `assume=`; DEMO never reaches it, so no test could catch it.
    # This scans every call to a pipeline function for unknown keywords.
    try:
        import ast as _ast
        import inspect as _inspect
        import sys as _sys
        _mod_src = None
        for _cand in ("pipeline_flat.py",):
            if Path(_cand).exists():
                _mod_src = Path(_cand).read_text()
                break
        if _mod_src is None:
            _mod_src = "".join(
                _inspect.getsource(_o) + "\n"
                for _o in list(globals().values())
                if callable(_o) and getattr(_o, "__module__", None)
                == __name__ and hasattr(_o, "__code__"))
        _tree = _ast.parse(_mod_src)
        _sigs = {}
        for _n in _ast.walk(_tree):
            if isinstance(_n, _ast.FunctionDef):
                _a = _n.args
                _sigs[_n.name] = (
                    {x.arg for x in _a.args} | {x.arg for x in _a.kwonlyargs},
                    _a.kwarg is not None)
        _bad = []
        for _n in _ast.walk(_tree):
            if (isinstance(_n, _ast.Call) and isinstance(_n.func, _ast.Name)
                    and _n.func.id in _sigs):
                _params, _has_kw = _sigs[_n.func.id]
                if _has_kw:
                    continue
                for _kw in _n.keywords:
                    if _kw.arg and _kw.arg not in _params:
                        _bad.append(f"{_n.func.id}({_kw.arg}=...) line "
                                    f"{getattr(_n, 'lineno', '?')}")
        check("No call passes an unknown keyword argument (catches LIVE-only "
              "branches no test can reach)",
              not _bad, f"mismatches: {_bad}" if _bad
              else f"{len(_sigs)} function signatures checked")
    except Exception as e:
        check("keyword-argument scan executed", False, repr(e))

    # 0c. ARTIFACT/SUMMARY CONSISTENCY: an analysis the run records as
    # SKIPPED must not have a results table in the exported archive. The
    # previous archive shipped table15_fusion_sensitivity.csv reporting "15
    # benchmarks, 11 in the top decile" from an earlier run while the summary
    # said the fusion analysis was skipped - a self-contradictory deliverable
    # that 37 passing tests did not notice.
    try:
        summ3 = json.load(open(DIRS["logs"] / "run_summary.json"))
        SKIP_MAP = {
            "fusion_sensitivity": "table15_fusion_sensitivity.csv",
            "psma_low_analysis_performed": "table17_psma_low_persistence.csv",
        }
        # TABLE_REGISTRY is the authoritative record of what THIS run
        # produced and is populated by the time the tests execute, whereas
        # archive_contents.csv is written later at export. The registry is
        # therefore what the assertion reads; the export stage copies from
        # the same registry, so a pass here guarantees the archive too.
        reg_names = {Path(r["path"]).name for r in TABLE_REGISTRY}
        for key, tbl in SKIP_MAP.items():
            val = summ3.get(key)
            skipped = (val is None) or (val is False)
            if skipped:
                check(f"Skipped analysis produced no result table ({tbl})",
                      tbl not in reg_names,
                      f"{key} = {val!r}; {tbl} "
                      f"{'WAS' if tbl in reg_names else 'was not'} registered "
                      f"by this run")
        # Any table sitting on disk that this run did not register is STALE
        # and must not reach the archive. Reported here so it is visible at
        # test time, not only at export.
        on_disk = {p.name for p in DIRS["tables"].glob("table*.csv")}
        # SELF-REVIEW: the first real version of this check flagged
        # table21_regression_tests.csv and table29_live_path_integration_
        # tests.csv as stale. They are not - they are THIS suite's own
        # result tables, written and registered after this check runs, so
        # the check was reading a snapshot that could not yet contain them.
        # A test that fires on its own output is a false alarm, and a false
        # alarm is how a real one gets ignored.
        _written_after_this_check = {"table21_regression_tests.csv",
                                     "table29_live_path_integration_tests.csv"}
        stale = sorted(on_disk - reg_names - _written_after_this_check)
        # BUG FIX (external review, correct and release-blocking): this
        # check was hardcoded to True. It computed `stale`, printed it, and
        # passed unconditionally - so it could never fail, and it is exactly
        # how an archive containing five unmanifested stale files was
        # delivered alongside a passing test asserting their exclusion. A
        # test with a literal True is not a test; it is a label.
        #
        # It now asserts the real condition. Stale files on disk are a
        # hazard even though the registry-based export excludes them,
        # because any packaging step that does not consult the registry -
        # including a hand-rolled `zip -r` over the output directory - will
        # pick them up. The authoritative post-packaging audit is
        # verify_release_archive(), which inspects the finished ZIP.
        check("No stale table from an earlier run is present on disk",
              not stale,
              f"{len(stale)} stale file(s) that this run did not register: "
              f"{stale[:6]}. The registry-based export excludes them, but "
              f"their presence means any packaging step that zips the "
              f"output directory directly will ship them."
              if stale else
              f"{len(reg_names)} table(s) registered this run, none stale")
    except Exception as e:
        check("artifact/summary consistency checks executed", False, repr(e))

    # 0d. Known clinical-stage targets must not appear in discovery shortlists.
    try:
        known_in_shortlist = {}
        for ctx in ["primary", "mcrpc", "nepc"]:
            fp = DIRS["tables"] / f"table2b_{ctx}_shortlist.csv"
            if fp.exists():
                d = pd.read_csv(fp)
                if "is_known_clinical_target" in d.columns:
                    n = int(d.is_known_clinical_target.fillna(False).sum())
                    if n:
                        known_in_shortlist[ctx] = n
        check("No known clinical-stage target appears in a discovery shortlist",
              not known_in_shortlist,
              f"contaminated: {known_in_shortlist}" if known_in_shortlist
              else "shortlists contain unlabeled candidates only")
    except Exception as e:
        check("shortlist purity check executed", False, repr(e))

    # 0e. Three-state modality outcomes: NOT ASSESSABLE must never be
    #     silently recorded as a failure, and must never vanish.
    try:
        rank_p = DIRS["tables"] / "table2_full_candidate_ranking.csv"
        if rank_p.exists():
            rk = pd.read_csv(rank_p)
            check("Modality outcome is reported even when nothing is "
                  "assessable", "modality_profile" in rk.columns,
                  "modality_profile present"
                  if "modality_profile" in rk.columns
                  else "COLUMN ABSENT - absence of a column reads as absence "
                       "of risk")
            if "n_modalities_assessable" in rk.columns:
                n_as = int(rk.n_modalities_assessable.max())
                n_pass = int(rk.n_modalities_passing_screen.max())
                check("Modalities passed never exceed modalities assessable",
                      n_pass <= n_as, f"max passed {n_pass} of {n_as} "
                      f"assessable")
    except Exception as e:
        check("modality three-state checks executed", False, repr(e))

    # 0f. The archive must not carry artifacts from a skipped analysis in ANY
    #     directory - the first fix covered tables but not the run logs.
    try:
        log_files = {p.name for p in DIRS["logs"].glob("*.csv")}
        fus_logs = sorted(n for n in log_files if "fusion" in n.lower())
        summ4 = json.load(open(DIRS["logs"] / "run_summary.json"))
        if summ4.get("fusion_sensitivity") is None and fus_logs:
            stale_ok = all(
                (DIRS["logs"] / n).stat().st_mtime < RUN_START_TIME - 1
                for n in fus_logs)
            check("Fusion logs from a skipped analysis are identifiable as "
                  "stale (and excluded from the archive by mtime)",
                  stale_ok,
                  f"{len(fus_logs)} fusion log(s) on disk, all predate this "
                  f"run" if stale_ok else
                  f"a fusion log was written during a run that skipped fusion")
    except Exception as e:
        check("archive log-isolation check executed", False, repr(e))

    # 0g. Benchmark provenance gate cannot be unlocked by one field.
    try:
        ba = audit_benchmarks(verbose=False)
        unlocked = (ba.get("n_unverified") == 0
                    and (ba.get("n_modality_unverified", 1) > 0
                         or ba.get("n_without_source_url", 1) > 0))
        check("Benchmark gate cannot pass on accessibility text alone",
              not unlocked,
              f"accessibility unverified={ba.get('n_unverified')}, "
              f"modality unverified={ba.get('n_modality_unverified')}, "
              f"missing source={ba.get('n_without_source_url')}")
        check("Benchmark set is declared non-independent when revised after "
              "seeing results",
              ba.get("revised_after_seeing_results") is not None,
              f"revised_after_seeing_results = "
              f"{ba.get('revised_after_seeing_results')}")
    except Exception as e:
        check("benchmark gate checks executed", False, repr(e))

    # 0h. Detection-floor boundary: a comparator exactly AT the floor is
    #     censored, therefore NOT measurable.
    try:
        from inspect import getsource as _gs
        msrc = _gs(modality_suitability)
        check("Comparator at exactly the detection floor counts as NOT "
              "measurable (strict >)",
              "bench_raw_mod > CFG.vital_tpm_detection_floor" in msrc,
              "strict inequality used at the floor boundary")
        check("Comparator assessability rule applies to every modality, not "
              "only low ceilings",
              "if not comparator_measurable:" in msrc,
              "single rule for all modalities")
    except Exception as e:
        check("floor-boundary checks executed", False, repr(e))

    # 0i. F2 assessability must follow the SAME comparator rule as the
    #     modality screens: an at-floor comparator cannot certify eligibility.
    try:
        rk2 = pd.read_csv(DIRS["tables"] / "table2_full_candidate_ranking.csv")
        if "f2_assessable" in rk2.columns:
            f2_ok = bool(rk2.f2_assessable.iloc[0])
            m19 = DIRS["tables"] / "table19_modality_expression_screen.csv"
            mod_any = True
            if m19.exists():
                md = pd.read_csv(m19)
                mod_any = bool(md.assessable.any()) if "assessable" in md else True
            check("F2 and the modality screens agree on comparator "
                  "assessability", f2_ok == mod_any,
                  f"F2 assessable = {f2_ok}, any modality assessable = "
                  f"{mod_any}")
            if not f2_ok and "eligibility_basis" in rk2.columns:
                basis = str(rk2.eligibility_basis.iloc[0])
                check("Eligibility is labelled provisional when the F2 "
                      "comparator is censored",
                      "censored" in basis.lower(), f"basis = {basis!r}")
    except Exception as e:
        check("F2 assessability checks executed", False, repr(e))

    # 0j. Both benchmark sets must face the model under identical conditions.
    try:
        _f3 = pd.DataFrame({"gene": sorted(
            set(EVALUATION_EXCLUDE) | {"FILLER1", "FILLER2"})})
        _y3, _hb3 = make_labels(_f3, verbose=False)
        _ex3 = evaluation_exclude_mask(_f3)
        _tx3 = _hb3 | off_class_mask(_f3) | _ex3
        leaked = sorted(set(_f3.gene[(~_tx3) & _f3.gene.isin(
            EVALUATION_EXCLUDE)]))
        check("Every benchmark in BOTH sets is withheld from training",
              not leaked,
              f"leaked into training: {leaked}" if leaked
              else f"all {len(EVALUATION_EXCLUDE)} union benchmarks withheld")
        check("Historical-only benchmarks are withheld but are NOT primary "
              "endpoints",
              int((_ex3 & ~_hb3).sum()) == len(
                  set(HISTORICAL_HELDOUT_POSITIVES)
                  - set(PROSTATE_HELDOUT_POSITIVES)),
              f"{int((_ex3 & ~_hb3).sum())} historical-only gene(s) withheld, "
              f"benchmark mask still {int(_hb3.sum())}")
    except Exception as e:
        check("training-equivalence checks executed", False, repr(e))

    # 0k. The experimental F3 route must not be gated by ECD length, and a
    #     positive assay claim must carry provenance.
    try:
        from inspect import getsource as _gs2
        gsrc = _gs2(apply_gatekeeping)
        check("Experimental F3 route is independent of the ECD-length rule",
              "route_experimental = _experimental" in gsrc
              and "route_anchored = _anchored & _ecd_ok" in gsrc,
              "ECD requirement applies to the anchored route only")
        check("F3 distinguishes route attempted from gate passed",
              "F3_route_attempted" in gsrc, "both fields recorded")
        esrc = _gs2(experimental_surface_evidence)
        check("A positive surface-assay claim requires provenance metadata",
              "SURFACE_EVIDENCE_REQUIRED_FIELDS" in esrc,
              f"required: {list(SURFACE_EVIDENCE_REQUIRED_FIELDS)}")
    except Exception as e:
        check("F3 route checks executed", False, repr(e))

    # 0l. Per-modality three-state outcomes must be EXPORTED, not just held
    #     in memory - a downstream analyst reads the CSV, not the session.
    try:
        rk3 = pd.read_csv(DIRS["tables"] / "table2_full_candidate_ranking.csv")
        need = [f"screen_outcome_{m}" for m in CFG.modality_risk_tissues]
        have = [c for c in need if c in rk3.columns]
        check("Per-modality three-state outcomes are exported in the ranking",
              len(have) == len(need),
              f"{len(have)}/{len(need)} outcome columns present"
              + (f"; missing {sorted(set(need)-set(have))}"
                 if len(have) < len(need) else ""))
        check("Modality assessability counts are exported",
              "n_modalities_assessable" in rk3.columns
              and "n_modalities_not_assessable" in rk3.columns,
              "assessable/not-assessable counts present")
    except Exception as e:
        check("modality export checks executed", False, repr(e))

    # 0m. Benchmark recoverability must be context-specific.
    try:
        rp = DIRS["tables"] / "table27_benchmark_recoverability_by_context.csv"
        check("Benchmark recoverability is reported per disease context",
              rp.exists() and set(pd.read_csv(rp).context) >= {
                  "primary", "mcrpc", "nepc"},
              "primary/mcrpc/nepc all reported" if rp.exists()
              else "table27 absent")
        ep = DIRS["tables"] / "table28_benchmark_endpoints_conditional.csv"
        if ep.exists():
            ed = pd.read_csv(ep)
            check("Unconditional and conditional endpoints are both reported",
                  ed.endpoint.nunique() >= 2,
                  f"{ed.endpoint.nunique()} endpoint type(s)")
    except Exception as e:
        check("recoverability checks executed", False, repr(e))

    # 0n. F2_outcome and F2_safety_definitive must agree. They disagreed
    #     because `~` on a PYTHON bool is bitwise NOT (~True == -2, ~False ==
    #     -1, both truthy), so np.where picked "NOT ASSESSABLE" for every
    #     gene while the definitive flag was computed correctly.
    try:
        rk4 = pd.read_csv(DIRS["tables"] / "table2_full_candidate_ranking.csv")
        if {"f2_assessable", "F2_outcome"} <= set(rk4.columns):
            assessable = bool(rk4.f2_assessable.iloc[0])
            outcomes = set(rk4.F2_outcome.dropna().unique())
            consistent = (({"NOT ASSESSABLE"} == outcomes) if not assessable
                          else ("NOT ASSESSABLE" not in outcomes))
            check("F2_outcome agrees with f2_assessable", consistent,
                  f"f2_assessable={assessable}, outcomes={sorted(outcomes)}")
        if {"F2_safety_definitive", "F2_outcome"} <= set(rk4.columns):
            n_def = int(rk4.F2_safety_definitive.fillna(False).sum())
            n_pass = int((rk4.F2_outcome == "PASS").sum())
            check("F2_outcome PASS count equals the definitive pass count",
                  n_def == n_pass,
                  f"definitive {n_def}, outcome PASS {n_pass}")
    except Exception as e:
        check("F2 outcome consistency checks executed", False, repr(e))

    # 0o. The comparator must DISCRIMINATE. A filter admitting most of the
    #     surfaceome is not a selectivity filter, and an anchor pinned at the
    #     detection floor is not a comparator. Both degenerate regimes have
    #     occurred in this pipeline, in consecutive rounds and in opposite
    #     directions, so both bounds are asserted.
    try:
        rk5 = pd.read_csv(DIRS["tables"] / "table2_full_candidate_ranking.csv")
        if "F2_safety" in rk5.columns:
            rate = float(rk5.F2_safety.fillna(False).mean())
            check("F2 discriminates (admits a minority of the surfaceome)",
                  0.02 <= rate <= 0.45,
                  f"F2 admits {rate:.1%}; outside 2-45% the comparator is "
                  f"either degenerate or permissive")
        if "f2_comparator_raw_tpm" in rk5.columns:
            comp = float(rk5.f2_comparator_raw_tpm.iloc[0])
            floor = float(CFG.vital_tpm_detection_floor)
            check("Comparator is measurable and above the detection floor",
                  comp > floor,
                  f"comparator {comp:.3f} TPM vs floor {floor} TPM")
            if "vital_max_tpm_raw" in rk5.columns:
                v = rk5.vital_max_tpm_raw.dropna()
                pctl = 100.0 * float((v < comp).mean())
                check("Comparator sits in a discriminating position in the "
                      "vital-organ distribution", pctl <= 45.0,
                      f"comparator is at the {pctl:.0f}th percentile of the "
                      f"background; at the median it filters nothing")
    except Exception as e:
        check("comparator calibration checks executed", False, repr(e))

    # 0b. Known clinical-stage targets must never be pseudo-negatives.
    try:
        _f = pd.DataFrame({"gene": sorted(
            set(CLINICAL_STAGE_TARGETS) - {"FOLH1_HELD"})})
        _y, _hb = make_labels(_f, verbose=False)
        _off = off_class_mask(_f)
        n_neg_known = int(((_y == 0) & ~_hb & ~_off).sum())
        check("No known clinical-stage target can enter the negative pool",
              n_neg_known == 0,
              f"{n_neg_known} known target(s) would be treated as unlabeled "
              f"background")
    except Exception as e:
        check("known-target negative-pool check executed", False, repr(e))

    # Everything below calls production code on fixtures. The guard makes a
    # write physically impossible for the rest of this function, regardless of
    # what any call site asks for - the second of the two defences.
    _guard = suppress_outputs(); _guard.__enter__()

    # 1. A perfectly tumour-restricted antigen must survive cleaning.
    #    This is the defect that would have deleted the best candidates on
    #    real data while being invisible in DEMO.
    t = pd.DataFrame(np.random.default_rng(0).normal(5, 1, (3, 12)),
                     index=["TUMOUR_ONLY", "VARIABLE", "FLAT_BOTH"])
    n = pd.DataFrame(np.zeros((3, 6)),
                     index=["TUMOUR_ONLY", "VARIABLE", "FLAT_BOTH"])
    n.loc["VARIABLE"] = np.random.default_rng(1).normal(3, 1, 6)
    t.loc["FLAT_BOTH"] = 0.0                      # constant in BOTH groups
    n.loc["FLAT_BOTH"] = 0.0
    tc = clean_expression(t.copy(), "test tumour", min_frac=0.0)
    nc = clean_expression(n.copy(), "test normal")
    keep, rep = joint_variance_filter(tc, nc, "test")
    check("tumour-restricted antigen survives cleaning (zero variance in "
          "normals only)", "TUMOUR_ONLY" in keep,
          f"retained genes: {keep}")
    check("gene constant across BOTH groups is still dropped",
          "FLAT_BOTH" not in keep, f"retained genes: {keep}")
    # External review, correctly: the previous version of this assertion
    # contained `"TUMOUN_ONLY" not in []`, a typo'd clause that is always
    # true and tested nothing. The default is now read from the signature.
    import inspect as _inspect
    _dzv = _inspect.signature(clean_expression).parameters["drop_zero_var"].default
    check("clean_expression defaults to drop_zero_var=False",
          _dzv is False, f"signature default = {_dzv!r}")

    # 2. Intracellular proteins must be excluded in EVERY disease context.
    #    The fixture includes the four filter controls, because
    #    apply_gatekeeping asserts their presence - a production safeguard
    #    that must not be bypassed just to make a test convenient.
    names = ["SURFACE_OK", "INTRACELLULAR"] + list(FILTER_CONTROLS)
    nrow = len(names)
    def col(default, overrides=None):
        v = [default] * nrow
        for k, x in (overrides or {}).items():
            v[names.index(k)] = x
        return v
    g = pd.DataFrame({
        "gene": names,
        "rna_log2fc": col(2.0, {"CTRL_NOT_DE": 0.0}),
        "rna_qval": col(1e-6, {"CTRL_NOT_DE": 0.9}),
        "frac_tumours_expressing": col(0.95, {"CTRL_NOT_DE": 0.05}),
        "safety_ratio_vs_psma": col(0.2, {"CTRL_HIGH_VITAL": 50.0}),
        "ecd_len_aa": col(200, {"CTRL_NO_ECD": 2}),
        "ecd_len": col(200, {"CTRL_NO_ECD": 2}),
        "n_transmem": col(1, {"CTRL_NO_ECD": 0}),
        "has_gpi": col(False),
        "n_antibodies": col(3, {"CTRL_NO_AB": 0}),
        "antibody_available": col(True, {"CTRL_NO_AB": False}),
        "hpa_membrane": col(True),
        # The flag is DERIVED from `subcellular` inside apply_gatekeeping, so
        # the fixture must supply the annotation, not the derived column -
        # setting the derived column would test nothing.
        "subcellular": col("Plasma membrane",
                           {"INTRACELLULAR": "Endoplasmic reticulum"}),
        "plasma_membrane_confidence": col(
            "high - HPA plasma membrane",
            {"INTRACELLULAR": "low - intracellular membrane"}),
        "mcrpc_frac_expressing": col(0.9),
        "retained_in_nepc": col(True),
        "protein_detected": col(True),
        "protein_corroborated": col(True),
        "prot_log2fc": col(1.5),
        "prot_qval": col(1e-4),
        "percentile": col(99.0),
        "is_surfaceome": col(True),
    })
    try:
        gg, _attr = apply_gatekeeping(g.copy(), save_outputs=False)
        for ctx in ["eligible_primary", "eligible_mcrpc", "eligible_nepc"]:
            val = bool(pd.Series(gg.loc[gg.gene == "INTRACELLULAR", ctx])
                       .fillna(False).astype(bool).iloc[0])
            check(f"intracellular protein excluded from {ctx}", not val,
                  f"{ctx} = {val}")
        check("surface protein still eligible in mCRPC",
              bool(gg.loc[gg.gene == "SURFACE_OK", "eligible_mcrpc"].iloc[0]))
    except Exception as e:
        check("intracellular exclusion test executed", False, repr(e))

    # 3. PSMA strata must be refused when the marker is tied.
    tied = pd.DataFrame(np.random.default_rng(2).normal(4, 1, (3, 40)),
                        index=["FOLH1", "G1", "G2"])
    tied.loc["FOLH1"] = 0.0          # completely tied -> q25 == q75
    check("PSMA-low analysis refuses tied/degenerate strata",
          psma_low_persistence(tied, benchmark="FOLH1") is None,
          "must return None, not a null result for every gene")
    sep = pd.DataFrame(np.random.default_rng(3).normal(4, 1, (3, 40)),
                       index=["FOLH1", "G1", "G2"])
    sep.loc["FOLH1"] = np.linspace(0, 8, 40)      # well separated
    check("PSMA-low analysis proceeds on well-separated strata",
          psma_low_persistence(sep, benchmark="FOLH1") is not None)

    # 4. The baseline table must not report a label-recovery metric as
    #    precision, and must report held-out recall.
    try:
        cols = set(pd.read_csv(DIRS["tables"] /
                               "table13_baseline_comparison.csv").columns)
        check("baseline table no longer reports 'precision_at_25'",
              "precision_at_25" not in cols, f"columns: {sorted(cols)}")
        check("baseline table reports held-out recall at K",
              "heldout_recall_at_25" in cols)
        check("baseline table discloses which methods saw the labels",
              "is_trained_on_labels" in cols)
    except Exception as e:
        check("baseline table check executed", False, repr(e))

    # 5. Shortlist exports must be complete, not truncated at 50.
    try:
        summ = json.load(open(DIRS["logs"] / "run_summary.json"))
        sizes = summ.get("shortlists_by_context", {})
        for ctx, n_claimed in sizes.items():
            fp = DIRS["tables"] / f"table2b_{ctx}_shortlist.csv"
            if fp.exists():
                n_file = len(pd.read_csv(fp))
                check(f"{ctx} shortlist exported complete "
                      f"({n_claimed} claimed)", n_file == n_claimed,
                      f"summary says {n_claimed}, file has {n_file}")
    except Exception as e:
        check("shortlist completeness check executed", False, repr(e))

    # 6. Vital-organ exposure must be censored at the detection floor, so no
    #    candidate can obtain an unbounded therapeutic window from a zero.
    try:
        rank = pd.read_csv(DIRS["tables"] / "table2_full_candidate_ranking.csv")
        if "safety_ratio_vs_psma" in rank.columns:
            n_zero_ratio = int((rank.safety_ratio_vs_psma <= 0).sum())
            check("no candidate has a zero (unbounded-window) safety ratio",
                  n_zero_ratio == 0, f"{n_zero_ratio} gene(s) at ratio 0")
        if "vital_max_at_detection_floor" in rank.columns:
            # SELF-REVIEW: a THIRD hardcoded-True check, beyond the two the
            # review found. It printed a count and asserted nothing. The
            # real property is that the flag agrees with the censored
            # values: a gene flagged as sitting at the floor must actually
            # have its vital maximum at the floor, and vice versa.
            _flag = rank.vital_max_at_detection_floor.fillna(False).astype(bool)
            if "vital_max_tpm_censored" in rank.columns:
                _at_floor = (pd.to_numeric(rank.vital_max_tpm_censored,
                                           errors="coerce")
                             <= CFG.vital_tpm_detection_floor + 1e-9)
                _mismatch = int((_flag != _at_floor).sum())
                check("The at-detection-floor flag agrees with the censored "
                      "vital maxima",
                      _mismatch == 0,
                      f"{_mismatch} gene(s) where the flag and the censored "
                      f"value disagree; {int(_flag.sum())} flagged")
            else:
                check("The censored vital maximum is exported alongside its "
                      "flag", False,
                      "the flag cannot be reconciled without it")
    except Exception as e:
        check("detection-floor check executed", False, repr(e))

    # 7. The blocking preflight must actually raise under strict mode. This
    #    path had never executed: strict is enabled only in LIVE, so the one
    #    gate specified as blocking was never exercised.
    try:
        y_t = np.zeros(40, dtype=int); y_t[:6] = 1
        f_t = pd.DataFrame({"gene": [f"G{i}" for i in range(40)],
                            "x1": np.arange(40.0), "x2": np.arange(40.0)})
        raised = False
        try:
            preflight_label_audit(f_t, y_t, np.zeros(40, dtype=bool),
                                  ["x1", "x2"],
                                  label_audit={"pct_verified": 0.0,
                                               "unresolved_rule_fields":
                                                   ["freeze_date"]},
                                  benchmark_audit={"n_unverified": 6,
                                                   "frozen_on": "UNSET"},
                                  benchmark_mask=np.zeros(40, dtype=bool),
                                  save_outputs=False, strict=True)
        except PreflightError:
            raised = True
        except Exception:
            raised = False
        check("blocking preflight RAISES on unresolved provenance under "
              "strict mode", raised,
              "strict=True must stop before training, not warn")
    except Exception as e:
        check("preflight strict test executed", False, repr(e))

    # 8. Calibration must be reported against the no-skill reference.
    try:
        summ = json.load(open(DIRS["logs"] / "run_summary.json"))
        cal = summ.get("calibration") or {}
        check("calibration reports a Brier SKILL score, not a bare Brier",
              "brier_skill_score" in cal,
              f"skill = {cal.get('brier_skill_score')}")
        check("score is not presented as a calibrated probability when its "
              "skill is negative",
              (cal.get("brier_skill_score", 0) >= 0)
              == bool(cal.get("scores_are_calibrated_probabilities", False)))
    except Exception as e:
        check("calibration reporting check executed", False, repr(e))

    # Functional proof that the guard blocks a write, tested INSIDE the
    # guard. The previous assertion checked that writes had been suppressed,
    # which failed for the right reason: the explicit save_outputs=False
    # arguments stop the writes before the guard is ever reached. Both
    # defences are now verified independently.
    # ---- Modality screen uses the SAME resolved organ set as F2 --------
    # Self-review: modality_suitability() matched tissue names literally
    # while safety_features() resolved them through harmonise_tissue_labels.
    # In the DEMO run the radioligand screen therefore ran WITHOUT salivary
    # gland - the organ that defines PSMA radioligand therapy's dose-limiting
    # xerostomia - and still reported assessable=True with 849 candidates.
    try:
        _declared = sorted({t for ts in CFG.modality_risk_tissues.values()
                            for t in ts})
        _unknown = [t for t in _declared if t not in set(CFG.vital_organs)]
        check("Every declared modality risk tissue is a declared vital organ "
              "(so one alias table covers both screens)",
              not _unknown, f"outside vital_organs: {_unknown}" if _unknown
              else f"{len(_declared)} tissues, all covered")
        _resolved_probe, _miss_probe = harmonise_tissue_labels(
            ["Minor Salivary Gland", "Kidney - Cortex", "Whole Blood",
             "Liver", "Lung", "Brain - Cortex", "Heart - Left Ventricle",
             "Nerve - Tibial", "Artery - Aorta"], verbose=False)
        check("'Salivary Gland' resolves for the radioligand screen on a "
              "GTEx v8-style panel",
              "Salivary Gland" in _resolved_probe,
              f"resolved to {_resolved_probe.get('Salivary Gland')!r}")
        _msrc = _inspect.getsource(modality_suitability)
        check("modality_suitability resolves tissues through the alias table",
              "harmonise_tissue_labels(" in _msrc,
              "literal matching silently drops aliased organs")
        check("modality_suitability gates on tissue coverage",
              "min_vital_organ_coverage" in _msrc,
              "a ratio over a degraded organ set is not a safety screen")
    except Exception as e:
        check("modality harmonisation checks executed", False, repr(e))

    # ---- One answer to "which organs are present" ----------------------
    # External review, explicitly requested: a run reporting 100% organ
    # coverage must not simultaneously report a missing declared organ. The
    # archived run did exactly that - feature table 100%, stop/go matrix
    # "['Salivary Gland'] absent" - because the audit and the screen resolved
    # tissue labels differently.
    try:
        _panel = ["Minor Salivary Gland", "Kidney - Cortex", "Whole Blood",
                  "Liver", "Lung", "Brain - Cortex",
                  "Heart - Left Ventricle", "Nerve - Tibial",
                  "Artery - Aorta"]
        _res_a, _miss_a = harmonise_tissue_labels(_panel, verbose=False)
        _cov_a = len(_res_a) / len(set(CFG.vital_organs))
        check("Organ coverage and the missing-organ audit cannot "
              "contradict each other",
              not (_cov_a >= 1.0 and _miss_a),
              f"coverage {_cov_a:.0%}, missing {_miss_a}")
        _asrc = _inspect.getsource(run_pipeline)
        check("run_pipeline audits organs on the RESOLVED panel",
              "harmonise_tissue_labels(\n        gtex.columns" in _asrc
              or "_resolved_vital, missing_vital" in _asrc,
              "a literal audit raises a false data-quality STOP")
    except Exception as e:
        check("organ audit consistency check executed", False, repr(e))

    # ---- Matched survival endpoint pairs -------------------------------
    try:
        _mixed = pd.DataFrame({"PATIENT_ID": ["P1", "P2"],
                               "DFS_MONTHS": [12, 18],
                               "PFS_STATUS": ["1:Recurred", "0:DISEASE FREE"]})
        try:
            select_survival_endpoint(_mixed)
            check("DFS time with only PFS status is REJECTED", False,
                  "an endpoint defined in no source document was assembled")
        except ValueError as _e:
            check("DFS time with only PFS status is REJECTED",
                  "Incomplete survival endpoint" in str(_e), str(_e)[:70])
        _complete = pd.DataFrame({"DFS_MONTHS": [12], "DFS_STATUS": ["1:Recurred"],
                                  "PFS_MONTHS": [9], "PFS_STATUS": ["0:DISEASE FREE"]})
        _sel = select_survival_endpoint(_complete)
        check("A complete endpoint pair loads with matching columns",
              _sel == ("DFS_MONTHS", "DFS_STATUS", "disease_free_survival"),
              f"selected {_sel}")
        check("A file with no survival columns returns None, not a guess",
              select_survival_endpoint(pd.DataFrame({"AGE": [60]})) is None)
        # Self-review of this round's own fix: a complete PFS pair is
        # correctly preferred over raising on a half-present DFS, but the
        # substitution must be DISCLOSED, not silent.
        _halfdfs = pd.DataFrame({"DFS_MONTHS": [12], "PFS_MONTHS": [9],
                                 "PFS_STATUS": ["0:DISEASE FREE"]})
        import io as _io
        import contextlib as _ctx
        _buf = _io.StringIO()
        with _ctx.redirect_stdout(_buf):
            _sel2 = select_survival_endpoint(_halfdfs)
        check("An endpoint substitution is announced, not silent",
              _sel2[2] == "progression_free_survival"
              and "INCOMPLETE" in _buf.getvalue(),
              f"selected {_sel2}, notice={'yes' if 'INCOMPLETE' in _buf.getvalue() else 'NO'}")
    except Exception as e:
        check("endpoint pairing checks executed", False, repr(e))

    # ---- PH diagnostic is three-state ----------------------------------
    try:
        _csrc = _inspect.getsource(cox_multivariable)
        check("A failed PH diagnostic reports NOT ASSESSED, never a pass",
              'ph_status, ph_p, ph_error = "NOT ASSESSED"' in _csrc
              and "ph_ok, ph_p = True" not in _csrc,
              "an unperformed test must not be recorded as met")
        check("PH failure reason is recorded rather than swallowed",
              "ph_diagnostic_error" in _csrc, "bare except hides the cause")
    except Exception as e:
        check("PH diagnostic check executed", False, repr(e))

    # ---- Strict molecular input validation -----------------------------
    try:
        _ok_counts = pd.DataFrame({"T1": [100.0, 200.0], "T2": [150.0, 60.0]},
                                  index=["A", "B"])
        _frac_counts = _ok_counts.copy()
        _frac_counts.iloc[0, 0] = 100.5
        try:
            differential_expression_nb(_frac_counts, _ok_counts)
            check("A single fractional count is REJECTED, not rounded",
                  False, "fractional abundance became a 'count' matrix")
        except ValueError as _e:
            check("A single fractional count is REJECTED, not rounded",
                  "fractional" in str(_e).lower(), str(_e)[:70])
        _neg_counts = _ok_counts.copy()
        _neg_counts.iloc[1, 1] = -5.0
        try:
            differential_expression_nb(_neg_counts, _ok_counts)
            check("Negative counts are REJECTED", False, "negatives accepted")
        except ValueError as _e:
            check("Negative counts are REJECTED",
                  "negative" in str(_e).lower(), str(_e)[:70])
        _psrc = _inspect.getsource(load_proteomics_live)
        check("Negative linear proteomics values are rejected, not clipped",
              "clip(lower=0) + 1.0" not in _psrc
              and "negative value(s) (min" in _psrc,
              "clipping fabricates a measurement from an invalid one")
    except Exception as e:
        check("molecular input validation checks executed", False, repr(e))

    # ---- Benchmark mask identity, not cardinality ----------------------
    try:
        _rsrc = _inspect.getsource(ranking_stability)
        check("ranking_stability checks benchmark IDENTITY, not just count",
              "actual != expected" in _rsrc,
              "a mask of the right size but wrong members passed before")
        _asrc2 = _inspect.getsource(load_clinical_live)
        check("A requested-but-missing ancestry file raises",
              "ANCESTRY_FILE was set to" in _asrc2,
              "silently substituting self-reported race for genetic "
              "ancestry mislabels every stratified result")
    except Exception as e:
        check("benchmark identity / ancestry checks executed", False, repr(e))

    # ---- Figure captions come from configuration -----------------------
    try:
        _ssrc = _inspect.getsource(run_survival_suite)
        check("KM caption is generated from CFG.expr_split, not hardcoded",
              "pre-declared median split" not in _ssrc
              and "pre-declared {CFG.expr_split} split" in _ssrc,
              "a caption travels into the manuscript with the figure")
    except Exception as e:
        check("caption configuration check executed", False, repr(e))

    # ---- Alias collisions are refused ----------------------------------
    # Self-review of this round's own fix: coverage is now a gate input, so
    # two declared organs resolving to one column must not pass as two.
    try:
        _saved_alias = dict(TISSUE_ALIASES)
        try:
            TISSUE_ALIASES["Lung"] = ["Liver"]   # deliberate collision
            try:
                harmonise_tissue_labels(["Liver"], verbose=False)
                check("Two organs resolving to one column is REFUSED", False,
                      "one measurement counted as two organs present")
            except ValueError as _e:
                check("Two organs resolving to one column is REFUSED",
                      "Ambiguous tissue harmonisation" in str(_e),
                      str(_e)[:70])
        finally:
            TISSUE_ALIASES.clear()
            TISSUE_ALIASES.update(_saved_alias)
        _r_ok, _ = harmonise_tissue_labels(["Liver", "Lung"], verbose=False)
        check("The real alias table has no collisions",
              len(set(_r_ok.values())) == len(_r_ok),
              f"{len(_r_ok)} organs -> {len(set(_r_ok.values()))} columns")
    except Exception as e:
        check("alias collision checks executed", False, repr(e))

    # ---- Count validation streams, and still checks every value --------
    # The strict validation materialised each matrix three times, ~2.2 GB
    # across both on LIVE TCGA PRAD. This asserts the properties that must
    # survive the rewrite: full coverage, no tolerance, and bounded memory.
    try:
        _vsrc = _inspect.getsource(differential_expression_nb)
        check("Count validation does not materialise the whole matrix",
              "_vals[~np.isnan(_vals)]" not in _vsrc and "_CHUNK" in _vsrc,
              "a LIVE-only MemoryError is unreachable from any DEMO test")
        # A fractional value in the LAST column must still be caught, which
        # is what proves the chunk loop covers the whole matrix rather than
        # just the first block.
        _wide = pd.DataFrame(
            {f"S{i}": [100.0, 200.0] for i in range(150)},
            index=["A", "B"])
        _tail = _wide.copy()
        _tail.iloc[1, 149] = 100.5
        try:
            differential_expression_nb(_tail, _wide)
            check("A fractional value in the LAST column is still caught",
                  False, "the chunk loop stopped before the final block")
        except ValueError as _e:
            check("A fractional value in the LAST column is still caught",
                  "fractional" in str(_e).lower(), str(_e)[:70])
        _inf = _wide.copy()
        _inf.iloc[0, 100] = np.inf
        try:
            differential_expression_nb(_inf, _wide)
            check("An inf count is REJECTED, not treated as missing", False,
                  "inf passed validation")
        except ValueError as _e:
            # The message now names missing and infinite counts separately,
            # so the assertion checks the count rather than the old wording.
            check("An inf count is REJECTED, not treated as missing",
                  "infinite" in str(_e).lower(), str(_e)[:70])
    except Exception as e:
        check("count validation property checks executed", False, repr(e))

    # ---- Ambiguous survival columns are refused ------------------------
    try:
        _amb = pd.DataFrame({"DFS_MONTHS_IMPUTED": [12],
                             "DFS_MONTHS_RAW": [13],
                             "DFS_STATUS": ["1:Recurred"]})
        try:
            select_survival_endpoint(_amb)
            check("Two candidate time columns is an error, not a coin toss",
                  False, "the first header match was used silently")
        except ValueError as _e:
            check("Two candidate time columns is an error, not a coin toss",
                  "Ambiguous survival column" in str(_e), str(_e)[:70])
        _exact = pd.DataFrame({"DFS_MONTHS": [12],
                               "DFS_MONTHS_IMPUTED": [13],
                               "DFS_STATUS": ["1:Recurred"]})
        check("An exact column name wins over a longer partial match",
              select_survival_endpoint(_exact)[0] == "DFS_MONTHS",
              f"selected {select_survival_endpoint(_exact)}")
    except Exception as e:
        check("endpoint ambiguity checks executed", False, repr(e))

    # ---- Unattainable modality thresholds ------------------------------
    # External review, correct: the previous guard caught only the case where
    # the comparator sits AT OR BELOW the floor. T_cell_engager sat in the
    # gap - comparator 0.967 TPM above the 0.5 floor, but 0.5/0.967 = 0.517
    # above a 0.5 ceiling - so a mathematically impossible screen reported
    # "assessable: True, 0 passing", which reads as a biological finding.
    try:
        _msrc2 = _inspect.getsource(modality_suitability)
        check("An unattainable ratio threshold is detected directly",
              "_threshold_attainable" in _msrc2, "narrow guard left a gap")
        for _m, _b, _c in (("T_cell_engager", 0.967, 0.5),
                           ("CAR_T", 1.034, 0.5)):
            _minr = CFG.vital_tpm_detection_floor / _b
            _att = _minr <= _c
            check(f"{_m}: attainability is computed, not assumed "
                  f"(min ratio {_minr:.3f} vs ceiling {_c})",
                  _att == (_minr <= _c),
                  f"attainable={_att}")
        _t19 = DIRS["tables"] / "table19_modality_expression_screen.csv"
        if _t19.exists():
            _d19 = pd.read_csv(_t19)
            check("Every configured modality has exactly one audit row",
                  sorted(_d19.modality) == sorted(CFG.modality_risk_tissues),
                  f"rows {sorted(_d19.modality)}")
            _bad = _d19[(_d19.assessable == True)
                        & (_d19.threshold_attainable == False)]
            check("No modality is reported assessable with an unattainable "
                  "threshold", _bad.empty,
                  f"offending: {_bad.modality.tolist()}" if not _bad.empty
                  else "none")
            _zero_pass = _d19[(_d19.assessable == True)
                              & (_d19.n_passing_expression_screen == 0)]
            check("A zero-pass modality is never reported as an assessable "
                  "biological finding", _zero_pass.empty,
                  f"offending: {_zero_pass.modality.tolist()}"
                  if not _zero_pass.empty else "none")
    except Exception as e:
        check("modality attainability checks executed", False, repr(e))

    # ---- Missing raw counts are rejected, not skipped ------------------
    try:
        _wide2 = pd.DataFrame({f"S{i}": [100.0, 200.0] for i in range(8)},
                              index=["A", "B"])
        _nan = _wide2.copy()
        _nan.iloc[0, 3] = np.nan
        try:
            differential_expression_nb(_nan, _wide2)
            check("A NaN raw count is REJECTED, not excluded from "
                  "validation", False,
                  "it would crash later inside pyDESeq2 with "
                  "IntCastingNaNError, far from its cause")
        except ValueError as _e:
            check("A NaN raw count is REJECTED, not excluded from "
                  "validation", "missing" in str(_e).lower(), str(_e)[:70])
        _vsrc2 = _inspect.getsource(differential_expression_nb)
        check("Missing counts are never silently zero-filled",
              "fillna(0)" not in _vsrc2,
              "a missing count and a true zero are different events")
    except Exception as e:
        check("missing-count checks executed", False, repr(e))

    # ---- Clinical and ancestry join cardinality ------------------------
    try:
        _asrc3 = _inspect.getsource(load_clinical_live)
        check("The ancestry join rejects duplicate patient keys",
              "duplicate patient" in _asrc3.lower()
              and 'validate="one_to_one"' in _asrc3,
              "a left merge on duplicated keys multiplies patient rows")
        check("The ancestry match rate is reported, not assumed",
              "min_ancestry_match_rate" in _asrc3,
              "a near-empty annotation must not pass for a merge")
        _bsrc3 = _inspect.getsource(validate_external_bcr)
        check("A very small non-zero endpoint overlap is refused",
              "min_bcr_match_rate" in _bsrc3,
              "4 of 500 patients is not the analysis the Methods describe")
    except Exception as e:
        check("join cardinality checks executed", False, repr(e))

    # ---- The declared endpoint is honoured -----------------------------
    try:
        _dfsonly = pd.DataFrame({"DFS_MONTHS": [12], "DFS_STATUS": ["1:Recurred"]})
        check("CFG.survival_endpoint is actually read",
              "survival_endpoint" in _inspect.getsource(
                  select_survival_endpoint),
              "a config field read nowhere documents an analysis the code "
              "does not perform")
        check("A declared endpoint is not substituted for another",
              _raises(lambda: select_survival_endpoint(
                  _dfsonly, requested="PFS"), (ValueError, KeyError)),
              "requesting PFS on a DFS-only file must refuse, not supply DFS")
        check("An unrecognised endpoint name is refused",
              _raises(lambda: select_survival_endpoint(
                  _dfsonly, requested="NONSENSE"), ValueError))
    except Exception as e:
        check("declared endpoint checks executed", False, repr(e))

    # ---- Both patient joins are guarded, not just one ------------------
    # Self-review: last round the ancestry merge gained a clinical-side
    # duplicate check and validate="one_to_one" while its sibling BCR merge,
    # three lines away in the same function, kept neither. This asserts the
    # policy holds on EVERY patient-level join rather than wherever it was
    # last remembered.
    try:
        _lsrc = _inspect.getsource(load_clinical_live)
        _merges = [ln.strip() for ln in _lsrc.splitlines()
                   if "out.merge(" in ln or "how=\"left\"" in ln]
        _unvalidated = [ln for ln in _merges
                        if "out.merge(" in ln and "validate=" not in ln
                        and "validate=" not in _lsrc[
                            _lsrc.index(ln):_lsrc.index(ln) + 200]]
        check("Every patient-level join declares its cardinality",
              not _unvalidated, f"unvalidated: {_unvalidated}"
              if _unvalidated else f"{len(_merges)} join line(s) checked")
        # SELF-REVIEW, found by mutation-testing this very assertion: the
        # first version counted occurrences of "out.patient.duplicated()",
        # which the ERROR MESSAGES also contain. Deleting a real guard left
        # the count at 2 and the test still passed - precisely the failure
        # the external review warned about ("a suite that passes when a
        # critical guard is deleted is not adequately protecting that
        # guard"). It now counts guard STATEMENTS, and deleting either one
        # fails.
        _n_guards = _lsrc.count("if int(out.patient.duplicated().sum()):")
        check("Both joins check the CLINICAL side for duplicate keys",
              _n_guards >= 2,
              f"{_n_guards} guard statement(s) - the BCR and ancestry "
              f"branches each need one")
    except Exception as e:
        check("join symmetry checks executed", False, repr(e))

    # ---- No silent rounding into the count model -----------------------
    try:
        _dsrc = _inspect.getsource(differential_expression_nb)
        check("pyDESeq2 input is asserted integral, not rounded into shape",
              ".round().astype(int)" not in _dsrc
              and "not integral despite validation" in _dsrc,
              "rounding is a latent re-entry path for the fabricated-count "
              "defect if validation is ever relaxed")
    except Exception as e:
        check("rounding check executed", False, repr(e))

    # NOTE: a test stood here asserting that an "ENDPOINT MISMATCH" notice
    # appears when CFG.survival_endpoint="BCR" and no BCR file is supplied.
    # That combination is now REFUSED outright by
    # validate_endpoint_configuration(), so the notice is unreachable and
    # the branch was removed. The test is deleted rather than relaxed,
    # because it asserted the opposite of the current contract - and the
    # suite caught the contradiction, which is what it is for.

    # ---- One survival-field contract on BOTH endpoint paths ------------
    # The non-numeric-time defect was fixed on the external BCR path and
    # left on the built-in DFS/PFS path, one branch over. Both now share
    # validate_survival_fields().
    try:
        check("Non-numeric times are refused on BOTH endpoint paths",
              _raises(lambda: validate_survival_fields(
                  pd.Series(["unknown", "12"]), pd.Series([1, 0]), "probe"),
                  ValueError))
        check("Event codes outside {0,1} are refused after decoding",
              _raises(lambda: validate_survival_fields(
                  pd.Series([10.0, 12.0]), pd.Series([1, 7]), "probe"),
                  ValueError))
        check("Blank follow-up parses as censored, not as an error",
              validate_survival_fields(pd.Series([np.nan, 12.0]),
                                       pd.Series([0, 1]),
                                       "probe").isna().sum() == 1)
        _lsrc2 = _inspect.getsource(load_clinical_live)
        check("The built-in DFS/PFS path routes through the shared "
              "validator", "validate_survival_fields(" in _lsrc2,
              "a bare pd.to_numeric(errors='coerce') hides malformed times")
        check("The dead ENDPOINT MISMATCH fallback is removed, not left to "
              "imply a fallback that no longer exists",
              "ENDPOINT MISMATCH" not in _lsrc2
              and "validate_endpoint_configuration(" in _lsrc2)
    except Exception as e:
        check("survival field contract checks executed", False, repr(e))

    # ---- Generic reporting carries no endpoint claim -------------------
    try:
        # SELF-REVIEW: the first version of this assertion searched the
        # function's SOURCE for "usable BCR follow-up" - and matched the
        # explanatory comment I had written above the fix, so it failed
        # against correct code. The third time this session a source-text
        # assertion has been fooled by surrounding prose. It now inspects
        # the function's STRING CONSTANTS via the AST, which comments cannot
        # reach.
        import ast as _ast
        _wtree = _ast.parse(_inspect.getsource(load_clinical).strip())
        _wstrings = [n.value for n in _ast.walk(_wtree)
                     if isinstance(n, _ast.Constant)
                     and isinstance(n.value, str)]
        _hardcoded = [t for t in _wstrings if "BCR" in t]
        check("Generic clinical reporting derives the endpoint label from "
              "metadata, not hardcoded BCR wording",
              not _hardcoded,
              f"hardcoded endpoint string(s): {_hardcoded}" if _hardcoded
              else f"{len(_wstrings)} string constant(s), none asserts an "
                   f"endpoint")
    except Exception as e:
        check("endpoint labelling check executed", False, repr(e))

    # ---- Ancestry coverage blocks only prespecified inference ----------
    try:
        _ansrc = _inspect.getsource(load_clinical_live)
        check("Insufficient ancestry coverage BLOCKS prespecified "
              "inference and is exploratory otherwise",
              "ancestry_inference_prespecified" in _ansrc
              and "EXPLORATORY ONLY" in _ansrc,
              "a warning is not equivalent to enforcing a minimum")
        check("Ancestry inference is not prespecified by default",
              CFG.ancestry_inference_prespecified is False,
              "the stricter gate must be opted into deliberately")
    except Exception as e:
        check("ancestry enforcement check executed", False, repr(e))

    # ---- SINGLE-IMPLEMENTATION invariants ------------------------------
    # Self-review, and the most valuable test in this suite if it holds: the
    # recurrent defect in this codebase is not a wrong line, it is ONE RULE
    # IMPLEMENTED TWICE and then changed once. It has now happened with the
    # tissue aliases, the detection floor, the two mask semantics, the
    # join-cardinality guards, the NaN policy and - last round, inside the
    # fix whose stated purpose was a single shared contract - the
    # survival-field validation. These assertions fail when a second
    # implementation of a shared rule appears, which is earlier than when
    # the two copies drift.
    try:
        import ast as _ast2
        import textwrap as _tw

        def _coerce_calls(fn):
            """Count REAL pd.to_numeric(..., errors="coerce") call nodes.

            SELF-REVIEW: the first version of this invariant counted the
            string 'errors="coerce"' in the source and failed on CORRECT
            code, because a comment of mine explaining the old defect
            contained the phrase. That is the FOURTH source-text assertion
            fooled by surrounding prose this session, so the rule is now:
            structural claims about code are made on the AST, never on its
            text.
            """
            _t = _ast2.parse(_tw.dedent(_inspect.getsource(fn)))
            _n = 0
            for _node in _ast2.walk(_t):
                if not isinstance(_node, _ast2.Call):
                    continue
                _f = _node.func
                _name = (_f.attr if isinstance(_f, _ast2.Attribute)
                         else getattr(_f, "id", ""))
                if _name != "to_numeric":
                    continue
                for _kw in _node.keywords:
                    if (_kw.arg == "errors"
                            and isinstance(_kw.value, _ast2.Constant)
                            and _kw.value.value == "coerce"):
                        _n += 1
            return _n

        _per_fn = {f.__name__: _coerce_calls(f) for f in
                   (validate_survival_fields, validate_external_bcr,
                    merge_clinical_to_samples)}
        check("Follow-up time coercion exists in exactly one "
              "implementation",
              _per_fn.get("validate_survival_fields") == 1
              and sum(v for k, v in _per_fn.items()
                      if k != "validate_survival_fields") == 0,
              f"{_per_fn} - a second coercion site is a second contract "
              f"that can be changed on its own")
        # Both endpoint paths must REACH that validator.
        _bsrc4 = _inspect.getsource(validate_external_bcr)
        _lsrc4 = _inspect.getsource(load_clinical_live)
        check("Both endpoint paths route through the shared validator",
              "validate_survival_fields(" in _bsrc4
              and "validate_survival_fields(" in _lsrc4,
              f"BCR={'yes' if 'validate_survival_fields(' in _bsrc4 else 'NO'}, "
              f"built-in={'yes' if 'validate_survival_fields(' in _lsrc4 else 'NO'}")
        # Tissue resolution: one alias table, one resolver, every consumer.
        _tsrcs = {n: _inspect.getsource(f) for n, f in
                  (("safety_features", safety_features),
                   ("modality_suitability", modality_suitability),
                   ("detection_floor_sensitivity", detection_floor_sensitivity),
                   ("run_pipeline", run_pipeline))}
        _missing_res = [n for n, t in _tsrcs.items()
                        if "harmonise_tissue_labels(" not in t]
        check("Every normal-tissue consumer resolves through one alias "
              "table", not _missing_res,
              f"not resolving: {_missing_res}" if _missing_res
              else f"{len(_tsrcs)} consumer(s) checked")
        # The detection floor must be read from CFG, never re-literalled.
        # AST again: a bare 0.5 in a comment or an unrelated ratio must not
        # register, so only numeric Constant nodes are inspected.
        def _floor_literals(fn):
            _t = _ast2.parse(_tw.dedent(_inspect.getsource(fn)))
            return sum(1 for _n in _ast2.walk(_t)
                       if isinstance(_n, _ast2.Constant)
                       and isinstance(_n.value, float)
                       and _n.value == float(CFG.vital_tpm_detection_floor))
        # detection_floor_sensitivity() is deliberately EXEMPT: its whole
        # purpose is to sweep alternative floors, and its default tuple
        # legitimately contains the configured value. Flagging it was a
        # false positive in the first version of this invariant - an
        # over-broad structural check is as much a defect as a missing one,
        # because a test that cries wolf gets silenced.
        _floor_by_fn = {n: _floor_literals(f) for n, f in
                        (("safety_features", safety_features),
                         ("modality_suitability", modality_suitability))}
        check("The detection floor is read from CFG, not re-literalled in "
              "the safety path",
              sum(_floor_by_fn.values()) == 0,
              f"literal {CFG.vital_tpm_detection_floor} occurrences: "
              f"{_floor_by_fn}")
    except Exception as e:
        check("single-implementation invariants executed", False, repr(e))

    # ---- Null join keys, identifier integrity --------------------------
    # Both external reviews reproduced this: pandas matches a null key to
    # another null key, so a 100% clinical match rate was reported while
    # linking a missing identifier to a missing identifier.
    try:
        _s_null = pd.DataFrame({"sample": ["S1", "S2"],
                                "patient": ["TCGA-AA-0001", np.nan]})
        _c_null = pd.DataFrame({"patient": ["TCGA-AA-0001", np.nan],
                                "surv_time": [20.0, 10.0]})
        check("Null patient keys are REFUSED, not matched to each other",
              _raises(lambda: merge_clinical_to_samples(_s_null, _c_null),
                      ValueError),
              "pandas matched NaN to NaN and reported a 100% match rate")
        # SELF-REVIEW: written as for/else with a literal True in the else.
        # Functionally correct - the else only runs if no iteration broke -
        # but it defeats the hardcoded-check scanner and turns into a
        # genuine always-pass test the moment someone removes the break.
        # Rewritten to accumulate a real condition.
        _ph_forms = ["", "nan", "None", "NULL", "  "]
        _ph_accepted = [b for b in _ph_forms
                        if not _raises(
                            lambda b=b: merge_clinical_to_samples(
                                pd.DataFrame({"sample": ["S1", "S2"],
                                              "patient": ["TCGA-AA-0001", b]}),
                                _c_null), ValueError)]
        check("Placeholder identifiers ('', 'nan', 'None', 'NULL') are "
              "REFUSED", not _ph_accepted,
              f"accepted: {_ph_accepted}" if _ph_accepted
              else f"{len(_ph_forms)} placeholder forms checked")
        # NOTE: a test stood here asserting that two barcodes sharing a
        # truncated patient key is a "collision" to be refused. That
        # encoded a WRONG policy of mine - for TCGA, several specimens per
        # patient is the normal relationship - so it is deleted rather than
        # relaxed. The correct contract (repeated specimens accepted,
        # malformed prefixes refused) is asserted in the barcode-policy
        # block below.
        _ok_ids = validate_identifier_column(
            pd.DataFrame({"barcode": ["TCGA-AA-0001-01A",
                                      "TCGA-BB-0002-01A"]}),
            "barcode", "probe", truncate_to=12)
        check("Distinct barcodes truncate without collision",
              list(_ok_ids) == ["TCGA-AA-0001", "TCGA-BB-0002"],
              f"{list(_ok_ids)}")
    except Exception as e:
        check("identifier integrity checks executed", False, repr(e))

    # ---- ONE survival eligibility rule --------------------------------
    # Newly identified by review: usability was a non-missing time in the
    # validator and surv_time > 0 in every model - one rule in five places
    # with two meanings, so zero-duration records passed the gate at 100%
    # and were then dropped by the analysis.
    try:
        _z = pd.DataFrame({"surv_time": [0.0, 5.0, np.nan, -1.0],
                           "surv_event": [1, 0, 1, 1]})
        _m = survival_eligibility_mask(_z)
        check("Zero-duration records are NOT eligible, matching the models",
              list(_m) == [False, True, False, False], f"{list(_m)}")
        # SELF-REVIEW, caught before running: the first draft of this block
        # referenced _bdir and _cohort, which belong to
        # test_live_path_integration() - a NameError waiting in the
        # regression suite. Its own fixture, scoped here.
        import tempfile as _tf2
        _zdir = Path(_tf2.mkdtemp(prefix=f"eligfix_{RUN_ID}_"))
        _zero_bcr = _zdir / "zerotime.csv"
        _zero_bcr.write_text("patient,time_months,event_status\n"
                             "TCGA-AA-0001,0,1:Recurred\n"
                             "TCGA-AA-0002,0,0:DiseaseFree\n"
                             "TCGA-AA-0003,0,0:DiseaseFree\n")
        with isolated_input_registry():
            check("An all-zero-duration endpoint file FAILS the usable gate",
                  _raises(lambda: validate_external_bcr(
                      _zero_bcr, ["TCGA-AA-0001", "TCGA-AA-0002",
                                  "TCGA-AA-0003"]), ValueError),
                  "100% identifier coverage on data the models discard "
                  "entirely")
        # The invariant that would have caught this: eligibility implemented
        # once. My previous single-implementation check covered only time
        # COERCION, which is why this rule slipped past it.
        # SELF-REVIEW: the first version banned the STRING "surv_time > 0"
        # across the consumers, and failed on correct code twice over - it
        # matched my own comment explaining the old defect, and it matched a
        # legitimate QC line that computes the valid-follow-up RATE rather
        # than filtering rows. That is the fifth source-text assertion
        # fooled by surrounding prose this session, so this invariant is
        # stated POSITIVELY instead: every site that FILTERS survival rows
        # must call the shared mask. A positive structural claim cannot be
        # satisfied by a comment, and cannot be broken by one either.
        _filtering_consumers = {
            "load_clinical": load_clinical,
            "validate_external_bcr": validate_external_bcr,
            "run_survival_suite": run_survival_suite,
            "cox_multivariable": cox_multivariable,
            "kaplan_meier_panel": globals().get("kaplan_meier_panel"),
        }
        # enforce_one_sample_per_patient() deliberately no longer touches
        # the eligibility mask: selection must not read the outcome at all,
        # which is the defect both reviews identified. It is checked by the
        # outcome-independence assertions instead, not by this invariant.
        # run_survival_suite() orchestrates rather than filters: it reduces
        # to one record per patient and delegates row filtering to the
        # models. It is checked for the REDUCTION call instead, because
        # that is the rule it is responsible for.
        _filtering_consumers.pop("run_survival_suite", None)
        _not_sharing = [n for n, f in _filtering_consumers.items()
                        if f is not None
                        and "survival_eligibility_mask("
                        not in _inspect.getsource(f)]
        check("Every survival consumer filters rows through the ONE shared "
              "eligibility rule",
              not _not_sharing,
              f"re-implementing eligibility: {_not_sharing}"
              if _not_sharing
              else f"{sum(1 for f in _filtering_consumers.values() if f)} "
                   f"consumer(s) share the rule")
    except Exception as e:
        check("eligibility rule checks executed", False, repr(e))

    # ---- Ancestry contract, file-present or not ------------------------
    try:
        _saved_pre = CFG.ancestry_inference_prespecified
        try:
            CFG.ancestry_inference_prespecified = True
            check("Prespecified genetic ancestry with NO file is REFUSED",
                  _raises(lambda: validate_ancestry_request(CFG, ""),
                          ValueError),
                  "the loader returned self-reported race as ancestry")
            # SELF-REVIEW: the first version referenced `tmp`, which
            # belongs to test_live_path_integration() - the SAME scoping
            # mistake I had already found and fixed two blocks earlier in
            # this round. _raises() swallowed the NameError and reported a
            # clean failure, which is exactly how a test can look like it
            # is checking something while checking nothing.
            check("Prespecified genetic ancestry with a missing file path "
                  "is REFUSED",
                  _raises(lambda: validate_ancestry_request(
                      CFG, str(_zdir / "no_such.csv")), FileNotFoundError))
            CFG.ancestry_inference_prespecified = False
            check("Without prespecified inference, no ancestry file is "
                  "required", validate_ancestry_request(CFG, "") is True,
                  "a race-based breakdown stays available as exploratory")
        finally:
            CFG.ancestry_inference_prespecified = _saved_pre
        _rsrc5 = _inspect.getsource(run_survival_suite)
        check("Stratified models consult ancestry_source before claiming "
              "genetic ancestry",
              "ancestry_source" in _rsrc5 and "_anc_is_genetic" in _rsrc5,
              "a race-stratified model was labelled ancestry_stratified")
    except Exception as e:
        check("ancestry contract checks executed", False, repr(e))

    # ---- The analysis unit is the patient ------------------------------
    try:
        # Outcomes AGREE within a patient: conflicting records are now a
        # reconciliation error, tested separately.
        _rep = pd.DataFrame({
            "patient": ["P1", "P1", "P2"],
            "sample": ["TCGA-AA-0001-01A", "TCGA-AA-0001-11A",
                       "TCGA-BB-0002-01A"],
            "surv_time": [25.0, 25.0, 30.0],
            "surv_event": [1, 1, 0]})
        _red, _n_drop = enforce_one_sample_per_patient(_rep, "probe")
        check("Repeated specimens are reduced to one record per patient",
              len(_red) == 2 and _n_drop == 1, f"{len(_red)} row(s), "
              f"{_n_drop} dropped")
        check("The prespecified rule is deterministic and outcome-blind",
              _red.loc[_red.patient == "P1", "sample"].iloc[0]
              == "TCGA-AA-0001-01A",
              "selection uses specimen type then sample ID, never the "
              "outcome")
        _already = pd.DataFrame({"patient": ["P1", "P2"],
                                 "surv_time": [10.0, 20.0],
                                 "surv_event": [1, 0]})
        check("A table already at one row per patient is unchanged",
              enforce_one_sample_per_patient(_already, "probe")[1] == 0)
        # The reduction must PREFER an eligible record: selecting purely on
        # the longest follow-up discarded a patient's one analysable record
        # when a longer, unusable specimen existed.
        # NOTE: this previously asserted that the reduction keeps the
        # ELIGIBLE record rather than the longest. That assertion belonged
        # to the outcome-dependent rule and is obsolete: selection no longer
        # reads the outcome at all, and records that disagree on the outcome
        # are refused by validate_patient_outcome_consistency() before any
        # specimen is chosen. Both properties are asserted in the
        # specimen-selection block.
        check("The survival suite reduces to patients before modelling",
              "enforce_one_sample_per_patient(" in _rsrc5,
              "two specimens from one patient would contribute the same "
              "event twice and narrow every interval")
    except Exception as e:
        check("analysis-unit checks executed", False, repr(e))

    # ---- The selector refuses what the contract forbids ----------------
    try:
        _dfs_only = pd.DataFrame({"DFS_MONTHS": [12, 18],
                                  "DFS_STATUS": ["1:Recurred",
                                                 "0:DiseaseFree"]})
        check("The standalone selector REFUSES requested='BCR' instead of "
              "returning DFS",
              _raises(lambda: select_survival_endpoint(
                  _dfs_only, requested="BCR"), ValueError),
              "a helper permitting what the contract forbids is one direct "
              "call away from the original defect")
        check("The selector still serves 'auto' and 'DFS'",
              select_survival_endpoint(_dfs_only, requested="auto")[2]
              == "disease_free_survival"
              and select_survival_endpoint(
                  _dfs_only, requested="DFS")[2] == "disease_free_survival")
        # Configuration must be checked BEFORE the download.
        _lsrc6 = _inspect.getsource(load_clinical_live)
        # UPDATED (round 25): the contract check inside load_clinical_live
        # now DELEGATES to validate_live_configuration(), which also runs at
        # the very top of run_pipeline - before the first loader, not merely
        # before the TCGA download. The assertion follows the delegation
        # rather than the literal call it replaced; the stronger ordering
        # claim is asserted separately against run_pipeline's own source.
        check("Configuration is validated before the archive is downloaded",
              _lsrc6.index("validate_live_configuration(")
              < _lsrc6.index("download("),
              "an invalid configuration wasted the download and failed late")
    except Exception as e:
        check("selector contract checks executed", False, repr(e))

    # ---- Patient-level counts and power, not sample-level --------------
    # Self-review: the reduction to one record per patient ran inside
    # run_survival_suite(), but the counts were REPORTED from the un-reduced
    # table and survival_power_report() received it too - so a sample count
    # was printed as "patients" and the power estimate was inflated. The
    # DEMO generator emits one sample per patient, so no DEMO assertion
    # could catch it; these use a deliberately repeated cohort.
    try:
        _csrc7 = _inspect.getsource(load_clinical)
        check("The cohort is reduced to one record per patient at the "
              "single entry point",
              "enforce_one_sample_per_patient(" in _csrc7,
              "reducing inside the models leaves the report and the power "
              "calculation on sample rows")
        check("Sample and patient counts are reported separately",
              "sample-level record(s) ->" in _csrc7
              and "PATIENTS with usable" in _csrc7,
              "a sample count printed as 'patients' overstates the cohort")
        _rep2 = pd.DataFrame({
            "patient": ["P1", "P1", "P2", "P3"],
            "sample": ["TCGA-AA-0001-01A", "TCGA-AA-0001-11A",
                       "TCGA-BB-0002-01A", "TCGA-CC-0003-01A"],
            "surv_time": [25.0, 25.0, 30.0, 40.0],
            "surv_event": [1, 1, 0, 1],
            "endpoint_name": ["x"] * 4,
            "ancestry": ["a"] * 4, "ancestry_source": ["genetic"] * 4})
        _red2, _nd2 = enforce_one_sample_per_patient(_rep2, "probe")
        check("A repeated-specimen cohort yields unique patients only",
              len(_red2) == _red2.patient.nunique() == 3 and _nd2 == 1,
              f"{len(_red2)} row(s), {_red2.patient.nunique()} patient(s)")
        # The power report must see the reduced table: on this cohort the
        # un-reduced event count is 3 and the reduced count is 2.
        check("Pseudoreplication would have inflated the event count",
              int(_rep2.surv_event.sum()) == 3
              and int(_red2.surv_event.sum()) == 2,
              f"un-reduced {int(_rep2.surv_event.sum())} event(s) vs "
              f"{int(_red2.surv_event.sum())} after reduction")
        check("The reduction is idempotent, so the guard inside the "
              "survival suite cannot change the cohort twice",
              enforce_one_sample_per_patient(_red2, "probe")[1] == 0)
    except Exception as e:
        check("analysis-unit reporting checks executed", False, repr(e))

    # ---- Unparseable survival values never leave silently --------------
    try:
        check("An unparseable follow-up time RAISES instead of being "
              "filtered out as ineligible",
              _raises(lambda: survival_eligibility_mask(pd.DataFrame(
                  {"surv_time": ["unknown", 12.0],
                   "surv_event": [1, 0]})), ValueError),
              "a dropped record would be indistinguishable from a censored "
              "patient")
        check("An unparseable event code RAISES",
              _raises(lambda: survival_eligibility_mask(pd.DataFrame(
                  {"surv_time": [10.0, 12.0],
                   "surv_event": ["maybe", 0]})), ValueError))
        check("Genuinely missing values remain eligible-or-not without "
              "raising",
              list(survival_eligibility_mask(pd.DataFrame(
                  {"surv_time": [np.nan, 12.0],
                   "surv_event": [1, 0]}))) == [False, True])
        check("A survival table missing a required column RAISES",
              _raises(lambda: survival_eligibility_mask(
                  pd.DataFrame({"surv_time": [1.0]})), ValueError))
    except Exception as e:
        check("mask strictness checks executed", False, repr(e))

    # ---- Specimen selection must not depend on the OUTCOME -------------
    # Both external reviews, independently, on the reduction rule I wrote
    # last round. Their scenario: (12 mo, recurrence) and (99 mo, censored)
    # for one patient selected the 99-month row and the recurrence event
    # vanished. Selecting on follow-up conditions the analysis on the
    # quantity being modelled.
    try:
        _conflict = pd.DataFrame({
            "patient": ["P1", "P1", "P2"],
            "sample": ["TCGA-AA-0001-01A", "TCGA-AA-0001-11A",
                       "TCGA-BB-0002-01A"],
            "surv_time": [12.0, 99.0, 16.0],
            "surv_event": [1, 0, 0]})
        check("Conflicting patient-level outcomes are FLAGGED for "
              "reconciliation, not silently resolved",
              _raises(lambda: validate_patient_outcome_consistency(
                  _conflict), ValueError),
              "the longest-follow-up rule made a recurrence event disappear")
        _flagged = validate_patient_outcome_consistency(_conflict,
                                                        strict=False)
        check("Non-strict mode reports the conflicting patients",
              _flagged == ["P1"], f"{_flagged}")
        # With outcomes agreeing, selection must use specimen metadata only.
        # SELF-REVIEW: this fixture used four-field barcodes
        # ("TCGA-AA-0001-01A"), which are sample IDs. Real TCGA ALIQUOT
        # barcodes carry three more fields, and the regex this test was
        # written against read the trailing plate code instead of the sample
        # type on those - so the test passed while production would not
        # have worked. The fixture is now the real form, which is what the
        # loader actually receives.
        _agree = pd.DataFrame({
            "patient": ["P1", "P1", "P2"],
            "sample": ["TCGA-AA-0001-11A-01R-A089-07",
                       "TCGA-AA-0001-01A-11R-A056-07",
                       "TCGA-BB-0002-01A-11R-A056-07"],
            "surv_time": [20.0, 20.0, 16.0],
            "surv_event": [1, 1, 0]})
        _sel1, _ = select_one_specimen_per_patient(_agree, "probe")
        check("The primary-tumour specimen (-01) is preferred over the "
              "normal specimen (-11)",
              _sel1.loc[_sel1.patient == "P1", "sample"].iloc[0]
              == "TCGA-AA-0001-01A-11R-A056-07",
              f"selected {_sel1.loc[_sel1.patient == 'P1', 'sample'].iloc[0]}")
        # The property the review asked to be testable: change the
        # follow-up and the SAME specimen must still be selected.
        _perturbed = _agree.copy()
        _perturbed.loc[:, "surv_time"] = [999.0, 1.0, 16.0]
        _perturbed.loc[:, "surv_event"] = [0, 1, 0]
        _sel2, _ = select_one_specimen_per_patient(_perturbed, "probe")
        check("Changing follow-up and event status does NOT change which "
              "specimen is selected",
              list(_sel2["sample"]) == list(_sel1["sample"]),
              f"before {list(_sel1['sample'])}, after {list(_sel2['sample'])}")
        # The sample-type code must be read from the SAME field whatever
        # the barcode length, including the full aliquot form whose trailing
        # field is a plate code.
        _bc_forms = (
            ("TCGA-ZZ-9001-01A-11R-A056-07",
             "TCGA-ZZ-9001-11A-01R-A089-07", "full aliquot"),
            ("TCGA-ZZ-9001-01A", "TCGA-ZZ-9001-11A", "sample+vial"),
            ("TCGA-ZZ-9001-01", "TCGA-ZZ-9001-11", "sample only"))
        _bc_wrong = []
        for _bc_t, _bc_n, _form in _bc_forms:
            _f = pd.DataFrame({
                "patient": ["Z1", "Z1"], "sample": [_bc_n, _bc_t],
                "surv_time": [30.0, 30.0], "surv_event": [1, 1]})
            _got = select_one_specimen_per_patient(
                _f, "probe")[0]["sample"].iloc[0]
            if _got != _bc_t:
                _bc_wrong.append(f"{_form}: got {_got}, wanted {_bc_t}")
        check("Primary tumour is preferred for every TCGA barcode length "
              "(aliquot, sample+vial, sample)",
              not _bc_wrong, "; ".join(_bc_wrong) if _bc_wrong
              else f"{len(_bc_forms)} barcode forms checked, including the "
                   f"full aliquot form whose trailing field is a plate code")
        _ssrc = _inspect.getsource(select_one_specimen_per_patient)
        check("The selection rule never reads surv_time or surv_event",
              "surv_time" not in _ssrc.split('"""')[-1]
              and "surv_event" not in _ssrc.split('"""')[-1],
              "an outcome-dependent sort key is selection bias")
    except Exception as e:
        check("specimen selection checks executed", False, repr(e))

    # ---- Kaplan-Meier edge cases ---------------------------------------
    try:
        _ties = pd.DataFrame({
            "patient": [f"P{i}" for i in range(36)],
            "sample": [f"S{i}" for i in range(36)],
            "surv_time": [float(10 + i) for i in range(36)],
            "surv_event": [1] * 12 + [0] * 24})
        _flat_expr = pd.DataFrame(
            [[5.0] * 36], index=["GENE1"],
            columns=[f"S{i}" for i in range(36)])
        _r_ties = km_by_expression(_flat_expr, _ties, "GENE1", split="tertile")
        check("Tied expression yields NOT FITTED, not 'Bin edges must be "
              "unique'",
              isinstance(_r_ties, dict)
              and _r_ties.get("status") == "NOT FITTED",
              f"{_r_ties if not isinstance(_r_ties, dict) else _r_ties.get('reason', '')[:60]}")
        _r_med = km_by_expression(_flat_expr, _ties, "GENE1", split="median")
        check("Constant expression yields NOT FITTED on a median split too, "
              "rather than a fabricated two-group comparison",
              isinstance(_r_med, dict)
              and _r_med.get("status") == "NOT FITTED")
        check("An unsupported split setting RAISES instead of silently "
              "becoming a tertile analysis",
              _raises(lambda: km_by_expression(
                  _flat_expr, _ties, "GENE1", split="quartile"), ValueError))
        _ksrc = _inspect.getsource(km_by_expression)
        # SELF-REVIEW: the first version asserted the STRING
        # "split=CFG.expr_split" was absent - and my own comment explaining
        # the old defect contains it, so the test failed against correct
        # code. Sixth time this session. The default is inspected on the
        # AST, where a comment cannot appear.
        import ast as _ast4
        import textwrap as _tw4
        _kfn = _ast4.parse(_tw4.dedent(_ksrc)).body[0]
        _split_default = next(
            (d for a, d in zip(
                _kfn.args.args[-len(_kfn.args.defaults):]
                if _kfn.args.defaults else [],
                _kfn.args.defaults) if a.arg == "split"), None)
        check("The split is read at CALL time, not captured in a default "
              "argument",
              isinstance(_split_default, _ast4.Constant)
              and _split_default.value is None,
              f"default is {_ast4.dump(_split_default)[:60] if _split_default else 'absent'}"
              f" - a CFG value captured at def time would not track a later "
              f"configuration change")
        check("Groups in the log-rank test are all plotted, including "
              "zero-event groups",
              "if sub.surv_event.sum() == 0:" not in _ksrc,
              "omitting a curve that the p-value includes makes the figure "
              "and the test describe different comparisons")
        _rssrc = _inspect.getsource(run_survival_suite)
        check("A NOT FITTED record is distinguished from a fitted result",
              'status") == "NOT FITTED"' in _rssrc
              and "NOT FITTED" in _rssrc,
              "a NOT FITTED dict is truthy and was appended as a result")
    except Exception as e:
        check("Kaplan-Meier edge-case checks executed", False, repr(e))

    # ---- Three ancestry sources, three labels --------------------------
    try:
        _asrc8 = _inspect.getsource(run_survival_suite)
        check("Simulated genetic ancestry is labelled DEMO, not "
              "self-reported race",
              "genetic_simulated" in _asrc8
              and "simulated_genetic_ancestry_DEMO" in _asrc8,
              "the DEMO variable was exported as "
              "self_reported_race_stratified_EXPLORATORY")
        _t8 = DIRS["tables"] / "table8_cox_models.csv"
        if _t8.exists():
            _d8 = pd.read_csv(_t8)
            _mis = _d8[_d8.get("label", pd.Series(dtype=str)).astype(str)
                       .str.contains("self_reported_race", na=False)]
            check("No exported Cox row calls the DEMO ancestry variable "
                  "self-reported race",
                  _mis.empty, f"{len(_mis)} mislabelled row(s)")
    except Exception as e:
        check("ancestry labelling checks executed", False, repr(e))

    # ---- Barcode policy: many specimens per patient is legal -----------
    try:
        _two_spec = pd.DataFrame({"sample": ["TCGA-AA-0001-01A",
                                             "TCGA-AA-0001-11A"]})
        _short = validate_identifier_column(
            _two_spec, "sample", "probe", truncate_to=12)
        check("Two specimens of ONE patient are accepted, not rejected as "
              "a collision",
              list(_short) == ["TCGA-AA-0001", "TCGA-AA-0001"],
              f"{list(_short)}")
        check("A barcode too short to carry a patient prefix is REFUSED",
              _raises(lambda: validate_identifier_column(
                  pd.DataFrame({"sample": ["TCGA-AA", "TCGA-BB-0002-01A"]}),
                  "sample", "probe", truncate_to=12), ValueError),
              "an unresolvable key would merge unrelated records")
        _lsrc9 = _inspect.getsource(load_clinical_live)
        # SELF-REVIEW: this asserted the loader uses truncate_to=12 - the
        # LENGTH check. That mechanism has been replaced by
        # parse_tcga_barcodes(), which validates the FORMAT, so the old
        # assertion now demands the weaker guard. Updated to the stronger
        # contract rather than kept passing by restoring the weaker one.
        check("The production loader parses TCGA barcodes by format, not "
              "by length", "parse_tcga_barcodes(" in _lsrc9,
              "a bare .str[:12] - or a length-only check - accepts "
              "'BAD-INPUT-1234-01A' and invents a patient ID")
    except Exception as e:
        check("barcode policy checks executed", False, repr(e))

    # ---- The lockfile is installable -----------------------------------
    try:
        # SELF-REVIEW: the first version read environment_lock.txt from
        # disk - but export_outputs() writes it AFTER the test suite runs,
        # so the assertion validated the PREVIOUS run's file and reported a
        # defect that was already fixed. A test that reads an artifact its
        # own run has not produced yet is checking history. The generator is
        # checked here (it also self-asserts at write time), and the file is
        # only inspected when it belongs to this run.
        _esrc9 = _inspect.getsource(export_outputs)
        check("The lockfile generator maps import names to DISTRIBUTION "
              "names and refuses prose",
              '"sklearn": "scikit-learn"' in _esrc9
              and "non-installable line(s)" in _esrc9,
              "'sklearn==x' and 'pkg==NOT INSTALLED in this run' are not "
              "installable requirements")
        _lock = DIRS["logs"] / "environment_lock.txt"
        if _lock.exists() and RUN_ID in _lock.read_text():
            import re as _re3
            _reqs = [ln.strip() for ln in _lock.read_text().splitlines()
                     if ln.strip() and not ln.strip().startswith("#")]
            _bad = [ln for ln in _reqs if not _re3.fullmatch(
                r"[A-Za-z0-9][A-Za-z0-9._-]*==[0-9][^\s]*", ln)]
            check("Every line of environment_lock.txt is a valid pinned "
                  "requirement", not _bad,
                  f"non-installable: {_bad}" if _bad
                  else f"{len(_reqs)} requirement(s)")
            check("scikit-learn is recorded under its DISTRIBUTION name",
                  not any(ln.startswith("sklearn==") for ln in _reqs),
                  "'sklearn' is the import name; pip install sklearn is "
                  "not the library")
    except Exception as e:
        check("lockfile checks executed", False, repr(e))

    # ---- Every KM row states its own outcome ---------------------------
    try:
        # SELF-REVIEW: the first version of this block looked for
        # "table7_km_logrank.csv", a filename that has never existed - the
        # table is table7_kaplan_meier.csv - and guarded the read with
        # `if _t.exists()`. So the assertion SILENTLY SKIPPED and reported
        # nothing, which is the failure mode I hardened _raises() against
        # one round ago, reappearing as an over-permissive existence guard.
        # A named artifact that is absent is now a failure, not a skip.
        _t = require_output(DIRS["tables"] / "table7_kaplan_meier.csv",
                            "the Kaplan-Meier results")
        if True:
            _d = pd.read_csv(_t)
            check("The KM table carries a status column",
                  "status" in _d.columns,
                  f"columns: {list(_d.columns)}")
            if "status" in _d.columns:
                check("Every exported KM row states FITTED or NOT FITTED, "
                      "so a blank cannot be read as success",
                      _d.status.notna().all()
                      and set(_d.status.unique()) <= {"FITTED", "NOT FITTED"},
                      f"statuses: {sorted(set(_d.status.dropna().unique()))}")
                _nf = _d[_d.status == "NOT FITTED"]
                check("No NOT FITTED row carries a p-value or a q-value",
                      _nf.empty or (_nf.get("logrank_p").isna().all()
                                    and _nf.get("qval_bh").isna().all()),
                      f"{len(_nf)} NOT FITTED row(s)")
    except Exception as e:
        check("KM row status checks executed", False, repr(e))

    # ---- A tertile request never becomes a two-group test --------------
    # External review, reproduced: expression taking two values gives
    # DISTINCT cutpoints (0 and 1) that passed last round's uniqueness
    # check, but groups Low=18, Mid=18, High=0 - so the two-group log-rank
    # ran while the record still said threshold_rule="tertile".
    try:
        _cols = [f"S{i}" for i in range(36)]
        _km_clin = pd.DataFrame({
            "patient": [f"P{i}" for i in range(36)],
            "sample": _cols,
            "surv_time": [float(10 + i) for i in range(36)],
            "surv_event": [1] * 12 + [0] * 24,
            "endpoint_name": ["probe"] * 36})
        _two_valued = pd.DataFrame([[0.0] * 18 + [1.0] * 18],
                                   index=["GENE1"], columns=_cols)
        _r2 = km_by_expression(_two_valued, _km_clin, "GENE1",
                               split="tertile")
        check("Two-valued expression yields NOT FITTED, not a two-group "
              "test mislabelled as a tertile analysis",
              isinstance(_r2, dict) and _r2.get("status") == "NOT FITTED",
              (_r2 or {}).get("reason", str(_r2))[:80])
        # A genuine three-group split must still fit.
        _spread = pd.DataFrame([[float(i) for i in range(36)]],
                               index=["GENE1"], columns=_cols)
        _r3 = km_by_expression(_spread, _km_clin, "GENE1", split="tertile")
        check("A genuine three-group tertile split still FITS",
              isinstance(_r3, dict) and _r3.get("status") == "FITTED",
              (_r3 or {}).get("reason", str(_r3))[:60])
        # Every skip path must be recorded, not silently dropped.
        _r4 = km_by_expression(_spread, _km_clin, "ABSENT_GENE")
        check("A gene absent from the matrix yields a RECORD, not None",
              isinstance(_r4, dict)
              and "absent" in _r4.get("reason", "").lower(),
              str(_r4)[:70])
        _small = _km_clin.head(12).copy()
        _r5 = km_by_expression(_spread[_cols[:12]], _small, "GENE1")
        check("Too few eligible patients yields a RECORD with its reason",
              isinstance(_r5, dict)
              and "insufficient" in _r5.get("reason", "").lower(),
              str(_r5)[:70])
        _few_ev = _km_clin.copy()
        _few_ev["surv_event"] = [1] * 3 + [0] * 33
        _r6 = km_by_expression(_spread, _few_ev, "GENE1")
        check("Too few events yields a RECORD with its reason",
              isinstance(_r6, dict)
              and "event" in _r6.get("reason", "").lower(), str(_r6)[:70])
    except Exception as e:
        check("KM grouping checks executed", False, repr(e))

    # ---- Unreconciled outcomes cannot reach modelling ------------------
    try:
        _conf = pd.DataFrame({
            "patient": ["P1", "P1"],
            "sample": ["TCGA-AA-0001-01A-11R-A056-07",
                       "TCGA-AA-0001-06A-11R-A056-07"],
            "surv_time": [12.0, 99.0], "surv_event": [1, 0]})
        _saved_strict = CFG.strict_patient_outcome_consistency
        try:
            _permissive = []
            for _mode in (True, False):
                CFG.strict_patient_outcome_consistency = _mode
                if not _raises(lambda: enforce_one_sample_per_patient(
                        _conf, "probe"), ValueError):
                    _permissive.append(_mode)
            check("Conflicting outcomes are refused in BOTH strict and "
                  "non-strict mode", not _permissive,
                  f"permitted modelling with strict={_permissive}"
                  if _permissive
                  else "non-strict audits the conflict; it never permits "
                       "modelling on unreconciled records")
        finally:
            CFG.strict_patient_outcome_consistency = _saved_strict
        check("Non-strict validation still RETURNS the conflicting patients "
              "for audit",
              validate_patient_outcome_consistency(_conf, strict=False)
              == ["P1"])
    except Exception as e:
        check("outcome reconciliation checks executed", False, repr(e))

    # ---- TCGA barcodes are validated by FORMAT, not length -------------
    try:
        check("A malformed barcode of the right LENGTH is REFUSED",
              _raises(lambda: parse_tcga_barcodes(
                  ["BAD-INPUT-1234-01A"], "probe"), ValueError),
              "a length check accepted it and yielded patient "
              "'BAD-INPUT-12'")
        _p = parse_tcga_barcodes(
            ["TCGA-AA-0001-01A-11R-A29S-07", "TCGA-AA-0001-11A-01R-1234-07"],
            "probe")
        check("The sample-type code is read from the correct field on FULL "
              "aliquot barcodes",
              list(_p.sample_type) == ["01", "11"],
              f"{list(_p.sample_type)} - the old regex returned ['07','07']")
        check("Both specimens resolve to the same patient",
              set(_p.patient) == {"TCGA-AA-0001"}, f"{set(_p.patient)}")
        check("A short sample-level barcode parses identically",
              list(parse_tcga_barcodes(["TCGA-AA-0001-01"],
                                       "probe").sample_type) == ["01"])
        check("SU2C-style identifiers are NOT forced through the TCGA "
              "pattern",
              "parse_tcga_barcodes" not in _inspect.getsource(
                  load_mcrpc_live),
              "imposing TCGA format on another cohort rejects valid data")
    except Exception as e:
        check("TCGA barcode checks executed", False, repr(e))

    # ---- Release packaging is complete and blocking --------------------
    try:
        _esrc10 = _inspect.getsource(export_outputs)
        _unpinned = [k for k in ("seaborn", "requests")
                     if f'"{k}": "{k}"' not in _esrc10]
        check("Unconditionally imported packages (seaborn, requests) are "
              "pinned", not _unpinned,
              f"missing from the pin map: {_unpinned}" if _unpinned
              else "both present in the pin map")
        check("An imported third-party module missing from the pin map is "
              "REPORTED", "are not in the pin map" in _esrc10,
              "a silently unpinned dependency is how seaborn and requests "
              "were omitted")
        check("A failed release artifact BLOCKS the release",
              "release packaging FAILED" in _esrc10
              and "release package NOT written" not in _esrc10,
              "a warning let a run ship with no lockfile and no source "
              "snapshot while reporting success")
    except Exception as e:
        check("release packaging checks executed", False, repr(e))

    # ---- The headline comparison is on a COMMON population -------------
    # Self-review of the statistic the STOP verdict rests on: the median
    # benchmark percentile was computed over all ranked genes, a population
    # that contains the ensemble's own training positives. All 57 of them
    # outrank the median benchmark, depressing the ensemble's percentile in
    # a way the unsupervised baselines are not subject to. Measured shift:
    # ensemble +1.59, B1 +0.64, so the gain moves -2.95 -> -2.00.
    try:
        _t13 = require_output(
            DIRS["tables"] / "table13_baseline_comparison.csv",
            "the primary benchmark comparison")
        if True:
            _d13 = pd.read_csv(_t13)
            check("A common-population percentile is reported for every "
                  "method",
                  "median_benchmark_percentile_common_population"
                  in _d13.columns
                  and _d13["median_benchmark_percentile_common_population"]
                  .notna().all(),
                  f"columns: {[c for c in _d13.columns if 'percentile' in c]}")
            _ens = _d13[_d13.method.str.startswith("PU ensemble")]
            _bas = _d13[~_d13.method.str.startswith("PU ensemble")]
            if len(_ens) and len(_bas):
                _shift_e = float(
                    _ens["median_benchmark_percentile_common_population"]
                    .iloc[0] - _ens["median_benchmark_percentile"].iloc[0])
                _shift_b = float(
                    _bas["median_benchmark_percentile_common_population"]
                    .max() - _bas["median_benchmark_percentile"].max())
                check("Excluding training labels helps the SUPERVISED "
                      "method more than the unsupervised baselines, which "
                      "is why the populations must match",
                      _shift_e >= _shift_b,
                      f"ensemble {_shift_e:+.2f}, best baseline "
                      f"{_shift_b:+.2f}")
        _vsrc = _inspect.getsource(simple_baselines)
        check("The verdict is computed on the common population",
              '_pcol = "median_benchmark_percentile_common_population"'
              in _vsrc,
              "comparing a label-memorising ranker against unsupervised "
              "scores on a pool containing its training set penalises it "
              "for something other than ranking quality")
        check("The all-genes figure is retained alongside, not replaced",
              "incremental_gain_pct_all_genes" in _vsrc,
              "both populations must stay visible")
        # The correction must NOT be what decides the verdict.
        if _t13.exists() and len(_ens) and len(_bas):
            _gain_common = float(
                _ens["median_benchmark_percentile_common_population"].iloc[0]
                - _bas["median_benchmark_percentile_common_population"].max())
            check("The verdict is unchanged by the correction: the gain "
                  "still falls short of the prespecified margin",
                  _gain_common < CFG.min_meaningful_margin_pct,
                  f"gain {_gain_common:+.2f} vs required "
                  f"+{CFG.min_meaningful_margin_pct}")
    except Exception as e:
        check("common-population comparison checks executed", False, repr(e))

    # ---- The release archive proves its own integrity ------------------
    # External review, confirmed against the delivered ZIP: five stale files
    # were present with no manifest entry, beside a regression test that
    # asserted their exclusion and was hardcoded to pass. The audit must
    # inspect the FINISHED archive, and this test must prove the audit
    # actually rejects a bad one.
    try:
        import hashlib as _h2
        from zipfile import ZipFile as _ZF2
        _zdir2 = Path(_tf2.mkdtemp(prefix=f"zipaudit_{RUN_ID}_"))
        _stage = _zdir2 / "stage" / "06_logs"
        _stage.mkdir(parents=True)
        (_zdir2 / "stage" / "05_tables").mkdir(parents=True)
        _good_csv = _zdir2 / "stage" / "05_tables" / "table_x.csv"
        _good_csv.write_text("a,b\n1,2\n")
        _digest = _h2.sha256(_good_csv.read_bytes()).hexdigest()
        pd.DataFrame([{"run_id": RUN_ID,
                       "archived_file": "05_tables/table_x.csv",
                       "bytes": _good_csv.stat().st_size,
                       "sha256": _digest}]).to_csv(
            _stage / "archive_contents.csv", index=False)
        _zip_ok = shutil.make_archive(str(_zdir2 / "ok"), "zip",
                                      root_dir=str(_zdir2 / "stage"))
        check("A correctly manifested archive VERIFIES",
              verify_release_archive(_zip_ok, require_release_members=False) is True)

        # Plant exactly the stale file the reviewer found.
        _stale_f = (_zdir2 / "stage" / "05_tables"
                    / "table19_modality_suitability.csv")
        _stale_f.write_text("obsolete,result\n1,2\n")
        _zip_bad = shutil.make_archive(str(_zdir2 / "bad"), "zip",
                                       root_dir=str(_zdir2 / "stage"))
        check("An UNMANIFESTED stale file fails the release audit",
              _raises(lambda: verify_release_archive(_zip_bad, require_release_members=False),
                      RuntimeError),
              "this is the exact file that shipped unmanifested")
        _stale_f.unlink()

        # A tampered or regenerated file must fail on checksum.
        _good_csv.write_text("a,b\n9,9\n")
        _zip_tamper = shutil.make_archive(str(_zdir2 / "tamper"), "zip",
                                          root_dir=str(_zdir2 / "stage"))
        check("A checksum mismatch fails the release audit",
              _raises(lambda: verify_release_archive(_zip_tamper, require_release_members=False),
                      RuntimeError))
        # A manifest-less archive must fail too.
        (_stage / "archive_contents.csv").unlink()
        _zip_noman = shutil.make_archive(str(_zdir2 / "noman"), "zip",
                                         root_dir=str(_zdir2 / "stage"))
        check("An archive with no checksum manifest fails the audit",
              _raises(lambda: verify_release_archive(_zip_noman, require_release_members=False),
                      RuntimeError))
        # A RELEASE-SHAPED archive that is internally consistent but is
        # missing run_manifest.json must still fail. This is the exact
        # archive the last three releases shipped: every listed file matched
        # its checksum, and the single most important reproducibility
        # artifact was simply not there.
        _rel = _zdir2 / "relstage"
        (_rel / "05_tables").mkdir(parents=True, exist_ok=True)
        (_rel / "06_logs" / "pipeline_source").mkdir(parents=True,
                                                     exist_ok=True)
        (_rel / "05_tables" / "t.csv").write_text("a\n1\n")
        (_rel / "06_logs" / "run_summary.json").write_text("{}")
        (_rel / "06_logs" / "environment_lock.txt").write_text("x==1\n")
        (_rel / "06_logs" / "pipeline_source"
         / "run_pipeline.py").write_text("pass\n")
        _relman = []
        for _f in sorted(_rel.rglob("*")):
            if _f.is_file():
                _relman.append({
                    "run_id": RUN_ID,
                    "archived_file": str(_f.relative_to(_rel)).replace(
                        "\\", "/"),
                    "bytes": _f.stat().st_size,
                    "sha256": _h2.sha256(_f.read_bytes()).hexdigest()})
        pd.DataFrame(_relman).to_csv(
            _rel / "06_logs" / "archive_contents.csv", index=False)
        _zip_rel = shutil.make_archive(str(_zdir2 / "rel"), "zip",
                                       root_dir=str(_rel))
        check("A release archive that is internally CONSISTENT but missing "
              "run_manifest.json is refused",
              _raises(lambda: verify_release_archive(_zip_rel), RuntimeError),
              "manifest reconciliation alone cannot detect an artifact that "
              "was never registered")
        check("export_outputs runs the audit on the archive it just wrote",
              "verify_release_archive(archive)"
              in _inspect.getsource(export_outputs),
              "every prior integrity claim was made before packaging")
        shutil.rmtree(_zdir2, ignore_errors=True)
    except Exception as e:
        check("release archive audit checks executed", False, repr(e))

    # ---- No hardcoded passing checks remain ----------------------------
    try:
        import ast as _ast5
        import textwrap as _tw5
        # SELF-REVIEW, twice over.
        #
        # (1) COVERAGE: the first version scanned only this function, so the
        #     defect the invariant exists to catch sat untouched one
        #     function over - test_live_path_integration() held seven
        #     literal-True checks. A guard that covers one of two suites is
        #     the same one-place-not-the-other failure it was built to stop.
        #
        # (2) PRECISION: a literal True inside an `except` handler, or in
        #     the `else` of a loop that breaks on failure, is reached ONLY
        #     when the test passed - correct, idiomatic, and not a defect.
        #     Flagging those would have forced 23 correct sites to be
        #     rewritten for the scanner's benefit, and a test that cries
        #     wolf gets silenced. The scan now reports a literal True only
        #     in straight-line code, which is exactly where all three
        #     genuine always-pass labels were found.
        def _hardcoded_checks(fn):
            _t = _ast5.parse(_tw5.dedent(_inspect.getsource(fn)))
            _safe = set()
            for _p in _ast5.walk(_t):
                _guarded = []
                if isinstance(_p, _ast5.ExceptHandler):
                    _guarded = _p.body
                elif isinstance(_p, (_ast5.For, _ast5.While)):
                    _guarded = _p.orelse
                for _st in _guarded:
                    for _c in _ast5.walk(_st):
                        _safe.add(id(_c))
            _bad = []
            for _n in _ast5.walk(_t):
                if id(_n) in _safe:
                    continue
                # Form 1: check(label, True, detail)
                if (isinstance(_n, _ast5.Call)
                        and getattr(_n.func, "id", "") == "check"
                        and len(_n.args) >= 2
                        and isinstance(_n.args[1], _ast5.Constant)
                        and _n.args[1].value is True):
                    _lab = (_n.args[0].value
                            if isinstance(_n.args[0], _ast5.Constant)
                            else "<joined string>")
                    _bad.append(str(_lab)[:60])
                # Form 2: checks.append((label, True, detail)) - the shape
                # verify() uses. The first version of this scanner only knew
                # form 1, so discovering verify() would still have missed
                # its two literal-True entries.
                if (isinstance(_n, _ast5.Tuple) and len(_n.elts) >= 2
                        and isinstance(_n.elts[1], _ast5.Constant)
                        and _n.elts[1].value is True):
                    _lab = (_n.elts[0].value
                            if isinstance(_n.elts[0], _ast5.Constant)
                            else "<joined string>")
                    _bad.append(str(_lab)[:60])
            return _bad

        # (3) COVERAGE, again: last round I extended this from one suite to
        #     three and still missed verify() - the POST-RUN verifier, which
        #     held two literal-True checks that both reviews found. Three
        #     rounds, three incomplete coverage lists. The set is now built
        #     by discovery rather than by hand, so a new test or verifier
        #     function is scanned without anyone remembering to add it.
        _suites = {}
        for _n, _o in sorted(globals().items()):
            if not callable(_o):
                continue
            if not (_n.startswith("test_") or _n == "verify"):
                continue
            try:
                _inspect.getsource(_o)
            except Exception:
                continue
            _suites[_n] = _o
        # SELF-REVIEW, fourth round running on this one guard's coverage,
        # and this time the failure was INVISIBLE: discovery reads
        # globals(), but verify() is defined in a later notebook cell than
        # the one that calls run_pipeline(), so at scan time it does not
        # exist. Discovery found nothing for it and the check PASSED -
        # reporting success for a function it never opened. A coverage
        # mechanism that cannot distinguish "clean" from "absent" is worse
        # than a hand-maintained list, because the gap is silent.
        #
        # The expected set is therefore asserted: a target that is not in
        # scope is a coverage FAILURE, not a pass. verify() is scanned where
        # it is defined (it scans itself in the verification cell), and that
        # is recorded here so the two cannot both assume the other did it.
        _expected_targets = {"test_review_round_regressions",
                             "test_live_path_integration",
                             "test_tier_reachability"}
        _not_in_scope = sorted(_expected_targets - set(_suites))
        check("The hardcoded-check scanner actually reached every expected "
              "target", not _not_in_scope,
              f"not in scope at scan time, so NOT scanned: {_not_in_scope}. "
              f"Discovery that silently finds nothing reports success for a "
              f"function it never opened."
              if _not_in_scope else
              f"scanned: {sorted(_suites)}")
        _hard_by_suite = {n: _hardcoded_checks(f)
                          for n, f in _suites.items()}
        _hard_total = sum(len(v) for v in _hard_by_suite.values())
        check("No check in ANY test suite is hardcoded to pass in "
              "straight-line code",
              _hard_total == 0,
              f"{_hard_total} unconditional check(s): "
              f"{ {k: v for k, v in _hard_by_suite.items() if v} }. A test "
              f"with a literal True is a label, not a test, and is how a "
              f"stale archive shipped beside a passing exclusion test"
              if _hard_total else
              f"{len(_suites)} suite(s) scanned, "
              f"{sum(1 for _ in _suites)} covered")
    except Exception as e:
        check("hardcoded-check scan executed", False, repr(e))

    # ---- One evaluation population across every benchmark statistic ----
    try:
        _t13b = require_output(
            DIRS["tables"] / "table13_baseline_comparison.csv",
            "the primary benchmark comparison")
        if True:
            _d13b = pd.read_csv(_t13b)
            _need = ("median_benchmark_percentile_common_population",
                     "heldout_recall_at_25_common_population",
                     "n_reference_genes_all", "n_reference_genes_common")
            _absent = [c for c in _need if c not in _d13b.columns]
            check("Percentile AND recall are both reported on the common "
                  "population, with the reference sizes",
                  not _absent,
                  f"missing columns: {_absent}" if _absent
                  else f"{len(_d13b)} method(s)")
            if not _absent:
                _ens13 = _d13b[_d13b.method.str.startswith("PU ensemble")]
                if len(_ens13):
                    check("The common population is smaller than the "
                          "all-gene population by the training positives",
                          int(_ens13.n_reference_genes_all.iloc[0])
                          > int(_ens13.n_reference_genes_common.iloc[0]),
                          f"{int(_ens13.n_reference_genes_all.iloc[0])} -> "
                          f"{int(_ens13.n_reference_genes_common.iloc[0])}")
        _fsrc = _inspect.getsource(fig_baselines)
        check("The primary-endpoint figure plots the primary metric",
              "median_benchmark_percentile_common_population" in _fsrc,
              "the panel was titled 'Primary endpoint' while plotting the "
              "all-gene metric, contradicting run_summary.json")
        _stsrc = _inspect.getsource(ranking_stability)
        check("The stability spread uses the same population as the gain",
              "_common_sb" in _stsrc,
              "a spread on 2,792 genes was compared with a deficit on 2,735")
        check("The spread is described as descriptive variability, not "
              "uncertainty", "not a confidence interval" in _stsrc,
              "a two-configuration range cannot establish statistical "
              "indistinguishability")
    except Exception as e:
        check("evaluation population checks executed", False, repr(e))

    # ---- Patient-level helpers require a patient identifier ------------
    try:
        _nopat = pd.DataFrame({"sample": ["S1", "S2"],
                               "surv_time": [10.0, 20.0],
                               "surv_event": [1, 0]})
        check("A patient-level helper REFUSES a table with no patient "
              "column",
              _raises(lambda: enforce_one_sample_per_patient(_nopat, "probe"),
                      ValueError),
              "a silent no-op let sample rows be reported as patients")
        check("The specimen selector refuses it too",
              _raises(lambda: select_one_specimen_per_patient(
                  _nopat, "probe"), ValueError))
    except Exception as e:
        check("patient-identifier guard checks executed", False, repr(e))

    # ---- BOTH verdict conditions on ONE population ---------------------
    # Self-review: the margin condition was moved to the common population
    # last round and the top-decile condition was left on all genes, so a
    # single prespecified rule was evaluated across two populations.
    try:
        _t13c = require_output(
            DIRS["tables"] / "table13_baseline_comparison.csv",
            "the primary benchmark comparison")
        if True:
            _d13c = pd.read_csv(_t13c)
            check("The top-decile count is reported on the common "
                  "population too",
                  "benchmarks_in_top_decile_common_population"
                  in _d13c.columns,
                  f"percentile columns: "
                  f"{[c for c in _d13c.columns if 'top_decile' in c]}")
        _vsrc2 = _inspect.getsource(simple_baselines)
        check("The verdict reads BOTH of its conditions from the common "
              "population",
              '_tdcol = "benchmarks_in_top_decile_common_population"'
              in _vsrc2 and '_pcol = "median_benchmark_percentile_common_population"'
              in _vsrc2,
              "a rule with one condition on 2,735 genes and the other on "
              "2,792 is not one rule")
        check("The all-genes top-decile gain is retained alongside",
              "top_decile_gain_all_genes" in _vsrc2)
        _fsrc2 = _inspect.getsource(fig_baselines)
        check("Both figure panels read the same population",
              "_tdcol_fig" in _fsrc2 and "_pcol_fig" in _fsrc2,
              "panel A on the common population and panel B on all genes "
              "would describe two analyses in one figure")
    except Exception as e:
        check("verdict population checks executed", False, repr(e))

    # ---- Every MANDATORY artifact exists -------------------------------
    # Both external reviews: the suites are full of `if table.exists():`
    # guards with no else, so a missing required output makes its
    # assertions vanish. Converting all eleven would still leave the next
    # one unguarded, so the mandatory set is DECLARED here and checked as a
    # whole - a missing output fails loudly whatever the individual guards
    # do, and anything not on this list is optional by explicit decision
    # rather than by omission.
    try:
        _mandatory = {
            "table1_filter_attrition.csv": "the filter attrition record",
            "table2_full_candidate_ranking.csv": "the candidate ranking",
            "table13_baseline_comparison.csv": "the primary comparison",
            "table14_stop_go_matrix.csv": "the advancement decision",
            "table19_modality_expression_screen.csv": "the modality screen",
            "table22_frozen_model_manifest.csv": "the frozen model record",
        }
        # SELF-REVIEW: the mandatory list covered only tables, so removing
        # the modification-time fallback dropped 03_models/
        # frozen_ensemble.joblib from the archive without any check
        # noticing - a required artifact that had been riding on the
        # fallback. Non-table release artifacts are declared too.
        _mandatory_other = {
            DIRS["models"] / "frozen_ensemble.joblib":
                "the frozen scoring model",
            DIRS["logs"] / "filter_unit_tests.csv":
                "the filter unit-test report",
            DIRS["logs"] / "run_summary.json": "the run summary",
            DIRS["logs"] / "run_manifest.json":
                "the reproducibility manifest (package versions, input "
                "checksums, cohort sample lists, resolved configuration)",
        }
        # Artifacts that are written AFTER this suite, because they record
        # the suite's own results. Declared once and used by BOTH the
        # existence check below and the registration check further down -
        # the first attempt at adding run_manifest.json exempted it from
        # registration only, so a run in a FRESH output directory failed the
        # existence check instead. (It passed here, because /home/claude
        # held a copy from the previous run - the stale-file trap again. The
        # extracted-archive run in an empty directory is what caught it.)
        #
        # The exemption is only safe because verify_release_archive()
        # asserts the membership of every name in this set against the
        # finished ZIP. Nothing may be added here without a matching entry
        # in that assertion.
        _written_after_this_suite = set(WRITTEN_AFTER_TEST_SUITE)
        _missing_mand = []
        for _fn, _why in _mandatory.items():
            try:
                require_output(DIRS["tables"] / _fn, _why)
            except AssertionError as _e:
                _missing_mand.append(str(_e))
        for _pth, _why in _mandatory_other.items():
            if Path(_pth).name in _written_after_this_suite:
                continue
            try:
                require_output(_pth, _why)
            except AssertionError as _e:
                _missing_mand.append(str(_e))
        # A mandatory artifact must also be REGISTERED, or it will not reach
        # the archive now that the fallback is gone.
        # SELF-REVIEW: run_summary.json is written AND registered after this
        # suite runs (the suite's own results go into it), so at scan time
        # the only copy on disk belongs to the previous run. Checking its
        # registration here compares against a stale file - the same
        # ordering mistake as the stale-table check two rounds ago. It is
        # excluded by name, with the reason stated, rather than silently
        # passing.
        # run_manifest.json joins this exemption for exactly the same
        # reason, and the exemption is REQUIRED to be narrow: both files are
        # written after this suite because both record the suite's results,
        # and both have a dedicated post-packaging membership assertion in
        # verify_release_archive() that reads the finished ZIP. An artifact
        # exempted here without that archive-level assertion would have no
        # check at all - which is precisely how run_manifest.json went
        # missing from every release in the first place.
        _registered_after_this_check = _written_after_this_suite
        _reg_resolved = {str(Path(r).resolve()) for r in ARTIFACT_REGISTRY}
        _unreg_mand = [p.name for p in _mandatory_other
                       if p.is_file()
                       and p.name not in _registered_after_this_check
                       and str(p.resolve()) not in _reg_resolved]
        check("Every mandatory non-table artifact is REGISTERED for this "
              "run, so it reaches the archive",
              not _unreg_mand,
              f"present on disk but unregistered: {_unreg_mand}"
              if _unreg_mand else
              f"{len(_mandatory_other)} artifact(s) registered")
        check("Every declared mandatory output exists and is non-empty",
              not _missing_mand,
              "; ".join(_missing_mand)[:200] if _missing_mand
              else f"{len(_mandatory)} mandatory artifact(s) present")
        check("require_output() fails on an absent file rather than "
              "returning quietly",
              _raises(lambda: require_output(
                  DIRS["tables"] / "no_such_table_xyz.csv"),
                  AssertionError),
              "a guard that can only skip is not a guard")
    except Exception as e:
        check("mandatory artifact checks executed", False, repr(e))

    # ---- The release gate fails closed ---------------------------------
    # Both reviews' top P0: the suites ran, failures were recorded, COMPLETE
    # was printed and the archive was built regardless. The checksum
    # machinery proves the files match the manifest; it proves nothing about
    # whether the code that made them passed its tests.
    try:
        _ok_df = pd.DataFrame({"test": ["a", "b"], "passed": [True, True]})
        check("A fully passing suite is approved",
              require_all_tests_pass(_ok_df, "probe") is True)
        _fail_df = pd.DataFrame({"test": ["a", "b"], "passed": [True, False]})
        check("A single failing test BLOCKS the release",
              _raises(lambda: require_all_tests_pass(_fail_df, "probe"),
                      RuntimeError),
              "a failed test must stop packaging, not annotate it")
        check("An EMPTY suite blocks the release",
              _raises(lambda: require_all_tests_pass(
                  pd.DataFrame(columns=["test", "passed"]), "probe"),
                  RuntimeError),
              "'no tests executed' must not read as 'nothing failed'")
        check("An unrecognised result schema blocks the release",
              _raises(lambda: require_all_tests_pass(
                  pd.DataFrame({"x": [1]}), "probe"), RuntimeError))
        _esrc11 = _inspect.getsource(export_outputs)
        # SELF-REVIEW: this asserted the exporter referenced
        # RELEASE_APPROVAL - the boolean flag that the state machine
        # replaced. Updated to the stronger contract rather than kept green
        # by restoring the weaker mechanism.
        check("The exporter requires a VERIFIED run, not merely passing "
              "tests", "release_is_packageable()" in _esrc11,
              "a direct call to export_outputs() must not bypass the gate")
        _rsrc11 = _inspect.getsource(run_pipeline)
        check("The pipeline gates on BOTH suites before packaging",
              _rsrc11.count("require_all_tests_pass(") >= 2,
              "one gated suite and one ungated suite is one ungated suite")
        check("A blocked run records WHY in the summary",
              "release_block_reason" in _rsrc11,
              "a blocked release must be diagnosable from the archive")
    except Exception as e:
        check("release gate checks executed", False, repr(e))

    # ---- Release approval is per-run, not per-session ------------------
    # Self-review: the artifact registry is reset at the start of every run
    # precisely so a second invocation cannot inherit the first's state. I
    # added RELEASE_APPROVAL last round without that treatment, so a run
    # that raised before reaching the gate still carried the previous run's
    # approval, and a direct export_outputs() call would have packaged its
    # partial outputs as validated.
    try:
        _rp = _inspect.getsource(run_pipeline)
        check("Release approval is cleared at the start of every run",
              "RELEASE_APPROVAL.update({" in _rp
              and "tests have not yet run for this invocation" in _rp,
              "an approval inherited from a previous run is not an approval")
        _ep = _inspect.getsource(export_outputs)
        check("The exporter refuses a verification recorded under a "
              "DIFFERENT run",
              "release_is_packageable()" in _ep,
              "release_is_packageable() compares the recorded run_id with "
              "the current one, so verification is not transferable")
        # Behavioural: simulate a stale approval and confirm it is refused.
        _saved_ra = dict(RELEASE_APPROVAL)
        try:
            # SELF-REVIEW: these drove the old RELEASE_APPROVAL flag, which
            # the state machine replaces. Rewritten against the state the
            # exporter now consults, so they test the live contract.
            _saved_rs2 = dict(RELEASE_STATE)
            try:
                RELEASE_STATE.update({"status": "VERIFIED", "verified": True,
                                      "run_id": "a-different-run",
                                      "reason": "probe"})
                check("A verification recorded under an earlier run BLOCKS "
                      "packaging",
                      _raises(lambda: export_outputs(download=False,
                                                     to_drive=False),
                              RuntimeError),
                      "run 1's verification must not authorise run 2")
                RELEASE_STATE.update({"status": "TESTED", "verified": False,
                                      "run_id": RUN_ID, "reason": "probe"})
                check("A run that passed tests but is NOT verified BLOCKS "
                      "packaging",
                      _raises(lambda: export_outputs(download=False,
                                                     to_drive=False),
                              RuntimeError))
            finally:
                RELEASE_STATE.clear()
                RELEASE_STATE.update(_saved_rs2)
        finally:
            RELEASE_APPROVAL.clear()
            RELEASE_APPROVAL.update(_saved_ra)
    except Exception as e:
        check("release approval scope checks executed", False, repr(e))

    # ---- The release STATE MACHINE, not a boolean -----------------------
    # Both external reviews, as the architectural correction: approval was a
    # flag set when the test suites passed, i.e. BEFORE post-run
    # verification, and a failed verify() never revoked it. These use their
    # own failure injections.
    try:
        _saved_rs = dict(RELEASE_STATE)
        try:
            # Tests passing must land on TESTED, which is NOT packageable.
            RELEASE_STATE.update({"status": "RUNNING", "run_id": RUN_ID,
                                  "verified": False, "invocation": 1})
            release_transition("TESTED", "probe", require_from="RUNNING")
            check("Passing the test suites alone is NOT packageable",
                  not release_is_packageable(),
                  f"state {RELEASE_STATE['status']} must not authorise "
                  f"packaging before verification")
            check("A direct export after passing tests but BEFORE "
                  "verification is refused",
                  _raises(lambda: export_outputs(download=False,
                                                 to_drive=False),
                          RuntimeError),
                  "this is the gap both reviews reproduced")
            # A failed verification must block, not leave approval standing.
            mark_verified(False, "probe failure")
            check("A failed post-run verification BLOCKS the release",
                  RELEASE_STATE["status"] == "BLOCKED"
                  and not release_is_packageable(),
                  f"state {RELEASE_STATE['status']}")
            check("A blocked run cannot be packaged",
                  _raises(lambda: export_outputs(download=False,
                                                 to_drive=False),
                          RuntimeError))
            # Only a successful verification makes it packageable.
            RELEASE_STATE.update({"status": "TESTED", "run_id": RUN_ID})
            mark_verified(True, "probe pass")
            check("A verified run IS packageable",
                  release_is_packageable()
                  and RELEASE_STATE["status"] == "VERIFIED")
            # Out-of-order transitions are themselves defects.
            check("Verification cannot be claimed from a RUNNING state",
                  _raises(lambda: (RELEASE_STATE.update({"status": "RUNNING"}),
                                   release_transition("VERIFIED", "x",
                                                      require_from="TESTED")),
                          RuntimeError),
                  "a stage running without its prerequisite is a defect")
            # A verification from another run must not authorise this one.
            RELEASE_STATE.update({"status": "VERIFIED", "verified": True,
                                  "run_id": "some-other-run"})
            check("A verification recorded under a DIFFERENT run_id is "
                  "refused", not release_is_packageable(),
                  "verification is not transferable between runs")
        finally:
            RELEASE_STATE.clear()
            RELEASE_STATE.update(_saved_rs)
    except Exception as e:
        check("release state machine checks executed", False, repr(e))

    # ---- Strict boolean test schema ------------------------------------
    try:
        check("passed=False blocks",
              _raises(lambda: require_all_tests_pass(
                  pd.DataFrame({"test": ["a"], "passed": [False]}), "probe"),
                  RuntimeError))
        check('passed="False" is REFUSED as a type error, not read as a pass',
              _raises(lambda: require_all_tests_pass(
                  pd.DataFrame({"test": ["a"], "passed": ["False"]}),
                  "probe"), TypeError),
              "a non-empty string is truthy in Python, so .astype(bool) "
              "counted \"False\" as a passing test")
        for _v, _lab in (("True", 'passed="True"'), (1, "passed=1"),
                         (None, "passed=None")):
            if not _raises(lambda _v=_v: require_all_tests_pass(
                    pd.DataFrame({"test": ["a"], "passed": [_v]}), "probe"),
                    (TypeError, RuntimeError)):
                check(f"{_lab} is refused", False,
                      "only real booleans may decide a release")
                break
        else:
            check("Only real booleans are accepted in the 'passed' column",
                  True, "string, int and None forms all refused")
        check("A genuinely passing suite is accepted",
              require_all_tests_pass(
                  pd.DataFrame({"test": ["a"], "passed": [True]}),
                  "probe") is True)
    except Exception as e:
        check("test schema checks executed", False, repr(e))

    # ---- No modification-time fallback into the archive ----------------
    try:
        _xsrc = _inspect.getsource(export_outputs)
        check("Archive membership comes from the registry alone, with no "
              "modification-time fallback",
              "st_mtime >= RUN_START_TIME" not in _xsrc
              and "unregistered file(s) NOT archived" in _xsrc,
              "RUN_START_TIME is bound at import, so on a second run every "
              "file from the first run satisfies it")
    except Exception as e:
        check("archive membership checks executed", False, repr(e))

    # ---- Duplicate archive members are refused -------------------------
    try:
        import zipfile as _zf3
        _ddir = Path(_tf2.mkdtemp(prefix=f"dupzip_{RUN_ID}_"))
        _dz = _ddir / "dup.zip"
        import hashlib as _h3
        _payload = b"a,b\n1,2\n"
        _man = pd.DataFrame([{
            "run_id": RUN_ID, "archived_file": "05_tables/t.csv",
            "bytes": len(_payload),
            "sha256": _h3.sha256(_payload).hexdigest()}])
        with _zf3.ZipFile(_dz, "w") as _z:
            _z.writestr("05_tables/t.csv", _payload)
            _z.writestr("05_tables/t.csv", _payload)   # deliberate duplicate
            _z.writestr("06_logs/archive_contents.csv",
                        _man.to_csv(index=False))
        check("A ZIP with duplicate member names is REFUSED even when the "
              "manifest and checksums are valid",
              _raises(lambda: verify_release_archive(_dz), RuntimeError),
              "a set of names collapses duplicates, so extraction order "
              "decided which copy a reader received")
        shutil.rmtree(_ddir, ignore_errors=True)
    except Exception as e:
        check("duplicate archive checks executed", False, repr(e))

    # ---- The sensitivity note carries numbers, not variable names ------
    try:
        # SELF-REVIEW: the first version searched the SOURCE for the banned
        # phrases - and my own comments quote them to explain the fix, so
        # both assertions failed against corrected code. That is the seventh
        # source-text assertion fooled by surrounding prose this session, so
        # the rule stands: structural claims are made on the AST, where a
        # comment cannot appear. This helper does it for any function.
        def _string_constants(fn):
            _t = _ast5.parse(_tw5.dedent(_inspect.getsource(fn)))
            return [n.value for n in _ast5.walk(_t)
                    if isinstance(n, _ast5.Constant)
                    and isinstance(n.value, str)]

        _bstr = " ".join(_string_constants(simple_baselines))
        check("The sensitivity note interpolates the measured spread, not "
              "the name of the variable holding it",
              "ensemble_percentile_spread points" not in _bstr,
              "the exported note told the reader a variable name")
        _sstr = " ".join(_string_constants(ranking_stability))
        check("The noise note no longer claims a resolution limit",
              "not resolvable by this design" not in _sstr
              and "not a resolution limit" in _sstr,
              "two configurations give a range for the ensemble percentile, "
              "not an interval for the difference between methods")
    except Exception as e:
        check("sensitivity wording checks executed", False, repr(e))

    # ---- The Colab setup contract --------------------------------------
    # The notebook is run on Colab, so the setup cell is part of the
    # deliverable: if it omits a package the pipeline imports, or the
    # cache-redirection env vars are not honoured, a LIVE run fails in a way
    # no DEMO test would see.
    try:
        _nb_path = Path("outputs_deliverables/prostate_surfaceome_pipeline.ipynb")
        _third_party = {"lifelines", "pydeseq2", "cptac"}
        _setup_ok, _detail = True, ""
        if _nb_path.is_file():
            _nbj = json.loads(_nb_path.read_text())
            _setup = "".join(
                "".join(c.get("source", [])) for c in _nbj["cells"][:6])
            _absent = [p for p in _third_party if p not in _setup]
            _setup_ok = not _absent
            _detail = (f"not installed by the setup cell: {_absent}"
                       if _absent else
                       f"{len(_third_party)} non-default package(s) handled")
        else:
            _detail = "notebook not built in this working directory"
        check("The Colab setup cell installs every package Colab lacks",
              _setup_ok, _detail)
        # The redirection the setup cell performs must actually be read.
        check("The raw-download cache honours SURFACEOME_RAW_DIR, so a "
              "dropped Colab session resumes instead of re-downloading",
              str(DIRS["raw"]) == os.environ.get(
                  "SURFACEOME_RAW_DIR", str(DIRS["raw"])),
              f"raw dir {DIRS['raw']}")
        for _ev in ("SURFACEOME_RAW_DIR", "SURFACEOME_OUT_DIR"):
            _saved_ev = os.environ.get(_ev)
            try:
                os.environ[_ev] = "/tmp/_probe_cache"
                _probe_ok = (Path(os.environ.get("SURFACEOME_RAW_DIR",
                                                 "outputs/00_raw"))
                             == Path("/tmp/_probe_cache")) or _ev.endswith(
                    "OUT_DIR")
                check(f"{_ev} is read from the environment", _probe_ok,
                      "the setup cell sets it; Section 0 must consume it")
            finally:
                if _saved_ev is None:
                    os.environ.pop(_ev, None)
                else:
                    os.environ[_ev] = _saved_ev
    except Exception as e:
        check("Colab setup checks executed", False, repr(e))

    # ---- The archived source runs as a plain script --------------------
    # Found by extracting the delivered ZIP and running it the obvious way:
    # the `if __name__ == "__main__"` entry point sat ~500 lines ABOVE the
    # release-state globals it depends on, so running the module as a script
    # died with NameError. Every development route reordered it by accident
    # - run_test.py strips the block, the notebook puts the call in a later
    # cell - so the defect was invisible until an independent reproduction
    # was actually attempted.
    try:
        import ast as _ast6
        import textwrap as _tw6
        _mod_file = None
        # SELF-REVIEW: the first list held only the development path and the
        # archived copy under the output tree. Run from an EXTRACTED
        # archive - the independent-reproduction route - the module sits in
        # the working directory instead, so the check reported "not found"
        # and failed. It fails rather than skips by design, which is how
        # this surfaced; the list simply needed the third location.
        for _p in (Path("src/s07_main.py"),
                   DIRS["logs"] / "pipeline_source" / "s07_main.py",
                   Path("s07_main.py"),
                   Path(__file__).parent / "s07_main.py"
                   if "__file__" in dir() else Path("s07_main.py")):
            if _p.is_file():
                _mod_file = _p
                break
        # BUG FIX, found by actually executing the delivered NOTEBOOK rather
        # than only the flat-module / archived-script paths this check
        # already covered. In a fresh Colab run this check executes from
        # INSIDE run_pipeline(), before export_outputs() has ever run - so
        # DIRS["logs"]/"pipeline_source"/"s07_main.py" does not exist yet,
        # that copy is only written by the later export stage - and
        # src/s07_main.py / ./s07_main.py / a __file__-relative path do not
        # exist either, because the notebook never wrote a copy of itself
        # to disk at all. Every development and archived-script route
        # happens to have a real file on disk at the moment this runs; the
        # one route that matters most - a user running the shipped notebook
        # top to bottom - does not, so this check failed on every single
        # correct Colab run, for a reason that had nothing to do with
        # entry-point placement.
        #
        # A first attempt grabbed the executing cell's own text from the
        # IPython input history instead. That is the wrong artifact to
        # check: build_notebook.py deliberately STRIPS the
        # `if __name__ == "__main__":` block out of the pipeline cell and
        # calls run_pipeline() directly as the cell's last line, because a
        # notebook has no "run as a script" ambiguity to guard against. The
        # executing cell's text can never contain that block, so grabbing
        # it would make this check fail a *different* assertion
        # ("exactly one script entry point") every time, for a structural
        # property the notebook was never meant to have in the first place.
        #
        # What this check actually needs to validate is the canonical
        # s07_main.py as it will be SHIPPED - the same file export_outputs()
        # later copies into the archive and the same file run as a script
        # outside any notebook - not whatever text happens to be executing
        # right now. The notebook builder embeds that exact text, verbatim,
        # as PIPELINE_SOURCE_FOR_SELFCHECK (see build_notebook.py) before
        # stripping anything for execution, so read that single canonical
        # declaration first; every other path below exists only for contexts
        # where the notebook builder had no chance to set it.
        _src_text = globals().get("PIPELINE_SOURCE_FOR_SELFCHECK")
        if not isinstance(_src_text, str) or not _src_text.strip():
            _src_text = _mod_file.read_text() if _mod_file is not None \
                else None
        if _src_text is None:
            check("The pipeline module source is available to check its "
                  "entry-point position", False,
                  "neither PIPELINE_SOURCE_FOR_SELFCHECK nor a source file "
                  "(src/, archived copy, or an extracted script) was found")
        else:
            _tree = _ast6.parse(_src_text)
            _main_guards = [
                n for n in _tree.body
                if isinstance(n, _ast6.If)
                and _ast6.dump(n.test).find("__main__") != -1]
            check("The module has exactly one script entry point",
                  len(_main_guards) == 1,
                  f"{len(_main_guards)} `if __name__` block(s)")
            if _main_guards:
                # SELF-REVIEW: first named `_guard`, which SHADOWED the
                # suppress_outputs context manager already bound to that
                # name earlier in this function - its __exit__ then resolved
                # to an ast.If node. A long test function is exactly where
                # a generic local name collides.
                _entry_guard = _main_guards[0]
                _assigns_after = [
                    n for n in _tree.body
                    if getattr(n, "lineno", 0) > _entry_guard.lineno
                    and isinstance(n, (_ast6.Assign, _ast6.FunctionDef,
                                       _ast6.ClassDef))]
                check("The script entry point is the LAST statement, so a "
                      "plain `python s07_main.py` cannot call the pipeline "
                      "before its globals exist",
                      not _assigns_after,
                      f"{len(_assigns_after)} definition(s)/assignment(s) "
                      f"follow it, e.g. line "
                      f"{getattr(_assigns_after[0], 'lineno', '?')} - "
                      f"running the module as a script would use them "
                      f"before they are bound"
                      if _assigns_after else
                      f"entry point at line {_entry_guard.lineno}, "
                      f"nothing after")
    except Exception as e:
        check("entry-point position checks executed", False, repr(e))

    # ---- Coverage threshold granularity --------------------------------
    # The config reads like a 10% tolerance. At 9 organs it is not one:
    # 8/9 = 88.9% fails. This asserts the EFFECT rather than the number, so
    # editing the organ list cannot silently open a tolerance nobody chose.
    try:
        _n_org = len(set(CFG.vital_organs))
        _max_missing = sum(
            1 for k in range(_n_org)
            if (_n_org - k - 1) / _n_org >= CFG.min_vital_organ_coverage)
        check("Vital-organ coverage policy permits zero missing organs",
              _max_missing == 0,
              f"{_n_org} organs at {CFG.min_vital_organ_coverage:.0%} "
              f"permits {_max_missing} missing - if that is not intended, "
              f"set the threshold deliberately")
    except Exception as e:
        check("coverage granularity check executed", False, repr(e))

    _guard_active = writes_suppressed()
    _probe_name = "table99_write_guard_probe.csv"
    _probe_path = DIRS["tables"] / _probe_name
    _pre_existing = _probe_path.exists()
    save_table(pd.DataFrame({"must_not": ["be written"]}), _probe_name,
               show=False)
    _blocked = (not _probe_path.exists()) and not _pre_existing
    _guard.__exit__(None, None, None)
    check("Production-write guard is active during the test suite",
          _guard_active, f"writes_suppressed() = {_guard_active}")
    check("Production-write guard physically blocks a write attempt",
          _blocked,
          "probe table was not created inside the guard" if _blocked
          else f"PROBE FILE WAS WRITTEN at {_probe_path}")
    # BUG FIX (external review, correct): also hardcoded True, while its
    # own diagnostic text said an unexpected suppressed write should mean
    # failure. Exactly one suppressed write is expected - the deliberate
    # guard probe - so anything beyond that is a call site relying on the
    # guard instead of passing save_outputs=False.
    _unexpected_sw = [n for _, n in SUPPRESSED_WRITE_LOG
                      if "probe" not in str(n)]
    check("Explicit save_outputs=False prevented test writes reaching disk",
          not _unexpected_sw,
          f"{len(SUPPRESSED_WRITE_LOG)} write(s) caught by the guard; "
          f"exactly 1 is expected - the deliberate probe above. More than "
          f"that means a call site is missing its explicit save_outputs="
          f"False and is relying on the guard alone: "
          f"{[n for _, n in SUPPRESSED_WRITE_LOG if 'probe' not in str(n)]}")

    # ----------------------------------------------------------------
    # ROUND 25: three external reviews of revisions 23/24, Colab + LIVE
    # ----------------------------------------------------------------
    # The finding that mattered most: run_manifest.json was WRITTEN but
    # never REGISTERED, so the registry-only packager dropped the single
    # most important reproducibility artifact from every release while
    # run_summary.json recorded "run_manifest_written": true. The archive
    # checksum audit cannot catch this, because an artifact that was never
    # registered is absent from BOTH the archive and its manifest - so the
    # reconciliation finds nothing to disagree about. This is the same
    # failure shape as the modification-time fallback removed earlier: a
    # mechanism that cannot tell "clean" from "absent".
    import inspect as _insp25
    import textwrap as _tw25
    import tempfile as _tmp25
    import ast as _ast25
    _wrm = _string_constants(write_run_manifest)
    check("write_run_manifest REGISTERS the manifest, so the registry-only "
          "packager ships it",
          "run_manifest" in _wrm or any(
              isinstance(_n, _ast25.Call)
              and getattr(_n.func, "id", "") == "register_artifact"
              for _n in _ast25.walk(_ast25.parse(
                  _tw25.dedent(_insp25.getsource(write_run_manifest))))),
          "register_artifact() call present in write_run_manifest")
    _vra_src = _insp25.getsource(verify_release_archive)
    # The exemption and its guarantee are tied together HERE, so the
    # exemption cannot outgrow the thing that makes it safe. Without this,
    # adding a name to WRITTEN_AFTER_TEST_SUITE silently removes every
    # check on that artifact.
    _unguaranteed = [_n for _n in WRITTEN_AFTER_TEST_SUITE
                     if not any(Path(_k).name == _n
                                for _k in RELEASE_REQUIRED_MEMBERS)]
    check("Every artifact exempted from the pre-packaging checks is "
          "asserted against the FINISHED archive instead",
          not _unguaranteed,
          f"exempted with NO archive-level check: {_unguaranteed}"
          if _unguaranteed else
          f"all {len(WRITTEN_AFTER_TEST_SUITE)} exempted artifact(s) are "
          f"required members of the release archive")
    check("The archive verifier asserts release-critical MEMBERSHIP from a "
          "list of its own, not only manifest agreement",
          "run_manifest.json" in _vra_src
          and "_required_members" in _vra_src,
          "membership of run_manifest/run_summary/lockfile/runner asserted")
    # Checked against the actual declaration object, not against this
    # function's own source text - grepping my own test for the string it
    # asserts would pass no matter what the pipeline declares. The manifest
    # cannot be checked in the ARTIFACT_REGISTRY here, because
    # write_run_manifest() runs AFTER this suite (the suite's own results go
    # into the manifest), so at this point the only copy on disk belongs to
    # the previous run - the same ordering caveat already documented for
    # run_summary.json.
    check("run_manifest.json is declared a mandatory release artifact",
          any(Path(_k).name == "run_manifest.json"
              for _k in _mandatory_other),
          f"mandatory non-table artifacts: "
          f"{sorted(Path(_k).name for _k in _mandatory_other)}")

    # The notebook setup cell hardcoded PROFILE=smoke into the environment,
    # overriding the pipeline's own correct LIVE->standard default. A user
    # who set RUN_MODE=LIVE and nothing else got a reportable-looking LIVE
    # run on 3 bags and 30 permutations.
    check("'full' is not an accepted profile",
          "full" not in _PROFILES
          and set(_PROFILES) == {"smoke", "standard", "manuscript"},
          f"valid profiles: {sorted(_PROFILES)}")
    check("The standalone runner does NOT hardcode a PROFILE default, so "
          "the pipeline's LIVE->standard default survives",
          'setdefault("PROFILE"' not in _insp25.getsource(export_outputs),
          "no PROFILE setdefault emitted into run_pipeline.py")
    check("The standalone runner's usage line names the real profiles",
          "smoke|standard|manuscript" in _insp25.getsource(export_outputs),
          "usage text lists the three valid profiles")

    # A LIVE run on the smoke profile is a software diagnostic. It used to
    # be silently permitted.
    check("A LIVE smoke run is refused unless explicitly opted into",
          "ALLOW_SMOKE_LIVE" in _insp25.getsource(validate_live_configuration),
          "validate_live_configuration gates smoke+LIVE")

    # Configuration contracts were validated inside load_clinical_live(),
    # the FOURTH acquisition step, so an invalid endpoint cost a
    # multi-gigabyte Toil download before being detected.
    _rp_src = _insp25.getsource(run_pipeline)
    _i_cfg = _rp_src.find("validate_live_configuration(")
    _i_acq = min([i for i in (_rp_src.find("load_protein_annotation("),
                              _rp_src.find("load_transcriptomes(")) if i > 0]
                 or [10 ** 9])
    check("Configuration is validated BEFORE the first acquisition call",
          0 < _i_cfg < _i_acq,
          f"validate_live_configuration at {_i_cfg}, first loader at {_i_acq}")

    # download(): only the HEAD probe was wrapped. The streaming GET and
    # raise_for_status() - the failures that actually happen - escaped raw.
    _dl_src = _insp25.getsource(download)
    check("The streaming GET is inside an exception handler, not only HEAD",
          _dl_src.find("for attempt in range") < _dl_src.find("requests.get("),
          "GET runs inside the bounded-retry try block")
    check("A permanent HTTP status is not retried",
          "_HTTP_PERMANENT" in _dl_src and 403 in _HTTP_PERMANENT,
          f"permanent statuses: {sorted(_HTTP_PERMANENT)}")
    check("A failed HEAD probe no longer aborts the download",
          "continuing to GET" in _dl_src,
          "HEAD is advisory; some object stores answer it incorrectly")

    # Cache reuse was gated on size alone.
    _tmpc = Path(_tmp25.mkdtemp(prefix="cacheval_"))
    try:
        _junk = _tmpc / "fake.gz"
        _junk.write_bytes(b"x" * 50_000)
        _ok_j, _why_j = validate_cached_input(_junk, 10_000)
        check("A large JUNK file is rejected as a cache hit, not reused",
              not _ok_j, f"rejected: {_why_j}")
        _html = _tmpc / "page.zip"
        _html.write_bytes(b"<html><body>Access Denied</body></html>"
                          + b" " * 50_000)
        _ok_h, _why_h = validate_cached_input(_html, 10_000)
        check("An HTML error page saved under a data filename is rejected",
              not _ok_h, f"rejected: {_why_h}")
        _good = _tmpc / "real.gz"
        with gzip.open(_good, "wb") as _fh:
            # NOTE on the fixture: the first version gzipped 20,000
            # identical lines, which compresses to 135 bytes and was
            # rejected for being below the size floor - the test failed
            # against correct code. Incompressible bytes instead.
            _fh.write(os.urandom(40_000))
        _ok_g, _why_g = validate_cached_input(_good, 10_000)
        check("A VALID gzip cache is still accepted, so the check is not "
              "merely refusing everything",
              _ok_g, f"accepted: {_why_g}")
        _trunc = _tmpc / "trunc.gz"
        _trunc.write_bytes(_good.read_bytes()[:len(_good.read_bytes()) // 3])
        _ok_t, _why_t = validate_cached_input(_trunc, 10_000)
        check("A TRUNCATED gzip cache is rejected",
              not _ok_t, f"rejected: {_why_t}")
        _sc = _tmpc / "sums.gz"
        _sh2 = __import__("shutil")
        _sh2.copy(_good, _sc)
        _sc.with_suffix(".gz.sha256").write_text("deadbeef  sums.gz\\n")
        _ok_s, _why_s = validate_cached_input(_sc, 10_000)
        check("A cache whose sidecar checksum disagrees is rejected",
              not _ok_s, f"rejected: {_why_s}")
    finally:
        __import__("shutil").rmtree(_tmpc, ignore_errors=True)

    # UniProt defines the GENE UNIVERSE and was neither cached nor
    # checksummed: two runs months apart analysed different gene sets with
    # nothing in the archive to show it.
    _up_src = _insp25.getsource(load_surfaceome_live)
    check("The UniProt gene universe is cached and keyed on the query",
          "query_sha256" in _up_src and "cache.is_file()" in _up_src,
          "cache invalidates when the query changes")
    check("The UniProt gene universe is registered as an analysis input",
          "register_input_file" in _up_src,
          "surfaceome definition appears in the input checksum manifest")
    # Asserted on the AST, not by comparing string offsets. The first
    # attempt compared find("raise_for_status") with find("except Exception")
    # and failed against CORRECT code, because the cache-reuse branch earlier
    # in the function has an except of its own - a text-ordering test cannot
    # express "inside the handler that converts network errors".
    def _guarded_by_network_failure(fn):
        _t = _ast25.parse(_tw25.dedent(_insp25.getsource(fn)))
        for _node in _ast25.walk(_t):
            if not isinstance(_node, _ast25.Try):
                continue
            _converts = any(
                getattr(getattr(_c, "func", None), "id", "")
                == "_network_failure"
                for _h in _node.handlers for _c in _ast25.walk(_h))
            if not _converts:
                continue
            _calls = {getattr(_c.func, "attr", "")
                      for _c in _ast25.walk(_node)
                      if isinstance(_c, _ast25.Call)}
            if "raise_for_status" in _calls and "get" in _calls:
                return True
        return False
    check("UniProt's GET and raise_for_status() are both inside the handler "
          "that produces the actionable network message",
          _guarded_by_network_failure(load_surfaceome_live),
          "an HTTP 503 from UniProt no longer escapes as a bare HTTPError")
    check("An EMPTY UniProt response stops the run rather than producing an "
          "empty ranking",
          "EMPTY membrane proteome" in _up_src,
          "empty gene universe is fatal")

    # The provenance template misdescribed 12 of its own rows.
    _roles = {}
    for _g in [g for g in CLINICAL_STAGE_TARGETS if not g.endswith("_HELD")]:
        _roles[_g] = _template_role_of(_g)
    _n_tp = sum(1 for v in _roles.values() if v == "training_positive")
    check("The provenance template marks exactly the genes that TRAIN the "
          "primary model as training_positive",
          _n_tp == len([g for g in PAYLOAD_DELIVERY_TARGETS
                        if g not in set(PROSTATE_HELDOUT_POSITIVES)]),
          f"{_n_tp} training_positive rows vs "
          f"{len(PAYLOAD_DELIVERY_TARGETS)} payload-delivery targets")
    check("Signal-blockade genes are NOT labelled training positives",
          all(_roles.get(_g, "") != "training_positive"
              for _g in SIGNAL_BLOCKADE_TARGETS
              if _g in _roles),
          f"{sum(1 for g in SIGNAL_BLOCKADE_TARGETS if _roles.get(g)== 'training_positive')} mislabelled")
    # The same defect lived in the startup banner, where it was the first
    # line of every run. Found during this round's work, not reported.
    check("The startup banner reports the PRIMARY training-set size, not "
          "the whole declared collection",
          len(_primary_pos) == _n_tp
          and len(_primary_pos) != len(
              [g for g in CLINICAL_STAGE_TARGETS
               if not g.endswith("_HELD")]),
          f"banner reports {len(_primary_pos)} primary positives; the "
          f"declared collection is "
          f"{len([g for g in CLINICAL_STAGE_TARGETS if not g.endswith('_HELD')])}")
    check("The template's role vocabulary distinguishes withheld classes "
          "from excluded ones",
          any("withheld_from_primary_model" in v for v in _roles.values()),
          f"roles emitted: {sorted(set(_roles.values()))}")

    # ---- ROUND 26: Colab end-to-end reliability and speed -------------
    # A digest computed by the cache validator was recomputed by
    # register_input_file over the identical bytes - ~5 GB hashed twice per
    # LIVE run, and through a Drive FUSE mount that is minutes.
    check("register_input_file can accept a precomputed digest, so cached "
          "inputs are hashed ONCE per run",
          "sha256=None" in _insp25.getsource(register_input_file).split(
              "\n")[0] or "sha256" in str(
              _insp25.signature(register_input_file).parameters),
          f"signature: {_insp25.signature(register_input_file)}")
    _tmph = Path(_tmp25.mkdtemp(prefix="digest_"))
    try:
        _f = _tmph / "x.bin"
        _f.write_bytes(b"abc" * 1000)
        _got = register_input_file(_f, role="probe", sha256="SENTINEL")
        check("A supplied digest is used verbatim rather than recomputed",
              _got == "SENTINEL", f"returned {_got}")
        INPUT_FILE_REGISTRY.pop(str(_f), None)
        _real = register_input_file(_f, role="probe")
        check("With no digest supplied, the real SHA-256 is still computed",
              _real == __import__("hashlib").sha256(
                  _f.read_bytes()).hexdigest(),
              "digest reuse did not break the normal path")
        INPUT_FILE_REGISTRY.pop(str(_f), None)
    finally:
        __import__("shutil").rmtree(_tmph, ignore_errors=True)

    # Nested joblib pools: a GridSearchCV with n_jobs=-1 whose estimator was
    # a RandomForest with n_jobs=-1, each worker then running threaded BLAS.
    # On 2 vCPUs that is 8 threads on 2 cores - slower AND more memory.
    _zoo_src = _insp25.getsource(model_zoo)
    check("The inner RandomForest does NOT re-parallelise inside a parallel "
          "grid search",
          "n_jobs=1" in _zoo_src and "n_jobs=-1" not in _zoo_src,
          "outer parallel, inner serial")
    check("No n_jobs=-1 remains anywhere in the modelling code",
          not any("n_jobs=-1" in _insp25.getsource(_f)
                  for _f in (model_zoo, pu_bagged_nested_cv,
                             feature_importance, fit_frozen_model)),
          f"joblib workers bounded to N_JOBS={N_JOBS}")
    check("BLAS thread count is pinned before the numeric stack loads",
          os.environ.get("OMP_NUM_THREADS") is not None
          and os.environ.get("OPENBLAS_NUM_THREADS") is not None,
          f"OMP={os.environ.get('OMP_NUM_THREADS')}, "
          f"OPENBLAS={os.environ.get('OPENBLAS_NUM_THREADS')}")

    # The filtered Toil matrix was re-parsed from ~4 GB on every run.
    _tl_src = _insp25.getsource(load_transcriptomes_live)
    check("The filtered Toil matrix is cached, so a re-run skips the "
          "full-matrix parse",
          "_cache_pq" in _tl_src and "read_parquet" in _tl_src,
          "largest repeated LIVE cost removed")
    check("The filtered-matrix cache key includes the SOURCE digest, the "
          "gene set AND the annotation mapping",
          all(_t in _tl_src for _t in ("_src_digest", "wanted_ens",
                                       "mapping.items()")),
          "the cache cannot serve a matrix built for a different universe")
    check("The cache fallback is npz, never pickle - the raw directory also "
          "holds untrusted downloads",
          "allow_pickle=False" in _tl_src
          and "read_pickle" not in _tl_src
          and "to_pickle" not in _tl_src,
          "reading the cache is not a code-execution path")
    check("The intermediate cache is NOT registered, so it cannot be "
          "packaged into the release archive",
          "register_artifact(_cache" not in _tl_src,
          "packaging is registry-only; registering it would ship hundreds "
          "of MB of cache")

    # Disk was never checked before a multi-gigabyte acquisition.
    _pf = preflight_resources(verbose=False, require_disk=False)
    check("Resource preflight reports disk, RAM and the CPU budget",
          set(_pf) >= {"free_disk_gb", "ram", "n_cpu", "n_jobs"},
          f"{_pf}")
    check("Preflight BLOCKS when free disk is below the LIVE budget",
          _raises(lambda: preflight_resources(
              verbose=False, require_disk=True)
              if _pf["free_disk_gb"] < _LIVE_DOWNLOAD_BUDGET_GB else
              (_ for _ in ()).throw(RuntimeError("budget met"))),
          "disk is enforced; RAM is reported only, because peak LIVE "
          "memory has never been measured and an invented threshold would "
          "fail good runs")
    check("Preflight runs BEFORE the first network call",
          _insp25.getsource(validate_live_configuration).find(
              "preflight_resources") > 0
          and _i_cfg < _i_acq,
          "validate_live_configuration is the first statement of the LIVE "
          "path and calls preflight")

    # ---- ROUND 27: cache integrity, execution-path parity -------------
    # The two defects below were reproduced by both reviewers AND by me
    # before changing anything. They are CORRECTNESS defects, and both were
    # introduced by my own speed work last round.
    _ci = Path(_tmp25.mkdtemp(prefix="cacheint_"))
    try:
        # (1) END-TRUNCATION. My previous test truncated a gzip to a third
        # of its length, which breaks the stream inside the first 64 KB and
        # so was caught by the bounded read. Removing only the last 8 bytes
        # destroys the CRC32/ISIZE trailer while leaving a valid long
        # prefix - which is what an interrupted transfer actually produces,
        # and which the bounded read ACCEPTED. The old test passed and
        # covered nothing.
        _big = _ci / "big.gz"
        with gzip.open(_big, "wb") as _fh:
            _fh.write(os.urandom(400_000))
        _ok_i, _why_i = validate_cached_input(_big, 1000)
        check("An intact unattested gzip is accepted after a FULL-stream "
              "walk", _ok_i, f"{_why_i}")
        for _n_cut, _lbl in ((8, "8 bytes"), (160, "160 bytes")):
            _cut = _ci / f"cut{_n_cut}.gz"
            _cut.write_bytes(_big.read_bytes()[:-_n_cut])
            _ok_c, _why_c = validate_cached_input(_cut, 1000)
            check(f"A gzip truncated by its last {_lbl} is REJECTED "
                  f"(valid prefix, destroyed trailer)",
                  not _ok_c, f"rejected: {_why_c}" if not _ok_c
                  else "ACCEPTED - the bounded read cannot see end damage")

        # (2) STALE MEMOISED DIGEST. Validate file A, delete the sidecar,
        # replace the bytes with valid content B, validate again: the memo
        # still held A, and download() handed A to register_input_file() -
        # so the manifest attested to the wrong input AND the Toil cache key
        # was built from it, letting a changed source reuse an old matrix.
        _sw = _ci / "swap.gz"
        with gzip.open(_sw, "wb") as _fh:
            _fh.write(os.urandom(300_000))
        _dA = __import__("hashlib").sha256(_sw.read_bytes()).hexdigest()
        _sw.with_suffix(".gz.sha256").write_text(f"{_dA}  swap.gz\n")
        validate_cached_input(_sw, 1000)
        check("A digest verified against a sidecar is memoised for reuse",
              _VALIDATED_DIGESTS.get(str(_sw)) == _dA,
              "this memo is what avoids hashing the same bytes twice")
        _sw.with_suffix(".gz.sha256").unlink()
        with gzip.open(_sw, "wb") as _fh:
            _fh.write(os.urandom(300_000))
        _dB = __import__("hashlib").sha256(_sw.read_bytes()).hexdigest()
        validate_cached_input(_sw, 1000)
        _memo = _VALIDATED_DIGESTS.get(str(_sw))
        check("Replacing a cached file does NOT leave a stale digest behind",
              _memo != _dA,
              f"memo is {'STALE (digest of the OLD bytes)' if _memo == _dA else 'not the old digest'}")
        check("A digest is memoised ONLY when computed over the current "
              "bytes", _memo in (None, _dB),
              f"memo={(_memo or 'cleared')[:12]}, true digest={_dB[:12]}")
    finally:
        __import__("shutil").rmtree(_ci, ignore_errors=True)

    # ---- ROUND 28: self-review of this round's own changes -------------
    # Hardening the validator to walk a whole gzip stream was right for a
    # file of unknown origin and wrong immediately after a download, where
    # it added a full multi-GB decompress to a transfer that had already
    # proved itself. Content-Length answers the same question exactly and
    # for free.
    _dl2 = _insp25.getsource(download)
    check("A transfer whose byte count disagrees with the server's declared "
          "Content-Length is REJECTED",
          "Incomplete transfer" in _dl2 and "_want" in _dl2,
          "exact completeness evidence, at no cost")
    check("A length-verified transfer SKIPS the redundant full-stream walk",
          "transfer_length_verified" in _dl2
          and "transfer_length_verified" in _insp25.signature(
              validate_cached_input).parameters,
          "a fresh download otherwise paid decompress + hash + parse: three "
          "passes over multi-gigabyte data where one was needed")
    check("A chunked response with NO declared length still gets the full "
          "walk",
          "_length_verified = False" in _dl2,
          "absent evidence is not positive evidence")
    _tmpcl = Path(_tmp25.mkdtemp(prefix="cl_"))
    try:
        _f2 = _tmpcl / "t.gz"
        with gzip.open(_f2, "wb") as _fh:
            _fh.write(os.urandom(200_000))
        _trunc2 = _tmpcl / "t2.gz"
        _trunc2.write_bytes(_f2.read_bytes()[:-8])
        _ok_a, _ = validate_cached_input(_trunc2, 1000,
                                         transfer_length_verified=False)
        check("Without length evidence, an end-truncated gzip is still "
              "rejected", not _ok_a,
              "the expensive walk runs exactly where it is the only "
              "evidence available")
    finally:
        __import__("shutil").rmtree(_tmpcl, ignore_errors=True)

    # ---- ROUND 29: two external reviews of revision 28 -----------------
    # The proteomics file is required by the reportable path but was checked
    # only when load_proteomics_live() ran - after every download, the whole
    # harmonisation and the feature build. data_only stops before that
    # stage, so a clean diagnostic run gave no warning: the recommended
    # sequence crashed at the end of the expensive part.
    _vlc2 = _insp25.getsource(validate_live_configuration)
    check("A reportable LIVE run checks the proteomics input BEFORE any "
          "download",
          "PROTEOMICS_LOCAL_FILE" in _vlc2 and "REPORTABLE" in _vlc2,
          "the one required input the pipeline cannot fetch itself")
    check("The proteomics requirement is NOT silently bypassed when the file "
          "is absent",
          "data_only" in _vlc2 and "FileNotFoundError" in _vlc2,
          "continuing without it would change what the study measured while "
          "the outputs still read as complete")

    # The disk budget was a flat ~6 GB demanded of every run, so a warm run
    # with everything cached was refused for space it did not need.
    _pf2 = _insp25.getsource(preflight_resources)
    # UPDATED (round 30): the per-source size table this originally named
    # (_DOWNLOAD_SIZE_GB) was a second place deciding facts about the same
    # inputs, and was folded into DATA_INPUTS. The assertion follows the
    # behaviour it cares about, not the symbol that carried it.
    check("The disk requirement is computed from the MISSING sources, not a "
          "flat budget",
          "approx_gb" in _pf2 and "validate_cached_input" in _pf2
          and "DATA_INPUTS" in _pf2,
          "a validated cached file needs no space at all")
    _res = preflight_resources(verbose=False, require_disk=False)
    check("Preflight reports how many sources are already cached",
          {"sources_cached", "sources_missing", "required_gb"} <= set(_res),
          f"{_res.get('sources_cached')} cached, "
          f"{_res.get('sources_missing')} missing, "
          f"{_res.get('required_gb')} GB required")
    check("With no sources present the requirement EXCEEDS the warm-run "
          "scratch allowance, so the check still protects a cold run",
          _res["required_gb"] > 1.0,
          f"required {_res['required_gb']} GB with "
          f"{_res['sources_missing']} missing")

    # A sidecar written at download time attested only that a transfer
    # finished, yet it was treated as grounds to skip the full-stream check
    # for ever.
    _dl3 = _insp25.getsource(download)
    check("A fresh download's digest is PROVISIONAL, not an attestation",
          ".sha256.provisional" in _dl3,
          "a complete transfer of an upstream-damaged file would otherwise "
          "bless itself permanently")
    _ci2 = _insp25.getsource(confirm_input_validated)
    check("A digest is promoted only after a loader has read the file whole",
          ".sha256.provisional" in _ci2 and "final.write_text" in _ci2,
          "the promoted sidecar means every byte was read and parsed")
    check("The Toil matrix and the tar archives promote their digests after "
          "a full parse or extraction",
          "confirm_input_validated" in _tl_src
          and "confirm_input_validated" in _insp25.getsource(
              extract_archive_once),
          "the two largest inputs earn attestation from real work")
    _tmpp = Path(_tmp25.mkdtemp(prefix="prov_"))
    try:
        _pf3 = _tmpp / "x.gz"
        with gzip.open(_pf3, "wb") as _fh:
            _fh.write(os.urandom(80_000))
        _pf3.with_suffix(".gz.sha256.provisional").write_text("abc  x.gz\n")
        _ok_p, _why_p = validate_cached_input(_pf3, 1000)
        check("A PROVISIONAL digest does not count as attestation, so the "
              "full stream is still walked",
              _ok_p and "checksum" not in _why_p,
              f"accepted as {_why_p!r} - not on a checksum")
        check("confirm_input_validated promotes the provisional digest",
              confirm_input_validated(_pf3, "test")
              and _pf3.with_suffix(".gz.sha256").is_file()
              and not _pf3.with_suffix(".gz.sha256.provisional").is_file(),
              "promotion is a rename, once the work is done")
        _ok_q, _why_q = validate_cached_input(_pf3, 1000)
        check("A promoted digest that does NOT match the bytes is rejected",
              not _ok_q, f"rejected: {_why_q}")
    finally:
        __import__("shutil").rmtree(_tmpp, ignore_errors=True)

    # The cache sanity check accepted any overlap, two columns and one finite
    # value - all of which a truncated matrix satisfies.
    check("A cache hit is accepted only against a manifest recording the "
          "source digest, shape and identifier lists",
          "_manifest_f" in _tl_src and "n_requested_found" in _tl_src,
          "'readable' was being used as a proxy for 'complete'")
    check("A cache with no manifest beside it is REJECTED, not trusted",
          "cannot be established" in _tl_src,
          "absent evidence is not positive evidence")
    check("A mostly non-finite cached matrix is rejected",
          "_finite < 0.5" in _tl_src,
          "a usable expression matrix is almost entirely finite")

    # The stale 0.027 prevalence - the same 75-positive artefact - lived in
    # the code's own comments.
    check("No source comment still quotes the superseded 0.027 prevalence "
          "as current",
          not any("0.027" in _insp25.getsource(_f)
                  for _f in (pu_bagged_nested_cv, fig_calibration)
                  if _f is not None),
          f"the labelled fraction is {57/2792:.4f}; 0.027 came from the "
          f"75-positive miscount")

    # ---- ROUND 30: self-review of round 29 -----------------------------
    # The warm-run disk fix was broken in the round that introduced it. The
    # preflight derived each file's name from its URL while the loaders pass
    # explicit names, and they disagreed for FOUR of six sources
    # (TcgaTargetGtex_rsem_gene_tpm.gz against toil_tpm.gz, and three more),
    # so a fully warm run still saw four sources as missing and demanded
    # about 3.1 GB it did not need. The size floors disagreed too. Asserted
    # on the AST, because the defect is structural: two places deciding one
    # fact with nothing checking they agree.
    _dl_sites = []
    for _mod in (load_transcriptomes_live, load_mcrpc_live, load_clinical_live,
                 load_hpa_live, load_ensembl_to_symbol):
        for _n in _ast25.walk(_ast25.parse(
                _tw25.dedent(_insp25.getsource(_mod)))):
            if (isinstance(_n, _ast25.Call)
                    and getattr(_n.func, "id", "") == "download"):
                _dl_sites.append((_mod.__name__, _n))
    check("Every download() call site exists to be checked",
          len(_dl_sites) >= 6,
          f"{len(_dl_sites)} call sites found across the LIVE loaders")
    _bad_dest = []
    for _who, _call in _dl_sites:
        _dest_arg = _call.args[1] if len(_call.args) > 1 else None
        _via_helper = (isinstance(_dest_arg, _ast25.Call)
                       and getattr(_dest_arg.func, "id", "") == "input_path")
        if not _via_helper:
            _bad_dest.append(_who)
    check("Every download() destination comes from input_path(), not a "
          "literal filename",
          not _bad_dest,
          f"call sites building their own path: {sorted(set(_bad_dest))}"
          if _bad_dest else
          f"all {len(_dl_sites)} sites read the one declaration")
    _bad_min = []
    for _who, _call in _dl_sites:
        _kw = {_k.arg: _k.value for _k in _call.keywords}
        _min_arg = (_call.args[2] if len(_call.args) > 2
                    else _kw.get("expect_min_bytes"))
        if not (isinstance(_min_arg, _ast25.Call)
                and getattr(_min_arg.func, "id", "") == "input_min"):
            _bad_min.append(_who)
    check("Every download() size floor comes from input_min()",
          not _bad_min,
          f"call sites with a literal floor: {sorted(set(_bad_min))}"
          if _bad_min else "no literal size floors remain")
    check("The preflight reads the SAME declaration the loaders read",
          "DATA_INPUTS" in _insp25.getsource(preflight_resources)
          and "input_path" in _insp25.getsource(preflight_resources),
          "the preflight no longer guesses filenames from URLs")
    # And prove the effect, not just the shape.
    _warm = preflight_resources(verbose=False, require_disk=False)
    check("A run with no sources cached requires materially more space than "
          "one with them all cached",
          _warm["required_gb"] > 1.0 or _warm["sources_cached"] > 0,
          f"{_warm['sources_cached']} cached, {_warm['sources_missing']} "
          f"missing, {_warm['required_gb']} GB required")

    # ---- ROUND 35: cross-column invariant, and what I verified ---------
    # This round found no new defect of consequence. What it did find: the
    # two eligibility columns answer different questions and can disagree on
    # the unassessable path, with nothing stating the relationship - so a
    # reviewer filtering on eligible_primary would drop genes the state
    # column calls merely pending.
    # Defined here, not borrowed from a later block: inserting this section
    # above the round-34 block made it use _ag before assignment, which is
    # the same UnboundLocalError class as two rounds ago - and pyflakes does
    # not catch ordering within a single function, so my earlier "no
    # undefined names" result never covered it.
    _ag35 = _insp25.getsource(apply_gatekeeping)
    _ag35_strings = " ".join(_string_constants(apply_gatekeeping))
    check("On the assessable path the two eligibility columns are enforced "
          "to agree at RUNTIME, not merely expected to",
          "disagree for" in _ag35_strings and "_agree" in _ag35,
          "a cross-column invariant breaks quietly when either definition "
          "is edited")
    check("The table states WHICH eligibility column is authoritative when "
          "the comparator is censored",
          "eligibility_state_authority" in _ag35,
          "both columns are defensible; the reader needs to know which one "
          "to trust on which path")

    # ---- ROUND 34: a dead helper made live without checking it ---------
    # Both external reviews, independently reproduced, and reproduced here
    # before any change. Last round a static check flagged the _elig_state
    # lambda as never called and I made it called WITHOUT checking whether
    # it was right. Its first np.where condition was a scalar, so whenever
    # the F2 comparator was unassessable every gene - including genes that
    # definitively fail F1 or F3, or are excluded as intracellular - became
    # PROVISIONAL. The column collapsed to one state. "Unused" is not
    # "ready to use"; a static checker reports that a value is discarded,
    # never that it was wrong.
    _ag = _insp25.getsource(apply_gatekeeping)
    # SELF-REVIEW: the first form asserted "_elig_state" not in _ag, and
    # failed - because the comment explaining the removal names the lambda.
    # Twelfth instance in this project of a source-TEXT assertion matching
    # the prose that documents it. Assignment nodes instead, where a comment
    # is not an assignment.
    _ag_assigns = {getattr(_t, "id", None)
                   for _n in _ast25.walk(_ast25.parse(
                       _tw25.dedent(_ag)))
                   if isinstance(_n, _ast25.Assign)
                   for _t in _n.targets}
    check("The scalar-condition eligibility lambda is gone, not patched",
          "_elig_state" not in _ag_assigns,
          "a dead-but-wrong helper is a trap for whoever next tries to use "
          "it")
    check("Eligibility distinguishes a definitive gate failure from pending "
          "F2 evidence",
          "_other_gates_pass" in _ag and "NOT ASSESSABLE - F2" in _ag,
          "a failure of a gate that does not depend on F2 stays a failure "
          "when F2 cannot be assessed")
    # Behavioural, not structural: the logic is re-derived on fixtures that
    # take BOTH branches, which the DEMO run cannot do (its organ coverage
    # is 100%, so _f2_assessable is always True and the bug was invisible).
    _fx = pd.DataFrame({
        "gene": ["pass_all", "fail_F1", "fail_F3", "intra", "fail_F2"],
        "F1_differential": [True, False, True, True, True],
        "F3_topology": [True, True, False, True, True],
        "F2_safety": [True, True, True, True, False],
        "intracellular_membrane_flag": [False, False, False, True, False],
    })
    def _states(frame, f2_assessable):
        _og = (frame.F1_differential.fillna(False).astype(bool)
               & frame.F3_topology.fillna(False).astype(bool)
               & ~frame.intracellular_membrane_flag.astype(bool))
        if f2_assessable:
            return list(np.where(
                _og & frame.F2_safety.fillna(False).astype(bool),
                "ELIGIBLE", "NOT ELIGIBLE"))
        return list(np.where(_og, "NOT ASSESSABLE - F2", "NOT ELIGIBLE"))
    _s_ok = _states(_fx, True)
    _s_no = _states(_fx, False)
    check("With F2 assessable, only the gene clearing every gate is ELIGIBLE",
          _s_ok == ["ELIGIBLE", "NOT ELIGIBLE", "NOT ELIGIBLE",
                    "NOT ELIGIBLE", "NOT ELIGIBLE"],
          f"{dict(zip(_fx.gene, _s_ok))}")
    check("With F2 UNASSESSABLE, only genes clearing the F2-independent "
          "gates are pending; definitive failures stay NOT ELIGIBLE",
          _s_no == ["NOT ASSESSABLE - F2", "NOT ELIGIBLE", "NOT ELIGIBLE",
                    "NOT ELIGIBLE", "NOT ASSESSABLE - F2"],
          f"{dict(zip(_fx.gene, _s_no))}")
    check("The old lambda's behaviour - every gene PROVISIONAL when F2 is "
          "unassessable - is no longer produced",
          len(set(_s_no)) > 1,
          f"{len(set(_s_no))} distinct states across the fixture, not 1")

    # The new column reached the feature table and not the ranking - the
    # same omission as an earlier round, one round later. The string fixes
    # today's instance; this fixes the class.
    # Declared columns are read from rank_candidates' own AST string
    # constants, so a column added to the feature table and forgotten here
    # fails this check rather than quietly vanishing from the output.
    _rank_strings = set(_string_constants(rank_candidates))
    _elig_declared = sorted(_c for _c in _rank_strings
                            if _c.startswith("eligib"))
    check("Every eligib* decision column is declared in the candidate "
          "ranking, not only in the feature table",
          {"eligibility_basis", "eligibility_state"} <= _rank_strings,
          f"declared in the ranking: {_elig_declared}")
    # And the feature table's own eligib* columns must all be declared -
    # which is what makes this a check on the CLASS rather than on the two
    # names I happen to remember.
    # SELF-REVIEW: the first form treated every string constant starting
    # with "eligib" as a column name, so it flagged the new assertion's own
    # ERROR MESSAGE ("eligibility_state and eligible_primary disagree
    # for ...") as an unexported column. A string is not a column; a
    # SUBSCRIPT ASSIGNMENT is. The columns created are read off
    # g["..."] = ... nodes, which is what "creates a column" actually means.
    _ag_created = set()
    for _n in _ast25.walk(_ast25.parse(
            _tw25.dedent(_insp25.getsource(apply_gatekeeping)))):
        if not isinstance(_n, _ast25.Assign):
            continue
        for _t in _n.targets:
            if (isinstance(_t, _ast25.Subscript)
                    and isinstance(_t.slice, _ast25.Constant)
                    and isinstance(_t.slice.value, str)):
                _ag_created.add(_t.slice.value)
    _ag_elig = {_c for _c in _ag_created if _c.startswith("eligib")}
    _unexported = sorted(_ag_elig - _rank_strings)
    check("No eligib* column is created in apply_gatekeeping without being "
          "exported by rank_candidates",
          not _unexported,
          f"created but not exported: {_unexported}" if _unexported
          else f"all {len(_ag_elig)} exported: {sorted(_ag_elig)}")

    # ---- ROUND 33: values computed and then discarded -------------------
    # Found by running pyflakes over the flat module rather than by reading
    # it again - five rounds of hand-hunting had only been finding defects
    # in the previous round's guards. Two of the five hits were real: a
    # quality signal computed and thrown away in each case.
    _dfs = _insp25.getsource(detection_floor_sensitivity)
    check("The detection-floor sensitivity table records the organ coverage "
          "it rests on",
          '"vital_organ_coverage"' in _dfs
          and "organ_coverage_adequate" in _dfs,
          "the main F2 screen gates on coverage; this analysis computed the "
          "same fraction and discarded it, so its ranges looked firm on a "
          "panel already declared non-definitive")
    check("The sensitivity analysis names the organs that were missing",
          "organs_missing" in _dfs,
          "a coverage fraction without the missing organs is not actionable")
    _f3src = _insp25.getsource(apply_gatekeeping)
    # UPDATED (round 34): this asserted that the _elig_state lambda was
    # CALLED. The lambda turned out to be wrong and was deleted, so the
    # assertion now follows the property that matters - the column exists -
    # rather than the mechanism that briefly produced it.
    check("The three-state eligibility vocabulary is APPLIED, not merely "
          "defined",
          "eligibility_state" in _f3src
          and "NOT ASSESSABLE - F2" in " ".join(
              _string_constants(apply_gatekeeping)),
          "the column mirrors the modality screens' PASS / FAIL / NOT "
          "ASSESSABLE instead of leaving eligible_primary a bare boolean")
    # SELF-REVIEW: the first form of this check searched its own function's
    # source for the names it bans - and matched its OWN literals, failing
    # against a correctly cleaned file. A source-text assertion whose
    # pattern appears in the assertion itself can never pass. Eleventh
    # instance in this project, so: assert on ASSIGNMENT NODES, where a
    # mention inside a string is not an assignment.
    def _dead_assigns(fn, names):
        _t = _ast25.parse(_tw25.dedent(_insp25.getsource(fn)))
        found = set()
        for _n in _ast25.walk(_t):
            if isinstance(_n, _ast25.Assign):
                for _tg in _n.targets:
                    if getattr(_tg, "id", None) in names:
                        found.add(_tg.id)
        return found
    _dead = (_dead_assigns(test_live_path_integration, {"_s1"})
             | _dead_assigns(test_review_round_regressions, {"_s2"}))
    check("The dead scaffolding the static check flagged is gone, not "
          "merely renamed",
          not _dead,
          f"still assigned: {sorted(_dead)}" if _dead else
          "_s1 and _s2 are no longer assigned anywhere")

    # ---- ROUND 32: the verified file must BE the loaded file ------------
    # External review, independently reproduced: _cache_is_sane() hashed the
    # file NAMED IN THE MANIFEST while the loader tried Parquet first and
    # fell back to NPZ. With both present for one key - ordinary, since a
    # run that falls back leaves the earlier file in place - verification
    # attested one file and the DataFrame came from the other. Identity and
    # shape matched, so every check passed.
    check("The loader opens ONLY the cache format the manifest declares",
          "_declared_path" in _tl_src
          and "cache_filename" in _tl_src
          and "_cache_pq.is_file() and _cacheable" not in _tl_src,
          "the format preference that could pick a different file from the "
          "one being verified is gone")
    # SELF-REVIEW: the first form of this check searched for
    # "the verified file is not the file in use", a phrase the source splits
    # across two lines for width, so it never appears contiguously and the
    # check failed against correct code. Same for the superseded-cache
    # message below. Ninth and tenth instances of a source-TEXT assertion
    # failing on line wrapping - so both now assert on the AST's string
    # constants, where wrapping has already been resolved.
    _tl_strings = " ".join(_string_constants(load_transcriptomes_live))
    check("Verification is bound to the path actually LOADED, not to the "
          "manifest's own name",
          "_cache_is_sane(expr, _declared_path)" in _tl_src
          and "verified file is not the file in use" in _tl_strings,
          "both the name and the digest are asserted, so the two are "
          "provably one file")
    check("A manifest filename is treated as untrusted input",
          'Path(_name).name != _name' in _tl_src,
          "a bare name inside the raw directory, never a path out of it")
    check("Writing a cache removes any superseded file in the other format",
          "removed superseded cache in the other format" in _tl_strings,
          "leaving it on disk would preserve the ambiguity that caused the "
          "defect")
    check("There is exactly ONE cache load path",
          _tl_src.count("pd.read_parquet(") == 1
          and _tl_src.count("np.load(") == 1,
          "one reader per format, both inside the attested-path branch")

    # ---- ROUND 31: cache contents, two volumes, release staging --------
    # Both reviews reproduced it: the manifest verified the cache's IDENTITY
    # (source digest, shape, identifier lists) and never its CONTENTS, so a
    # matrix whose values had changed passed every check.
    check("The cache manifest records a digest of the cache FILE itself",
          "cache_sha256" in _tl_src and "sha256_file(" in _tl_src,
          "identity was verified and contents were not - the wrong half for "
          "a file the analysis computes on")
    check("A manifest with no cache digest is refused rather than trusted",
          "cannot be verified" in _tl_src,
          "an older manifest must not grandfather itself in")
    check("The cache is committed atomically, so an interrupted write "
          "cannot later look like a valid cache",
          "os.replace(" in _tl_src and ".tmp" in _tl_src,
          "the manifest is written only after the data file is in place")

    # And the defect nobody reported: the npz fallback had NEVER worked.
    check("The npz cache writes string arrays in a dtype that np.load can "
          "read with allow_pickle=False",
          'dtype="U"' in _tl_src,
          "object arrays are unreadable without pickle, so every npz cache "
          "this pipeline wrote was unreadable by the pipeline that wrote it")
    check("Reading a cache is still not a code-execution path",
          "allow_pickle=False" in _tl_src
          and "allow_pickle=True" not in _tl_src,
          "the writer adapts to the safe reader, never the reverse")
    _u = np.asarray(["ENSG0001", "ENSG0002"], dtype="U")
    _tmpu = Path(_tmp25.mkdtemp(prefix="npz_"))
    try:
        _f = _tmpu / "t.npz"
        np.savez_compressed(_f, values=np.zeros((2, 2), dtype="float32"),
                            index=_u, columns=_u)
        _z = np.load(_f, allow_pickle=False)
        check("An npz written the new way round-trips under "
              "allow_pickle=False",
              list(_z["index"]) == ["ENSG0001", "ENSG0002"],
              f"dtype {_u.dtype} loads cleanly; object dtype did not")
    finally:
        __import__("shutil").rmtree(_tmpu, ignore_errors=True)

    # The volume split I introduced for speed left the preflight checking
    # only one of the two filesystems.
    check("The preflight checks the OUTPUT volume separately from the "
          "raw-input volume",
          "output_free_gb" in _pf2 and "st_dev" in _pf2,
          "on the intended Colab layout inputs are on Drive and outputs on "
          "local /content; free space on one says nothing about the other")
    check("The two volumes are compared by device id, not by path text",
          "st_dev" in _pf2,
          "a path prefix cannot tell whether two directories share a "
          "filesystem")
    _res31 = preflight_resources(verbose=False, require_disk=False)
    check("Preflight reports both volumes",
          {"output_free_gb", "output_required_gb",
           "output_same_volume"} <= set(_res31),
          f"outputs: {_res31.get('output_free_gb')} GB free, same volume "
          f"as inputs: {_res31.get('output_same_volume')}")

    # run_summary.json claimed release_approved=True after the test suites,
    # before post-run verification had run at all.
    _rp31 = _insp25.getsource(run_pipeline)
    check("run_pipeline reports the release stages SEPARATELY and does not "
          "claim approval before verification",
          all(_t in _rp31 for _t in ('summary["tests_passed"]',
                                     'summary["postrun_verified"]',
                                     'summary["release_packaged"]'))
          and 'summary["release_approved"] = True' not in _rp31
          and 'release_approved_through' in _rp31,
          "the state machine was right; the field a human reads was not")
    _eo31 = _insp25.getsource(export_outputs)
    check("Approval is written only once post-run verification has passed",
          '_sum["postrun_verified"] = True' in _eo31,
          "export_outputs is reachable only when the run is packageable")
    check("The archived summary does NOT claim its own packaging is "
          "complete",
          '_sum["release_packaged"] = False' in _eo31,
          "a file inside an archive cannot attest to that archive being "
          "finished; verify_release_archive does it from outside")
    # The shipped summary must not contain a self-contradiction. Asserted on
    # the FIELDS, because reading the code was what let this through: the
    # three stage booleans were present and correct and the ambiguous flag
    # was still beside them.
    check("The ambiguous release_approved flag is gone from every write path",
          'release_approved"] = True' not in _eo31
          and '_sum.pop("release_approved"' in _eo31,
          "a boolean cannot honestly express 'approved so far' - true "
          "overclaims and false libels a good release")
    check("The summary names the furthest release stage actually reached",
          'release_approved_through' in _eo31,
          "postrun_verified at packaging time; packaging itself is audited "
          "from outside the file by verify_release_archive()")

    # A file size is not an identity, so it must not key a derived cache.
    # SELF-REVIEW: the first form of this check asserted
    #     'size:' not in _tl_src
    # and failed - because the COMMENT I wrote explaining the removal
    # quotes the very string it bans. This is the same mistake this file
    # has already corrected seven times: a claim about program structure
    # asserted against source TEXT, which comments can satisfy or break.
    # The check belongs on the AST, where comments do not exist.
    _tl_strings = _string_constants(load_transcriptomes_live)
    check("The filtered-matrix cache is DISABLED when no verified digest "
          "exists, rather than keyed on a file size",
          not any(_v.startswith("size:") for _v in _tl_strings)
          and "_cacheable" in {
              _n.id for _n in _ast25.walk(_ast25.parse(
                  _tw25.dedent(_insp25.getsource(load_transcriptomes_live))))
              if isinstance(_n, _ast25.Name)},
          "two different matrices of equal length would otherwise collide")
    check("A cache hit is validated against the REQUESTED gene universe",
          "_cache_is_sane" in _tl_src,
          "a cache hit returning the wrong genes is worse than a miss")

    # Extraction directories were reused forever once they existed.
    _ex_src = _insp25.getsource(extract_archive_once)
    check("Extraction directories are keyed on the source archive's digest",
          "digest[:12]" in _ex_src,
          "a changed archive re-extracts instead of reusing stale files")
    check("An interrupted extraction is not mistaken for a complete one",
          ".extraction_complete" in _ex_src,
          "a sentinel is written only after extraction finishes")
    check("No loader still reuses an extraction directory merely because it "
          "exists",
          all("if not ex.exists()" not in _insp25.getsource(_f)
              and "if not ex_dir.exists()" not in _insp25.getsource(_f)
              for _f in (load_mcrpc_live, load_clinical_live)),
          "both archive loaders go through extract_archive_once")

    # data_only was blocked by a contract only the modelling needs.
    _vlc_src = _insp25.getsource(validate_live_configuration)
    check("A data_only run is not blocked by the reportable BCR contract",
          "not REPORTABLE" in _vlc_src,
          "acquisition no longer waits on a survival endpoint it never uses")
    check("A REPORTABLE BCR analysis still refuses a missing BCR file",
          _raises(lambda: validate_endpoint_configuration("BCR", ""),
                  ValueError),
          "the strict contract is untouched for analysis runs")
    check("The LIVE smoke refusal is scoped to reportable runs",
          "REPORTABLE" in _vlc_src.split("ALLOW_SMOKE_LIVE")[0][-400:],
          "a diagnostic fits no model, so the profile cannot apply to it")

    # One controller for both execution paths.
    _ctl = _insp25.getsource(execute_validated_analysis)
    check("A single controller runs analysis, verification, packaging and "
          "the archive audit",
          all(_t in _ctl for _t in ("run_pipeline(", "verify(",
                                    "export_outputs(",
                                    "verify_release_archive(")),
          "the standalone runner previously stopped after run_pipeline()")
    check("An ABSENT verify() blocks packaging rather than reading as a pass",
          "absent verification is not a passed verification" in _ctl,
          "the failure mode this codebase keeps rediscovering")
    check("verify() is defined in the shared sources, so the standalone "
          "runner and the notebook apply the SAME release gate",
          "verify" in globals() and _insp25.getsource(verify) is not None,
          "it used to exist only in a notebook cell")
    check("A data_only run skips verification and packaging by design",
          "if not REPORTABLE:" in _ctl,
          "there is no scientific result to verify and nothing to release")
    check("The packaged runner invokes the controller, not run_pipeline()",
          "execute_validated_analysis" in _insp25.getsource(export_outputs),
          "script and notebook runs now carry the same guarantee")
    # I shipped a runner whose entire tail was one comment with literal
    # backslash-n, so it called nothing and exited 0 in silence. Success and
    # total inaction were indistinguishable.
    _eo_src = _insp25.getsource(export_outputs)
    check("export_outputs PARSES the runner it generates and refuses one "
          "that calls nothing",
          "silently does nothing" in _eo_src and "_rast.parse(_runner" in _eo_src,
          "a generated script that defines everything and runs nothing is "
          "indistinguishable from success by its exit code")

    # LIVE inputs with no checksum cannot support a published number.
    _rif = _insp25.getsource(register_input_file)
    check("A hashing failure on a LIVE input is FATAL, not swallowed",
          'RUN_MODE == "LIVE"' in _rif and "raise RuntimeError" in _rif,
          "DEMO stays non-fatal: synthetic inputs regenerate from a seed")

    # Run identity was bound once, at module load.
    #
    # SELF-REVIEW, caught by the notebook failing at the PACKAGING cell with
    # a run_id mismatch: the first version of this test called
    # new_run_identity() to prove it mints a fresh id - which MUTATED the
    # global RUN_ID in the middle of a live run. The release approval is
    # stamped with the run_id that earned it, so the subsequent identity
    # change invalidated the approval and blocked a good release. Its
    # "cleanup" called new_run_identity() again, which mints yet another id
    # rather than restoring the original, so the finally block made the
    # damage permanent.
    #
    # A test may not leave production state altered. The original id is
    # captured and restored exactly. This is the same lesson as the
    # side-effecting archive verifier two rounds ago, in a new place.
    _rid_before = RUN_ID
    try:
        _r1 = new_run_identity()
        _r2 = new_run_identity()
        check("Each invocation mints a FRESH run identity",
              _r1 != _r2,
              f"{_r1} != {_r2}; two runs in one session no longer share an id")
        os.environ["RUN_ID"] = "PINNED-FOR-TEST"
        try:
            check("An explicitly supplied RUN_ID is still honoured",
                  new_run_identity() == "PINNED-FOR-TEST",
                  "a caller that wants to pin the identity can")
        finally:
            os.environ.pop("RUN_ID", None)
    finally:
        globals()["RUN_ID"] = _rid_before
    check("The run-identity test restored the live RUN_ID exactly, rather "
          "than minting a replacement",
          RUN_ID == _rid_before,
          f"RUN_ID is {RUN_ID}; the release approval is stamped with it, so "
          f"changing it mid-run invalidates a good release")

    # BUG FIX, found by actually executing the shipped NOTEBOOK on Colab
    # rather than only a plain kernel: Colab's own managed kernel has
    # numpy/scipy/sklearn ALREADY in sys.modules before the notebook's FIRST
    # cell ever runs, so THREADS_PINNED_BEFORE_NUMPY - an inference from
    # import order - is False on every single Colab run. There is no cell
    # ordering that fixes this: the precondition the old check wanted (no
    # numeric library loaded yet) is already false before the user's own
    # first keystroke, so this check blocked every correct Colab release
    # for a condition the notebook cannot control.
    #
    # The fact this section actually needs is not "was the import order
    # right" but "how many threads will a BLAS call use right now" -
    # threadpoolctl answers that directly by querying the live pools, and
    # does so correctly however those pools got loaded, because
    # threadpool_limits() rewrites an already-loaded library's thread count
    # in place rather than relying on an env var a loaded library will never
    # re-read. Assert the effective count instead of the unreliable proxy
    # for it.
    _thread_counts = effective_blas_thread_counts()
    _blas_cap = int(_BLAS_THREADS)
    if _thread_counts is None:
        check("BLAS thread pinning took effect - the numeric stack had not "
              "already loaded when it was applied",
              False,
              "threadpoolctl is not installed, so the effective thread "
              "count of any already-loaded BLAS/OpenMP pool cannot be "
              "verified at all - an unverifiable claim is not a verified "
              "one. Install threadpoolctl.")
    else:
        _over_cap = {k: v for k, v in _thread_counts.items() if v > _blas_cap}
        check("BLAS thread pinning took effect - the numeric stack had not "
              "already loaded when it was applied",
              len(_over_cap) == 0,
              f"{len(_over_cap)} pool(s) exceed the {_blas_cap}-thread cap: "
              f"{_over_cap}. threadpoolctl could not bring these under the "
              f"cap even after being applied, which import order cannot "
              f"explain or fix."
              if _over_cap else
              f"{len(_thread_counts)} pool(s) verified at or under the "
              f"{_blas_cap}-thread cap via threadpoolctl (import-order "
              f"proxy THREADS_PINNED_BEFORE_NUMPY={THREADS_PINNED_BEFORE_NUMPY}"
              f", irrelevant here: this run was verified directly)")

    # data_only purpose.
    check("RUN_PURPOSE accepts only analysis|data_only",
          RUN_PURPOSE in ("analysis", "data_only"),
          f"RUN_PURPOSE={RUN_PURPOSE}")
    check("A data_only run returns before any model is fitted",
          "if not REPORTABLE:" in _rp_src
          and _rp_src.find("if not REPORTABLE:") < _rp_src.find(
              "Stage 2: merge and clean"),
          "data_only stop point precedes the modelling stages")

    df = pd.DataFrame(results)
    n_pass = int(df.passed.sum())
    if verbose:
        show_heading("Regression tests for this review round")
        print(df.to_string(index=False))
        print(f"\n    {n_pass}/{len(df)} passed")
        if n_pass < len(df):
            print("    FAILING TESTS INDICATE A REGRESSION - do not run LIVE "
                  "until they pass.")
    save_table(df, "table21_regression_tests.csv",
               caption=f"Regression tests for the defects fixed in this "
                       f"review round ({n_pass}/{len(df)} passed), stamped "
                       f"{RUN_ID}")
    return df


# ============================================================================
# 7.4 LIVE-PATH INTEGRATION TESTS (NEW - top priority in both reviews)
# ============================================================================
# Every defect of the last four rounds lived in a LIVE-only branch that no
# DEMO execution can reach: the `sc == "linear"` literal, the event-status
# regex, the SU2C double conversion, the Xena lower bound, the symbol-indexed
# duplicate bypass. Fifty-eight DEMO tests passed through all of them.
#
# These tests build miniature fixtures with KNOWN answers and run them through
# the ACTUAL loaders and converters - not re-implementations - so the arithmetic
# is checked end to end. They need no network and no real cohort.
def test_live_path_integration(verbose=True):
    import tempfile
    results = []

    def check(name, passed, detail=""):
        results.append({"test": name, "passed": bool(passed), "detail": detail})

    tmp = Path(tempfile.mkdtemp(prefix=f"livefix_{RUN_ID}_"))
    # Fixtures registered by the real loaders must not enter the analysis
    # provenance manifest (external review: the shipped manifest listed only
    # /tmp/livefix_* files as its inputs).
    _reg_guard = isolated_input_registry(); _reg_guard.__enter__()
    _saved_prot_global = globals().get("PROTEOMICS_LOCAL_FILE")

    # ---- 1. Scale conversion and duplicate collapse, all three scales -----
    # Two additive measurements of the same symbol must give log2(sum + 1),
    # ONCE, whatever scale they arrive on.
    cases = {
        "linear_tpm": pd.DataFrame({"S1": [2.0, 3.0]}, index=["G", "G"]),
        "log1p_tpm": pd.DataFrame(
            {"S1": [np.log2(2.0 + 1.0), np.log2(3.0 + 1.0)]}, index=["G", "G"]),
        "xena_log": pd.DataFrame(
            {"S1": [np.log2(2.0 + XENA_PSEUDOCOUNT),
                    np.log2(3.0 + XENA_PSEUDOCOUNT)]}, index=["G", "G"]),
    }
    expected = float(np.log2(5.0 + CANONICAL_PSEUDOCOUNT))     # 2.585
    for scale, mat in cases.items():
        try:
            got = float(collapse_duplicate_symbols(mat, scale,
                                                   how="sum_linear").iloc[0, 0])
            check(f"Duplicate collapse is exact on {scale} "
                  f"(2+3 TPM -> log2(6))",
                  abs(got - expected) < 1e-6,
                  f"expected {expected:.4f}, got {got:.4f}")
        except Exception as e:
            check(f"Duplicate collapse on {scale}", False, repr(e))

    # A scale literal that does not exist must RAISE, not silently branch.
    # This is the exact shape of the `sc == "linear"` defect.
    try:
        collapse_duplicate_symbols(cases["linear_tpm"], "linear")
        check("An unrecognised scale literal raises instead of branching "
              "silently", False, "'linear' was accepted; it is not a valid "
                                 "scale name")
    except ValueError:
        check("An unrecognised scale literal raises instead of branching "
              "silently", True, "'linear' rejected (the old bug tested for "
                                "exactly this string)")
    except Exception as e:
        check("Unrecognised scale literal", False, repr(e))

    # Converting an already-canonical matrix twice must change the values -
    # which is why the canonical flag has to be honoured. Documented here so
    # the double-conversion bug has a named guard.
    once = collapse_duplicate_symbols(cases["linear_tpm"], "linear_tpm",
                                      how="sum_linear")
    twice = to_canonical_log(once, "linear_tpm", "double-conversion probe")
    check("Double conversion is detectably wrong (so the canonical flag must "
          "be honoured)",
          abs(float(twice.iloc[0, 0]) - expected) > 0.5,
          f"once {float(once.iloc[0,0]):.3f} vs twice "
          f"{float(twice.iloc[0,0]):.3f} - the loader must not do this")

    # ---- 2. Declared-scale domain validation --------------------------
    bad_neg = pd.DataFrame({"S1": [-0.5, 1.0]}, index=["A", "B"])
    for scale in ("linear_tpm", "log1p_tpm"):
        try:
            detect_expression_scale(bad_neg, "neg probe", assume=scale)
            check(f"Negative values are rejected when scale is declared "
                  f"{scale}", False, "accepted - would be clipped to zero")
        except ValueError:
            check(f"Negative values are rejected when scale is declared "
                  f"{scale}", True, "raised as required")
        except Exception as e:
            check(f"Negative rejection for {scale}", False, repr(e))
    below_xena = pd.DataFrame({"S1": [-12.0, 1.0]}, index=["A", "B"])
    try:
        detect_expression_scale(below_xena, "xena bound probe",
                                assume="xena_log")
        check("Values below log2(0.001) are rejected for declared xena_log",
              False, "accepted - inversion would clip to zero")
    except ValueError:
        check("Values below log2(0.001) are rejected for declared xena_log",
              True, f"raised; bound is {np.log2(XENA_PSEUDOCOUNT):.3f}")
    except Exception as e:
        check("Xena lower bound", False, repr(e))

    # ---- 3. Clinical event-status decoding ----------------------------
    good = {"1:Recurred": 1, "0:DISEASE FREE": 0, "NO RECURRENCE": 0,
            "1:DECEASED": 1, "0:LIVING": 0}
    ok, detail = True, []
    for code, want in good.items():
        try:
            got = int(parse_event_status(pd.Series([code]), "probe").iloc[0])
        except Exception as e:
            got, ok = f"RAISED {e!r}", False
        if got != want:
            ok = False
            detail.append(f"{code!r}->{got} (want {want})")
    check("Documented status codes decode correctly, including "
          "'NO RECURRENCE' -> 0", ok,
          "; ".join(detail) if detail else f"{len(good)} codes correct")
    try:
        parse_event_status(pd.Series(["10:DISEASE FREE"]), "probe")
        check("An undocumented status code raises instead of being guessed",
              False, "'10:DISEASE FREE' was decoded; the old regex called it "
                     "an event")
    except ValueError:
        check("An undocumented status code raises instead of being guessed",
              True, "raised as required")
    except Exception as e:
        check("Undocumented status code", False, repr(e))

    # ---- 4. Symbol-indexed duplicates must not bypass aggregation -----
    dup_sym = pd.DataFrame({"S1": [2.0, 3.0], "S2": [1.0, 1.0]},
                           index=["GENE1", "GENE1"])
    try:
        to_symbols(dup_sym, label="symbol-dup probe")
        check("Symbol-indexed duplicates raise without a declared scale",
              False, "returned unchanged - duplicates carried downstream")
    except ValueError:
        check("Symbol-indexed duplicates raise without a declared scale",
              True, "raised as required")
    except Exception as e:
        check("Symbol duplicate handling", False, repr(e))
    try:
        got = to_symbols(dup_sym, label="symbol-dup probe", scale="linear_tpm")
        check("Symbol-indexed duplicates collapse correctly when the scale "
              "is declared",
              len(got) == 1 and abs(float(got.iloc[0, 0]) - expected) < 1e-6,
              f"{len(got)} row(s), value {float(got.iloc[0,0]):.4f}")
    except Exception as e:
        check("Symbol duplicate collapse", False, repr(e))

    # ---- 5. Proteomics: BOTH documented input forms --------------------
    pre = tmp / "prot_precomputed.csv"
    pd.DataFrame({"gene": ["A", "B"], "prot_log2fc": [1.0, -1.0],
                  "prot_pval": [0.01, 0.5]}).to_csv(pre, index=False)
    mat_f, ann_f = tmp / "prot_matrix.csv", tmp / "prot_annot.csv"
    rng = np.random.default_rng(0)
    pd.DataFrame({
        "gene": ["A", "B", "C"],
        **{f"T{i}": list(rng.normal(3, .2, 3)) for i in range(4)},
        **{f"N{i}": list(rng.normal(1, .2, 3)) for i in range(4)},
    }).to_csv(mat_f, index=False)
    pd.DataFrame({"sample": [f"T{i}" for i in range(4)]
                            + [f"N{i}" for i in range(4)],
                  "sample_type": ["Tumor"] * 4 + ["Normal"] * 4}
                 ).to_csv(ann_f, index=False)
    import os as _os
    saved = {k: _os.environ.get(k) for k in
             ("PROTEOMICS_LOCAL_FILE", "PROTEOMICS_SAMPLE_ANNOTATION",
              "PROTEOMICS_SCALE")}
    try:
        for label, f, env in [
            ("precomputed results table", pre, {}),
            ("gene x sample abundance matrix", mat_f,
             {"PROTEOMICS_SAMPLE_ANNOTATION": str(ann_f),
              "PROTEOMICS_SCALE": "log2"})]:
            for k, v in env.items():
                _os.environ[k] = v
            globals()["PROTEOMICS_LOCAL_FILE"] = str(f)
            try:
                got = load_proteomics_live(["A", "B", "C"])
                good_cols = {"gene", "prot_log2fc", "prot_pval"} <= set(got.columns)
                check(f"Proteomics loader accepts its documented "
                      f"{label}", good_cols and len(got) > 0,
                      f"{len(got)} row(s), columns ok={good_cols}")
            except Exception as e:
                check(f"Proteomics loader accepts its documented {label}",
                      False, repr(e))
            for k in env:
                _os.environ.pop(k, None)
    finally:
        for k, v in saved.items():
            if v is None:
                _os.environ.pop(k, None)
            else:
                _os.environ[k] = v
        # BUG FIX (external review, correct): the module-level global was set
        # but never restored, so a later run in the same notebook session
        # would inherit the last temporary fixture as its proteomics input.
        globals()["PROTEOMICS_LOCAL_FILE"] = _saved_prot_global
        _reg_guard.__exit__(None, None, None)

    # ---- 6. END-TO-END loader behaviour, not just helpers ------------
    # External review, correctly: the tests above exercise
    # collapse_duplicate_symbols, detect_expression_scale, parse_event_status
    # and to_symbols, but did not prove that load_mcrpc_live() HONOURS the
    # canonical-scale guard or that load_clinical_live() validates an external
    # BCR file. Those are the functions the defects actually lived in.
    #
    # The loaders download and untar real archives, so they cannot run here.
    # What is testable without the network is the CONTRACT each loader must
    # satisfy, asserted against its own source: the guard must be read, not
    # merely set, and the validation must be reached before the merge.
    try:
        from inspect import getsource as _gs3
        msrc = _gs3(load_mcrpc_live)
        sets_flag = "scale_already_canonical = True" in msrc
        reads_flag = "if scale_already_canonical:" in msrc
        check("load_mcrpc_live READS the canonical-scale guard, not just "
              "sets it", sets_flag and reads_flag,
              f"set={sets_flag}, read={reads_flag} - an unread flag is how "
              f"the matrix was logged twice")
        guard_pos = msrc.find("if scale_already_canonical:")
        conv_pos = msrc.find("to_canonical_log(expr")
        check("The canonical guard precedes the second conversion",
              guard_pos != -1 and conv_pos != -1 and guard_pos < conv_pos,
              f"guard at {guard_pos}, conversion at {conv_pos}")
        # BEHAVIOURAL, not source-text. Self-review: the three assertions
        # that used to stand here searched load_clinical_live's source for
        # phrases ("duplicate patient", "BCR_ENDPOINT_FILE was set to") and
        # so would have passed if the raise were deleted and a comment
        # mentioning it survived. The validation is now a standalone function
        # that runs without the network, so these call it with fixture files
        # and assert what it DOES.
        _bdir = tmp / "bcr_fixtures"
        _bdir.mkdir(parents=True, exist_ok=True)
        _cohort = ["TCGA-AA-0001", "TCGA-AA-0002", "TCGA-AA-0003"]

        _good = _bdir / "good.csv"
        _good.write_text("patient,time_months,event_status\n"
                         "TCGA-AA-0001,12.5,1:Recurred\n"
                         "TCGA-AA-0002,30.0,0:DiseaseFree\n")
        _bcr, _nm = validate_external_bcr(_good, _cohort)
        check("validate_external_bcr accepts a well-formed BCR file",
              _nm == 2 and set(_bcr.surv_event.unique()) == {0, 1},
              f"{_nm} matched, events {sorted(_bcr.surv_event.unique())}")

        _dup = _bdir / "dup.csv"
        _dup.write_text("patient,time_months,event_status\n"
                        "TCGA-AA-0001,12.5,1:Recurred\n"
                        "TCGA-AA-0001,30.0,0:DiseaseFree\n")
        try:
            validate_external_bcr(_dup, _cohort)
            check("Duplicate patient keys are REJECTED, not merged", False,
                  "a left merge on duplicated keys multiplies rows and "
                  "inflates the event count")
        except ValueError as _e:
            check("Duplicate patient keys are REJECTED, not merged",
                  "duplicate patient" in str(_e).lower(), str(_e)[:70])

        _neg = _bdir / "neg.csv"
        _neg.write_text("patient,time_months,event_status\n"
                        "TCGA-AA-0001,-3.0,1:Recurred\n"
                        "TCGA-AA-0002,30.0,0:DiseaseFree\n")
        try:
            validate_external_bcr(_neg, _cohort)
            check("Negative follow-up times are REJECTED", False,
                  "negative time to event accepted")
        except ValueError as _e:
            check("Negative follow-up times are REJECTED",
                  "negative" in str(_e).lower(), str(_e)[:70])

        _nomatch = _bdir / "nomatch.csv"
        _nomatch.write_text("patient,time_months,event_status\n"
                            "OTHER-XX-9999,12.5,1:Recurred\n")
        try:
            validate_external_bcr(_nomatch, _cohort)
            check("A zero-match cohort join is REJECTED", False,
                  "an all-missing endpoint would yield an empty analysis")
        except ValueError as _e:
            check("A zero-match cohort join is REJECTED",
                  "no patient identifier matched" in str(_e), str(_e)[:70])

        _junk = _bdir / "junk.csv"
        _junk.write_text("patient,time_months,event_status\n"
                         "TCGA-AA-0001,12.5,probably recurred?\n")
        try:
            validate_external_bcr(_junk, _cohort)
            check("Unrecognised event codes are REJECTED, not guessed", False,
                  "free text entered the survival analysis unchallenged")
        except Exception as _e:
            check("Unrecognised event codes are REJECTED, not guessed",
                  True, type(_e).__name__ + ": " + str(_e)[:60])

        # ---- REAL end-to-end execution of load_mcrpc_live ------------
        # Both external reviews, correctly: asserting that
        # "if scale_already_canonical:" appears in the source proves a line
        # exists, not that the matrix is transformed exactly once. The
        # loader's only untestable step is the network download, so that one
        # call is mocked and the ACTUAL loader runs against a miniature
        # cBioPortal-format archive with a known answer.
        import tarfile as _tf, shutil as _sh
        _sudir = tmp / "su2c_fixture"
        _sudir.mkdir(parents=True, exist_ok=True)
        _inner = _sudir / "prad_su2c_2019"
        _inner.mkdir(exist_ok=True)
        # Two samples; FOLH1 duplicated on purpose; one gene outside the
        # universe; one lineage marker that must survive the universe filter.
        (_inner / "data_mrna_seq_tpm.txt").write_text(
            "Hugo_Symbol\tEntrez_Gene_Id\tS1\tS2\n"
            "FOLH1\t2346\t2.0\t4.0\n"
            "FOLH1\t2346\t3.0\t4.0\n"
            "STEAP1\t26872\t8.0\t1.0\n"
            "AR\t367\t5.0\t6.0\n"
            "NOTINUNIVERSE\t999999\t7.0\t7.0\n")
        (_inner / "data_clinical_sample.txt").write_text(
            "#SAMPLE_ID\tPATIENT_ID\nSAMPLE_ID\tPATIENT_ID\n"
            "S1\tP1\nS2\tP2\n")
        _tar = _sudir / "prad_su2c_2019.tar.gz"
        with _tf.open(_tar, "w:gz") as _t:
            _t.add(_inner, arcname="prad_su2c_2019")

        _orig_download = globals()["download"]
        _orig_raw = DIRS["raw"]
        _saved_pol = os.environ.get("SU2C_DUPLICATE_POLICY")
        _saved_suscale = os.environ.get("SU2C_SCALE")
        try:
            DIRS["raw"] = tmp / "su2c_raw"
            DIRS["raw"].mkdir(parents=True, exist_ok=True)
            globals()["download"] = (
                lambda url, dest, minsize=0, **kw: (
                    _sh.copy(_tar, dest), dest)[1])
            _universe = ["FOLH1", "STEAP1"]

            # (a) Duplicate symbols must REFUSE without a declared policy.
            os.environ.pop("SU2C_DUPLICATE_POLICY", None)
            os.environ["SU2C_SCALE"] = "linear_tpm"
            try:
                load_mcrpc_live(_universe)
                check("load_mcrpc_live REFUSES undeclared duplicate symbols",
                      False, "a silent collapse policy was applied")
            except ValueError as _e:
                check("load_mcrpc_live REFUSES undeclared duplicate symbols",
                      "duplicate gene symbol" in str(_e), str(_e)[:70])

            # (b) The canonical-scale guard: 2 + 3 TPM summed on the LINEAR
            # scale is log2(5+1) = 2.585. A second transformation of that
            # value would give log2(2.585+1) = 1.842. This asserts the
            # ARITHMETIC, so deleting the guard fails the test.
            # SELF-REVIEW: rmtree on a path read back from a MUTABLE global
            # is one failed assignment away from deleting the real raw-data
            # directory. The path is asserted to be inside this test's
            # temporary directory before anything is removed.
            _rawp = Path(DIRS["raw"]).resolve()
            if not str(_rawp).startswith(str(Path(tmp).resolve())):
                raise RuntimeError(
                    f"refusing to remove {_rawp}: not inside the test "
                    f"temporary directory {tmp}")
            _sh.rmtree(_rawp, ignore_errors=True)
            DIRS["raw"].mkdir(parents=True, exist_ok=True)
            os.environ["SU2C_DUPLICATE_POLICY"] = "sum_linear"
            _expr, _ph = load_mcrpc_live(_universe)[:2]
            _got = float(_expr.loc["FOLH1", "S1"])
            _want = float(np.log2(5.0 + 1.0))
            check("load_mcrpc_live transforms the matrix EXACTLY ONCE "
                  "(2+3 TPM -> log2(6), not log2(log2(6)+1))",
                  abs(_got - _want) < 1e-9,
                  f"got {_got:.4f}, want {_want:.4f}; a second conversion "
                  f"would give {np.log2(_want + 1):.4f}")
            check("load_mcrpc_live keeps lineage markers through the "
                  "universe filter", "AR" in _expr.index,
                  f"index: {sorted(_expr.index)}")
            check("load_mcrpc_live drops genes outside the universe",
                  "NOTINUNIVERSE" not in _expr.index,
                  f"index: {sorted(_expr.index)}")
            check("load_mcrpc_live returns both cohort samples",
                  list(_expr.columns) == ["S1", "S2"],
                  f"columns: {list(_expr.columns)}")
        finally:
            globals()["download"] = _orig_download
            DIRS["raw"] = _orig_raw
            for _k, _v in (("SU2C_DUPLICATE_POLICY", _saved_pol),
                           ("SU2C_SCALE", _saved_suscale)):
                if _v is None:
                    os.environ.pop(_k, None)
                else:
                    os.environ[_k] = _v

        # ---- REAL end-to-end execution of load_clinical_live ---------
        # External review, correctly: the BCR helpers are behaviourally
        # tested but the successful load_clinical_live() pathway was only
        # asserted by source text. Same technique as the SU2C fixture - a
        # miniature cBioPortal clinical archive with the download mocked -
        # so the loader itself runs and both endpoint paths are exercised.
        _cdir = tmp / "clin_fixture"
        _cdir.mkdir(parents=True, exist_ok=True)
        _cinner = _cdir / "prad_tcga"
        _cinner.mkdir(exist_ok=True)
        (_cinner / "data_clinical_patient.txt").write_text(
            "#Patient Identifier\tDFS months\tDFS status\tRace\tAge\n"
            "PATIENT_ID\tDFS_MONTHS\tDFS_STATUS\tRACE\tAGE\n"
            "TCGA-AA-0001\t12.5\t1:Recurred\tWHITE\t61\n"
            "TCGA-AA-0002\t30.0\t0:DiseaseFree\tBLACK\t66\n"
            "TCGA-AA-0003\t44.0\t0:DiseaseFree\tASIAN\t58\n")
        _ctar = _cdir / "prad_tcga.tar.gz"
        with _tf.open(_ctar, "w:gz") as _t:
            _t.add(_cinner, arcname="prad_tcga")
        _saved_ep = CFG.survival_endpoint
        try:
            DIRS["raw"] = tmp / "clin_raw"
            DIRS["raw"].mkdir(parents=True, exist_ok=True)
            globals()["download"] = (
                lambda url, dest, minsize=0, **kw: (
                    _sh.copy(_ctar, dest), dest)[1])
            _samples = ["TCGA-AA-0001-01", "TCGA-AA-0002-01",
                        "TCGA-AA-0003-01"]
            CFG.survival_endpoint = "auto"
            _clin = load_clinical_live(_samples)
            check("load_clinical_live runs end to end on a miniature "
                  "archive", len(_clin) == 3, f"{len(_clin)} row(s)")
            check("load_clinical_live decodes event codes through the "
                  "strict parser",
                  sorted(pd.to_numeric(_clin.surv_event,
                                       errors="coerce").dropna().unique())
                  == [0.0, 1.0],
                  f"events {sorted(_clin.surv_event.dropna().unique())}")
            check("load_clinical_live labels the endpoint it actually used",
                  set(_clin.endpoint_name) == {"disease_free_survival"},
                  f"{set(_clin.endpoint_name)}")
            check("load_clinical_live names BOTH source columns, so a mixed "
                  "pair is visible in the provenance",
                  all("DFS_MONTHS" in str(v) and "DFS_STATUS" in str(v)
                      for v in _clin.endpoint_source_column),
                  f"{set(_clin.endpoint_source_column)}")
            check("Self-reported race is NOT relabelled as genetic ancestry",
                  set(_clin.get("ancestry_source", pd.Series(["self"])))
                  == {"self_reported_race"}
                  or "genetic" not in set(
                      _clin.get("ancestry_source", pd.Series([]))),
                  f"{set(_clin.get('ancestry_source', pd.Series([])))}")

            # A DECLARED endpoint that the archive cannot supply must be
            # refused, not quietly swapped for the other composite.
            _sh.rmtree(DIRS["raw"], ignore_errors=True)
            DIRS["raw"].mkdir(parents=True, exist_ok=True)
            CFG.survival_endpoint = "PFS"
            try:
                load_clinical_live(_samples)
                check("A declared endpoint absent from the file is REFUSED",
                      False, "PFS was requested; DFS was substituted")
            except (ValueError, KeyError) as _e:
                check("A declared endpoint absent from the file is REFUSED",
                      True, f"{type(_e).__name__}: {str(_e)[:60]}")
        finally:
            CFG.survival_endpoint = _saved_ep
            globals()["download"] = _orig_download
            DIRS["raw"] = _orig_raw

        # ---- The endpoint CONTRACT (both reviews' top P0) -------------
        # Each reviewer reproduced this by executing the function: a
        # declared BCR endpoint with no BCR file returned DFS. These use
        # their exact scenarios.
        check("BCR requested with NO BCR file is REFUSED, even when "
              "complete DFS columns exist",
              _raises(lambda: validate_endpoint_configuration("BCR", ""),
                      ValueError),
              "returning DFS for a declared BCR endpoint invalidates every "
              "survival figure and sentence naming the endpoint")
        check("BCR requested with a non-existent BCR file is REFUSED",
              _raises(lambda: validate_endpoint_configuration(
                  "BCR", str(tmp / "nope.csv")), FileNotFoundError))
        check("DFS requested WITH a BCR file set is a configuration "
              "CONFLICT, not a silent override",
              _raises(lambda: validate_endpoint_configuration(
                  "DFS", str(_good)), ValueError),
              "the BCR branch would take precedence over the declaration")
        check("An unknown endpoint name is REFUSED",
              _raises(lambda: validate_endpoint_configuration("XYZ", ""),
                      ValueError))
        check("'auto' with no BCR file is permitted - the escape hatch is "
              "explicit",
              validate_endpoint_configuration("auto", "") == "AUTO")
        check("BCR with a real BCR file is permitted",
              validate_endpoint_configuration("BCR", str(_good)) == "BCR")

        # ---- Usable endpoint coverage, not ID matching ----------------
        # The reviewers' scenario exactly: 2/2 identifiers matched, 100%
        # reported coverage, 0/2 usable follow-up times, accepted.
        _badtime = _bdir / "badtime.csv"
        _badtime.write_text("patient,time_months,event_status\n"
                            "TCGA-AA-0001,unknown,1:Recurred\n"
                            "TCGA-AA-0002,missing,0:DiseaseFree\n")
        try:
            validate_external_bcr(_badtime, _cohort)
            check("Non-numeric follow-up times are REJECTED, not coerced "
                  "to missing", False,
                  "100% identifier coverage was reported on data with no "
                  "analysable follow-up")
        except ValueError as _e:
            check("Non-numeric follow-up times are REJECTED, not coerced "
                  "to missing", "non-numeric" in str(_e).lower(),
                  str(_e)[:70])
        # Blank follow-up IS a legitimate censoring representation, so it
        # parses - but it must not count toward usable coverage.
        _blank = _bdir / "blank.csv"
        _blank.write_text("patient,time_months,event_status\n"
                          "TCGA-AA-0001,,1:Recurred\n"
                          "TCGA-AA-0002,,0:DiseaseFree\n"
                          "TCGA-AA-0003,,0:DiseaseFree\n")
        try:
            validate_external_bcr(_blank, _cohort)
            check("All-missing follow-up fails the USABLE coverage gate",
                  False, "identifier matching was 100%")
        except ValueError as _e:
            check("All-missing follow-up fails the USABLE coverage gate",
                  "usable" in str(_e).lower() or "valid follow-up"
                  in str(_e).lower(), str(_e)[:70])

        # ---- The central clinical join --------------------------------
        _smap = pd.DataFrame({"sample": ["S1", "S2", "S3"],
                              "patient": ["P1", "P1", "P2"]})
        _clin1 = pd.DataFrame({"patient": ["P1", "P2"],
                               "surv_time": [10.0, 20.0]})
        _m, _rate = merge_clinical_to_samples(_smap, _clin1)
        check("Multiple samples per patient are preserved without "
              "duplicating clinical rows",
              len(_m) == 3 and _rate == 1.0, f"{len(_m)} row(s), "
              f"match {_rate:.0%}")
        _clin_dup = pd.DataFrame({"patient": ["P1", "P1", "P2"],
                                  "surv_time": [10.0, 11.0, 20.0]})
        check("A duplicate patient-level clinical record is REFUSED even "
              "with no ancestry or BCR file supplied",
              _raises(lambda: merge_clinical_to_samples(_smap, _clin_dup),
                      ValueError),
              "the guards added last round were conditional on those "
              "optional inputs")
        _smap_dup = pd.DataFrame({"sample": ["S1", "S1"],
                                  "patient": ["P1", "P2"]})
        check("A duplicate SAMPLE identifier is REFUSED",
              _raises(lambda: merge_clinical_to_samples(_smap_dup, _clin1),
                      ValueError))

        csrc = _gs3(load_clinical_live)
        check("load_clinical_live raises on a requested-but-missing BCR file",
              "BCR_ENDPOINT_FILE was set to" in csrc,
              "a missing requested file must not fall back to DFS/PFS")
        check("load_clinical_live routes external BCR through the validator",
              "validate_external_bcr(bcr_file" in csrc,
              "validation must not be re-inlined and diverge")
    except Exception as e:
        check("end-to-end loader contract checks executed", False, repr(e))

    # ---- 7. Tissue-label harmonisation must actually resolve -----------
    try:
        probe_cols = ["Liver", "Lung", "Minor Salivary Gland",
                      "Kidney - Cortex", "Brain - Cortex",
                      "Heart - Left Ventricle", "Whole Blood",
                      "Nerve - Tibial"]
        res, miss = harmonise_tissue_labels(probe_cols, verbose=False)
        check("GTEx v8 'Minor Salivary Gland' resolves to the declared "
              "'Salivary Gland'",
              res.get("Salivary Gland") == "Minor Salivary Gland",
              f"resolved to {res.get('Salivary Gland')!r}")
        check("Unmatched declared organs are reported, not silently dropped",
              isinstance(miss, list),
              f"{len(miss)} organ(s) reported missing from the probe panel")
    except Exception as e:
        check("tissue harmonisation checks executed", False, repr(e))

    # ---- LIVE FIXTURE: load_transcriptomes_live and its new cache ------
    # This function had NO executable test - only source inspection. The
    # filtered-matrix cache added this round would therefore have run for
    # the very first time on Colab, against real data, which is exactly the
    # situation these fixtures exist to prevent. A miniature synthetic Toil
    # matrix exercises the parse, the cache WRITE, and the cache HIT.
    _tl = Path(tempfile.mkdtemp(prefix="toilfix_"))
    _dirs_save = dict(DIRS)
    _dl_save = globals().get("download")
    try:
        (_tl / "raw").mkdir(parents=True, exist_ok=True)
        DIRS["raw"] = _tl / "raw"
        _genes = ["ERBB2", "FOLH1", "EPCAM", "KLK3", "AR", "PTPRC", "EGFR"]
        _ens = [f"ENSG{i:011d}" for i in range(len(_genes))]
        _samples = ([f"TCGA-PR-{i:04d}-01" for i in range(6)]
                    + [f"TCGA-PR-{i:04d}-11" for i in range(3)]
                    + [f"GTEX-{i:04d}-0001" for i in range(5)])
        _n_gtex_tissues = 5
        _rng_t = np.random.default_rng(11)
        _vals = _rng_t.uniform(0.5, 9.0, size=(len(_ens), len(_samples)))
        _tpm = pd.DataFrame(_vals, index=[f"{e}.1" for e in _ens],
                            columns=_samples)
        _tpm_f = _tl / "raw" / "toil_tpm.gz"
        _tpm.to_csv(_tpm_f, sep="\t", compression="gzip")
        _pheno = pd.DataFrame({
            # Five DISTINCT GTEx tissues. The first version of this
            # fixture gave all five the same tissue and then expected five
            # GTEx columns - but GTEx is deliberately collapsed to a
            # per-tissue MEDIAN, so one tissue is one column. The fixture
            # was wrong, not the loader; distinct tissues also exercise the
            # aggregation rather than bypassing it.
            "primary_site": (["Prostate"] * 9)
                            + ["Lung", "Liver", "Heart", "Kidney", "Brain"],
            "sample_type": (["Primary Tumor"] * 6
                            + ["Solid Tissue Normal"] * 3
                            + ["Normal Tissue"] * 5),
            "_study": (["TCGA"] * 9) + ["GTEX"] * 5,
        }, index=_samples)
        _phe_f = _tl / "raw" / "toil_pheno.txt.gz"
        _pheno.to_csv(_phe_f, sep="\t", compression="gzip")
        _pm = pd.DataFrame({"id": [f"{e}.1" for e in _ens], "gene": _genes})
        _pm_f = _tl / "raw" / "gencode_probemap.tsv"
        _pm.to_csv(_pm_f, sep="\t", index=False)

        # SELF-REVIEW: the first version of this counted download() calls,
        # but download() runs on every invocation (it is a cheap cache hit
        # on the FILE) - so the count could never distinguish a working
        # matrix cache from a broken one. The pipeline now counts the
        # expensive thing itself, the full-matrix parse, in
        # TOIL_PARSE_COUNT.
        TOIL_PARSE_COUNT["n"] = 0
        _parse_count = TOIL_PARSE_COUNT

        def _fake_download(url, dest, expect_min_bytes=0, **kw):
            _name = Path(dest).name
            _src = {"toil_tpm.gz": _tpm_f, "toil_pheno.txt.gz": _phe_f,
                    "gencode_probemap.tsv": _pm_f}.get(_name)
            if _src is None:
                raise AssertionError(f"unexpected download: {_name}")
            # SELF-REVIEW: the first version of this stub returned the file
            # without registering it. The real download() always registers
            # an input digest, and the filtered-matrix cache now REFUSES to
            # key itself on anything weaker than a digest - so the stub's
            # omission disabled the cache and the cache tests failed
            # against correct code. A stub that skips the side effect its
            # caller depends on does not stand in for the real function.
            register_input_file(_src, role=f"fixture input: {url}")
            return _src
        globals()["download"] = _fake_download

        _t0 = time.time()
        _pt, _pn, _gt = load_transcriptomes_live(set(_genes))
        _first_s = time.time() - _t0
        check("load_transcriptomes_live runs end to end on a miniature Toil "
              "matrix", _pt.shape[1] == 6 and _pn.shape[1] == 3
              and _gt.shape[1] == _n_gtex_tissues,
              f"tumour {_pt.shape}, normal {_pn.shape}, gtex {_gt.shape} "
              f"(GTEx is one column per TISSUE, not per sample)")
        # Count the cache DATA files only. The strengthened cache writes a
        # .manifest.json beside each one, so a bare glob counts two per
        # cache and these assertions failed against correct code.
        def _cache_files():
            # Excludes temp files. An earlier version matched on suffix
            # alone, so a stray "...npz.tmp.npz" left behind by a failed
            # atomic commit satisfied "the cache file is WRITTEN" - the test
            # passed on the debris of the bug it was meant to catch.
            return [f for f in (_tl / "raw").glob("toil_surfaceome_filtered_*")
                    if f.suffix in (".parquet", ".npz")
                    and ".tmp" not in f.name]
        _caches = _cache_files()
        check("The filtered-matrix cache file is actually WRITTEN during a "
              "parse", len(_caches) == 1,
              f"{[c.name for c in _caches]}")
        # Second call must hit the cache and return the SAME numbers.
        _t1 = time.time()
        _pt2, _pn2, _gt2 = load_transcriptomes_live(set(_genes))
        _second_s = time.time() - _t1
        check("A cached re-run returns IDENTICAL expression values, so the "
              "cache cannot change a result",
              bool(np.allclose(_pt.to_numpy(dtype=float),
                               _pt2.to_numpy(dtype=float), atol=0,
                               rtol=0, equal_nan=True))
              and list(_pt.index) == list(_pt2.index)
              and list(_pt.columns) == list(_pt2.columns),
              f"max |delta| "
              f"{np.nanmax(np.abs(_pt.to_numpy(dtype=float) - _pt2.to_numpy(dtype=float))):.3e}")
        check("A cached re-run does not re-read the source matrix",
              _parse_count["n"] == 1,
              f"full-matrix parses across two runs: "
              f"{_parse_count['n']}; 2 means the cache was written and then "
              f"silently discarded (first {_first_s:.2f}s, second "
              f"{_second_s:.2f}s)")
        check("The cache that was written is READABLE by the code that "
              "wrote it",
              _parse_count["n"] == 1 and _pt2.shape == _pt.shape,
              "an npz cache of object arrays is unreadable with "
              "allow_pickle=False, and the failure looks like a cold start")
        # Tamper with the cached VALUES, leaving genes, samples and shape
        # untouched: the defect both reviews reproduced.
        _cf = _cache_files()[0]
        if _cf.suffix == ".npz":
            _z = np.load(_cf, allow_pickle=False)
            _vals = _z["values"].copy()
            _vals[0, 0] = _vals[0, 0] * 7.0 + 1.0
            np.savez_compressed(_cf, values=_vals, index=_z["index"],
                                columns=_z["columns"])
        else:
            _d = pd.read_parquet(_cf)
            _d.iloc[0, 0] = _d.iloc[0, 0] * 7.0 + 1.0
            _d.to_parquet(_cf, compression="snappy")
        _parse_count["n"] = 0
        _pt5, _, _ = load_transcriptomes_live(set(_genes))
        check("A cached matrix whose VALUES were altered is rejected and "
              "rebuilt from the verified source",
              _parse_count["n"] == 1,
              "identifiers and shape are unchanged, so only a digest of the "
              "cache file itself can detect this")
        check("The rebuilt matrix matches the original, so rejection "
              "recovers rather than propagating the tampered values",
              bool(np.allclose(_pt5.to_numpy(dtype=float),
                               _pt.to_numpy(dtype=float), equal_nan=True)),
              "the source is the authority, not the cache")
        # A DIFFERENT gene universe must MISS the cache, not silently serve
        # the previous matrix - the failure that would be hardest to notice.
        _pt3, _, _ = load_transcriptomes_live(set(_genes[:4]))
        check("A different gene universe MISSES the cache rather than "
              "serving a matrix built for another universe",
              len(_cache_files()) == 2
              and _pt3.shape[0] <= _pt.shape[0],
              f"{_pt3.shape[0]} genes vs {_pt.shape[0]} for the full set")
        # A corrupt cache must be re-parsed, not crash the run.
        _caches2 = sorted(_cache_files())
        _caches2[0].write_bytes(b"not a parquet or npz file at all")
        _pt4, _, _ = load_transcriptomes_live(set(_genes))
        check("A CORRUPT filtered-matrix cache falls back to re-parsing "
              "instead of raising",
              _pt4.shape == _pt.shape,
              f"recovered to {_pt4.shape}")
        # ROUND 32 fixtures: the attested file must BE the loaded file.
        _pt5, _, _ = load_transcriptomes_live(set(_genes))
        _cf = _cache_files()
        _man = sorted((_tl / "raw").glob("*.manifest.json"))
        # SELF-REVIEW: the first form asserted len(_cf) == 1, but earlier
        # steps in this fixture deliberately create a SECOND cache under a
        # different key (the changed-gene-universe case), so the count was
        # never going to be 1 and the check failed against correct code.
        # The property that actually matters is one data file PER KEY.
        _by_key = {}
        for _f in _cf:
            _by_key.setdefault(
                _f.stem.replace("toil_surfaceome_filtered_", ""),
                []).append(_f.suffix)
        check("No cache key has more than one data file",
              all(len(_v) == 1 for _v in _by_key.values()),
              f"{ {k: v for k, v in _by_key.items()} }")
        _attested = sorted(_cf, key=lambda f: f.stat().st_mtime)[-1]

        def _write_as(path, frame):
            """Write `frame` to `path` in the format its suffix implies.

            SELF-REVIEW: the first version always used to_parquet() for the
            .parquet case, which raises here because pyarrow is not
            installed - so the conflict fixture errored instead of testing
            anything. When parquet cannot be written, arbitrary bytes are
            planted instead: the property under test is that the loader
            never OPENS the unattested file, and a file it must not open
            need not be a valid matrix. Returns what it planted.
            """
            if path.suffix == ".npz":
                np.savez_compressed(
                    path, values=frame.to_numpy(dtype="float32"),
                    index=np.asarray(frame.index.astype(str), dtype="U"),
                    columns=np.asarray(frame.columns.astype(str), dtype="U"))
                return "matrix"
            try:
                frame.to_parquet(path)
                return "matrix"
            except Exception:
                path.write_bytes(b"UNATTESTED - must never be opened\n" * 64)
                return "bytes"

        # Plant a SECOND file in the other format, with different contents,
        # for the same key. This is the exact state the defect needed.
        _other = (_attested.with_suffix(".npz")
                  if _attested.suffix == ".parquet"
                  else _attested.with_suffix(".parquet"))
        try:
            _bogus = _pt5.copy()
            if _bogus.size:
                _bogus.iloc[0, 0] = float(_bogus.iloc[0, 0]) + 999.0
            _write_as(_other, _bogus)
            _pt6, _, _ = load_transcriptomes_live(set(_genes))
            check("With both cache formats present and DISAGREEING, the "
                  "attested file is the one used",
                  bool(np.allclose(_pt5.to_numpy(dtype=float),
                                   _pt6.to_numpy(dtype=float),
                                   equal_nan=True)),
                  "the planted file in the other format is ignored")
        except Exception as _e:
            check("Both-format conflict fixture ran", False, repr(_e))
        finally:
            if _other.is_file():
                _other.unlink()

        # A manifest attesting a file that is not the one present.
        _mf = _man[0]
        _saved = _mf.read_text()
        try:
            _m = json.loads(_saved)
            _m["cache_filename"] = "some_other_file.parquet"
            _mf.write_text(json.dumps(_m))
            _pt7, _, _ = load_transcriptomes_live(set(_genes))
            check("A manifest naming a file other than the one present "
                  "causes a rebuild, not a false pass",
                  _pt7.shape == _pt5.shape,
                  "recovered by re-parsing from the verified source")
        except Exception as _e:
            check("Manifest-mismatch fixture ran", False, repr(_e))
        finally:
            if _mf.is_file():
                _mf.write_text(_saved)

        # One altered expression value; identifiers and shape untouched.
        try:
            _att2 = _cache_files()
            if _att2:
                _tamper = _pt5.copy()
                _tamper.iloc[0, 0] = float(_tamper.iloc[0, 0]) + 123.0
                _write_as(_att2[0], _tamper)
                _pt8, _, _ = load_transcriptomes_live(set(_genes))
                check("A single ALTERED expression value, identifiers and "
                      "shape unchanged, is rejected and rebuilt",
                      bool(np.allclose(_pt5.to_numpy(dtype=float),
                                       _pt8.to_numpy(dtype=float),
                                       equal_nan=True)),
                      "the contents digest catches what identity checks "
                      "cannot")
        except Exception as _e:
            check("Altered-value fixture ran", False, repr(_e))
    except Exception as _e:
        check("load_transcriptomes_live fixture executed", False, repr(_e))
    finally:
        if _dl_save is not None:
            globals()["download"] = _dl_save
        DIRS.clear()
        DIRS.update(_dirs_save)
        __import__("shutil").rmtree(_tl, ignore_errors=True)

    df = pd.DataFrame(results)
    n_pass = int(df.passed.sum())
    if verbose:
        show_heading("LIVE-path integration tests (miniature fixtures)")
        print(df.to_string(index=False))
        print(f"\n    {n_pass}/{len(df)} passed")
        if n_pass < len(df):
            print("    A FAILURE HERE IS A LIVE-DATA DEFECT. These paths are "
                  "unreachable from DEMO, so nothing else in the suite can "
                  "catch them.")
    save_table(df, "table29_live_path_integration_tests.csv",
               caption=f"LIVE-path integration tests on miniature fixtures "
                       f"({n_pass}/{len(df)} passed): expression scales and "
                       f"duplicate collapse, declared-scale domain limits, "
                       f"clinical event decoding, symbol duplicates, and both "
                       f"documented proteomics input formats. These exercise "
                       f"code DEMO cannot reach.")
    return df


# =============================================================================
# SCRIPT ENTRY POINT - must be the LAST statement in this module
# =============================================================================
# BUG FIX (found by extracting the delivered archive and running it the
# obvious way): this block sat at line ~1485, roughly 500 lines BEFORE
# RELEASE_STATE and RELEASE_APPROVAL were defined. Executing the module as a
# script therefore called run_pipeline() while those globals did not yet
# exist, and died with
#
#     NameError: name 'RELEASE_APPROVAL' is not defined
#
# The defect was invisible from every route actually used during development:
# run_test.py STRIPS this block before exec'ing, and the notebook puts the
# call in a later cell, so both reorder it by accident. Only an independent
# analyst running the archived source as a plain script would hit it - which
# is precisely the reproducibility claim both external reviews asked to be
# demonstrated, and the first time anyone tried it, it failed.
#
# The entry point belongs last, after every definition. Keeping it here is
# asserted by a regression test so it cannot drift back up the file.
# =============================================================================
# POST-RUN VERIFICATION
# =============================================================================
# MOVED HERE from a notebook cell (both external reviews, correct). While it
# lived only in the notebook, the packaged standalone runner had no access to
# it, so a script run could never pass the release gate the notebook applies.
# Defining it in the shared sections is what actually makes the two
# execution paths equivalent; execute_validated_analysis() below calls it.
def verify(summary=None):
    # GRACEFUL DEGRADATION: if the pipeline cell raised, `summary` never got
    # bound and this cell previously failed with a bare NameError that told
    # the reader nothing about the real problem. Say what actually happened.
    #
    # BUG FIX: this used to `return False`, the same value a run that
    # executed and failed its checks returns. Both outcomes then hit
    # `if _verified is not True: raise RuntimeError("...verification
    # failed...")`, so a pipeline that never ran was reported with the exact
    # same "verification failed" wording as a pipeline that ran and failed -
    # directly contradicting the "scroll up, that's the real error" message
    # printed two lines above it. A boolean that conflates "failed" with
    # "could not be assessed" is the defect this project's own checklist
    # (1.6) warns against; it reached this function anyway. Returning None
    # lets every caller tell the two apart and raise the right message.
    if summary is None:
        summary = globals().get("summary")
    if summary is None:
        print("=" * 74)
        print("VERIFICATION SKIPPED - the pipeline did not complete.")
        print("Scroll up to the first traceback: that is the real error.")
        print("This cell depends on `summary`, which the pipeline cell binds "
              "on success, so it cannot run until that cell does.")
        print("=" * 74)
        return None  # NOT ASSESSABLE - distinct from False (ran and failed)
    checks, warn = [], []

    # BUG FIX (both external reviews, independently reproduced): this was
    # read behind `if ut.exists():` with no else, so DELETING the filter
    # unit-test report left verify() returning True - the check simply
    # vanished. A mandatory piece of release evidence that can disappear
    # without failing verification is not evidence.
    # SELF-REVIEW, caught by verify()'s own self-scan on its first run: my
    # first version of this check put a literal True in the else branch.
    # Reached only on success, but the scanner bans literal True in
    # straight-line code for good reason - all three genuine always-pass
    # labels found in this project sat in exactly such a branch - and
    # whitelisting if/else would have gutted it. A real boolean is clearer
    # anyway.
    ut = DIRS["logs"] / "filter_unit_tests.csv"
    _ut_ok = ut.is_file() and ut.stat().st_size > 0
    checks.append(("Filter unit-test report present", _ut_ok,
                   f"{ut.name} present" if _ut_ok else
                   f"{ut.name} is missing or empty - its checks cannot run, "
                   f"and a check that does not run has not passed"))
    if ut.exists():
        u = pd.read_csv(ut)
        checks.append(("Filter unit tests", (u.test == "PASS").all(),
                       f"{(u.test=='PASS').sum()}/{len(u)} controls behaved as designed"))

    hb = summary["heldout_benchmark_result"]
    checks.append(("Held-out benchmarks in top decile",
                   hb["n_in_top_decile"] >= max(1, hb["n_heldout"] // 2),
                   f"{hb['n_in_top_decile']}/{hb['n_heldout']} "
                   f"(median percentile {hb['median_heldout_percentile']:.1f})"))
    if "permutation_p" in hb:
        checks.append(("Benchmark enrichment vs background",
                       hb["permutation_p"] < 0.05,
                       f"permutation p = {hb['permutation_p']:.2e}"))

    c = summary["curves"]
    # The null and the observed statistic must come from the SAME procedure,
    # so the matched observed AUC is what gets compared - not the out-of-fold
    # value, which a different procedure produced.
    obs_m = c.get("matched_observed_auc", c.get("ensemble_oof_auc"))
    pv = c.get("permutation_p")
    floor = c.get("permutation_resolution")
    ptxt = (f"p < {floor:.3g}" if c.get("permutation_p_at_resolution_floor")
            else f"p = {pv:.3g}") if pv is not None else "p unavailable"
    checks.append(("Matched observed AUC exceeds the permutation null",
                   obs_m > c["null_auc_p95"],
                   f"matched observed {obs_m:.3f} vs null 95th pct "
                   f"{c['null_auc_p95']:.3f} ({ptxt}, B="
                   f"{c.get('permutation_n_resamples', '?')})"))

    # Each interval must contain its own point estimate. This caught a real
    # bug: the exported "auc_ci" was the average-precision interval.
    for pt, ci_key, name in [("ensemble_oof_auc", "roc_auc_ci", "ROC-AUC"),
                             ("ensemble_ap", "average_precision_ci",
                              "average precision")]:
        if pt in c and ci_key in c and c[ci_key]:
            lo, hi = c[ci_key]
            checks.append((f"{name} interval contains its own estimate",
                           lo <= c[pt] <= hi,
                           f"{name} {c[pt]:.3f} in [{lo:.3f}, {hi:.3f}]"))

    perf = pd.DataFrame(summary["model_performance_auc"])
    if len(perf):
        full = perf[perf.feature_set == "biology+research_intensity"]["mean"].mean()
        bio = perf[perf.feature_set == "biology_only"]["mean"].mean()
        drop = full - bio
        checks.append(("Leakage ablation: biology alone still performs",
                       bio > 0.65,
                       f"AUC {bio:.3f} biology-only vs {full:.3f} with "
                       f"research-intensity features (drop {drop:+.3f})"))
        if drop > 0.05:
            warn.append(f"Research-intensity features add {drop:.3f} AUC. Report the "
                        f"biology-only model as the primary result and state this gap.")

    if summary["cox_refused_low_epv"] > 0:
        warn.append(f"{summary['cox_refused_low_epv']} Cox models refused for "
                    f"insufficient events per variable. Any ancestry-stratified claim "
                    f"must be reduced to a pre-specified descriptive analysis.")
    by_ctx = summary.get("shortlists_by_context") or {}
    empty = [k for k, v in by_ctx.items() if v == 0]
    if by_ctx and len(empty) == len(by_ctx):
        warn.append("No candidate reached any disease-context shortlist. "
                    "Loosen a threshold in CFG deliberately and transparently, "
                    "or report the null result.")
    elif empty:
        warn.append(f"No candidates shortlisted for: {', '.join(empty)}. "
                    f"Check whether that context was assessable at all before "
                    f"reporting it as an empty result.")

    # BUG FIX (both external reviews, correct): this appended a literal
    # True. It reported a COUNT of controls removed and asserted nothing
    # about whether any control actually survived into a shortlist - the
    # property the label claims. The check now reads the exported ranking
    # and looks for control genes in the candidate output, so planting one
    # back makes it fail.
    n_ctrl = summary.get("n_synthetic_controls_excluded", 0)
    if n_ctrl:
        _ctrl_found = []
        try:
            _rk = pd.read_csv(DIRS["tables"] /
                              "table2_full_candidate_ranking.csv")
            _is_ctrl = _rk.gene.astype(str).str.startswith("CTRL_")
            # SELF-REVIEW, caught by the new release gate on its first run:
            # my first version included `readiness` among the "candidate
            # columns" and treated any non-empty string as a flag. readiness
            # is a DESCRIPTIVE label carried by every gene, so all four
            # controls matched and a correct pipeline failed verification.
            # A check must test the property it names: candidacy is carried
            # by the boolean candidacy flags only.
            _cand_cols = [c for c in ("is_unlabeled_candidate", "is_novel")
                          if c in _rk.columns]
            for _c in _cand_cols:
                _flag = _rk[_c].fillna(False).astype(bool)
                _ctrl_found += _rk.gene[_is_ctrl & _flag].tolist()
        except Exception as _e:
            _ctrl_found = [f"<could not read ranking: {_e!r}>"]
        checks.append(("Engineered controls excluded from candidacy",
                       not _ctrl_found,
                       f"{n_ctrl} control(s) removed; "
                       + ("none present in any candidate column"
                          if not _ctrl_found
                          else f"STILL PRESENT: {sorted(set(_ctrl_found))[:5]}")))

    tr = summary.get("tier_reachability") or {}
    if tr:
        checks.append(("Every declared evidence tier is reachable",
                       not tr.get("unreachable"),
                       f"unreachable: {tr.get('unreachable')}"
                       if tr.get("unreachable") else "all tiers reachable"))

    # Regression tests for the defects fixed in the latest review round.
    # Every one of those defects ran to completion and produced plausible
    # output, so a clean execution proves nothing on its own.
    rt = summary.get("regression_tests") or {}
    if rt:
        checks.append(("Regression tests for reviewed defects",
                       rt.get("n_passed") == rt.get("n_tests"),
                       f"{rt.get('n_passed')}/{rt.get('n_tests')} passed"
                       + (f"; FAILED: {[f['test'] for f in rt.get('failed', [])]}"
                          if rt.get("failed") else "")))

    # Calibration must be reported against the no-skill reference, and the
    # score must not be described as a probability when its skill is negative.
    cal = summary.get("calibration") or {}
    if cal:
        bss = cal.get("brier_skill_score")
        checks.append(("Score is honestly labelled (rank vs probability)",
                       (bss is not None)
                       and (bss >= 0) == bool(cal.get(
                           "scores_are_calibrated_probabilities")),
                       f"Brier {cal.get('brier'):.4f} vs no-skill "
                       f"{cal.get('brier_noskill_prevalence_rule'):.4f} "
                       f"(skill {bss:+.2f}); "
                       f"{cal.get('over_prediction_factor', float('nan')):.1f}x "
                       f"over-prediction -> treated as a RANKING"))
        if bss is not None and bss < 0:
            warn.append("Ensemble scores are NOT calibrated probabilities "
                        "(negative Brier skill). Report ranks and percentiles, "
                        "never 'probability of being a viable target'.")

    # Shortlist exports must match the sizes the summary claims.
    for ctx, n_claimed in (summary.get("shortlists_by_context") or {}).items():
        fp = DIRS["tables"] / f"table2b_{ctx}_shortlist.csv"
        if fp.exists():
            n_file = len(pd.read_csv(fp))
            checks.append((f"{ctx} shortlist exported complete",
                           n_file == n_claimed,
                           f"summary {n_claimed}, file {n_file}"))

    # The architecture-prior ablation, when it ran.
    ap = summary.get("architecture_prior_ablation") or {}
    if ap:
        d = ap.get("auc_attributable_to_architecture_prior")
        checks.append(("Ranking survives without protein-architecture features",
                       d is not None and d < 0.10,
                       f"biology-only {ap.get('biology_only_auc')} -> without "
                       f"architecture {ap.get('biology_minus_architecture_auc')} "
                       f"(delta {d:+.3f}); a large drop would mean the model "
                       f"detects druggable architecture, not prostate biology"))

    # BUG FIX (both external reviews, correct): also a literal True. It
    # printed the scorer name and a correlation while asserting neither. The
    # claim is that every exported score came from ONE frozen procedure, so
    # the check now requires the frozen-model manifest to exist, to record a
    # verified reload, and the out-of-fold/frozen agreement to be present
    # and high - a swapped or re-fitted model breaks at least one of those.
    if summary.get("final_scorer") == "frozen_ensemble_all_genes":
        rho = summary.get("oof_vs_frozen_spearman")
        _mf = DIRS["tables"] / "table22_frozen_model_manifest.csv"
        _prov_fail = []
        if not _mf.is_file():
            _prov_fail.append("frozen-model manifest missing")
        else:
            try:
                _m22 = pd.read_csv(_mf)
                if not len(_m22):
                    _prov_fail.append("frozen-model manifest empty")
                _rl = [c for c in _m22.columns if "reload" in c.lower()]
                if _rl and not _m22[_rl[0]].astype(str).str.lower().isin(
                        ["true", "1", "yes", "verified"]).all():
                    _prov_fail.append("reload verification not recorded")
            except Exception as _e:
                _prov_fail.append(f"manifest unreadable: {_e!r}")
        if rho is None:
            _prov_fail.append("no out-of-fold/frozen agreement recorded")
        elif rho < 0.80:
            _prov_fail.append(f"out-of-fold/frozen Spearman {rho:.3f} < 0.80")
        checks.append(("Final scores come from one identical procedure",
                       not _prov_fail,
                       (f"frozen ensemble scored every gene"
                        + (f"; Spearman {rho:.3f} vs the out-of-fold ranking"
                           if rho is not None else ""))
                       if not _prov_fail else "; ".join(_prov_fail)))

    # SELF-REVIEW: the regression suite's hardcoded-check scanner cannot
    # reach this function - verify() is defined in a later cell than the one
    # that runs the suite, so discovery found nothing for it and reported
    # success. The two literal-True checks that both external reviews found
    # here survived exactly that gap. verify() therefore scans ITSELF, where
    # it is certainly in scope, and the regression suite asserts its own
    # coverage separately so neither can assume the other did it.
    try:
        import ast as _ast_v, textwrap as _tw_v, inspect as _insp_v
        _vt = _ast_v.parse(_tw_v.dedent(_insp_v.getsource(verify)))
        _vsafe = set()
        for _p in _ast_v.walk(_vt):
            _g = (_p.body if isinstance(_p, _ast_v.ExceptHandler)
                  else _p.orelse if isinstance(_p, (_ast_v.For, _ast_v.While))
                  else [])
            for _st in _g:
                for _c in _ast_v.walk(_st):
                    _vsafe.add(id(_c))
        _vhard = []
        for _n in _ast_v.walk(_vt):
            if id(_n) in _vsafe:
                continue
            if (isinstance(_n, _ast_v.Tuple) and len(_n.elts) >= 2
                    and isinstance(_n.elts[1], _ast_v.Constant)
                    and _n.elts[1].value is True):
                _lab = (_n.elts[0].value
                        if isinstance(_n.elts[0], _ast_v.Constant)
                        else "<joined string>")
                _vhard.append(str(_lab)[:60])
        checks.append(("No post-run check is hardcoded to pass",
                       not _vhard,
                       f"unconditional check(s): {_vhard}" if _vhard
                       else "verify() self-scanned; no literal-True checks "
                            "in straight-line code"))
    except Exception as _e:
        checks.append(("No post-run check is hardcoded to pass", False,
                       f"self-scan failed: {_e!r}"))

    print("=" * 74)
    print("POST-RUN VERIFICATION")
    print("=" * 74)
    for name, ok, detail in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
        print(f"          {detail}")
    if warn:
        print("\\n  INTERPRETATION WARNINGS")
        for w in warn:
            print(f"    - {w}")
    print("=" * 74)
    if RUN_MODE == "DEMO":
        print("  REMINDER: DEMO mode. These checks validate the CODE, not biology.")
        print("  Set RUN_MODE = 'LIVE' and re-run for real results.")
        print("=" * 74)
    _ok_all = all(ok for _, ok, _ in checks)
    # Record the outcome in the run's release state. Both external reviews:
    # verify()'s result was returned and discarded, so the exporter could
    # not tell a verified run from an unverified one. The state machine now
    # carries it, and export_outputs() requires VERIFIED.
    try:
        _failed = [n for n, ok, _ in checks if not ok]
        mark_verified(_ok_all,
                      "post-run verification passed" if _ok_all
                      else f"{len(_failed)} post-run check(s) failed: "
                           f"{_failed[:6]}")
    except Exception as _e:
        print(f"  WARNING: could not record verification state: {_e!r}")
    return _ok_all


def execute_validated_analysis(download_archive=False, to_drive=False):
    """The ONE end-to-end path: analyse, verify, package, audit.

    NEW (both external reviews asked for this, and they are right about the
    gap). The notebook ran the pipeline in one cell, verify() in another and
    export_outputs() in a third. The packaged standalone runner called
    run_pipeline() and stopped. run_pipeline() does run both internal test
    suites and blocks on failure - that part was never missing - but it does
    NOT run the post-run verify() or the packaging and archive audit. So a
    script run produced analysis outputs that had never been through the
    release gate, while looking like the same thing the notebook produces.
    Two execution paths with different guarantees is precisely the
    "mechanism that cannot tell clean from absent" pattern, one level up.

    A data_only run gets its own completion path. It fits no model, so
    post-run verification has nothing to verify and release packaging has
    nothing to release; running them would either fail on absent results or,
    worse, pass and stamp a diagnostic as a validated release.

    Returns (summary, archive_path_or_None).
    """
    result = run_pipeline()
    summary = result[0] if isinstance(result, tuple) else result
    if not REPORTABLE:
        print("\n" + "=" * 78)
        print("DATA-ONLY RUN: post-run verification and release packaging "
              "SKIPPED by design")
        print("=" * 78)
        print("  A diagnostic run fits no model, so there is no scientific "
              "result to verify and")
        print("  nothing to release. The acquisition report, input "
              "checksums and run manifest are")
        print("  on disk. Re-run with RUN_PURPOSE=analysis for a scientific "
              "result.")
        return summary, None
    # BUG FIX: verify() itself can now also return None, meaning "ran but
    # could not assess anything because `summary` was missing" - a different
    # fact from "the verify() function itself is not defined", which is what
    # this block originally used None to mean. Collapsing both into one
    # `_v is None` check would blame a missing function for a missing
    # summary (or vice versa) and misdirect whoever reads the error. They
    # are told apart explicitly below; in this path summary already came
    # from a successful run_pipeline() call, so reaching the "could not
    # assess" branch would itself indicate a new defect worth reporting as
    # such rather than silently swallowing.
    if "verify" not in globals():
        # verify() lives in a notebook cell, not in the packaged sections.
        # Say so rather than silently treating its absence as a pass - an
        # absent check must never read as a satisfied one.
        raise RuntimeError(
            "Post-run verify() is not defined in this execution context, so "
            "the release gate cannot be evaluated. Refusing to package: an "
            "absent verification is not a passed verification.")
    _v = verify(summary)
    if _v is None:
        raise RuntimeError(
            "Release blocked: post-run verification could not assess this "
            "run (no summary was produced) even though run_pipeline() "
            "returned. The outputs remain on disk for diagnosis but are "
            "NOT a validated release.")
    if _v is not True:
        raise RuntimeError(
            "Release blocked: post-run verification failed. The outputs "
            "remain on disk for diagnosis but are NOT a validated release.")
    archive = export_outputs(download=download_archive, to_drive=to_drive)
    if verify_release_archive(archive) is not True:
        raise RuntimeError(
            f"Release blocked: the packaged archive {archive} failed its "
            f"own integrity audit.")
    print(f"\nVALIDATED RELEASE: {archive}")
    return summary, archive


# Execute
summary, ranked, shortlists, feat = run_pipeline()

SURFACEOME-WIDE MULTI-OMIC TARGET PRIORITISATION PIPELINE
Mode: DEMO | seed: 42 | profile: smoke | purpose: analysis
[0] Label and benchmark provenance audit
  LABEL_PROVENANCE_CSV not set - the label registry stays empty and the LIVE run will report it as unverified.
  label audit: 0/74 genes with a recorded trial ID (0%)
  inclusion rule fields still unresolved: discontinued_programmes, freeze_date
  ACTION REQUIRED before the LIVE run: populate LABEL_PROVENANCE from ClinicalTrials.gov, record the freeze date, and decide the discontinued-programme rule. Deposit the completed table as a supplementary file.
  label registry template written: outputs/06_logs/label_provenance_TEMPLATE.csv (80 gene(s) to curate). Fill it in, then set LABEL_PROVENANCE_CSV to its path and re-run.
  benchmark audit: 6 eligible benchmarks, 1 exclusion control(s) (KLK3)
  ACTION REQUIRED: extracellular-accessibility and modality evidence unverified for 6 benchmark(s): FOLH1, STEAP1, STEAP2, TMEFF2, PSCA, KLK2


/usr/lib/python3.13/zipfile/__init__.py:1686: UserWarning: Duplicate name: '05_tables/t.csv'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


    validated end to end (test): x.gz - later runs reuse it on its checksum

Regression tests for this review round
--------------------------------------
                                                                                                                               test  passed                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    detail
                                                                                  Candidate ranking matches the analyse

---
## Section 7 — Post-run verification

Run this after the pipeline. It re-checks the claims the pipeline makes about itself: that the filter unit tests passed, that the held-out benchmarks validated, that the observed AUC exceeds the permutation null, and that the survival models that were fitted had adequate events per variable. **A green check here is a check on internal consistency, not on biology.**

In [9]:
# =============================================================================
# SECTION 7 - POST-RUN VERIFICATION
# =============================================================================
# verify() is defined in the pipeline source (Section 7 of the flat module),
# not here, so the standalone runner and this notebook run the IDENTICAL
# post-run verification. This cell only invokes it.
# BUG FIX, and one I had already claimed to fix. A data_only run fits no
# model, so run_pipeline() returns Nones and there is nothing to verify.
# I added that skip to execute_validated_analysis(), pointed the STANDALONE
# runner at it, verified data_only through the runner, and reported the
# problem solved - without ever running the NOTEBOOK in data_only mode.
# The notebook does not call that controller: it runs the pipeline,
# verification, gallery and export in four separate cells, by design, so a
# user can inspect intermediate state. "Run all" with RUN_PURPOSE=data_only
# therefore still failed in three cells. The fix covered the path I had
# changed and not the path a Colab user actually takes.
if not REPORTABLE:
    print("data_only run: post-run verification SKIPPED by design - a "
          "diagnostic fits no model,\n"
          "so there is no scientific result to verify. Nothing from this "
          "run is reportable.")
    _verified = None
else:
    _verified = verify(globals().get('summary'))
    # BUG FIX: verify() returns None (not False) when the pipeline cell
    # above never ran to completion - distinct from False, which means the
    # checks ran and some failed. The two used to share one message
    # ("post-run verification failed"), which told a user whose pipeline
    # cell had crashed that *verification* was the problem, flatly
    # contradicting the "scroll up, that is the real error" text verify()
    # prints in that exact situation. Report each truthfully.
    if _verified is None:
        raise RuntimeError(
            "Release blocked: the pipeline did not complete, so post-run "
            "verification never ran - see the message above and the first "
            "traceback in the pipeline cell. This is NOT a verification "
            "failure; there is nothing yet to verify.")
    if _verified is not True:
        raise RuntimeError(
            "Release blocked: post-run verification failed. The outputs "
            "remain on disk for diagnosis but are NOT a validated release.")

POST-RUN VERIFICATION
  [PASS]  Filter unit-test report present
          filter_unit_tests.csv present
  [PASS]  Filter unit tests
          4/4 controls behaved as designed
  [PASS]  Held-out benchmarks in top decile
          3/6 (median percentile 91.6)
  [PASS]  Benchmark enrichment vs background
          permutation p = 2.00e-04
  [PASS]  Matched observed AUC exceeds the permutation null
          matched observed 0.926 vs null 95th pct 0.623 (p < 0.0323, B=30)
  [PASS]  ROC-AUC interval contains its own estimate
          ROC-AUC 0.941 in [0.917, 0.966]
  [PASS]  average precision interval contains its own estimate
          average precision 0.606 in [0.484, 0.720]
  [PASS]  Leakage ablation: biology alone still performs
          AUC 0.904 biology-only vs 0.920 with research-intensity features (drop +0.016)
  [PASS]  Engineered controls excluded from candidacy
          4 control(s) removed; none present in any candidate column
  [PASS]  Every declared evidence tier is reacha

---
## Section 8 — Output gallery

Every figure and table has already been displayed inline above, as each section produced it. This cell re-displays the complete set in one place, which is useful for scrolling back through a finished run or for screenshotting a panel.

In [10]:
# Re-display everything the run produced, in one place.
if not REPORTABLE:
    print("data_only run: no figures or analysis tables were produced. "
          "The acquisition report is at\n"
          "06_logs/data_only_acquisition_report.csv.")
else:
    show_gallery(figures=True, tables=True)

Gallery: 11 figures, 34 tables in /content/outputs


---
## Section 9 — Export before you close the tab

**Colab's filesystem is ephemeral.** The `outputs/` directory — every figure, every table, and `run_summary.json` — is deleted when the session ends. Run this cell to package it and pull it down.

Pass `to_drive=True` to copy the archive to your Google Drive instead of, or as well as, downloading it. For a long `manuscript` run, mount Drive *before* starting, so a disconnect part-way through does not cost you the whole run.

In [11]:
# Package outputs/ and download. to_drive is driven by the setup cell:
# when USE_DRIVE mounted Drive, the finished archive is copied there too.
# Outputs themselves were written to fast local disk, not through the Drive
# FUSE mount - only this one small, final write goes to Drive.
if not REPORTABLE:
    print("data_only run: release packaging SKIPPED by design - there is no "
          "validated analysis to\n"
          "release. The acquisition report, input checksums and run "
          "manifest are in outputs/06_logs/.\n"
          "Re-run with RUN_PURPOSE=analysis for a scientific result.")
    archive = None
else:
    archive = export_outputs(download=True,
                             to_drive=bool(globals().get("SYNC_ZIP_TO_DRIVE",
                                                         False)))

  1 unregistered file(s) NOT archived (no modification-time fallback - an unregistered file may belong to an earlier run in this session): ['06_logs/label_provenance_TEMPLATE.csv']
  source fingerprint: e3b0c44298fc1c14... (0 file(s))
  release package: 11 pinned dependency version(s), 1 source file(s) archived (including run_pipeline.py)
  archive verified: 59 entr(ies), all manifested, all checksums match, no duplicates, no stale files
  EXCLUDED 1 stale artifact(s) not produced by run 20261123T171033-01f789:
    - 06_logs/label_provenance_TEMPLATE.csv
  These remain on disk but are NOT in the archive: they belong to an earlier execution and would contradict this run.
Packaged /content/surfaceome_outputs_DEMO_smoke_20261123_1723.zip (7.0 MB)
  11 figures, 34 tables, plus run_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  browser download started


---
## Section 10 — What to inspect before writing anything up

Work through these in order. Each one has sunk a target-discovery paper at review.

| # | Check | Where | What a problem looks like |
|---|---|---|---|
| 1 | Positive-label provenance | `[0]` output | Any gene without a trial ID is an unaudited label, and labels are the model's entire supervision |
| 2 | Benchmark provenance and freeze date | `[0]` output | Unverified accessibility or modality evidence; an unset freeze date invites post-hoc revision |
| 3 | Gene-identifier mapping rate | `[1.3]` output | Below ~85% retention means harmonisation failed, not that biology is quiet |
| 4 | Expression scale decisions | `scale inferred as …` | A wrong call makes every fold-change wrong by an exponent while still looking plausible |
| 5 | Which DE model ran | `de_method_used` | Welch on abundance is a fallback; with counts, the NB/DESeq2 path is primary |
| 6 | Filter unit tests | Table in `[2.6]` | Any FAIL means the filter logic is inverted |
| 7 | Biological screen vs strict four-gate | `[2.6]` output | A large gap means reagent availability was doing much of the filtering |
| 8 | Positives actually matched | `[3.1]` output | Far fewer than declared means symbol mismatch, not biology |
| 9 | Leakage ablation gap | Fig 10 | A large drop without research-intensity features means the model learned fame |
| 10 | Observed AUC vs permutation null | Fig 4C | Observed near the null 95th percentile means no signal |
| 11 | Held-out benchmark ranks | Table 4 | PSMA outside the top decile triggers the documented failure investigation |
| 12 | Baseline comparison | Fig 11, Table 13 | If the ensemble loses, report the simplest adequate method |
| 13 | Ranking stability | Table 10 | Median ρ < 0.8 means report rank intervals, never a point-ranked top-20 |
| 14 | External transfer | Table 11 | Not run = not validated. Say so rather than implying otherwise |
| 15 | Single-cell coverage | Table 12 | Not run = heterogeneity unassessed, which is the study's own stated motivation |
| 16 | Survival events per variable | Table 6 | `min_detectable_HR_3group_logrank_as_run` above ~2 means the subgroup is uninformative; median follow-up is the reverse-KM estimate |
| 17 | Proportional hazards | Table 8 | `ph_assumption_met = False` means the HR is a time-averaged artefact |
| 18 | Coefficient signs | Fig 9B | A backwards sign is usually collinearity — read importance at group level |

---
## Section 11 — Modality-specific experimental follow-up

The pipeline's output is a prioritised hypothesis list. What the computational evidence cannot establish, and what each gap requires:

| Unresolved question | Required experiment | Why the pipeline cannot answer it |
|---|---|---|
| Is the antigen accessible on intact cells? | Flow cytometry on viable tumour cells | Transcript and total-protein evidence both include intracellular pools |
| How many copies per cell? | Quantitative flow / receptor density | Abundance ranking is not absolute density, and ADC efficacy is density-dependent |
| Does it internalise, and how fast? | Trafficking assay with a labelled binder | Expression says nothing about internalisation kinetics |
| Is the epitope reachable in tissue? | IHC plus binder validation on tissue | Topology annotation is sequence-derived, not structural |
| Is expression uniform across malignant cells? | Single-cell or spatial profiling | Bulk RNA averages over exactly the heterogeneity the study targets |
| Rare normal-cell expression? | Single-cell normal atlas | Bulk GTEx dilutes a strong signal in a small cell population |

**Select 5–10 candidates** from Tier 1 and Tier 2 of the novel shortlist, and match the assay to the intended modality — an ADC programme needs internalisation data a radioligand programme does not, and a CAR-T programme needs density and normal-cell data above all.

Record these as **proposed** experiments. The distinction between a planned experiment and a performed one is the difference between an honest proposal and an overstated result.